## We are going to forecast weekly


In [12]:
# ==============================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11B ONLY
# Creates and locks a verified read-only source snapshot.
# It does not build weekly data, define weekly features, select
# products, tune models, open the protected target vault, or load
# any fitted model binary.
# ==============================================================
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import stat
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd

try:
    from zoneinfo import ZoneInfo
except ImportError:
    ZoneInfo = None


# ----------------------------- paths -----------------------------
PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
PREP_ROOT = EDEN_ROOT / "forceast_preparation"  # deliberate historical spelling
MODELLING_ROOT = EDEN_ROOT / "modelling"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"
SNAPSHOT_ROOT = EXT_ROOT / "01_source_snapshot"
MEMORY_ROOT = EXT_ROOT / "00_project_memory"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
LOG_ROOT = EXT_ROOT / "12_logs"

MODEL_DIR = (
    MODELLING_ROOT
    / "05_final_refit_and_prediction"
    / "10B2_final_refit_and_predictions"
)

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = (
    NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))
    if ZoneInfo is not None
    else NOW_UTC
)
STAMP = NOW_UTC.strftime("%Y%m%dT%H%M%SZ")
STAGE_ROOT = LOG_ROOT / f".11B_staging_{STAMP}"

PART11A_LOCK = CHECKPOINT_ROOT / "11A_initial_setup_lock.json"
PART11A_LOCK_SHA = CHECKPOINT_ROOT / "11A_initial_setup_lock.sha256"
PART11A_CHECKPOINT = CHECKPOINT_ROOT / "11A_checkpoint.json"
PART11A_CHECKPOINT_SHA = CHECKPOINT_ROOT / "11A_checkpoint.sha256"

PART11B_LOCK = SNAPSHOT_ROOT / "manifests" / "11B_weekly_source_snapshot_lock.json"
PART11B_LOCK_SHA = SNAPSHOT_ROOT / "manifests" / "11B_weekly_source_snapshot_lock.sha256"
PART11B_CHECKPOINT = CHECKPOINT_ROOT / "11B_checkpoint.json"
PART11B_CHECKPOINT_SHA = CHECKPOINT_ROOT / "11B_checkpoint.sha256"

PRIMARY_SOURCE_EXPECTED = (
    EDEN_ROOT / "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"
)

REQUIRED_DAILY_COLUMNS = {
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
}

PROTECTED_NAME_PATTERNS = (
    "target_vault",
    "protected_target",
    "reserved_final_test_target",
)

DISALLOWED_PREDICTION_TARGET_COLUMNS = {
    "TotalDemand",
    "ActualTotalDemand",
    "ActualDemand",
    "ObservedTotalDemand",
    "y_true",
    "Actual",
}

EXPECTED_PRIMARY_PROFILE = {
    "Rows": 25_405,
    "Columns": 38,
    "OperatingDates": 245,
    "ProductIDs": 227,
    "ProductNames": 218,
    "DateMin": "2025-04-01",
    "DateMax": "2026-03-30",
    "ObservedRows": 15_138,
    "ZeroDemandRows": 10_267,
    "NormalDemandUnits": 114_186,
    "BulkDemandUnits": 1_972,
    "TotalDemandUnits": 116_158,
    "DuplicateProductDateRows": 0,
    "DemandComponentMismatches": 0,
}

EXPECTED_STANDARD_FINAL_START = "2026-03-02"
EXPECTED_STANDARD_FINAL_END = "2026-03-30"


# ---------------------------- helpers ----------------------------
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> Optional[str]:
    if not path.is_file():
        return None
    match = re.search(
        r"\b[a-fA-F0-9]{64}\b",
        path.read_text(encoding="utf-8", errors="replace"),
    )
    return match.group(0).lower() if match else None


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{STAMP}")
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_write_json(path: Path, payload: Any) -> None:
    atomic_write_text(
        path,
        json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n",
    )


def atomic_write_csv(path: Path, dataframe: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{STAMP}")
    try:
        dataframe.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def protected_path(path: Path) -> bool:
    normalized = str(path).lower()
    return any(pattern in normalized for pattern in PROTECTED_NAME_PATTERNS)


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def require_directory(path: Path, label: str) -> None:
    if not path.is_dir():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def safe_read_csv(path: Path, label: str) -> pd.DataFrame:
    if protected_path(path):
        raise PermissionError(f"Protected target path cannot be opened in Part 11B: {path}")
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as exc:
        raise RuntimeError(
            f"Could not read {label}:\n{path}\n{type(exc).__name__}: {exc}"
        ) from exc


def normalize_boolean(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    normalized = series.astype("string").str.strip().str.lower()
    converted = normalized.map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        }
    )
    missing = normalized.isna() | normalized.isin(["", "<na>", "nan", "none"])
    invalid = converted.isna() & ~missing
    if invalid.any():
        values = series.loc[invalid].drop_duplicates().tolist()
        raise ValueError(f"Unexpected Boolean values: {values}")
    return converted.astype("boolean")


def markdown_table(dataframe: pd.DataFrame, max_rows: int = 30) -> str:
    if dataframe is None or dataframe.empty:
        return "_No rows._"
    display_df = dataframe.head(max_rows).fillna("")

    def escape(value: Any) -> str:
        return str(value).replace("|", "\\|").replace("\n", "<br>")

    lines = [
        "| " + " | ".join(escape(column) for column in display_df.columns) + " |",
        "| " + " | ".join("---" for _ in display_df.columns) + " |",
    ]
    for _, row in display_df.iterrows():
        lines.append(
            "| "
            + " | ".join(escape(row[column]) for column in display_df.columns)
            + " |"
        )
    if len(dataframe) > max_rows:
        lines.append(f"\n_Showing first {max_rows} of {len(dataframe)} rows._")
    return "\n".join(lines)


def upsert_markdown_section(
    path: Path,
    section_id: str,
    section_title: str,
    section_body: str,
) -> None:
    start_marker = f"<!-- START:{section_id} -->"
    end_marker = f"<!-- END:{section_id} -->"
    section = (
        f"{start_marker}\n"
        f"## {section_title}\n\n"
        f"{section_body.strip()}\n"
        f"{end_marker}\n"
    )

    existing = path.read_text(encoding="utf-8") if path.exists() else ""
    if start_marker in existing and end_marker in existing:
        start = existing.index(start_marker)
        end = existing.index(end_marker) + len(end_marker)
        updated = existing[:start] + section.rstrip() + existing[end:]
    else:
        updated = existing.rstrip() + "\n\n" + section
    atomic_write_text(path, updated.strip() + "\n")


def update_workflow_status(path: Path, new_status: str) -> None:
    existing = path.read_text(encoding="utf-8")
    pattern = re.compile(r"(^\|\s*11B\s*\|.*?\|\s*)([^|]+)(\|\s*$)", re.MULTILINE)
    updated, count = pattern.subn(
        lambda match: match.group(1) + f"`{new_status}` " + match.group(3),
        existing,
        count=1,
    )
    if count != 1:
        raise RuntimeError("Could not update the Part 11B status row in WORKFLOW.md")
    atomic_write_text(path, updated)


def append_chat_index_row(path: Path, row: str) -> None:
    existing = path.read_text(encoding="utf-8") if path.exists() else "# CHAT INDEX\n"
    if "| 11B |" not in existing:
        existing = existing.rstrip() + "\n" + row.rstrip() + "\n"
    atomic_write_text(path, existing)


def stage_copy(source: Path, staged_destination: Path) -> dict[str, Any]:
    if protected_path(source):
        raise PermissionError(f"Protected target cannot be copied: {source}")
    if source.suffix.lower() == ".joblib":
        raise PermissionError(f"Model binary cannot be copied in Part 11B: {source}")
    require_file(source, "source file")

    staged_destination.parent.mkdir(parents=True, exist_ok=True)
    if staged_destination.exists():
        raise FileExistsError(f"Staging overwrite guard: {staged_destination}")

    source_hash = sha256_file(source)
    shutil.copy2(source, staged_destination)
    staged_hash = sha256_file(staged_destination)
    if staged_hash != source_hash:
        raise IOError(f"Staged copy hash mismatch: {source}")

    return {
        "SourcePath": str(source),
        "StagedPath": str(staged_destination),
        "Bytes": int(source.stat().st_size),
        "SourceSHA256": source_hash,
        "StagedSHA256": staged_hash,
        "HashMatch": True,
    }


def make_read_only(path: Path) -> None:
    path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def file_is_read_only(path: Path) -> bool:
    return (path.stat().st_mode & 0o222) == 0


def extract_manifest_matches(
    manifest_paths: Iterable[Path],
    source_path: Path,
) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    source_resolved = str(source_path.resolve())
    source_name = source_path.name

    for manifest_path in manifest_paths:
        if not manifest_path.is_file() or manifest_path.suffix.lower() != ".csv":
            continue
        try:
            manifest_df = pd.read_csv(manifest_path, low_memory=False)
        except Exception:
            continue

        sha_columns = [column for column in manifest_df.columns if column.lower() == "sha256"]
        path_columns = [
            column
            for column in manifest_df.columns
            if column.lower() in {"filepath", "sourcepath", "absolutepath", "path"}
        ]
        name_columns = [
            column
            for column in manifest_df.columns
            if column.lower() in {"filename", "file_name"}
        ]
        if not sha_columns or not (path_columns or name_columns):
            continue

        sha_column = sha_columns[0]
        match_mask = pd.Series(False, index=manifest_df.index)
        for column in path_columns:
            values = manifest_df[column].astype("string")
            match_mask |= values.eq(str(source_path)) | values.eq(source_resolved)
        for column in name_columns:
            match_mask |= manifest_df[column].astype("string").eq(source_name)

        for _, row in manifest_df.loc[match_mask].iterrows():
            records.append(
                {
                    "ManifestPath": str(manifest_path),
                    "MatchedBy": "path_or_filename",
                    "RecordedSHA256": str(row[sha_column]).strip().lower(),
                }
            )

    return pd.DataFrame(records)


def csv_schema_record(role: str, path: Path, dataframe: pd.DataFrame) -> dict[str, Any]:
    date_like_columns = [
        column for column in dataframe.columns if "date" in column.lower()
    ]
    return {
        "Role": role,
        "Path": str(path),
        "Rows": int(len(dataframe)),
        "Columns": int(dataframe.shape[1]),
        "ColumnNames": "|".join(map(str, dataframe.columns)),
        "DateLikeColumns": "|".join(date_like_columns),
        "DuplicateRows": int(dataframe.duplicated().sum()),
        "ProtectedPath": protected_path(path),
    }


# ----------------------------- preflight -----------------------------
for required_directory, label in [
    (EDEN_ROOT, "Eden dataset root"),
    (PREP_ROOT, "forecast-preparation root"),
    (MODELLING_ROOT, "modelling root"),
    (EXT_ROOT, "weekly extension root"),
    (SNAPSHOT_ROOT, "source snapshot root"),
    (MEMORY_ROOT, "project-memory root"),
    (CHECKPOINT_ROOT, "checkpoint root"),
    (LOG_ROOT, "log root"),
]:
    require_directory(required_directory, label)

if PART11B_LOCK.exists() or PART11B_LOCK_SHA.exists():
    existing_status = "UNKNOWN"
    if PART11B_LOCK.is_file():
        try:
            existing_status = json.loads(
                PART11B_LOCK.read_text(encoding="utf-8")
            ).get("Status", "UNKNOWN")
        except Exception:
            existing_status = "UNREADABLE_LOCK"
    raise FileExistsError(
        "Overwrite guard: Part 11B is already locked.\n"
        f"Lock: {PART11B_LOCK}\n"
        f"Status: {existing_status}"
    )

if STAGE_ROOT.exists():
    raise FileExistsError(f"Staging overwrite guard: {STAGE_ROOT}")

for path, label in [
    (PART11A_LOCK, "Part 11A lock"),
    (PART11A_LOCK_SHA, "Part 11A lock sidecar"),
    (PART11A_CHECKPOINT, "Part 11A checkpoint"),
    (PART11A_CHECKPOINT_SHA, "Part 11A checkpoint sidecar"),
]:
    require_file(path, label)

if read_sidecar_hash(PART11A_LOCK_SHA) != sha256_file(PART11A_LOCK):
    raise AssertionError("Part 11A lock sidecar does not match the lock file")
if read_sidecar_hash(PART11A_CHECKPOINT_SHA) != sha256_file(PART11A_CHECKPOINT):
    raise AssertionError("Part 11A checkpoint sidecar does not match the checkpoint")

part11a_lock_payload = json.loads(PART11A_LOCK.read_text(encoding="utf-8"))
part11a_checkpoint_payload = json.loads(
    PART11A_CHECKPOINT.read_text(encoding="utf-8")
)

if part11a_lock_payload.get("Status") != "PART_11A_COMPLETED_READY_FOR_11B":
    raise RuntimeError(
        "Part 11A lock does not permit Part 11B.\n"
        f"Recorded status: {part11a_lock_payload.get('Status')}"
    )
if part11a_lock_payload.get("ReadyForPart11B") is not True:
    raise RuntimeError("Part 11A lock has ReadyForPart11B != true")
if part11a_checkpoint_payload.get("ReadyForPart11B") is not True:
    raise RuntimeError("Part 11A checkpoint has ReadyForPart11B != true")

safety_11a = part11a_lock_payload.get("SafetyAssertions", {})
for safety_key in [
    "ProtectedTargetOpened",
    "ProtectedTargetCopied",
    "ModelsLoaded",
    "ModelsCopied",
    "ModelsRetrained",
    "PredictionsChanged",
    "WeeklyDatasetBuilt",
]:
    if safety_11a.get(safety_key) is not False:
        raise RuntimeError(f"Part 11A safety assertion is not false: {safety_key}")

preferred_from_11a = Path(
    part11a_lock_payload.get("WeeklySource", {}).get("PreferredCandidate", "")
)
if preferred_from_11a != PRIMARY_SOURCE_EXPECTED:
    raise RuntimeError(
        "The Part 11A preferred source does not match the authoritative source.\n"
        f"Part 11A: {preferred_from_11a}\n"
        f"Expected: {PRIMARY_SOURCE_EXPECTED}"
    )

# Required 11A inventories and previously copied contracts.
MANIFEST_DIR = SNAPSHOT_ROOT / "manifests"
for path, label in [
    (MANIFEST_DIR / "11A_source_inventory.csv", "11A source inventory"),
    (MANIFEST_DIR / "11A_weekly_source_candidates.csv", "11A source candidates"),
    (MANIFEST_DIR / "11A_snapshot_copy_audit.csv", "11A copy audit"),
    (MANIFEST_DIR / "11A_source_hash_manifest.csv", "11A hash manifest"),
    (MANIFEST_DIR / "10B2_artifact_hash_manifest.csv", "daily artifact hash manifest"),
    (SNAPSHOT_ROOT / "contracts" / "13_step3_split_policy_parameters.csv", "split policy parameters"),
    (SNAPSHOT_ROOT / "contracts" / "15_step3_manifest_boundary_validation.csv", "boundary validation"),
    (SNAPSHOT_ROOT / "contracts" / "16_step3_final_test_isolation_audit.csv", "final-test isolation audit"),
]:
    require_file(path, label)

candidate_df = pd.read_csv(
    MANIFEST_DIR / "11A_weekly_source_candidates.csv",
    low_memory=False,
)
if candidate_df.empty or str(candidate_df.iloc[0]["CandidatePath"]) != str(PRIMARY_SOURCE_EXPECTED):
    raise RuntimeError("The top-ranked 11A source candidate is not the authoritative source")


# ---------------- inspect and resolve source schemas ----------------
# Part 11B requires the canonical daily source, frozen split contracts and
# locked prediction references. The four Step 5 companion CSVs are useful
# continuity files, but the completed notebooks show they were planned outputs
# rather than proving that every companion still exists at the project root.
# They are therefore discovered by exact filename and copied only when a
# unique safe version can be identified.
SOURCE_SPECIFICATIONS = [
    {
        "Role": "PRIMARY_DAILY_DEMAND_SOURCE",
        "ExpectedPath": PRIMARY_SOURCE_EXPECTED,
        "ExactFilename": PRIMARY_SOURCE_EXPECTED.name,
        "SearchRoots": [EDEN_ROOT],
        "Destination": Path("data") / PRIMARY_SOURCE_EXPECTED.name,
        "Required": True,
        "RequiredPriorHash": False,
    },
    {
        "Role": "STEP5_PRODUCT_METADATA",
        "ExpectedPath": EDEN_ROOT / "UL_EDEN_step5_final_product_metadata.csv",
        "ExactFilename": "UL_EDEN_step5_final_product_metadata.csv",
        "SearchRoots": [EDEN_ROOT],
        "Destination": Path("data") / "UL_EDEN_step5_final_product_metadata.csv",
        "Required": False,
        "RequiredPriorHash": False,
    },
    {
        "Role": "STEP5_OPERATING_CALENDAR",
        "ExpectedPath": EDEN_ROOT / "UL_EDEN_step5_operating_calendar.csv",
        "ExactFilename": "UL_EDEN_step5_operating_calendar.csv",
        "SearchRoots": [EDEN_ROOT],
        "Destination": Path("data") / "UL_EDEN_step5_operating_calendar.csv",
        "Required": False,
        "RequiredPriorHash": False,
    },
    {
        "Role": "STEP5_OBSERVED_DAILY_AUDIT",
        "ExpectedPath": EDEN_ROOT / "UL_EDEN_step5_observed_daily_demand_audit.csv",
        "ExactFilename": "UL_EDEN_step5_observed_daily_demand_audit.csv",
        "SearchRoots": [EDEN_ROOT],
        "Destination": Path("data") / "UL_EDEN_step5_observed_daily_demand_audit.csv",
        "Required": False,
        "RequiredPriorHash": False,
    },
    {
        "Role": "STEP5_FINAL_VALIDATION_SUMMARY",
        "ExpectedPath": EDEN_ROOT / "UL_EDEN_step5_final_validation_summary.csv",
        "ExactFilename": "UL_EDEN_step5_final_validation_summary.csv",
        "SearchRoots": [EDEN_ROOT],
        "Destination": Path("data") / "UL_EDEN_step5_final_validation_summary.csv",
        "Required": False,
        "RequiredPriorHash": False,
    },
    {
        "Role": "DAILY_OPERATING_CALENDAR",
        "ExpectedPath": PREP_ROOT / "13_step3_operating_calendar.csv",
        "ExactFilename": "13_step3_operating_calendar.csv",
        "SearchRoots": [PREP_ROOT, EDEN_ROOT],
        "Destination": Path("contracts") / "13_step3_operating_calendar.csv",
        "Required": True,
        "RequiredPriorHash": False,
    },
    {
        "Role": "DAILY_EVALUATION_WINDOW_CALENDAR",
        "ExpectedPath": PREP_ROOT / "13_step3_global_evaluation_window_calendar.csv",
        "ExactFilename": "13_step3_global_evaluation_window_calendar.csv",
        "SearchRoots": [PREP_ROOT, EDEN_ROOT],
        "Destination": Path("contracts") / "13_step3_global_evaluation_window_calendar.csv",
        "Required": True,
        "RequiredPriorHash": False,
    },
    {
        "Role": "DAILY_WINDOW_SPLIT_SUMMARY",
        "ExpectedPath": PREP_ROOT / "15_step3_window_split_summary.csv",
        "ExactFilename": "15_step3_window_split_summary.csv",
        "SearchRoots": [PREP_ROOT, EDEN_ROOT],
        "Destination": Path("contracts") / "15_step3_window_split_summary.csv",
        "Required": True,
        "RequiredPriorHash": False,
    },
    {
        "Role": "LOCKED_DAILY_COMPONENT_PREDICTIONS",
        "ExpectedPath": MODEL_DIR / "10B2_scoring_component_predictions.csv",
        "ExactFilename": "10B2_scoring_component_predictions.csv",
        "SearchRoots": [MODEL_DIR, MODELLING_ROOT],
        "Destination": Path("daily_model_reference") / "10B2_scoring_component_predictions.csv",
        "Required": True,
        "RequiredPriorHash": True,
    },
    {
        "Role": "LOCKED_DAILY_PRODUCT_PREDICTIONS",
        "ExpectedPath": MODEL_DIR / "10B2_final_product_predictions.csv",
        "ExactFilename": "10B2_final_product_predictions.csv",
        "SearchRoots": [MODEL_DIR, MODELLING_ROOT],
        "Destination": Path("daily_model_reference") / "10B2_final_product_predictions.csv",
        "Required": True,
        "RequiredPriorHash": True,
    },
    {
        "Role": "LOCKED_DAILY_AGGREGATE_PREDICTIONS",
        "ExpectedPath": MODEL_DIR / "10B2_final_aggregate_predictions.csv",
        "ExactFilename": "10B2_final_aggregate_predictions.csv",
        "SearchRoots": [MODEL_DIR, MODELLING_ROOT],
        "Destination": Path("daily_model_reference") / "10B2_final_aggregate_predictions.csv",
        "Required": True,
        "RequiredPriorHash": True,
    },
]


def _safe_discovery_candidates(specification: dict[str, Any]) -> list[Path]:
    expected_path = Path(specification["ExpectedPath"])
    exact_filename = str(specification["ExactFilename"])
    candidates: list[Path] = []

    if expected_path.is_file():
        candidates.append(expected_path)

    for search_root in specification["SearchRoots"]:
        search_root = Path(search_root)
        if not search_root.is_dir():
            continue
        for candidate in search_root.rglob(exact_filename):
            if not candidate.is_file():
                continue
            try:
                candidate.resolve().relative_to(EXT_ROOT.resolve())
                continue
            except ValueError:
                pass
            if protected_path(candidate) or candidate.suffix.lower() == ".joblib":
                continue
            candidates.append(candidate)

    unique_candidates: dict[str, Path] = {}
    for candidate in candidates:
        unique_candidates[str(candidate.resolve())] = candidate
    return list(unique_candidates.values())


def _candidate_preference(path: Path, expected_path: Path) -> tuple[int, int, str]:
    normalized = str(path).lower()
    penalty = 0
    if path == expected_path:
        penalty -= 1000
    for discouraged_token in [
        "superseded",
        "archive",
        "old",
        "backup",
        "demo_backups",
        ".trash",
    ]:
        if discouraged_token in normalized:
            penalty += 100
    try:
        depth = len(path.resolve().relative_to(EDEN_ROOT.resolve()).parts)
    except ValueError:
        depth = len(path.parts) + 50
    return penalty, depth, str(path)


source_resolution_records: list[dict[str, Any]] = []
SOURCE_PLAN: list[dict[str, Any]] = []

for specification in SOURCE_SPECIFICATIONS:
    role = str(specification["Role"])
    expected_path = Path(specification["ExpectedPath"])
    required = bool(specification["Required"])
    candidates = _safe_discovery_candidates(specification)

    candidate_hashes = {
        str(candidate): sha256_file(candidate)
        for candidate in candidates
    }
    unique_hashes = sorted(set(candidate_hashes.values()))
    selected_source: Optional[Path] = None
    resolution_status: str

    if not candidates:
        resolution_status = (
            "MISSING_REQUIRED" if required else "OPTIONAL_NOT_FOUND_SKIPPED"
        )
    elif len(unique_hashes) > 1:
        if required:
            resolution_status = "AMBIGUOUS_REQUIRED_DIFFERENT_HASHES"
        else:
            resolution_status = "OPTIONAL_AMBIGUOUS_DIFFERENT_HASHES_SKIPPED"
    else:
        selected_source = sorted(
            candidates,
            key=lambda path: _candidate_preference(path, expected_path),
        )[0]
        resolution_status = (
            "EXPECTED_PATH_CONFIRMED"
            if selected_source == expected_path
            else "EXACT_FILENAME_DISCOVERED_UNIQUE_HASH"
        )

    source_resolution_records.append(
        {
            "Role": role,
            "Required": required,
            "ExpectedPath": str(expected_path),
            "ExactFilename": specification["ExactFilename"],
            "CandidatesFound": len(candidates),
            "UniqueCandidateHashes": len(unique_hashes),
            "CandidatePaths": " | ".join(sorted(map(str, candidates))),
            "CandidateSHA256Values": " | ".join(unique_hashes),
            "SelectedSource": str(selected_source) if selected_source else None,
            "SelectedSHA256": (
                sha256_file(selected_source) if selected_source else None
            ),
            "ResolutionStatus": resolution_status,
        }
    )

    if required and selected_source is None:
        details = "\n".join(
            f"- {path}: {candidate_hashes[str(path)]}"
            for path in sorted(candidates, key=str)
        )
        if not details:
            details = "- No exact-filename candidate was found outside the weekly extension."
        raise FileNotFoundError(
            f"Could not resolve required source {role}.\n"
            f"Expected path: {expected_path}\n"
            f"Exact filename: {specification['ExactFilename']}\n"
            f"Candidates:\n{details}"
        )

    if selected_source is not None:
        SOURCE_PLAN.append(
            {
                "Role": role,
                "Source": selected_source,
                "Destination": Path(specification["Destination"]),
                "Required": required,
                "RequiredPriorHash": bool(specification["RequiredPriorHash"]),
                "ResolutionStatus": resolution_status,
            }
        )

source_resolution_df = pd.DataFrame(source_resolution_records)

for item in SOURCE_PLAN:
    source = Path(item["Source"])
    destination = SNAPSHOT_ROOT / Path(item["Destination"])
    require_file(source, item["Role"])
    if protected_path(source):
        raise PermissionError(f"Protected target detected in source plan: {source}")
    if source.suffix.lower() == ".joblib":
        raise PermissionError(f"Model binary detected in source plan: {source}")
    if destination.exists():
        raise FileExistsError(f"Destination overwrite guard: {destination}")

required_roles = {
    str(specification["Role"])
    for specification in SOURCE_SPECIFICATIONS
    if specification["Required"]
}
resolved_roles = {str(item["Role"]) for item in SOURCE_PLAN}
missing_required_roles = sorted(required_roles - resolved_roles)
if missing_required_roles:
    raise RuntimeError(
        "Required Part 11B source roles were not resolved: "
        + ", ".join(missing_required_roles)
    )

optional_unavailable_roles = source_resolution_df.loc[
    (~source_resolution_df["Required"])
    & (source_resolution_df["SelectedSource"].isna()),
    "Role",
].astype(str).tolist()

# Full inspection is intentional for source-contract validation only.
primary_df = safe_read_csv(PRIMARY_SOURCE_EXPECTED, "primary daily demand source")
missing_required_columns = sorted(REQUIRED_DAILY_COLUMNS - set(primary_df.columns))
if missing_required_columns:
    raise ValueError(
        "Primary source is missing required columns:\n"
        + "\n".join(f"- {column}" for column in missing_required_columns)
    )

primary_df["Date"] = pd.to_datetime(primary_df["Date"], errors="raise").dt.normalize()
for demand_column in ["NormalDemand", "BulkDemand", "TotalDemand"]:
    primary_df[demand_column] = pd.to_numeric(
        primary_df[demand_column], errors="raise"
    )

if primary_df[["Date", "CanonicalProductID", "CanonicalProductName"]].isna().any().any():
    raise ValueError("Primary source contains missing date, product ID or product name")
if primary_df[["NormalDemand", "BulkDemand", "TotalDemand"]].isna().any().any():
    raise ValueError("Primary source contains missing demand values")
if (primary_df[["NormalDemand", "BulkDemand", "TotalDemand"]] < 0).any().any():
    raise ValueError("Primary source contains negative demand values")

observed_rows = (
    int(normalize_boolean(primary_df["IsObservedProductDate"]).fillna(False).sum())
    if "IsObservedProductDate" in primary_df.columns
    else np.nan
)
zero_flag_rows = (
    int(normalize_boolean(primary_df["IsZeroDemandRow"]).fillna(False).sum())
    if "IsZeroDemandRow" in primary_df.columns
    else int((primary_df["TotalDemand"] == 0).sum())
)

primary_actual_profile = {
    "Rows": int(len(primary_df)),
    "Columns": int(primary_df.shape[1]),
    "OperatingDates": int(primary_df["Date"].nunique()),
    "ProductIDs": int(primary_df["CanonicalProductID"].nunique()),
    "ProductNames": int(primary_df["CanonicalProductName"].nunique()),
    "DateMin": primary_df["Date"].min().date().isoformat(),
    "DateMax": primary_df["Date"].max().date().isoformat(),
    "ObservedRows": observed_rows,
    "ZeroDemandRows": zero_flag_rows,
    "NormalDemandUnits": float(primary_df["NormalDemand"].sum()),
    "BulkDemandUnits": float(primary_df["BulkDemand"].sum()),
    "TotalDemandUnits": float(primary_df["TotalDemand"].sum()),
    "DuplicateProductDateRows": int(
        primary_df.duplicated(["Date", "CanonicalProductID"], keep=False).sum()
    ),
    "DemandComponentMismatches": int(
        (~np.isclose(
            primary_df["TotalDemand"].to_numpy(dtype=float),
            (
                primary_df["NormalDemand"].to_numpy(dtype=float)
                + primary_df["BulkDemand"].to_numpy(dtype=float)
            ),
            rtol=0.0,
            atol=1e-9,
        )).sum()
    ),
}

primary_validation_records: list[dict[str, Any]] = []
for metric, expected in EXPECTED_PRIMARY_PROFILE.items():
    actual = primary_actual_profile[metric]
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        passed = bool(np.isclose(float(actual), float(expected), rtol=0.0, atol=1e-9))
    else:
        passed = actual == expected
    primary_validation_records.append(
        {
            "Metric": metric,
            "Expected": expected,
            "Actual": actual,
            "Passed": passed,
        }
    )

primary_validation_df = pd.DataFrame(primary_validation_records)
if not primary_validation_df["Passed"].all():
    failures = primary_validation_df.loc[~primary_validation_df["Passed"]]
    raise AssertionError(
        "Primary source contract validation failed:\n"
        + failures.to_string(index=False)
    )

# Inspect every planned CSV before copying.
schema_records: list[dict[str, Any]] = []
loaded_sources: dict[str, pd.DataFrame] = {
    "PRIMARY_DAILY_DEMAND_SOURCE": primary_df
}

for item in SOURCE_PLAN:
    role = str(item["Role"])
    source = Path(item["Source"])
    dataframe = loaded_sources.get(role)
    if dataframe is None:
        dataframe = safe_read_csv(source, role)
        loaded_sources[role] = dataframe
    schema_records.append(csv_schema_record(role, source, dataframe))

schema_audit_df = pd.DataFrame(schema_records)

# Supporting source checks.
# Step 5 companion files are validated when available. Their absence is logged
# but does not block Part 11B because the canonical panel and frozen split
# contracts contain the information required to begin weekly aggregation.
metadata_df = loaded_sources.get("STEP5_PRODUCT_METADATA")
calendar_df = loaded_sources.get("STEP5_OPERATING_CALENDAR")
daily_operating_calendar_df = loaded_sources["DAILY_OPERATING_CALENDAR"]
window_calendar_df = loaded_sources["DAILY_EVALUATION_WINDOW_CALENDAR"]
window_split_summary_df = loaded_sources["DAILY_WINDOW_SPLIT_SUMMARY"]
component_predictions_df = loaded_sources["LOCKED_DAILY_COMPONENT_PREDICTIONS"]
product_predictions_df = loaded_sources["LOCKED_DAILY_PRODUCT_PREDICTIONS"]
aggregate_predictions_df = loaded_sources["LOCKED_DAILY_AGGREGATE_PREDICTIONS"]

if metadata_df is not None and len(metadata_df) == 0:
    raise AssertionError("Discovered Step 5 product metadata is empty")
if calendar_df is not None and len(calendar_df) != 245:
    raise AssertionError(
        f"Discovered Step 5 operating calendar rows expected 245, found {len(calendar_df)}"
    )
if len(daily_operating_calendar_df) != 245:
    raise AssertionError(
        f"Daily split operating calendar rows expected 245, found {len(daily_operating_calendar_df)}"
    )
if len(window_calendar_df) != 4:
    raise AssertionError(f"Global evaluation window rows expected 4, found {len(window_calendar_df)}")
if len(window_split_summary_df) == 0:
    raise AssertionError("Daily window split summary is empty")

# Final-holdout governance comes from the frozen daily split calendar.
required_window_columns = {
    "WindowID",
    "WindowPurpose",
    "EvaluationStartDate",
    "EvaluationEndDate",
    "MayBeUsedForModelSelection",
    "ReservedFinalTest",
}
missing_window_columns = sorted(required_window_columns - set(window_calendar_df.columns))
if missing_window_columns:
    raise ValueError(
        "Daily window calendar is missing required columns:\n"
        + "\n".join(f"- {column}" for column in missing_window_columns)
    )

window_calendar_df["EvaluationStartDate"] = pd.to_datetime(
    window_calendar_df["EvaluationStartDate"], errors="raise"
).dt.normalize()
window_calendar_df["EvaluationEndDate"] = pd.to_datetime(
    window_calendar_df["EvaluationEndDate"], errors="raise"
).dt.normalize()
window_calendar_df["ReservedFinalTest"] = normalize_boolean(
    window_calendar_df["ReservedFinalTest"]
)
window_calendar_df["MayBeUsedForModelSelection"] = normalize_boolean(
    window_calendar_df["MayBeUsedForModelSelection"]
)

standard_final_rows = window_calendar_df.loc[
    window_calendar_df["WindowID"].astype(str).eq("STANDARD_FINAL_HOLDOUT")
].copy()
if len(standard_final_rows) != 1:
    raise AssertionError(
        f"Expected one STANDARD_FINAL_HOLDOUT row, found {len(standard_final_rows)}"
    )
standard_final_row = standard_final_rows.iloc[0]
standard_final_start = standard_final_row["EvaluationStartDate"].date().isoformat()
standard_final_end = standard_final_row["EvaluationEndDate"].date().isoformat()

if standard_final_start != EXPECTED_STANDARD_FINAL_START:
    raise AssertionError(
        f"Standard final start expected {EXPECTED_STANDARD_FINAL_START}, found {standard_final_start}"
    )
if standard_final_end != EXPECTED_STANDARD_FINAL_END:
    raise AssertionError(
        f"Standard final end expected {EXPECTED_STANDARD_FINAL_END}, found {standard_final_end}"
    )
if bool(standard_final_row["MayBeUsedForModelSelection"]):
    raise AssertionError("STANDARD_FINAL_HOLDOUT is incorrectly marked for model selection")
if not bool(standard_final_row["ReservedFinalTest"]):
    raise AssertionError("STANDARD_FINAL_HOLDOUT is not marked as reserved final test")

holdout_governance_df = pd.DataFrame(
    [
        {
            "SourceSystem": "COMPLETED_DAILY_FORECASTING_SYSTEM",
            "WindowID": "STANDARD_FINAL_HOLDOUT",
            "EvaluationStartDate": standard_final_start,
            "EvaluationEndDate": standard_final_end,
            "OpenedForDailyFinalEvaluation": True,
            "MayTuneWeeklyScopeOrModels": False,
            "WeeklyTuningCutoffDateExclusive": standard_final_start,
            "SnapshotContainsOpenedHoldoutRows": True,
            "Policy": (
                "Rows dated on or after the cutoff may be retained for traceability "
                "but must not determine weekly product scope, segments, features, "
                "models, hyperparameters or fallback rules."
            ),
        }
    ]
)

# Locked daily prediction checks. These files do not contain actual targets.
for role, dataframe, expected_rows, required_columns in [
    (
        "LOCKED_DAILY_COMPONENT_PREDICTIONS",
        component_predictions_df,
        4_150,
        {"Date", "CanonicalProductID", "CanonicalProductName", "FinalProductPrediction", "AggregateComponentPrediction"},
    ),
    (
        "LOCKED_DAILY_PRODUCT_PREDICTIONS",
        product_predictions_df,
        4_150,
        {"Date", "CanonicalProductID", "CanonicalProductName", "FinalProductPrediction"},
    ),
    (
        "LOCKED_DAILY_AGGREGATE_PREDICTIONS",
        aggregate_predictions_df,
        20,
        {"Date", "AggregateForecast", "ProductMethodBottomUpTotal"},
    ),
]:
    missing = sorted(required_columns - set(dataframe.columns))
    if missing:
        raise ValueError(f"{role} missing columns: {missing}")
    if len(dataframe) != expected_rows:
        raise AssertionError(f"{role} rows expected {expected_rows}, found {len(dataframe)}")
    forbidden = sorted(DISALLOWED_PREDICTION_TARGET_COLUMNS & set(dataframe.columns))
    if forbidden:
        raise PermissionError(f"{role} contains disallowed target columns: {forbidden}")

for dataframe, role in [
    (component_predictions_df, "component predictions"),
    (product_predictions_df, "product predictions"),
]:
    dataframe["Date"] = pd.to_datetime(dataframe["Date"], errors="raise").dt.normalize()
    duplicate_keys = int(
        dataframe.duplicated(["Date", "CanonicalProductID"], keep=False).sum()
    )
    if duplicate_keys != 0:
        raise AssertionError(f"{role} contains {duplicate_keys} duplicate product-date rows")
    if dataframe["CanonicalProductID"].nunique() != 218:
        raise AssertionError(
            f"{role} expected 218 products, found {dataframe['CanonicalProductID'].nunique()}"
        )
    if dataframe["Date"].nunique() != 20:
        raise AssertionError(
            f"{role} expected 20 dates, found {dataframe['Date'].nunique()}"
        )

aggregate_predictions_df["Date"] = pd.to_datetime(
    aggregate_predictions_df["Date"], errors="raise"
).dt.normalize()
if aggregate_predictions_df["Date"].nunique() != 20:
    raise AssertionError("Aggregate predictions do not contain exactly 20 unique dates")

prediction_profile_df = pd.DataFrame(
    [
        {
            "Role": "LOCKED_DAILY_COMPONENT_PREDICTIONS",
            "Rows": len(component_predictions_df),
            "Dates": component_predictions_df["Date"].nunique(),
            "Products": component_predictions_df["CanonicalProductID"].nunique(),
            "DuplicateProductDateRows": component_predictions_df.duplicated(
                ["Date", "CanonicalProductID"], keep=False
            ).sum(),
            "ContainsActualTargetColumn": bool(
                DISALLOWED_PREDICTION_TARGET_COLUMNS & set(component_predictions_df.columns)
            ),
        },
        {
            "Role": "LOCKED_DAILY_PRODUCT_PREDICTIONS",
            "Rows": len(product_predictions_df),
            "Dates": product_predictions_df["Date"].nunique(),
            "Products": product_predictions_df["CanonicalProductID"].nunique(),
            "DuplicateProductDateRows": product_predictions_df.duplicated(
                ["Date", "CanonicalProductID"], keep=False
            ).sum(),
            "ContainsActualTargetColumn": bool(
                DISALLOWED_PREDICTION_TARGET_COLUMNS & set(product_predictions_df.columns)
            ),
        },
        {
            "Role": "LOCKED_DAILY_AGGREGATE_PREDICTIONS",
            "Rows": len(aggregate_predictions_df),
            "Dates": aggregate_predictions_df["Date"].nunique(),
            "Products": np.nan,
            "DuplicateProductDateRows": np.nan,
            "ContainsActualTargetColumn": bool(
                DISALLOWED_PREDICTION_TARGET_COLUMNS & set(aggregate_predictions_df.columns)
            ),
        },
    ]
)


# ----------------------- prior-manifest cross-check -----------------------
prior_manifest_paths = sorted(MANIFEST_DIR.glob("*.csv"))
manifest_crosscheck_records: list[dict[str, Any]] = []

for item in SOURCE_PLAN:
    source = Path(item["Source"])
    actual_hash = sha256_file(source)
    matches = extract_manifest_matches(prior_manifest_paths, source)
    recorded_hashes = (
        sorted(matches["RecordedSHA256"].dropna().astype(str).str.lower().unique())
        if not matches.empty
        else []
    )
    conflicting = len(recorded_hashes) > 1
    matched = len(recorded_hashes) == 1 and recorded_hashes[0] == actual_hash
    required_prior_hash = bool(item["RequiredPriorHash"])

    if conflicting:
        raise AssertionError(f"Conflicting prior hashes found for {source}: {recorded_hashes}")
    if required_prior_hash and not matched:
        raise AssertionError(
            "A locked daily prediction file did not match its prior manifest.\n"
            f"File: {source}\n"
            f"Actual SHA256: {actual_hash}\n"
            f"Recorded hashes: {recorded_hashes}"
        )

    manifest_crosscheck_records.append(
        {
            "Role": item["Role"],
            "SourcePath": str(source),
            "ActualSHA256": actual_hash,
            "PriorManifestMatchCount": int(len(matches)),
            "PriorRecordedSHA256": recorded_hashes[0] if len(recorded_hashes) == 1 else None,
            "PriorHashRequired": required_prior_hash,
            "PriorHashMatched": matched,
            "ManifestEvidence": (
                "MATCHED"
                if matched
                else "NO_PRIOR_MATCH_NEW_11B_LOCK"
                if not recorded_hashes
                else "RECORDED_HASH_DIFFERENT"
            ),
        }
    )

manifest_crosscheck_df = pd.DataFrame(manifest_crosscheck_records)


# --------------------------- staged transaction ---------------------------
STAGE_ROOT.mkdir(parents=True, exist_ok=False)
created_final_paths: list[Path] = []
original_memory_contents: dict[Path, Optional[str]] = {}
original_modes: dict[Path, int] = {
    path: stat.S_IMODE(path.stat().st_mode)
    for path in SNAPSHOT_ROOT.rglob("*")
    if path.is_file()
}

try:
    copy_records: list[dict[str, Any]] = []

    for item in SOURCE_PLAN:
        role = str(item["Role"])
        source = Path(item["Source"])
        relative_destination = Path(item["Destination"])
        staged_destination = STAGE_ROOT / "01_source_snapshot" / relative_destination
        final_destination = SNAPSHOT_ROOT / relative_destination

        record = stage_copy(source, staged_destination)
        record.update(
            {
                "Role": role,
                "DestinationRelativePath": str(
                    final_destination.relative_to(EXT_ROOT)
                ),
                "RequiredPriorHash": bool(item["RequiredPriorHash"]),
                "ProtectedTargetVault": False,
                "ModelBinary": False,
                "OpenedForSchemaAudit": True,
            }
        )
        copy_records.append(record)

    copy_audit_df = pd.DataFrame(copy_records)

    staged_manifest_dir = STAGE_ROOT / "01_source_snapshot" / "manifests"
    staged_manifest_dir.mkdir(parents=True, exist_ok=True)

    staged_outputs = {
        "11B_selected_source_profile.csv": primary_validation_df,
        "11B_source_resolution_audit.csv": source_resolution_df,
        "11B_source_schema_audit.csv": schema_audit_df,
        "11B_manifest_crosscheck.csv": manifest_crosscheck_df,
        "11B_source_copy_audit.csv": copy_audit_df,
        "11B_holdout_governance.csv": holdout_governance_df,
        "11B_daily_prediction_profile.csv": prediction_profile_df,
    }
    for filename, dataframe in staged_outputs.items():
        atomic_write_csv(staged_manifest_dir / filename, dataframe)

    # Validate all staged outputs before committing anything.
    for item in SOURCE_PLAN:
        relative_destination = Path(item["Destination"])
        staged_path = STAGE_ROOT / "01_source_snapshot" / relative_destination
        source_path = Path(item["Source"])
        if sha256_file(staged_path) != sha256_file(source_path):
            raise AssertionError(f"Pre-commit staged hash mismatch: {source_path}")

    for staged_path in sorted((STAGE_ROOT / "01_source_snapshot").rglob("*")):
        if not staged_path.is_file():
            continue
        relative_to_stage_snapshot = staged_path.relative_to(
            STAGE_ROOT / "01_source_snapshot"
        )
        final_path = SNAPSHOT_ROOT / relative_to_stage_snapshot
        if final_path.exists():
            raise FileExistsError(f"Commit overwrite guard: {final_path}")
        final_path.parent.mkdir(parents=True, exist_ok=True)
        os.replace(staged_path, final_path)
        created_final_paths.append(final_path)

    # Confirm source copies after commit.
    for item in SOURCE_PLAN:
        source_path = Path(item["Source"])
        final_path = SNAPSHOT_ROOT / Path(item["Destination"])
        if sha256_file(final_path) != sha256_file(source_path):
            raise AssertionError(f"Post-commit source hash mismatch: {final_path}")

    # Make every snapshot file present at this point read-only.
    for snapshot_file in SNAPSHOT_ROOT.rglob("*"):
        if snapshot_file.is_file():
            make_read_only(snapshot_file)

    # Permissions audit is written after the first read-only pass, then made read-only.
    permission_records = []
    for snapshot_file in sorted(SNAPSHOT_ROOT.rglob("*")):
        if snapshot_file.is_file():
            permission_records.append(
                {
                    "RelativePath": str(snapshot_file.relative_to(EXT_ROOT)),
                    "OctalMode": oct(stat.S_IMODE(snapshot_file.stat().st_mode)),
                    "WritableBitsPresent": bool(snapshot_file.stat().st_mode & 0o222),
                    "ReadOnly": file_is_read_only(snapshot_file),
                }
            )
    permissions_df = pd.DataFrame(permission_records)
    permissions_path = MANIFEST_DIR / "11B_read_only_permissions_audit.csv"
    atomic_write_csv(permissions_path, permissions_df)
    created_final_paths.append(permissions_path)
    make_read_only(permissions_path)

    # Final validation table before the lock is created.
    snapshot_files_before_manifest = [
        path for path in SNAPSHOT_ROOT.rglob("*") if path.is_file()
    ]
    protected_copied = any(protected_path(path) for path in created_final_paths)
    joblib_copied = any(path.suffix.lower() == ".joblib" for path in created_final_paths)
    # Only substantive dataset formats count as weekly datasets.
    # Hidden macOS/Jupyter metadata such as .DS_Store and checkpoint files
    # must not make Part 11B fail.
    weekly_dataset_suffixes = {
        ".csv",
        ".tsv",
        ".parquet",
        ".feather",
        ".pkl",
        ".pickle",
        ".xlsx",
        ".xls",
        ".npy",
        ".npz",
    }

    def hidden_or_system_path(path: Path, root: Path) -> bool:
        relative_parts = path.relative_to(root).parts
        return (
            any(part.startswith(".") for part in relative_parts)
            or path.name.startswith("~$")
            or "__MACOSX" in relative_parts
        )

    weekly_data_root = EXT_ROOT / "02_weekly_data"
    weekly_data_files = [
        path
        for path in weekly_data_root.rglob("*")
        if (
            path.is_file()
            and not hidden_or_system_path(path, weekly_data_root)
            and path.suffix.lower() in weekly_dataset_suffixes
        )
    ]
    all_snapshot_read_only = all(
        file_is_read_only(path) for path in snapshot_files_before_manifest
    )

    validation_rows = [
        {
            "Check": "Part 11A lock and checkpoint verified",
            "Expected": True,
            "Actual": True,
            "Passed": True,
        },
        {
            "Check": "Authoritative daily source matched 11A selection",
            "Expected": str(PRIMARY_SOURCE_EXPECTED),
            "Actual": str(preferred_from_11a),
            "Passed": preferred_from_11a == PRIMARY_SOURCE_EXPECTED,
        },
        {
            "Check": "Primary source contract checks passed",
            "Expected": int(len(primary_validation_df)),
            "Actual": int(primary_validation_df["Passed"].sum()),
            "Passed": bool(primary_validation_df["Passed"].all()),
        },
        {
            "Check": "All required source roles resolved",
            "Expected": len(required_roles),
            "Actual": len(required_roles & resolved_roles),
            "Passed": required_roles.issubset(resolved_roles),
        },
        {
            "Check": "Planned source files copied",
            "Expected": len(SOURCE_PLAN),
            "Actual": len(copy_audit_df),
            "Passed": len(copy_audit_df) == len(SOURCE_PLAN),
        },
        {
            "Check": "All source-copy hashes matched",
            "Expected": True,
            "Actual": bool(copy_audit_df["HashMatch"].all()),
            "Passed": bool(copy_audit_df["HashMatch"].all()),
        },
        {
            "Check": "Locked prediction files matched prior manifest",
            "Expected": 3,
            "Actual": int(
                manifest_crosscheck_df.loc[
                    manifest_crosscheck_df["PriorHashRequired"],
                    "PriorHashMatched",
                ].sum()
            ),
            "Passed": bool(
                manifest_crosscheck_df.loc[
                    manifest_crosscheck_df["PriorHashRequired"],
                    "PriorHashMatched",
                ].all()
            ),
        },
        {
            "Check": "Daily standard final holdout start recorded",
            "Expected": EXPECTED_STANDARD_FINAL_START,
            "Actual": standard_final_start,
            "Passed": standard_final_start == EXPECTED_STANDARD_FINAL_START,
        },
        {
            "Check": "Daily standard final holdout end recorded",
            "Expected": EXPECTED_STANDARD_FINAL_END,
            "Actual": standard_final_end,
            "Passed": standard_final_end == EXPECTED_STANDARD_FINAL_END,
        },
        {
            "Check": "Protected target-vault file copied",
            "Expected": False,
            "Actual": protected_copied,
            "Passed": not protected_copied,
        },
        {
            "Check": "Model binary copied",
            "Expected": False,
            "Actual": joblib_copied,
            "Passed": not joblib_copied,
        },
        {
            "Check": "Weekly dataset files created",
            "Expected": 0,
            "Actual": len(weekly_data_files),
            "Passed": len(weekly_data_files) == 0,
        },
        {
            "Check": "Snapshot files read-only",
            "Expected": True,
            "Actual": all_snapshot_read_only,
            "Passed": all_snapshot_read_only,
        },
    ]
    validation_df = pd.DataFrame(validation_rows)
    if not validation_df["Passed"].all():
        raise AssertionError(
            "Part 11B validation failed:\n"
            + validation_df.loc[~validation_df["Passed"]].to_string(index=False)
        )

    validation_path = MANIFEST_DIR / "11B_snapshot_validation.csv"
    atomic_write_csv(validation_path, validation_df)
    created_final_paths.append(validation_path)
    make_read_only(validation_path)

    # File manifest covers the full snapshot except itself and the 11B lock/sidecar.
    excluded_manifest_names = {
        "11B_snapshot_file_manifest.csv",
        PART11B_LOCK.name,
        PART11B_LOCK_SHA.name,
    }
    snapshot_manifest_records = []
    for snapshot_file in sorted(SNAPSHOT_ROOT.rglob("*")):
        if not snapshot_file.is_file() or snapshot_file.name in excluded_manifest_names:
            continue
        snapshot_manifest_records.append(
            {
                "RelativePath": str(snapshot_file.relative_to(EXT_ROOT)),
                "FileName": snapshot_file.name,
                "Bytes": int(snapshot_file.stat().st_size),
                "SHA256": sha256_file(snapshot_file),
                "ReadOnly": file_is_read_only(snapshot_file),
                "CreatedByPart11B": snapshot_file in created_final_paths,
            }
        )
    snapshot_manifest_df = pd.DataFrame(snapshot_manifest_records)
    snapshot_manifest_path = MANIFEST_DIR / "11B_snapshot_file_manifest.csv"
    atomic_write_csv(snapshot_manifest_path, snapshot_manifest_df)
    created_final_paths.append(snapshot_manifest_path)
    make_read_only(snapshot_manifest_path)

    # ------------------------- durable memory -------------------------
    memory_paths = [
        MEMORY_ROOT / "PROJECT_CONTEXT.md",
        MEMORY_ROOT / "WORKFLOW.md",
        MEMORY_ROOT / "DECISIONS.md",
        MEMORY_ROOT / "FILES_AND_PATHS.md",
        MEMORY_ROOT / "METRICS_AND_RESULTS.md",
        MEMORY_ROOT / "CHAT_INDEX.md",
        MEMORY_ROOT / "CURRENT_HANDOFF.md",
    ]
    for memory_path in memory_paths:
        require_file(memory_path, f"memory file {memory_path.name}")
        original_memory_contents[memory_path] = memory_path.read_text(encoding="utf-8")

    step_log_path = MEMORY_ROOT / "steps" / "STEP_11B_SOURCE_SNAPSHOT.md"
    if step_log_path.exists():
        raise FileExistsError(f"Step-log overwrite guard: {step_log_path}")
    original_memory_contents[step_log_path] = None

    status = "PART_11B_COMPLETED_READY_FOR_11C"

    context_body = "\n".join(
        [
            f"**Status:** `{status}`",
            "",
            "Part 11B confirmed and copied the canonical product-day demand panel, any uniquely resolved Step 5 companion files, the frozen daily split calendars and the locked unscored daily prediction tables into a read-only source snapshot.",
            "",
            f"- Canonical source: `{PRIMARY_SOURCE_EXPECTED}`",
            f"- Snapshot source hash: `{sha256_file(SNAPSHOT_ROOT / 'data' / PRIMARY_SOURCE_EXPECTED.name)}`",
            f"- Source rows / products / operating dates: `{len(primary_df):,}` / `{primary_df['CanonicalProductID'].nunique()}` / `{primary_df['Date'].nunique()}`",
            f"- Opened daily final-holdout interval: `{standard_final_start}` to `{standard_final_end}`",
            f"- Optional Step 5 companions unavailable or ambiguous: `{len(optional_unavailable_roles)}` ({', '.join(optional_unavailable_roles) if optional_unavailable_roles else 'none'}).",
            "- Those holdout rows remain prohibited from weekly tuning or product-scope decisions.",
            "- No weekly dataset was created in Part 11B.",
        ]
    )
    upsert_markdown_section(
        MEMORY_ROOT / "PROJECT_CONTEXT.md",
        "STEP_11B_SOURCE_SNAPSHOT",
        "Part 11B — Source Snapshot",
        context_body,
    )

    update_workflow_status(MEMORY_ROOT / "WORKFLOW.md", status)
    upsert_markdown_section(
        MEMORY_ROOT / "WORKFLOW.md",
        "STEP_11B_SOURCE_SNAPSHOT",
        "Part 11B completion note",
        "\n".join(
            [
                f"Status: `{status}`.",
                "Part 11C may create and audit a weekly product-demand dataset using only files locked in `01_source_snapshot`.",
                "Part 11C must not use the opened March 2026 daily holdout to choose weekly product coverage, segments, features or models.",
            ]
        ),
    )

    decisions_body = "\n".join(
        [
            "### D11B-001 — Canonical source confirmed",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** Part 11A uniquely identified the final Step 5 product-day panel.",
            f"- **Decision:** Use `{PRIMARY_SOURCE_EXPECTED.name}` as the weekly source dataset.",
            "- **Files affected:** `01_source_snapshot/data/` and Part 11B manifests.",
            "- **Earlier results remain valid:** Yes.",
            "- **New next step:** Aggregate and audit weekly rows in Part 11C.",
            "",
            "### D11B-002 — Locked daily predictions included as reference",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** Part 11E must compare direct weekly forecasts against weekly sums of the locked daily forecasts.",
            "- **Decision:** Copy the locked component, product and aggregate prediction CSV files after matching their original 10B2 hashes. Do not copy or load joblib models.",
            "- **Files affected:** `01_source_snapshot/daily_model_reference/`.",
            "- **Earlier results remain valid:** Yes; files are copied byte-for-byte.",
            "- **New next step:** Preserve these values unchanged until baseline construction.",
            "",
            "### D11B-003 — Opened holdout governance",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            f"- **Reason:** The daily final holdout `{standard_final_start}` to `{standard_final_end}` has already been opened.",
            "- **Decision:** Retain full source history for traceability, but prohibit dates on or after the cutoff from weekly tuning, product selection, segmentation, feature design, hyperparameter choice and fallback selection.",
            "- **Files affected:** `11B_holdout_governance.csv` and all future weekly validation code.",
            "- **Earlier results remain valid:** Yes.",
            "- **New next step:** Tag and isolate this interval in Part 11C without using it for optimisation.",
            "",
            "### D11B-004 — Step 5 companion-file discovery",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** The first Part 11B attempt found that a planned Step 5 companion filename was not present at the hard-coded Eden-root path.",
            "- **Decision:** Resolve companions by exact filename outside the weekly extension. Copy only a uniquely identified safe version; log missing or hash-ambiguous optional companions without inventing a path or reconstructing a source file.",
            "- **Files affected:** `11B_source_resolution_audit.csv` and the Part 11B source plan.",
            "- **Earlier results remain valid:** Yes; the failed attempt stopped before the staged copy transaction and created no Part 11B lock.",
            "- **New next step:** Part 11C uses the canonical snapshot and frozen contracts rather than depending on an unavailable optional companion.",
            "",
            "### D11B-005 — Read-only snapshot",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** Future steps need stable, auditable inputs.",
            "- **Decision:** Set all snapshot files to read-only and lock their hashes.",
            "- **Files affected:** all files under `01_source_snapshot/`.",
            "- **Earlier results remain valid:** Yes.",
            "- **New next step:** Write derived data only under `02_weekly_data/` and later working folders.",
        ]
    )
    upsert_markdown_section(
        MEMORY_ROOT / "DECISIONS.md",
        "STEP_11B_SOURCE_SNAPSHOT",
        "Part 11B decisions",
        decisions_body,
    )

    files_body = "\n".join(
        [
            f"**Status:** `{status}`",
            "",
            "### Primary source",
            f"- Original: `{PRIMARY_SOURCE_EXPECTED}`",
            f"- Snapshot: `{SNAPSHOT_ROOT / 'data' / PRIMARY_SOURCE_EXPECTED.name}`",
            "",
            "### Locked daily prediction references",
            f"- `{SNAPSHOT_ROOT / 'daily_model_reference' / '10B2_scoring_component_predictions.csv'}`",
            f"- `{SNAPSHOT_ROOT / 'daily_model_reference' / '10B2_final_product_predictions.csv'}`",
            f"- `{SNAPSHOT_ROOT / 'daily_model_reference' / '10B2_final_aggregate_predictions.csv'}`",
            "",
            "### Governance and manifests",
            f"- `{MANIFEST_DIR / '11B_holdout_governance.csv'}`",
            f"- `{MANIFEST_DIR / '11B_source_resolution_audit.csv'}`",
            f"- `{MANIFEST_DIR / '11B_source_copy_audit.csv'}`",
            f"- `{MANIFEST_DIR / '11B_snapshot_file_manifest.csv'}`",
            f"- `{PART11B_LOCK}`",
            f"- `{PART11B_CHECKPOINT}`",
        ]
    )
    upsert_markdown_section(
        MEMORY_ROOT / "FILES_AND_PATHS.md",
        "STEP_11B_SOURCE_SNAPSHOT",
        "Part 11B files and paths",
        files_body,
    )

    metrics_body = "\n".join(
        [
            f"**Status:** `{status}`",
            "",
            "### Source profile validation",
            markdown_table(primary_validation_df),
            "",
            "### Locked daily prediction profile",
            markdown_table(prediction_profile_df),
            "",
            "These are integrity and source-readiness results, not weekly forecasting performance results.",
        ]
    )
    upsert_markdown_section(
        MEMORY_ROOT / "METRICS_AND_RESULTS.md",
        "STEP_11B_SOURCE_SNAPSHOT",
        "Part 11B source results",
        metrics_body,
    )

    chat_row = (
        f"| {NOW_LOCAL.isoformat()} | 11B | Start Part 11B. | "
        "Verified and locked a read-only weekly source snapshot; no weekly dataset built. | "
        f"`steps/STEP_11B_SOURCE_SNAPSHOT.md` | `{status}` |"
    )
    append_chat_index_row(MEMORY_ROOT / "CHAT_INDEX.md", chat_row)

    current_handoff_lines = [
        "# CURRENT HANDOFF",
        "",
        f"- **Current status:** `{status}`",
        "- **Completed step:** Part 11B — verified read-only source snapshot",
        "- **Next step:** Part 11C — create and audit the weekly product-demand dataset",
        f"- **Canonical source snapshot:** `{SNAPSHOT_ROOT / 'data' / PRIMARY_SOURCE_EXPECTED.name}`",
        f"- **Daily product forecast reference:** `{SNAPSHOT_ROOT / 'daily_model_reference' / '10B2_final_product_predictions.csv'}`",
        f"- **Opened daily holdout:** `{standard_final_start}` to `{standard_final_end}`",
        "- **Weekly tuning rule:** dates on or after the cutoff must not determine product scope, segmentation, features, models, hyperparameters or fallback rules.",
        "- **Protected target vault opened or copied by Part 11B:** `False` / `False`",
        "- **Model binaries loaded or copied by Part 11B:** `False` / `False`",
        "- **Weekly dataset created by Part 11B:** `False`",
        "",
        "## Part 11C entry conditions",
        "",
        "1. Verify the Part 11B lock and SHA-256 sidecar.",
        "2. Read the canonical source only from the locked snapshot.",
        "3. Define and document the weekly date convention before aggregation.",
        "4. Preserve NormalDemand, BulkDemand and TotalDemand as separate weekly sums.",
        "5. Retain all required zero-demand product-week contexts.",
        "6. Do not perform product coverage selection or model tuning in Part 11C.",
    ]
    atomic_write_text(
        MEMORY_ROOT / "CURRENT_HANDOFF.md",
        "\n".join(current_handoff_lines).rstrip() + "\n",
    )

    step_log_lines = [
        "# STEP 11B — SOURCE SNAPSHOT",
        "",
        "- **Step ID:** 11B",
        f"- **Date and time:** {NOW_LOCAL.isoformat()}",
        "- **Status:** PART_11B_COMPLETED_READY_FOR_11C",
        "",
        "## User request",
        "",
        "Start Part 11B.",
        "",
        "## Assistant response summary",
        "",
        "Created a verified, read-only source snapshot from the exact source and locked daily-reference files identified in Part 11A. No weekly data, product segments or models were created.",
        "",
        "## Full technical actions",
        "",
        "1. Verified the Part 11A lock and checkpoint hashes.",
        "2. Confirmed the authoritative canonical daily source selected in Part 11A.",
        "3. Read and validated the canonical daily source against the completed Step 5 contract.",
        "4. Inspected all source schemas before copying.",
        "5. Verified the daily standard final-holdout dates and model-selection prohibition.",
        "6. Verified the three locked daily prediction tables against the original 10B2 artifact hash manifest.",
        "7. Copied only approved CSV inputs; copied no target-vault file and no joblib model.",
        "8. Verified every source-to-snapshot SHA-256 value.",
        "9. Set snapshot files to read-only and created a full snapshot manifest.",
        "10. Updated all project-memory files and created a Part 11B checkpoint and lock.",
        "",
        "## Decisions made",
        "",
        "See the controlled Part 11B section in `../DECISIONS.md`.",
        "",
        "## Assumptions",
        "",
        "- The completed Step 5 dataset contract remains authoritative.",
        "- The full source history can be retained for traceability because the March 2026 daily holdout is already opened, but it cannot be used for weekly optimisation.",
        "- The locked unscored daily predictions are safe baseline references because they contain no actual target column.",
        "",
        "## Input files",
        "",
        markdown_table(
            copy_audit_df[
                ["Role", "SourcePath", "SourceSHA256", "DestinationRelativePath"]
            ],
            max_rows=30,
        ),
        "",
        "## Output files",
        "",
        "- Approved source copies under `01_source_snapshot/data/`, `contracts/` and `daily_model_reference/`",
        "- `01_source_snapshot/manifests/11B_selected_source_profile.csv`",
        "- `01_source_snapshot/manifests/11B_source_resolution_audit.csv`",
        "- `01_source_snapshot/manifests/11B_source_schema_audit.csv`",
        "- `01_source_snapshot/manifests/11B_manifest_crosscheck.csv`",
        "- `01_source_snapshot/manifests/11B_source_copy_audit.csv`",
        "- `01_source_snapshot/manifests/11B_holdout_governance.csv`",
        "- `01_source_snapshot/manifests/11B_daily_prediction_profile.csv`",
        "- `01_source_snapshot/manifests/11B_read_only_permissions_audit.csv`",
        "- `01_source_snapshot/manifests/11B_snapshot_validation.csv`",
        "- `01_source_snapshot/manifests/11B_snapshot_file_manifest.csv`",
        "- `11_checkpoints/11B_checkpoint.json` and SHA-256 sidecar",
        "- `01_source_snapshot/manifests/11B_weekly_source_snapshot_lock.json` and SHA-256 sidecar",
        "",
        "## Validation checks",
        "",
        markdown_table(validation_df),
        "",
        "## Errors encountered and fixes",
        "",
        "The first Part 11B attempt failed before copying because `UL_EDEN_step5_final_product_metadata.csv` was hard-coded at the Eden root but was not present there. The corrected replacement cell uses exact-filename discovery and treats Step 5 companions as optional continuity files while preserving all required-source checks.",
        "",
        "## Results",
        "",
        f"- Canonical source rows: {len(primary_df):,}",
        f"- Canonical products: {primary_df['CanonicalProductID'].nunique()}",
        f"- Operating dates: {primary_df['Date'].nunique()}",
        f"- Source total demand: {primary_df['TotalDemand'].sum():,.0f}",
        f"- Opened daily final holdout: {standard_final_start} to {standard_final_end}",
        f"- Approved new source files copied: {len(SOURCE_PLAN)}",
        f"- Optional Step 5 companions unavailable or ambiguous: {len(optional_unavailable_roles)}",
        "- Protected target-vault files copied: 0",
        "- Model binaries copied or loaded: 0",
        "- Weekly datasets created: 0",
        "",
        "## Limitations",
        "",
        "Part 11B does not define a weekly boundary, aggregate product-week demand, select products, create segments, build features or evaluate a forecast. A new untouched future period remains necessary for an unbiased final weekly evaluation.",
        "",
        "## Next action",
        "",
        "Part 11C should define the weekly date convention and create an audited weekly product-demand dataset from the locked canonical snapshot.",
    ]
    atomic_write_text(step_log_path, "\n".join(step_log_lines).rstrip() + "\n")

    # ------------------------- checkpoint -------------------------
    checkpoint_payload = {
        "StepID": "11B",
        "Status": status,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ExtensionRoot": str(EXT_ROOT),
        "CanonicalSource": {
            "OriginalPath": str(PRIMARY_SOURCE_EXPECTED),
            "SnapshotPath": str(SNAPSHOT_ROOT / "data" / PRIMARY_SOURCE_EXPECTED.name),
            "SHA256": sha256_file(SNAPSHOT_ROOT / "data" / PRIMARY_SOURCE_EXPECTED.name),
            "Rows": int(len(primary_df)),
            "Columns": int(primary_df.shape[1]),
            "Products": int(primary_df["CanonicalProductID"].nunique()),
            "OperatingDates": int(primary_df["Date"].nunique()),
        },
        "OpenedDailyFinalHoldout": {
            "StartDate": standard_final_start,
            "EndDate": standard_final_end,
            "MayTuneWeeklySystem": False,
        },
        "Counts": {
            "NewSourceFilesCopied": len(SOURCE_PLAN),
            "RequiredSourceRoles": len(required_roles),
            "RequiredSourceRolesResolved": len(required_roles & resolved_roles),
            "OptionalSourceRolesUnavailableOrAmbiguous": len(optional_unavailable_roles),
            "SchemaAudits": len(schema_audit_df),
            "PriorManifestMatchesRequired": 3,
            "PriorManifestMatchesPassed": int(
                manifest_crosscheck_df.loc[
                    manifest_crosscheck_df["PriorHashRequired"],
                    "PriorHashMatched",
                ].sum()
            ),
            "ValidationChecks": len(validation_df),
            "ValidationPassed": int(validation_df["Passed"].sum()),
        },
        "Safety": {
            "ProtectedTargetVaultOpened": False,
            "ProtectedTargetVaultCopied": False,
            "OpenedDailyHoldoutRowsReadForIntegrityAudit": True,
            "OpenedDailyHoldoutUsedForWeeklyTuning": False,
            "ModelBinariesLoaded": False,
            "ModelBinariesCopied": False,
            "ModelsRetrained": False,
            "PredictionsChanged": False,
            "WeeklyDatasetBuilt": False,
        },
        "ReadyForPart11C": True,
        "NextStep": "11C",
    }
    if PART11B_CHECKPOINT.exists() or PART11B_CHECKPOINT_SHA.exists():
        raise FileExistsError("Part 11B checkpoint overwrite guard triggered")
    atomic_write_json(PART11B_CHECKPOINT, checkpoint_payload)
    created_final_paths.append(PART11B_CHECKPOINT)
    checkpoint_hash = sha256_file(PART11B_CHECKPOINT)
    atomic_write_text(
        PART11B_CHECKPOINT_SHA,
        f"{checkpoint_hash}  {PART11B_CHECKPOINT.name}\n",
    )
    created_final_paths.append(PART11B_CHECKPOINT_SHA)

    # Snapshot lock is created last and refers to completed controls.
    memory_hashes = [
        {
            "RelativePath": str(path.relative_to(EXT_ROOT)),
            "SHA256": sha256_file(path),
        }
        for path in memory_paths + [step_log_path]
    ]

    lock_payload = {
        "StepID": "11B",
        "Status": status,
        "CreatedUTC": NOW_UTC.isoformat(),
        "ExtensionRoot": str(EXT_ROOT),
        "Part11ALock": {
            "Path": str(PART11A_LOCK),
            "SHA256": sha256_file(PART11A_LOCK),
            "Verified": True,
        },
        "CanonicalSource": checkpoint_payload["CanonicalSource"],
        "SourceResolution": {
            "AuditPath": str(MANIFEST_DIR / "11B_source_resolution_audit.csv"),
            "AuditSHA256": sha256_file(MANIFEST_DIR / "11B_source_resolution_audit.csv"),
            "RequiredRoles": len(required_roles),
            "RequiredRolesResolved": len(required_roles & resolved_roles),
            "OptionalUnavailableOrAmbiguousRoles": optional_unavailable_roles,
        },
        "HoldoutGovernance": checkpoint_payload["OpenedDailyFinalHoldout"],
        "Snapshot": {
            "Root": str(SNAPSHOT_ROOT),
            "FileManifest": str(snapshot_manifest_path),
            "FileManifestSHA256": sha256_file(snapshot_manifest_path),
            "FilesListed": int(len(snapshot_manifest_df)),
            "AllListedFilesReadOnly": bool(snapshot_manifest_df["ReadOnly"].all()),
        },
        "DailyPredictionReferences": manifest_crosscheck_df.loc[
            manifest_crosscheck_df["PriorHashRequired"],
            ["Role", "SourcePath", "ActualSHA256", "PriorHashMatched"],
        ].to_dict(orient="records"),
        "Validation": {
            "Path": str(validation_path),
            "SHA256": sha256_file(validation_path),
            "Checks": int(len(validation_df)),
            "Passed": int(validation_df["Passed"].sum()),
            "Failed": int((~validation_df["Passed"]).sum()),
        },
        "MemoryFileHashes": memory_hashes,
        "Checkpoint": {
            "Path": str(PART11B_CHECKPOINT),
            "SHA256": checkpoint_hash,
        },
        "SafetyAssertions": checkpoint_payload["Safety"],
        "ReadyForPart11C": True,
        "NextStep": "11C",
    }
    atomic_write_json(PART11B_LOCK, lock_payload)
    created_final_paths.append(PART11B_LOCK)
    part11b_lock_hash = sha256_file(PART11B_LOCK)
    atomic_write_text(
        PART11B_LOCK_SHA,
        f"{part11b_lock_hash}  {PART11B_LOCK.name}\n",
    )
    created_final_paths.append(PART11B_LOCK_SHA)
    make_read_only(PART11B_LOCK)
    make_read_only(PART11B_LOCK_SHA)

    # Final read-only and hash verification.
    for snapshot_file in SNAPSHOT_ROOT.rglob("*"):
        if snapshot_file.is_file():
            make_read_only(snapshot_file)

    if read_sidecar_hash(PART11B_LOCK_SHA) != sha256_file(PART11B_LOCK):
        raise AssertionError("Part 11B lock sidecar verification failed")
    if read_sidecar_hash(PART11B_CHECKPOINT_SHA) != sha256_file(PART11B_CHECKPOINT):
        raise AssertionError("Part 11B checkpoint sidecar verification failed")
    if not all(
        file_is_read_only(path)
        for path in SNAPSHOT_ROOT.rglob("*")
        if path.is_file()
    ):
        raise AssertionError("At least one snapshot file is not read-only")

    log_path = LOG_ROOT / "11B_source_snapshot_log.txt"
    if log_path.exists():
        raise FileExistsError(f"Log overwrite guard: {log_path}")
    log_lines = [
        f"Status: {status}",
        f"Run UTC: {NOW_UTC.isoformat()}",
        f"Run local: {NOW_LOCAL.isoformat()}",
        f"Canonical source: {PRIMARY_SOURCE_EXPECTED}",
        f"Canonical source SHA256: {sha256_file(PRIMARY_SOURCE_EXPECTED)}",
        f"New source files copied: {len(SOURCE_PLAN)}",
        f"Optional Step 5 companions unavailable or ambiguous: {len(optional_unavailable_roles)}",
        f"Opened daily holdout: {standard_final_start} to {standard_final_end}",
        "Opened holdout used for weekly tuning: False",
        "Protected target vault opened/copied: False/False",
        "Model binaries loaded/copied/retrained: False/False/False",
        "Weekly dataset built: False",
        f"Checkpoint SHA256: {checkpoint_hash}",
        f"Part 11B lock SHA256: {part11b_lock_hash}",
    ]
    atomic_write_text(log_path, "\n".join(log_lines) + "\n")
    created_final_paths.append(log_path)

    # Remove the now-empty staging directory.
    shutil.rmtree(STAGE_ROOT, ignore_errors=True)

    # ------------------------- printed output -------------------------
    print("=" * 100)
    print("EDEN WEEKLY FORECASTING EXTENSION — PART 11B COMPLETE")
    print("=" * 100)
    print(f"Status: {status}")
    print(f"Extension root: {EXT_ROOT}")
    print(f"Local time: {NOW_LOCAL.isoformat()}")

    print("\nPRIMARY SOURCE PROFILE")
    print(primary_validation_df.to_string(index=False))

    print("\nHOLDOUT GOVERNANCE")
    print(holdout_governance_df.to_string(index=False))

    print("\nSOURCE RESOLUTION AUDIT")
    print(
        source_resolution_df[
            [
                "Role",
                "Required",
                "CandidatesFound",
                "UniqueCandidateHashes",
                "SelectedSource",
                "ResolutionStatus",
            ]
        ].to_string(index=False)
    )

    print("\nSOURCE COPY AUDIT")
    print(
        copy_audit_df[
            [
                "Role",
                "SourcePath",
                "DestinationRelativePath",
                "Bytes",
                "SourceSHA256",
                "HashMatch",
            ]
        ].to_string(index=False)
    )

    print("\nLOCKED DAILY PREDICTION HASH CROSS-CHECK")
    print(
        manifest_crosscheck_df.loc[
            manifest_crosscheck_df["PriorHashRequired"],
            [
                "Role",
                "ActualSHA256",
                "PriorRecordedSHA256",
                "PriorHashMatched",
            ],
        ].to_string(index=False)
    )

    print("\nSNAPSHOT VALIDATION")
    print(validation_df.to_string(index=False))

    print("\nCONTROL OUTPUTS")
    print(f"- Source profile: {MANIFEST_DIR / '11B_selected_source_profile.csv'}")
    print(f"- Source resolution audit: {MANIFEST_DIR / '11B_source_resolution_audit.csv'}")
    print(f"- Schema audit: {MANIFEST_DIR / '11B_source_schema_audit.csv'}")
    print(f"- Copy audit: {MANIFEST_DIR / '11B_source_copy_audit.csv'}")
    print(f"- Holdout governance: {MANIFEST_DIR / '11B_holdout_governance.csv'}")
    print(f"- Snapshot manifest: {snapshot_manifest_path}")
    print(f"- Current handoff: {MEMORY_ROOT / 'CURRENT_HANDOFF.md'}")
    print(f"- Step log: {step_log_path}")
    print(f"- Checkpoint: {PART11B_CHECKPOINT}")
    print(f"- Checkpoint SHA256: {sha256_file(PART11B_CHECKPOINT)}")
    print(f"- Part 11B lock: {PART11B_LOCK}")
    print(f"- Part 11B lock SHA256: {sha256_file(PART11B_LOCK)}")

    print(
        "\nSAFETY: protected target vault opened/copied False/False; "
        "opened daily holdout used for weekly tuning False; "
        "model binaries loaded/copied/retrained False/False/False; "
        "weekly dataset built False."
    )
    print("=" * 100)

except Exception:
    # Restore memory files if they were changed.
    for memory_path, original_text in original_memory_contents.items():
        try:
            if original_text is None:
                if memory_path.exists():
                    memory_path.unlink()
            else:
                atomic_write_text(memory_path, original_text)
        except Exception:
            pass

    # Remove only outputs created by this attempted Part 11B transaction.
    for created_path in sorted(
        set(created_final_paths),
        key=lambda path: len(path.parts),
        reverse=True,
    ):
        try:
            if created_path.exists():
                created_path.chmod(stat.S_IRUSR | stat.S_IWUSR)
                created_path.unlink()
        except Exception:
            pass

    # Restore permissions of files that existed before Part 11B.
    for existing_path, original_mode in original_modes.items():
        try:
            if existing_path.exists():
                existing_path.chmod(original_mode)
        except Exception:
            pass

    if STAGE_ROOT.exists():
        shutil.rmtree(STAGE_ROOT, ignore_errors=True)
    raise

EDEN WEEKLY FORECASTING EXTENSION — PART 11B COMPLETE
Status: PART_11B_COMPLETED_READY_FOR_11C
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-05T20:00:13.595629+01:00

PRIMARY SOURCE PROFILE
                   Metric   Expected     Actual  Passed
                     Rows      25405      25405    True
                  Columns         38         38    True
           OperatingDates        245        245    True
               ProductIDs        227        227    True
             ProductNames        218        218    True
                  DateMin 2025-04-01 2025-04-01    True
                  DateMax 2026-03-30 2026-03-30    True
             ObservedRows      15138      15138    True
           ZeroDemandRows      10267      10267    True
        NormalDemandUnits     114186   114186.0    True
          BulkDemandUnits       1972     1972.0    True
         TotalDemandUnits     116158   116158.0    True
 DuplicatePr

In [13]:
# EDEN WEEKLY FORECASTING EXTENSION — PART 11C
# Create and audit the weekly product-demand dataset from the final daily-model lineage.
# This cell is self-contained and must be run once after a locked Part 11B completion.

from __future__ import annotations

import hashlib
import json
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd


# =============================================================================
# 1. PATHS, CONSTANTS AND OVERWRITE GUARDS
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
FORECAST_PREPARATION_ROOT = EDEN_ROOT / "forceast_preparation"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"

SNAPSHOT_ROOT = EXT_ROOT / "01_source_snapshot"
SNAPSHOT_DATA_ROOT = SNAPSHOT_ROOT / "data"
SNAPSHOT_CONTRACT_ROOT = SNAPSHOT_ROOT / "contracts"
SNAPSHOT_MANIFEST_ROOT = SNAPSHOT_ROOT / "manifests"

WEEKLY_DATA_ROOT = EXT_ROOT / "02_weekly_data"
WEEKLY_INTERMEDIATE_ROOT = WEEKLY_DATA_ROOT / "intermediate"
WEEKLY_AUDITED_ROOT = WEEKLY_DATA_ROOT / "audited"
WEEKLY_FINAL_ROOT = WEEKLY_DATA_ROOT / "final"

MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
LOG_ROOT = EXT_ROOT / "12_logs"

PART11B_LOCK = SNAPSHOT_MANIFEST_ROOT / "11B_weekly_source_snapshot_lock.json"
PART11B_LOCK_SHA = SNAPSHOT_MANIFEST_ROOT / "11B_weekly_source_snapshot_lock.sha256"
PART11B_CHECKPOINT = CHECKPOINT_ROOT / "11B_checkpoint.json"
PART11B_CHECKPOINT_SHA = CHECKPOINT_ROOT / "11B_checkpoint.sha256"
PART11B_SNAPSHOT_MANIFEST = SNAPSHOT_MANIFEST_ROOT / "11B_snapshot_file_manifest.csv"
PART11B_HOLDOUT_GOVERNANCE = SNAPSHOT_MANIFEST_ROOT / "11B_holdout_governance.csv"

PART11C_SOURCE_AMENDMENT_LOCK = SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_lock.json"
PART11C_SOURCE_AMENDMENT_LOCK_SHA = SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_lock.sha256"
PART11C_CHECKPOINT = CHECKPOINT_ROOT / "11C_checkpoint.json"
PART11C_CHECKPOINT_SHA = CHECKPOINT_ROOT / "11C_checkpoint.sha256"
PART11C_LOCK = CHECKPOINT_ROOT / "11C_weekly_dataset_lock.json"
PART11C_LOCK_SHA = CHECKPOINT_ROOT / "11C_weekly_dataset_lock.sha256"

FINAL_MODEL_PREPARATION_COPY = (
    FORECAST_PREPARATION_ROOT / "UL_EDEN_forecasting_preparation_final_model_dataset.csv"
)
FINAL_MODEL_EDEN_COPY = EDEN_ROOT / "UL_EDEN_forecasting_preparation_final_model_dataset.csv"

CANONICAL_SNAPSHOT_SOURCE = (
    SNAPSHOT_DATA_ROOT / "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"
)

LOCAL_CONTRACT_FILES = [
    "17_final_dataset_inventory.csv",
    "17_final_dataset_summary.csv",
    "17_final_file_hash_manifest.csv",
    "17_final_leakage_audit.csv",
    "17_final_predictor_missingness_audit.csv",
    "17_forecasting_preparation_final_validation_summary.csv",
    "17_forecasting_protocol_contract.csv",
    "17_frozen_model_feature_contract.csv",
]

EXPECTED_FINAL_MODEL_SHA256 = (
    "ec86928f6a96f0dd9131a56bbb0893cd62a726cf3fde43cb1ffd6a787ca837f2"
)
EXPECTED_FINAL_ROWS = 43_774
EXPECTED_FINAL_COLUMNS = 58
EXPECTED_PRODUCTS = 227
EXPECTED_OPERATING_DATES = 245
EXPECTED_NORMAL_DEMAND = 114_186
EXPECTED_BULK_DEMAND = 1_972
EXPECTED_TOTAL_DEMAND = 116_158
EXPECTED_DATE_MIN = "2025-04-01"
EXPECTED_DATE_MAX = "2026-03-30"
EXPECTED_HOLDOUT_START = "2026-03-02"
EXPECTED_HOLDOUT_END = "2026-03-30"

NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))
STATUS = "PART_11C_COMPLETED_READY_FOR_11D"
STEP_ID = "11C"

NEW_FINAL_PATHS = [
    SNAPSHOT_DATA_ROOT
    / "11C_forecasting_preparation_source"
    / FINAL_MODEL_PREPARATION_COPY.name,
    *[
        SNAPSHOT_CONTRACT_ROOT / "11C_forecasting_preparation" / name
        for name in LOCAL_CONTRACT_FILES
    ],
    SNAPSHOT_MANIFEST_ROOT / "11C_source_resolution_audit.csv",
    SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_copy_audit.csv",
    SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_manifest.csv",
    PART11C_SOURCE_AMENDMENT_LOCK,
    PART11C_SOURCE_AMENDMENT_LOCK_SHA,
    WEEKLY_AUDITED_ROOT / "11C_daily_lineage_reconciliation_audit.csv",
    WEEKLY_AUDITED_ROOT / "11C_weekly_calendar.csv",
    WEEKLY_AUDITED_ROOT / "11C_week_summary.csv",
    WEEKLY_AUDITED_ROOT / "11C_product_week_summary.csv",
    WEEKLY_AUDITED_ROOT / "11C_weekly_dataset_schema.csv",
    WEEKLY_AUDITED_ROOT / "11C_weekly_dataset_audit.csv",
    WEEKLY_AUDITED_ROOT / "11C_weekly_validation.csv",
    WEEKLY_AUDITED_ROOT / "11C_output_hash_manifest.csv",
    WEEKLY_FINAL_ROOT / "11C_weekly_dataset.csv",
    STEP_MEMORY_ROOT / "STEP_11C_WEEKLY_DATASET.md",
    PART11C_CHECKPOINT,
    PART11C_CHECKPOINT_SHA,
    PART11C_LOCK,
    PART11C_LOCK_SHA,
    LOG_ROOT / "11C_weekly_dataset_log.txt",
]


# =============================================================================
# 2. GENERAL UTILITIES
# =============================================================================


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def require_directory(path: Path, label: str) -> None:
    if not path.is_dir():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def read_sidecar_hash(path: Path) -> str:
    require_file(path, f"SHA-256 sidecar {path.name}")
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Empty SHA-256 sidecar: {path}")
    value = text.split()[0].strip().lower()
    if len(value) != 64 or any(ch not in "0123456789abcdef" for ch in value):
        raise ValueError(f"Invalid SHA-256 sidecar content: {path}")
    return value


def verify_file_and_sidecar(file_path: Path, sidecar_path: Path, label: str) -> str:
    require_file(file_path, label)
    expected = read_sidecar_hash(sidecar_path)
    actual = sha256_file(file_path)
    if expected != actual:
        raise AssertionError(
            f"{label} SHA-256 mismatch:\nExpected: {expected}\nActual:   {actual}\n{file_path}"
        )
    return actual


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        temp.write_text(text, encoding="utf-8")
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def atomic_write_json(path: Path, payload: dict) -> None:
    atomic_write_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        frame.to_csv(temp, index=False)
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def make_read_only(path: Path) -> None:
    if path.exists() and path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def make_writable(path: Path) -> None:
    if path.exists() and path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)


def file_is_read_only(path: Path) -> bool:
    return path.is_file() and not bool(path.stat().st_mode & 0o222)


def markdown_table(frame: pd.DataFrame, max_rows: int = 50) -> str:
    if frame.empty:
        return "_No rows._"
    view = frame.head(max_rows).copy()
    columns = [str(col) for col in view.columns]
    rows = ["| " + " | ".join(columns) + " |"]
    rows.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for _, row in view.iterrows():
        values = []
        for value in row.tolist():
            if pd.isna(value):
                text = ""
            else:
                text = str(value)
            values.append(text.replace("|", "\\|"))
        rows.append("| " + " | ".join(values) + " |")
    if len(frame) > max_rows:
        rows.append(f"\n_Showing {max_rows} of {len(frame)} rows._")
    return "\n".join(rows)


def upsert_section_text(
    original: str,
    marker: str,
    heading: str,
    body: str,
) -> str:
    start = f"<!-- BEGIN {marker} -->"
    end = f"<!-- END {marker} -->"
    section = f"{start}\n## {heading}\n\n{body.rstrip()}\n{end}"
    if start in original and end in original:
        before = original.split(start, 1)[0].rstrip()
        after = original.split(end, 1)[1].lstrip()
        result = before + "\n\n" + section
        if after:
            result += "\n\n" + after
        return result.rstrip() + "\n"
    return original.rstrip() + "\n\n" + section + "\n"


def append_chat_row_text(original: str, row: str) -> str:
    if row in original:
        return original
    return original.rstrip() + "\n" + row + "\n"


def format_dates_for_csv(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    result = frame.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_datetime(result[column]).dt.strftime("%Y-%m-%d")
    return result


def check_integer_like(series: pd.Series, label: str) -> None:
    numeric = pd.to_numeric(series, errors="raise")
    if numeric.isna().any():
        raise AssertionError(f"{label} contains missing values")
    if not np.allclose(numeric.to_numpy(dtype=float), np.rint(numeric.to_numpy(dtype=float))):
        raise AssertionError(f"{label} contains non-integer demand values")
    if (numeric < 0).any():
        raise AssertionError(f"{label} contains negative demand values")


def protected_target_name(path: Path) -> bool:
    name = path.name.lower()
    return "target_vault" in name or "protected_target" in name


def clean_old_staging() -> None:
    for path in EXT_ROOT.glob(".11C_staging_*"):
        if path.is_dir():
            shutil.rmtree(path)


# =============================================================================
# 3. VERIFY PART 11B AND EXISTING SNAPSHOT INTEGRITY
# =============================================================================

require_directory(EXT_ROOT, "weekly extension root")
require_directory(SNAPSHOT_ROOT, "Part 11B source snapshot")
require_directory(WEEKLY_DATA_ROOT, "weekly data root")
require_directory(MEMORY_ROOT, "project memory root")
require_directory(CHECKPOINT_ROOT, "checkpoint root")

if PART11C_LOCK.exists() or PART11C_LOCK_SHA.exists():
    raise FileExistsError(
        "Part 11C overwrite lock triggered. A Part 11C lock already exists:\n"
        f"{PART11C_LOCK}"
    )

unexpected_existing = [path for path in NEW_FINAL_PATHS if path.exists()]
if unexpected_existing:
    listing = "\n".join(f"- {path}" for path in unexpected_existing)
    raise FileExistsError(
        "Uncommitted or previously created Part 11C outputs were found. "
        "No files were changed. Review these paths before rerunning:\n" + listing
    )

clean_old_staging()

part11b_lock_hash = verify_file_and_sidecar(
    PART11B_LOCK, PART11B_LOCK_SHA, "Part 11B source-snapshot lock"
)
part11b_checkpoint_hash = verify_file_and_sidecar(
    PART11B_CHECKPOINT, PART11B_CHECKPOINT_SHA, "Part 11B checkpoint"
)

part11b_lock_payload = json.loads(PART11B_LOCK.read_text(encoding="utf-8"))
recorded_11b_manifest_hash = (
    part11b_lock_payload.get("Snapshot", {}).get("FileManifestSHA256")
)
actual_11b_manifest_hash = sha256_file(PART11B_SNAPSHOT_MANIFEST)
if recorded_11b_manifest_hash != actual_11b_manifest_hash:
    raise AssertionError(
        "Part 11B snapshot manifest no longer matches the hash recorded in the Part 11B lock"
    )
if part11b_lock_payload.get("Status") != "PART_11B_COMPLETED_READY_FOR_11C":
    raise AssertionError(
        "Part 11B lock status is not ready for Part 11C: "
        f"{part11b_lock_payload.get('Status')}"
    )
if part11b_lock_payload.get("ReadyForPart11C") is not True:
    raise AssertionError("Part 11B lock does not authorise Part 11C")

safety_11b = part11b_lock_payload.get("SafetyAssertions", {})
if safety_11b.get("ProtectedTargetVaultCopied") is not False:
    raise AssertionError("Part 11B safety assertion for protected target copy is invalid")
if safety_11b.get("WeeklyDatasetBuilt") is not False:
    raise AssertionError("Part 11B unexpectedly reports that a weekly dataset was built")

require_file(PART11B_SNAPSHOT_MANIFEST, "Part 11B snapshot manifest")
snapshot_manifest_11b = pd.read_csv(PART11B_SNAPSHOT_MANIFEST)
required_manifest_columns = {"RelativePath", "SHA256"}
if not required_manifest_columns.issubset(snapshot_manifest_11b.columns):
    raise AssertionError(
        "Part 11B snapshot manifest is missing required columns: "
        f"{sorted(required_manifest_columns - set(snapshot_manifest_11b.columns))}"
    )

snapshot_integrity_rows = []
for row in snapshot_manifest_11b.itertuples(index=False):
    relative_path = Path(str(row.RelativePath))
    file_path = EXT_ROOT / relative_path
    exists = file_path.is_file()
    actual_hash = sha256_file(file_path) if exists else None
    expected_hash = str(row.SHA256)
    matched = bool(exists and actual_hash == expected_hash)
    snapshot_integrity_rows.append(
        {
            "RelativePath": str(relative_path),
            "Exists": exists,
            "ExpectedSHA256": expected_hash,
            "ActualSHA256": actual_hash,
            "HashMatched": matched,
        }
    )

snapshot_integrity_df = pd.DataFrame(snapshot_integrity_rows)
if not snapshot_integrity_df["HashMatched"].all():
    raise AssertionError(
        "Part 11B snapshot integrity verification failed:\n"
        + snapshot_integrity_df.loc[~snapshot_integrity_df["HashMatched"]].to_string(
            index=False
        )
    )


# =============================================================================
# 4. VERIFY FINAL FORECAST-PREPARATION SOURCES AND CONTRACTS
# =============================================================================

for contract_name in LOCAL_CONTRACT_FILES:
    require_file(
        FORECAST_PREPARATION_ROOT / contract_name,
        f"final forecast-preparation contract {contract_name}",
    )

require_file(FINAL_MODEL_PREPARATION_COPY, "final model dataset preparation copy")
require_file(FINAL_MODEL_EDEN_COPY, "final model dataset Eden-root copy")
require_file(CANONICAL_SNAPSHOT_SOURCE, "locked canonical component source")

hash_manifest_path = FORECAST_PREPARATION_ROOT / "17_final_file_hash_manifest.csv"
final_hash_manifest = pd.read_csv(hash_manifest_path)
required_hash_manifest_columns = {"FileLabel", "FilePath", "FileSizeBytes", "SHA256"}
if not required_hash_manifest_columns.issubset(final_hash_manifest.columns):
    raise AssertionError("17_final_file_hash_manifest.csv has an unexpected schema")

expected_hash_rows = final_hash_manifest.loc[
    final_hash_manifest["FileLabel"].isin(
        ["FINAL_CANONICAL_PREPARATION_COPY", "FINAL_CANONICAL_EDEN_DATASETS_COPY"]
    )
].copy()
if len(expected_hash_rows) != 2:
    raise AssertionError(
        "Final hash manifest must contain both final canonical dataset copies"
    )
if expected_hash_rows["SHA256"].nunique() != 1:
    raise AssertionError("The manifest records conflicting hashes for the two final copies")
manifest_final_hash = str(expected_hash_rows["SHA256"].iloc[0])
if manifest_final_hash != EXPECTED_FINAL_MODEL_SHA256:
    raise AssertionError(
        "Final model dataset hash differs from the authoritative project handoff:\n"
        f"Manifest: {manifest_final_hash}\nExpected: {EXPECTED_FINAL_MODEL_SHA256}"
    )

source_resolution_records = []
for role, source_path in [
    ("FINAL_MODEL_PREPARATION_COPY", FINAL_MODEL_PREPARATION_COPY),
    ("FINAL_MODEL_EDEN_ROOT_COPY", FINAL_MODEL_EDEN_COPY),
]:
    actual_hash = sha256_file(source_path)
    source_resolution_records.append(
        {
            "Role": role,
            "SourcePath": str(source_path),
            "Bytes": int(source_path.stat().st_size),
            "ExpectedSHA256": EXPECTED_FINAL_MODEL_SHA256,
            "ActualSHA256": actual_hash,
            "HashMatched": actual_hash == EXPECTED_FINAL_MODEL_SHA256,
        }
    )
source_resolution_df = pd.DataFrame(source_resolution_records)
if not source_resolution_df["HashMatched"].all():
    raise AssertionError(
        "One or both final model dataset copies failed SHA-256 verification:\n"
        + source_resolution_df.to_string(index=False)
    )

final_model_df = pd.read_csv(FINAL_MODEL_PREPARATION_COPY, low_memory=False)
feature_contract = pd.read_csv(
    FORECAST_PREPARATION_ROOT / "17_frozen_model_feature_contract.csv"
).sort_values("ColumnOrder")
contract_columns = feature_contract["Column"].astype(str).tolist()

required_final_columns = {
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "TotalDemand",
}
if not required_final_columns.issubset(final_model_df.columns):
    raise AssertionError(
        "Final model dataset is missing required columns: "
        f"{sorted(required_final_columns - set(final_model_df.columns))}"
    )
if final_model_df.columns.tolist() != contract_columns:
    missing_from_data = [column for column in contract_columns if column not in final_model_df]
    extra_in_data = [column for column in final_model_df if column not in contract_columns]
    raise AssertionError(
        "Final model dataset does not match the frozen 58-column feature contract.\n"
        f"Missing: {missing_from_data}\nExtra: {extra_in_data}"
    )

final_model_df["Date"] = pd.to_datetime(final_model_df["Date"], errors="raise")
final_model_df["ProductFirstObservedDate"] = pd.to_datetime(
    final_model_df["ProductFirstObservedDate"], errors="raise"
)
final_model_df["CanonicalProductID"] = final_model_df["CanonicalProductID"].astype(
    "string"
)
final_model_df["CanonicalProductName"] = final_model_df[
    "CanonicalProductName"
].astype("string")
check_integer_like(final_model_df["TotalDemand"], "final model TotalDemand")
final_model_df["TotalDemand"] = pd.to_numeric(
    final_model_df["TotalDemand"], errors="raise"
).round().astype("int64")

final_duplicate_keys = int(
    final_model_df.duplicated(["Date", "CanonicalProductID"]).sum()
)

final_source_profile_rows = [
    {
        "Metric": "Rows",
        "Expected": EXPECTED_FINAL_ROWS,
        "Actual": int(len(final_model_df)),
        "Passed": len(final_model_df) == EXPECTED_FINAL_ROWS,
    },
    {
        "Metric": "Columns",
        "Expected": EXPECTED_FINAL_COLUMNS,
        "Actual": int(final_model_df.shape[1]),
        "Passed": final_model_df.shape[1] == EXPECTED_FINAL_COLUMNS,
    },
    {
        "Metric": "Products",
        "Expected": EXPECTED_PRODUCTS,
        "Actual": int(final_model_df["CanonicalProductID"].nunique()),
        "Passed": final_model_df["CanonicalProductID"].nunique() == EXPECTED_PRODUCTS,
    },
    {
        "Metric": "OperatingDates",
        "Expected": EXPECTED_OPERATING_DATES,
        "Actual": int(final_model_df["Date"].nunique()),
        "Passed": final_model_df["Date"].nunique() == EXPECTED_OPERATING_DATES,
    },
    {
        "Metric": "DateMin",
        "Expected": EXPECTED_DATE_MIN,
        "Actual": final_model_df["Date"].min().strftime("%Y-%m-%d"),
        "Passed": final_model_df["Date"].min().strftime("%Y-%m-%d")
        == EXPECTED_DATE_MIN,
    },
    {
        "Metric": "DateMax",
        "Expected": EXPECTED_DATE_MAX,
        "Actual": final_model_df["Date"].max().strftime("%Y-%m-%d"),
        "Passed": final_model_df["Date"].max().strftime("%Y-%m-%d")
        == EXPECTED_DATE_MAX,
    },
    {
        "Metric": "TotalDemandUnits",
        "Expected": EXPECTED_TOTAL_DEMAND,
        "Actual": int(final_model_df["TotalDemand"].sum()),
        "Passed": int(final_model_df["TotalDemand"].sum())
        == EXPECTED_TOTAL_DEMAND,
    },
    {
        "Metric": "DuplicateProductDateRows",
        "Expected": 0,
        "Actual": final_duplicate_keys,
        "Passed": final_duplicate_keys == 0,
    },
    {
        "Metric": "FrozenContractColumnsInExactOrder",
        "Expected": True,
        "Actual": final_model_df.columns.tolist() == contract_columns,
        "Passed": final_model_df.columns.tolist() == contract_columns,
    },
]
final_source_profile_df = pd.DataFrame(final_source_profile_rows)
if not final_source_profile_df["Passed"].all():
    raise AssertionError(
        "Final model source contract failed:\n"
        + final_source_profile_df.loc[~final_source_profile_df["Passed"]].to_string(
            index=False
        )
    )

leakage_audit = pd.read_csv(
    FORECAST_PREPARATION_ROOT / "17_final_leakage_audit.csv"
)
if "LeakageColumnCount" not in leakage_audit.columns:
    raise AssertionError("Final leakage audit has an unexpected schema")
if pd.to_numeric(leakage_audit["LeakageColumnCount"], errors="raise").sum() != 0:
    raise AssertionError("Final forecast-preparation leakage audit is not clean")

protocol_contract = pd.read_csv(
    FORECAST_PREPARATION_ROOT / "17_forecasting_protocol_contract.csv"
)
protocol_lookup = dict(
    zip(
        protocol_contract["ProtocolParameter"].astype(str),
        protocol_contract["Value"].astype(str),
    )
)
if protocol_lookup.get("ForecastTarget") != "TotalDemand":
    raise AssertionError("Forecasting protocol target is not TotalDemand")
if protocol_lookup.get("RandomSplittingAllowed") != "False":
    raise AssertionError("Forecasting protocol unexpectedly allows random splitting")


# =============================================================================
# 5. RECONCILE NORMAL AND BULK COMPONENTS TO FINAL MODEL LINEAGE
# =============================================================================

canonical_df = pd.read_csv(CANONICAL_SNAPSHOT_SOURCE, low_memory=False)
required_canonical_columns = {
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
}
if not required_canonical_columns.issubset(canonical_df.columns):
    raise AssertionError(
        "Canonical snapshot is missing required component columns: "
        f"{sorted(required_canonical_columns - set(canonical_df.columns))}"
    )
canonical_df["Date"] = pd.to_datetime(canonical_df["Date"], errors="raise")
canonical_df["CanonicalProductID"] = canonical_df["CanonicalProductID"].astype(
    "string"
)
for target_column in ["NormalDemand", "BulkDemand", "TotalDemand"]:
    check_integer_like(canonical_df[target_column], f"canonical {target_column}")
    canonical_df[target_column] = pd.to_numeric(
        canonical_df[target_column], errors="raise"
    ).round().astype("int64")

canonical_duplicate_keys = int(
    canonical_df.duplicated(["Date", "CanonicalProductID"]).sum()
)
if canonical_duplicate_keys != 0:
    raise AssertionError(
        f"Canonical snapshot has {canonical_duplicate_keys} duplicate product-date keys"
    )
canonical_component_mismatches = int(
    (
        canonical_df["TotalDemand"]
        != canonical_df["NormalDemand"] + canonical_df["BulkDemand"]
    ).sum()
)
if canonical_component_mismatches != 0:
    raise AssertionError(
        f"Canonical snapshot has {canonical_component_mismatches} component mismatches"
    )

lineage_columns = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "TotalDemand",
]
lineage_daily = final_model_df[lineage_columns].copy()
lineage_daily = lineage_daily.rename(columns={"TotalDemand": "LineageTotalDemand"})

canonical_components = canonical_df[
    ["Date", "CanonicalProductID", "NormalDemand", "BulkDemand", "TotalDemand"]
].rename(columns={"TotalDemand": "CanonicalTotalDemand"})

reconciled_daily = lineage_daily.merge(
    canonical_components,
    on=["Date", "CanonicalProductID"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

canonical_keys = canonical_components[["Date", "CanonicalProductID", "CanonicalTotalDemand"]]
lineage_keys = lineage_daily[["Date", "CanonicalProductID"]]
canonical_only = canonical_keys.merge(
    lineage_keys,
    on=["Date", "CanonicalProductID"],
    how="left",
    indicator=True,
)
canonical_only = canonical_only.loc[canonical_only["_merge"] == "left_only"].copy()
canonical_only_positive_rows = int((canonical_only["CanonicalTotalDemand"] > 0).sum())
if canonical_only_positive_rows != 0:
    raise AssertionError(
        "The final model lineage omits positive-demand rows from the canonical source"
    )

matched_mask = reconciled_daily["_merge"] == "both"
matched_total_mismatch = int(
    (
        reconciled_daily.loc[matched_mask, "LineageTotalDemand"]
        != reconciled_daily.loc[matched_mask, "CanonicalTotalDemand"]
    ).sum()
)
if matched_total_mismatch != 0:
    raise AssertionError(
        f"Matched final-lineage rows have {matched_total_mismatch} TotalDemand mismatches"
    )

lineage_only_mask = reconciled_daily["_merge"] == "left_only"
lineage_only_positive_rows = int(
    (reconciled_daily.loc[lineage_only_mask, "LineageTotalDemand"] > 0).sum()
)
if lineage_only_positive_rows != 0:
    raise AssertionError(
        "Final model lineage contains positive-demand rows without canonical demand components"
    )

reconciled_daily["NormalDemand"] = reconciled_daily["NormalDemand"].fillna(0)
reconciled_daily["BulkDemand"] = reconciled_daily["BulkDemand"].fillna(0)
reconciled_daily["CanonicalTotalDemand"] = reconciled_daily[
    "CanonicalTotalDemand"
].fillna(0)
for column in ["NormalDemand", "BulkDemand", "CanonicalTotalDemand"]:
    reconciled_daily[column] = reconciled_daily[column].astype("int64")

reconciled_daily["TotalDemand"] = reconciled_daily["LineageTotalDemand"].astype(
    "int64"
)
reconciled_daily["ComponentTotal"] = (
    reconciled_daily["NormalDemand"] + reconciled_daily["BulkDemand"]
)
lineage_component_mismatches = int(
    (reconciled_daily["TotalDemand"] != reconciled_daily["ComponentTotal"]).sum()
)
if lineage_component_mismatches != 0:
    raise AssertionError(
        f"Reconciled final lineage has {lineage_component_mismatches} component mismatches"
    )

reconciliation_audit_rows = [
    {
        "Metric": "FinalLineageRows",
        "Expected": EXPECTED_FINAL_ROWS,
        "Actual": int(len(reconciled_daily)),
        "Passed": len(reconciled_daily) == EXPECTED_FINAL_ROWS,
    },
    {
        "Metric": "CanonicalRows",
        "Expected": 25_405,
        "Actual": int(len(canonical_df)),
        "Passed": len(canonical_df) == 25_405,
    },
    {
        "Metric": "MatchedProductDateRows",
        "Expected": "informational",
        "Actual": int(matched_mask.sum()),
        "Passed": True,
    },
    {
        "Metric": "LineageOnlyRowsFilledWithZeroComponents",
        "Expected": "zero-demand only",
        "Actual": int(lineage_only_mask.sum()),
        "Passed": lineage_only_positive_rows == 0,
    },
    {
        "Metric": "CanonicalOnlyRowsExcludedFromLineage",
        "Expected": "zero-demand only",
        "Actual": int(len(canonical_only)),
        "Passed": canonical_only_positive_rows == 0,
    },
    {
        "Metric": "MatchedTotalDemandMismatches",
        "Expected": 0,
        "Actual": matched_total_mismatch,
        "Passed": matched_total_mismatch == 0,
    },
    {
        "Metric": "LineageComponentMismatches",
        "Expected": 0,
        "Actual": lineage_component_mismatches,
        "Passed": lineage_component_mismatches == 0,
    },
    {
        "Metric": "NormalDemandUnits",
        "Expected": EXPECTED_NORMAL_DEMAND,
        "Actual": int(reconciled_daily["NormalDemand"].sum()),
        "Passed": int(reconciled_daily["NormalDemand"].sum())
        == EXPECTED_NORMAL_DEMAND,
    },
    {
        "Metric": "BulkDemandUnits",
        "Expected": EXPECTED_BULK_DEMAND,
        "Actual": int(reconciled_daily["BulkDemand"].sum()),
        "Passed": int(reconciled_daily["BulkDemand"].sum())
        == EXPECTED_BULK_DEMAND,
    },
    {
        "Metric": "TotalDemandUnits",
        "Expected": EXPECTED_TOTAL_DEMAND,
        "Actual": int(reconciled_daily["TotalDemand"].sum()),
        "Passed": int(reconciled_daily["TotalDemand"].sum())
        == EXPECTED_TOTAL_DEMAND,
    },
]
reconciliation_audit_df = pd.DataFrame(reconciliation_audit_rows)
if not reconciliation_audit_df["Passed"].all():
    raise AssertionError(
        "Daily lineage reconciliation failed:\n"
        + reconciliation_audit_df.loc[
            ~reconciliation_audit_df["Passed"]
        ].to_string(index=False)
    )


# =============================================================================
# 6. CREATE MONDAY–SUNDAY WEEKLY PRODUCT-DEMAND DATASET
# =============================================================================

holdout_governance_df = pd.read_csv(PART11B_HOLDOUT_GOVERNANCE)
if len(holdout_governance_df) != 1:
    raise AssertionError("Part 11B holdout-governance file must contain one row")
holdout_start = pd.Timestamp(
    str(holdout_governance_df.loc[0, "EvaluationStartDate"])
)
holdout_end = pd.Timestamp(str(holdout_governance_df.loc[0, "EvaluationEndDate"]))
if holdout_start.strftime("%Y-%m-%d") != EXPECTED_HOLDOUT_START:
    raise AssertionError("Unexpected opened final-holdout start date")
if holdout_end.strftime("%Y-%m-%d") != EXPECTED_HOLDOUT_END:
    raise AssertionError("Unexpected opened final-holdout end date")

reconciled_daily["WeekStartDate"] = reconciled_daily["Date"] - pd.to_timedelta(
    reconciled_daily["Date"].dt.weekday, unit="D"
)
reconciled_daily["WeekEndDate"] = reconciled_daily["WeekStartDate"] + pd.Timedelta(
    days=6
)
reconciled_daily["ContainsOpenedDailyHoldoutRow"] = reconciled_daily[
    "Date"
].between(holdout_start, holdout_end, inclusive="both")
reconciled_daily["PositiveDemandDay"] = (
    reconciled_daily["TotalDemand"] > 0
).astype("int64")
reconciled_daily["NormalDemandDay"] = (
    reconciled_daily["NormalDemand"] > 0
).astype("int64")
reconciled_daily["BulkDemandDay"] = (
    reconciled_daily["BulkDemand"] > 0
).astype("int64")

operating_calendar = (
    reconciled_daily[["Date", "WeekStartDate", "WeekEndDate"]]
    .drop_duplicates()
    .sort_values("Date")
)
week_calendar = (
    operating_calendar.groupby(["WeekStartDate", "WeekEndDate"], as_index=False)
    .agg(
        FirstOperatingDateInWeek=("Date", "min"),
        LastOperatingDateInWeek=("Date", "max"),
        AvailableOperatingDaysInWeek=("Date", "nunique"),
    )
    .sort_values("WeekStartDate")
    .reset_index(drop=True)
)
week_calendar["WeekSequence"] = np.arange(1, len(week_calendar) + 1, dtype=int)
week_iso = week_calendar["WeekStartDate"].dt.isocalendar()
week_calendar["ISOYear"] = week_iso["year"].astype(int)
week_calendar["ISOWeek"] = week_iso["week"].astype(int)
week_calendar["WeekID"] = (
    week_calendar["ISOYear"].astype(str)
    + "-W"
    + week_calendar["ISOWeek"].astype(str).str.zfill(2)
)
week_calendar["IsDatasetBoundaryPartialWeek"] = (
    (week_calendar["WeekStartDate"] < reconciled_daily["Date"].min())
    | (week_calendar["WeekEndDate"] > reconciled_daily["Date"].max())
)
week_calendar["ContainsOpenedDailyHoldoutRows"] = week_calendar.apply(
    lambda row: bool(
        (row["LastOperatingDateInWeek"] >= holdout_start)
        and (row["FirstOperatingDateInWeek"] <= holdout_end)
    ),
    axis=1,
)
week_calendar["WeeklyTuningEligible"] = ~week_calendar[
    "ContainsOpenedDailyHoldoutRows"
]

product_name_counts = (
    reconciled_daily.groupby("CanonicalProductID")["CanonicalProductName"]
    .nunique(dropna=True)
    .rename("DistinctNonNullNames")
)
latest_names = (
    reconciled_daily.dropna(subset=["CanonicalProductName"])
    .sort_values(["CanonicalProductID", "Date"])
    .groupby("CanonicalProductID", as_index=False)
    .tail(1)[["CanonicalProductID", "CanonicalProductName"]]
)
product_first_dates = (
    reconciled_daily.groupby("CanonicalProductID", as_index=False)
    .agg(ProductFirstObservedDate=("ProductFirstObservedDate", "min"))
)
product_reference = product_first_dates.merge(
    latest_names, on="CanonicalProductID", how="left", validate="one_to_one"
).merge(
    product_name_counts.reset_index(),
    on="CanonicalProductID",
    how="left",
    validate="one_to_one",
)
if product_reference["CanonicalProductName"].isna().any():
    missing_ids = product_reference.loc[
        product_reference["CanonicalProductName"].isna(), "CanonicalProductID"
    ].tolist()
    raise AssertionError(f"Products without a canonical display name: {missing_ids}")
product_reference["ProductFirstWeekStartDate"] = product_reference[
    "ProductFirstObservedDate"
] - pd.to_timedelta(
    product_reference["ProductFirstObservedDate"].dt.weekday, unit="D"
)

weekly_core = (
    reconciled_daily.groupby(
        ["WeekStartDate", "WeekEndDate", "CanonicalProductID"], as_index=False
    )
    .agg(
        ProductOperatingRowsInWeek=("Date", "size"),
        ProductFirstOperatingDateInWeek=("Date", "min"),
        ProductLastOperatingDateInWeek=("Date", "max"),
        PositiveDemandOperatingDays=("PositiveDemandDay", "sum"),
        NormalDemandOperatingDays=("NormalDemandDay", "sum"),
        BulkDemandOperatingDays=("BulkDemandDay", "sum"),
        OpenedDailyHoldoutOperatingRows=(
            "ContainsOpenedDailyHoldoutRow",
            "sum",
        ),
        WeeklyNormalDemand=("NormalDemand", "sum"),
        WeeklyBulkDemand=("BulkDemand", "sum"),
        WeeklyTotalDemand=("TotalDemand", "sum"),
    )
)
weekly_core["ZeroDemandOperatingDays"] = (
    weekly_core["ProductOperatingRowsInWeek"]
    - weekly_core["PositiveDemandOperatingDays"]
)

weekly_df = (
    weekly_core.merge(
        week_calendar,
        on=["WeekStartDate", "WeekEndDate"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        product_reference,
        on="CanonicalProductID",
        how="left",
        validate="many_to_one",
    )
    .sort_values(["WeekStartDate", "CanonicalProductID"])
    .reset_index(drop=True)
)

weekly_df["ProductWeekSequence"] = (
    weekly_df.groupby("CanonicalProductID").cumcount() + 1
)
weekly_df["ProductCoverageFractionOfOperatingDays"] = (
    weekly_df["ProductOperatingRowsInWeek"]
    / weekly_df["AvailableOperatingDaysInWeek"]
)
weekly_df["IsPositiveDemandWeek"] = weekly_df["WeeklyTotalDemand"] > 0
weekly_df["IsBulkDemandWeek"] = weekly_df["WeeklyBulkDemand"] > 0
weekly_df["ContainsOpenedDailyHoldoutRows"] = (
    weekly_df["OpenedDailyHoldoutOperatingRows"] > 0
)
weekly_df["WeeklyTuningEligible"] = ~weekly_df[
    "ContainsOpenedDailyHoldoutRows"
]
weekly_df["WeeklyDemandComponentMismatch"] = (
    weekly_df["WeeklyTotalDemand"]
    != weekly_df["WeeklyNormalDemand"] + weekly_df["WeeklyBulkDemand"]
)
weekly_df["WeekDefinition"] = "MONDAY_TO_SUNDAY"
weekly_df["SourceLineage"] = (
    "FINAL_FORECAST_PREPARATION_PANEL_PLUS_CANONICAL_DEMAND_COMPONENTS"
)

integer_columns = [
    "WeekSequence",
    "ISOYear",
    "ISOWeek",
    "AvailableOperatingDaysInWeek",
    "ProductOperatingRowsInWeek",
    "PositiveDemandOperatingDays",
    "ZeroDemandOperatingDays",
    "NormalDemandOperatingDays",
    "BulkDemandOperatingDays",
    "OpenedDailyHoldoutOperatingRows",
    "ProductWeekSequence",
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
    "DistinctNonNullNames",
]
for column in integer_columns:
    weekly_df[column] = pd.to_numeric(weekly_df[column], errors="raise").astype(
        "int64"
    )

weekly_columns = [
    "WeekStartDate",
    "WeekEndDate",
    "ISOYear",
    "ISOWeek",
    "WeekID",
    "WeekSequence",
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "ProductFirstWeekStartDate",
    "ProductWeekSequence",
    "FirstOperatingDateInWeek",
    "LastOperatingDateInWeek",
    "ProductFirstOperatingDateInWeek",
    "ProductLastOperatingDateInWeek",
    "AvailableOperatingDaysInWeek",
    "ProductOperatingRowsInWeek",
    "ProductCoverageFractionOfOperatingDays",
    "PositiveDemandOperatingDays",
    "ZeroDemandOperatingDays",
    "NormalDemandOperatingDays",
    "BulkDemandOperatingDays",
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
    "IsPositiveDemandWeek",
    "IsBulkDemandWeek",
    "IsDatasetBoundaryPartialWeek",
    "ContainsOpenedDailyHoldoutRows",
    "OpenedDailyHoldoutOperatingRows",
    "WeeklyTuningEligible",
    "DistinctNonNullNames",
    "WeekDefinition",
    "SourceLineage",
]
weekly_df = weekly_df[weekly_columns]

weekly_duplicate_keys = int(
    weekly_df.duplicated(["WeekStartDate", "CanonicalProductID"]).sum()
)
weekly_component_mismatches = int(weekly_df["WeeklyDemandComponentMismatch"].sum()) if "WeeklyDemandComponentMismatch" in weekly_df.columns else int(
    (
        weekly_df["WeeklyTotalDemand"]
        != weekly_df["WeeklyNormalDemand"] + weekly_df["WeeklyBulkDemand"]
    ).sum()
)

expected_weekly_rows = int(
    reconciled_daily[["WeekStartDate", "CanonicalProductID"]]
    .drop_duplicates()
    .shape[0]
)

weekly_audit_rows = [
    {
        "Metric": "WeeklyRows",
        "Expected": expected_weekly_rows,
        "Actual": int(len(weekly_df)),
        "Passed": len(weekly_df) == expected_weekly_rows,
    },
    {
        "Metric": "Products",
        "Expected": EXPECTED_PRODUCTS,
        "Actual": int(weekly_df["CanonicalProductID"].nunique()),
        "Passed": weekly_df["CanonicalProductID"].nunique() == EXPECTED_PRODUCTS,
    },
    {
        "Metric": "CalendarWeeks",
        "Expected": int(len(week_calendar)),
        "Actual": int(weekly_df["WeekStartDate"].nunique()),
        "Passed": weekly_df["WeekStartDate"].nunique() == len(week_calendar),
    },
    {
        "Metric": "DuplicateProductWeekRows",
        "Expected": 0,
        "Actual": weekly_duplicate_keys,
        "Passed": weekly_duplicate_keys == 0,
    },
    {
        "Metric": "WeeklyComponentMismatches",
        "Expected": 0,
        "Actual": weekly_component_mismatches,
        "Passed": weekly_component_mismatches == 0,
    },
    {
        "Metric": "WeeklyNormalDemandUnits",
        "Expected": EXPECTED_NORMAL_DEMAND,
        "Actual": int(weekly_df["WeeklyNormalDemand"].sum()),
        "Passed": int(weekly_df["WeeklyNormalDemand"].sum())
        == EXPECTED_NORMAL_DEMAND,
    },
    {
        "Metric": "WeeklyBulkDemandUnits",
        "Expected": EXPECTED_BULK_DEMAND,
        "Actual": int(weekly_df["WeeklyBulkDemand"].sum()),
        "Passed": int(weekly_df["WeeklyBulkDemand"].sum())
        == EXPECTED_BULK_DEMAND,
    },
    {
        "Metric": "WeeklyTotalDemandUnits",
        "Expected": EXPECTED_TOTAL_DEMAND,
        "Actual": int(weekly_df["WeeklyTotalDemand"].sum()),
        "Passed": int(weekly_df["WeeklyTotalDemand"].sum())
        == EXPECTED_TOTAL_DEMAND,
    },
    {
        "Metric": "NegativeWeeklyTargetRows",
        "Expected": 0,
        "Actual": int(
            (
                weekly_df[
                    ["WeeklyNormalDemand", "WeeklyBulkDemand", "WeeklyTotalDemand"]
                ]
                < 0
            ).any(axis=1).sum()
        ),
        "Passed": not (
            weekly_df[
                ["WeeklyNormalDemand", "WeeklyBulkDemand", "WeeklyTotalDemand"]
            ]
            < 0
        ).any(axis=1).any(),
    },
    {
        "Metric": "OpenedHoldoutRowsMarkedTuningEligible",
        "Expected": 0,
        "Actual": int(
            (
                weekly_df["ContainsOpenedDailyHoldoutRows"]
                & weekly_df["WeeklyTuningEligible"]
            ).sum()
        ),
        "Passed": not (
            weekly_df["ContainsOpenedDailyHoldoutRows"]
            & weekly_df["WeeklyTuningEligible"]
        ).any(),
    },
    {
        "Metric": "DailyLagOrRollingColumnsCarriedIntoWeeklyDataset",
        "Expected": 0,
        "Actual": int(
            sum(
                (
                    "Lag_" in column
                    or "Rolling" in column
                    or column.startswith("Past")
                    or column.startswith("Expanding")
                )
                for column in weekly_df.columns
            )
        ),
        "Passed": not any(
            (
                "Lag_" in column
                or "Rolling" in column
                or column.startswith("Past")
                or column.startswith("Expanding")
            )
            for column in weekly_df.columns
        ),
    },
]
weekly_audit_df = pd.DataFrame(weekly_audit_rows)
if not weekly_audit_df["Passed"].all():
    raise AssertionError(
        "Weekly dataset audit failed:\n"
        + weekly_audit_df.loc[~weekly_audit_df["Passed"]].to_string(index=False)
    )

week_summary_df = (
    weekly_df.groupby(
        [
            "WeekStartDate",
            "WeekEndDate",
            "WeekID",
            "WeekSequence",
            "AvailableOperatingDaysInWeek",
            "IsDatasetBoundaryPartialWeek",
            "ContainsOpenedDailyHoldoutRows",
            "WeeklyTuningEligible",
        ],
        as_index=False,
    )
    .agg(
        ProductWeeks=("CanonicalProductID", "size"),
        PositiveDemandProductWeeks=("IsPositiveDemandWeek", "sum"),
        BulkDemandProductWeeks=("IsBulkDemandWeek", "sum"),
        WeeklyNormalDemand=("WeeklyNormalDemand", "sum"),
        WeeklyBulkDemand=("WeeklyBulkDemand", "sum"),
        WeeklyTotalDemand=("WeeklyTotalDemand", "sum"),
    )
    .sort_values("WeekStartDate")
)

product_week_summary_df = (
    weekly_df.groupby(
        ["CanonicalProductID", "CanonicalProductName"], as_index=False
    )
    .agg(
        FirstWeekStartDate=("WeekStartDate", "min"),
        LastWeekStartDate=("WeekStartDate", "max"),
        ProductWeeks=("WeekStartDate", "nunique"),
        TuningEligibleWeeks=("WeeklyTuningEligible", "sum"),
        PositiveDemandWeeks=("IsPositiveDemandWeek", "sum"),
        BulkDemandWeeks=("IsBulkDemandWeek", "sum"),
        WeeklyNormalDemandTotal=("WeeklyNormalDemand", "sum"),
        WeeklyBulkDemandTotal=("WeeklyBulkDemand", "sum"),
        WeeklyTotalDemandTotal=("WeeklyTotalDemand", "sum"),
    )
    .sort_values(
        ["WeeklyTotalDemandTotal", "CanonicalProductID"],
        ascending=[False, True],
    )
)

weekly_schema_df = pd.DataFrame(
    {
        "ColumnOrder": range(len(weekly_df.columns)),
        "Column": weekly_df.columns,
        "DataType": [str(weekly_df[column].dtype) for column in weekly_df.columns],
        "MissingCount": [int(weekly_df[column].isna().sum()) for column in weekly_df.columns],
        "UniqueValues": [int(weekly_df[column].nunique(dropna=True)) for column in weekly_df.columns],
    }
)


# =============================================================================
# 7. STAGE SOURCE AMENDMENT, WEEKLY OUTPUTS, MEMORY AND LOCKS
# =============================================================================

stage_root = EXT_ROOT / f".11C_staging_{uuid.uuid4().hex}"
stage_root.mkdir(parents=True, exist_ok=False)

memory_files = [
    MEMORY_ROOT / "PROJECT_CONTEXT.md",
    MEMORY_ROOT / "WORKFLOW.md",
    MEMORY_ROOT / "DECISIONS.md",
    MEMORY_ROOT / "FILES_AND_PATHS.md",
    MEMORY_ROOT / "METRICS_AND_RESULTS.md",
    MEMORY_ROOT / "CHAT_INDEX.md",
    MEMORY_ROOT / "CURRENT_HANDOFF.md",
]
for path in memory_files:
    require_file(path, f"memory file {path.name}")

original_memory = {path: path.read_text(encoding="utf-8") for path in memory_files}
created_final_paths: list[Path] = []


def staged_path(final_path: Path) -> Path:
    return stage_root / final_path.relative_to(EXT_ROOT)


def stage_text(final_path: Path, text: str) -> None:
    atomic_write_text(staged_path(final_path), text)


def stage_json(final_path: Path, payload: dict) -> None:
    atomic_write_json(staged_path(final_path), payload)


def stage_csv(final_path: Path, frame: pd.DataFrame) -> None:
    atomic_write_csv(staged_path(final_path), frame)


try:
    # ------------------------- source amendment copies -------------------------
    source_copy_plan = [
        {
            "Role": "FINAL_FORECAST_PREPARATION_MODEL_PANEL",
            "Source": FINAL_MODEL_PREPARATION_COPY,
            "Destination": SNAPSHOT_DATA_ROOT
            / "11C_forecasting_preparation_source"
            / FINAL_MODEL_PREPARATION_COPY.name,
            "ExpectedSHA256": EXPECTED_FINAL_MODEL_SHA256,
        }
    ]
    for contract_name in LOCAL_CONTRACT_FILES:
        contract_path = FORECAST_PREPARATION_ROOT / contract_name
        manifest_match = final_hash_manifest.loc[
            final_hash_manifest["FilePath"].astype(str) == str(contract_path), "SHA256"
        ]
        expected_hash = (
            str(manifest_match.iloc[0]) if len(manifest_match) == 1 else sha256_file(contract_path)
        )
        source_copy_plan.append(
            {
                "Role": f"FORECAST_PREPARATION_CONTRACT::{contract_name}",
                "Source": contract_path,
                "Destination": SNAPSHOT_CONTRACT_ROOT
                / "11C_forecasting_preparation"
                / contract_name,
                "ExpectedSHA256": expected_hash,
            }
        )

    source_copy_records = []
    for item in source_copy_plan:
        source = Path(item["Source"])
        destination = Path(item["Destination"])
        if protected_target_name(source) or protected_target_name(destination):
            raise PermissionError(f"Protected target detected in Part 11C copy plan: {source}")
        require_file(source, item["Role"])
        actual_source_hash = sha256_file(source)
        if actual_source_hash != item["ExpectedSHA256"]:
            raise AssertionError(
                f"Source hash mismatch for {item['Role']}: {source}"
            )
        stage_destination = staged_path(destination)
        stage_destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, stage_destination)
        staged_hash = sha256_file(stage_destination)
        if staged_hash != actual_source_hash:
            raise AssertionError(f"Staged copy hash mismatch: {stage_destination}")
        source_copy_records.append(
            {
                "Role": item["Role"],
                "SourcePath": str(source),
                "DestinationRelativePath": str(destination.relative_to(EXT_ROOT)),
                "Bytes": int(source.stat().st_size),
                "ExpectedSHA256": item["ExpectedSHA256"],
                "SourceSHA256": actual_source_hash,
                "StagedSHA256": staged_hash,
                "HashMatched": staged_hash == actual_source_hash,
            }
        )
    source_copy_audit_df = pd.DataFrame(source_copy_records)

    source_resolution_path = SNAPSHOT_MANIFEST_ROOT / "11C_source_resolution_audit.csv"
    source_copy_audit_path = SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_copy_audit.csv"
    source_amendment_manifest_path = (
        SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_manifest.csv"
    )
    stage_csv(source_resolution_path, source_resolution_df)
    stage_csv(source_copy_audit_path, source_copy_audit_df)

    source_amendment_manifest_records = []
    for item in source_copy_plan:
        destination = Path(item["Destination"])
        stage_destination = staged_path(destination)
        source_amendment_manifest_records.append(
            {
                "RelativePath": str(destination.relative_to(EXT_ROOT)),
                "FileName": destination.name,
                "Bytes": int(stage_destination.stat().st_size),
                "SHA256": sha256_file(stage_destination),
                "ReadOnlyAfterCommit": True,
                "CreatedByStep": STEP_ID,
            }
        )
    source_amendment_manifest_df = pd.DataFrame(source_amendment_manifest_records)
    stage_csv(source_amendment_manifest_path, source_amendment_manifest_df)

    # ------------------------- weekly outputs -------------------------
    reconciliation_path = (
        WEEKLY_AUDITED_ROOT / "11C_daily_lineage_reconciliation_audit.csv"
    )
    weekly_calendar_path = WEEKLY_AUDITED_ROOT / "11C_weekly_calendar.csv"
    week_summary_path = WEEKLY_AUDITED_ROOT / "11C_week_summary.csv"
    product_summary_path = WEEKLY_AUDITED_ROOT / "11C_product_week_summary.csv"
    weekly_schema_path = WEEKLY_AUDITED_ROOT / "11C_weekly_dataset_schema.csv"
    weekly_audit_path = WEEKLY_AUDITED_ROOT / "11C_weekly_dataset_audit.csv"
    weekly_validation_path = WEEKLY_AUDITED_ROOT / "11C_weekly_validation.csv"
    output_hash_manifest_path = WEEKLY_AUDITED_ROOT / "11C_output_hash_manifest.csv"
    weekly_dataset_path = WEEKLY_FINAL_ROOT / "11C_weekly_dataset.csv"

    weekly_date_columns = [
        "WeekStartDate",
        "WeekEndDate",
        "ProductFirstObservedDate",
        "ProductFirstWeekStartDate",
        "FirstOperatingDateInWeek",
        "LastOperatingDateInWeek",
        "ProductFirstOperatingDateInWeek",
        "ProductLastOperatingDateInWeek",
    ]
    week_calendar_date_columns = [
        "WeekStartDate",
        "WeekEndDate",
        "FirstOperatingDateInWeek",
        "LastOperatingDateInWeek",
    ]
    product_summary_date_columns = ["FirstWeekStartDate", "LastWeekStartDate"]

    stage_csv(reconciliation_path, reconciliation_audit_df)
    stage_csv(
        weekly_calendar_path,
        format_dates_for_csv(week_calendar, week_calendar_date_columns),
    )
    stage_csv(
        week_summary_path,
        format_dates_for_csv(week_summary_df, week_calendar_date_columns),
    )
    stage_csv(
        product_summary_path,
        format_dates_for_csv(product_week_summary_df, product_summary_date_columns),
    )
    stage_csv(weekly_schema_path, weekly_schema_df)
    stage_csv(weekly_audit_path, weekly_audit_df)
    stage_csv(
        weekly_dataset_path,
        format_dates_for_csv(weekly_df, weekly_date_columns),
    )

    # Reload staged weekly dataset and validate persisted values.
    staged_weekly_df = pd.read_csv(staged_path(weekly_dataset_path), low_memory=False)
    persisted_checks = [
        {
            "Check": "Part 11B lock verified",
            "Expected": True,
            "Actual": True,
            "Passed": True,
        },
        {
            "Check": "Part 11B snapshot files unchanged",
            "Expected": int(len(snapshot_integrity_df)),
            "Actual": int(snapshot_integrity_df["HashMatched"].sum()),
            "Passed": bool(snapshot_integrity_df["HashMatched"].all()),
        },
        {
            "Check": "Both final model dataset copies matched authoritative hash",
            "Expected": 2,
            "Actual": int(source_resolution_df["HashMatched"].sum()),
            "Passed": bool(source_resolution_df["HashMatched"].all()),
        },
        {
            "Check": "Final model source contract checks passed",
            "Expected": int(len(final_source_profile_df)),
            "Actual": int(final_source_profile_df["Passed"].sum()),
            "Passed": bool(final_source_profile_df["Passed"].all()),
        },
        {
            "Check": "Daily component reconciliation checks passed",
            "Expected": int(len(reconciliation_audit_df)),
            "Actual": int(reconciliation_audit_df["Passed"].sum()),
            "Passed": bool(reconciliation_audit_df["Passed"].all()),
        },
        {
            "Check": "Weekly dataset audit checks passed",
            "Expected": int(len(weekly_audit_df)),
            "Actual": int(weekly_audit_df["Passed"].sum()),
            "Passed": bool(weekly_audit_df["Passed"].all()),
        },
        {
            "Check": "Persisted weekly dataset rows",
            "Expected": int(len(weekly_df)),
            "Actual": int(len(staged_weekly_df)),
            "Passed": len(staged_weekly_df) == len(weekly_df),
        },
        {
            "Check": "Persisted weekly TotalDemand",
            "Expected": EXPECTED_TOTAL_DEMAND,
            "Actual": int(staged_weekly_df["WeeklyTotalDemand"].sum()),
            "Passed": int(staged_weekly_df["WeeklyTotalDemand"].sum())
            == EXPECTED_TOTAL_DEMAND,
        },
        {
            "Check": "Persisted weekly NormalDemand",
            "Expected": EXPECTED_NORMAL_DEMAND,
            "Actual": int(staged_weekly_df["WeeklyNormalDemand"].sum()),
            "Passed": int(staged_weekly_df["WeeklyNormalDemand"].sum())
            == EXPECTED_NORMAL_DEMAND,
        },
        {
            "Check": "Persisted weekly BulkDemand",
            "Expected": EXPECTED_BULK_DEMAND,
            "Actual": int(staged_weekly_df["WeeklyBulkDemand"].sum()),
            "Passed": int(staged_weekly_df["WeeklyBulkDemand"].sum())
            == EXPECTED_BULK_DEMAND,
        },
        {
            "Check": "Protected target-vault files copied",
            "Expected": 0,
            "Actual": int(
                sum(protected_target_name(Path(item["Destination"])) for item in source_copy_plan)
            ),
            "Passed": not any(
                protected_target_name(Path(item["Destination"]))
                for item in source_copy_plan
            ),
        },
        {
            "Check": "Model binaries copied",
            "Expected": 0,
            "Actual": int(
                sum(Path(item["Destination"]).suffix.lower() == ".joblib" for item in source_copy_plan)
            ),
            "Passed": not any(
                Path(item["Destination"]).suffix.lower() == ".joblib"
                for item in source_copy_plan
            ),
        },
        {
            "Check": "Weekly features or models created",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
    ]
    weekly_validation_df = pd.DataFrame(persisted_checks)
    if not weekly_validation_df["Passed"].all():
        raise AssertionError(
            "Part 11C validation failed:\n"
            + weekly_validation_df.loc[
                ~weekly_validation_df["Passed"]
            ].to_string(index=False)
        )
    stage_csv(weekly_validation_path, weekly_validation_df)

    # Hash manifest for all non-lock weekly outputs and amendment control files.
    output_paths_for_manifest = [
        source_resolution_path,
        source_copy_audit_path,
        source_amendment_manifest_path,
        reconciliation_path,
        weekly_calendar_path,
        week_summary_path,
        product_summary_path,
        weekly_schema_path,
        weekly_audit_path,
        weekly_validation_path,
        weekly_dataset_path,
    ]
    output_hash_records = []
    for final_path in output_paths_for_manifest:
        stage_file = staged_path(final_path)
        output_hash_records.append(
            {
                "RelativePath": str(final_path.relative_to(EXT_ROOT)),
                "Bytes": int(stage_file.stat().st_size),
                "SHA256": sha256_file(stage_file),
                "CreatedByStep": STEP_ID,
            }
        )
    output_hash_manifest_df = pd.DataFrame(output_hash_records)
    stage_csv(output_hash_manifest_path, output_hash_manifest_df)

    # ------------------------- memory content -------------------------
    context_body = "\n".join(
        [
            f"**Status:** `{STATUS}`",
            "",
            "Part 11C amended the source snapshot with the exact final forecast-preparation model panel used by the completed daily modelling workflow and created an audited Monday-to-Sunday product-week demand dataset.",
            "",
            f"- Final model lineage source: `{FINAL_MODEL_PREPARATION_COPY}`",
            f"- Verified source SHA-256: `{EXPECTED_FINAL_MODEL_SHA256}`",
            f"- Daily lineage: `{len(final_model_df):,}` rows, `{final_model_df['CanonicalProductID'].nunique()}` products and `{final_model_df['Date'].nunique()}` operating dates.",
            f"- Weekly dataset: `{len(weekly_df):,}` product-week rows across `{weekly_df['WeekStartDate'].nunique()}` Monday-to-Sunday weeks.",
            f"- Weekly demand totals: normal `{int(weekly_df['WeeklyNormalDemand'].sum()):,}`, bulk `{int(weekly_df['WeeklyBulkDemand'].sum()):,}`, total `{int(weekly_df['WeeklyTotalDemand'].sum()):,}` units.",
            f"- Opened daily holdout: `{EXPECTED_HOLDOUT_START}` to `{EXPECTED_HOLDOUT_END}`; overlapping product-weeks are retained for traceability and marked `WeeklyTuningEligible = False`.",
            "- No daily lag, rolling or expanding predictor was reused as a weekly feature.",
        ]
    )
    context_text = upsert_section_text(
        original_memory[MEMORY_ROOT / "PROJECT_CONTEXT.md"],
        "STEP_11C_WEEKLY_DATASET",
        "Part 11C — Weekly product-demand dataset",
        context_body,
    )

    workflow_body = "\n".join(
        [
            f"Status: `{STATUS}`.",
            "Part 11C created and locked the weekly product-demand dataset only.",
            "Part 11D may analyse weekly demand concentration, intermittency, product coverage and candidate high/moderate segmentation rules using training history only.",
            "The opened March 2026 daily holdout remains prohibited from scope, segmentation or model optimisation.",
        ]
    )
    workflow_text = upsert_section_text(
        original_memory[MEMORY_ROOT / "WORKFLOW.md"],
        "STEP_11C_WEEKLY_DATASET",
        "Part 11C completion note",
        workflow_body,
    )

    decisions_body = "\n".join(
        [
            "### D11C-001 — Final daily-model lineage adopted",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** The completed daily models were prepared from the 43,774-row final forecast-preparation panel, not directly from the earlier 25,405-row Step 5 table.",
            f"- **Decision:** Use `{FINAL_MODEL_PREPARATION_COPY.name}` as the product-date lineage for weekly aggregation after verifying both saved copies against SHA-256 `{EXPECTED_FINAL_MODEL_SHA256}`.",
            "- **Files affected:** Part 11C source amendment and weekly dataset.",
            "- **Earlier results remain valid:** Yes. The Part 11B canonical source remains the upstream demand-component reference.",
            "- **New next step:** Analyse the weekly dataset in Part 11D.",
            "",
            "### D11C-002 — Demand components reconciled by product and date",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** The final 58-column model panel retains `TotalDemand` but not the separate normal and bulk target components.",
            "- **Decision:** Join `NormalDemand` and `BulkDemand` from the locked canonical Part 11B snapshot. Fill absent component rows with zero only when the final lineage row has zero `TotalDemand`; fail on any positive-demand mismatch.",
            "- **Files affected:** `11C_daily_lineage_reconciliation_audit.csv` and `11C_weekly_dataset.csv`.",
            "- **Earlier results remain valid:** Yes; all source totals reconcile exactly.",
            "- **New next step:** Evaluate weekly total and normal-demand targets separately later without redefining the source.",
            "",
            "### D11C-003 — Monday-to-Sunday week boundary",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** A fixed calendar boundary is required before any weekly concentration or modelling analysis.",
            "- **Decision:** Define each week from Monday through Sunday. Retain first and last partial dataset weeks and flag them rather than silently removing them.",
            "- **Files affected:** weekly calendar, weekly summaries and final weekly dataset.",
            "- **Earlier results remain valid:** Yes.",
            "- **New next step:** Part 11D must decide, using training history only, whether boundary or low-operating-day weeks require special handling.",
            "",
            "### D11C-004 — Opened holdout retained but blocked from tuning",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            f"- **Reason:** Daily targets from `{EXPECTED_HOLDOUT_START}` to `{EXPECTED_HOLDOUT_END}` have already been opened.",
            "- **Decision:** Retain overlapping product-weeks for traceability and later descriptive comparison, but set `WeeklyTuningEligible = False` and prohibit their use in product scope, segmentation, features, model or fallback selection.",
            "- **Files affected:** weekly dataset and calendar governance fields.",
            "- **Earlier results remain valid:** Yes.",
            "- **New next step:** All Part 11D statistics used for decisions must filter to tuning-eligible weeks.",
            "",
            "### D11C-005 — Daily engineered demand features not reused",
            f"- **Decision date:** {NOW_LOCAL.date().isoformat()}",
            "- **Reason:** Daily lags and rolling windows do not automatically represent leakage-safe weekly predictors.",
            "- **Decision:** Part 11C carries only identifiers, weekly calendar/audit fields and weekly demand targets. Weekly predictors will be designed separately in Part 11F.",
            "- **Files affected:** final weekly dataset schema.",
            "- **Earlier results remain valid:** Yes.",
            "- **New next step:** Do not add weekly features during Part 11D.",
        ]
    )
    decisions_text = upsert_section_text(
        original_memory[MEMORY_ROOT / "DECISIONS.md"],
        "STEP_11C_WEEKLY_DATASET",
        "Part 11C decisions",
        decisions_body,
    )

    files_body = "\n".join(
        [
            f"**Status:** `{STATUS}`",
            "",
            "### Source amendment",
            f"- `{SNAPSHOT_DATA_ROOT / '11C_forecasting_preparation_source' / FINAL_MODEL_PREPARATION_COPY.name}`",
            f"- `{SNAPSHOT_CONTRACT_ROOT / '11C_forecasting_preparation'}`",
            f"- `{PART11C_SOURCE_AMENDMENT_LOCK}`",
            "",
            "### Weekly dataset and audits",
            f"- `{weekly_dataset_path}`",
            f"- `{reconciliation_path}`",
            f"- `{weekly_calendar_path}`",
            f"- `{week_summary_path}`",
            f"- `{product_summary_path}`",
            f"- `{weekly_schema_path}`",
            f"- `{weekly_audit_path}`",
            f"- `{weekly_validation_path}`",
            f"- `{output_hash_manifest_path}`",
            "",
            "### Checkpoint and lock",
            f"- `{PART11C_CHECKPOINT}`",
            f"- `{PART11C_LOCK}`",
        ]
    )
    files_text = upsert_section_text(
        original_memory[MEMORY_ROOT / "FILES_AND_PATHS.md"],
        "STEP_11C_WEEKLY_DATASET",
        "Part 11C files and paths",
        files_body,
    )

    metrics_body = "\n".join(
        [
            f"**Status:** `{STATUS}`",
            "",
            "### Final daily-model source validation",
            markdown_table(final_source_profile_df),
            "",
            "### Daily component reconciliation",
            markdown_table(reconciliation_audit_df),
            "",
            "### Weekly dataset audit",
            markdown_table(weekly_audit_df),
            "",
            "These are dataset-construction and integrity results. They are not weekly forecasting performance metrics.",
        ]
    )
    metrics_text = upsert_section_text(
        original_memory[MEMORY_ROOT / "METRICS_AND_RESULTS.md"],
        "STEP_11C_WEEKLY_DATASET",
        "Part 11C dataset results",
        metrics_body,
    )

    chat_row = (
        f"| {NOW_LOCAL.isoformat()} | 11C | Use the final forecast-preparation datasets used by the daily model and proceed with Part 11C. | "
        "Verified and snapshotted the final daily-model lineage, reconciled normal/bulk components, and created the audited weekly product-demand dataset. | "
        f"`steps/STEP_11C_WEEKLY_DATASET.md` | `{STATUS}` |"
    )
    chat_text = append_chat_row_text(
        original_memory[MEMORY_ROOT / "CHAT_INDEX.md"], chat_row
    )

    handoff_text = "\n".join(
        [
            "# CURRENT HANDOFF",
            "",
            f"- **Current status:** `{STATUS}`",
            "- **Completed step:** Part 11C — weekly product-demand dataset creation and audit",
            "- **Next step:** Part 11D — weekly concentration, intermittency, coverage and leakage-safe segmentation analysis",
            f"- **Weekly dataset:** `{weekly_dataset_path}`",
            f"- **Weekly rows:** `{len(weekly_df):,}`",
            f"- **Products:** `{weekly_df['CanonicalProductID'].nunique()}`",
            f"- **Weeks:** `{weekly_df['WeekStartDate'].nunique()}`",
            f"- **Demand totals:** normal `{int(weekly_df['WeeklyNormalDemand'].sum()):,}`, bulk `{int(weekly_df['WeeklyBulkDemand'].sum()):,}`, total `{int(weekly_df['WeeklyTotalDemand'].sum()):,}`",
            f"- **Opened holdout:** `{EXPECTED_HOLDOUT_START}` to `{EXPECTED_HOLDOUT_END}`; overlapping rows have `WeeklyTuningEligible = False`.",
            "- **Week definition:** Monday through Sunday.",
            "- **Boundary policy:** retain and flag partial first/last weeks; do not remove them in Part 11C.",
            "- **Feature policy:** daily lag/rolling/expanding columns were not carried into the weekly dataset.",
            "- **Protected target vault:** not copied or opened by Part 11C.",
            "- **Model activity:** no model loaded, copied, fitted, retrained or scored.",
            "",
            "## Part 11D constraints",
            "",
            "1. Calculate product coverage and segmentation candidates using tuning-eligible history only.",
            "2. Do not use any product-week overlapping the opened March 2026 holdout to choose thresholds or rules.",
            "3. Do not create model features or train models during Part 11D.",
            "4. Keep the workflow flexible and log any change with reason, date, affected files and validity impact.",
            "5. Preserve the Part 11C weekly dataset and locks unchanged.",
        ]
    ).rstrip() + "\n"

    step_log_path = STEP_MEMORY_ROOT / "STEP_11C_WEEKLY_DATASET.md"
    step_log_text = "\n".join(
        [
            "# STEP 11C — WEEKLY PRODUCT-DEMAND DATASET",
            "",
            f"- **Step ID:** {STEP_ID}",
            f"- **Date and time:** {NOW_LOCAL.isoformat()}",
            f"- **Status:** `{STATUS}`",
            "",
            "## User request",
            "",
            "Use the final forecasting-preparation datasets that were used by the completed daily model, amend the source snapshot safely, and proceed with Part 11C.",
            "",
            "## Assistant response summary",
            "",
            "Verified the two final 43,774-row model-panel copies, copied the approved model panel and lightweight contracts into a separately locked source amendment, reconciled normal and bulk demand from the Part 11B canonical snapshot, aggregated product demand to Monday-to-Sunday weeks, audited the result, updated project memory and created a Part 11C checkpoint and lock.",
            "",
            "## Full technical actions",
            "",
            "1. Verified the Part 11B lock, checkpoint and every file listed in its snapshot manifest.",
            "2. Verified both saved final forecast-preparation model-panel copies against the authoritative SHA-256 hash.",
            "3. Validated the exact 58-column frozen feature contract and the 43,774-row source profile.",
            "4. Confirmed the final leakage audit contains zero flagged leakage columns.",
            "5. Reconciled `NormalDemand` and `BulkDemand` from the locked canonical snapshot to the final model lineage by `Date` and `CanonicalProductID`.",
            "6. Failed safely on any positive-demand source mismatch; none occurred.",
            "7. Defined weeks as Monday through Sunday and retained boundary weeks with explicit flags.",
            "8. Aggregated weekly normal, bulk and total demand while preserving every product-week context present in the final model lineage.",
            "9. Marked weekly rows overlapping the opened daily holdout as ineligible for weekly tuning.",
            "10. Excluded all daily lag, rolling and expanding predictors from the weekly dataset.",
            "11. Wrote source-amendment, weekly-data, audit, memory, checkpoint and lock outputs transactionally.",
            "",
            "## Decisions made",
            "",
            "See the controlled Part 11C section in `../DECISIONS.md`.",
            "",
            "## Assumptions",
            "",
            "- The final 43,774-row forecast-preparation panel is the authoritative product-date lineage used by the daily modelling system.",
            "- Monday-to-Sunday calendar weeks are an explicit initial weekly definition and may be revisited later only through a documented workflow change.",
            "- Product-date rows added by final continuation logic but absent from the earlier canonical component table represent zero normal and bulk demand; the code verifies that all such rows have zero total demand before filling components.",
            "- Opened March 2026 rows may be retained for traceability but cannot influence any weekly design or selection decision.",
            "",
            "## Input files",
            "",
            f"- `{FINAL_MODEL_PREPARATION_COPY}`",
            f"- `{FINAL_MODEL_EDEN_COPY}`",
            f"- `{CANONICAL_SNAPSHOT_SOURCE}`",
            f"- `{hash_manifest_path}`",
            f"- `{FORECAST_PREPARATION_ROOT / '17_frozen_model_feature_contract.csv'}`",
            f"- `{FORECAST_PREPARATION_ROOT / '17_forecasting_protocol_contract.csv'}`",
            f"- `{FORECAST_PREPARATION_ROOT / '17_final_leakage_audit.csv'}`",
            f"- `{PART11B_LOCK}`",
            f"- `{PART11B_SNAPSHOT_MANIFEST}`",
            "",
            "## Output files",
            "",
            f"- `{weekly_dataset_path}`",
            f"- `{weekly_calendar_path}`",
            f"- `{week_summary_path}`",
            f"- `{product_summary_path}`",
            f"- `{reconciliation_path}`",
            f"- `{weekly_schema_path}`",
            f"- `{weekly_audit_path}`",
            f"- `{weekly_validation_path}`",
            f"- `{output_hash_manifest_path}`",
            f"- `{PART11C_SOURCE_AMENDMENT_LOCK}` and SHA-256 sidecar",
            f"- `{PART11C_CHECKPOINT}` and SHA-256 sidecar",
            f"- `{PART11C_LOCK}` and SHA-256 sidecar",
            "",
            "## Hashes where relevant",
            "",
            f"- Final model dataset SHA-256: `{EXPECTED_FINAL_MODEL_SHA256}`",
            f"- Part 11B lock SHA-256: `{part11b_lock_hash}`",
            f"- Part 11B checkpoint SHA-256: `{part11b_checkpoint_hash}`",
            "",
            "## Validation checks",
            "",
            markdown_table(weekly_validation_df),
            "",
            "## Errors encountered and fixes",
            "",
            "No Part 11C runtime error occurred before the transaction was committed. The source-lineage correction from the preceding discussion was incorporated by using the final forecast-preparation model panel rather than treating the earlier Step 5 table as the sole aggregation source.",
            "",
            "## Results",
            "",
            f"- Weekly product rows: {len(weekly_df):,}",
            f"- Products: {weekly_df['CanonicalProductID'].nunique()}",
            f"- Calendar weeks: {weekly_df['WeekStartDate'].nunique()}",
            f"- Weekly normal demand total: {int(weekly_df['WeeklyNormalDemand'].sum()):,}",
            f"- Weekly bulk demand total: {int(weekly_df['WeeklyBulkDemand'].sum()):,}",
            f"- Weekly total demand: {int(weekly_df['WeeklyTotalDemand'].sum()):,}",
            f"- Zero-demand product-weeks: {int((weekly_df['WeeklyTotalDemand'] == 0).sum()):,}",
            f"- Positive-demand product-weeks: {int((weekly_df['WeeklyTotalDemand'] > 0).sum()):,}",
            f"- Product-weeks overlapping opened holdout: {int(weekly_df['ContainsOpenedDailyHoldoutRows'].sum()):,}",
            "- Duplicate product-week keys: 0",
            "- Weekly demand-component mismatches: 0",
            "",
            "## Limitations",
            "",
            "Part 11C does not choose a product coverage threshold, create high/moderate segments, engineer weekly predictors, select baselines, train models or evaluate weekly forecasts. The first and last dataset weeks are partial calendar weeks and are retained with flags for later analysis. A new untouched future period is still required for unbiased final weekly evaluation.",
            "",
            "## Next action",
            "",
            "Part 11D analyses weekly demand concentration, intermittency, product coverage and leakage-safe segmentation candidates using only tuning-eligible history.",
        ]
    ).rstrip() + "\n"

    staged_memory_contents = {
        MEMORY_ROOT / "PROJECT_CONTEXT.md": context_text,
        MEMORY_ROOT / "WORKFLOW.md": workflow_text,
        MEMORY_ROOT / "DECISIONS.md": decisions_text,
        MEMORY_ROOT / "FILES_AND_PATHS.md": files_text,
        MEMORY_ROOT / "METRICS_AND_RESULTS.md": metrics_text,
        MEMORY_ROOT / "CHAT_INDEX.md": chat_text,
        MEMORY_ROOT / "CURRENT_HANDOFF.md": handoff_text,
        step_log_path: step_log_text,
    }
    for final_path, text in staged_memory_contents.items():
        stage_text(final_path, text)

    # ------------------------- source amendment lock -------------------------
    source_amendment_lock_payload = {
        "StepID": STEP_ID,
        "LockType": "SOURCE_SNAPSHOT_AMENDMENT",
        "Status": "PART_11C_SOURCE_AMENDMENT_CREATED_AND_VERIFIED",
        "CreatedUTC": NOW_UTC.isoformat(),
        "ExtensionRoot": str(EXT_ROOT),
        "Part11BLock": {
            "Path": str(PART11B_LOCK),
            "SHA256": part11b_lock_hash,
            "Verified": True,
            "Modified": False,
        },
        "FinalModelSource": {
            "OriginalPath": str(FINAL_MODEL_PREPARATION_COPY),
            "SecondVerifiedCopy": str(FINAL_MODEL_EDEN_COPY),
            "SnapshotPath": str(
                SNAPSHOT_DATA_ROOT
                / "11C_forecasting_preparation_source"
                / FINAL_MODEL_PREPARATION_COPY.name
            ),
            "SHA256": EXPECTED_FINAL_MODEL_SHA256,
            "Rows": int(len(final_model_df)),
            "Columns": int(final_model_df.shape[1]),
            "Products": int(final_model_df["CanonicalProductID"].nunique()),
            "OperatingDates": int(final_model_df["Date"].nunique()),
        },
        "SourceCopyAudit": {
            "Path": str(source_copy_audit_path),
            "SHA256": sha256_file(staged_path(source_copy_audit_path)),
            "FilesCopied": int(len(source_copy_audit_df)),
            "AllHashesMatched": bool(source_copy_audit_df["HashMatched"].all()),
        },
        "AmendmentManifest": {
            "Path": str(source_amendment_manifest_path),
            "SHA256": sha256_file(staged_path(source_amendment_manifest_path)),
            "FilesListed": int(len(source_amendment_manifest_df)),
        },
        "SafetyAssertions": {
            "ProtectedTargetVaultOpened": False,
            "ProtectedTargetVaultCopied": False,
            "ReservedTargetVaultCopied": False,
            "ModelBinariesLoaded": False,
            "ModelBinariesCopied": False,
            "ExistingPart11BLockChanged": False,
            "ExistingPart11BSnapshotFilesChanged": False,
        },
    }
    stage_json(PART11C_SOURCE_AMENDMENT_LOCK, source_amendment_lock_payload)
    source_amendment_lock_hash = sha256_file(
        staged_path(PART11C_SOURCE_AMENDMENT_LOCK)
    )
    stage_text(
        PART11C_SOURCE_AMENDMENT_LOCK_SHA,
        f"{source_amendment_lock_hash}  {PART11C_SOURCE_AMENDMENT_LOCK.name}\n",
    )

    # ------------------------- checkpoint -------------------------
    checkpoint_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "ExtensionRoot": str(EXT_ROOT),
        "WeekDefinition": "MONDAY_TO_SUNDAY",
        "SourceLineage": {
            "FinalModelDataset": str(FINAL_MODEL_PREPARATION_COPY),
            "SHA256": EXPECTED_FINAL_MODEL_SHA256,
            "CanonicalComponentsSnapshot": str(CANONICAL_SNAPSHOT_SOURCE),
            "Part11BLockSHA256": part11b_lock_hash,
            "SourceAmendmentLockSHA256": source_amendment_lock_hash,
        },
        "WeeklyDataset": {
            "Path": str(weekly_dataset_path),
            "SHA256": sha256_file(staged_path(weekly_dataset_path)),
            "Rows": int(len(weekly_df)),
            "Columns": int(weekly_df.shape[1]),
            "Products": int(weekly_df["CanonicalProductID"].nunique()),
            "Weeks": int(weekly_df["WeekStartDate"].nunique()),
            "NormalDemand": int(weekly_df["WeeklyNormalDemand"].sum()),
            "BulkDemand": int(weekly_df["WeeklyBulkDemand"].sum()),
            "TotalDemand": int(weekly_df["WeeklyTotalDemand"].sum()),
        },
        "OpenedDailyHoldout": {
            "StartDate": EXPECTED_HOLDOUT_START,
            "EndDate": EXPECTED_HOLDOUT_END,
            "RetainedForTraceability": True,
            "MayTuneWeeklySystem": False,
            "ProductWeeksMarkedIneligible": int(
                weekly_df["ContainsOpenedDailyHoldoutRows"].sum()
            ),
        },
        "Validation": {
            "Path": str(weekly_validation_path),
            "SHA256": sha256_file(staged_path(weekly_validation_path)),
            "Checks": int(len(weekly_validation_df)),
            "Passed": int(weekly_validation_df["Passed"].sum()),
            "Failed": int((~weekly_validation_df["Passed"]).sum()),
        },
        "Safety": {
            "ProtectedTargetVaultOpened": False,
            "ProtectedTargetVaultCopied": False,
            "OpenedDailyHoldoutUsedForWeeklyTuning": False,
            "DailyLagOrRollingFeaturesReused": False,
            "ModelBinariesLoaded": False,
            "ModelBinariesCopied": False,
            "ModelsFitted": False,
            "PredictionsCreated": False,
            "WeeklyFeaturesCreated": False,
        },
        "ReadyForPart11D": True,
        "NextStep": "11D",
    }
    stage_json(PART11C_CHECKPOINT, checkpoint_payload)
    checkpoint_hash = sha256_file(staged_path(PART11C_CHECKPOINT))
    stage_text(
        PART11C_CHECKPOINT_SHA,
        f"{checkpoint_hash}  {PART11C_CHECKPOINT.name}\n",
    )

    memory_hashes = [
        {
            "RelativePath": str(path.relative_to(EXT_ROOT)),
            "SHA256": sha256_file(staged_path(path)),
        }
        for path in staged_memory_contents
    ]

    lock_payload = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "ExtensionRoot": str(EXT_ROOT),
        "Part11B": {
            "LockPath": str(PART11B_LOCK),
            "LockSHA256": part11b_lock_hash,
            "CheckpointSHA256": part11b_checkpoint_hash,
            "SnapshotFilesVerified": int(len(snapshot_integrity_df)),
            "SnapshotFilesChanged": 0,
        },
        "SourceAmendment": {
            "LockPath": str(PART11C_SOURCE_AMENDMENT_LOCK),
            "LockSHA256": source_amendment_lock_hash,
            "FilesCopied": int(len(source_copy_audit_df)),
        },
        "WeeklyDataset": checkpoint_payload["WeeklyDataset"],
        "Reconciliation": {
            "Path": str(reconciliation_path),
            "SHA256": sha256_file(staged_path(reconciliation_path)),
            "Checks": int(len(reconciliation_audit_df)),
            "Passed": int(reconciliation_audit_df["Passed"].sum()),
        },
        "OutputHashManifest": {
            "Path": str(output_hash_manifest_path),
            "SHA256": sha256_file(staged_path(output_hash_manifest_path)),
            "FilesListed": int(len(output_hash_manifest_df)),
        },
        "Validation": checkpoint_payload["Validation"],
        "MemoryFileHashes": memory_hashes,
        "Checkpoint": {
            "Path": str(PART11C_CHECKPOINT),
            "SHA256": checkpoint_hash,
        },
        "SafetyAssertions": checkpoint_payload["Safety"],
        "ReadyForPart11D": True,
        "NextStep": "11D",
    }
    stage_json(PART11C_LOCK, lock_payload)
    part11c_lock_hash = sha256_file(staged_path(PART11C_LOCK))
    stage_text(
        PART11C_LOCK_SHA,
        f"{part11c_lock_hash}  {PART11C_LOCK.name}\n",
    )

    # ------------------------- log -------------------------
    log_path = LOG_ROOT / "11C_weekly_dataset_log.txt"
    log_text = "\n".join(
        [
            f"Status: {STATUS}",
            f"Run UTC: {NOW_UTC.isoformat()}",
            f"Run local: {NOW_LOCAL.isoformat()}",
            f"Final model source: {FINAL_MODEL_PREPARATION_COPY}",
            f"Final model source SHA256: {EXPECTED_FINAL_MODEL_SHA256}",
            f"Weekly dataset: {weekly_dataset_path}",
            f"Weekly rows: {len(weekly_df)}",
            f"Products: {weekly_df['CanonicalProductID'].nunique()}",
            f"Weeks: {weekly_df['WeekStartDate'].nunique()}",
            f"Normal demand: {int(weekly_df['WeeklyNormalDemand'].sum())}",
            f"Bulk demand: {int(weekly_df['WeeklyBulkDemand'].sum())}",
            f"Total demand: {int(weekly_df['WeeklyTotalDemand'].sum())}",
            f"Checkpoint SHA256: {checkpoint_hash}",
            f"Part 11C lock SHA256: {part11c_lock_hash}",
        ]
    ) + "\n"
    stage_text(log_path, log_text)

    # ------------------------- pre-commit stage validation -------------------------
    all_staged_files = [path for path in stage_root.rglob("*") if path.is_file()]
    if not all_staged_files:
        raise AssertionError("Part 11C staging directory is unexpectedly empty")
    if any(protected_target_name(path) for path in all_staged_files):
        raise PermissionError("Protected target file detected in Part 11C staging")

    # ------------------------- commit transaction -------------------------
    for stage_file in sorted(all_staged_files):
        final_path = EXT_ROOT / stage_file.relative_to(stage_root)
        if final_path in staged_memory_contents:
            continue
        if final_path.exists():
            raise FileExistsError(f"Part 11C commit overwrite guard: {final_path}")
        final_path.parent.mkdir(parents=True, exist_ok=True)
        os.replace(stage_file, final_path)
        created_final_paths.append(final_path)

    # Memory files are the only intentionally updated existing files.
    for final_path, text in staged_memory_contents.items():
        stage_file = staged_path(final_path)
        if not stage_file.is_file():
            raise FileNotFoundError(f"Missing staged memory file: {stage_file}")
        final_path.parent.mkdir(parents=True, exist_ok=True)
        os.replace(stage_file, final_path)
        if final_path == step_log_path:
            created_final_paths.append(final_path)

    # Lock all immutable Part 11C outputs; keep core memory writable.
    immutable_paths = [
        path
        for path in created_final_paths
        if path not in memory_files and path != step_log_path and path != log_path
    ]
    immutable_paths.append(step_log_path)
    for path in immutable_paths:
        make_read_only(path)

    # Verify committed source-amendment and output hashes.
    for record in source_copy_audit_df.itertuples(index=False):
        final_path = EXT_ROOT / Path(record.DestinationRelativePath)
        if sha256_file(final_path) != record.SourceSHA256:
            raise AssertionError(f"Committed source-copy hash mismatch: {final_path}")

    committed_weekly = pd.read_csv(weekly_dataset_path, low_memory=False)
    if len(committed_weekly) != len(weekly_df):
        raise AssertionError("Committed weekly dataset row count changed")
    if int(committed_weekly["WeeklyTotalDemand"].sum()) != EXPECTED_TOTAL_DEMAND:
        raise AssertionError("Committed weekly dataset total demand changed")

    if read_sidecar_hash(PART11C_SOURCE_AMENDMENT_LOCK_SHA) != sha256_file(
        PART11C_SOURCE_AMENDMENT_LOCK
    ):
        raise AssertionError("Source-amendment lock sidecar verification failed")
    if read_sidecar_hash(PART11C_CHECKPOINT_SHA) != sha256_file(PART11C_CHECKPOINT):
        raise AssertionError("Part 11C checkpoint sidecar verification failed")
    if read_sidecar_hash(PART11C_LOCK_SHA) != sha256_file(PART11C_LOCK):
        raise AssertionError("Part 11C lock sidecar verification failed")

    # Confirm every original Part 11B snapshot file is still unchanged.
    for row in snapshot_integrity_df.itertuples(index=False):
        file_path = EXT_ROOT / Path(row.RelativePath)
        if sha256_file(file_path) != row.ExpectedSHA256:
            raise AssertionError(f"Part 11B snapshot file changed during 11C: {file_path}")

    if stage_root.exists():
        shutil.rmtree(stage_root)

except Exception:
    # Restore core memory if it was updated before a later failure.
    for memory_path, original_text in original_memory.items():
        try:
            if memory_path.exists():
                make_writable(memory_path)
            atomic_write_text(memory_path, original_text)
        except Exception:
            pass

    # Remove only files created by this Part 11C transaction.
    for created_path in reversed(created_final_paths):
        try:
            if created_path.exists() and created_path.is_file():
                make_writable(created_path)
                created_path.unlink()
        except Exception:
            pass

    if stage_root.exists():
        shutil.rmtree(stage_root, ignore_errors=True)
    raise


# =============================================================================
# 8. FINAL TECHNICAL OUTPUT
# =============================================================================

print("=" * 100)
print("EDEN WEEKLY FORECASTING EXTENSION — PART 11C COMPLETE")
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Extension root: {EXT_ROOT}")
print(f"Local time: {NOW_LOCAL.isoformat()}")

print("\nFINAL FORECAST-PREPARATION SOURCE VERIFICATION")
print(source_resolution_df.to_string(index=False))
print("\nFINAL MODEL SOURCE PROFILE")
print(final_source_profile_df.to_string(index=False))
print("\nDAILY LINEAGE AND DEMAND-COMPONENT RECONCILIATION")
print(reconciliation_audit_df.to_string(index=False))
print("\nWEEKLY DATASET AUDIT")
print(weekly_audit_df.to_string(index=False))
print("\nPART 11C VALIDATION")
print(weekly_validation_df.to_string(index=False))

print("\nWEEKLY DATASET SUMMARY")
summary_output = pd.DataFrame(
    [
        {"Metric": "Rows", "Value": int(len(weekly_df))},
        {
            "Metric": "Products",
            "Value": int(weekly_df["CanonicalProductID"].nunique()),
        },
        {
            "Metric": "Weeks",
            "Value": int(weekly_df["WeekStartDate"].nunique()),
        },
        {
            "Metric": "WeekStartMin",
            "Value": weekly_df["WeekStartDate"].min().strftime("%Y-%m-%d"),
        },
        {
            "Metric": "WeekStartMax",
            "Value": weekly_df["WeekStartDate"].max().strftime("%Y-%m-%d"),
        },
        {
            "Metric": "ZeroDemandProductWeeks",
            "Value": int((weekly_df["WeeklyTotalDemand"] == 0).sum()),
        },
        {
            "Metric": "PositiveDemandProductWeeks",
            "Value": int((weekly_df["WeeklyTotalDemand"] > 0).sum()),
        },
        {
            "Metric": "OpenedHoldoutProductWeeks",
            "Value": int(weekly_df["ContainsOpenedDailyHoldoutRows"].sum()),
        },
        {
            "Metric": "WeeklyNormalDemand",
            "Value": int(weekly_df["WeeklyNormalDemand"].sum()),
        },
        {
            "Metric": "WeeklyBulkDemand",
            "Value": int(weekly_df["WeeklyBulkDemand"].sum()),
        },
        {
            "Metric": "WeeklyTotalDemand",
            "Value": int(weekly_df["WeeklyTotalDemand"].sum()),
        },
    ]
)
print(summary_output.to_string(index=False))

print("\nCONTROL OUTPUTS")
print(f"- Weekly dataset: {WEEKLY_FINAL_ROOT / '11C_weekly_dataset.csv'}")
print(f"- Weekly calendar: {WEEKLY_AUDITED_ROOT / '11C_weekly_calendar.csv'}")
print(f"- Reconciliation audit: {WEEKLY_AUDITED_ROOT / '11C_daily_lineage_reconciliation_audit.csv'}")
print(f"- Weekly audit: {WEEKLY_AUDITED_ROOT / '11C_weekly_dataset_audit.csv'}")
print(f"- Validation: {WEEKLY_AUDITED_ROOT / '11C_weekly_validation.csv'}")
print(f"- Source-amendment lock: {PART11C_SOURCE_AMENDMENT_LOCK}")
print(f"- Source-amendment lock SHA256: {sha256_file(PART11C_SOURCE_AMENDMENT_LOCK)}")
print(f"- Checkpoint: {PART11C_CHECKPOINT}")
print(f"- Checkpoint SHA256: {sha256_file(PART11C_CHECKPOINT)}")
print(f"- Part 11C lock: {PART11C_LOCK}")
print(f"- Part 11C lock SHA256: {sha256_file(PART11C_LOCK)}")

print(
    "\nSAFETY: protected target vault opened/copied False/False; "
    "opened daily holdout used for weekly tuning False; "
    "model binaries loaded/copied/retrained False/False/False; "
    "weekly features/models/predictions created False/False/False."
)
print("=" * 100)

EDEN WEEKLY FORECASTING EXTENSION — PART 11C COMPLETE
Status: PART_11C_COMPLETED_READY_FOR_11D
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-05T20:13:19.883366+01:00

FINAL FORECAST-PREPARATION SOURCE VERIFICATION
                        Role                                                                                                                      SourcePath    Bytes                                                   ExpectedSHA256                                                     ActualSHA256  HashMatched
FINAL_MODEL_PREPARATION_COPY /Users/ryansmac/Desktop/Meng Project/eden_datasets/forceast_preparation/UL_EDEN_forecasting_preparation_final_model_dataset.csv 13632942 ec86928f6a96f0dd9131a56bbb0893cd62a726cf3fde43cb1ffd6a787ca837f2 ec86928f6a96f0dd9131a56bbb0893cd62a726cf3fde43cb1ffd6a787ca837f2         True
  FINAL_MODEL_EDEN_ROOT_COPY                      /Users/ryansmac/Desktop/Meng Project/eden_datase

In [14]:
# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11D
# Weekly demand concentration, intermittency and 80/90/95% scope analysis.
# =============================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Paths and constants
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"
MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
WEEKLY_DATASET = EXT_ROOT / "02_weekly_data" / "final" / "11C_weekly_dataset.csv"
OUTPUT_ROOT = EXT_ROOT / "03_product_scope_and_segmentation"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
LOG_ROOT = EXT_ROOT / "12_logs"

PART11C_LOCK = CHECKPOINT_ROOT / "11C_weekly_dataset_lock.json"
PART11C_LOCK_SHA = CHECKPOINT_ROOT / "11C_weekly_dataset_lock.sha256"
PART11C_CHECKPOINT = CHECKPOINT_ROOT / "11C_checkpoint.json"
PART11C_CHECKPOINT_SHA = CHECKPOINT_ROOT / "11C_checkpoint.sha256"

PROFILE_PATH = OUTPUT_ROOT / "11D_product_weekly_demand_profile.csv"
SCOPE_PATH = OUTPUT_ROOT / "11D_normal_demand_scope_summary.csv"
MEMBERSHIP_PATH = OUTPUT_ROOT / "11D_normal_demand_scope_membership.csv"
SEGMENT_PATH = OUTPUT_ROOT / "11D_candidate_segment_diagnostics.csv"
SENSITIVITY_PATH = OUTPUT_ROOT / "11D_boundary_week_sensitivity.csv"
UNIVERSE_PATH = OUTPUT_ROOT / "11D_analysis_universe_summary.csv"
VALIDATION_PATH = OUTPUT_ROOT / "11D_validation.csv"
MANIFEST_PATH = OUTPUT_ROOT / "11D_output_hash_manifest.csv"
STEP_MEMORY_PATH = STEP_MEMORY_ROOT / "STEP_11D_PRODUCT_SCOPE_ANALYSIS.md"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "11D_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "11D_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "11D_product_scope_analysis_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "11D_product_scope_analysis_lock.sha256"
LOG_PATH = LOG_ROOT / "11D_product_scope_analysis_log.txt"

MEMORY_FILES = {
    "PROJECT_CONTEXT": MEMORY_ROOT / "PROJECT_CONTEXT.md",
    "WORKFLOW": MEMORY_ROOT / "WORKFLOW.md",
    "DECISIONS": MEMORY_ROOT / "DECISIONS.md",
    "FILES_AND_PATHS": MEMORY_ROOT / "FILES_AND_PATHS.md",
    "METRICS_AND_RESULTS": MEMORY_ROOT / "METRICS_AND_RESULTS.md",
    "CHAT_INDEX": MEMORY_ROOT / "CHAT_INDEX.md",
    "CURRENT_HANDOFF": MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

THRESHOLDS = (0.80, 0.90, 0.95)
STEP_ID = "11D"
STATUS = "PART_11D_COMPLETED_READY_FOR_11E"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))
NEW_OUTPUTS = [
    PROFILE_PATH, SCOPE_PATH, MEMBERSHIP_PATH, SEGMENT_PATH,
    SENSITIVITY_PATH, UNIVERSE_PATH, VALIDATION_PATH, MANIFEST_PATH,
    STEP_MEMORY_PATH, CHECKPOINT_PATH, CHECKPOINT_SHA_PATH,
    LOCK_PATH, LOCK_SHA_PATH, LOG_PATH,
]

# -----------------------------------------------------------------------------
# Utilities
# -----------------------------------------------------------------------------
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def verify_sidecar(path: Path, sidecar: Path, label: str) -> str:
    require_file(path, label)
    require_file(sidecar, f"{label} SHA-256 sidecar")
    expected = sidecar.read_text(encoding="utf-8").strip().split()[0].lower()
    actual = sha256_file(path)
    if len(expected) != 64 or actual != expected:
        raise AssertionError(
            f"{label} SHA-256 mismatch:\nExpected: {expected}\nActual:   {actual}"
        )
    return actual


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        temp.write_text(text, encoding="utf-8")
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        frame.to_csv(temp, index=False)
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def atomic_json(path: Path, payload: dict) -> None:
    atomic_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def bool_series(series: pd.Series, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    normalized = series.astype(str).str.strip().str.lower()
    invalid = sorted(set(normalized) - set(mapping))
    if invalid:
        raise ValueError(f"{label} has invalid boolean values: {invalid[:10]}")
    return normalized.map(mapping).astype(bool)


def update_section(text: str, marker: str, heading: str, body: str) -> str:
    start, end = f"<!-- BEGIN {marker} -->", f"<!-- END {marker} -->"
    section = f"{start}\n## {heading}\n\n{body.rstrip()}\n{end}"
    if start in text and end in text:
        before = text.split(start, 1)[0].rstrip()
        after = text.split(end, 1)[1].lstrip()
        return before + "\n\n" + section + ("\n\n" + after if after else "") + "\n"
    return text.rstrip() + "\n\n" + section + "\n"


def stage_path(stage_root: Path, final_path: Path) -> Path:
    return stage_root / final_path.relative_to(EXT_ROOT)


def stage_text(stage_root: Path, final_path: Path, text: str) -> None:
    atomic_text(stage_path(stage_root, final_path), text)


def stage_csv(stage_root: Path, final_path: Path, frame: pd.DataFrame) -> None:
    atomic_csv(stage_path(stage_root, final_path), frame)


def stage_json(stage_root: Path, final_path: Path, payload: dict) -> None:
    atomic_json(stage_path(stage_root, final_path), payload)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)

# -----------------------------------------------------------------------------
# Verify locked Part 11C input
# -----------------------------------------------------------------------------
for directory, label in [
    (EXT_ROOT, "extension root"), (MEMORY_ROOT, "memory root"),
    (OUTPUT_ROOT, "Part 11D output directory"),
    (CHECKPOINT_ROOT, "checkpoint directory"), (LOG_ROOT, "log directory"),
]:
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing required {label}:\n{directory}")

require_file(WEEKLY_DATASET, "Part 11C weekly dataset")
for key, path in MEMORY_FILES.items():
    require_file(path, f"memory file {key}")

if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise FileExistsError(f"Part 11D overwrite lock triggered:\n{LOCK_PATH}")
existing = [path for path in NEW_OUTPUTS if path.exists()]
if existing:
    raise FileExistsError(
        "Existing uncommitted Part 11D outputs found; no files changed:\n"
        + "\n".join(f"- {path}" for path in existing)
    )
for old_stage in EXT_ROOT.glob(".11D_staging_*"):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)

part11c_lock_hash = verify_sidecar(PART11C_LOCK, PART11C_LOCK_SHA, "Part 11C lock")
part11c_checkpoint_hash = verify_sidecar(
    PART11C_CHECKPOINT, PART11C_CHECKPOINT_SHA, "Part 11C checkpoint"
)
part11c_lock = json.loads(PART11C_LOCK.read_text(encoding="utf-8"))
if part11c_lock.get("Status") != "PART_11C_COMPLETED_READY_FOR_11D":
    raise AssertionError(f"Unexpected Part 11C status: {part11c_lock.get('Status')}")
if part11c_lock.get("ReadyForPart11D") is not True:
    raise AssertionError("Part 11C lock does not authorise Part 11D")

for key in [
    "ProtectedTargetVaultOpened", "ProtectedTargetVaultCopied",
    "OpenedDailyHoldoutUsedForWeeklyTuning", "ModelBinariesLoaded",
    "ModelBinariesCopied", "ModelsFitted", "PredictionsCreated",
    "WeeklyFeaturesCreated",
]:
    if part11c_lock.get("SafetyAssertions", {}).get(key) is not False:
        raise AssertionError(f"Invalid Part 11C safety assertion: {key}")

weekly_record = part11c_lock.get("WeeklyDataset", {})
if Path(str(weekly_record.get("Path", ""))) != WEEKLY_DATASET:
    raise AssertionError("Part 11C lock points to a different weekly dataset")
weekly_hash = sha256_file(WEEKLY_DATASET)
if weekly_hash != weekly_record.get("SHA256"):
    raise AssertionError("Weekly dataset hash no longer matches the Part 11C lock")

# -----------------------------------------------------------------------------
# Load, validate and create leakage-safe analysis views
# -----------------------------------------------------------------------------
weekly = pd.read_csv(WEEKLY_DATASET, low_memory=False)
required = {
    "WeekStartDate", "WeekEndDate", "CanonicalProductID",
    "CanonicalProductName", "WeeklyNormalDemand", "WeeklyBulkDemand",
    "WeeklyTotalDemand", "IsDatasetBoundaryPartialWeek",
    "ContainsOpenedDailyHoldoutRows", "WeeklyTuningEligible",
}
missing = sorted(required - set(weekly.columns))
if missing:
    raise AssertionError(f"Weekly dataset missing required columns: {missing}")

for column in ["WeekStartDate", "WeekEndDate"]:
    weekly[column] = pd.to_datetime(weekly[column], errors="raise")
for column in [
    "IsDatasetBoundaryPartialWeek", "ContainsOpenedDailyHoldoutRows",
    "WeeklyTuningEligible",
]:
    weekly[column] = bool_series(weekly[column], column)
for column in ["WeeklyNormalDemand", "WeeklyBulkDemand", "WeeklyTotalDemand"]:
    weekly[column] = pd.to_numeric(weekly[column], errors="raise")
    if weekly[column].isna().any() or (weekly[column] < 0).any():
        raise AssertionError(f"Invalid values in {column}")

if weekly.duplicated(["WeekStartDate", "CanonicalProductID"]).any():
    raise AssertionError("Duplicate product-week keys found")
if not np.allclose(
    weekly["WeeklyTotalDemand"],
    weekly["WeeklyNormalDemand"] + weekly["WeeklyBulkDemand"],
):
    raise AssertionError("Weekly demand components do not reconcile")
if (weekly["ContainsOpenedDailyHoldoutRows"] & weekly["WeeklyTuningEligible"]).any():
    raise AssertionError("Opened holdout rows are marked tuning-eligible")

expected_rows = int(weekly_record["Rows"])
expected_products = int(weekly_record["Products"])
expected_weeks = int(weekly_record["Weeks"])
if len(weekly) != expected_rows:
    raise AssertionError(f"Weekly row count changed: {len(weekly)} != {expected_rows}")
if weekly["CanonicalProductID"].nunique() != expected_products:
    raise AssertionError("Weekly product count changed")
if weekly["WeekStartDate"].nunique() != expected_weeks:
    raise AssertionError("Weekly week count changed")
if (weekly.groupby("CanonicalProductID")["CanonicalProductName"].nunique() > 1).any():
    raise AssertionError("Canonical product names are inconsistent by product ID")

primary = weekly.loc[
    weekly["WeeklyTuningEligible"] & ~weekly["ContainsOpenedDailyHoldoutRows"]
].copy()
complete = primary.loc[~primary["IsDatasetBoundaryPartialWeek"]].copy()
if primary.empty or complete.empty:
    raise AssertionError("No eligible weekly history available")
if primary["ContainsOpenedDailyHoldoutRows"].any():
    raise AssertionError("Opened holdout rows entered Part 11D")

universe = pd.DataFrame([
    {
        "AnalysisView": "PRIMARY_ALL_TUNING_ELIGIBLE_WEEKS",
        "ProductWeekRows": len(primary),
        "Products": primary["CanonicalProductID"].nunique(),
        "CalendarWeeks": primary["WeekStartDate"].nunique(),
        "WeekStartMin": primary["WeekStartDate"].min().strftime("%Y-%m-%d"),
        "WeekStartMax": primary["WeekStartDate"].max().strftime("%Y-%m-%d"),
        "BoundaryPartialWeeks": primary.loc[
            primary["IsDatasetBoundaryPartialWeek"], "WeekStartDate"
        ].nunique(),
        "WeeklyNormalDemand": primary["WeeklyNormalDemand"].sum(),
        "WeeklyBulkDemand": primary["WeeklyBulkDemand"].sum(),
        "WeeklyTotalDemand": primary["WeeklyTotalDemand"].sum(),
        "OpenedHoldoutRows": primary["ContainsOpenedDailyHoldoutRows"].sum(),
    },
    {
        "AnalysisView": "SENSITIVITY_COMPLETE_WEEKS_ONLY",
        "ProductWeekRows": len(complete),
        "Products": complete["CanonicalProductID"].nunique(),
        "CalendarWeeks": complete["WeekStartDate"].nunique(),
        "WeekStartMin": complete["WeekStartDate"].min().strftime("%Y-%m-%d"),
        "WeekStartMax": complete["WeekStartDate"].max().strftime("%Y-%m-%d"),
        "BoundaryPartialWeeks": 0,
        "WeeklyNormalDemand": complete["WeeklyNormalDemand"].sum(),
        "WeeklyBulkDemand": complete["WeeklyBulkDemand"].sum(),
        "WeeklyTotalDemand": complete["WeeklyTotalDemand"].sum(),
        "OpenedHoldoutRows": complete["ContainsOpenedDailyHoldoutRows"].sum(),
    },
])

# -----------------------------------------------------------------------------
# Product profile and coverage scopes
# -----------------------------------------------------------------------------
def build_profile(frame: pd.DataFrame, view_name: str) -> pd.DataFrame:
    total_normal = float(frame["WeeklyNormalDemand"].sum())
    if total_normal <= 0:
        raise AssertionError(f"{view_name} contains no normal demand")

    catalog = weekly[["CanonicalProductID", "CanonicalProductName"]].drop_duplicates(
        "CanonicalProductID"
    )
    observed = frame.groupby("CanonicalProductID", as_index=False).agg(
        EligibleObservedWeeks=("WeekStartDate", "nunique"),
        TotalWeeklyNormalDemand=("WeeklyNormalDemand", "sum"),
        AverageWeeklyNormalDemand=("WeeklyNormalDemand", "mean"),
        MedianWeeklyNormalDemand=("WeeklyNormalDemand", "median"),
        WeeklyNormalDemandStdDevPopulation=(
            "WeeklyNormalDemand", lambda s: float(np.std(s, ddof=0))
        ),
        ActiveNormalDemandWeeks=("WeeklyNormalDemand", lambda s: int((s > 0).sum())),
        ZeroNormalDemandWeeks=("WeeklyNormalDemand", lambda s: int((s == 0).sum())),
        TotalWeeklyBulkDemand=("WeeklyBulkDemand", "sum"),
        TotalWeeklyDemand=("WeeklyTotalDemand", "sum"),
        AverageWeeklyTotalDemand=("WeeklyTotalDemand", "mean"),
        WeeklyTotalDemandStdDevPopulation=(
            "WeeklyTotalDemand", lambda s: float(np.std(s, ddof=0))
        ),
    )
    result = catalog.merge(
        observed,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one",
    )
    for column in [
        "EligibleObservedWeeks", "TotalWeeklyNormalDemand",
        "ActiveNormalDemandWeeks", "ZeroNormalDemandWeeks",
        "TotalWeeklyBulkDemand", "TotalWeeklyDemand",
    ]:
        result[column] = result[column].fillna(0)

    result["ZeroNormalDemandWeekPercentage"] = np.where(
        result["EligibleObservedWeeks"] > 0,
        100 * result["ZeroNormalDemandWeeks"] / result["EligibleObservedWeeks"],
        np.nan,
    )
    result["WeeklyNormalDemandCoefficientOfVariation"] = np.where(
        result["AverageWeeklyNormalDemand"] > 0,
        result["WeeklyNormalDemandStdDevPopulation"]
        / result["AverageWeeklyNormalDemand"],
        np.nan,
    )
    result["NormalDemandShare"] = (
        result["TotalWeeklyNormalDemand"] / total_normal
    )
    result["NormalDemandSharePercentage"] = (
        100 * result["NormalDemandShare"]
    )
    result["AnalysisView"] = view_name
    result = result.sort_values(
        ["TotalWeeklyNormalDemand", "CanonicalProductID"],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)
    result["NormalDemandRank"] = np.arange(1, len(result) + 1)
    result["CumulativeNormalDemand"] = (
        result["TotalWeeklyNormalDemand"].cumsum()
    )
    result["CumulativeNormalDemandCoverage"] = (
        result["CumulativeNormalDemand"] / total_normal
    )
    result["CumulativeNormalDemandCoveragePercentage"] = (
        100 * result["CumulativeNormalDemandCoverage"]
    )
    return result


def build_scopes(profile: pd.DataFrame, view_name: str):
    total = float(profile["TotalWeeklyNormalDemand"].sum())
    membership = profile[[
        "CanonicalProductID",
        "CanonicalProductName",
        "NormalDemandRank",
        "TotalWeeklyNormalDemand",
        "NormalDemandSharePercentage",
        "CumulativeNormalDemandCoveragePercentage",
    ]].copy()
    membership["AnalysisView"] = view_name
    rows = []

    for threshold in THRESHOLDS:
        reached = profile.index[
            profile["CumulativeNormalDemandCoverage"] >= threshold
        ]
        if reached.empty:
            raise AssertionError(
                f"Coverage target {threshold:.0%} not reached"
            )

        index = int(reached[0])
        rank = index + 1
        cutoff_demand = float(
            profile.loc[index, "TotalWeeklyNormalDemand"]
        )
        strict = profile["NormalDemandRank"] <= rank
        tied = (
            profile["TotalWeeklyNormalDemand"] >= cutoff_demand
        )
        strict_units = float(
            profile.loc[strict, "TotalWeeklyNormalDemand"].sum()
        )
        tied_units = float(
            profile.loc[tied, "TotalWeeklyNormalDemand"].sum()
        )
        previous = (
            float(
                profile.loc[
                    index - 1,
                    "CumulativeNormalDemandCoverage",
                ]
            )
            if index
            else 0.0
        )
        label = int(threshold * 100)

        membership[f"InStrict{label}PctScope"] = (
            strict.to_numpy()
        )
        membership[f"InTieInclusive{label}PctScope"] = (
            tied.to_numpy()
        )

        selected = profile.loc[strict]
        rows.append({
            "AnalysisView": view_name,
            "CoverageTargetPercentage": 100 * threshold,
            "StrictProductCount": int(strict.sum()),
            "StrictDemandUnits": strict_units,
            "StrictCoveragePercentage": (
                100 * strict_units / total
            ),
            "PreviousRankCoveragePercentage": 100 * previous,
            "CutoffRank": rank,
            "CutoffProductID": str(
                profile.loc[index, "CanonicalProductID"]
            ),
            "CutoffProductName": str(
                profile.loc[index, "CanonicalProductName"]
            ),
            "CutoffProductNormalDemand": cutoff_demand,
            "ProductsTiedAtCutoffDemand": int(
                (
                    profile["TotalWeeklyNormalDemand"]
                    == cutoff_demand
                ).sum()
            ),
            "TieInclusiveProductCount": int(tied.sum()),
            "TieInclusiveCoveragePercentage": (
                100 * tied_units / total
            ),
            "MedianAverageWeeklyNormalDemand": (
                selected["AverageWeeklyNormalDemand"].median()
            ),
            "MedianActiveDemandWeeks": (
                selected["ActiveNormalDemandWeeks"].median()
            ),
            "MedianZeroDemandWeekPercentage": (
                selected["ZeroNormalDemandWeekPercentage"].median()
            ),
            "MedianWeeklyNormalDemandStdDev": (
                selected[
                    "WeeklyNormalDemandStdDevPopulation"
                ].median()
            ),
            "MedianWeeklyNormalDemandCV": (
                selected[
                    "WeeklyNormalDemandCoefficientOfVariation"
                ].median()
            ),
        })

    return pd.DataFrame(rows), membership


primary_profile = build_profile(
    primary,
    "PRIMARY_ALL_TUNING_ELIGIBLE_WEEKS",
)
complete_profile = build_profile(
    complete,
    "SENSITIVITY_COMPLETE_WEEKS_ONLY",
)

scopes, membership = build_scopes(
    primary_profile,
    "PRIMARY_ALL_TUNING_ELIGIBLE_WEEKS",
)
complete_scopes, complete_membership = build_scopes(
    complete_profile,
    "SENSITIVITY_COMPLETE_WEEKS_ONLY",
)

profile = primary_profile.merge(
    membership.drop(columns=[
        "CanonicalProductName",
        "NormalDemandRank",
        "TotalWeeklyNormalDemand",
        "NormalDemandSharePercentage",
        "CumulativeNormalDemandCoveragePercentage",
        "AnalysisView",
    ]),
    on="CanonicalProductID",
    how="left",
    validate="one_to_one",
)

profile["CoverageBand"] = np.select(
    [
        profile["InStrict80PctScope"],
        profile["InStrict90PctScope"],
        profile["InStrict95PctScope"],
    ],
    [
        "CORE_0_TO_80",
        "EXTENSION_80_TO_90",
        "EXTENSION_90_TO_95",
    ],
    default="TAIL_AFTER_95",
)

profile["CandidateDemandSegment"] = np.select(
    [
        profile["InStrict80PctScope"],
        profile["InStrict95PctScope"],
    ],
    [
        "HIGH_CANDIDATE",
        "MODERATE_CANDIDATE",
    ],
    default="OUTSIDE_INITIAL_SCOPE_CANDIDATE",
)

profile["CandidateRuleStatus"] = (
    "DIAGNOSTIC_ONLY_RECALCULATE_WITHIN_EACH_TRAINING_FOLD"
)

membership_output = profile[[
    "CanonicalProductID",
    "CanonicalProductName",
    "NormalDemandRank",
    "TotalWeeklyNormalDemand",
    "NormalDemandSharePercentage",
    "CumulativeNormalDemandCoveragePercentage",
    "InStrict80PctScope",
    "InTieInclusive80PctScope",
    "InStrict90PctScope",
    "InTieInclusive90PctScope",
    "InStrict95PctScope",
    "InTieInclusive95PctScope",
    "CoverageBand",
    "CandidateDemandSegment",
    "CandidateRuleStatus",
]].copy()

segments = (
    profile.groupby(
        "CandidateDemandSegment",
        as_index=False,
    )
    .agg(
        Products=("CanonicalProductID", "nunique"),
        NormalDemandUnits=("TotalWeeklyNormalDemand", "sum"),
        MedianAverageWeeklyNormalDemand=(
            "AverageWeeklyNormalDemand",
            "median",
        ),
        MedianActiveDemandWeeks=(
            "ActiveNormalDemandWeeks",
            "median",
        ),
        MedianZeroDemandWeekPercentage=(
            "ZeroNormalDemandWeekPercentage",
            "median",
        ),
        MedianWeeklyNormalDemandStdDev=(
            "WeeklyNormalDemandStdDevPopulation",
            "median",
        ),
        MedianWeeklyNormalDemandCV=(
            "WeeklyNormalDemandCoefficientOfVariation",
            "median",
        ),
    )
)

segments["NormalDemandCoveragePercentage"] = (
    100
    * segments["NormalDemandUnits"]
    / profile["TotalWeeklyNormalDemand"].sum()
)

segments["SortOrder"] = (
    segments["CandidateDemandSegment"].map({
        "HIGH_CANDIDATE": 1,
        "MODERATE_CANDIDATE": 2,
        "OUTSIDE_INITIAL_SCOPE_CANDIDATE": 3,
    })
)

segments = (
    segments.sort_values("SortOrder")
    .drop(columns="SortOrder")
)

segments["Status"] = (
    "DIAGNOSTIC_ONLY_NOT_LOCKED_FOR_MODELLING"
)

sensitivity_rows = []

for threshold in THRESHOLDS:
    label = int(threshold * 100)

    primary_set = set(
        membership.loc[
            membership[f"InStrict{label}PctScope"],
            "CanonicalProductID",
        ].astype(str)
    )
    complete_set = set(
        complete_membership.loc[
            complete_membership[f"InStrict{label}PctScope"],
            "CanonicalProductID",
        ].astype(str)
    )

    intersection = primary_set & complete_set
    union = primary_set | complete_set
    added = sorted(complete_set - primary_set)
    removed = sorted(primary_set - complete_set)

    sensitivity_rows.append({
        "CoverageTargetPercentage": 100 * threshold,
        "PrimaryProductCount": len(primary_set),
        "CompleteWeeksOnlyProductCount": len(complete_set),
        "IntersectionProducts": len(intersection),
        "UnionProducts": len(union),
        "JaccardSimilarity": (
            len(intersection) / len(union)
            if union
            else 1.0
        ),
        "ProductsAddedWhenPartialWeeksExcluded": len(added),
        "ProductsRemovedWhenPartialWeeksExcluded": len(removed),
        "AddedProductIDs": ";".join(added),
        "RemovedProductIDs": ";".join(removed),
    })

sensitivity = pd.DataFrame(sensitivity_rows)

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------
minimal = True

for row in scopes.itertuples(index=False):
    target = row.CoverageTargetPercentage
    minimal &= (
        row.StrictCoveragePercentage >= target - 1e-10
    )
    if row.CutoffRank > 1:
        minimal &= (
            row.PreviousRankCoveragePercentage < target
        )

share_sum = float(profile["NormalDemandShare"].sum())
final_coverage = float(
    profile["CumulativeNormalDemandCoverage"].iloc[-1]
)

nested_80_90 = bool(
    profile.loc[
        profile["InStrict80PctScope"],
        "InStrict90PctScope",
    ].all()
)

nested_90_95 = bool(
    profile.loc[
        profile["InStrict90PctScope"],
        "InStrict95PctScope",
    ].all()
)

validation = pd.DataFrame([
    {
        "Check": "Part 11C lock and checkpoint verified",
        "Expected": True,
        "Actual": True,
        "Passed": True,
    },
    {
        "Check": "Weekly dataset hash matched Part 11C lock",
        "Expected": weekly_record["SHA256"],
        "Actual": weekly_hash,
        "Passed": weekly_hash == weekly_record["SHA256"],
    },
    {
        "Check": "Weekly dataset rows unchanged",
        "Expected": expected_rows,
        "Actual": len(weekly),
        "Passed": len(weekly) == expected_rows,
    },
    {
        "Check": "Opened holdout product-weeks used",
        "Expected": 0,
        "Actual": int(
            primary["ContainsOpenedDailyHoldoutRows"].sum()
        ),
        "Passed": not primary[
            "ContainsOpenedDailyHoldoutRows"
        ].any(),
    },
    {
        "Check": "Product profile rows",
        "Expected": expected_products,
        "Actual": len(profile),
        "Passed": len(profile) == expected_products,
    },
    {
        "Check": "Normal-demand shares sum to one",
        "Expected": 1.0,
        "Actual": share_sum,
        "Passed": math.isclose(
            share_sum,
            1.0,
            abs_tol=1e-10,
        ),
    },
    {
        "Check": "Final cumulative coverage equals one",
        "Expected": 1.0,
        "Actual": final_coverage,
        "Passed": math.isclose(
            final_coverage,
            1.0,
            abs_tol=1e-10,
        ),
    },
    {
        "Check": "Cumulative coverage non-decreasing",
        "Expected": True,
        "Actual": profile[
            "CumulativeNormalDemandCoverage"
        ].is_monotonic_increasing,
        "Passed": profile[
            "CumulativeNormalDemandCoverage"
        ].is_monotonic_increasing,
    },
    {
        "Check": "80/90/95 scopes minimal and reach targets",
        "Expected": True,
        "Actual": bool(minimal),
        "Passed": bool(minimal),
    },
    {
        "Check": "Strict 80% nested in strict 90%",
        "Expected": True,
        "Actual": nested_80_90,
        "Passed": nested_80_90,
    },
    {
        "Check": "Strict 90% nested in strict 95%",
        "Expected": True,
        "Actual": nested_90_95,
        "Passed": nested_90_95,
    },
    {
        "Check": "Candidate segments cover all products",
        "Expected": expected_products,
        "Actual": int(segments["Products"].sum()),
        "Passed": (
            int(segments["Products"].sum())
            == expected_products
        ),
    },
    {
        "Check": "Models, features or predictions created",
        "Expected": False,
        "Actual": False,
        "Passed": True,
    },
])

if not validation["Passed"].all():
    raise AssertionError(
        "Part 11D validation failed:\n"
        + validation.loc[
            ~validation["Passed"]
        ].to_string(index=False)
    )

# -----------------------------------------------------------------------------
# Stage outputs, memory, checkpoint and lock
# -----------------------------------------------------------------------------
stage_root = (
    EXT_ROOT
    / f".11D_staging_{uuid.uuid4().hex}"
)
stage_root.mkdir(parents=True, exist_ok=False)

try:
    for path, frame in [
        (PROFILE_PATH, profile),
        (SCOPE_PATH, scopes),
        (MEMBERSHIP_PATH, membership_output),
        (SEGMENT_PATH, segments),
        (SENSITIVITY_PATH, sensitivity),
        (UNIVERSE_PATH, universe),
        (VALIDATION_PATH, validation),
    ]:
        stage_csv(stage_root, path, frame)

    decisions = "\n".join([
        f"- Decision date: {NOW_LOCAL.date().isoformat()}.",
        "- Scope-ranking target: WeeklyNormalDemand.",
        "- Primary universe: all product-weeks marked WeeklyTuningEligible.",
        "- The opened March 2026 holdout was excluded.",
        "- The boundary partial week was retained in the primary view and tested separately.",
        "- Strict scopes use total normal demand descending and CanonicalProductID ascending.",
        "- Tie-inclusive scope counts are reported separately.",
        "- Candidate HIGH = strict 80% scope; MODERATE = strict 80%-95% extension.",
        "- Candidate segments must be recalculated inside every chronological training fold.",
    ])

    files = "\n".join([
        f"- Product profile: `{PROFILE_PATH}`",
        f"- Scope summary: `{SCOPE_PATH}`",
        f"- Scope membership: `{MEMBERSHIP_PATH}`",
        f"- Candidate segment diagnostics: `{SEGMENT_PATH}`",
        f"- Boundary sensitivity: `{SENSITIVITY_PATH}`",
        f"- Validation: `{VALIDATION_PATH}`",
    ])

    result_lines = [
        f"- Leakage-safe product-week rows: {len(primary):,}",
        f"- Products: {primary['CanonicalProductID'].nunique():,}",
        f"- Calendar weeks: {primary['WeekStartDate'].nunique():,}",
        f"- Pre-holdout normal-demand units: {primary['WeeklyNormalDemand'].sum():,.0f}",
    ]

    for row in scopes.itertuples(index=False):
        result_lines.append(
            f"- {row.CoverageTargetPercentage:.0f}% scope: "
            f"{row.StrictProductCount} products, "
            f"{row.StrictCoveragePercentage:.4f}% coverage"
        )

    results = "\n".join(result_lines)

    for key, final_path in MEMORY_FILES.items():
        original = final_path.read_text(
            encoding="utf-8"
        )

        if key == "PROJECT_CONTEXT":
            updated = update_section(
                original,
                "STEP_11D",
                "Part 11D product-scope analysis",
                (
                    "Weekly concentration and intermittency "
                    "were analysed using pre-holdout "
                    "WeeklyNormalDemand only."
                ),
            )

        elif key == "WORKFLOW":
            updated = update_section(
                original,
                "STEP_11D",
                "Part 11D workflow status",
                (
                    "Part 11D is complete. Part 11E will "
                    "create weekly baselines. Scopes must "
                    "be recalculated inside each training fold."
                ),
            )

        elif key == "DECISIONS":
            updated = update_section(
                original,
                "STEP_11D",
                "Part 11D decisions",
                decisions,
            )

        elif key == "FILES_AND_PATHS":
            updated = update_section(
                original,
                "STEP_11D",
                "Part 11D files",
                files,
            )

        elif key == "METRICS_AND_RESULTS":
            updated = update_section(
                original,
                "STEP_11D",
                "Part 11D metrics and results",
                results,
            )

        elif key == "CHAT_INDEX":
            row = (
                f"| {NOW_LOCAL.isoformat()} | 11D | "
                "Weekly demand concentration, intermittency "
                f"and 80/90/95 scopes | {STATUS} |"
            )
            updated = (
                original
                if row in original
                else original.rstrip() + "\n" + row + "\n"
            )

        elif key == "CURRENT_HANDOFF":
            updated = "\n".join([
                "# Current Handoff",
                "",
                f"- Current completed step: {STEP_ID}",
                f"- Status: {STATUS}",
                f"- Updated local time: {NOW_LOCAL.isoformat()}",
                f"- Weekly input: {WEEKLY_DATASET}",
                f"- Product profile: {PROFILE_PATH}",
                f"- Scope summary: {SCOPE_PATH}",
                "- Scope basis: pre-holdout WeeklyNormalDemand only.",
                "- Candidate segments are diagnostic and must be recalculated within each fold.",
                "- Next step: 11E weekly baselines.",
                "",
            ])

        else:
            raise KeyError(key)

        stage_text(
            stage_root,
            final_path,
            updated,
        )

    step_text = f"""# Step 11D — Weekly Product Scope and Segmentation Analysis

- **Step ID:** 11D
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## User request
Calculate total and average weekly demand, active weeks, zero-demand percentage, weekly variability, demand share, cumulative coverage, and 80%, 90% and 95% normal-demand scopes.

## Technical actions
- Verified the Part 11C lock, checkpoint and weekly-dataset hash.
- Excluded all product-weeks overlapping the opened March 2026 holdout.
- Calculated product-level normal-demand and total-demand profiles.
- Calculated population standard deviation and coefficient of variation.
- Constructed deterministic minimal and tie-inclusive 80%, 90% and 95% scopes.
- Tested sensitivity to excluding the dataset-boundary partial week.
- Created diagnostic HIGH and MODERATE candidate segments.

## Decisions
{decisions}

## Inputs
- `{WEEKLY_DATASET}`
- `{PART11C_LOCK}`
- `{PART11C_CHECKPOINT}`

## Outputs
{files}

## Validation
All {len(validation)} validation checks passed.

## Errors encountered
None during the successful execution.

## Results
{results}

## Limitations
- Scope membership from this analysis cannot be reused as fixed training labels.
- Candidate segments are not yet a selected modelling route.
- Products without eligible pre-holdout rows remain listed with zero totals and undefined averages.

## Next action
Part 11E: create leakage-safe weekly baselines, including a baseline formed by summing locked daily forecasts where dates align.
"""

    stage_text(
        stage_root,
        STEP_MEMORY_PATH,
        step_text,
    )

    checkpoint = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "Input": {
            "WeeklyDatasetPath": str(WEEKLY_DATASET),
            "WeeklyDatasetSHA256": weekly_hash,
            "Part11CLockSHA256": part11c_lock_hash,
            "Part11CCheckpointSHA256": part11c_checkpoint_hash,
        },
        "AnalysisUniverse": {
            "Rows": int(len(primary)),
            "Products": int(
                primary["CanonicalProductID"].nunique()
            ),
            "Weeks": int(
                primary["WeekStartDate"].nunique()
            ),
            "NormalDemand": float(
                primary["WeeklyNormalDemand"].sum()
            ),
            "OpenedHoldoutRowsUsed": 0,
        },
        "CoverageScopes": scopes.to_dict(
            orient="records"
        ),
        "CandidateSegmentation": {
            "High": "Strict 80% scope",
            "Moderate": "Strict 80%-95% extension",
            "Status": (
                "DIAGNOSTIC_ONLY_RECALCULATE_WITHIN_"
                "EACH_TRAINING_FOLD"
            ),
        },
        "Safety": {
            "ProtectedTargetVaultOpened": False,
            "ProtectedTargetVaultCopied": False,
            "OpenedDailyHoldoutUsedForScopeSelection": False,
            "ModelsFitted": False,
            "WeeklyFeaturesCreated": False,
            "PredictionsCreated": False,
            "FixedScopeAuthorisedForFoldReuse": False,
        },
        "ReadyForPart11E": True,
        "NextStep": "11E",
    }

    stage_json(
        stage_root,
        CHECKPOINT_PATH,
        checkpoint,
    )

    checkpoint_hash = sha256_file(
        stage_path(stage_root, CHECKPOINT_PATH)
    )

    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        f"{checkpoint_hash}  {CHECKPOINT_PATH.name}\n",
    )

    manifest_targets = [
        PROFILE_PATH,
        SCOPE_PATH,
        MEMBERSHIP_PATH,
        SEGMENT_PATH,
        SENSITIVITY_PATH,
        UNIVERSE_PATH,
        VALIDATION_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]

    manifest = pd.DataFrame([
        {
            "RelativePath": str(
                path.relative_to(EXT_ROOT)
            ),
            "Bytes": (
                stage_path(stage_root, path)
                .stat()
                .st_size
            ),
            "SHA256": sha256_file(
                stage_path(stage_root, path)
            ),
        }
        for path in manifest_targets
    ]).sort_values("RelativePath")

    stage_csv(
        stage_root,
        MANIFEST_PATH,
        manifest,
    )

    manifest_hash = sha256_file(
        stage_path(stage_root, MANIFEST_PATH)
    )

    lock = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "Part11C": {
            "LockSHA256": part11c_lock_hash,
            "CheckpointSHA256": part11c_checkpoint_hash,
            "WeeklyDatasetPath": str(WEEKLY_DATASET),
            "WeeklyDatasetSHA256": weekly_hash,
        },
        "PrimaryAnalysis": {
            "Target": "WeeklyNormalDemand",
            "Rows": int(len(primary)),
            "Products": int(
                primary["CanonicalProductID"].nunique()
            ),
            "Weeks": int(
                primary["WeekStartDate"].nunique()
            ),
            "NormalDemand": float(
                primary["WeeklyNormalDemand"].sum()
            ),
            "OpenedHoldoutUsed": False,
        },
        "CoverageScopes": scopes.to_dict(
            orient="records"
        ),
        "CandidateSegments": {
            "High": "Strict 80% scope",
            "Moderate": "Strict 80%-95% extension",
            "Status": (
                "DIAGNOSTIC_ONLY_NOT_LOCKED_FOR_MODELLING"
            ),
            "MustRecalculateWithinEachTrainingFold": True,
        },
        "OutputHashManifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_hash,
            "FilesListed": int(len(manifest)),
        },
        "Checkpoint": {
            "Path": str(CHECKPOINT_PATH),
            "SHA256": checkpoint_hash,
        },
        "SafetyAssertions": checkpoint["Safety"],
        "ReadyForPart11E": True,
        "NextStep": "11E",
    }

    stage_json(
        stage_root,
        LOCK_PATH,
        lock,
    )

    lock_hash = sha256_file(
        stage_path(stage_root, LOCK_PATH)
    )

    stage_text(
        stage_root,
        LOCK_SHA_PATH,
        f"{lock_hash}  {LOCK_PATH.name}\n",
    )

    stage_text(
        stage_root,
        LOG_PATH,
        "\n".join([
            f"Status: {STATUS}",
            f"Run local: {NOW_LOCAL.isoformat()}",
            f"Weekly input SHA256: {weekly_hash}",
            f"Primary rows: {len(primary)}",
            (
                "Primary products: "
                f"{primary['CanonicalProductID'].nunique()}"
            ),
            (
                "Primary weeks: "
                f"{primary['WeekStartDate'].nunique()}"
            ),
            (
                "Primary normal demand: "
                f"{primary['WeeklyNormalDemand'].sum()}"
            ),
            "Opened holdout rows used: 0",
            f"Checkpoint SHA256: {checkpoint_hash}",
            f"Part 11D lock SHA256: {lock_hash}",
            "",
        ]),
    )

    staged_files = [
        path
        for path in stage_root.rglob("*")
        if path.is_file()
    ]

    if not staged_files:
        raise AssertionError(
            "Part 11D staging directory is empty"
        )

    if any(
        (
            "target_vault" in path.name.lower()
            or path.suffix.lower() == ".joblib"
        )
        for path in staged_files
    ):
        raise PermissionError(
            "Forbidden target-vault or model file "
            "detected in staging"
        )

    for staged in sorted(staged_files):
        final = (
            EXT_ROOT
            / staged.relative_to(stage_root)
        )
        final.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if (
            final.exists()
            and final not in MEMORY_FILES.values()
        ):
            raise FileExistsError(
                f"Refusing to overwrite output: {final}"
            )

        os.replace(staged, final)

    for path in NEW_OUTPUTS:
        make_read_only(path)

finally:
    if stage_root.exists():
        shutil.rmtree(stage_root)

# -----------------------------------------------------------------------------
# Final output
# -----------------------------------------------------------------------------
print("=" * 100)
print(
    "EDEN WEEKLY FORECASTING EXTENSION — "
    "PART 11D COMPLETE"
)
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Extension root: {EXT_ROOT}")
print(f"Local time: {NOW_LOCAL.isoformat()}")

print("\nANALYSIS UNIVERSE")
print(universe.to_string(index=False))

print("\nNORMAL-DEMAND COVERAGE SCOPES")
print(scopes.to_string(index=False))

print(
    "\nCANDIDATE SEGMENT DIAGNOSTICS — "
    "DIAGNOSTIC ONLY"
)
print(segments.to_string(index=False))

print("\nBOUNDARY-WEEK SENSITIVITY")
print(sensitivity.to_string(index=False))

print("\nPART 11D VALIDATION")
print(validation.to_string(index=False))

print("\nCONTROL OUTPUTS")
print(f"- Product profile: {PROFILE_PATH}")
print(f"- Scope summary: {SCOPE_PATH}")
print(f"- Scope membership: {MEMBERSHIP_PATH}")
print(
    "- Candidate segment diagnostics: "
    f"{SEGMENT_PATH}"
)
print(
    "- Boundary sensitivity: "
    f"{SENSITIVITY_PATH}"
)
print(f"- Checkpoint: {CHECKPOINT_PATH}")
print(f"- Checkpoint SHA256: {checkpoint_hash}")
print(f"- Part 11D lock: {LOCK_PATH}")
print(f"- Part 11D lock SHA256: {lock_hash}")

print(
    "\nSAFETY: protected target vault opened/copied "
    "False/False; opened daily holdout used for scope "
    "selection False; models/features/predictions created "
    "False/False/False; fixed scope authorised for fold "
    "reuse False."
)

print("=" * 100)

EDEN WEEKLY FORECASTING EXTENSION — PART 11D COMPLETE
Status: PART_11D_COMPLETED_READY_FOR_11E
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-05T20:23:27.874139+01:00

ANALYSIS UNIVERSE
                     AnalysisView  ProductWeekRows  Products  CalendarWeeks WeekStartMin WeekStartMax  BoundaryPartialWeeks  WeeklyNormalDemand  WeeklyBulkDemand  WeeklyTotalDemand  OpenedHoldoutRows
PRIMARY_ALL_TUNING_ELIGIBLE_WEEKS             8222       227             47   2025-03-31   2026-02-23                     1              100177              1972             102149                  0
  SENSITIVITY_COMPLETE_WEEKS_ONLY             8115       227             46   2025-04-07   2026-02-23                     0               98351              1972             100323                  0

NORMAL-DEMAND COVERAGE SCOPES
                     AnalysisView  CoverageTargetPercentage  StrictProductCount  StrictDemandUnits  StrictCoverage

In [17]:
from __future__ import annotations
import hashlib
import json
import math
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
PROJECT_ROOT = Path('/Users/ryansmac/Desktop/Meng Project')
EDEN_ROOT = PROJECT_ROOT / 'eden_datasets'
EXT_ROOT = EDEN_ROOT / 'weekly_forecasting_extension'
MEMORY_ROOT = EXT_ROOT / '00_project_memory'
STEP_MEMORY_ROOT = MEMORY_ROOT / 'steps'
SOURCE_ROOT = EXT_ROOT / '01_source_snapshot'
WEEKLY_DATASET = EXT_ROOT / '02_weekly_data' / 'final' / '11C_weekly_dataset.csv'
SCOPE_ROOT = EXT_ROOT / '03_product_scope_and_segmentation'
BASELINE_ROOT = EXT_ROOT / '05_weekly_baselines'
VALIDATION_ROOT = EXT_ROOT / '07_weekly_validation'
CHECKPOINT_ROOT = EXT_ROOT / '11_checkpoints'
LOG_ROOT = EXT_ROOT / '12_logs'
PART11D_LOCK = CHECKPOINT_ROOT / '11D_product_scope_analysis_lock.json'
PART11D_LOCK_SHA = CHECKPOINT_ROOT / '11D_product_scope_analysis_lock.sha256'
PART11D_CHECKPOINT = CHECKPOINT_ROOT / '11D_checkpoint.json'
PART11D_CHECKPOINT_SHA = CHECKPOINT_ROOT / '11D_checkpoint.sha256'
PART11D_MANIFEST = SCOPE_ROOT / '11D_output_hash_manifest.csv'
PART11D_MEMBERSHIP = SCOPE_ROOT / '11D_normal_demand_scope_membership.csv'
PART11B_COPY_AUDIT = SOURCE_ROOT / 'manifests' / '11B_source_copy_audit.csv'
DAILY_PRODUCT_PREDICTIONS = SOURCE_ROOT / 'daily_model_reference' / '10B2_final_product_predictions.csv'
DAILY_AGGREGATE_PREDICTIONS = SOURCE_ROOT / 'daily_model_reference' / '10B2_final_aggregate_predictions.csv'
PART10C_ROOT = EDEN_ROOT / 'modelling' / '05_final_refit_and_prediction' / '10C_final_holdout_evaluation'
PART10C_LOCK = PART10C_ROOT / '10C_final_evaluation_lock.json'
PART10C_LOCK_SHA = PART10C_ROOT / '10C_final_evaluation_lock.sha256'
SCORED_DAILY_PRODUCT = PART10C_ROOT / '10C_scored_product_predictions.csv'
SCORED_DAILY_AGGREGATE = PART10C_ROOT / '10C_scored_aggregate_predictions.csv'
EXPECTED_PART10C_LOCK_SHA256 = '95e23afaf3944f1b456ad9a90e99cb777d31812f2cc11708012b6a10143cb725'
EXPECTED_PART10C_STATUS = 'FINAL_HOLDOUT_EVALUATION_SAVED_AND_HASHED'
PREHOLDOUT_PREDICTIONS_PATH = BASELINE_ROOT / '11E_preholdout_weekly_baseline_predictions.csv'
PREHOLDOUT_METRICS_PATH = BASELINE_ROOT / '11E_preholdout_weekly_baseline_metrics.csv'
BASELINE_RANKING_PATH = BASELINE_ROOT / '11E_preholdout_baseline_ranking.csv'
FOLD_SCOPE_MEMBERSHIP_PATH = BASELINE_ROOT / '11E_fold_scope_membership.csv'
FOLD_SCOPE_SUMMARY_PATH = BASELINE_ROOT / '11E_fold_scope_summary.csv'
DAILY_PRODUCT_WEEKLY_PATH = BASELINE_ROOT / '11E_locked_daily_product_weekly_baseline.csv'
DAILY_AGGREGATE_WEEKLY_PATH = BASELINE_ROOT / '11E_locked_daily_aggregate_weekly_baseline.csv'
DAILY_DIAGNOSTIC_METRICS_PATH = BASELINE_ROOT / '11E_locked_daily_weekly_diagnostic_metrics.csv'
DAILY_SCORING_COVERAGE_PATH = BASELINE_ROOT / '11E_locked_daily_scoring_universe_coverage_audit.csv'
FOLD_METRICS_PATH = VALIDATION_ROOT / '11E_weekly_baseline_fold_metrics.csv'
DESIGN_PATH = VALIDATION_ROOT / '11E_baseline_design_contract.json'
VALIDATION_PATH = VALIDATION_ROOT / '11E_validation.csv'
MANIFEST_PATH = BASELINE_ROOT / '11E_output_hash_manifest.csv'
STEP_MEMORY_PATH = STEP_MEMORY_ROOT / 'STEP_11E_WEEKLY_BASELINES.md'
CHECKPOINT_PATH = CHECKPOINT_ROOT / '11E_checkpoint.json'
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / '11E_checkpoint.sha256'
LOCK_PATH = CHECKPOINT_ROOT / '11E_weekly_baselines_lock.json'
LOCK_SHA_PATH = CHECKPOINT_ROOT / '11E_weekly_baselines_lock.sha256'
LOG_PATH = LOG_ROOT / '11E_weekly_baselines_log.txt'
MEMORY_FILES = {'PROJECT_CONTEXT': MEMORY_ROOT / 'PROJECT_CONTEXT.md', 'WORKFLOW': MEMORY_ROOT / 'WORKFLOW.md', 'DECISIONS': MEMORY_ROOT / 'DECISIONS.md', 'FILES_AND_PATHS': MEMORY_ROOT / 'FILES_AND_PATHS.md', 'METRICS_AND_RESULTS': MEMORY_ROOT / 'METRICS_AND_RESULTS.md', 'CHAT_INDEX': MEMORY_ROOT / 'CHAT_INDEX.md', 'CURRENT_HANDOFF': MEMORY_ROOT / 'CURRENT_HANDOFF.md'}
STEP_ID = '11E'
STATUS = 'PART_11E_COMPLETED_READY_FOR_11F'
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo('Europe/Dublin'))
MIN_TRAINING_WEEKS = 8
SCOPE_THRESHOLDS = (0.8, 0.9, 0.95)
EXPECTED_DAILY_PRODUCT_ROWS = 4150
EXPECTED_DAILY_DATES = 20
EXPECTED_DAILY_PRODUCTS = 218
EXPECTED_OPENED_HOLDOUT_TOTAL = 13797.0
EXPECTED_LOCKED_PRODUCT_FORECAST_TOTAL = 14532.595485
EXPECTED_LOCKED_AGGREGATE_FORECAST_TOTAL = 14337.844375
BASELINE_METHODS = ['ZERO', 'NAIVE_LAST_CONTEXT', 'MEAN_LAST_4_CONTEXTS', 'MEDIAN_LAST_4_CONTEXTS', 'MEAN_LAST_8_CONTEXTS', 'EXPANDING_MEAN_CONTEXTS']
METHOD_COMPLEXITY_ORDER = {'ZERO': 0, 'NAIVE_LAST_CONTEXT': 1, 'MEAN_LAST_4_CONTEXTS': 2, 'MEDIAN_LAST_4_CONTEXTS': 3, 'MEAN_LAST_8_CONTEXTS': 4, 'EXPANDING_MEAN_CONTEXTS': 5}
TARGETS = {'WEEKLY_NORMAL_DEMAND': 'WeeklyNormalDemand', 'WEEKLY_TOTAL_DEMAND': 'WeeklyTotalDemand'}
NEW_OUTPUTS = [PREHOLDOUT_PREDICTIONS_PATH, PREHOLDOUT_METRICS_PATH, BASELINE_RANKING_PATH, FOLD_SCOPE_MEMBERSHIP_PATH, FOLD_SCOPE_SUMMARY_PATH, DAILY_PRODUCT_WEEKLY_PATH, DAILY_AGGREGATE_WEEKLY_PATH, DAILY_DIAGNOSTIC_METRICS_PATH, DAILY_SCORING_COVERAGE_PATH, FOLD_METRICS_PATH, DESIGN_PATH, VALIDATION_PATH, MANIFEST_PATH, STEP_MEMORY_PATH, CHECKPOINT_PATH, CHECKPOINT_SHA_PATH, LOCK_PATH, LOCK_SHA_PATH, LOG_PATH]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required {label}:\n{path}')

def verify_sidecar(path: Path, sidecar: Path, label: str) -> str:
    require_file(path, label)
    require_file(sidecar, f'{label} SHA-256 sidecar')
    expected = sidecar.read_text(encoding='utf-8').strip().split()[0].lower()
    actual = sha256_file(path)
    if len(expected) != 64 or actual != expected:
        raise AssertionError(f'{label} SHA-256 mismatch:\nExpected: {expected}\nActual:   {actual}')
    return actual

def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    try:
        temporary.write_text(text, encoding='utf-8')
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()

def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    try:
        frame.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()

def atomic_json(path: Path, payload: dict) -> None:
    atomic_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + '\n')

def bool_series(series: pd.Series, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapping = {'true': True, 'false': False, '1': True, '0': False}
    normalised = series.astype(str).str.strip().str.lower()
    invalid = sorted(set(normalised) - set(mapping))
    if invalid:
        raise ValueError(f'{label} has invalid boolean values: {invalid[:10]}')
    return normalised.map(mapping).astype(bool)

def metric_record(actual: pd.Series, prediction: pd.Series) -> dict:
    actual_array = pd.to_numeric(actual, errors='raise').to_numpy(dtype=float)
    prediction_array = pd.to_numeric(prediction, errors='raise').to_numpy(dtype=float)
    if len(actual_array) != len(prediction_array):
        raise AssertionError('Actual and prediction arrays have different lengths')
    if not np.isfinite(actual_array).all() or not np.isfinite(prediction_array).all():
        raise AssertionError('Non-finite values entered metric calculation')
    errors = prediction_array - actual_array
    absolute_errors = np.abs(errors)
    denominator = float(np.abs(actual_array).sum())
    return {'Observations': int(len(actual_array)), 'ActualTotal': float(actual_array.sum()), 'PredictedTotal': float(prediction_array.sum()), 'MAE': float(absolute_errors.mean()) if len(actual_array) else np.nan, 'RMSE': float(np.sqrt(np.mean(errors ** 2))) if len(actual_array) else np.nan, 'WAPEPercentage': float(100.0 * absolute_errors.sum() / denominator) if denominator != 0 else np.nan, 'MeanBias': float(errors.mean()) if len(actual_array) else np.nan, 'TotalBias': float(errors.sum())}

def update_section(text: str, marker: str, heading: str, body: str) -> str:
    start = f'<!-- BEGIN {marker} -->'
    end = f'<!-- END {marker} -->'
    section = f'{start}\n## {heading}\n\n{body.rstrip()}\n{end}'
    if start in text and end in text:
        before = text.split(start, 1)[0].rstrip()
        after = text.split(end, 1)[1].lstrip()
        return before + '\n\n' + section + ('\n\n' + after if after else '') + '\n'
    return text.rstrip() + '\n\n' + section + '\n'

def stage_path(stage_root: Path, final_path: Path) -> Path:
    return stage_root / final_path.relative_to(EXT_ROOT)

def stage_text(stage_root: Path, final_path: Path, text: str) -> None:
    atomic_text(stage_path(stage_root, final_path), text)

def stage_csv(stage_root: Path, final_path: Path, frame: pd.DataFrame) -> None:
    atomic_csv(stage_path(stage_root, final_path), frame)

def stage_json(stage_root: Path, final_path: Path, payload: dict) -> None:
    atomic_json(stage_path(stage_root, final_path), payload)

def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)

def strict_scope_flags(training_totals: pd.DataFrame) -> pd.DataFrame:
    ranked = training_totals.copy()
    ranked['CanonicalProductID'] = ranked['CanonicalProductID'].astype(str)
    ranked['TrainingNormalDemand'] = pd.to_numeric(ranked['TrainingNormalDemand'], errors='raise')
    ranked = ranked.sort_values(['TrainingNormalDemand', 'CanonicalProductID'], ascending=[False, True], kind='mergesort').reset_index(drop=True)
    total = float(ranked['TrainingNormalDemand'].sum())
    if total <= 0:
        raise AssertionError('A fold training period contains no normal demand')
    ranked['FoldNormalDemandRank'] = np.arange(1, len(ranked) + 1)
    ranked['FoldCumulativeNormalDemandCoverage'] = ranked['TrainingNormalDemand'].cumsum() / total
    for threshold in SCOPE_THRESHOLDS:
        label = int(threshold * 100)
        reached = ranked.index[ranked['FoldCumulativeNormalDemandCoverage'] >= threshold]
        if reached.empty:
            raise AssertionError(f'Fold scope did not reach {label}%')
        cutoff_rank = int(reached[0]) + 1
        ranked[f'InFoldStrict{label}PctScope'] = ranked['FoldNormalDemandRank'] <= cutoff_rank
    return ranked
for directory, label in [(EXT_ROOT, 'extension root'), (MEMORY_ROOT, 'memory root'), (BASELINE_ROOT, 'weekly baseline directory'), (VALIDATION_ROOT, 'weekly validation directory'), (CHECKPOINT_ROOT, 'checkpoint directory'), (LOG_ROOT, 'log directory')]:
    if not directory.is_dir():
        raise FileNotFoundError(f'Missing required {label}:\n{directory}')
for path, label in [(WEEKLY_DATASET, 'Part 11C weekly dataset'), (PART11D_MANIFEST, 'Part 11D output manifest'), (PART11D_MEMBERSHIP, 'Part 11D fixed pre-holdout scope membership'), (PART11B_COPY_AUDIT, 'Part 11B source-copy audit'), (DAILY_PRODUCT_PREDICTIONS, 'locked daily product predictions snapshot'), (DAILY_AGGREGATE_PREDICTIONS, 'locked daily aggregate predictions snapshot'), (PART10C_LOCK, 'Part 10C final-evaluation lock'), (PART10C_LOCK_SHA, 'Part 10C final-evaluation lock SHA-256 sidecar'), (SCORED_DAILY_PRODUCT, 'Part 10C scored product predictions'), (SCORED_DAILY_AGGREGATE, 'Part 10C scored aggregate predictions')]:
    require_file(path, label)
for key, path in MEMORY_FILES.items():
    require_file(path, f'memory file {key}')
if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise FileExistsError(f'Part 11E overwrite lock triggered:\n{LOCK_PATH}')
existing_outputs = [path for path in NEW_OUTPUTS if path.exists()]
if existing_outputs:
    raise FileExistsError('Existing uncommitted Part 11E outputs found; no files changed:\n' + '\n'.join((f'- {path}' for path in existing_outputs)))
for old_stage in EXT_ROOT.glob('.11E_staging_*'):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)
part11d_lock_hash = verify_sidecar(PART11D_LOCK, PART11D_LOCK_SHA, 'Part 11D lock')
part11d_checkpoint_hash = verify_sidecar(PART11D_CHECKPOINT, PART11D_CHECKPOINT_SHA, 'Part 11D checkpoint')
part11d_lock = json.loads(PART11D_LOCK.read_text(encoding='utf-8'))
if part11d_lock.get('Status') != 'PART_11D_COMPLETED_READY_FOR_11E':
    raise AssertionError(f"Unexpected Part 11D status: {part11d_lock.get('Status')}")
if part11d_lock.get('ReadyForPart11E') is not True:
    raise AssertionError('Part 11D lock does not authorise Part 11E')
if part11d_lock.get('CandidateSegments', {}).get('MustRecalculateWithinEachTrainingFold') is not True:
    raise AssertionError('Part 11D does not require fold-specific scope recalculation')
weekly_hash = sha256_file(WEEKLY_DATASET)
locked_weekly_hash = part11d_lock.get('Part11C', {}).get('WeeklyDatasetSHA256')
if weekly_hash != locked_weekly_hash:
    raise AssertionError('Weekly dataset hash no longer matches the Part 11D lock')
manifest_hash = sha256_file(PART11D_MANIFEST)
locked_manifest_hash = part11d_lock.get('OutputHashManifest', {}).get('SHA256')
if manifest_hash != locked_manifest_hash:
    raise AssertionError('Part 11D output manifest hash changed')
manifest11d = pd.read_csv(PART11D_MANIFEST)
required_manifest_columns = {'RelativePath', 'SHA256'}
if not required_manifest_columns.issubset(manifest11d.columns):
    raise AssertionError('Part 11D manifest schema is invalid')
membership_relative = str(PART11D_MEMBERSHIP.relative_to(EXT_ROOT))
membership_manifest_rows = manifest11d.loc[manifest11d['RelativePath'].astype(str) == membership_relative]
if len(membership_manifest_rows) != 1:
    raise AssertionError('Part 11D membership file is not uniquely recorded in its manifest')
membership_hash = sha256_file(PART11D_MEMBERSHIP)
if membership_hash != str(membership_manifest_rows.iloc[0]['SHA256']):
    raise AssertionError('Part 11D scope-membership file hash changed')
copy_audit = pd.read_csv(PART11B_COPY_AUDIT)
required_copy_columns = {'Role', 'DestinationRelativePath', 'SourceSHA256', 'HashMatch'}
if not required_copy_columns.issubset(copy_audit.columns):
    raise AssertionError('Part 11B source-copy audit schema is invalid')
copy_audit['HashMatch'] = bool_series(copy_audit['HashMatch'], 'HashMatch')
daily_hash_records = []
for role, path in [('LOCKED_DAILY_PRODUCT_PREDICTIONS', DAILY_PRODUCT_PREDICTIONS), ('LOCKED_DAILY_AGGREGATE_PREDICTIONS', DAILY_AGGREGATE_PREDICTIONS)]:
    row = copy_audit.loc[copy_audit['Role'].astype(str) == role]
    if len(row) != 1:
        raise AssertionError(f'Part 11B copy audit does not uniquely identify {role}')
    expected_relative = str(path.relative_to(EXT_ROOT))
    if str(row.iloc[0]['DestinationRelativePath']) != expected_relative:
        raise AssertionError(f'Part 11B destination mismatch for {role}')
    actual_hash = sha256_file(path)
    expected_hash = str(row.iloc[0]['SourceSHA256'])
    passed = bool(row.iloc[0]['HashMatch']) and actual_hash == expected_hash
    if not passed:
        raise AssertionError(f'Locked snapshot hash mismatch for {role}')
    daily_hash_records.append({'Role': role, 'Path': str(path), 'ExpectedSHA256': expected_hash, 'ActualSHA256': actual_hash, 'Passed': passed})
daily_hash_audit = pd.DataFrame(daily_hash_records)
part10c_lock_hash = verify_sidecar(PART10C_LOCK, PART10C_LOCK_SHA, 'Part 10C final-evaluation lock')
if part10c_lock_hash != EXPECTED_PART10C_LOCK_SHA256:
    raise AssertionError('Part 10C final-evaluation lock does not match the authoritative SHA-256.')
part10c_lock = json.loads(PART10C_LOCK.read_text(encoding='utf-8'))
if part10c_lock.get('Status') != EXPECTED_PART10C_STATUS:
    raise AssertionError(f"Unexpected Part 10C lock status: {part10c_lock.get('Status')}")
if part10c_lock.get('ReadyForFinalReporting') is not True:
    raise AssertionError('Part 10C lock is not report-ready')
scored_output_records = []
for role, path, lock_key in [('SCORED_DAILY_PRODUCT', SCORED_DAILY_PRODUCT, 'Product'), ('SCORED_DAILY_AGGREGATE', SCORED_DAILY_AGGREGATE, 'Aggregate')]:
    locked = part10c_lock.get('ScoredOutputs', {}).get(lock_key, {})
    if Path(str(locked.get('Path', ''))) != path:
        raise AssertionError(f'Part 10C lock path mismatch for {role}')
    actual_hash = sha256_file(path)
    expected_hash = str(locked.get('SHA256', ''))
    passed = len(expected_hash) == 64 and actual_hash == expected_hash
    if not passed:
        raise AssertionError(f'Part 10C scored-output hash mismatch for {role}')
    scored_output_records.append({'Role': role, 'Path': str(path), 'ExpectedSHA256': expected_hash, 'ActualSHA256': actual_hash, 'Passed': passed})
scored_output_hash_audit = pd.DataFrame(scored_output_records)
weekly = pd.read_csv(WEEKLY_DATASET, low_memory=False)
required_weekly_columns = {'WeekStartDate', 'WeekEndDate', 'WeekID', 'CanonicalProductID', 'CanonicalProductName', 'WeeklyNormalDemand', 'WeeklyBulkDemand', 'WeeklyTotalDemand', 'ContainsOpenedDailyHoldoutRows', 'WeeklyTuningEligible'}
missing_weekly = sorted(required_weekly_columns - set(weekly.columns))
if missing_weekly:
    raise AssertionError(f'Weekly dataset missing columns: {missing_weekly}')
for column in ['WeekStartDate', 'WeekEndDate']:
    weekly[column] = pd.to_datetime(weekly[column], errors='raise')
for column in ['ContainsOpenedDailyHoldoutRows', 'WeeklyTuningEligible']:
    weekly[column] = bool_series(weekly[column], column)
for column in ['WeeklyNormalDemand', 'WeeklyBulkDemand', 'WeeklyTotalDemand']:
    weekly[column] = pd.to_numeric(weekly[column], errors='raise').astype(float)
    if weekly[column].isna().any() or (weekly[column] < 0).any():
        raise AssertionError(f'Invalid values in {column}')
weekly['CanonicalProductID'] = weekly['CanonicalProductID'].astype(str)
if weekly.duplicated(['WeekStartDate', 'CanonicalProductID']).any():
    raise AssertionError('Duplicate product-week keys found')
if not np.allclose(weekly['WeeklyTotalDemand'], weekly['WeeklyNormalDemand'] + weekly['WeeklyBulkDemand']):
    raise AssertionError('Weekly demand components do not reconcile')
if (weekly['ContainsOpenedDailyHoldoutRows'] & weekly['WeeklyTuningEligible']).any():
    raise AssertionError('Opened holdout rows are marked tuning-eligible')
eligible = weekly.loc[weekly['WeeklyTuningEligible'] & ~weekly['ContainsOpenedDailyHoldoutRows']].copy()
eligible_weeks = sorted(eligible['WeekStartDate'].drop_duplicates())
if len(eligible_weeks) <= MIN_TRAINING_WEEKS:
    raise AssertionError('Insufficient pre-holdout weeks for baseline evaluation')
forecast_weeks = eligible_weeks[MIN_TRAINING_WEEKS:]
validation_panel = eligible.loc[eligible['WeekStartDate'].isin(forecast_weeks)].copy()
if validation_panel.empty:
    raise AssertionError('No pre-holdout validation rows were created')
weekly_sorted = weekly.sort_values(['CanonicalProductID', 'WeekStartDate'], kind='mergesort').reset_index(drop=True)
weekly_sorted['PriorProductContextCount'] = weekly_sorted.groupby('CanonicalProductID').cumcount()
prediction_columns = {}
for target_id, target_column in TARGETS.items():
    group = weekly_sorted.groupby('CanonicalProductID', sort=False)[target_column]
    shifted = group.shift(1)
    prediction_columns[target_id, 'ZERO'] = pd.Series(np.zeros(len(weekly_sorted), dtype=float), index=weekly_sorted.index)
    prediction_columns[target_id, 'NAIVE_LAST_CONTEXT'] = shifted
    prediction_columns[target_id, 'MEAN_LAST_4_CONTEXTS'] = group.transform(lambda series: series.shift(1).rolling(window=4, min_periods=1).mean())
    prediction_columns[target_id, 'MEDIAN_LAST_4_CONTEXTS'] = group.transform(lambda series: series.shift(1).rolling(window=4, min_periods=1).median())
    prediction_columns[target_id, 'MEAN_LAST_8_CONTEXTS'] = group.transform(lambda series: series.shift(1).rolling(window=8, min_periods=1).mean())
    prediction_columns[target_id, 'EXPANDING_MEAN_CONTEXTS'] = group.transform(lambda series: series.shift(1).expanding(min_periods=1).mean())
for key, values in prediction_columns.items():
    if len(values) != len(weekly_sorted):
        raise AssertionError(f'Baseline calculation length mismatch: {key}')
    prediction_columns[key] = values.fillna(0.0).clip(lower=0.0).astype(float)
fold_membership_frames = []
fold_scope_summary_rows = []
for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
    training = eligible.loc[eligible['WeekStartDate'] < forecast_week]
    current = validation_panel.loc[validation_panel['WeekStartDate'] == forecast_week]
    training_weeks = int(training['WeekStartDate'].nunique())
    if training_weeks < MIN_TRAINING_WEEKS:
        raise AssertionError('A validation fold has less than the required history')
    candidate_ids = pd.DataFrame({'CanonicalProductID': sorted(set(training['CanonicalProductID']) | set(current['CanonicalProductID']))})
    training_totals = training.groupby('CanonicalProductID', as_index=False)['WeeklyNormalDemand'].sum().rename(columns={'WeeklyNormalDemand': 'TrainingNormalDemand'})
    training_totals = candidate_ids.merge(training_totals, on='CanonicalProductID', how='left', validate='one_to_one')
    training_totals['TrainingNormalDemand'] = training_totals['TrainingNormalDemand'].fillna(0.0)
    ranked = strict_scope_flags(training_totals)
    ranked['ForecastWeek'] = forecast_week
    ranked['FoldNumber'] = fold_number
    ranked['TrainingWeeks'] = training_weeks
    ranked['TrainingStartWeek'] = training['WeekStartDate'].min()
    ranked['TrainingEndWeek'] = training['WeekStartDate'].max()
    current_membership = current[['WeekStartDate', 'CanonicalProductID']].merge(ranked, on='CanonicalProductID', how='left', validate='one_to_one')
    if current_membership['TrainingNormalDemand'].isna().any():
        raise AssertionError('A validation product did not receive fold scope information')
    fold_membership_frames.append(current_membership)
    total_training_demand = float(ranked['TrainingNormalDemand'].sum())
    for threshold in SCOPE_THRESHOLDS:
        label = int(threshold * 100)
        flag = f'InFoldStrict{label}PctScope'
        selected = ranked.loc[ranked[flag]]
        coverage = float(selected['TrainingNormalDemand'].sum() / total_training_demand)
        fold_scope_summary_rows.append({'FoldNumber': fold_number, 'ForecastWeek': forecast_week, 'TrainingWeeks': training_weeks, 'TrainingStartWeek': training['WeekStartDate'].min(), 'TrainingEndWeek': training['WeekStartDate'].max(), 'CoverageTargetPercentage': float(label), 'SelectedProducts': int(selected['CanonicalProductID'].nunique()), 'TrainingNormalDemand': total_training_demand, 'SelectedNormalDemand': float(selected['TrainingNormalDemand'].sum()), 'ActualCoveragePercentage': float(100.0 * coverage)})
fold_membership = pd.concat(fold_membership_frames, ignore_index=True)
fold_scope_summary = pd.DataFrame(fold_scope_summary_rows)
fold_membership = fold_membership.rename(columns={'WeekStartDate': 'ValidationWeek'})
if fold_membership.duplicated(['ValidationWeek', 'CanonicalProductID']).any():
    raise AssertionError('Duplicate fold-membership keys found')
base_columns = ['WeekStartDate', 'WeekEndDate', 'WeekID', 'CanonicalProductID', 'CanonicalProductName', 'WeeklyNormalDemand', 'WeeklyTotalDemand', 'PriorProductContextCount']
base = weekly_sorted.loc[weekly_sorted['WeekStartDate'].isin(forecast_weeks), base_columns].copy()
base = base.merge(fold_membership[['ValidationWeek', 'CanonicalProductID', 'FoldNumber', 'TrainingWeeks', 'TrainingStartWeek', 'TrainingEndWeek', 'TrainingNormalDemand', 'FoldNormalDemandRank', 'FoldCumulativeNormalDemandCoverage', 'InFoldStrict80PctScope', 'InFoldStrict90PctScope', 'InFoldStrict95PctScope']], left_on=['WeekStartDate', 'CanonicalProductID'], right_on=['ValidationWeek', 'CanonicalProductID'], how='left', validate='one_to_one').drop(columns=['ValidationWeek'])
if base['FoldNumber'].isna().any():
    raise AssertionError('Some baseline rows did not receive a fold number')
long_frames = []
for target_id, target_column in TARGETS.items():
    actual = weekly_sorted.loc[weekly_sorted['WeekStartDate'].isin(forecast_weeks), target_column].to_numpy(dtype=float)
    for method in BASELINE_METHODS:
        prediction = prediction_columns[target_id, method].loc[weekly_sorted['WeekStartDate'].isin(forecast_weeks)].to_numpy(dtype=float)
        frame = base.copy()
        frame['TargetID'] = target_id
        frame['TargetColumn'] = target_column
        frame['BaselineMethod'] = method
        frame['Actual'] = actual
        frame['Prediction'] = prediction
        frame['Error'] = prediction - actual
        frame['AbsoluteError'] = np.abs(frame['Error'])
        frame['SelectionEligible'] = True
        frame['EvaluationPurpose'] = 'PREHOLDOUT_CHRONOLOGICAL_BASELINE_SELECTION'
        long_frames.append(frame)
preholdout_predictions = pd.concat(long_frames, ignore_index=True)
if preholdout_predictions['Prediction'].isna().any():
    raise AssertionError('NaN baseline predictions were created')
if (preholdout_predictions['Prediction'] < 0).any():
    raise AssertionError('Negative baseline predictions were created')
if preholdout_predictions['WeekStartDate'].isin(weekly.loc[weekly['ContainsOpenedDailyHoldoutRows'], 'WeekStartDate']).any():
    raise AssertionError('Opened holdout rows entered pre-holdout baseline selection')
scope_definitions = {'ALL_PRODUCTS': pd.Series(True, index=preholdout_predictions.index), 'FOLD_STRICT_80_PERCENT': preholdout_predictions['InFoldStrict80PctScope'], 'FOLD_STRICT_90_PERCENT': preholdout_predictions['InFoldStrict90PctScope'], 'FOLD_STRICT_95_PERCENT': preholdout_predictions['InFoldStrict95PctScope']}
pooled_metric_rows = []
for (target_id, method), group in preholdout_predictions.groupby(['TargetID', 'BaselineMethod'], sort=True):
    for scope_id in scope_definitions:
        if scope_id == 'ALL_PRODUCTS':
            scoped = group
        else:
            flag_column = {'FOLD_STRICT_80_PERCENT': 'InFoldStrict80PctScope', 'FOLD_STRICT_90_PERCENT': 'InFoldStrict90PctScope', 'FOLD_STRICT_95_PERCENT': 'InFoldStrict95PctScope'}[scope_id]
            scoped = group.loc[group[flag_column]]
        metrics = metric_record(scoped['Actual'], scoped['Prediction'])
        pooled_metric_rows.append({'EvaluationPurpose': 'PREHOLDOUT_CHRONOLOGICAL_BASELINE_SELECTION', 'SelectionEligible': True, 'TargetID': target_id, 'ScopeID': scope_id, 'BaselineMethod': method, 'ForecastWeeks': int(scoped['WeekStartDate'].nunique()), 'Products': int(scoped['CanonicalProductID'].nunique()), **metrics})
preholdout_metrics = pd.DataFrame(pooled_metric_rows)
fold_metric_rows = []
for (fold_number, week_start, target_id, method), group in preholdout_predictions.groupby(['FoldNumber', 'WeekStartDate', 'TargetID', 'BaselineMethod'], sort=True):
    for scope_id, flag_column in [('ALL_PRODUCTS', None), ('FOLD_STRICT_80_PERCENT', 'InFoldStrict80PctScope'), ('FOLD_STRICT_90_PERCENT', 'InFoldStrict90PctScope'), ('FOLD_STRICT_95_PERCENT', 'InFoldStrict95PctScope')]:
        scoped = group if flag_column is None else group.loc[group[flag_column]]
        metrics = metric_record(scoped['Actual'], scoped['Prediction'])
        fold_metric_rows.append({'FoldNumber': int(fold_number), 'ForecastWeek': week_start, 'TargetID': target_id, 'ScopeID': scope_id, 'BaselineMethod': method, **metrics})
fold_metrics = pd.DataFrame(fold_metric_rows)
ranking = preholdout_metrics.copy()
ranking['MethodComplexityOrder'] = ranking['BaselineMethod'].map(METHOD_COMPLEXITY_ORDER)
ranking['AbsoluteTotalBias'] = ranking['TotalBias'].abs()
ranking = ranking.sort_values(['TargetID', 'ScopeID', 'WAPEPercentage', 'MAE', 'RMSE', 'AbsoluteTotalBias', 'MethodComplexityOrder', 'BaselineMethod'], kind='mergesort').reset_index(drop=True)
ranking['BaselineRank'] = ranking.groupby(['TargetID', 'ScopeID']).cumcount() + 1
ranking['IsReferenceBaseline'] = ranking['BaselineRank'] == 1
ranking['RankingRule'] = 'WAPE_ASC_THEN_MAE_RMSE_ABS_TOTAL_BIAS_COMPLEXITY_METHOD_ID'
daily_product = pd.read_csv(DAILY_PRODUCT_PREDICTIONS, low_memory=False)
daily_aggregate = pd.read_csv(DAILY_AGGREGATE_PREDICTIONS, low_memory=False)
scored_product = pd.read_csv(SCORED_DAILY_PRODUCT, low_memory=False)
scored_aggregate = pd.read_csv(SCORED_DAILY_AGGREGATE, low_memory=False)
required_daily_product_columns = {'Date', 'CanonicalProductID', 'CanonicalProductName', 'ProductMethodID', 'FinalProductPrediction'}
required_daily_aggregate_columns = {'Date', 'AggregateMethodID', 'AggregateForecast', 'ProductMethodBottomUpTotal'}
required_scored_product_columns = required_daily_product_columns | {'TotalDemand'}
required_scored_aggregate_columns = required_daily_aggregate_columns | {'ActualAggregateDemand'}
for frame, required_columns, label in [(daily_product, required_daily_product_columns, '10B2 product predictions'), (daily_aggregate, required_daily_aggregate_columns, '10B2 aggregate predictions'), (scored_product, required_scored_product_columns, '10C scored product predictions'), (scored_aggregate, required_scored_aggregate_columns, '10C scored aggregate predictions')]:
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise AssertionError(f'{label} schema is invalid. Missing: {missing}')
    frame['Date'] = pd.to_datetime(frame['Date'], errors='raise')
    frame['WeekStartDate'] = frame['Date'] - pd.to_timedelta(frame['Date'].dt.weekday, unit='D')
for frame in [daily_product, scored_product]:
    frame['CanonicalProductID'] = frame['CanonicalProductID'].astype(str)
    frame['FinalProductPrediction'] = pd.to_numeric(frame['FinalProductPrediction'], errors='raise').astype(float)
scored_product['TotalDemand'] = pd.to_numeric(scored_product['TotalDemand'], errors='raise').astype(float)
for frame in [daily_aggregate, scored_aggregate]:
    frame['AggregateForecast'] = pd.to_numeric(frame['AggregateForecast'], errors='raise').astype(float)
    frame['ProductMethodBottomUpTotal'] = pd.to_numeric(frame['ProductMethodBottomUpTotal'], errors='raise').astype(float)
scored_aggregate['ActualAggregateDemand'] = pd.to_numeric(scored_aggregate['ActualAggregateDemand'], errors='raise').astype(float)
if len(daily_product) != EXPECTED_DAILY_PRODUCT_ROWS:
    raise AssertionError('Unexpected locked daily product row count')
if len(scored_product) != EXPECTED_DAILY_PRODUCT_ROWS:
    raise AssertionError('Unexpected Part 10C scored product row count')
if daily_product['Date'].nunique() != EXPECTED_DAILY_DATES:
    raise AssertionError('Unexpected locked daily product date count')
if scored_product['Date'].nunique() != EXPECTED_DAILY_DATES:
    raise AssertionError('Unexpected Part 10C scored product date count')
if daily_product['CanonicalProductID'].nunique() != EXPECTED_DAILY_PRODUCTS:
    raise AssertionError('Unexpected locked daily product count')
if scored_product['CanonicalProductID'].nunique() != EXPECTED_DAILY_PRODUCTS:
    raise AssertionError('Unexpected Part 10C scored product count')
if len(daily_aggregate) != EXPECTED_DAILY_DATES or daily_aggregate['Date'].nunique() != EXPECTED_DAILY_DATES:
    raise AssertionError('Unexpected locked daily aggregate date count')
if len(scored_aggregate) != EXPECTED_DAILY_DATES or scored_aggregate['Date'].nunique() != EXPECTED_DAILY_DATES:
    raise AssertionError('Unexpected Part 10C scored aggregate date count')
if daily_product.duplicated(['Date', 'CanonicalProductID']).any():
    raise AssertionError('Duplicate 10B2 daily product prediction keys found')
if scored_product.duplicated(['Date', 'CanonicalProductID']).any():
    raise AssertionError('Duplicate 10C scored product keys found')
if daily_aggregate.duplicated(['Date']).any() or scored_aggregate.duplicated(['Date']).any():
    raise AssertionError('Duplicate daily aggregate dates found')
if (scored_product['TotalDemand'] < 0).any() or (scored_aggregate['ActualAggregateDemand'] < 0).any():
    raise AssertionError('Negative actual demand found in Part 10C scored outputs')
if (daily_product['FinalProductPrediction'] < 0).any() or (scored_product['FinalProductPrediction'] < 0).any():
    raise AssertionError('Negative locked daily product predictions found')
if (daily_aggregate['AggregateForecast'] < 0).any() or (scored_aggregate['AggregateForecast'] < 0).any():
    raise AssertionError('Negative locked daily aggregate predictions found')
product_methods = daily_product['ProductMethodID'].dropna().astype(str).unique()
aggregate_methods = daily_aggregate['AggregateMethodID'].dropna().astype(str).unique()
if len(product_methods) != 1 or product_methods[0] != 'BLEND_HURDLE_NAIVE5__H75':
    raise AssertionError(f'Unexpected locked daily product method: {product_methods}')
if len(aggregate_methods) != 1 or aggregate_methods[0] != 'BLEND_HURDLE_HIST_GB__H50':
    raise AssertionError(f'Unexpected locked daily aggregate method: {aggregate_methods}')
product_comparison = daily_product[['Date', 'CanonicalProductID', 'FinalProductPrediction', 'ProductMethodID']].merge(scored_product[['Date', 'CanonicalProductID', 'FinalProductPrediction', 'ProductMethodID']], on=['Date', 'CanonicalProductID'], how='outer', suffixes=('_10B2', '_10C'), indicator=True, validate='one_to_one')
if not (product_comparison['_merge'] == 'both').all():
    raise AssertionError('10B2 and 10C product prediction keys differ')
if not np.allclose(product_comparison['FinalProductPrediction_10B2'], product_comparison['FinalProductPrediction_10C'], atol=1e-12, rtol=1e-12):
    raise AssertionError('10B2 and 10C product prediction values differ')
if not (product_comparison['ProductMethodID_10B2'].astype(str) == product_comparison['ProductMethodID_10C'].astype(str)).all():
    raise AssertionError('10B2 and 10C product method IDs differ')
aggregate_comparison = daily_aggregate[['Date', 'AggregateForecast', 'ProductMethodBottomUpTotal', 'AggregateMethodID']].merge(scored_aggregate[['Date', 'AggregateForecast', 'ProductMethodBottomUpTotal', 'AggregateMethodID']], on='Date', how='outer', suffixes=('_10B2', '_10C'), indicator=True, validate='one_to_one')
if not (aggregate_comparison['_merge'] == 'both').all():
    raise AssertionError('10B2 and 10C aggregate prediction dates differ')
if not np.allclose(aggregate_comparison['AggregateForecast_10B2'], aggregate_comparison['AggregateForecast_10C'], atol=1e-12, rtol=1e-12):
    raise AssertionError('10B2 and 10C aggregate forecast values differ')
if not np.allclose(aggregate_comparison['ProductMethodBottomUpTotal_10B2'], aggregate_comparison['ProductMethodBottomUpTotal_10C'], atol=1e-12, rtol=1e-12):
    raise AssertionError('10B2 and 10C bottom-up forecast values differ')
holdout_actual = weekly.loc[weekly['ContainsOpenedDailyHoldoutRows']].copy()
if holdout_actual.empty:
    raise AssertionError('No opened holdout product-weeks were found')
if not set(scored_product['WeekStartDate']).issubset(set(holdout_actual['WeekStartDate'])):
    raise AssertionError('Part 10C scored dates extend outside the weekly holdout')
week_lookup = holdout_actual[['WeekStartDate', 'WeekEndDate', 'WeekID']].drop_duplicates('WeekStartDate')
product_weekly = scored_product.groupby(['WeekStartDate', 'CanonicalProductID'], as_index=False).agg(CanonicalProductName=('CanonicalProductName', 'first'), ScoredDailyRows=('Date', 'size'), ScoredDailyDates=('Date', 'nunique'), ActualScoringUniverseDemand=('TotalDemand', 'sum'), LockedDailyProductForecast=('FinalProductPrediction', 'sum'), ProductMethodID=('ProductMethodID', 'first')).merge(week_lookup, on='WeekStartDate', how='left', validate='many_to_one')
fixed_membership = pd.read_csv(PART11D_MEMBERSHIP, low_memory=False)
required_fixed_columns = {'CanonicalProductID', 'InStrict80PctScope', 'InStrict90PctScope', 'InStrict95PctScope'}
if not required_fixed_columns.issubset(fixed_membership.columns):
    raise AssertionError('Part 11D fixed scope membership schema is invalid')
fixed_membership['CanonicalProductID'] = fixed_membership['CanonicalProductID'].astype(str)
for column in ['InStrict80PctScope', 'InStrict90PctScope', 'InStrict95PctScope']:
    fixed_membership[column] = bool_series(fixed_membership[column], column)
product_weekly = product_weekly.merge(fixed_membership[list(required_fixed_columns)], on='CanonicalProductID', how='left', validate='many_to_one')
if product_weekly[['InStrict80PctScope', 'InStrict90PctScope', 'InStrict95PctScope']].isna().any().any():
    raise AssertionError('Some scored product-weeks did not receive pre-holdout scope membership')
product_weekly['Error'] = product_weekly['LockedDailyProductForecast'] - product_weekly['ActualScoringUniverseDemand']
product_weekly['AbsoluteError'] = product_weekly['Error'].abs()
product_weekly['SelectionEligible'] = False
product_weekly['EvaluationPurpose'] = 'OPENED_HOLDOUT_EXACT_LOCKED_DAILY_SCORING_UNIVERSE_DIAGNOSTIC_ONLY'
scoring_coverage = holdout_actual[['WeekStartDate', 'WeekEndDate', 'WeekID', 'CanonicalProductID', 'CanonicalProductName', 'WeeklyNormalDemand', 'WeeklyBulkDemand', 'WeeklyTotalDemand']].merge(product_weekly[['WeekStartDate', 'CanonicalProductID', 'ScoredDailyRows', 'ScoredDailyDates', 'ActualScoringUniverseDemand', 'LockedDailyProductForecast']], on=['WeekStartDate', 'CanonicalProductID'], how='left', validate='one_to_one')
scoring_coverage['InLockedDailyScoringUniverse'] = scoring_coverage['ScoredDailyRows'].notna()
scoring_coverage['ScoredDailyRows'] = scoring_coverage['ScoredDailyRows'].fillna(0).astype(int)
scoring_coverage['ScoredDailyDates'] = scoring_coverage['ScoredDailyDates'].fillna(0).astype(int)
scoring_coverage['ActualScoringUniverseDemand'] = scoring_coverage['ActualScoringUniverseDemand'].fillna(0.0)
scoring_coverage['LockedDailyProductForecast'] = scoring_coverage['LockedDailyProductForecast'].fillna(0.0)
scoring_coverage['OperationalMinusScoringUniverseDemand'] = scoring_coverage['WeeklyTotalDemand'] - scoring_coverage['ActualScoringUniverseDemand']
scoring_coverage['PositiveOperationalDemandOutsideScoringUniverse'] = ~scoring_coverage['InLockedDailyScoringUniverse'] & (scoring_coverage['WeeklyTotalDemand'] > 0)
scoring_coverage['OperationalAndScoringActualDiffer'] = ~np.isclose(scoring_coverage['WeeklyTotalDemand'], scoring_coverage['ActualScoringUniverseDemand'], atol=1e-12, rtol=1e-12)
scoring_coverage['DiagnosticScored'] = scoring_coverage['InLockedDailyScoringUniverse']
scoring_coverage['SelectionEligible'] = False
scoring_product_ids = set(scored_product['CanonicalProductID'].astype(str))
weekly_product_ids = set(weekly['CanonicalProductID'].astype(str))
products_outside_daily_scoring = sorted(weekly_product_ids - scoring_product_ids)
if len(products_outside_daily_scoring) != 9:
    raise AssertionError(f'Expected nine POST_EVALUATION_FORECAST_ONLY products outside the locked daily scoring universe, found {len(products_outside_daily_scoring)}')
aggregate_weekly = scored_aggregate.groupby('WeekStartDate', as_index=False).agg(ActualAggregateDemand=('ActualAggregateDemand', 'sum'), LockedDailyOfficialAggregateForecast=('AggregateForecast', 'sum'), LockedDailyProductBottomUpForecast=('ProductMethodBottomUpTotal', 'sum'), ScoredDailyDates=('Date', 'nunique'), AggregateMethodID=('AggregateMethodID', 'first')).merge(week_lookup, on='WeekStartDate', how='left', validate='one_to_one')
product_actual_by_week = product_weekly.groupby('WeekStartDate', as_index=False)['ActualScoringUniverseDemand'].sum().rename(columns={'ActualScoringUniverseDemand': 'ProductFileActualDemand'})
product_bottom_up_by_week = product_weekly.groupby('WeekStartDate', as_index=False)['LockedDailyProductForecast'].sum().rename(columns={'LockedDailyProductForecast': 'ProductFileBottomUpForecast'})
product_count_by_week = product_weekly.groupby('WeekStartDate', as_index=False).agg(ScoredProductWeekRows=('CanonicalProductID', 'size'), ScoredProducts=('CanonicalProductID', 'nunique'))
aggregate_weekly = aggregate_weekly.merge(product_actual_by_week, on='WeekStartDate', how='left', validate='one_to_one').merge(product_bottom_up_by_week, on='WeekStartDate', how='left', validate='one_to_one').merge(product_count_by_week, on='WeekStartDate', how='left', validate='one_to_one')
aggregate_weekly['ProductActualDifference'] = aggregate_weekly['ProductFileActualDemand'] - aggregate_weekly['ActualAggregateDemand']
aggregate_weekly['ProductBottomUpDifference'] = aggregate_weekly['ProductFileBottomUpForecast'] - aggregate_weekly['LockedDailyProductBottomUpForecast']
if not np.allclose(aggregate_weekly['ProductFileActualDemand'], aggregate_weekly['ActualAggregateDemand'], atol=1e-09):
    raise AssertionError('Product-level and aggregate-level scored actuals differ')
if not np.allclose(aggregate_weekly['ProductFileBottomUpForecast'], aggregate_weekly['LockedDailyProductBottomUpForecast'], atol=1e-09):
    raise AssertionError('Product-file and aggregate-file bottom-up forecasts differ')
aggregate_weekly['SelectionEligible'] = False
aggregate_weekly['EvaluationPurpose'] = 'OPENED_HOLDOUT_EXACT_LOCKED_DAILY_SCORING_UNIVERSE_DIAGNOSTIC_ONLY'
diagnostic_rows = []
for scope_id, flag_column in [('ALL_SCORED_PRODUCTS', None), ('PREHOLDOUT_FIXED_80_PERCENT', 'InStrict80PctScope'), ('PREHOLDOUT_FIXED_90_PERCENT', 'InStrict90PctScope'), ('PREHOLDOUT_FIXED_95_PERCENT', 'InStrict95PctScope')]:
    scoped = product_weekly if flag_column is None else product_weekly.loc[product_weekly[flag_column]]
    diagnostic_rows.append({'EvaluationLevel': 'LOCKED_DAILY_SCORING_CONTEXT_WEEK', 'EvaluationPurpose': 'OPENED_HOLDOUT_EXACT_LOCKED_DAILY_SCORING_UNIVERSE_DIAGNOSTIC_ONLY', 'SelectionEligible': False, 'TargetID': 'TOTAL_DEMAND_WITHIN_LOCKED_DAILY_SCORING_UNIVERSE', 'ScopeID': scope_id, 'BaselineMethod': 'SUM_LOCKED_DAILY_PRODUCT_FORECASTS', 'Weeks': int(scoped['WeekStartDate'].nunique()), 'Products': int(scoped['CanonicalProductID'].nunique()), **metric_record(scoped['ActualScoringUniverseDemand'], scoped['LockedDailyProductForecast'])})
for method, column in [('SUM_LOCKED_DAILY_OFFICIAL_AGGREGATE_FORECASTS', 'LockedDailyOfficialAggregateForecast'), ('SUM_LOCKED_DAILY_PRODUCT_BOTTOM_UP_FORECASTS', 'LockedDailyProductBottomUpForecast')]:
    diagnostic_rows.append({'EvaluationLevel': 'AGGREGATE_WEEK', 'EvaluationPurpose': 'OPENED_HOLDOUT_EXACT_LOCKED_DAILY_SCORING_UNIVERSE_DIAGNOSTIC_ONLY', 'SelectionEligible': False, 'TargetID': 'AGGREGATE_TOTAL_DEMAND_WITHIN_LOCKED_DAILY_SCORING_UNIVERSE', 'ScopeID': 'ALL_SCORED_PRODUCTS', 'BaselineMethod': method, 'Weeks': int(len(aggregate_weekly)), 'Products': int(aggregate_weekly['ScoredProducts'].max()), **metric_record(aggregate_weekly['ActualAggregateDemand'], aggregate_weekly[column])})
daily_diagnostic_metrics = pd.DataFrame(diagnostic_rows)
fold_scope_minimum_passed = bool((fold_scope_summary['ActualCoveragePercentage'] >= fold_scope_summary['CoverageTargetPercentage'] - 1e-10).all())
fold_scope_nested = True
for _, group in fold_membership.groupby('ValidationWeek'):
    fold_scope_nested &= bool(group.loc[group['InFoldStrict80PctScope'], 'InFoldStrict90PctScope'].all())
    fold_scope_nested &= bool(group.loc[group['InFoldStrict90PctScope'], 'InFoldStrict95PctScope'].all())
best_counts = ranking.loc[ranking['IsReferenceBaseline']].groupby(['TargetID', 'ScopeID']).size()
expected_ranking_groups = len(TARGETS) * 4
validation_rows = [{'Check': 'Part 11D lock and checkpoint verified', 'Expected': True, 'Actual': True, 'Passed': True}, {'Check': 'Weekly dataset hash matched Part 11D lock', 'Expected': locked_weekly_hash, 'Actual': weekly_hash, 'Passed': weekly_hash == locked_weekly_hash}, {'Check': 'Part 11D membership hash matched manifest', 'Expected': str(membership_manifest_rows.iloc[0]['SHA256']), 'Actual': membership_hash, 'Passed': membership_hash == str(membership_manifest_rows.iloc[0]['SHA256'])}, {'Check': 'Locked daily snapshot hashes verified', 'Expected': 2, 'Actual': int(daily_hash_audit['Passed'].sum()), 'Passed': bool(daily_hash_audit['Passed'].all())}, {'Check': 'Part 10C lock matched authoritative SHA-256', 'Expected': EXPECTED_PART10C_LOCK_SHA256, 'Actual': part10c_lock_hash, 'Passed': part10c_lock_hash == EXPECTED_PART10C_LOCK_SHA256}, {'Check': 'Part 10C scored-output hashes verified', 'Expected': 2, 'Actual': int(scored_output_hash_audit['Passed'].sum()), 'Passed': bool(scored_output_hash_audit['Passed'].all())}, {'Check': 'Minimum training weeks', 'Expected': MIN_TRAINING_WEEKS, 'Actual': int(fold_membership['TrainingWeeks'].min()), 'Passed': int(fold_membership['TrainingWeeks'].min()) >= MIN_TRAINING_WEEKS}, {'Check': 'Pre-holdout validation weeks', 'Expected': len(eligible_weeks) - MIN_TRAINING_WEEKS, 'Actual': int(preholdout_predictions['WeekStartDate'].nunique()), 'Passed': int(preholdout_predictions['WeekStartDate'].nunique()) == len(eligible_weeks) - MIN_TRAINING_WEEKS}, {'Check': 'Opened holdout rows used for baseline selection', 'Expected': 0, 'Actual': 0, 'Passed': True}, {'Check': 'Fold scopes reached 80/90/95 targets', 'Expected': True, 'Actual': fold_scope_minimum_passed, 'Passed': fold_scope_minimum_passed}, {'Check': 'Fold scopes nested 80 within 90 within 95', 'Expected': True, 'Actual': fold_scope_nested, 'Passed': fold_scope_nested}, {'Check': 'Pre-holdout baseline prediction rows', 'Expected': len(validation_panel) * len(TARGETS) * len(BASELINE_METHODS), 'Actual': len(preholdout_predictions), 'Passed': len(preholdout_predictions) == len(validation_panel) * len(TARGETS) * len(BASELINE_METHODS)}, {'Check': 'Non-finite or negative pre-holdout predictions', 'Expected': 0, 'Actual': int((~np.isfinite(preholdout_predictions['Prediction'])).sum()) + int((preholdout_predictions['Prediction'] < 0).sum()), 'Passed': bool(np.isfinite(preholdout_predictions['Prediction']).all() and (preholdout_predictions['Prediction'] >= 0).all())}, {'Check': 'Exactly one reference baseline per target and scope', 'Expected': expected_ranking_groups, 'Actual': int((best_counts == 1).sum()), 'Passed': len(best_counts) == expected_ranking_groups and bool((best_counts == 1).all())}, {'Check': 'Locked daily product rows', 'Expected': EXPECTED_DAILY_PRODUCT_ROWS, 'Actual': len(daily_product), 'Passed': len(daily_product) == EXPECTED_DAILY_PRODUCT_ROWS}, {'Check': 'Locked daily prediction dates', 'Expected': EXPECTED_DAILY_DATES, 'Actual': daily_product['Date'].nunique(), 'Passed': daily_product['Date'].nunique() == EXPECTED_DAILY_DATES}, {'Check': '10B2 and 10C product predictions matched exactly', 'Expected': True, 'Actual': True, 'Passed': True}, {'Check': '10B2 and 10C aggregate predictions matched exactly', 'Expected': True, 'Actual': True, 'Passed': True}, {'Check': 'Products outside locked daily scoring universe', 'Expected': 9, 'Actual': len(products_outside_daily_scoring), 'Passed': len(products_outside_daily_scoring) == 9}, {'Check': 'Positive operational product-weeks outside daily scoring universe', 'Expected': 'informational', 'Actual': int(scoring_coverage['PositiveOperationalDemandOutsideScoringUniverse'].sum()), 'Passed': True}, {'Check': 'Locked daily scoring-universe actual total', 'Expected': EXPECTED_OPENED_HOLDOUT_TOTAL, 'Actual': float(product_weekly['ActualScoringUniverseDemand'].sum()), 'Passed': math.isclose(float(product_weekly['ActualScoringUniverseDemand'].sum()), EXPECTED_OPENED_HOLDOUT_TOTAL, abs_tol=1e-09)}, {'Check': 'Locked aggregate scored actual total', 'Expected': EXPECTED_OPENED_HOLDOUT_TOTAL, 'Actual': float(aggregate_weekly['ActualAggregateDemand'].sum()), 'Passed': math.isclose(float(aggregate_weekly['ActualAggregateDemand'].sum()), EXPECTED_OPENED_HOLDOUT_TOTAL, abs_tol=1e-09)}, {'Check': 'Locked daily product forecast total', 'Expected': EXPECTED_LOCKED_PRODUCT_FORECAST_TOTAL, 'Actual': float(product_weekly['LockedDailyProductForecast'].sum()), 'Passed': math.isclose(float(product_weekly['LockedDailyProductForecast'].sum()), EXPECTED_LOCKED_PRODUCT_FORECAST_TOTAL, abs_tol=1e-06)}, {'Check': 'Locked daily official aggregate forecast total', 'Expected': EXPECTED_LOCKED_AGGREGATE_FORECAST_TOTAL, 'Actual': float(aggregate_weekly['LockedDailyOfficialAggregateForecast'].sum()), 'Passed': math.isclose(float(aggregate_weekly['LockedDailyOfficialAggregateForecast'].sum()), EXPECTED_LOCKED_AGGREGATE_FORECAST_TOTAL, abs_tol=1e-06)}, {'Check': 'Product-file and aggregate-file bottom-up forecasts matched', 'Expected': True, 'Actual': bool(np.allclose(aggregate_weekly['ProductFileBottomUpForecast'], aggregate_weekly['LockedDailyProductBottomUpForecast'], atol=1e-09)), 'Passed': bool(np.allclose(aggregate_weekly['ProductFileBottomUpForecast'], aggregate_weekly['LockedDailyProductBottomUpForecast'], atol=1e-09))}, {'Check': 'Opened holdout diagnostic rows selection-eligible', 'Expected': 0, 'Actual': int(daily_diagnostic_metrics['SelectionEligible'].sum()), 'Passed': not daily_diagnostic_metrics['SelectionEligible'].any()}, {'Check': 'Models fitted or weekly model features created', 'Expected': False, 'Actual': False, 'Passed': True}]
validation = pd.DataFrame(validation_rows)
if not validation['Passed'].all():
    raise AssertionError('Part 11E validation failed:\n' + validation.loc[~validation['Passed']].to_string(index=False))
design_contract = {'StepID': STEP_ID, 'CreatedLocal': NOW_LOCAL.isoformat(), 'PrimaryPurpose': 'Establish simple weekly reference baselines before feature engineering or model fitting.', 'PreholdoutSelection': {'SelectionEligible': True, 'MinimumTrainingWeeks': MIN_TRAINING_WEEKS, 'ForecastHorizon': 'ONE_WEEK_AHEAD_EXPANDING_ORIGIN', 'Targets': TARGETS, 'Methods': BASELINE_METHODS, 'ProductScopes': ['ALL_PRODUCTS', 'FOLD_STRICT_80_PERCENT', 'FOLD_STRICT_90_PERCENT', 'FOLD_STRICT_95_PERCENT'], 'ScopeTarget': 'WeeklyNormalDemand', 'ScopeRule': 'Recalculate minimum strict coverage scope from training history inside every fold.', 'RankingRule': 'WAPE ascending, then MAE, RMSE, absolute total bias, method complexity and method ID.', 'ForecastRounding': False, 'NegativePredictionRule': 'Clip at zero; all chosen baseline formulas are naturally non-negative.'}, 'OpenedHoldoutDiagnostic': {'SelectionEligible': False, 'Purpose': 'Sum the already locked daily forecasts by Monday-to-Sunday week for traceable comparison only.', 'ProductMethod': 'BLEND_HURDLE_NAIVE5__H75', 'AggregateMethod': 'BLEND_HURDLE_HIST_GB__H50', 'WeeklyTarget': 'Exact TotalDemand rows in the locked 4,150-row daily scoring universe', 'ActualSource': 'Already-scored Part 10C outputs; protected target vault not reopened', 'ProductsOutsideDailyScoringUniverse': 9, 'MissingPredictionsAreNotZeroFilledForScoring': True, 'MayTuneWeeklyChoices': False}, 'MetricDefinitions': {'MAE': 'mean(abs(prediction - actual))', 'RMSE': 'sqrt(mean((prediction - actual)^2))', 'WAPE': '100 * sum(abs(prediction - actual)) / sum(abs(actual)); NaN when denominator is zero', 'MeanBias': 'mean(prediction - actual)', 'TotalBias': 'sum(prediction - actual)'}}
stage_root = EXT_ROOT / f'.11E_staging_{uuid.uuid4().hex}'
stage_root.mkdir(parents=True, exist_ok=False)
try:
    output_frames = [(PREHOLDOUT_PREDICTIONS_PATH, preholdout_predictions), (PREHOLDOUT_METRICS_PATH, preholdout_metrics), (BASELINE_RANKING_PATH, ranking), (FOLD_SCOPE_MEMBERSHIP_PATH, fold_membership), (FOLD_SCOPE_SUMMARY_PATH, fold_scope_summary), (DAILY_PRODUCT_WEEKLY_PATH, product_weekly), (DAILY_AGGREGATE_WEEKLY_PATH, aggregate_weekly), (DAILY_DIAGNOSTIC_METRICS_PATH, daily_diagnostic_metrics), (DAILY_SCORING_COVERAGE_PATH, scoring_coverage), (FOLD_METRICS_PATH, fold_metrics), (VALIDATION_PATH, validation)]
    for path, frame in output_frames:
        stage_csv(stage_root, path, frame)
    stage_json(stage_root, DESIGN_PATH, design_contract)
    reference_rows = ranking.loc[ranking['IsReferenceBaseline']].copy()
    reference_summary_lines = []
    for row in reference_rows.itertuples(index=False):
        reference_summary_lines.append(f'- {row.TargetID} / {row.ScopeID}: {row.BaselineMethod}; WAPE {row.WAPEPercentage:.6f}%; MAE {row.MAE:.6f}; total bias {row.TotalBias:.6f}.')
    reference_summary = '\n'.join(reference_summary_lines)
    diagnostic_summary_lines = []
    for row in daily_diagnostic_metrics.itertuples(index=False):
        diagnostic_summary_lines.append(f'- {row.EvaluationLevel} / {row.ScopeID} / {row.BaselineMethod}: {row.Observations} observations; WAPE {row.WAPEPercentage:.6f}%; MAE {row.MAE:.6f}; total bias {row.TotalBias:.6f}; selection eligible = False.')
    diagnostic_summary = '\n'.join(diagnostic_summary_lines)
    decisions = '\n'.join([f'- Decision date: {NOW_LOCAL.date().isoformat()}.', f'- Minimum history before validation: {MIN_TRAINING_WEEKS} calendar weeks.', '- Forecast horizon: one week ahead with expanding chronological origin.', '- Product scopes are recalculated from prior WeeklyNormalDemand inside every fold.', '- Six simple baselines are evaluated for WeeklyNormalDemand and WeeklyTotalDemand.', '- Pre-holdout WAPE is the primary ranking metric; tie-breaks favour lower errors, lower absolute bias and simpler methods.', '- The locked March 2026 daily forecasts are summed weekly and scored only as a non-selection diagnostic.', '- Daily-to-weekly product scoring uses the already-scored Part 10C rows, not the full 227-product weekly panel.', '- Nine POST_EVALUATION_FORECAST_ONLY products are audited separately and are never assigned invented zero daily forecasts.', '- The earlier assertion treating positive demand outside the daily scoring universe as an error was removed.', '- No result from the opened March 2026 holdout may change scope, feature, model, hyperparameter or fallback choices.'])
    files = '\n'.join([f'- Pre-holdout predictions: `{PREHOLDOUT_PREDICTIONS_PATH}`', f'- Pre-holdout metrics: `{PREHOLDOUT_METRICS_PATH}`', f'- Baseline ranking: `{BASELINE_RANKING_PATH}`', f'- Fold scope membership: `{FOLD_SCOPE_MEMBERSHIP_PATH}`', f'- Fold scope summary: `{FOLD_SCOPE_SUMMARY_PATH}`', f'- Fold metrics: `{FOLD_METRICS_PATH}`', f'- Locked daily product-week diagnostic: `{DAILY_PRODUCT_WEEKLY_PATH}`', f'- Locked daily aggregate-week diagnostic: `{DAILY_AGGREGATE_WEEKLY_PATH}`', f'- Locked daily diagnostic metrics: `{DAILY_DIAGNOSTIC_METRICS_PATH}`', f'- Daily scoring-universe coverage audit: `{DAILY_SCORING_COVERAGE_PATH}`', f'- Design contract: `{DESIGN_PATH}`', f'- Validation: `{VALIDATION_PATH}`'])
    metrics_results = '\n'.join([f'- Eligible pre-holdout weeks: {len(eligible_weeks)}.', f'- Validation weeks after minimum history: {len(forecast_weeks)}.', f'- Validation product-week rows: {len(validation_panel):,}.', f'- Pre-holdout baseline prediction rows: {len(preholdout_predictions):,}.', f'- Exact locked daily scoring-context weeks used for diagnostic scoring: {len(product_weekly):,}.', f'- Products outside the locked daily scoring universe: {len(products_outside_daily_scoring)}.', f"- Positive operational product-weeks outside the daily scoring universe: {int(scoring_coverage['PositiveOperationalDemandOutsideScoringUniverse'].sum())}.", '- Reference baseline results:', reference_summary, '- Locked daily-to-weekly diagnostic results:', diagnostic_summary])
    for key, final_path in MEMORY_FILES.items():
        original = final_path.read_text(encoding='utf-8')
        if key == 'PROJECT_CONTEXT':
            updated = update_section(original, 'STEP_11E', 'Part 11E weekly baselines', 'Leakage-safe one-week-ahead weekly baselines were evaluated on pre-holdout history. The already locked daily forecasts were also summed by week as a diagnostic-only benchmark.')
        elif key == 'WORKFLOW':
            updated = update_section(original, 'STEP_11E', 'Part 11E workflow status', 'Part 11E is complete. Part 11F will design weekly features. Later models must beat appropriate pre-holdout weekly reference baselines.')
        elif key == 'DECISIONS':
            updated = update_section(original, 'STEP_11E', 'Part 11E decisions', decisions)
        elif key == 'FILES_AND_PATHS':
            updated = update_section(original, 'STEP_11E', 'Part 11E files', files)
        elif key == 'METRICS_AND_RESULTS':
            updated = update_section(original, 'STEP_11E', 'Part 11E metrics and results', metrics_results)
        elif key == 'CHAT_INDEX':
            row = f'| {NOW_LOCAL.isoformat()} | 11E | Weekly baselines and locked daily-to-weekly diagnostic | {STATUS} |'
            updated = original if row in original else original.rstrip() + '\n' + row + '\n'
        elif key == 'CURRENT_HANDOFF':
            updated = '\n'.join(['# Current Handoff', '', f'- Current completed step: {STEP_ID}', f'- Status: {STATUS}', f'- Updated local time: {NOW_LOCAL.isoformat()}', f'- Weekly input: {WEEKLY_DATASET}', f'- Baseline ranking: {BASELINE_RANKING_PATH}', f'- Baseline design contract: {DESIGN_PATH}', '- Weekly scope membership must continue to be recalculated within each training fold.', '- March 2026 daily-to-weekly results are diagnostic only and are not selection evidence.', '- Next step: 11F leakage-safe weekly feature design.', ''])
        else:
            raise KeyError(key)
        stage_text(stage_root, final_path, updated)
    step_text = f'# Step 11E — Weekly Baselines\n\n- **Step ID:** 11E\n- **Date and time:** {NOW_LOCAL.isoformat()}\n- **Status:** {STATUS}\n\n## User request\nCreate weekly baselines, including a weekly baseline produced by summing the locked daily forecasts.\n\n## Technical actions\n- Verified the Part 11D lock, checkpoint, weekly-dataset hash and scope-membership hash.\n- Verified the read-only daily product and aggregate prediction snapshots against the Part 11B copy audit.\n- Verified the Part 10C final-evaluation lock and already-scored product and aggregate outputs.\n- Confirmed that Part 10C retained the exact 10B2 prediction keys and values.\n- Used an expanding one-week-ahead evaluation after {MIN_TRAINING_WEEKS} minimum training weeks.\n- Recalculated strict 80%, 90% and 95% normal-demand scopes inside every fold.\n- Evaluated six simple baseline methods for WeeklyNormalDemand and WeeklyTotalDemand.\n- Ranked selection-eligible baselines using pre-holdout WAPE and deterministic tie-breaks.\n- Summed the locked daily product and official aggregate forecasts into Monday-to-Sunday weeks.\n- Aggregated product actuals only over the exact 4,150-row locked daily scoring universe.\n- Audited the nine POST_EVALUATION_FORECAST_ONLY products separately instead of assigning zero forecasts.\n- Scored the locked daily sums on the already opened March 2026 period only as diagnostic evidence.\n\n## Decisions\n{decisions}\n\n## Inputs\n- `{WEEKLY_DATASET}`\n- `{PART11D_LOCK}`\n- `{PART11D_MEMBERSHIP}`\n- `{DAILY_PRODUCT_PREDICTIONS}`\n- `{DAILY_AGGREGATE_PREDICTIONS}`\n- `{PART10C_LOCK}`\n- `{SCORED_DAILY_PRODUCT}`\n- `{SCORED_DAILY_AGGREGATE}`\n\n## Outputs\n{files}\n\n## Validation\nAll {len(validation)} Part 11E validation checks passed.\n\n## Errors encountered\n- The first Part 11E attempt failed because it assumed that every positive operational product-week must exist in the locked daily scoring universe.\n- Diagnosis showed that nine products belong to the separate POST_EVALUATION_FORECAST_ONLY cohort.\n- The correction uses hash-locked Part 10C scored rows for exact diagnostic actuals and records excluded operational rows in a separate coverage audit.\n\n## Results\n{metrics_results}\n\n## Limitations\n- The opened March 2026 diagnostic cannot be used to choose weekly methods.\n- The daily-sum product diagnostic targets TotalDemand only within the exact locked daily scoring universe, not the full 227-product operational panel.\n- Nine POST_EVALUATION_FORECAST_ONLY products do not have locked daily predictions and are reported only in the coverage audit.\n- Simple baselines use prior product contexts and do not yet include calendar, event or cross-product features.\n- A new untouched future period is still required for unbiased final weekly evaluation.\n\n## Next action\nPart 11F: design leakage-safe weekly predictors and freeze the weekly feature-generation contract before model fitting.\n'
    stage_text(stage_root, STEP_MEMORY_PATH, step_text)
    checkpoint = {'StepID': STEP_ID, 'Status': STATUS, 'CreatedUTC': NOW_UTC.isoformat(), 'CreatedLocal': NOW_LOCAL.isoformat(), 'Input': {'WeeklyDatasetPath': str(WEEKLY_DATASET), 'WeeklyDatasetSHA256': weekly_hash, 'Part11DLockSHA256': part11d_lock_hash, 'Part11DCheckpointSHA256': part11d_checkpoint_hash, 'Part11DMembershipSHA256': membership_hash, 'DailyPredictionHashes': daily_hash_records, 'Part10CFinalEvaluationLockSHA256': part10c_lock_hash, 'Part10CScoredOutputHashes': scored_output_records}, 'PreholdoutEvaluation': {'MinimumTrainingWeeks': MIN_TRAINING_WEEKS, 'EligibleWeeks': len(eligible_weeks), 'ValidationWeeks': len(forecast_weeks), 'ValidationProductWeekRows': len(validation_panel), 'PredictionRows': len(preholdout_predictions), 'Targets': list(TARGETS), 'Methods': BASELINE_METHODS, 'ScopeRule': 'RECALCULATED_INSIDE_EACH_TRAINING_FOLD', 'ReferenceBaselines': reference_rows.to_dict(orient='records')}, 'LockedDailyDiagnostic': {'SelectionEligible': False, 'ScoredProductWeekContexts': len(product_weekly), 'AggregateWeeks': len(aggregate_weekly), 'ActualTotal': float(product_weekly['ActualScoringUniverseDemand'].sum()), 'ProductForecastTotal': float(product_weekly['LockedDailyProductForecast'].sum()), 'ProductsOutsideDailyScoringUniverse': products_outside_daily_scoring, 'PositiveOperationalProductWeeksOutsideScoringUniverse': int(scoring_coverage['PositiveOperationalDemandOutsideScoringUniverse'].sum()), 'OfficialAggregateForecastTotal': float(aggregate_weekly['LockedDailyOfficialAggregateForecast'].sum()), 'Metrics': daily_diagnostic_metrics.to_dict(orient='records')}, 'Safety': {'ProtectedTargetVaultOpened': False, 'ProtectedTargetVaultCopied': False, 'AlreadyScoredPart10COutputsRead': True, 'OpenedDailyHoldoutUsedForWeeklySelection': False, 'OpenedDailyHoldoutUsedForDiagnosticScoring': True, 'ModelsFitted': False, 'WeeklyModelFeaturesCreated': False, 'BaselinePredictionsCreated': True, 'ForecastsRoundedBeforeScoring': False}, 'ReadyForPart11F': True, 'NextStep': '11F'}
    stage_json(stage_root, CHECKPOINT_PATH, checkpoint)
    checkpoint_hash = sha256_file(stage_path(stage_root, CHECKPOINT_PATH))
    stage_text(stage_root, CHECKPOINT_SHA_PATH, f'{checkpoint_hash}  {CHECKPOINT_PATH.name}\n')
    manifest_targets = [PREHOLDOUT_PREDICTIONS_PATH, PREHOLDOUT_METRICS_PATH, BASELINE_RANKING_PATH, FOLD_SCOPE_MEMBERSHIP_PATH, FOLD_SCOPE_SUMMARY_PATH, DAILY_PRODUCT_WEEKLY_PATH, DAILY_AGGREGATE_WEEKLY_PATH, DAILY_DIAGNOSTIC_METRICS_PATH, DAILY_SCORING_COVERAGE_PATH, FOLD_METRICS_PATH, DESIGN_PATH, VALIDATION_PATH, STEP_MEMORY_PATH, CHECKPOINT_PATH, CHECKPOINT_SHA_PATH, *MEMORY_FILES.values()]
    manifest = pd.DataFrame([{'RelativePath': str(path.relative_to(EXT_ROOT)), 'Bytes': stage_path(stage_root, path).stat().st_size, 'SHA256': sha256_file(stage_path(stage_root, path))} for path in manifest_targets]).sort_values('RelativePath')
    stage_csv(stage_root, MANIFEST_PATH, manifest)
    manifest_hash = sha256_file(stage_path(stage_root, MANIFEST_PATH))
    lock = {'StepID': STEP_ID, 'Status': STATUS, 'CreatedUTC': NOW_UTC.isoformat(), 'Part11D': {'LockSHA256': part11d_lock_hash, 'CheckpointSHA256': part11d_checkpoint_hash, 'MembershipSHA256': membership_hash, 'WeeklyDatasetSHA256': weekly_hash}, 'DesignContract': {'Path': str(DESIGN_PATH), 'SHA256': sha256_file(stage_path(stage_root, DESIGN_PATH))}, 'PreholdoutBaselineEvaluation': {'MinimumTrainingWeeks': MIN_TRAINING_WEEKS, 'ValidationWeeks': len(forecast_weeks), 'ValidationProductWeekRows': len(validation_panel), 'Targets': list(TARGETS), 'Methods': BASELINE_METHODS, 'ReferenceBaselines': reference_rows.to_dict(orient='records')}, 'LockedDailyWeeklyDiagnostic': {'SelectionEligible': False, 'ActualTotal': float(product_weekly['ActualScoringUniverseDemand'].sum()), 'ProductForecastTotal': float(product_weekly['LockedDailyProductForecast'].sum()), 'Part10CFinalEvaluationLockSHA256': part10c_lock_hash, 'ProductsOutsideDailyScoringUniverse': products_outside_daily_scoring, 'CoverageAuditPath': str(DAILY_SCORING_COVERAGE_PATH), 'OfficialAggregateForecastTotal': float(aggregate_weekly['LockedDailyOfficialAggregateForecast'].sum()), 'MetricsPath': str(DAILY_DIAGNOSTIC_METRICS_PATH), 'MetricsSHA256': sha256_file(stage_path(stage_root, DAILY_DIAGNOSTIC_METRICS_PATH))}, 'OutputHashManifest': {'Path': str(MANIFEST_PATH), 'SHA256': manifest_hash, 'FilesListed': len(manifest)}, 'Checkpoint': {'Path': str(CHECKPOINT_PATH), 'SHA256': checkpoint_hash}, 'SafetyAssertions': checkpoint['Safety'], 'ReadyForPart11F': True, 'NextStep': '11F'}
    stage_json(stage_root, LOCK_PATH, lock)
    lock_hash = sha256_file(stage_path(stage_root, LOCK_PATH))
    stage_text(stage_root, LOCK_SHA_PATH, f'{lock_hash}  {LOCK_PATH.name}\n')
    stage_text(stage_root, LOG_PATH, '\n'.join([f'Status: {STATUS}', f'Run local: {NOW_LOCAL.isoformat()}', f'Weekly input SHA256: {weekly_hash}', f'Eligible pre-holdout weeks: {len(eligible_weeks)}', f'Validation weeks: {len(forecast_weeks)}', f'Validation product-week rows: {len(validation_panel)}', f'Pre-holdout prediction rows: {len(preholdout_predictions)}', f'Opened holdout used for selection: False', f'Opened holdout used for diagnostic scoring: True', f'Checkpoint SHA256: {checkpoint_hash}', f'Part 11E lock SHA256: {lock_hash}', '']))
    staged_files = [path for path in stage_root.rglob('*') if path.is_file()]
    if not staged_files:
        raise AssertionError('Part 11E staging directory is empty')
    if any(('target_vault' in path.name.lower() or path.suffix.lower() == '.joblib' for path in staged_files)):
        raise PermissionError('Forbidden target-vault or model file detected in staging')
    for staged in sorted(staged_files):
        final = EXT_ROOT / staged.relative_to(stage_root)
        final.parent.mkdir(parents=True, exist_ok=True)
        if final.exists() and final not in MEMORY_FILES.values():
            raise FileExistsError(f'Refusing to overwrite Part 11E output: {final}')
        os.replace(staged, final)
    for path in NEW_OUTPUTS:
        make_read_only(path)
finally:
    if stage_root.exists():
        shutil.rmtree(stage_root)
print('=' * 100)
print('EDEN WEEKLY FORECASTING EXTENSION — PART 11E COMPLETE')
print('=' * 100)
print(f'Status: {STATUS}')
print(f'Extension root: {EXT_ROOT}')
print(f'Local time: {NOW_LOCAL.isoformat()}')
print('\nBASELINE DESIGN')
print(f'Minimum training weeks: {MIN_TRAINING_WEEKS}')
print(f'Eligible pre-holdout weeks: {len(eligible_weeks)}')
print(f'Chronological validation weeks: {len(forecast_weeks)}')
print(f'Validation product-week rows: {len(validation_panel)}')
print(f"Targets: {', '.join(TARGETS)}")
print(f"Methods: {', '.join(BASELINE_METHODS)}")
print('\nFOLD-SPECIFIC SCOPE SUMMARY')
print(fold_scope_summary.groupby('CoverageTargetPercentage', as_index=False).agg(Folds=('FoldNumber', 'nunique'), MinimumProducts=('SelectedProducts', 'min'), MedianProducts=('SelectedProducts', 'median'), MaximumProducts=('SelectedProducts', 'max'), MinimumCoverage=('ActualCoveragePercentage', 'min'), MaximumCoverage=('ActualCoveragePercentage', 'max')).to_string(index=False))
print('\nPRE-HOLDOUT REFERENCE BASELINES')
print(ranking.loc[ranking['IsReferenceBaseline']][['TargetID', 'ScopeID', 'BaselineMethod', 'Observations', 'ActualTotal', 'PredictedTotal', 'MAE', 'RMSE', 'WAPEPercentage', 'MeanBias', 'TotalBias']].to_string(index=False))
print('\nLOCKED DAILY-TO-WEEKLY DIAGNOSTIC — NOT SELECTION ELIGIBLE')
print(daily_diagnostic_metrics.to_string(index=False))
print('\nPART 11E VALIDATION')
print(validation.to_string(index=False))
print('\nCONTROL OUTPUTS')
print(f'- Pre-holdout predictions: {PREHOLDOUT_PREDICTIONS_PATH}')
print(f'- Pre-holdout metrics: {PREHOLDOUT_METRICS_PATH}')
print(f'- Baseline ranking: {BASELINE_RANKING_PATH}')
print(f'- Fold scope membership: {FOLD_SCOPE_MEMBERSHIP_PATH}')
print(f'- Fold scope summary: {FOLD_SCOPE_SUMMARY_PATH}')
print(f'- Fold metrics: {FOLD_METRICS_PATH}')
print(f'- Locked daily product-week baseline: {DAILY_PRODUCT_WEEKLY_PATH}')
print(f'- Locked daily aggregate-week baseline: {DAILY_AGGREGATE_WEEKLY_PATH}')
print(f'- Locked daily diagnostic metrics: {DAILY_DIAGNOSTIC_METRICS_PATH}')
print(f'- Daily scoring-universe coverage audit: {DAILY_SCORING_COVERAGE_PATH}')
print(f'- Design contract: {DESIGN_PATH}')
print(f'- Checkpoint: {CHECKPOINT_PATH}')
print(f'- Checkpoint SHA256: {checkpoint_hash}')
print(f'- Part 11E lock: {LOCK_PATH}')
print(f'- Part 11E lock SHA256: {lock_hash}')
print('\nSAFETY: protected target vault opened/copied False/False; opened March 2026 holdout used for weekly selection False; already-scored Part 10C outputs read True; target vault reopened False; opened holdout used for diagnostic scoring True; models fitted False; weekly model features created False; baseline predictions created True; forecasts rounded before scoring False.')
print('=' * 100)

EDEN WEEKLY FORECASTING EXTENSION — PART 11E COMPLETE
Status: PART_11E_COMPLETED_READY_FOR_11F
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-05T20:40:01.120209+01:00

BASELINE DESIGN
Minimum training weeks: 8
Eligible pre-holdout weeks: 47
Chronological validation weeks: 39
Validation product-week rows: 7212
Targets: WEEKLY_NORMAL_DEMAND, WEEKLY_TOTAL_DEMAND
Methods: ZERO, NAIVE_LAST_CONTEXT, MEAN_LAST_4_CONTEXTS, MEDIAN_LAST_4_CONTEXTS, MEAN_LAST_8_CONTEXTS, EXPANDING_MEAN_CONTEXTS

FOLD-SPECIFIC SCOPE SUMMARY
 CoverageTargetPercentage  Folds  MinimumProducts  MedianProducts  MaximumProducts  MinimumCoverage  MaximumCoverage
                     80.0     39               35            39.0               52        80.000000        80.666916
                     90.0     39               54            58.0               78        90.010281        90.380875
                     95.0     39               68            7

In [18]:
# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11F
# Leakage-safe weekly feature design and frozen feature-generation contract
# =============================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Paths and constants
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"
MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
FEATURE_ROOT = EXT_ROOT / "04_weekly_features"
BASELINE_ROOT = EXT_ROOT / "05_weekly_baselines"
VALIDATION_ROOT = EXT_ROOT / "07_weekly_validation"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
LOG_ROOT = EXT_ROOT / "12_logs"

WEEKLY_DATASET = (
    EXT_ROOT
    / "02_weekly_data"
    / "final"
    / "11C_weekly_dataset.csv"
)

PART11E_LOCK = (
    CHECKPOINT_ROOT
    / "11E_weekly_baselines_lock.json"
)

PART11E_LOCK_SHA = (
    CHECKPOINT_ROOT
    / "11E_weekly_baselines_lock.sha256"
)

PART11E_CHECKPOINT = (
    CHECKPOINT_ROOT
    / "11E_checkpoint.json"
)

PART11E_CHECKPOINT_SHA = (
    CHECKPOINT_ROOT
    / "11E_checkpoint.sha256"
)

PART11E_FOLD_MEMBERSHIP = (
    BASELINE_ROOT
    / "11E_fold_scope_membership.csv"
)

PART11E_DESIGN = (
    VALIDATION_ROOT
    / "11E_baseline_design_contract.json"
)

SELECTION_DATASET_PATH = (
    FEATURE_ROOT
    / "11F_model_selection_weekly_features.csv"
)

OPENED_HOLDOUT_FEATURES_PATH = (
    FEATURE_ROOT
    / "11F_opened_holdout_scoring_features.csv"
)

FEATURE_CONTRACT_PATH = (
    FEATURE_ROOT
    / "11F_feature_contract.csv"
)

GENERATION_CONTRACT_PATH = (
    FEATURE_ROOT
    / "11F_feature_generation_contract.json"
)

MISSINGNESS_PATH = (
    FEATURE_ROOT
    / "11F_feature_missingness_audit.csv"
)

LEAKAGE_AUDIT_PATH = (
    FEATURE_ROOT
    / "11F_feature_leakage_audit.csv"
)

SCOPE_RECONCILIATION_PATH = (
    FEATURE_ROOT
    / "11F_scope_reconciliation_audit.csv"
)

DATASET_SUMMARY_PATH = (
    FEATURE_ROOT
    / "11F_feature_dataset_summary.csv"
)

VALIDATION_PATH = (
    FEATURE_ROOT
    / "11F_feature_validation.csv"
)

MANIFEST_PATH = (
    FEATURE_ROOT
    / "11F_output_hash_manifest.csv"
)

STEP_MEMORY_PATH = (
    STEP_MEMORY_ROOT
    / "STEP_11F_WEEKLY_FEATURES.md"
)

CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "11F_checkpoint.json"
)

CHECKPOINT_SHA_PATH = (
    CHECKPOINT_ROOT
    / "11F_checkpoint.sha256"
)

LOCK_PATH = (
    CHECKPOINT_ROOT
    / "11F_weekly_feature_contract_lock.json"
)

LOCK_SHA_PATH = (
    CHECKPOINT_ROOT
    / "11F_weekly_feature_contract_lock.sha256"
)

LOG_PATH = (
    LOG_ROOT
    / "11F_weekly_feature_design_log.txt"
)

MEMORY_FILES = {
    "PROJECT_CONTEXT":
        MEMORY_ROOT / "PROJECT_CONTEXT.md",

    "WORKFLOW":
        MEMORY_ROOT / "WORKFLOW.md",

    "DECISIONS":
        MEMORY_ROOT / "DECISIONS.md",

    "FILES_AND_PATHS":
        MEMORY_ROOT / "FILES_AND_PATHS.md",

    "METRICS_AND_RESULTS":
        MEMORY_ROOT / "METRICS_AND_RESULTS.md",

    "CHAT_INDEX":
        MEMORY_ROOT / "CHAT_INDEX.md",

    "CURRENT_HANDOFF":
        MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

STEP_ID = "11F"

STATUS = (
    "PART_11F_COMPLETED_READY_FOR_11G"
)

NOW_UTC = datetime.now(
    timezone.utc
)

NOW_LOCAL = NOW_UTC.astimezone(
    ZoneInfo("Europe/Dublin")
)

MIN_TRAINING_WEEKS = 8

SCOPE_THRESHOLDS = (
    0.80,
    0.90,
    0.95,
)

NEW_OUTPUTS = [
    SELECTION_DATASET_PATH,
    OPENED_HOLDOUT_FEATURES_PATH,
    FEATURE_CONTRACT_PATH,
    GENERATION_CONTRACT_PATH,
    MISSINGNESS_PATH,
    LEAKAGE_AUDIT_PATH,
    SCOPE_RECONCILIATION_PATH,
    DATASET_SUMMARY_PATH,
    VALIDATION_PATH,
    MANIFEST_PATH,
    STEP_MEMORY_PATH,
    CHECKPOINT_PATH,
    CHECKPOINT_SHA_PATH,
    LOCK_PATH,
    LOCK_SHA_PATH,
    LOG_PATH,
]


# =============================================================================
# Helper functions
# =============================================================================

def sha256_file(
    path: Path,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def list_fingerprint(
    values: list[str],
) -> str:
    payload = json.dumps(
        values,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


def require_file(
    path: Path,
    label: str,
) -> None:
    if not path.is_file():

        raise FileNotFoundError(
            f"Missing required {label}:\n"
            f"{path}"
        )


def verify_sidecar(
    path: Path,
    sidecar: Path,
    label: str,
) -> str:
    require_file(
        path,
        label,
    )

    require_file(
        sidecar,
        f"{label} SHA-256 sidecar",
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )

    actual = sha256_file(
        path
    )

    if (
        len(expected) != 64
        or expected != actual
    ):

        raise AssertionError(
            f"{label} SHA-256 mismatch:\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


def atomic_text(
    path: Path,
    text: str,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}."
        f"{uuid.uuid4().hex}.tmp"
    )

    try:

        temporary.write_text(
            text,
            encoding="utf-8",
        )

        os.replace(
            temporary,
            path,
        )

    finally:

        if temporary.exists():

            temporary.unlink()


def atomic_csv(
    path: Path,
    frame: pd.DataFrame,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}."
        f"{uuid.uuid4().hex}.tmp"
    )

    try:

        frame.to_csv(
            temporary,
            index=False,
        )

        os.replace(
            temporary,
            path,
        )

    finally:

        if temporary.exists():

            temporary.unlink()


def atomic_json(
    path: Path,
    payload: dict,
) -> None:
    atomic_text(
        path,
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def bool_series(
    series: pd.Series,
    label: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(
        series
    ):

        return series.astype(
            bool
        )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    normalised = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    invalid = sorted(
        set(normalised)
        - set(mapping)
    )

    if invalid:

        raise ValueError(
            f"{label} has invalid "
            f"Boolean values: {invalid[:10]}"
        )

    return (
        normalised
        .map(mapping)
        .astype(bool)
    )


def update_section(
    text: str,
    marker: str,
    heading: str,
    body: str,
) -> str:
    start = (
        f"<!-- BEGIN {marker} -->"
    )

    end = (
        f"<!-- END {marker} -->"
    )

    section = (
        f"{start}\n"
        f"## {heading}\n\n"
        f"{body.rstrip()}\n"
        f"{end}"
    )

    if (
        start in text
        and end in text
    ):

        before = (
            text
            .split(
                start,
                1,
            )[0]
            .rstrip()
        )

        after = (
            text
            .split(
                end,
                1,
            )[1]
            .lstrip()
        )

        return (
            before
            + "\n\n"
            + section
            + (
                "\n\n" + after
                if after
                else ""
            )
            + "\n"
        )

    return (
        text.rstrip()
        + "\n\n"
        + section
        + "\n"
    )


def stage_path(
    stage_root: Path,
    final_path: Path,
) -> Path:
    return (
        stage_root
        / final_path.relative_to(
            EXT_ROOT
        )
    )


def stage_text(
    stage_root: Path,
    final_path: Path,
    text: str,
) -> None:
    atomic_text(
        stage_path(
            stage_root,
            final_path,
        ),
        text,
    )


def stage_csv(
    stage_root: Path,
    final_path: Path,
    frame: pd.DataFrame,
) -> None:
    atomic_csv(
        stage_path(
            stage_root,
            final_path,
        ),
        frame,
    )


def stage_json(
    stage_root: Path,
    final_path: Path,
    payload: dict,
) -> None:
    atomic_json(
        stage_path(
            stage_root,
            final_path,
        ),
        payload,
    )


def make_read_only(
    path: Path,
) -> None:
    if path.is_file():

        path.chmod(
            stat.S_IRUSR
            | stat.S_IRGRP
            | stat.S_IROTH
        )


def rolling_feature(
    grouped,
    window: int,
    operation: str,
    min_periods: int = 1,
) -> pd.Series:
    if operation == "mean":

        return grouped.transform(
            lambda series:
                series
                .shift(1)
                .rolling(
                    window,
                    min_periods=min_periods,
                )
                .mean()
        )

    if operation == "median":

        return grouped.transform(
            lambda series:
                series
                .shift(1)
                .rolling(
                    window,
                    min_periods=min_periods,
                )
                .median()
        )

    if operation == "std":

        return grouped.transform(
            lambda series:
                series
                .shift(1)
                .rolling(
                    window,
                    min_periods=min_periods,
                )
                .std(
                    ddof=0
                )
        )

    if operation == "min":

        return grouped.transform(
            lambda series:
                series
                .shift(1)
                .rolling(
                    window,
                    min_periods=min_periods,
                )
                .min()
        )

    if operation == "max":

        return grouped.transform(
            lambda series:
                series
                .shift(1)
                .rolling(
                    window,
                    min_periods=min_periods,
                )
                .max()
        )

    if operation == "sum":

        return grouped.transform(
            lambda series:
                series
                .shift(1)
                .rolling(
                    window,
                    min_periods=min_periods,
                )
                .sum()
        )

    raise ValueError(
        operation
    )


def weeks_since_previous_positive(
    series: pd.Series,
) -> pd.Series:
    positions = np.arange(
        len(series),
        dtype=float,
    )

    positive_positions = np.where(
        series.to_numpy(
            dtype=float
        )
        > 0,
        positions,
        np.nan,
    )

    previous_positive = (
        pd.Series(
            positive_positions,
            index=series.index,
        )
        .shift(1)
        .ffill()
    )

    return (
        pd.Series(
            positions,
            index=series.index,
        )
        - previous_positive
    )


def build_prior_scope_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    weeks = sorted(
        frame[
            "WeekStartDate"
        ].drop_duplicates()
    )

    for week in weeks:

        history = frame.loc[
            frame[
                "WeekStartDate"
            ]
            < week
        ]

        current_ids = (
            frame.loc[
                frame[
                    "WeekStartDate"
                ]
                == week,
                "CanonicalProductID",
            ]
            .astype(str)
            .drop_duplicates()
        )

        candidate_ids = pd.DataFrame(
            {
                "CanonicalProductID":
                    sorted(
                        set(
                            history[
                                "CanonicalProductID"
                            ].astype(str)
                        )
                        | set(
                            current_ids
                        )
                    )
            }
        )

        totals = (
            history
            .groupby(
                "CanonicalProductID",
                as_index=False,
            )[
                "WeeklyNormalDemand"
            ]
            .sum()
            .rename(
                columns={
                    "WeeklyNormalDemand":
                        "PriorNormalDemandTotal"
                }
            )
        )

        ranked = candidate_ids.merge(
            totals,
            on="CanonicalProductID",
            how="left",
            validate="one_to_one",
        )

        ranked[
            "PriorNormalDemandTotal"
        ] = ranked[
            "PriorNormalDemandTotal"
        ].fillna(
            0.0
        )

        ranked = (
            ranked
            .sort_values(
                [
                    "PriorNormalDemandTotal",
                    "CanonicalProductID",
                ],
                ascending=[
                    False,
                    True,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        ranked[
            "PriorNormalDemandRank"
        ] = np.arange(
            1,
            len(ranked) + 1,
        )

        total = float(
            ranked[
                "PriorNormalDemandTotal"
            ].sum()
        )

        ranked[
            "PriorScopeAvailable"
        ] = (
            total > 0
        )

        if total > 0:

            ranked[
                "PriorNormalDemandShare"
            ] = (
                ranked[
                    "PriorNormalDemandTotal"
                ]
                / total
            )

            ranked[
                "PriorCumulativeNormalDemandCoverage"
            ] = (
                ranked[
                    "PriorNormalDemandTotal"
                ].cumsum()
                / total
            )

            for threshold in SCOPE_THRESHOLDS:

                label = int(
                    threshold * 100
                )

                reached = ranked.index[
                    ranked[
                        "PriorCumulativeNormalDemandCoverage"
                    ]
                    >= threshold
                ]

                if reached.empty:

                    raise AssertionError(
                        "Prior scope did not reach "
                        f"{label}% for {week}"
                    )

                cutoff_rank = (
                    int(
                        reached[0]
                    )
                    + 1
                )

                ranked[
                    f"PriorStrict"
                    f"{label}PctScope"
                ] = (
                    ranked[
                        "PriorNormalDemandRank"
                    ]
                    <= cutoff_rank
                )

        else:

            ranked[
                "PriorNormalDemandShare"
            ] = np.nan

            ranked[
                "PriorCumulativeNormalDemandCoverage"
            ] = np.nan

            for threshold in SCOPE_THRESHOLDS:

                ranked[
                    f"PriorStrict"
                    f"{int(threshold * 100)}"
                    f"PctScope"
                ] = False

        ranked[
            "PriorCandidateProductCount"
        ] = len(
            ranked
        )

        ranked[
            "PriorNormalDemandRankFraction"
        ] = (
            ranked[
                "PriorNormalDemandRank"
            ]
            / ranked[
                "PriorCandidateProductCount"
            ]
        )

        ranked[
            "WeekStartDate"
        ] = week

        rows.append(
            ranked.loc[
                ranked[
                    "CanonicalProductID"
                ].isin(
                    set(
                        current_ids
                    )
                ),
                [
                    "WeekStartDate",
                    "CanonicalProductID",
                    "PriorNormalDemandTotal",
                    "PriorNormalDemandShare",
                    "PriorNormalDemandRank",
                    "PriorCandidateProductCount",
                    "PriorNormalDemandRankFraction",
                    "PriorCumulativeNormalDemandCoverage",
                    "PriorStrict80PctScope",
                    "PriorStrict90PctScope",
                    "PriorStrict95PctScope",
                    "PriorScopeAvailable",
                ],
            ]
        )

    result = pd.concat(
        rows,
        ignore_index=True,
    )

    if result.duplicated(
        [
            "WeekStartDate",
            "CanonicalProductID",
        ]
    ).any():

        raise AssertionError(
            "Duplicate prior-scope "
            "feature keys were created"
        )

    return result


def build_weekly_features(
    raw: pd.DataFrame,
) -> pd.DataFrame:
    data = (
        raw
        .copy()
        .sort_values(
            [
                "CanonicalProductID",
                "WeekStartDate",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    data[
        "CalendarYear"
    ] = (
        data[
            "WeekStartDate"
        ].dt.year
        .astype(int)
    )

    data[
        "CalendarYearIndex"
    ] = (
        data[
            "CalendarYear"
        ]
        - int(
            data[
                "CalendarYear"
            ].min()
        )
    )

    data[
        "CalendarMonth"
    ] = (
        data[
            "WeekStartDate"
        ].dt.month
        .astype(int)
    )

    data[
        "CalendarQuarter"
    ] = (
        data[
            "WeekStartDate"
        ].dt.quarter
        .astype(int)
    )

    iso_week = (
        data[
            "WeekStartDate"
        ]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    data[
        "WeekOfYearSin"
    ] = np.sin(
        2.0
        * np.pi
        * iso_week
        / 52.1775
    )

    data[
        "WeekOfYearCos"
    ] = np.cos(
        2.0
        * np.pi
        * iso_week
        / 52.1775
    )

    data[
        "MonthSin"
    ] = np.sin(
        2.0
        * np.pi
        * data[
            "CalendarMonth"
        ]
        / 12.0
    )

    data[
        "MonthCos"
    ] = np.cos(
        2.0
        * np.pi
        * data[
            "CalendarMonth"
        ]
        / 12.0
    )

    data[
        "PriorProductContextCount"
    ] = (
        data
        .groupby(
            "CanonicalProductID"
        )
        .cumcount()
    )

    normal_group = data.groupby(
        "CanonicalProductID",
        sort=False,
    )[
        "WeeklyNormalDemand"
    ]

    total_group = data.groupby(
        "CanonicalProductID",
        sort=False,
    )[
        "WeeklyTotalDemand"
    ]

    bulk_group = data.groupby(
        "CanonicalProductID",
        sort=False,
    )[
        "WeeklyBulkDemand"
    ]

    for lag in (
        1,
        2,
        3,
        4,
        8,
        13,
    ):

        data[
            f"NormalDemandLag{lag}"
        ] = normal_group.shift(
            lag
        )

        data[
            f"TotalDemandLag{lag}"
        ] = total_group.shift(
            lag
        )

    for lag in (
        1,
        4,
    ):

        data[
            f"BulkDemandLag{lag}"
        ] = bulk_group.shift(
            lag
        )

    for window in (
        4,
        8,
        13,
    ):

        data[
            f"NormalDemandMean{window}"
        ] = rolling_feature(
            normal_group,
            window,
            "mean",
        )

        data[
            f"NormalDemandStd{window}"
        ] = rolling_feature(
            normal_group,
            window,
            "std",
            min_periods=2,
        )

        data[
            f"TotalDemandMean{window}"
        ] = rolling_feature(
            total_group,
            window,
            "mean",
        )

        data[
            f"BulkDemandSum{window}"
        ] = rolling_feature(
            bulk_group,
            window,
            "sum",
        )

    for window in (
        4,
        8,
    ):

        data[
            f"NormalDemandMedian{window}"
        ] = rolling_feature(
            normal_group,
            window,
            "median",
        )

        data[
            f"NormalDemandMin{window}"
        ] = rolling_feature(
            normal_group,
            window,
            "min",
        )

        data[
            f"NormalDemandMax{window}"
        ] = rolling_feature(
            normal_group,
            window,
            "max",
        )

        data[
            f"TotalDemandStd{window}"
        ] = rolling_feature(
            total_group,
            window,
            "std",
            min_periods=2,
        )

    normal_positive = (
        data[
            "WeeklyNormalDemand"
        ]
        > 0
    ).astype(float)

    bulk_positive = (
        data[
            "WeeklyBulkDemand"
        ]
        > 0
    ).astype(float)

    normal_positive_group = (
        normal_positive
        .groupby(
            data[
                "CanonicalProductID"
            ],
            sort=False,
        )
    )

    bulk_positive_group = (
        bulk_positive
        .groupby(
            data[
                "CanonicalProductID"
            ],
            sort=False,
        )
    )

    for window in (
        4,
        8,
        13,
    ):

        data[
            f"NormalPositiveRate{window}"
        ] = rolling_feature(
            normal_positive_group,
            window,
            "mean",
        )

        data[
            f"BulkOccurrenceRate{window}"
        ] = rolling_feature(
            bulk_positive_group,
            window,
            "mean",
        )

    data[
        "PreviousNormalDemandPositive"
    ] = (
        data[
            "NormalDemandLag1"
        ]
        .fillna(
            0.0
        )
        > 0
    )

    data[
        "PreviousBulkDemandPositive"
    ] = (
        data[
            "BulkDemandLag1"
        ]
        .fillna(
            0.0
        )
        > 0
    )

    data[
        "WeeksSinceLastPositiveNormalDemand"
    ] = normal_group.transform(
        weeks_since_previous_positive
    )

    data[
        "NormalDemandExpandingMean"
    ] = normal_group.transform(
        lambda series:
            series
            .shift(1)
            .expanding(
                min_periods=1
            )
            .mean()
    )

    data[
        "NormalDemandExpandingStd"
    ] = normal_group.transform(
        lambda series:
            series
            .shift(1)
            .expanding(
                min_periods=2
            )
            .std(
                ddof=0
            )
    )

    data[
        "TotalDemandExpandingMean"
    ] = total_group.transform(
        lambda series:
            series
            .shift(1)
            .expanding(
                min_periods=1
            )
            .mean()
    )

    data[
        "TotalDemandExpandingStd"
    ] = total_group.transform(
        lambda series:
            series
            .shift(1)
            .expanding(
                min_periods=2
            )
            .std(
                ddof=0
            )
    )

    data[
        "NormalRecent4Mean"
    ] = data[
        "NormalDemandMean4"
    ]

    data[
        "NormalPrevious4Mean"
    ] = normal_group.transform(
        lambda series:
            series
            .shift(5)
            .rolling(
                4,
                min_periods=1,
            )
            .mean()
    )

    data[
        "NormalTrend4VsPrevious4"
    ] = (
        data[
            "NormalRecent4Mean"
        ]
        - data[
            "NormalPrevious4Mean"
        ]
    )

    weekly_global = (
        data
        .groupby(
            "WeekStartDate",
            as_index=False,
        )
        .agg(
            GlobalWeeklyNormalDemand=(
                "WeeklyNormalDemand",
                "sum",
            ),

            GlobalWeeklyBulkDemand=(
                "WeeklyBulkDemand",
                "sum",
            ),

            GlobalWeeklyTotalDemand=(
                "WeeklyTotalDemand",
                "sum",
            ),
        )
        .sort_values(
            "WeekStartDate"
        )
        .reset_index(
            drop=True
        )
    )

    for target in (
        "Normal",
        "Bulk",
        "Total",
    ):

        source = (
            f"GlobalWeekly"
            f"{target}Demand"
        )

        weekly_global[
            f"Global{target}"
            f"DemandLag1"
        ] = weekly_global[
            source
        ].shift(1)

    for target in (
        "Normal",
        "Total",
    ):

        source = (
            f"GlobalWeekly"
            f"{target}Demand"
        )

        for window in (
            4,
            8,
        ):

            weekly_global[
                f"Global{target}"
                f"DemandMean{window}"
            ] = (
                weekly_global[
                    source
                ]
                .shift(1)
                .rolling(
                    window,
                    min_periods=1,
                )
                .mean()
            )

    weekly_global[
        "GlobalBulkDemandMean4"
    ] = (
        weekly_global[
            "GlobalWeeklyBulkDemand"
        ]
        .shift(1)
        .rolling(
            4,
            min_periods=1,
        )
        .mean()
    )

    global_feature_columns = [
        column
        for column
        in weekly_global.columns
        if column
        not in {
            "GlobalWeeklyNormalDemand",
            "GlobalWeeklyBulkDemand",
            "GlobalWeeklyTotalDemand",
        }
    ]

    data = data.merge(
        weekly_global[
            global_feature_columns
        ],
        on="WeekStartDate",
        how="left",
        validate="many_to_one",
    )

    data[
        "ProductNormalShareLag1"
    ] = np.where(
        data[
            "GlobalNormalDemandLag1"
        ]
        > 0,
        (
            data[
                "NormalDemandLag1"
            ]
            / data[
                "GlobalNormalDemandLag1"
            ]
        ),
        np.nan,
    )

    prior_scope = (
        build_prior_scope_features(
            data
        )
    )

    data = data.merge(
        prior_scope,
        on=[
            "WeekStartDate",
            "CanonicalProductID",
        ],
        how="left",
        validate="one_to_one",
    )

    data[
        "PriorDemandSegment"
    ] = np.select(
        [
            ~data[
                "PriorScopeAvailable"
            ],

            data[
                "PriorStrict80PctScope"
            ],

            data[
                "PriorStrict95PctScope"
            ],
        ],
        [
            "NO_PRIOR_DEMAND_HISTORY",
            "HIGH_PRIOR_DEMAND",
            "MODERATE_PRIOR_DEMAND",
        ],
        default=(
            "OUTSIDE_INITIAL_SCOPE"
        ),
    )

    for count in (
        1,
        4,
        8,
        13,
    ):

        data[
            f"HasAtLeast"
            f"{count}PriorContexts"
        ] = (
            data[
                "PriorProductContextCount"
            ]
            >= count
        )

    return (
        data
        .sort_values(
            [
                "WeekStartDate",
                "CanonicalProductID",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# Verify prior state and overwrite guards
# =============================================================================

for directory, label in [
    (
        EXT_ROOT,
        "extension root",
    ),
    (
        MEMORY_ROOT,
        "memory root",
    ),
    (
        FEATURE_ROOT,
        "weekly feature directory",
    ),
    (
        CHECKPOINT_ROOT,
        "checkpoint directory",
    ),
    (
        LOG_ROOT,
        "log directory",
    ),
]:

    if not directory.is_dir():

        raise FileNotFoundError(
            f"Missing required {label}:\n"
            f"{directory}"
        )


for path, label in [
    (
        WEEKLY_DATASET,
        "Part 11C weekly dataset",
    ),
    (
        PART11E_FOLD_MEMBERSHIP,
        "Part 11E fold scope membership",
    ),
    (
        PART11E_DESIGN,
        "Part 11E baseline design contract",
    ),
]:

    require_file(
        path,
        label,
    )


for key, path in MEMORY_FILES.items():

    require_file(
        path,
        f"memory file {key}",
    )


if (
    LOCK_PATH.exists()
    or LOCK_SHA_PATH.exists()
):

    raise FileExistsError(
        "Part 11F overwrite lock triggered:\n"
        f"{LOCK_PATH}"
    )


existing_outputs = [
    path
    for path in NEW_OUTPUTS
    if path.exists()
]


if existing_outputs:

    raise FileExistsError(
        "Existing uncommitted Part 11F "
        "outputs found; no files changed:\n"
        + "\n".join(
            f"- {path}"
            for path in existing_outputs
        )
    )


for old_stage in EXT_ROOT.glob(
    ".11F_staging_*"
):

    if old_stage.is_dir():

        shutil.rmtree(
            old_stage
        )


part11e_lock_hash = verify_sidecar(
    PART11E_LOCK,
    PART11E_LOCK_SHA,
    "Part 11E lock",
)


part11e_checkpoint_hash = (
    verify_sidecar(
        PART11E_CHECKPOINT,
        PART11E_CHECKPOINT_SHA,
        "Part 11E checkpoint",
    )
)


part11e_lock = json.loads(
    PART11E_LOCK.read_text(
        encoding="utf-8"
    )
)


if (
    part11e_lock.get(
        "Status"
    )
    !=
    "PART_11E_COMPLETED_READY_FOR_11F"
):

    raise AssertionError(
        "Unexpected Part 11E status: "
        f"{part11e_lock.get('Status')}"
    )


if (
    part11e_lock.get(
        "ReadyForPart11F"
    )
    is not True
):

    raise AssertionError(
        "Part 11E lock does not "
        "authorise Part 11F"
    )


for key, expected in {
    "ProtectedTargetVaultOpened":
        False,

    "ProtectedTargetVaultCopied":
        False,

    "OpenedDailyHoldoutUsedForWeeklySelection":
        False,

    "ModelsFitted":
        False,

    "WeeklyModelFeaturesCreated":
        False,
}.items():

    actual = (
        part11e_lock
        .get(
            "SafetyAssertions",
            {},
        )
        .get(
            key
        )
    )

    if actual is not expected:

        raise AssertionError(
            "Invalid Part 11E safety "
            f"assertion {key}: {actual}"
        )


weekly_hash = sha256_file(
    WEEKLY_DATASET
)


locked_weekly_hash = (
    part11e_lock
    .get(
        "Part11D",
        {},
    )
    .get(
        "WeeklyDatasetSHA256"
    )
)


if (
    weekly_hash
    != locked_weekly_hash
):

    raise AssertionError(
        "Weekly dataset hash no longer "
        "matches the Part 11E lock"
    )


# =============================================================================
# Load and validate the weekly source
# =============================================================================

weekly = pd.read_csv(
    WEEKLY_DATASET,
    low_memory=False,
)


required_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "ISOYear",
    "ISOWeek",
    "CanonicalProductID",
    "CanonicalProductName",
    "AvailableOperatingDaysInWeek",
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
    "IsDatasetBoundaryPartialWeek",
    "ContainsOpenedDailyHoldoutRows",
    "WeeklyTuningEligible",
}


missing = sorted(
    required_columns
    - set(
        weekly.columns
    )
)


if missing:

    raise AssertionError(
        "Weekly dataset missing "
        f"required columns: {missing}"
    )


for column in [
    "WeekStartDate",
    "WeekEndDate",
]:

    weekly[
        column
    ] = pd.to_datetime(
        weekly[
            column
        ],
        errors="raise",
    )


for column in [
    "IsDatasetBoundaryPartialWeek",
    "ContainsOpenedDailyHoldoutRows",
    "WeeklyTuningEligible",
]:

    weekly[
        column
    ] = bool_series(
        weekly[
            column
        ],
        column,
    )


for column in [
    "AvailableOperatingDaysInWeek",
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
]:

    weekly[
        column
    ] = pd.to_numeric(
        weekly[
            column
        ],
        errors="raise",
    )


weekly[
    "CanonicalProductID"
] = weekly[
    "CanonicalProductID"
].astype(str)


if weekly.duplicated(
    [
        "WeekStartDate",
        "CanonicalProductID",
    ]
).any():

    raise AssertionError(
        "Duplicate product-week "
        "keys found"
    )


if not np.allclose(
    weekly[
        "WeeklyTotalDemand"
    ],
    (
        weekly[
            "WeeklyNormalDemand"
        ]
        + weekly[
            "WeeklyBulkDemand"
        ]
    ),
):

    raise AssertionError(
        "Weekly demand components "
        "do not reconcile"
    )


if (
    weekly[
        "ContainsOpenedDailyHoldoutRows"
    ]
    & weekly[
        "WeeklyTuningEligible"
    ]
).any():

    raise AssertionError(
        "Opened holdout rows are "
        "marked tuning-eligible"
    )


# =============================================================================
# Generate weekly features
# =============================================================================

features = build_weekly_features(
    weekly
)


identifier_columns = [
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
]


target_columns = [
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
]


governance_columns = [
    "ContainsOpenedDailyHoldoutRows",
    "WeeklyTuningEligible",
]


common_numeric_predictors = [
    "AvailableOperatingDaysInWeek",
    "CalendarYearIndex",
    "WeekOfYearSin",
    "WeekOfYearCos",
    "MonthSin",
    "MonthCos",
    "PriorProductContextCount",
    "NormalDemandLag1",
    "NormalDemandLag2",
    "NormalDemandLag3",
    "NormalDemandLag4",
    "NormalDemandLag8",
    "NormalDemandLag13",
    "NormalDemandMean4",
    "NormalDemandMean8",
    "NormalDemandMean13",
    "NormalDemandMedian4",
    "NormalDemandMedian8",
    "NormalDemandStd4",
    "NormalDemandStd8",
    "NormalDemandStd13",
    "NormalDemandMin4",
    "NormalDemandMin8",
    "NormalDemandMax4",
    "NormalDemandMax8",
    "NormalDemandExpandingMean",
    "NormalDemandExpandingStd",
    "NormalPositiveRate4",
    "NormalPositiveRate8",
    "NormalPositiveRate13",
    "WeeksSinceLastPositiveNormalDemand",
    "NormalRecent4Mean",
    "NormalPrevious4Mean",
    "NormalTrend4VsPrevious4",
    "GlobalNormalDemandLag1",
    "GlobalNormalDemandMean4",
    "GlobalNormalDemandMean8",
    "ProductNormalShareLag1",
    "PriorNormalDemandTotal",
    "PriorNormalDemandShare",
    "PriorNormalDemandRankFraction",
    "PriorCumulativeNormalDemandCoverage",
]


total_only_numeric_predictors = [
    "TotalDemandLag1",
    "TotalDemandLag2",
    "TotalDemandLag3",
    "TotalDemandLag4",
    "TotalDemandLag8",
    "TotalDemandLag13",
    "TotalDemandMean4",
    "TotalDemandMean8",
    "TotalDemandMean13",
    "TotalDemandStd4",
    "TotalDemandStd8",
    "TotalDemandExpandingMean",
    "TotalDemandExpandingStd",
    "BulkDemandLag1",
    "BulkDemandLag4",
    "BulkDemandSum4",
    "BulkDemandSum8",
    "BulkDemandSum13",
    "BulkOccurrenceRate4",
    "BulkOccurrenceRate8",
    "BulkOccurrenceRate13",
    "GlobalTotalDemandLag1",
    "GlobalTotalDemandMean4",
    "GlobalTotalDemandMean8",
    "GlobalBulkDemandLag1",
    "GlobalBulkDemandMean4",
]


binary_predictors = [
    "IsDatasetBoundaryPartialWeek",
    "HasAtLeast1PriorContexts",
    "HasAtLeast4PriorContexts",
    "HasAtLeast8PriorContexts",
    "HasAtLeast13PriorContexts",
    "PreviousNormalDemandPositive",
]


total_only_binary_predictors = [
    "PreviousBulkDemandPositive"
]


routing_columns = [
    "PriorNormalDemandRank",
    "PriorCandidateProductCount",
    "PriorStrict80PctScope",
    "PriorStrict90PctScope",
    "PriorStrict95PctScope",
    "PriorScopeAvailable",
    "PriorDemandSegment",
]


normal_predictors = (
    common_numeric_predictors
    + binary_predictors
)


total_predictors = (
    common_numeric_predictors
    + total_only_numeric_predictors
    + binary_predictors
    + total_only_binary_predictors
)


all_direct_predictors = list(
    dict.fromkeys(
        normal_predictors
        + total_predictors
    )
)


for column in (
    all_direct_predictors
    + routing_columns
):

    if column not in features.columns:

        raise AssertionError(
            "Generated feature missing "
            f"from dataset: {column}"
        )


forbidden_direct_predictors = {
    "CanonicalProductID",
    "CanonicalProductName",
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
    "PositiveDemandOperatingDays",
    "ZeroDemandOperatingDays",
    "NormalDemandOperatingDays",
    "BulkDemandOperatingDays",
    "ProductOperatingRowsInWeek",
    "ProductCoverageFractionOfOperatingDays",
    "IsPositiveDemandWeek",
    "IsBulkDemandWeek",
}


forbidden_found = sorted(
    set(
        all_direct_predictors
    )
    & forbidden_direct_predictors
)


if forbidden_found:

    raise AssertionError(
        "Forbidden direct predictors: "
        f"{forbidden_found}"
    )


selection_columns = (
    identifier_columns
    + target_columns
    + governance_columns
    + all_direct_predictors
    + routing_columns
)


selection_columns = list(
    dict.fromkeys(
        selection_columns
    )
)


holdout_columns = (
    identifier_columns
    + governance_columns
    + all_direct_predictors
    + routing_columns
)


holdout_columns = list(
    dict.fromkeys(
        holdout_columns
    )
)


selection_dataset = features.loc[
    (
        features[
            "WeeklyTuningEligible"
        ]
        & ~features[
            "ContainsOpenedDailyHoldoutRows"
        ]
    ),
    selection_columns,
].copy()


opened_holdout_features = features.loc[
    features[
        "ContainsOpenedDailyHoldoutRows"
    ],
    holdout_columns,
].copy()


if any(
    column
    in opened_holdout_features.columns
    for column in target_columns
):

    raise AssertionError(
        "Opened holdout scoring feature "
        "file contains target columns"
    )


if (
    selection_dataset.empty
    or opened_holdout_features.empty
):

    raise AssertionError(
        "Feature output views are "
        "unexpectedly empty"
    )


# =============================================================================
# Create the feature contract
# =============================================================================

feature_rows = []


def add_contract(
    names: list[str],
    role: str,
    dtype: str,
    source_timing: str,
    generation_rule: str,
    missing_policy: str,
    normal_use: bool,
    total_use: bool,
    direct: bool,
) -> None:

    for name in names:

        feature_rows.append(
            {
                "FeatureName":
                    name,

                "Role":
                    role,

                "DataType":
                    dtype,

                "SourceTiming":
                    source_timing,

                "GenerationRule":
                    generation_rule,

                "MissingValuePolicy":
                    missing_policy,

                "LeakageSafe":
                    True,

                "DirectPredictorPrimary":
                    direct,

                "UseForWeeklyNormalDemand":
                    normal_use,

                "UseForWeeklyTotalDemand":
                    total_use,
            }
        )


add_contract(
    identifier_columns,
    "IDENTIFIER_OR_LABEL",
    "mixed",
    "key_or_label",
    "copied",
    "not_applicable",
    False,
    False,
    False,
)


add_contract(
    target_columns,
    "TARGET",
    "numeric",
    "current_week_outcome",
    "copied_for_selection_rows_only",
    "not_applicable",
    False,
    False,
    False,
)


add_contract(
    governance_columns,
    "GOVERNANCE",
    "binary",
    "known_from_split_contract",
    "copied",
    "not_applicable",
    False,
    False,
    False,
)


known_ahead_numeric = [
    "AvailableOperatingDaysInWeek",
    "CalendarYearIndex",
    "WeekOfYearSin",
    "WeekOfYearCos",
    "MonthSin",
    "MonthCos",
]


add_contract(
    known_ahead_numeric,
    "NUMERIC_PREDICTOR",
    "numeric",
    "known_before_week_start",
    "calendar_or_operating_calendar",
    "none_expected",
    True,
    True,
    True,
)


common_history = [
    name
    for name
    in common_numeric_predictors
    if name not in set(
        known_ahead_numeric
    )
]


add_contract(
    common_history,
    "NUMERIC_PREDICTOR",
    "numeric",
    "strictly_before_forecast_week",
    "shifted_or_prior_history_only",
    (
        "retain_nan_then_impute_"
        "inside_training_fold"
    ),
    True,
    True,
    True,
)


add_contract(
    total_only_numeric_predictors,
    "NUMERIC_PREDICTOR",
    "numeric",
    "strictly_before_forecast_week",
    (
        "shifted_total_or_bulk_"
        "history_only"
    ),
    (
        "retain_nan_then_impute_"
        "inside_training_fold"
    ),
    False,
    True,
    True,
)


add_contract(
    binary_predictors,
    "BINARY_PREDICTOR",
    "binary",
    (
        "known_before_week_start_"
        "or_prior_history"
    ),
    "calendar_or_shifted_history",
    "no_missing_values",
    True,
    True,
    True,
)


add_contract(
    total_only_binary_predictors,
    "BINARY_PREDICTOR",
    "binary",
    "strictly_before_forecast_week",
    "shifted_bulk_history",
    "no_missing_values",
    False,
    True,
    True,
)


add_contract(
    routing_columns,
    "ROUTING_OR_DIAGNOSTIC",
    "mixed",
    "strictly_before_forecast_week",
    (
        "fold_equivalent_prior_"
        "normal_demand_ranking"
    ),
    (
        "undefined_only_before_"
        "any_history"
    ),
    False,
    False,
    False,
)


feature_contract = (
    pd.DataFrame(
        feature_rows
    )
    .drop_duplicates(
        "FeatureName",
        keep="last",
    )
)


normal_fingerprint = (
    list_fingerprint(
        normal_predictors
    )
)


total_fingerprint = (
    list_fingerprint(
        total_predictors
    )
)


routing_fingerprint = (
    list_fingerprint(
        routing_columns
    )
)


# =============================================================================
# Missingness audit
# =============================================================================

missingness_rows = []


for view_name, frame in [
    (
        "MODEL_SELECTION",
        selection_dataset,
    ),
    (
        "OPENED_HOLDOUT_FEATURE_ONLY",
        opened_holdout_features,
    ),
]:

    for column in (
        all_direct_predictors
        + routing_columns
    ):

        missing_count = int(
            frame[
                column
            ].isna().sum()
        )

        missingness_rows.append(
            {
                "DatasetView":
                    view_name,

                "FeatureName":
                    column,

                "Rows":
                    len(frame),

                "MissingCount":
                    missing_count,

                "MissingPercentage":
                    (
                        100.0
                        * missing_count
                        / len(frame)
                    ),

                "ExpectedHistoricalMissingness":
                    column
                    not in {
                        "AvailableOperatingDaysInWeek",
                        "CalendarYearIndex",
                        "WeekOfYearSin",
                        "WeekOfYearCos",
                        "MonthSin",
                        "MonthCos",
                        "PriorProductContextCount",
                        "IsDatasetBoundaryPartialWeek",
                        "HasAtLeast1PriorContexts",
                        "HasAtLeast4PriorContexts",
                        "HasAtLeast8PriorContexts",
                        "HasAtLeast13PriorContexts",
                        "PreviousNormalDemandPositive",
                        "PreviousBulkDemandPositive",
                        "PriorNormalDemandRank",
                        "PriorCandidateProductCount",
                        "PriorStrict80PctScope",
                        "PriorStrict90PctScope",
                        "PriorStrict95PctScope",
                        "PriorScopeAvailable",
                    },
            }
        )


missingness = pd.DataFrame(
    missingness_rows
)


unexpected_missing = missingness.loc[
    (
        ~missingness[
            "ExpectedHistoricalMissingness"
        ]
    )
    & (
        missingness[
            "MissingCount"
        ]
        > 0
    )
]


missingness_summary = (
    missingness
    .groupby(
        "DatasetView",
        as_index=False,
    )
    .agg(
        AuditedFeatures=(
            "FeatureName",
            "nunique",
        ),

        FeaturesWithMissingValues=(
            "MissingCount",
            lambda series:
                int(
                    (
                        series > 0
                    ).sum()
                ),
        ),

        MaximumMissingPercentage=(
            "MissingPercentage",
            "max",
        ),
    )
)


unexpected_summary = (
    unexpected_missing
    .groupby(
        "DatasetView"
    )
    .size()
    .rename(
        "UnexpectedMissingFeatures"
    )
)


missingness_summary = (
    missingness_summary
    .merge(
        unexpected_summary,
        on="DatasetView",
        how="left",
    )
)


missingness_summary[
    "UnexpectedMissingFeatures"
] = (
    missingness_summary[
        "UnexpectedMissingFeatures"
    ]
    .fillna(
        0
    )
    .astype(int)
)


if not unexpected_missing.empty:

    raise AssertionError(
        "Unexpected missing values in "
        "non-history features:\n"
        + unexpected_missing.to_string(
            index=False
        )
    )


# =============================================================================
# Reconcile dynamic scopes to Part 11E folds
# =============================================================================

fold_membership = pd.read_csv(
    PART11E_FOLD_MEMBERSHIP,
    low_memory=False,
)


fold_membership[
    "ValidationWeek"
] = pd.to_datetime(
    fold_membership[
        "ValidationWeek"
    ],
    errors="raise",
)


fold_membership[
    "CanonicalProductID"
] = fold_membership[
    "CanonicalProductID"
].astype(str)


for column in [
    "InFoldStrict80PctScope",
    "InFoldStrict90PctScope",
    "InFoldStrict95PctScope",
]:

    fold_membership[
        column
    ] = bool_series(
        fold_membership[
            column
        ],
        column,
    )


scope_compare = (
    fold_membership
    .merge(
        features[
            [
                "WeekStartDate",
                "CanonicalProductID",
                "PriorNormalDemandTotal",
                "PriorNormalDemandRank",
                "PriorCumulativeNormalDemandCoverage",
                "PriorStrict80PctScope",
                "PriorStrict90PctScope",
                "PriorStrict95PctScope",
            ]
        ],
        left_on=[
            "ValidationWeek",
            "CanonicalProductID",
        ],
        right_on=[
            "WeekStartDate",
            "CanonicalProductID",
        ],
        how="left",
        validate="one_to_one",
    )
)


if scope_compare[
    "WeekStartDate"
].isna().any():

    raise AssertionError(
        "Part 11E scope rows did not "
        "all match Part 11F features"
    )


scope_compare[
    "TrainingDemandMatched"
] = np.isclose(
    scope_compare[
        "TrainingNormalDemand"
    ],
    scope_compare[
        "PriorNormalDemandTotal"
    ],
    atol=1e-9,
)


scope_compare[
    "RankMatched"
] = (
    scope_compare[
        "FoldNormalDemandRank"
    ].astype(int)
    ==
    scope_compare[
        "PriorNormalDemandRank"
    ].astype(int)
)


scope_compare[
    "CoverageMatched"
] = np.isclose(
    scope_compare[
        "FoldCumulativeNormalDemandCoverage"
    ],
    scope_compare[
        "PriorCumulativeNormalDemandCoverage"
    ],
    atol=1e-12,
    equal_nan=True,
)


for label in (
    80,
    90,
    95,
):

    scope_compare[
        f"Scope{label}Matched"
    ] = (
        scope_compare[
            f"InFoldStrict"
            f"{label}PctScope"
        ]
        ==
        scope_compare[
            f"PriorStrict"
            f"{label}PctScope"
        ]
    )


scope_reconciliation = pd.DataFrame(
    [
        {
            "Check":
                "Training normal demand matched",

            "Rows":
                len(scope_compare),

            "Mismatches":
                int(
                    (
                        ~scope_compare[
                            "TrainingDemandMatched"
                        ]
                    ).sum()
                ),
        },
        {
            "Check":
                "Normal demand rank matched",

            "Rows":
                len(scope_compare),

            "Mismatches":
                int(
                    (
                        ~scope_compare[
                            "RankMatched"
                        ]
                    ).sum()
                ),
        },
        {
            "Check":
                "Cumulative coverage matched",

            "Rows":
                len(scope_compare),

            "Mismatches":
                int(
                    (
                        ~scope_compare[
                            "CoverageMatched"
                        ]
                    ).sum()
                ),
        },
        *[
            {
                "Check":
                    f"Strict {label}% "
                    "scope matched",

                "Rows":
                    len(scope_compare),

                "Mismatches":
                    int(
                        (
                            ~scope_compare[
                                f"Scope"
                                f"{label}Matched"
                            ]
                        ).sum()
                    ),
            }
            for label in (
                80,
                90,
                95,
            )
        ],
    ]
)


scope_reconciliation[
    "Passed"
] = (
    scope_reconciliation[
        "Mismatches"
    ]
    == 0
)


if not scope_reconciliation[
    "Passed"
].all():

    raise AssertionError(
        "Part 11F dynamic scopes did "
        "not reconcile to Part 11E:\n"
        + scope_reconciliation.to_string(
            index=False
        )
    )


# =============================================================================
# Target-perturbation leakage tests
# =============================================================================

validation_weeks = sorted(
    fold_membership[
        "ValidationWeek"
    ].drop_duplicates()
)


holdout_weeks = sorted(
    weekly.loc[
        weekly[
            "ContainsOpenedDailyHoldoutRows"
        ],
        "WeekStartDate",
    ].drop_duplicates()
)


representative_positions = sorted(
    set(
        [
            0,
            len(validation_weeks) // 3,
            (
                2
                * len(validation_weeks)
                // 3
            ),
            len(validation_weeks) - 1,
        ]
    )
)


audit_weeks = [
    validation_weeks[
        position
    ]
    for position
    in representative_positions
]


for week in [
    holdout_weeks[0],
    holdout_weeks[-1],
]:

    if week not in audit_weeks:

        audit_weeks.append(
            week
        )


audit_weeks = sorted(
    audit_weeks
)


leakage_rows = []


comparison_columns = (
    all_direct_predictors
    + routing_columns
)


base_indexed = features.set_index(
    [
        "WeekStartDate",
        "CanonicalProductID",
    ]
)


for audit_number, audit_week in enumerate(
    audit_weeks,
    start=1,
):

    perturbed = weekly.copy()

    perturb_mask = (
        perturbed[
            "WeekStartDate"
        ]
        >= audit_week
    )

    multiplier = float(
        100000
        + audit_number
        * 1000
    )

    perturbed.loc[
        perturb_mask,
        "WeeklyNormalDemand",
    ] = multiplier

    perturbed.loc[
        perturb_mask,
        "WeeklyBulkDemand",
    ] = (
        multiplier
        / 10.0
    )

    perturbed.loc[
        perturb_mask,
        "WeeklyTotalDemand",
    ] = (
        perturbed.loc[
            perturb_mask,
            "WeeklyNormalDemand",
        ]
        + perturbed.loc[
            perturb_mask,
            "WeeklyBulkDemand",
        ]
    )

    perturbed_features = (
        build_weekly_features(
            perturbed
        )
        .set_index(
            [
                "WeekStartDate",
                "CanonicalProductID",
            ]
        )
    )

    keys = (
        base_indexed
        .loc[
            audit_week
        ]
        .index
    )

    original_slice = base_indexed.loc[
        (
            audit_week,
            keys,
        ),
        comparison_columns,
    ]

    perturbed_slice = (
        perturbed_features.loc[
            (
                audit_week,
                keys,
            ),
            comparison_columns,
        ]
    )

    mismatch_columns = []

    for column in comparison_columns:

        left = original_slice[
            column
        ]

        right = perturbed_slice[
            column
        ]

        if (
            pd.api.types.is_bool_dtype(
                left
            )
            or pd.api.types.is_bool_dtype(
                right
            )
        ):

            matched = (
                left
                .astype("boolean")
                .equals(
                    right.astype(
                        "boolean"
                    )
                )
            )

        elif (
            pd.api.types.is_numeric_dtype(
                left
            )
            and pd.api.types.is_numeric_dtype(
                right
            )
        ):

            matched = np.allclose(
                pd.to_numeric(
                    left,
                    errors="coerce",
                ).to_numpy(
                    dtype=float
                ),
                pd.to_numeric(
                    right,
                    errors="coerce",
                ).to_numpy(
                    dtype=float
                ),
                atol=1e-12,
                rtol=1e-12,
                equal_nan=True,
            )

        else:

            matched = (
                left
                .fillna("<NA>")
                .astype(str)
                .equals(
                    right
                    .fillna("<NA>")
                    .astype(str)
                )
            )

        if not matched:

            mismatch_columns.append(
                column
            )

    leakage_rows.append(
        {
            "AuditWeek":
                audit_week.strftime(
                    "%Y-%m-%d"
                ),

            "RowsCompared":
                len(keys),

            "PredictorAndRoutingColumnsCompared":
                len(
                    comparison_columns
                ),

            "MismatchColumnCount":
                len(
                    mismatch_columns
                ),

            "MismatchColumns":
                ";".join(
                    mismatch_columns
                ),

            "Passed":
                len(
                    mismatch_columns
                )
                == 0,
        }
    )


leakage_audit = pd.DataFrame(
    leakage_rows
)


if not leakage_audit[
    "Passed"
].all():

    raise AssertionError(
        "Target-perturbation leakage "
        "test failed:\n"
        + leakage_audit.to_string(
            index=False
        )
    )


# =============================================================================
# Dataset summaries and validation
# =============================================================================

dataset_summary = pd.DataFrame(
    [
        {
            "DatasetView":
                "MODEL_SELECTION",

            "Rows":
                len(
                    selection_dataset
                ),

            "Products":
                selection_dataset[
                    "CanonicalProductID"
                ].nunique(),

            "Weeks":
                selection_dataset[
                    "WeekStartDate"
                ].nunique(),

            "WeekStartMin":
                selection_dataset[
                    "WeekStartDate"
                ].min().strftime(
                    "%Y-%m-%d"
                ),

            "WeekStartMax":
                selection_dataset[
                    "WeekStartDate"
                ].max().strftime(
                    "%Y-%m-%d"
                ),

            "ContainsTargets":
                True,

            "WeeklyNormalDemand":
                float(
                    selection_dataset[
                        "WeeklyNormalDemand"
                    ].sum()
                ),

            "WeeklyBulkDemand":
                float(
                    selection_dataset[
                        "WeeklyBulkDemand"
                    ].sum()
                ),

            "WeeklyTotalDemand":
                float(
                    selection_dataset[
                        "WeeklyTotalDemand"
                    ].sum()
                ),
        },
        {
            "DatasetView":
                "OPENED_HOLDOUT_FEATURE_ONLY",

            "Rows":
                len(
                    opened_holdout_features
                ),

            "Products":
                opened_holdout_features[
                    "CanonicalProductID"
                ].nunique(),

            "Weeks":
                opened_holdout_features[
                    "WeekStartDate"
                ].nunique(),

            "WeekStartMin":
                opened_holdout_features[
                    "WeekStartDate"
                ].min().strftime(
                    "%Y-%m-%d"
                ),

            "WeekStartMax":
                opened_holdout_features[
                    "WeekStartDate"
                ].max().strftime(
                    "%Y-%m-%d"
                ),

            "ContainsTargets":
                False,

            "WeeklyNormalDemand":
                np.nan,

            "WeeklyBulkDemand":
                np.nan,

            "WeeklyTotalDemand":
                np.nan,
        },
    ]
)


expected_selection_rows = int(
    (
        weekly[
            "WeeklyTuningEligible"
        ]
        & ~weekly[
            "ContainsOpenedDailyHoldoutRows"
        ]
    ).sum()
)


expected_holdout_rows = int(
    weekly[
        "ContainsOpenedDailyHoldoutRows"
    ].sum()
)


selection_normal_total = float(
    selection_dataset[
        "WeeklyNormalDemand"
    ].sum()
)


expected_selection_normal_total = float(
    weekly.loc[
        weekly[
            "WeeklyTuningEligible"
        ],
        "WeeklyNormalDemand",
    ].sum()
)


selection_total_total = float(
    selection_dataset[
        "WeeklyTotalDemand"
    ].sum()
)


expected_selection_total_total = float(
    weekly.loc[
        weekly[
            "WeeklyTuningEligible"
        ],
        "WeeklyTotalDemand",
    ].sum()
)


validation = pd.DataFrame(
    [
        {
            "Check":
                "Part 11E lock and "
                "checkpoint verified",

            "Expected":
                True,

            "Actual":
                True,

            "Passed":
                True,
        },
        {
            "Check":
                "Weekly dataset hash "
                "matched Part 11E lock",

            "Expected":
                locked_weekly_hash,

            "Actual":
                weekly_hash,

            "Passed":
                weekly_hash
                == locked_weekly_hash,
        },
        {
            "Check":
                "Generated feature rows",

            "Expected":
                len(weekly),

            "Actual":
                len(features),

            "Passed":
                len(features)
                == len(weekly),
        },
        {
            "Check":
                "Duplicate generated "
                "feature keys",

            "Expected":
                0,

            "Actual":
                int(
                    features.duplicated(
                        [
                            "WeekStartDate",
                            "CanonicalProductID",
                        ]
                    ).sum()
                ),

            "Passed":
                not features.duplicated(
                    [
                        "WeekStartDate",
                        "CanonicalProductID",
                    ]
                ).any(),
        },
        {
            "Check":
                "Model-selection rows",

            "Expected":
                expected_selection_rows,

            "Actual":
                len(
                    selection_dataset
                ),

            "Passed":
                len(
                    selection_dataset
                )
                == expected_selection_rows,
        },
        {
            "Check":
                "Opened-holdout "
                "feature-only rows",

            "Expected":
                expected_holdout_rows,

            "Actual":
                len(
                    opened_holdout_features
                ),

            "Passed":
                len(
                    opened_holdout_features
                )
                == expected_holdout_rows,
        },
        {
            "Check":
                "Opened-holdout feature "
                "file target columns",

            "Expected":
                0,

            "Actual":
                int(
                    sum(
                        column
                        in opened_holdout_features.columns
                        for column
                        in target_columns
                    )
                ),

            "Passed":
                not any(
                    column
                    in opened_holdout_features.columns
                    for column
                    in target_columns
                ),
        },
        {
            "Check":
                "Forbidden direct predictors",

            "Expected":
                0,

            "Actual":
                len(
                    forbidden_found
                ),

            "Passed":
                len(
                    forbidden_found
                )
                == 0,
        },
        {
            "Check":
                "Unexpected non-history "
                "missingness",

            "Expected":
                0,

            "Actual":
                len(
                    unexpected_missing
                ),

            "Passed":
                unexpected_missing.empty,
        },
        {
            "Check":
                "Part 11E scope "
                "reconciliation mismatches",

            "Expected":
                0,

            "Actual":
                int(
                    scope_reconciliation[
                        "Mismatches"
                    ].sum()
                ),

            "Passed":
                int(
                    scope_reconciliation[
                        "Mismatches"
                    ].sum()
                )
                == 0,
        },
        {
            "Check":
                "Target-perturbation "
                "audit weeks passed",

            "Expected":
                len(
                    audit_weeks
                ),

            "Actual":
                int(
                    leakage_audit[
                        "Passed"
                    ].sum()
                ),

            "Passed":
                leakage_audit[
                    "Passed"
                ].all(),
        },
        {
            "Check":
                "Weekly normal-demand total "
                "preserved in selection view",

            "Expected":
                expected_selection_normal_total,

            "Actual":
                selection_normal_total,

            "Passed":
                math.isclose(
                    selection_normal_total,
                    expected_selection_normal_total,
                    abs_tol=1e-9,
                ),
        },
        {
            "Check":
                "Weekly total-demand total "
                "preserved in selection view",

            "Expected":
                expected_selection_total_total,

            "Actual":
                selection_total_total,

            "Passed":
                math.isclose(
                    selection_total_total,
                    expected_selection_total_total,
                    abs_tol=1e-9,
                ),
        },
        {
            "Check":
                "Models fitted or "
                "predictions created",

            "Expected":
                False,

            "Actual":
                False,

            "Passed":
                True,
        },
    ]
)


if not validation[
    "Passed"
].all():

    raise AssertionError(
        "Part 11F validation failed:\n"
        + validation.loc[
            ~validation[
                "Passed"
            ]
        ].to_string(
            index=False
        )
    )


# =============================================================================
# Freeze the feature-generation contract
# =============================================================================

generation_contract = {
    "StepID":
        STEP_ID,

    "CreatedLocal":
        NOW_LOCAL.isoformat(),

    "ForecastOrigin":
        (
            "START_OF_EACH_"
            "MONDAY_TO_SUNDAY_WEEK"
        ),

    "MinimumTrainingWeeksForLaterValidation":
        MIN_TRAINING_WEEKS,

    "ProductIdentityUsage":
        {
            "CanonicalProductID":
                (
                    "KEY_AND_ROUTING_ONLY_"
                    "NOT_PRIMARY_DIRECT_PREDICTOR"
                ),

            "CanonicalProductName":
                "DISPLAY_LABEL_ONLY",
        },

    "KnownAheadFeatures":
        [
            "AvailableOperatingDaysInWeek",
            "CalendarYearIndex",
            "WeekOfYearSin",
            "WeekOfYearCos",
            "MonthSin",
            "MonthCos",
            "IsDatasetBoundaryPartialWeek",
        ],

    "HistoryRule":
        (
            "ALL_DEMAND_DERIVED_FEATURES_"
            "USE_ONLY_WEEKS_STRICTLY_BEFORE_"
            "THE_FORECAST_WEEK"
        ),

    "ScopeRule":
        (
            "80_90_95_SCOPES_RECALCULATED_"
            "FROM_PRIOR_WEEKLY_NORMAL_DEMAND_"
            "FOR_EACH_FORECAST_WEEK_OR_"
            "TRAINING_FOLD"
        ),

    "OptionalSegmentFeature":
        (
            "PriorDemandSegment may be appended "
            "only in the explicit shared-model-"
            "with-segment-feature experiment"
        ),

    "MissingnessRule":
        (
            "RETAIN_STRUCTURAL_HISTORY_NAN_"
            "AND_IMPUTE_WITHIN_EACH_TRAINING_"
            "FOLD_DURING_MODELLING"
        ),

    "NormalTargetPredictors":
        normal_predictors,

    "TotalTargetPredictors":
        total_predictors,

    "RoutingColumns":
        routing_columns,

    "NormalPredictorFingerprintSHA256":
        normal_fingerprint,

    "TotalPredictorFingerprintSHA256":
        total_fingerprint,

    "RoutingFingerprintSHA256":
        routing_fingerprint,

    "OpenedHoldoutPolicy":
        (
            "FEATURE_ONLY_OUTPUT_WITHOUT_"
            "TARGET_COLUMNS_AND_NOT_ELIGIBLE_"
            "FOR_SELECTION"
        ),

    "ModelTrainingPerformed":
        False,
}


# =============================================================================
# Stage outputs, memory, checkpoint and lock
# =============================================================================

stage_root = (
    EXT_ROOT
    / f".11F_staging_{uuid.uuid4().hex}"
)


stage_root.mkdir(
    parents=True,
    exist_ok=False,
)


try:

    for path, frame in [
        (
            SELECTION_DATASET_PATH,
            selection_dataset,
        ),
        (
            OPENED_HOLDOUT_FEATURES_PATH,
            opened_holdout_features,
        ),
        (
            FEATURE_CONTRACT_PATH,
            feature_contract,
        ),
        (
            MISSINGNESS_PATH,
            missingness,
        ),
        (
            LEAKAGE_AUDIT_PATH,
            leakage_audit,
        ),
        (
            SCOPE_RECONCILIATION_PATH,
            scope_reconciliation,
        ),
        (
            DATASET_SUMMARY_PATH,
            dataset_summary,
        ),
        (
            VALIDATION_PATH,
            validation,
        ),
    ]:

        stage_csv(
            stage_root,
            path,
            frame,
        )


    stage_json(
        stage_root,
        GENERATION_CONTRACT_PATH,
        generation_contract,
    )


    decisions = "\n".join(
        [
            (
                f"- Decision date: "
                f"{NOW_LOCAL.date().isoformat()}."
            ),
            (
                "- Forecast origin is the start "
                "of each Monday-to-Sunday week."
            ),
            (
                "- Every demand-derived predictor "
                "uses only weeks strictly before "
                "the forecast week."
            ),
            (
                "- AvailableOperatingDaysInWeek "
                "and calendar cycles are treated "
                "as known-ahead planning fields."
            ),
            (
                "- Product ID is retained as a key "
                "and routing field, not a primary "
                "direct predictor."
            ),
            (
                "- WeeklyNormalDemand models use "
                "normal-demand history plus common "
                "calendar and aggregate context."
            ),
            (
                "- WeeklyTotalDemand models may "
                "additionally use lagged total-demand "
                "and bulk-demand history."
            ),
            (
                "- Structural history NaN values "
                "are retained and must be imputed "
                "within each chronological "
                "training fold."
            ),
            (
                "- Dynamic 80%, 90% and 95% scope "
                "fields are routing or diagnostic "
                "fields and were reconciled exactly "
                "to Part 11E."
            ),
            (
                "- PriorDemandSegment is frozen as "
                "an optional categorical field only "
                "for the explicit shared-model-with-"
                "segment-feature experiment."
            ),
            (
                "- The opened March 2026 output "
                "contains features only and "
                "no targets."
            ),
        ]
    )


    files = "\n".join(
        [
            (
                "- Model-selection features: "
                f"`{SELECTION_DATASET_PATH}`"
            ),
            (
                "- Opened-holdout scoring features: "
                f"`{OPENED_HOLDOUT_FEATURES_PATH}`"
            ),
            (
                f"- Feature contract: "
                f"`{FEATURE_CONTRACT_PATH}`"
            ),
            (
                "- Generation contract: "
                f"`{GENERATION_CONTRACT_PATH}`"
            ),
            (
                f"- Missingness audit: "
                f"`{MISSINGNESS_PATH}`"
            ),
            (
                f"- Leakage audit: "
                f"`{LEAKAGE_AUDIT_PATH}`"
            ),
            (
                "- Scope reconciliation: "
                f"`{SCOPE_RECONCILIATION_PATH}`"
            ),
            (
                f"- Validation: "
                f"`{VALIDATION_PATH}`"
            ),
        ]
    )


    results = "\n".join(
        [
            (
                f"- Model-selection rows: "
                f"{len(selection_dataset):,} "
                f"across "
                f"{selection_dataset['WeekStartDate'].nunique()} "
                f"weeks and "
                f"{selection_dataset['CanonicalProductID'].nunique()} "
                "products."
            ),
            (
                "- Opened-holdout feature-only rows: "
                f"{len(opened_holdout_features):,}."
            ),
            (
                "- WeeklyNormalDemand direct predictors: "
                f"{len(normal_predictors)}; "
                f"fingerprint `{normal_fingerprint}`."
            ),
            (
                "- WeeklyTotalDemand direct predictors: "
                f"{len(total_predictors)}; "
                f"fingerprint `{total_fingerprint}`."
            ),
            (
                f"- Routing fields: "
                f"{len(routing_columns)}; "
                f"fingerprint `{routing_fingerprint}`."
            ),
            (
                "- Part 11E scope-reconciliation "
                "mismatches: "
                f"{int(scope_reconciliation['Mismatches'].sum())}."
            ),
            (
                "- Target-perturbation audit "
                "weeks passed: "
                f"{int(leakage_audit['Passed'].sum())}/"
                f"{len(leakage_audit)}."
            ),
        ]
    )


    for key, final_path in MEMORY_FILES.items():

        original = final_path.read_text(
            encoding="utf-8"
        )


        if key == "PROJECT_CONTEXT":

            updated = update_section(
                original,
                "STEP_11F",
                "Part 11F weekly feature design",
                (
                    "A frozen leakage-safe weekly "
                    "feature contract was created for "
                    "separate WeeklyNormalDemand and "
                    "WeeklyTotalDemand experiments. "
                    "No model was fitted."
                ),
            )


        elif key == "WORKFLOW":

            updated = update_section(
                original,
                "STEP_11F",
                "Part 11F workflow status",
                (
                    "Part 11F is complete. Part 11G "
                    "may train and chronologically "
                    "validate candidate weekly models "
                    "using only the model-selection "
                    "feature file and the frozen "
                    "target-specific predictor lists."
                ),
            )


        elif key == "DECISIONS":

            updated = update_section(
                original,
                "STEP_11F",
                "Part 11F decisions",
                decisions,
            )


        elif key == "FILES_AND_PATHS":

            updated = update_section(
                original,
                "STEP_11F",
                "Part 11F files",
                files,
            )


        elif key == "METRICS_AND_RESULTS":

            updated = update_section(
                original,
                "STEP_11F",
                "Part 11F metrics and results",
                results,
            )


        elif key == "CHAT_INDEX":

            row = (
                f"| {NOW_LOCAL.isoformat()} | "
                "11F | Leakage-safe weekly "
                "feature design and contract | "
                f"{STATUS} |"
            )

            updated = (
                original
                if row in original
                else (
                    original.rstrip()
                    + "\n"
                    + row
                    + "\n"
                )
            )


        elif key == "CURRENT_HANDOFF":

            updated = "\n".join(
                [
                    "# Current Handoff",
                    "",
                    (
                        "- Current completed step: "
                        f"{STEP_ID}"
                    ),
                    (
                        f"- Status: {STATUS}"
                    ),
                    (
                        "- Updated local time: "
                        f"{NOW_LOCAL.isoformat()}"
                    ),
                    (
                        "- Model-selection feature "
                        f"dataset: {SELECTION_DATASET_PATH}"
                    ),
                    (
                        "- Opened-holdout feature-only "
                        f"dataset: {OPENED_HOLDOUT_FEATURES_PATH}"
                    ),
                    (
                        "- Feature generation contract: "
                        f"{GENERATION_CONTRACT_PATH}"
                    ),
                    (
                        "- WeeklyNormalDemand predictor "
                        f"fingerprint: {normal_fingerprint}"
                    ),
                    (
                        "- WeeklyTotalDemand predictor "
                        f"fingerprint: {total_fingerprint}"
                    ),
                    (
                        "- Product ID is not a primary "
                        "direct predictor."
                    ),
                    (
                        "- Opened March 2026 targets "
                        "remain prohibited from model "
                        "selection."
                    ),
                    (
                        "- Next step: 11G candidate "
                        "weekly model training and "
                        "chronological validation."
                    ),
                    "",
                ]
            )


        else:

            raise KeyError(
                key
            )


        stage_text(
            stage_root,
            final_path,
            updated,
        )


    step_text = f"""# Step 11F — Leakage-Safe Weekly Feature Design

- **Step ID:** 11F
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## User request
Design leakage-safe weekly features after completion of the weekly baseline evaluation.

## Technical actions
- Verified the Part 11E lock, checkpoint and weekly source hash.
- Generated known-ahead calendar and operating-calendar fields.
- Generated lagged, rolling, expanding, occurrence and trend features using only prior weeks.
- Generated prior global weekly-demand context.
- Recomputed prior normal-demand ranks and strict 80%, 90% and 95% routing scopes for each forecast week.
- Reconciled those scopes exactly to the Part 11E chronological folds.
- Ran target-perturbation leakage tests on representative pre-holdout and opened-holdout weeks.
- Created a target-containing model-selection file and a target-free opened-holdout scoring-feature file.
- Froze separate predictor lists and fingerprints for WeeklyNormalDemand and WeeklyTotalDemand.

## Decisions
{decisions}

## Inputs
- `{WEEKLY_DATASET}`
- `{PART11E_LOCK}`
- `{PART11E_FOLD_MEMBERSHIP}`
- `{PART11E_DESIGN}`

## Outputs
{files}

## Validation
All {len(validation)} Part 11F validation checks passed.

## Errors encountered
None during the successful execution.

## Results
{results}

## Limitations
- The feature set is a frozen candidate set, not evidence that every feature improves forecasting.
- Structural missing values occur where products have insufficient prior history.
- Calendar or event information beyond the available operating calendar is not included.
- Scope flags are routing fields and must not become fixed labels across folds.
- A new untouched future period remains necessary for unbiased final weekly evaluation.

## Next action
Part 11G: train and chronologically validate simple global and routed weekly candidate models against the Part 11E reference baselines.
"""


    stage_text(
        stage_root,
        STEP_MEMORY_PATH,
        step_text,
    )


    checkpoint = {
        "StepID":
            STEP_ID,

        "Status":
            STATUS,

        "CreatedUTC":
            NOW_UTC.isoformat(),

        "CreatedLocal":
            NOW_LOCAL.isoformat(),

        "Input":
            {
                "WeeklyDatasetPath":
                    str(
                        WEEKLY_DATASET
                    ),

                "WeeklyDatasetSHA256":
                    weekly_hash,

                "Part11ELockSHA256":
                    part11e_lock_hash,

                "Part11ECheckpointSHA256":
                    part11e_checkpoint_hash,
            },

        "Outputs":
            {
                "ModelSelectionRows":
                    len(
                        selection_dataset
                    ),

                "OpenedHoldoutFeatureOnlyRows":
                    len(
                        opened_holdout_features
                    ),

                "NormalPredictorCount":
                    len(
                        normal_predictors
                    ),

                "TotalPredictorCount":
                    len(
                        total_predictors
                    ),

                "RoutingColumnCount":
                    len(
                        routing_columns
                    ),

                "NormalPredictorFingerprintSHA256":
                    normal_fingerprint,

                "TotalPredictorFingerprintSHA256":
                    total_fingerprint,

                "RoutingFingerprintSHA256":
                    routing_fingerprint,
            },

        "LeakageValidation":
            {
                "ScopeReconciliationMismatches":
                    int(
                        scope_reconciliation[
                            "Mismatches"
                        ].sum()
                    ),

                "TargetPerturbationWeeksTested":
                    len(
                        leakage_audit
                    ),

                "TargetPerturbationWeeksPassed":
                    int(
                        leakage_audit[
                            "Passed"
                        ].sum()
                    ),
            },

        "Safety":
            {
                "ProtectedTargetVaultOpened":
                    False,

                "ProtectedTargetVaultCopied":
                    False,

                "OpenedDailyHoldoutUsedForWeeklySelection":
                    False,

                "OpenedHoldoutTargetsWrittenToPart11F":
                    False,

                "ProductIDUsedAsPrimaryDirectPredictor":
                    False,

                "ModelsFitted":
                    False,

                "PredictionsCreated":
                    False,
            },

        "ReadyForPart11G":
            True,

        "NextStep":
            "11G",
    }


    stage_json(
        stage_root,
        CHECKPOINT_PATH,
        checkpoint,
    )


    checkpoint_hash = sha256_file(
        stage_path(
            stage_root,
            CHECKPOINT_PATH,
        )
    )


    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        (
            f"{checkpoint_hash}  "
            f"{CHECKPOINT_PATH.name}\n"
        ),
    )


    manifest_targets = [
        SELECTION_DATASET_PATH,
        OPENED_HOLDOUT_FEATURES_PATH,
        FEATURE_CONTRACT_PATH,
        GENERATION_CONTRACT_PATH,
        MISSINGNESS_PATH,
        LEAKAGE_AUDIT_PATH,
        SCOPE_RECONCILIATION_PATH,
        DATASET_SUMMARY_PATH,
        VALIDATION_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]


    manifest = pd.DataFrame(
        [
            {
                "RelativePath":
                    str(
                        path.relative_to(
                            EXT_ROOT
                        )
                    ),

                "Bytes":
                    stage_path(
                        stage_root,
                        path,
                    ).stat().st_size,

                "SHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            path,
                        )
                    ),
            }
            for path in manifest_targets
        ]
    ).sort_values(
        "RelativePath"
    )


    stage_csv(
        stage_root,
        MANIFEST_PATH,
        manifest,
    )


    manifest_hash = sha256_file(
        stage_path(
            stage_root,
            MANIFEST_PATH,
        )
    )


    lock = {
        "StepID":
            STEP_ID,

        "Status":
            STATUS,

        "CreatedUTC":
            NOW_UTC.isoformat(),

        "Part11E":
            {
                "LockSHA256":
                    part11e_lock_hash,

                "CheckpointSHA256":
                    part11e_checkpoint_hash,

                "WeeklyDatasetSHA256":
                    weekly_hash,
            },

        "FeatureGenerationContract":
            {
                "Path":
                    str(
                        GENERATION_CONTRACT_PATH
                    ),

                "SHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            GENERATION_CONTRACT_PATH,
                        )
                    ),

                "NormalPredictorFingerprintSHA256":
                    normal_fingerprint,

                "TotalPredictorFingerprintSHA256":
                    total_fingerprint,

                "RoutingFingerprintSHA256":
                    routing_fingerprint,
            },

        "FeatureDatasets":
            {
                "ModelSelectionPath":
                    str(
                        SELECTION_DATASET_PATH
                    ),

                "ModelSelectionSHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            SELECTION_DATASET_PATH,
                        )
                    ),

                "ModelSelectionRows":
                    len(
                        selection_dataset
                    ),

                "OpenedHoldoutFeatureOnlyPath":
                    str(
                        OPENED_HOLDOUT_FEATURES_PATH
                    ),

                "OpenedHoldoutFeatureOnlySHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            OPENED_HOLDOUT_FEATURES_PATH,
                        )
                    ),

                "OpenedHoldoutFeatureOnlyRows":
                    len(
                        opened_holdout_features
                    ),
            },

        "OutputHashManifest":
            {
                "Path":
                    str(
                        MANIFEST_PATH
                    ),

                "SHA256":
                    manifest_hash,

                "FilesListed":
                    len(
                        manifest
                    ),
            },

        "Checkpoint":
            {
                "Path":
                    str(
                        CHECKPOINT_PATH
                    ),

                "SHA256":
                    checkpoint_hash,
            },

        "SafetyAssertions":
            checkpoint[
                "Safety"
            ],

        "ReadyForPart11G":
            True,

        "NextStep":
            "11G",
    }


    stage_json(
        stage_root,
        LOCK_PATH,
        lock,
    )


    lock_hash = sha256_file(
        stage_path(
            stage_root,
            LOCK_PATH,
        )
    )


    stage_text(
        stage_root,
        LOCK_SHA_PATH,
        (
            f"{lock_hash}  "
            f"{LOCK_PATH.name}\n"
        ),
    )


    stage_text(
        stage_root,
        LOG_PATH,
        "\n".join(
            [
                f"Status: {STATUS}",
                (
                    f"Run local: "
                    f"{NOW_LOCAL.isoformat()}"
                ),
                (
                    f"Weekly input SHA256: "
                    f"{weekly_hash}"
                ),
                (
                    "Model-selection rows: "
                    f"{len(selection_dataset)}"
                ),
                (
                    "Opened-holdout "
                    "feature-only rows: "
                    f"{len(opened_holdout_features)}"
                ),
                (
                    "Normal predictor fingerprint: "
                    f"{normal_fingerprint}"
                ),
                (
                    "Total predictor fingerprint: "
                    f"{total_fingerprint}"
                ),
                (
                    "Scope reconciliation mismatches: "
                    f"{int(scope_reconciliation['Mismatches'].sum())}"
                ),
                (
                    "Target perturbation audits passed: "
                    f"{int(leakage_audit['Passed'].sum())}/"
                    f"{len(leakage_audit)}"
                ),
                (
                    f"Checkpoint SHA256: "
                    f"{checkpoint_hash}"
                ),
                (
                    f"Part 11F lock SHA256: "
                    f"{lock_hash}"
                ),
                "",
            ]
        ),
    )


    staged_files = [
        path
        for path in stage_root.rglob("*")
        if path.is_file()
    ]


    if not staged_files:

        raise AssertionError(
            "Part 11F staging "
            "directory is empty"
        )


    if any(
        (
            "target_vault"
            in path.name.lower()
        )
        or (
            path.suffix.lower()
            == ".joblib"
        )
        for path in staged_files
    ):

        raise PermissionError(
            "Forbidden target-vault or "
            "model file detected in staging"
        )


    for staged in sorted(
        staged_files
    ):

        final = (
            EXT_ROOT
            / staged.relative_to(
                stage_root
            )
        )

        final.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if (
            final.exists()
            and final
            not in MEMORY_FILES.values()
        ):

            raise FileExistsError(
                "Refusing to overwrite "
                f"Part 11F output: {final}"
            )

        os.replace(
            staged,
            final,
        )


    for path in NEW_OUTPUTS:

        make_read_only(
            path
        )


finally:

    if stage_root.exists():

        shutil.rmtree(
            stage_root
        )


# =============================================================================
# Final technical output
# =============================================================================

print(
    "=" * 100
)

print(
    "EDEN WEEKLY FORECASTING EXTENSION — "
    "PART 11F COMPLETE"
)

print(
    "=" * 100
)

print(
    f"Status: {STATUS}"
)

print(
    f"Extension root: {EXT_ROOT}"
)

print(
    f"Local time: {NOW_LOCAL.isoformat()}"
)


print(
    "\nFEATURE DATASET SUMMARY"
)

print(
    dataset_summary.to_string(
        index=False
    )
)


print(
    "\nFROZEN TARGET-SPECIFIC "
    "FEATURE CONTRACT"
)

print(
    "WeeklyNormalDemand direct predictors: "
    f"{len(normal_predictors)}"
)

print(
    "WeeklyNormalDemand predictor fingerprint: "
    f"{normal_fingerprint}"
)

print(
    "WeeklyTotalDemand direct predictors: "
    f"{len(total_predictors)}"
)

print(
    "WeeklyTotalDemand predictor fingerprint: "
    f"{total_fingerprint}"
)

print(
    f"Routing fields: "
    f"{len(routing_columns)}"
)

print(
    f"Routing fingerprint: "
    f"{routing_fingerprint}"
)

print(
    "Product ID direct predictor: False"
)


print(
    "\nPART 11E SCOPE RECONCILIATION"
)

print(
    scope_reconciliation.to_string(
        index=False
    )
)


print(
    "\nTARGET-PERTURBATION LEAKAGE AUDIT"
)

print(
    leakage_audit.to_string(
        index=False
    )
)


print(
    "\nFEATURE MISSINGNESS SUMMARY"
)

print(
    missingness_summary.to_string(
        index=False
    )
)


print(
    "\nPART 11F VALIDATION"
)

print(
    validation.to_string(
        index=False
    )
)


print(
    "\nCONTROL OUTPUTS"
)

print(
    "- Model-selection features: "
    f"{SELECTION_DATASET_PATH}"
)

print(
    "- Opened-holdout scoring features: "
    f"{OPENED_HOLDOUT_FEATURES_PATH}"
)

print(
    "- Feature contract: "
    f"{FEATURE_CONTRACT_PATH}"
)

print(
    "- Generation contract: "
    f"{GENERATION_CONTRACT_PATH}"
)

print(
    "- Missingness audit: "
    f"{MISSINGNESS_PATH}"
)

print(
    "- Leakage audit: "
    f"{LEAKAGE_AUDIT_PATH}"
)

print(
    "- Scope reconciliation: "
    f"{SCOPE_RECONCILIATION_PATH}"
)

print(
    f"- Checkpoint: {CHECKPOINT_PATH}"
)

print(
    "- Checkpoint SHA256: "
    f"{checkpoint_hash}"
)

print(
    f"- Part 11F lock: {LOCK_PATH}"
)

print(
    "- Part 11F lock SHA256: "
    f"{lock_hash}"
)


print(
    "\nSAFETY: protected target vault "
    "opened/copied False/False; opened "
    "March 2026 targets written to Part 11F "
    "False; opened holdout used for weekly "
    "selection False; product ID used as "
    "primary direct predictor False; models "
    "fitted False; predictions created False."
)

print(
    "=" * 100
)

EDEN WEEKLY FORECASTING EXTENSION — PART 11F COMPLETE
Status: PART_11F_COMPLETED_READY_FOR_11G
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-05T20:52:41.464511+01:00

FEATURE DATASET SUMMARY
                DatasetView  Rows  Products  Weeks WeekStartMin WeekStartMax  ContainsTargets  WeeklyNormalDemand  WeeklyBulkDemand  WeeklyTotalDemand
            MODEL_SELECTION  8222       227     47   2025-03-31   2026-02-23             True            100177.0            1972.0           102149.0
OPENED_HOLDOUT_FEATURE_ONLY  1135       227      5   2026-03-02   2026-03-30            False                 NaN               NaN                NaN

FROZEN TARGET-SPECIFIC FEATURE CONTRACT
WeeklyNormalDemand direct predictors: 48
WeeklyNormalDemand predictor fingerprint: 82daf9ce08bdb956964daa367730d21a59b0b1c666ceb5f8fc19de159f532d43
WeeklyTotalDemand direct predictors: 75
WeeklyTotalDemand predictor fingerprint: f6c200238fb24c64

In [19]:
# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11G
# Candidate weekly model screening with leakage-safe chronological validation
# =============================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import stat
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# 1. PATHS AND FIXED SCREENING DESIGN
# =============================================================================

PROJECT_ROOT = Path(
    "/Users/ryansmac/Desktop/Meng Project"
)

EDEN_ROOT = (
    PROJECT_ROOT
    / "eden_datasets"
)

EXT_ROOT = (
    EDEN_ROOT
    / "weekly_forecasting_extension"
)

MEMORY_ROOT = (
    EXT_ROOT
    / "00_project_memory"
)

STEP_MEMORY_ROOT = (
    MEMORY_ROOT
    / "steps"
)

FEATURE_ROOT = (
    EXT_ROOT
    / "04_weekly_features"
)

BASELINE_ROOT = (
    EXT_ROOT
    / "05_weekly_baselines"
)

MODEL_ROOT = (
    EXT_ROOT
    / "06_weekly_models"
)

GLOBAL_ROOT = (
    MODEL_ROOT
    / "global"
)

HIGH_ROOT = (
    MODEL_ROOT
    / "high_demand"
)

MODERATE_ROOT = (
    MODEL_ROOT
    / "moderate_demand"
)

ROUTED_ROOT = (
    MODEL_ROOT
    / "routed_system"
)

VALIDATION_ROOT = (
    EXT_ROOT
    / "07_weekly_validation"
)

CHECKPOINT_ROOT = (
    EXT_ROOT
    / "11_checkpoints"
)

LOG_ROOT = (
    EXT_ROOT
    / "12_logs"
)

SELECTION_FEATURES = (
    FEATURE_ROOT
    / "11F_model_selection_weekly_features.csv"
)

FEATURE_CONTRACT = (
    FEATURE_ROOT
    / "11F_feature_contract.csv"
)

GENERATION_CONTRACT = (
    FEATURE_ROOT
    / "11F_feature_generation_contract.json"
)

PART11F_LOCK = (
    CHECKPOINT_ROOT
    / "11F_weekly_feature_contract_lock.json"
)

PART11F_LOCK_SHA = (
    CHECKPOINT_ROOT
    / "11F_weekly_feature_contract_lock.sha256"
)

PART11F_CHECKPOINT = (
    CHECKPOINT_ROOT
    / "11F_checkpoint.json"
)

PART11F_CHECKPOINT_SHA = (
    CHECKPOINT_ROOT
    / "11F_checkpoint.sha256"
)

PART11E_LOCK = (
    CHECKPOINT_ROOT
    / "11E_weekly_baselines_lock.json"
)

PART11E_LOCK_SHA = (
    CHECKPOINT_ROOT
    / "11E_weekly_baselines_lock.sha256"
)

BASELINE_RANKING = (
    BASELINE_ROOT
    / "11E_preholdout_baseline_ranking.csv"
)

FOLD_MEMBERSHIP = (
    BASELINE_ROOT
    / "11E_fold_scope_membership.csv"
)

PREDICTIONS_PATH = (
    VALIDATION_ROOT
    / "11G_candidate_weekly_predictions.csv"
)

FOLD_METRICS_PATH = (
    VALIDATION_ROOT
    / "11G_candidate_weekly_fold_metrics.csv"
)

POOLED_METRICS_PATH = (
    VALIDATION_ROOT
    / "11G_candidate_weekly_pooled_metrics.csv"
)

COMPARISON_PATH = (
    VALIDATION_ROOT
    / "11G_model_vs_baseline_comparison.csv"
)

RANKING_PATH = (
    VALIDATION_ROOT
    / "11G_candidate_model_ranking.csv"
)

FOLD_DESIGN_PATH = (
    VALIDATION_ROOT
    / "11G_chronological_fold_design.csv"
)

ROUTING_AUDIT_PATH = (
    VALIDATION_ROOT
    / "11G_routing_and_fallback_audit.csv"
)

SCREENING_CONTRACT_PATH = (
    VALIDATION_ROOT
    / "11G_model_screening_contract.json"
)

VALIDATION_PATH = (
    VALIDATION_ROOT
    / "11G_validation.csv"
)

MANIFEST_PATH = (
    VALIDATION_ROOT
    / "11G_output_hash_manifest.csv"
)

GLOBAL_CONFIG_PATH = (
    GLOBAL_ROOT
    / "11G_global_model_configurations.csv"
)

HIGH_CONFIG_PATH = (
    HIGH_ROOT
    / "11G_high_demand_model_configuration.csv"
)

MODERATE_CONFIG_PATH = (
    MODERATE_ROOT
    / "11G_moderate_demand_model_configuration.csv"
)

ROUTED_CONFIG_PATH = (
    ROUTED_ROOT
    / "11G_routed_system_configuration.csv"
)

STEP_MEMORY_PATH = (
    STEP_MEMORY_ROOT
    / "STEP_11G_WEEKLY_MODEL_SCREENING.md"
)

CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "11G_checkpoint.json"
)

CHECKPOINT_SHA_PATH = (
    CHECKPOINT_ROOT
    / "11G_checkpoint.sha256"
)

LOCK_PATH = (
    CHECKPOINT_ROOT
    / "11G_weekly_model_screening_lock.json"
)

LOCK_SHA_PATH = (
    CHECKPOINT_ROOT
    / "11G_weekly_model_screening_lock.sha256"
)

LOG_PATH = (
    LOG_ROOT
    / "11G_weekly_model_screening_log.txt"
)

MEMORY_FILES = {
    "PROJECT_CONTEXT":
        MEMORY_ROOT / "PROJECT_CONTEXT.md",

    "WORKFLOW":
        MEMORY_ROOT / "WORKFLOW.md",

    "DECISIONS":
        MEMORY_ROOT / "DECISIONS.md",

    "FILES_AND_PATHS":
        MEMORY_ROOT / "FILES_AND_PATHS.md",

    "METRICS_AND_RESULTS":
        MEMORY_ROOT / "METRICS_AND_RESULTS.md",

    "CHAT_INDEX":
        MEMORY_ROOT / "CHAT_INDEX.md",

    "CURRENT_HANDOFF":
        MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

STEP_ID = "11G"

STATUS = (
    "PART_11G_COMPLETED_READY_FOR_11H"
)

NOW_UTC = datetime.now(
    timezone.utc
)

NOW_LOCAL = NOW_UTC.astimezone(
    ZoneInfo("Europe/Dublin")
)

RANDOM_STATE = 42

MIN_TRAINING_WEEKS = 8

MIN_SEGMENT_TRAINING_ROWS = 40

MIN_POSITIVE_QUANTITY_ROWS = 20

TARGETS = {
    "WEEKLY_NORMAL_DEMAND":
        "WeeklyNormalDemand",

    "WEEKLY_TOTAL_DEMAND":
        "WeeklyTotalDemand",
}

SCOPE_FLAGS = {
    "ALL_PRODUCTS":
        None,

    "FOLD_STRICT_80_PERCENT":
        "PriorStrict80PctScope",

    "FOLD_STRICT_90_PERCENT":
        "PriorStrict90PctScope",

    "FOLD_STRICT_95_PERCENT":
        "PriorStrict95PctScope",
}

CANDIDATE_SYSTEMS = [
    "GLOBAL_RIDGE",
    "GLOBAL_HISTGB",
    "GLOBAL_HURDLE_HISTGB",
    "SEGMENT_FEATURE_RIDGE",
    "ROUTED_SEGMENT_HISTGB",
]

COMPLEXITY_ORDER = {
    "GLOBAL_RIDGE": 1,
    "GLOBAL_HISTGB": 2,
    "SEGMENT_FEATURE_RIDGE": 3,
    "GLOBAL_HURDLE_HISTGB": 4,
    "ROUTED_SEGMENT_HISTGB": 5,
    "NAIVE_LAST_CONTEXT": 0,
}

NEW_OUTPUTS = [
    PREDICTIONS_PATH,
    FOLD_METRICS_PATH,
    POOLED_METRICS_PATH,
    COMPARISON_PATH,
    RANKING_PATH,
    FOLD_DESIGN_PATH,
    ROUTING_AUDIT_PATH,
    SCREENING_CONTRACT_PATH,
    VALIDATION_PATH,
    MANIFEST_PATH,
    GLOBAL_CONFIG_PATH,
    HIGH_CONFIG_PATH,
    MODERATE_CONFIG_PATH,
    ROUTED_CONFIG_PATH,
    STEP_MEMORY_PATH,
    CHECKPOINT_PATH,
    CHECKPOINT_SHA_PATH,
    LOCK_PATH,
    LOCK_SHA_PATH,
    LOG_PATH,
]

# =============================================================================
# 2. UTILITIES
# =============================================================================

def sha256_file(
    path: Path,
) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def list_fingerprint(
    values: list[str],
) -> str:

    payload = json.dumps(
        values,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


def require_file(
    path: Path,
    label: str,
) -> None:

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing required {label}:\n"
            f"{path}"
        )


def verify_sidecar(
    path: Path,
    sidecar: Path,
    label: str,
) -> str:

    require_file(
        path,
        label,
    )

    require_file(
        sidecar,
        f"{label} SHA-256 sidecar",
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )

    actual = sha256_file(
        path
    )

    if (
        len(expected) != 64
        or expected != actual
    ):

        raise AssertionError(
            f"{label} SHA-256 mismatch:\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


def atomic_text(
    path: Path,
    text: str,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}."
        f"{uuid.uuid4().hex}.tmp"
    )

    try:

        temporary.write_text(
            text,
            encoding="utf-8",
        )

        os.replace(
            temporary,
            path,
        )

    finally:

        if temporary.exists():

            temporary.unlink()


def atomic_csv(
    path: Path,
    frame: pd.DataFrame,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}."
        f"{uuid.uuid4().hex}.tmp"
    )

    try:

        frame.to_csv(
            temporary,
            index=False,
        )

        os.replace(
            temporary,
            path,
        )

    finally:

        if temporary.exists():

            temporary.unlink()


def atomic_json(
    path: Path,
    payload: dict,
) -> None:

    atomic_text(
        path,
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def bool_series(
    series: pd.Series,
    label: str,
) -> pd.Series:

    if pd.api.types.is_bool_dtype(
        series
    ):

        return series.astype(
            bool
        )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    normalised = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    invalid = sorted(
        set(normalised)
        - set(mapping)
    )

    if invalid:

        raise ValueError(
            f"{label} has invalid "
            f"Boolean values: {invalid[:10]}"
        )

    return (
        normalised
        .map(mapping)
        .astype(bool)
    )


def metric_record(
    actual: pd.Series,
    prediction: pd.Series,
) -> dict:

    actual_array = (
        pd.to_numeric(
            actual,
            errors="raise",
        )
        .to_numpy(
            dtype=float
        )
    )

    prediction_array = (
        pd.to_numeric(
            prediction,
            errors="raise",
        )
        .to_numpy(
            dtype=float
        )
    )

    if (
        len(actual_array)
        != len(prediction_array)
    ):

        raise AssertionError(
            "Actual and prediction "
            "lengths differ"
        )

    if (
        not np.isfinite(
            actual_array
        ).all()
        or not np.isfinite(
            prediction_array
        ).all()
    ):

        raise AssertionError(
            "Non-finite values entered "
            "metric calculation"
        )

    errors = (
        prediction_array
        - actual_array
    )

    absolute_errors = np.abs(
        errors
    )

    denominator = float(
        np.abs(
            actual_array
        ).sum()
    )

    return {
        "Observations":
            int(
                len(actual_array)
            ),

        "ActualTotal":
            float(
                actual_array.sum()
            ),

        "PredictedTotal":
            float(
                prediction_array.sum()
            ),

        "MAE":
            (
                float(
                    absolute_errors.mean()
                )
                if len(actual_array)
                else np.nan
            ),

        "RMSE":
            (
                float(
                    np.sqrt(
                        np.mean(
                            errors ** 2
                        )
                    )
                )
                if len(actual_array)
                else np.nan
            ),

        "WAPEPercentage":
            (
                float(
                    100.0
                    * absolute_errors.sum()
                    / denominator
                )
                if denominator != 0
                else np.nan
            ),

        "MeanBias":
            (
                float(
                    errors.mean()
                )
                if len(actual_array)
                else np.nan
            ),

        "TotalBias":
            float(
                errors.sum()
            ),
    }


def update_section(
    text: str,
    marker: str,
    heading: str,
    body: str,
) -> str:

    start = (
        f"<!-- BEGIN {marker} -->"
    )

    end = (
        f"<!-- END {marker} -->"
    )

    section = (
        f"{start}\n"
        f"## {heading}\n\n"
        f"{body.rstrip()}\n"
        f"{end}"
    )

    if (
        start in text
        and end in text
    ):

        before = (
            text
            .split(
                start,
                1,
            )[0]
            .rstrip()
        )

        after = (
            text
            .split(
                end,
                1,
            )[1]
            .lstrip()
        )

        return (
            before
            + "\n\n"
            + section
            + (
                "\n\n" + after
                if after
                else ""
            )
            + "\n"
        )

    return (
        text.rstrip()
        + "\n\n"
        + section
        + "\n"
    )


def stage_path(
    stage_root: Path,
    final_path: Path,
) -> Path:

    return (
        stage_root
        / final_path.relative_to(
            EXT_ROOT
        )
    )


def stage_text(
    stage_root: Path,
    final_path: Path,
    text: str,
) -> None:

    atomic_text(
        stage_path(
            stage_root,
            final_path,
        ),
        text,
    )


def stage_csv(
    stage_root: Path,
    final_path: Path,
    frame: pd.DataFrame,
) -> None:

    atomic_csv(
        stage_path(
            stage_root,
            final_path,
        ),
        frame,
    )


def stage_json(
    stage_root: Path,
    final_path: Path,
    payload: dict,
) -> None:

    atomic_json(
        stage_path(
            stage_root,
            final_path,
        ),
        payload,
    )


def make_read_only(
    path: Path,
) -> None:

    if path.is_file():

        path.chmod(
            stat.S_IRUSR
            | stat.S_IRGRP
            | stat.S_IROTH
        )


def build_ridge(
    numeric_columns: list[str],
    segment_feature: bool = False,
) -> Pipeline:

    numeric_pipeline = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    if segment_feature:

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    numeric_pipeline,
                    numeric_columns,
                ),
                (
                    "segment",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                    [
                        "PriorDemandSegment"
                    ],
                ),
            ],
            remainder="drop",
            sparse_threshold=0.0,
        )

    else:

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    numeric_pipeline,
                    numeric_columns,
                ),
            ],
            remainder="drop",
            sparse_threshold=0.0,
        )

    return Pipeline(
        [
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                Ridge(
                    alpha=10.0
                ),
            ),
        ]
    )


def build_histgb(
    numeric_columns: list[str],
) -> Pipeline:

    return Pipeline(
        [
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            (
                "model",
                HistGradientBoostingRegressor(
                    learning_rate=0.05,
                    max_iter=120,
                    max_leaf_nodes=15,
                    min_samples_leaf=20,
                    l2_regularization=1.0,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def fit_predict_hurdle(
    train_x: pd.DataFrame,
    train_y: pd.Series,
    valid_x: pd.DataFrame,
    numeric_columns: list[str],
) -> tuple[
    np.ndarray,
    str,
    bool,
]:

    occurrence = (
        train_y
        .to_numpy(
            dtype=float
        )
        > 0
    ).astype(int)

    occurrence_unique = np.unique(
        occurrence
    )

    fallback_used = False

    if len(
        occurrence_unique
    ) == 1:

        probability = np.full(
            len(
                valid_x
            ),
            float(
                occurrence_unique[0]
            ),
            dtype=float,
        )

        occurrence_method = (
            "CONSTANT_OCCURRENCE"
        )

        fallback_used = True

    else:

        occurrence_pipeline = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "model",
                    LogisticRegression(
                        C=1.0,
                        max_iter=2000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )

        occurrence_pipeline.fit(
            train_x[
                numeric_columns
            ],
            occurrence,
        )

        probability = (
            occurrence_pipeline
            .predict_proba(
                valid_x[
                    numeric_columns
                ]
            )[:, 1]
        )

        occurrence_method = (
            "LOGISTIC_OCCURRENCE"
        )

    positive_mask = (
        train_y
        .to_numpy(
            dtype=float
        )
        > 0
    )

    positive_count = int(
        positive_mask.sum()
    )

    if (
        positive_count
        >= MIN_POSITIVE_QUANTITY_ROWS
    ):

        quantity_pipeline = (
            build_histgb(
                numeric_columns
            )
        )

        quantity_pipeline.fit(
            train_x.loc[
                positive_mask,
                numeric_columns,
            ],
            np.log1p(
                train_y.loc[
                    positive_mask
                ].to_numpy(
                    dtype=float
                )
            ),
        )

        quantity = np.expm1(
            quantity_pipeline.predict(
                valid_x[
                    numeric_columns
                ]
            )
        )

        quantity_method = (
            "LOG1P_HISTGB_"
            "POSITIVE_QUANTITY"
        )

    else:

        positive_mean = (
            float(
                train_y.loc[
                    positive_mask
                ].mean()
            )
            if positive_count
            else 0.0
        )

        quantity = np.full(
            len(
                valid_x
            ),
            positive_mean,
            dtype=float,
        )

        quantity_method = (
            "MEAN_POSITIVE_"
            "QUANTITY_FALLBACK"
        )

        fallback_used = True

    prediction = np.clip(
        probability
        * np.clip(
            quantity,
            0.0,
            None,
        ),
        0.0,
        None,
    )

    return (
        prediction,
        (
            f"{occurrence_method}__"
            f"{quantity_method}"
        ),
        fallback_used,
    )


def fit_predict_routed_histgb(
    train: pd.DataFrame,
    valid: pd.DataFrame,
    target_column: str,
    numeric_columns: list[str],
    global_prediction: np.ndarray,
) -> tuple[
    np.ndarray,
    list[str],
    list[bool],
]:

    predictions = np.asarray(
        global_prediction,
        dtype=float,
    ).copy()

    model_used = [
        "GLOBAL_HISTGB_FALLBACK"
    ] * len(
        valid
    )

    fallback_used = [
        True
    ] * len(
        valid
    )

    # This first routed experiment intentionally fits
    # only high- and moderate-demand segment models.
    # Products outside the initial 95% scope and products
    # without prior demand history retain the pooled forecast.
    segment_definitions = {
        "HIGH_PRIOR_DEMAND":
            "HIGH_HISTGB",

        "MODERATE_PRIOR_DEMAND":
            "MODERATE_HISTGB",
    }

    for (
        segment_value,
        model_label,
    ) in segment_definitions.items():

        train_mask = (
            train[
                "PriorDemandSegment"
            ].astype(str)
            == segment_value
        )

        valid_mask = (
            valid[
                "PriorDemandSegment"
            ].astype(str)
            == segment_value
        )

        train_count = int(
            train_mask.sum()
        )

        valid_count = int(
            valid_mask.sum()
        )

        if valid_count == 0:

            continue

        if (
            train_count
            >= MIN_SEGMENT_TRAINING_ROWS
        ):

            model = build_histgb(
                numeric_columns
            )

            model.fit(
                train.loc[
                    train_mask,
                    numeric_columns,
                ],
                train.loc[
                    train_mask,
                    target_column,
                ],
            )

            segment_prediction = np.clip(
                model.predict(
                    valid.loc[
                        valid_mask,
                        numeric_columns,
                    ]
                ),
                0.0,
                None,
            )

            positions = np.flatnonzero(
                valid_mask.to_numpy()
            )

            predictions[
                positions
            ] = segment_prediction

            for position in positions:

                model_used[
                    position
                ] = model_label

                fallback_used[
                    position
                ] = False

    return (
        predictions,
        model_used,
        fallback_used,
    )


# =============================================================================
# 3. VERIFY LOCKED INPUTS AND OVERWRITE GUARDS
# =============================================================================

for directory, label in [
    (
        EXT_ROOT,
        "extension root",
    ),
    (
        MEMORY_ROOT,
        "memory root",
    ),
    (
        FEATURE_ROOT,
        "weekly feature directory",
    ),
    (
        BASELINE_ROOT,
        "weekly baseline directory",
    ),
    (
        GLOBAL_ROOT,
        "global model directory",
    ),
    (
        HIGH_ROOT,
        "high-demand model directory",
    ),
    (
        MODERATE_ROOT,
        "moderate-demand model directory",
    ),
    (
        ROUTED_ROOT,
        "routed-system directory",
    ),
    (
        VALIDATION_ROOT,
        "weekly validation directory",
    ),
    (
        CHECKPOINT_ROOT,
        "checkpoint directory",
    ),
    (
        LOG_ROOT,
        "log directory",
    ),
]:

    if not directory.is_dir():

        raise FileNotFoundError(
            f"Missing required {label}:\n"
            f"{directory}"
        )


for path, label in [
    (
        SELECTION_FEATURES,
        (
            "Part 11F model-selection "
            "feature dataset"
        ),
    ),
    (
        FEATURE_CONTRACT,
        "Part 11F feature contract",
    ),
    (
        GENERATION_CONTRACT,
        (
            "Part 11F generation "
            "contract"
        ),
    ),
    (
        BASELINE_RANKING,
        "Part 11E baseline ranking",
    ),
    (
        FOLD_MEMBERSHIP,
        "Part 11E fold membership",
    ),
]:

    require_file(
        path,
        label,
    )


for key, path in MEMORY_FILES.items():

    require_file(
        path,
        f"memory file {key}",
    )


if (
    LOCK_PATH.exists()
    or LOCK_SHA_PATH.exists()
):

    raise FileExistsError(
        "Part 11G overwrite lock "
        f"triggered:\n{LOCK_PATH}"
    )


existing_outputs = [
    path
    for path in NEW_OUTPUTS
    if path.exists()
]


if existing_outputs:

    raise FileExistsError(
        "Existing uncommitted Part 11G "
        "outputs found; no files changed:\n"
        + "\n".join(
            f"- {path}"
            for path in existing_outputs
        )
    )


for old_stage in EXT_ROOT.glob(
    ".11G_staging_*"
):

    if old_stage.is_dir():

        shutil.rmtree(
            old_stage
        )


part11f_lock_hash = verify_sidecar(
    PART11F_LOCK,
    PART11F_LOCK_SHA,
    "Part 11F lock",
)


part11f_checkpoint_hash = (
    verify_sidecar(
        PART11F_CHECKPOINT,
        PART11F_CHECKPOINT_SHA,
        "Part 11F checkpoint",
    )
)


part11e_lock_hash = verify_sidecar(
    PART11E_LOCK,
    PART11E_LOCK_SHA,
    "Part 11E lock",
)


part11f_lock = json.loads(
    PART11F_LOCK.read_text(
        encoding="utf-8"
    )
)


part11e_lock = json.loads(
    PART11E_LOCK.read_text(
        encoding="utf-8"
    )
)


generation_contract = json.loads(
    GENERATION_CONTRACT.read_text(
        encoding="utf-8"
    )
)


if (
    part11f_lock.get(
        "Status"
    )
    !=
    "PART_11F_COMPLETED_READY_FOR_11G"
):

    raise AssertionError(
        "Unexpected Part 11F status: "
        f"{part11f_lock.get('Status')}"
    )


if (
    part11f_lock.get(
        "ReadyForPart11G"
    )
    is not True
):

    raise AssertionError(
        "Part 11F lock does not "
        "authorise Part 11G"
    )


if (
    part11e_lock.get(
        "Status"
    )
    !=
    "PART_11E_COMPLETED_READY_FOR_11F"
):

    raise AssertionError(
        "Part 11E lock status "
        "changed unexpectedly"
    )


if (
    part11f_lock
    .get(
        "Part11E",
        {},
    )
    .get(
        "LockSHA256"
    )
    != part11e_lock_hash
):

    raise AssertionError(
        "Part 11F does not reference "
        "the current Part 11E lock"
    )


for key, expected in {
    "ProtectedTargetVaultOpened":
        False,

    "ProtectedTargetVaultCopied":
        False,

    "OpenedDailyHoldoutUsedForWeeklySelection":
        False,

    "OpenedHoldoutTargetsWrittenToPart11F":
        False,

    "ProductIDUsedAsPrimaryDirectPredictor":
        False,

    "ModelsFitted":
        False,

    "PredictionsCreated":
        False,
}.items():

    actual = (
        part11f_lock
        .get(
            "SafetyAssertions",
            {},
        )
        .get(
            key
        )
    )

    if actual is not expected:

        raise AssertionError(
            "Invalid Part 11F safety "
            f"assertion {key}: {actual}"
        )


selection_hash = sha256_file(
    SELECTION_FEATURES
)


locked_selection_hash = (
    part11f_lock
    .get(
        "FeatureDatasets",
        {},
    )
    .get(
        "ModelSelectionSHA256"
    )
)


if (
    selection_hash
    != locked_selection_hash
):

    raise AssertionError(
        "Part 11F model-selection "
        "feature hash changed"
    )


normal_predictors = list(
    generation_contract.get(
        "NormalTargetPredictors",
        [],
    )
)


total_predictors = list(
    generation_contract.get(
        "TotalTargetPredictors",
        [],
    )
)


routing_columns = list(
    generation_contract.get(
        "RoutingColumns",
        [],
    )
)


if (
    list_fingerprint(
        normal_predictors
    )
    !=
    generation_contract.get(
        "NormalPredictorFingerprintSHA256"
    )
):

    raise AssertionError(
        "Normal predictor "
        "fingerprint mismatch"
    )


if (
    list_fingerprint(
        total_predictors
    )
    !=
    generation_contract.get(
        "TotalPredictorFingerprintSHA256"
    )
):

    raise AssertionError(
        "Total predictor "
        "fingerprint mismatch"
    )


if (
    list_fingerprint(
        routing_columns
    )
    !=
    generation_contract.get(
        "RoutingFingerprintSHA256"
    )
):

    raise AssertionError(
        "Routing predictor "
        "fingerprint mismatch"
    )


# =============================================================================
# 4. LOAD DATA AND RECONCILE THE 39 LOCKED FOLDS
# =============================================================================

features = pd.read_csv(
    SELECTION_FEATURES,
    low_memory=False,
)


feature_contract = pd.read_csv(
    FEATURE_CONTRACT,
    low_memory=False,
)


baseline_ranking = pd.read_csv(
    BASELINE_RANKING,
    low_memory=False,
)


fold_membership = pd.read_csv(
    FOLD_MEMBERSHIP,
    low_memory=False,
)


for column in [
    "WeekStartDate",
    "WeekEndDate",
]:

    features[
        column
    ] = pd.to_datetime(
        features[
            column
        ],
        errors="raise",
    )


fold_membership[
    "ValidationWeek"
] = pd.to_datetime(
    fold_membership[
        "ValidationWeek"
    ],
    errors="raise",
)


features[
    "CanonicalProductID"
] = features[
    "CanonicalProductID"
].astype(str)


fold_membership[
    "CanonicalProductID"
] = fold_membership[
    "CanonicalProductID"
].astype(str)


for column in [
    "PriorStrict80PctScope",
    "PriorStrict90PctScope",
    "PriorStrict95PctScope",
    "PriorScopeAvailable",
]:

    features[
        column
    ] = bool_series(
        features[
            column
        ],
        column,
    )


for column in [
    "InFoldStrict80PctScope",
    "InFoldStrict90PctScope",
    "InFoldStrict95PctScope",
]:

    fold_membership[
        column
    ] = bool_series(
        fold_membership[
            column
        ],
        column,
    )


required_feature_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "WeeklyNormalDemand",
    "WeeklyTotalDemand",
    "PriorDemandSegment",
    *normal_predictors,
    *total_predictors,
    *routing_columns,
}


missing_feature_columns = sorted(
    required_feature_columns
    - set(
        features.columns
    )
)


if missing_feature_columns:

    raise AssertionError(
        "Feature dataset missing "
        f"columns: {missing_feature_columns}"
    )


for target_column in TARGETS.values():

    features[
        target_column
    ] = pd.to_numeric(
        features[
            target_column
        ],
        errors="raise",
    ).astype(float)

    if (
        features[
            target_column
        ]
        < 0
    ).any():

        raise AssertionError(
            "Negative target values "
            f"found in {target_column}"
        )


if features.duplicated(
    [
        "WeekStartDate",
        "CanonicalProductID",
    ]
).any():

    raise AssertionError(
        "Duplicate model-selection "
        "feature keys found"
    )


fold_scope_check = (
    fold_membership
    .merge(
        features[
            [
                "WeekStartDate",
                "CanonicalProductID",
                "PriorStrict80PctScope",
                "PriorStrict90PctScope",
                "PriorStrict95PctScope",
            ]
        ],
        left_on=[
            "ValidationWeek",
            "CanonicalProductID",
        ],
        right_on=[
            "WeekStartDate",
            "CanonicalProductID",
        ],
        how="left",
        validate="one_to_one",
    )
)


if fold_scope_check[
    "WeekStartDate"
].isna().any():

    raise AssertionError(
        "Part 11E fold rows did not "
        "all match Part 11F features"
    )


scope_mismatches = 0


for label in [
    80,
    90,
    95,
]:

    scope_mismatches += int(
        (
            fold_scope_check[
                f"InFoldStrict"
                f"{label}PctScope"
            ]
            !=
            fold_scope_check[
                f"PriorStrict"
                f"{label}PctScope"
            ]
        ).sum()
    )


if scope_mismatches != 0:

    raise AssertionError(
        "Part 11E/11F scope "
        f"mismatches: {scope_mismatches}"
    )


forecast_weeks = sorted(
    fold_membership[
        "ValidationWeek"
    ].drop_duplicates()
)


if len(
    forecast_weeks
) != 39:

    raise AssertionError(
        "Expected 39 chronological "
        f"forecast weeks, found "
        f"{len(forecast_weeks)}"
    )


fold_design_rows = []


for (
    fold_number,
    forecast_week,
) in enumerate(
    forecast_weeks,
    start=1,
):

    training = features.loc[
        features[
            "WeekStartDate"
        ]
        < forecast_week
    ]

    validation_week = features.loc[
        features[
            "WeekStartDate"
        ]
        == forecast_week
    ]

    training_weeks = int(
        training[
            "WeekStartDate"
        ].nunique()
    )

    if (
        training_weeks
        < MIN_TRAINING_WEEKS
    ):

        raise AssertionError(
            "A Part 11G fold has "
            "insufficient history"
        )

    fold_design_rows.append(
        {
            "FoldNumber":
                fold_number,

            "ForecastWeek":
                forecast_week,

            "TrainingStartWeek":
                training[
                    "WeekStartDate"
                ].min(),

            "TrainingEndWeek":
                training[
                    "WeekStartDate"
                ].max(),

            "TrainingWeeks":
                training_weeks,

            "TrainingRows":
                len(
                    training
                ),

            "ValidationRows":
                len(
                    validation_week
                ),

            "ValidationProducts":
                validation_week[
                    "CanonicalProductID"
                ].nunique(),
        }
    )


fold_design = pd.DataFrame(
    fold_design_rows
)


# =============================================================================
# 5. CHRONOLOGICAL MODEL SCREENING
# =============================================================================

prediction_frames = []


for (
    target_id,
    target_column,
) in TARGETS.items():

    predictor_columns = (
        normal_predictors
        if target_id
        == "WEEKLY_NORMAL_DEMAND"
        else total_predictors
    )

    for (
        fold_number,
        forecast_week,
    ) in enumerate(
        forecast_weeks,
        start=1,
    ):

        train = features.loc[
            features[
                "WeekStartDate"
            ]
            < forecast_week
        ].copy()

        valid = features.loc[
            features[
                "WeekStartDate"
            ]
            == forecast_week
        ].copy()

        if valid.empty:

            raise AssertionError(
                "No validation rows for "
                f"{forecast_week}"
            )

        model_predictions: dict[
            str,
            np.ndarray,
        ] = {}

        model_used: dict[
            str,
            list[str],
        ] = {}

        fallback_used: dict[
            str,
            list[bool],
        ] = {}

        ridge = build_ridge(
            predictor_columns,
            segment_feature=False,
        )

        ridge.fit(
            train[
                predictor_columns
            ],
            train[
                target_column
            ],
        )

        ridge_prediction = np.clip(
            ridge.predict(
                valid[
                    predictor_columns
                ]
            ),
            0.0,
            None,
        )

        model_predictions[
            "GLOBAL_RIDGE"
        ] = ridge_prediction

        model_used[
            "GLOBAL_RIDGE"
        ] = [
            "GLOBAL_RIDGE"
        ] * len(
            valid
        )

        fallback_used[
            "GLOBAL_RIDGE"
        ] = [
            False
        ] * len(
            valid
        )

        histgb = build_histgb(
            predictor_columns
        )

        histgb.fit(
            train[
                predictor_columns
            ],
            train[
                target_column
            ],
        )

        histgb_prediction = np.clip(
            histgb.predict(
                valid[
                    predictor_columns
                ]
            ),
            0.0,
            None,
        )

        model_predictions[
            "GLOBAL_HISTGB"
        ] = histgb_prediction

        model_used[
            "GLOBAL_HISTGB"
        ] = [
            "GLOBAL_HISTGB"
        ] * len(
            valid
        )

        fallback_used[
            "GLOBAL_HISTGB"
        ] = [
            False
        ] * len(
            valid
        )

        (
            hurdle_prediction,
            hurdle_method,
            hurdle_fallback,
        ) = fit_predict_hurdle(
            train,
            train[
                target_column
            ],
            valid,
            predictor_columns,
        )

        model_predictions[
            "GLOBAL_HURDLE_HISTGB"
        ] = hurdle_prediction

        model_used[
            "GLOBAL_HURDLE_HISTGB"
        ] = [
            hurdle_method
        ] * len(
            valid
        )

        fallback_used[
            "GLOBAL_HURDLE_HISTGB"
        ] = [
            hurdle_fallback
        ] * len(
            valid
        )

        segment_ridge = build_ridge(
            predictor_columns,
            segment_feature=True,
        )

        segment_ridge.fit(
            train[
                predictor_columns
                + [
                    "PriorDemandSegment"
                ]
            ],
            train[
                target_column
            ],
        )

        segment_ridge_prediction = (
            np.clip(
                segment_ridge.predict(
                    valid[
                        predictor_columns
                        + [
                            "PriorDemandSegment"
                        ]
                    ]
                ),
                0.0,
                None,
            )
        )

        model_predictions[
            "SEGMENT_FEATURE_RIDGE"
        ] = segment_ridge_prediction

        model_used[
            "SEGMENT_FEATURE_RIDGE"
        ] = [
            "RIDGE_WITH_SEGMENT_FEATURE"
        ] * len(
            valid
        )

        fallback_used[
            "SEGMENT_FEATURE_RIDGE"
        ] = [
            False
        ] * len(
            valid
        )

        (
            routed_prediction,
            routed_models,
            routed_fallbacks,
        ) = fit_predict_routed_histgb(
            train,
            valid,
            target_column,
            predictor_columns,
            histgb_prediction,
        )

        model_predictions[
            "ROUTED_SEGMENT_HISTGB"
        ] = routed_prediction

        model_used[
            "ROUTED_SEGMENT_HISTGB"
        ] = routed_models

        fallback_used[
            "ROUTED_SEGMENT_HISTGB"
        ] = routed_fallbacks

        for system_id in CANDIDATE_SYSTEMS:

            frame = valid[
                [
                    "WeekStartDate",
                    "WeekEndDate",
                    "WeekID",
                    "CanonicalProductID",
                    "CanonicalProductName",
                    "PriorDemandSegment",
                    "PriorStrict80PctScope",
                    "PriorStrict90PctScope",
                    "PriorStrict95PctScope",
                ]
            ].copy()

            frame[
                "FoldNumber"
            ] = fold_number

            frame[
                "TrainingWeeks"
            ] = int(
                train[
                    "WeekStartDate"
                ].nunique()
            )

            frame[
                "TargetID"
            ] = target_id

            frame[
                "TargetColumn"
            ] = target_column

            frame[
                "CandidateSystemID"
            ] = system_id

            frame[
                "PredictionModelUsed"
            ] = model_used[
                system_id
            ]

            frame[
                "FallbackUsed"
            ] = fallback_used[
                system_id
            ]

            frame[
                "Actual"
            ] = valid[
                target_column
            ].to_numpy(
                dtype=float
            )

            frame[
                "Prediction"
            ] = np.clip(
                model_predictions[
                    system_id
                ],
                0.0,
                None,
            )

            frame[
                "Error"
            ] = (
                frame[
                    "Prediction"
                ]
                - frame[
                    "Actual"
                ]
            )

            frame[
                "AbsoluteError"
            ] = frame[
                "Error"
            ].abs()

            frame[
                "SelectionEligible"
            ] = True

            frame[
                "EvaluationPurpose"
            ] = (
                "PREHOLDOUT_CHRONOLOGICAL_"
                "MODEL_SCREENING"
            )

            prediction_frames.append(
                frame
            )


predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)


if (
    predictions[
        "Prediction"
    ].isna().any()
    or not np.isfinite(
        predictions[
            "Prediction"
        ]
    ).all()
):

    raise AssertionError(
        "Non-finite candidate "
        "predictions were created"
    )


if (
    predictions[
        "Prediction"
    ]
    < 0
).any():

    raise AssertionError(
        "Negative candidate "
        "predictions were created"
    )


expected_prediction_rows = (
    sum(
        int(
            (
                features[
                    "WeekStartDate"
                ]
                == week
            ).sum()
        )
        for week
        in forecast_weeks
    )
    * len(
        TARGETS
    )
    * len(
        CANDIDATE_SYSTEMS
    )
)


if (
    len(
        predictions
    )
    != expected_prediction_rows
):

    raise AssertionError(
        "Prediction row mismatch: "
        f"{len(predictions)} != "
        f"{expected_prediction_rows}"
    )


# =============================================================================
# 6. FOLD AND POOLED METRICS
# =============================================================================

fold_metric_rows = []

pooled_metric_rows = []


for (
    fold_number,
    forecast_week,
    target_id,
    system_id,
), group in predictions.groupby(
    [
        "FoldNumber",
        "WeekStartDate",
        "TargetID",
        "CandidateSystemID",
    ],
    sort=True,
):

    for (
        scope_id,
        flag_column,
    ) in SCOPE_FLAGS.items():

        scoped = (
            group
            if flag_column is None
            else group.loc[
                group[
                    flag_column
                ]
            ]
        )

        fold_metric_rows.append(
            {
                "FoldNumber":
                    int(
                        fold_number
                    ),

                "ForecastWeek":
                    forecast_week,

                "TargetID":
                    target_id,

                "ScopeID":
                    scope_id,

                "CandidateSystemID":
                    system_id,

                **metric_record(
                    scoped[
                        "Actual"
                    ],
                    scoped[
                        "Prediction"
                    ],
                ),
            }
        )


for (
    target_id,
    system_id,
), group in predictions.groupby(
    [
        "TargetID",
        "CandidateSystemID",
    ],
    sort=True,
):

    for (
        scope_id,
        flag_column,
    ) in SCOPE_FLAGS.items():

        scoped = (
            group
            if flag_column is None
            else group.loc[
                group[
                    flag_column
                ]
            ]
        )

        pooled_metric_rows.append(
            {
                "EvaluationPurpose":
                    (
                        "PREHOLDOUT_CHRONOLOGICAL_"
                        "MODEL_SCREENING"
                    ),

                "SelectionEligible":
                    True,

                "TargetID":
                    target_id,

                "ScopeID":
                    scope_id,

                "CandidateSystemID":
                    system_id,

                "ForecastWeeks":
                    int(
                        scoped[
                            "WeekStartDate"
                        ].nunique()
                    ),

                "Products":
                    int(
                        scoped[
                            "CanonicalProductID"
                        ].nunique()
                    ),

                **metric_record(
                    scoped[
                        "Actual"
                    ],
                    scoped[
                        "Prediction"
                    ],
                ),
            }
        )


fold_metrics = pd.DataFrame(
    fold_metric_rows
)


pooled_metrics = pd.DataFrame(
    pooled_metric_rows
)


# =============================================================================
# 7. COMPARE TO LOCKED PART 11E REFERENCE BASELINES
# =============================================================================

baseline_reference = (
    baseline_ranking.loc[
        baseline_ranking[
            "IsReferenceBaseline"
        ]
        .astype(str)
        .str.lower()
        .isin(
            [
                "true",
                "1",
            ]
        )
    ]
    .copy()
)


required_baseline_columns = {
    "TargetID",
    "ScopeID",
    "BaselineMethod",
    "Observations",
    "ActualTotal",
    "PredictedTotal",
    "MAE",
    "RMSE",
    "WAPEPercentage",
    "MeanBias",
    "TotalBias",
}


if not required_baseline_columns.issubset(
    baseline_reference.columns
):

    raise AssertionError(
        "Part 11E baseline reference "
        "schema is incomplete"
    )


if len(
    baseline_reference
) != (
    len(
        TARGETS
    )
    * len(
        SCOPE_FLAGS
    )
):

    raise AssertionError(
        "Expected exactly eight "
        "Part 11E reference baselines"
    )


comparison = (
    pooled_metrics
    .merge(
        baseline_reference[
            [
                "TargetID",
                "ScopeID",
                "BaselineMethod",
                "WAPEPercentage",
                "MAE",
                "RMSE",
                "TotalBias",
            ]
        ].rename(
            columns={
                "BaselineMethod":
                    "ReferenceBaselineMethod",

                "WAPEPercentage":
                    (
                        "ReferenceBaseline"
                        "WAPEPercentage"
                    ),

                "MAE":
                    "ReferenceBaselineMAE",

                "RMSE":
                    "ReferenceBaselineRMSE",

                "TotalBias":
                    (
                        "ReferenceBaseline"
                        "TotalBias"
                    ),
            }
        ),
        on=[
            "TargetID",
            "ScopeID",
        ],
        how="left",
        validate="many_to_one",
    )
)


if comparison[
    "ReferenceBaselineMethod"
].isna().any():

    raise AssertionError(
        "Candidate metrics did not all "
        "receive a baseline reference"
    )


comparison[
    "WAPEImprovementPercentagePoints"
] = (
    comparison[
        "ReferenceBaselineWAPEPercentage"
    ]
    - comparison[
        "WAPEPercentage"
    ]
)


comparison[
    "RelativeWAPEReductionPercentage"
] = np.where(
    comparison[
        "ReferenceBaselineWAPEPercentage"
    ]
    != 0,
    (
        100.0
        * comparison[
            "WAPEImprovementPercentagePoints"
        ]
        / comparison[
            "ReferenceBaselineWAPEPercentage"
        ]
    ),
    np.nan,
)


comparison[
    "BeatReferenceBaselineOnWAPE"
] = (
    comparison[
        "WAPEPercentage"
    ]
    < comparison[
        "ReferenceBaselineWAPEPercentage"
    ]
)


comparison[
    "AbsoluteTotalBias"
] = comparison[
    "TotalBias"
].abs()


comparison[
    "ComplexityOrder"
] = comparison[
    "CandidateSystemID"
].map(
    COMPLEXITY_ORDER
)


ranking = (
    comparison
    .sort_values(
        [
            "TargetID",
            "ScopeID",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "AbsoluteTotalBias",
            "ComplexityOrder",
            "CandidateSystemID",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


ranking[
    "CandidateRank"
] = (
    ranking
    .groupby(
        [
            "TargetID",
            "ScopeID",
        ]
    )
    .cumcount()
    + 1
)


ranking[
    "IsScreeningLeader"
] = (
    ranking[
        "CandidateRank"
    ]
    == 1
)


ranking[
    "SelectionStatus"
] = (
    "SCREENING_ONLY_"
    "NOT_FINAL_MODEL_SELECTION"
)


# =============================================================================
# 8. ROUTING AND FALLBACK AUDIT
# =============================================================================

routing_audit = (
    predictions
    .groupby(
        [
            "TargetID",
            "CandidateSystemID",
            "PriorDemandSegment",
            "PredictionModelUsed",
            "FallbackUsed",
        ],
        as_index=False,
    )
    .agg(
        PredictionRows=(
            "Prediction",
            "size",
        ),

        ForecastWeeks=(
            "WeekStartDate",
            "nunique",
        ),

        Products=(
            "CanonicalProductID",
            "nunique",
        ),

        ActualTotal=(
            "Actual",
            "sum",
        ),

        PredictedTotal=(
            "Prediction",
            "sum",
        ),
    )
)


# =============================================================================
# 9. VALIDATION
# =============================================================================

screening_leaders = ranking.loc[
    ranking[
        "IsScreeningLeader"
    ]
].copy()


expected_metric_groups = (
    len(
        TARGETS
    )
    * len(
        SCOPE_FLAGS
    )
    * len(
        CANDIDATE_SYSTEMS
    )
)


expected_leaders = (
    len(
        TARGETS
    )
    * len(
        SCOPE_FLAGS
    )
)


validation = pd.DataFrame(
    [
        {
            "Check":
                (
                    "Part 11F lock and "
                    "checkpoint verified"
                ),

            "Expected":
                True,

            "Actual":
                True,

            "Passed":
                True,
        },
        {
            "Check":
                (
                    "Part 11E lock verified "
                    "and referenced by Part 11F"
                ),

            "Expected":
                part11e_lock_hash,

            "Actual":
                part11f_lock
                .get(
                    "Part11E",
                    {},
                )
                .get(
                    "LockSHA256"
                ),

            "Passed":
                (
                    part11f_lock
                    .get(
                        "Part11E",
                        {},
                    )
                    .get(
                        "LockSHA256"
                    )
                    == part11e_lock_hash
                ),
        },
        {
            "Check":
                (
                    "Model-selection feature hash "
                    "matched Part 11F lock"
                ),

            "Expected":
                locked_selection_hash,

            "Actual":
                selection_hash,

            "Passed":
                (
                    selection_hash
                    == locked_selection_hash
                ),
        },
        {
            "Check":
                (
                    "Normal predictor "
                    "fingerprint matched"
                ),

            "Expected":
                generation_contract.get(
                    "NormalPredictorFingerprintSHA256"
                ),

            "Actual":
                list_fingerprint(
                    normal_predictors
                ),

            "Passed":
                (
                    list_fingerprint(
                        normal_predictors
                    )
                    ==
                    generation_contract.get(
                        "NormalPredictorFingerprintSHA256"
                    )
                ),
        },
        {
            "Check":
                (
                    "Total predictor "
                    "fingerprint matched"
                ),

            "Expected":
                generation_contract.get(
                    "TotalPredictorFingerprintSHA256"
                ),

            "Actual":
                list_fingerprint(
                    total_predictors
                ),

            "Passed":
                (
                    list_fingerprint(
                        total_predictors
                    )
                    ==
                    generation_contract.get(
                        "TotalPredictorFingerprintSHA256"
                    )
                ),
        },
        {
            "Check":
                (
                    "Part 11E and Part 11F "
                    "scope mismatches"
                ),

            "Expected":
                0,

            "Actual":
                scope_mismatches,

            "Passed":
                scope_mismatches == 0,
        },
        {
            "Check":
                "Chronological validation folds",

            "Expected":
                39,

            "Actual":
                len(
                    forecast_weeks
                ),

            "Passed":
                len(
                    forecast_weeks
                )
                == 39,
        },
        {
            "Check":
                "Minimum training weeks",

            "Expected":
                MIN_TRAINING_WEEKS,

            "Actual":
                int(
                    fold_design[
                        "TrainingWeeks"
                    ].min()
                ),

            "Passed":
                (
                    int(
                        fold_design[
                            "TrainingWeeks"
                        ].min()
                    )
                    >= MIN_TRAINING_WEEKS
                ),
        },
        {
            "Check":
                "Candidate prediction rows",

            "Expected":
                expected_prediction_rows,

            "Actual":
                len(
                    predictions
                ),

            "Passed":
                (
                    len(
                        predictions
                    )
                    == expected_prediction_rows
                ),
        },
        {
            "Check":
                (
                    "Non-finite or negative "
                    "candidate predictions"
                ),

            "Expected":
                0,

            "Actual":
                (
                    int(
                        (
                            ~np.isfinite(
                                predictions[
                                    "Prediction"
                                ]
                            )
                        ).sum()
                    )
                    + int(
                        (
                            predictions[
                                "Prediction"
                            ]
                            < 0
                        ).sum()
                    )
                ),

            "Passed":
                (
                    np.isfinite(
                        predictions[
                            "Prediction"
                        ]
                    ).all()
                    and (
                        predictions[
                            "Prediction"
                        ]
                        >= 0
                    ).all()
                ),
        },
        {
            "Check":
                (
                    "Pooled candidate "
                    "metric groups"
                ),

            "Expected":
                expected_metric_groups,

            "Actual":
                len(
                    pooled_metrics
                ),

            "Passed":
                (
                    len(
                        pooled_metrics
                    )
                    == expected_metric_groups
                ),
        },
        {
            "Check":
                (
                    "Exactly one screening "
                    "leader per target and scope"
                ),

            "Expected":
                expected_leaders,

            "Actual":
                len(
                    screening_leaders
                ),

            "Passed":
                (
                    len(
                        screening_leaders
                    )
                    == expected_leaders
                ),
        },
        {
            "Check":
                (
                    "All candidate metrics linked "
                    "to reference baseline"
                ),

            "Expected":
                expected_metric_groups,

            "Actual":
                int(
                    comparison[
                        "ReferenceBaselineMethod"
                    ].notna().sum()
                ),

            "Passed":
                comparison[
                    "ReferenceBaselineMethod"
                ].notna().all(),
        },
        {
            "Check":
                "Opened March 2026 targets used",

            "Expected":
                0,

            "Actual":
                0,

            "Passed":
                True,
        },
        {
            "Check":
                "Forecasts rounded before scoring",

            "Expected":
                False,

            "Actual":
                False,

            "Passed":
                True,
        },
        {
            "Check":
                (
                    "Final weekly model "
                    "selected or refitted"
                ),

            "Expected":
                False,

            "Actual":
                False,

            "Passed":
                True,
        },
    ]
)


if not validation[
    "Passed"
].all():

    raise AssertionError(
        "Part 11G validation failed:\n"
        + validation.loc[
            ~validation[
                "Passed"
            ]
        ].to_string(
            index=False
        )
    )


# =============================================================================
# 10. SCREENING CONTRACT AND MODEL CONFIGURATION TABLES
# =============================================================================

screening_contract = {
    "StepID":
        STEP_ID,

    "CreatedLocal":
        NOW_LOCAL.isoformat(),

    "Purpose":
        (
            "Chronological screening of "
            "pooled, hurdle, segment-feature "
            "and routed weekly candidate systems."
        ),

    "SelectionStatus":
        (
            "SCREENING_ONLY_"
            "NOT_FINAL_MODEL_SELECTION"
        ),

    "Targets":
        TARGETS,

    "Scopes":
        list(
            SCOPE_FLAGS
        ),

    "Folds":
        39,

    "MinimumTrainingWeeks":
        MIN_TRAINING_WEEKS,

    "CandidateSystems":
        CANDIDATE_SYSTEMS,

    "RankingRule":
        (
            "WAPE ascending, then MAE, "
            "RMSE, absolute total bias, "
            "model complexity and system ID."
        ),

    "PredictorContracts":
        {
            "WeeklyNormalDemand":
                {
                    "Count":
                        len(
                            normal_predictors
                        ),

                    "FingerprintSHA256":
                        list_fingerprint(
                            normal_predictors
                        ),
                },

            "WeeklyTotalDemand":
                {
                    "Count":
                        len(
                            total_predictors
                        ),

                    "FingerprintSHA256":
                        list_fingerprint(
                            total_predictors
                        ),
                },
        },

    "ModelConfigurations":
        {
            "GLOBAL_RIDGE":
                (
                    "Median imputation with "
                    "missing indicators, "
                    "standardisation and "
                    "Ridge(alpha=10)."
                ),

            "GLOBAL_HISTGB":
                (
                    "Median imputation with "
                    "missing indicators and fixed "
                    "HistGradientBoostingRegressor."
                ),

            "GLOBAL_HURDLE_HISTGB":
                (
                    "Logistic occurrence probability "
                    "multiplied by a log1p HistGB "
                    "positive-quantity estimate."
                ),

            "SEGMENT_FEATURE_RIDGE":
                (
                    "Shared Ridge model with "
                    "one-hot encoded "
                    "PriorDemandSegment."
                ),

            "ROUTED_SEGMENT_HISTGB":
                (
                    "Separate high- and moderate-"
                    "demand HistGB models with the "
                    "pooled global HistGB forecast "
                    "used outside those segments "
                    "or when segment history is "
                    "insufficient."
                ),
        },

    "ProductSpecificModels":
        {
            "Included":
                False,

            "Reason":
                (
                    "Only 47 eligible weeks are "
                    "available. Product-specific "
                    "fitting is deferred until "
                    "pooled and routed systems are "
                    "screened because each product "
                    "would have very limited "
                    "chronological training history."
                ),
        },

    "HyperparameterSearchPerformed":
        False,

    "OpenedHoldoutPolicy":
        (
            "March 2026 targets were not "
            "read or used in Part 11G."
        ),

    "FinalModelSelected":
        False,
}


global_config = pd.DataFrame(
    [
        {
            "CandidateSystemID":
                "GLOBAL_RIDGE",

            "Family":
                "LINEAR_REGRESSION",

            "ScopeTraining":
                "ALL_AVAILABLE_TRAINING_ROWS",

            "SegmentHandling":
                "NONE",

            "FixedConfiguration":
                (
                    "median imputer + missing "
                    "indicators + standard scaler "
                    "+ Ridge(alpha=10)"
                ),
        },
        {
            "CandidateSystemID":
                "GLOBAL_HISTGB",

            "Family":
                "TREE_ENSEMBLE",

            "ScopeTraining":
                "ALL_AVAILABLE_TRAINING_ROWS",

            "SegmentHandling":
                "NONE",

            "FixedConfiguration":
                (
                    "median imputer + missing "
                    "indicators + HistGB("
                    "lr=.05,max_iter=120,"
                    "max_leaf_nodes=15,"
                    "min_samples_leaf=20,l2=1)"
                ),
        },
        {
            "CandidateSystemID":
                "GLOBAL_HURDLE_HISTGB",

            "Family":
                "TWO_PART_HURDLE",

            "ScopeTraining":
                "ALL_AVAILABLE_TRAINING_ROWS",

            "SegmentHandling":
                "NONE",

            "FixedConfiguration":
                (
                    "balanced logistic occurrence "
                    "x log1p HistGB "
                    "positive quantity"
                ),
        },
        {
            "CandidateSystemID":
                "SEGMENT_FEATURE_RIDGE",

            "Family":
                "LINEAR_REGRESSION",

            "ScopeTraining":
                "ALL_AVAILABLE_TRAINING_ROWS",

            "SegmentHandling":
                (
                    "PriorDemandSegment "
                    "one-hot encoded"
                ),

            "FixedConfiguration":
                (
                    "numeric median imputer + "
                    "standard scaler + segment "
                    "one-hot + Ridge(alpha=10)"
                ),
        },
    ]
)


high_config = pd.DataFrame(
    [
        {
            "Segment":
                "HIGH_PRIOR_DEMAND",

            "CandidateSystemID":
                "ROUTED_SEGMENT_HISTGB",

            "MinimumTrainingRows":
                MIN_SEGMENT_TRAINING_ROWS,

            "Fallback":
                "GLOBAL_HISTGB",

            "Configuration":
                "fixed HistGB configuration",
        }
    ]
)


moderate_config = pd.DataFrame(
    [
        {
            "Segment":
                "MODERATE_PRIOR_DEMAND",

            "CandidateSystemID":
                "ROUTED_SEGMENT_HISTGB",

            "MinimumTrainingRows":
                MIN_SEGMENT_TRAINING_ROWS,

            "Fallback":
                "GLOBAL_HISTGB",

            "Configuration":
                "fixed HistGB configuration",
        }
    ]
)


routed_config = pd.DataFrame(
    [
        {
            "CandidateSystemID":
                "ROUTED_SEGMENT_HISTGB",

            "RoutingField":
                "PriorDemandSegment",

            "Segments":
                (
                    "HIGH_PRIOR_DEMAND;"
                    "MODERATE_PRIOR_DEMAND"
                ),

            "MinimumSegmentTrainingRows":
                MIN_SEGMENT_TRAINING_ROWS,

            "Fallback":
                "GLOBAL_HISTGB",

            "ScopeMembershipRule":
                (
                    "recalculated from prior "
                    "normal-demand history for "
                    "every forecast week"
                ),
        }
    ]
)


# =============================================================================
# 11. STAGE OUTPUTS, MEMORY, CHECKPOINT AND LOCK
# =============================================================================

stage_root = (
    EXT_ROOT
    / f".11G_staging_{uuid.uuid4().hex}"
)


stage_root.mkdir(
    parents=True,
    exist_ok=False,
)


try:

    for path, frame in [
        (
            PREDICTIONS_PATH,
            predictions,
        ),
        (
            FOLD_METRICS_PATH,
            fold_metrics,
        ),
        (
            POOLED_METRICS_PATH,
            pooled_metrics,
        ),
        (
            COMPARISON_PATH,
            comparison,
        ),
        (
            RANKING_PATH,
            ranking,
        ),
        (
            FOLD_DESIGN_PATH,
            fold_design,
        ),
        (
            ROUTING_AUDIT_PATH,
            routing_audit,
        ),
        (
            VALIDATION_PATH,
            validation,
        ),
        (
            GLOBAL_CONFIG_PATH,
            global_config,
        ),
        (
            HIGH_CONFIG_PATH,
            high_config,
        ),
        (
            MODERATE_CONFIG_PATH,
            moderate_config,
        ),
        (
            ROUTED_CONFIG_PATH,
            routed_config,
        ),
    ]:

        stage_csv(
            stage_root,
            path,
            frame,
        )


    stage_json(
        stage_root,
        SCREENING_CONTRACT_PATH,
        screening_contract,
    )


    leader_lines = []


    for row in screening_leaders.itertuples(
        index=False
    ):

        leader_lines.append(
            f"- {row.TargetID} / "
            f"{row.ScopeID}: "
            f"{row.CandidateSystemID}; "
            f"WAPE "
            f"{row.WAPEPercentage:.6f}%; "
            f"reference "
            f"{row.ReferenceBaselineMethod} "
            f"{row.ReferenceBaselineWAPEPercentage:.6f}%; "
            f"improvement "
            f"{row.WAPEImprovementPercentagePoints:.6f} "
            "percentage points."
        )


    leader_summary = "\n".join(
        leader_lines
    )


    decisions = "\n".join(
        [
            (
                f"- Decision date: "
                f"{NOW_LOCAL.date().isoformat()}."
            ),
            (
                "- Part 11G is a screening stage; "
                "no final weekly model is selected."
            ),
            (
                "- The exact 39 Part 11E "
                "chronological forecast weeks "
                "are reused."
            ),
            (
                "- Models are refitted separately "
                "inside every fold using only "
                "earlier weeks."
            ),
            (
                "- Both WeeklyNormalDemand and "
                "WeeklyTotalDemand are screened "
                "for all, 80%, 90% and 95% scopes."
            ),
            (
                "- Five fixed candidate systems "
                "are tested without "
                "hyperparameter search."
            ),
            (
                "- Product ID remains excluded "
                "as a primary direct predictor."
            ),
            (
                "- Scope and segment fields are "
                "recalculated from prior normal-"
                "demand history for each "
                "forecast week."
            ),
            (
                "- Product-specific models are "
                "deferred because only 47 eligible "
                "weeks are available per product."
            ),
            (
                "- WAPE is the primary screening "
                "metric; simpler models are "
                "preferred when performance is "
                "effectively tied."
            ),
            (
                "- March 2026 targets are not "
                "read or used."
            ),
        ]
    )


    files = "\n".join(
        [
            (
                "- Candidate predictions: "
                f"`{PREDICTIONS_PATH}`"
            ),
            (
                "- Fold metrics: "
                f"`{FOLD_METRICS_PATH}`"
            ),
            (
                "- Pooled metrics: "
                f"`{POOLED_METRICS_PATH}`"
            ),
            (
                "- Model versus baseline "
                f"comparison: `{COMPARISON_PATH}`"
            ),
            (
                "- Candidate ranking: "
                f"`{RANKING_PATH}`"
            ),
            (
                "- Routing audit: "
                f"`{ROUTING_AUDIT_PATH}`"
            ),
            (
                "- Screening contract: "
                f"`{SCREENING_CONTRACT_PATH}`"
            ),
            (
                "- Validation: "
                f"`{VALIDATION_PATH}`"
            ),
        ]
    )


    results = "\n".join(
        [
            (
                f"- Chronological folds: "
                f"{len(forecast_weeks)}."
            ),
            (
                f"- Candidate systems: "
                f"{len(CANDIDATE_SYSTEMS)}."
            ),
            (
                f"- Targets: "
                f"{len(TARGETS)}."
            ),
            (
                f"- Candidate prediction rows: "
                f"{len(predictions):,}."
            ),
            (
                f"- Pooled metric groups: "
                f"{len(pooled_metrics)}."
            ),
            "- Screening leaders:",
            leader_summary,
        ]
    )


    for key, final_path in MEMORY_FILES.items():

        original = final_path.read_text(
            encoding="utf-8"
        )


        if key == "PROJECT_CONTEXT":

            updated = update_section(
                original,
                "STEP_11G",
                (
                    "Part 11G weekly "
                    "model screening"
                ),
                (
                    "Five pooled, hurdle, "
                    "segment-feature and routed "
                    "candidate systems were "
                    "chronologically screened "
                    "against the Part 11E "
                    "baselines. No final weekly "
                    "model was selected."
                ),
            )


        elif key == "WORKFLOW":

            updated = update_section(
                original,
                "STEP_11G",
                (
                    "Part 11G "
                    "workflow status"
                ),
                (
                    "Part 11G is complete. "
                    "Part 11H will compare the "
                    "WeeklyNormalDemand and "
                    "WeeklyTotalDemand experiments, "
                    "including bulk-demand "
                    "implications, before any final "
                    "approach is selected in "
                    "Part 11I."
                ),
            )


        elif key == "DECISIONS":

            updated = update_section(
                original,
                "STEP_11G",
                "Part 11G decisions",
                decisions,
            )


        elif key == "FILES_AND_PATHS":

            updated = update_section(
                original,
                "STEP_11G",
                "Part 11G files",
                files,
            )


        elif key == "METRICS_AND_RESULTS":

            updated = update_section(
                original,
                "STEP_11G",
                (
                    "Part 11G metrics "
                    "and results"
                ),
                results,
            )


        elif key == "CHAT_INDEX":

            row = (
                f"| {NOW_LOCAL.isoformat()} | "
                "11G | Candidate weekly model "
                f"screening | {STATUS} |"
            )

            updated = (
                original
                if row in original
                else (
                    original.rstrip()
                    + "\n"
                    + row
                    + "\n"
                )
            )


        elif key == "CURRENT_HANDOFF":

            updated = "\n".join(
                [
                    "# Current Handoff",
                    "",
                    (
                        "- Current completed step: "
                        f"{STEP_ID}"
                    ),
                    (
                        f"- Status: {STATUS}"
                    ),
                    (
                        "- Updated local time: "
                        f"{NOW_LOCAL.isoformat()}"
                    ),
                    (
                        "- Candidate ranking: "
                        f"{RANKING_PATH}"
                    ),
                    (
                        "- Model-versus-baseline "
                        f"comparison: {COMPARISON_PATH}"
                    ),
                    (
                        "- Screening contract: "
                        f"{SCREENING_CONTRACT_PATH}"
                    ),
                    (
                        "- Part 11G leaders are "
                        "screening results only, "
                        "not final selected models."
                    ),
                    (
                        "- March 2026 targets "
                        "were not used."
                    ),
                    (
                        "- Next step: 11H compare "
                        "WeeklyNormalDemand and "
                        "WeeklyTotalDemand experiments "
                        "and bulk-demand treatment."
                    ),
                    "",
                ]
            )


        else:

            raise KeyError(
                key
            )


        stage_text(
            stage_root,
            final_path,
            updated,
        )


    step_text = f"""# Step 11G — Candidate Weekly Model Screening

- **Step ID:** 11G
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## User request
Train and chronologically validate candidate global and segmented weekly models.

## Technical actions
- Verified the Part 11F feature lock, checkpoint, feature hashes and predictor fingerprints.
- Reused the exact 39 chronological forecast weeks from Part 11E.
- Refitted every candidate separately inside every fold using only earlier weeks.
- Screened global Ridge, global HistGradientBoosting, a hurdle HistGradientBoosting system, a shared Ridge model with demand segment as a feature and a routed segment-specific HistGradientBoosting system.
- Evaluated both WeeklyNormalDemand and WeeklyTotalDemand across all products and fold-specific 80%, 90% and 95% scopes.
- Compared every candidate to the locked Part 11E NAIVE_LAST_CONTEXT reference baseline.
- Recorded the routed segment and fallback model used for every prediction.

## Decisions
{decisions}

## Inputs
- `{SELECTION_FEATURES}`
- `{GENERATION_CONTRACT}`
- `{PART11F_LOCK}`
- `{BASELINE_RANKING}`
- `{FOLD_MEMBERSHIP}`

## Outputs
{files}

## Validation
All {len(validation)} Part 11G validation checks passed.

## Results
{results}

## Limitations
- Part 11G is a fixed-configuration screening stage and performs no hyperparameter optimisation.
- Product-specific models were deferred because the available per-product weekly history is short.
- Screening leaders are not final selected models.
- March 2026 remains excluded from model screening and selection.
- A new untouched future period is still required for unbiased final weekly evaluation.

## Next action
Part 11H: compare the WeeklyNormalDemand and WeeklyTotalDemand experiments and determine how bulk demand should be handled before final method selection.
"""


    stage_text(
        stage_root,
        STEP_MEMORY_PATH,
        step_text,
    )


    checkpoint = {
        "StepID":
            STEP_ID,

        "Status":
            STATUS,

        "CreatedUTC":
            NOW_UTC.isoformat(),

        "CreatedLocal":
            NOW_LOCAL.isoformat(),

        "Input":
            {
                "ModelSelectionFeaturesPath":
                    str(
                        SELECTION_FEATURES
                    ),

                "ModelSelectionFeaturesSHA256":
                    selection_hash,

                "Part11FLockSHA256":
                    part11f_lock_hash,

                "Part11FCheckpointSHA256":
                    part11f_checkpoint_hash,

                "Part11ELockSHA256":
                    part11e_lock_hash,
            },

        "Screening":
            {
                "Folds":
                    len(
                        forecast_weeks
                    ),

                "Targets":
                    list(
                        TARGETS
                    ),

                "Scopes":
                    list(
                        SCOPE_FLAGS
                    ),

                "CandidateSystems":
                    CANDIDATE_SYSTEMS,

                "PredictionRows":
                    len(
                        predictions
                    ),

                "PooledMetricGroups":
                    len(
                        pooled_metrics
                    ),

                "Leaders":
                    screening_leaders.to_dict(
                        orient="records"
                    ),

                "FinalModelSelected":
                    False,
            },

        "Safety":
            {
                "ProtectedTargetVaultOpened":
                    False,

                "ProtectedTargetVaultCopied":
                    False,

                "OpenedMarch2026TargetsRead":
                    False,

                "ProductIDUsedAsPrimaryDirectPredictor":
                    False,

                "HyperparameterSearchPerformed":
                    False,

                "ModelsFittedInsideChronologicalFolds":
                    True,

                "FinalModelsRefitted":
                    False,

                "FinalPredictionsCreated":
                    False,

                "ForecastsRoundedBeforeScoring":
                    False,
            },

        "ReadyForPart11H":
            True,

        "NextStep":
            "11H",
    }


    stage_json(
        stage_root,
        CHECKPOINT_PATH,
        checkpoint,
    )


    checkpoint_hash = sha256_file(
        stage_path(
            stage_root,
            CHECKPOINT_PATH,
        )
    )


    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        (
            f"{checkpoint_hash}  "
            f"{CHECKPOINT_PATH.name}\n"
        ),
    )


    manifest_targets = [
        PREDICTIONS_PATH,
        FOLD_METRICS_PATH,
        POOLED_METRICS_PATH,
        COMPARISON_PATH,
        RANKING_PATH,
        FOLD_DESIGN_PATH,
        ROUTING_AUDIT_PATH,
        SCREENING_CONTRACT_PATH,
        VALIDATION_PATH,
        GLOBAL_CONFIG_PATH,
        HIGH_CONFIG_PATH,
        MODERATE_CONFIG_PATH,
        ROUTED_CONFIG_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]


    manifest = pd.DataFrame(
        [
            {
                "RelativePath":
                    str(
                        path.relative_to(
                            EXT_ROOT
                        )
                    ),

                "Bytes":
                    stage_path(
                        stage_root,
                        path,
                    ).stat().st_size,

                "SHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            path,
                        )
                    ),
            }
            for path in manifest_targets
        ]
    ).sort_values(
        "RelativePath"
    )


    stage_csv(
        stage_root,
        MANIFEST_PATH,
        manifest,
    )


    manifest_hash = sha256_file(
        stage_path(
            stage_root,
            MANIFEST_PATH,
        )
    )


    lock = {
        "StepID":
            STEP_ID,

        "Status":
            STATUS,

        "CreatedUTC":
            NOW_UTC.isoformat(),

        "Part11F":
            {
                "LockSHA256":
                    part11f_lock_hash,

                "CheckpointSHA256":
                    part11f_checkpoint_hash,

                "ModelSelectionFeaturesSHA256":
                    selection_hash,

                "NormalPredictorFingerprintSHA256":
                    list_fingerprint(
                        normal_predictors
                    ),

                "TotalPredictorFingerprintSHA256":
                    list_fingerprint(
                        total_predictors
                    ),
            },

        "Part11E":
            {
                "LockSHA256":
                    part11e_lock_hash,

                "ReferenceBaselineRankingPath":
                    str(
                        BASELINE_RANKING
                    ),

                "ReferenceBaselineRankingSHA256":
                    sha256_file(
                        BASELINE_RANKING
                    ),
            },

        "Screening":
            {
                "ContractPath":
                    str(
                        SCREENING_CONTRACT_PATH
                    ),

                "ContractSHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            SCREENING_CONTRACT_PATH,
                        )
                    ),

                "CandidateRankingPath":
                    str(
                        RANKING_PATH
                    ),

                "CandidateRankingSHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            RANKING_PATH,
                        )
                    ),

                "Leaders":
                    screening_leaders.to_dict(
                        orient="records"
                    ),

                "FinalModelSelected":
                    False,
            },

        "OutputHashManifest":
            {
                "Path":
                    str(
                        MANIFEST_PATH
                    ),

                "SHA256":
                    manifest_hash,

                "FilesListed":
                    len(
                        manifest
                    ),
            },

        "Checkpoint":
            {
                "Path":
                    str(
                        CHECKPOINT_PATH
                    ),

                "SHA256":
                    checkpoint_hash,
            },

        "SafetyAssertions":
            checkpoint[
                "Safety"
            ],

        "ReadyForPart11H":
            True,

        "NextStep":
            "11H",
    }


    stage_json(
        stage_root,
        LOCK_PATH,
        lock,
    )


    lock_hash = sha256_file(
        stage_path(
            stage_root,
            LOCK_PATH,
        )
    )


    stage_text(
        stage_root,
        LOCK_SHA_PATH,
        (
            f"{lock_hash}  "
            f"{LOCK_PATH.name}\n"
        ),
    )


    stage_text(
        stage_root,
        LOG_PATH,
        "\n".join(
            [
                f"Status: {STATUS}",
                (
                    f"Run local: "
                    f"{NOW_LOCAL.isoformat()}"
                ),
                (
                    "Model-selection features "
                    f"SHA256: {selection_hash}"
                ),
                (
                    f"Chronological folds: "
                    f"{len(forecast_weeks)}"
                ),
                (
                    f"Candidate systems: "
                    f"{len(CANDIDATE_SYSTEMS)}"
                ),
                (
                    f"Prediction rows: "
                    f"{len(predictions)}"
                ),
                (
                    f"Pooled metric groups: "
                    f"{len(pooled_metrics)}"
                ),
                (
                    "Opened March 2026 "
                    "targets read: False"
                ),
                (
                    "Final weekly model "
                    "selected: False"
                ),
                (
                    f"Checkpoint SHA256: "
                    f"{checkpoint_hash}"
                ),
                (
                    f"Part 11G lock SHA256: "
                    f"{lock_hash}"
                ),
                "",
            ]
        ),
    )


    staged_files = [
        path
        for path in stage_root.rglob("*")
        if path.is_file()
    ]


    if not staged_files:

        raise AssertionError(
            "Part 11G staging "
            "directory is empty"
        )


    if any(
        (
            "target_vault"
            in path.name.lower()
        )
        or (
            path.suffix.lower()
            == ".joblib"
        )
        for path in staged_files
    ):

        raise PermissionError(
            "Forbidden target-vault "
            "or persisted-model file "
            "detected in staging"
        )


    for staged in sorted(
        staged_files
    ):

        final = (
            EXT_ROOT
            / staged.relative_to(
                stage_root
            )
        )

        final.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if (
            final.exists()
            and final
            not in MEMORY_FILES.values()
        ):

            raise FileExistsError(
                "Refusing to overwrite "
                f"Part 11G output: {final}"
            )

        os.replace(
            staged,
            final,
        )


    for path in NEW_OUTPUTS:

        make_read_only(
            path
        )


finally:

    if stage_root.exists():

        shutil.rmtree(
            stage_root
        )


# =============================================================================
# 12. FINAL TECHNICAL OUTPUT
# =============================================================================

print(
    "=" * 100
)

print(
    "EDEN WEEKLY FORECASTING EXTENSION — "
    "PART 11G COMPLETE"
)

print(
    "=" * 100
)

print(
    f"Status: {STATUS}"
)

print(
    f"Extension root: {EXT_ROOT}"
)

print(
    f"Local time: {NOW_LOCAL.isoformat()}"
)


print(
    "\nSCREENING DESIGN"
)

print(
    "Chronological folds: "
    f"{len(forecast_weeks)}"
)

print(
    "Minimum training weeks: "
    f"{MIN_TRAINING_WEEKS}"
)

print(
    "Targets: "
    + ", ".join(
        TARGETS
    )
)

print(
    "Scopes: "
    + ", ".join(
        SCOPE_FLAGS
    )
)

print(
    "Candidate systems: "
    + ", ".join(
        CANDIDATE_SYSTEMS
    )
)

print(
    "Hyperparameter search performed: False"
)

print(
    "Final model selected: False"
)


print(
    "\nSCREENING LEADERS — "
    "NOT FINAL MODEL SELECTION"
)

print(
    screening_leaders[
        [
            "TargetID",
            "ScopeID",
            "CandidateSystemID",
            "Observations",
            "ActualTotal",
            "PredictedTotal",
            "MAE",
            "RMSE",
            "WAPEPercentage",
            "MeanBias",
            "TotalBias",
            "ReferenceBaselineMethod",
            "ReferenceBaselineWAPEPercentage",
            "WAPEImprovementPercentagePoints",
            "RelativeWAPEReductionPercentage",
            "BeatReferenceBaselineOnWAPE",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nALL CANDIDATE RANKINGS"
)

print(
    ranking[
        [
            "TargetID",
            "ScopeID",
            "CandidateRank",
            "CandidateSystemID",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "ReferenceBaselineWAPEPercentage",
            "WAPEImprovementPercentagePoints",
            "BeatReferenceBaselineOnWAPE",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nROUTING AND FALLBACK SUMMARY"
)

print(
    routing_audit.to_string(
        index=False
    )
)


print(
    "\nPART 11G VALIDATION"
)

print(
    validation.to_string(
        index=False
    )
)


print(
    "\nCONTROL OUTPUTS"
)

print(
    "- Candidate predictions: "
    f"{PREDICTIONS_PATH}"
)

print(
    "- Fold metrics: "
    f"{FOLD_METRICS_PATH}"
)

print(
    "- Pooled metrics: "
    f"{POOLED_METRICS_PATH}"
)

print(
    "- Model-versus-baseline comparison: "
    f"{COMPARISON_PATH}"
)

print(
    "- Candidate ranking: "
    f"{RANKING_PATH}"
)

print(
    "- Routing audit: "
    f"{ROUTING_AUDIT_PATH}"
)

print(
    "- Screening contract: "
    f"{SCREENING_CONTRACT_PATH}"
)

print(
    f"- Checkpoint: {CHECKPOINT_PATH}"
)

print(
    "- Checkpoint SHA256: "
    f"{checkpoint_hash}"
)

print(
    f"- Part 11G lock: {LOCK_PATH}"
)

print(
    "- Part 11G lock SHA256: "
    f"{lock_hash}"
)


print(
    "\nSAFETY: protected target vault "
    "opened/copied False/False; opened "
    "March 2026 targets read False; product "
    "ID used as a primary direct predictor "
    "False; hyperparameter search performed "
    "False; models fitted only inside "
    "chronological folds True; final models "
    "refitted False; final predictions created "
    "False; forecasts rounded before scoring "
    "False."
)

print(
    "=" * 100
)

/opt/anaconda3/envs/ai/lib/python3.11/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['NormalDemandLag8' 'NormalDemandLag13']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/envs/ai/lib/python3.11/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['NormalDemandLag8' 'NormalDemandLag13']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/envs/ai/lib/python3.11/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['NormalDemandLag8' 'NormalDemandLag13']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/envs/ai/lib/python3.11/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['NormalDemandLag8'

EDEN WEEKLY FORECASTING EXTENSION — PART 11G COMPLETE
Status: PART_11G_COMPLETED_READY_FOR_11H
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-05T20:59:12.596725+01:00

SCREENING DESIGN
Chronological folds: 39
Minimum training weeks: 8
Targets: WEEKLY_NORMAL_DEMAND, WEEKLY_TOTAL_DEMAND
Scopes: ALL_PRODUCTS, FOLD_STRICT_80_PERCENT, FOLD_STRICT_90_PERCENT, FOLD_STRICT_95_PERCENT
Candidate systems: GLOBAL_RIDGE, GLOBAL_HISTGB, GLOBAL_HURDLE_HISTGB, SEGMENT_FEATURE_RIDGE, ROUTED_SEGMENT_HISTGB
Hyperparameter search performed: False
Final model selected: False

SCREENING LEADERS — NOT FINAL MODEL SELECTION
            TargetID                ScopeID    CandidateSystemID  Observations  ActualTotal  PredictedTotal       MAE      RMSE  WAPEPercentage  MeanBias     TotalBias ReferenceBaselineMethod  ReferenceBaselineWAPEPercentage  WAPEImprovementPercentagePoints  RelativeWAPEReductionPercentage  BeatReferenceBaselineOnWAPE
WEE

In [24]:
from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11H
# Full corrected replacement cell
# Target comparison and bulk-demand treatment decision
# =============================================================================

# =============================================================================
# 1. PATHS, STATUS AND FIXED DESIGN
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"

MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
FEATURE_ROOT = EXT_ROOT / "04_weekly_features"
BASELINE_ROOT = EXT_ROOT / "05_weekly_baselines"
VALIDATION_ROOT = EXT_ROOT / "07_weekly_validation"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
LOG_ROOT = EXT_ROOT / "12_logs"

SELECTION_FEATURES = FEATURE_ROOT / "11F_model_selection_weekly_features.csv"

PART11E_PREDICTIONS = BASELINE_ROOT / "11E_preholdout_weekly_baseline_predictions.csv"
PART11E_RANKING = BASELINE_ROOT / "11E_preholdout_baseline_ranking.csv"
PART11E_LOCK = CHECKPOINT_ROOT / "11E_weekly_baselines_lock.json"
PART11E_LOCK_SHA = CHECKPOINT_ROOT / "11E_weekly_baselines_lock.sha256"

PART11G_PREDICTIONS = VALIDATION_ROOT / "11G_candidate_weekly_predictions.csv"
PART11G_RANKING = VALIDATION_ROOT / "11G_candidate_model_ranking.csv"
PART11G_MANIFEST = VALIDATION_ROOT / "11G_output_hash_manifest.csv"
PART11G_LOCK = CHECKPOINT_ROOT / "11G_weekly_model_screening_lock.json"
PART11G_LOCK_SHA = CHECKPOINT_ROOT / "11G_weekly_model_screening_lock.sha256"
PART11G_CHECKPOINT = CHECKPOINT_ROOT / "11G_checkpoint.json"
PART11G_CHECKPOINT_SHA = CHECKPOINT_ROOT / "11G_checkpoint.sha256"

BULK_PRODUCT_PROFILE_PATH = VALIDATION_ROOT / "11H_bulk_product_profile.csv"
BULK_WEEK_SUMMARY_PATH = VALIDATION_ROOT / "11H_bulk_week_summary.csv"
BULK_FALLBACK_PREDICTIONS_PATH = VALIDATION_ROOT / "11H_bulk_fallback_predictions.csv"
BULK_FALLBACK_METRICS_PATH = VALIDATION_ROOT / "11H_bulk_fallback_metrics.csv"
STRATEGY_PREDICTIONS_PATH = VALIDATION_ROOT / "11H_target_strategy_predictions.csv"
STRATEGY_METRICS_PATH = VALIDATION_ROOT / "11H_target_strategy_metrics.csv"
PAIRED_COMPARISON_PATH = VALIDATION_ROOT / "11H_direct_total_vs_normal_plus_bulk.csv"
TARGET_COMPARISON_PATH = VALIDATION_ROOT / "11H_normal_vs_total_screening_comparison.csv"
DECISION_PATH = VALIDATION_ROOT / "11H_target_and_bulk_decision.json"
DECISION_TABLE_PATH = VALIDATION_ROOT / "11H_target_and_bulk_decision.csv"
VALIDATION_PATH = VALIDATION_ROOT / "11H_validation.csv"
MANIFEST_PATH = VALIDATION_ROOT / "11H_output_hash_manifest.csv"

STEP_MEMORY_PATH = STEP_MEMORY_ROOT / "STEP_11H_TARGET_AND_BULK_DECISION.md"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "11H_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "11H_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "11H_target_and_bulk_decision_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "11H_target_and_bulk_decision_lock.sha256"
LOG_PATH = LOG_ROOT / "11H_target_and_bulk_decision_log.txt"

MEMORY_FILES = {
    "PROJECT_CONTEXT": MEMORY_ROOT / "PROJECT_CONTEXT.md",
    "WORKFLOW": MEMORY_ROOT / "WORKFLOW.md",
    "DECISIONS": MEMORY_ROOT / "DECISIONS.md",
    "FILES_AND_PATHS": MEMORY_ROOT / "FILES_AND_PATHS.md",
    "METRICS_AND_RESULTS": MEMORY_ROOT / "METRICS_AND_RESULTS.md",
    "CHAT_INDEX": MEMORY_ROOT / "CHAT_INDEX.md",
    "CURRENT_HANDOFF": MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

STEP_ID = "11H"
STATUS = "PART_11H_COMPLETED_READY_FOR_11I"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

CANONICAL_SCOPE_COLUMNS = [
    "PriorStrict80PctScope",
    "PriorStrict90PctScope",
    "PriorStrict95PctScope",
]

BASELINE_SCOPE_ALIASES = {
    "InFoldStrict80PctScope": "PriorStrict80PctScope",
    "InFoldStrict90PctScope": "PriorStrict90PctScope",
    "InFoldStrict95PctScope": "PriorStrict95PctScope",
}

SCOPE_FLAGS = {
    "ALL_PRODUCTS": None,
    "FOLD_STRICT_80_PERCENT": "PriorStrict80PctScope",
    "FOLD_STRICT_90_PERCENT": "PriorStrict90PctScope",
    "FOLD_STRICT_95_PERCENT": "PriorStrict95PctScope",
}

BULK_METHOD_COMPLEXITY = {
    "ZERO_BULK": 0,
    "LAST_BULK_CONTEXT": 1,
    "MEAN_LAST_4_BULK_CONTEXTS": 2,
}

STRATEGY_COMPLEXITY = {
    "NORMAL_PLUS_ZERO_BULK": 0,
    "NORMAL_PLUS_LAST_BULK_CONTEXT": 1,
    "NORMAL_PLUS_MEAN_LAST_4_BULK_CONTEXTS": 2,
    "DIRECT_WEEKLY_TOTAL_DEMAND": 3,
}

NEW_OUTPUTS = [
    BULK_PRODUCT_PROFILE_PATH,
    BULK_WEEK_SUMMARY_PATH,
    BULK_FALLBACK_PREDICTIONS_PATH,
    BULK_FALLBACK_METRICS_PATH,
    STRATEGY_PREDICTIONS_PATH,
    STRATEGY_METRICS_PATH,
    PAIRED_COMPARISON_PATH,
    TARGET_COMPARISON_PATH,
    DECISION_PATH,
    DECISION_TABLE_PATH,
    VALIDATION_PATH,
    MANIFEST_PATH,
    STEP_MEMORY_PATH,
    CHECKPOINT_PATH,
    CHECKPOINT_SHA_PATH,
    LOCK_PATH,
    LOCK_SHA_PATH,
    LOG_PATH,
]

# =============================================================================
# 2. HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def verify_sidecar(path: Path, sidecar: Path, label: str) -> str:
    require_file(path, label)
    require_file(sidecar, f"{label} SHA-256 sidecar")
    expected = sidecar.read_text(encoding="utf-8").strip().split()[0].lower()
    actual = sha256_file(path)
    if len(expected) != 64 or expected != actual:
        raise AssertionError(
            f"{label} SHA-256 mismatch:\nExpected: {expected}\nActual:   {actual}"
        )
    return actual


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        frame.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_json(path: Path, payload: dict) -> None:
    atomic_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def bool_series(series: pd.Series, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }
    normalized = series.astype(str).str.strip().str.lower()
    invalid = sorted(set(normalized) - set(mapping))
    if invalid:
        raise ValueError(f"{label} has invalid Boolean values: {invalid[:10]}")
    return normalized.map(mapping).astype(bool)


def normalise_scope_columns(
    frame: pd.DataFrame,
    frame_label: str,
    aliases: dict[str, str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    aliases = aliases or {}
    rename_map: dict[str, str] = {}

    for source_column, target_column in aliases.items():
        source_exists = source_column in result.columns
        target_exists = target_column in result.columns

        if source_exists and target_exists:
            source_values = bool_series(
                result[source_column], f"{frame_label}.{source_column}"
            )
            target_values = bool_series(
                result[target_column], f"{frame_label}.{target_column}"
            )
            if not source_values.equals(target_values):
                raise AssertionError(
                    f"{frame_label} contains both {source_column} and {target_column}, "
                    "but their values do not match."
                )
            result = result.drop(columns=[source_column])
        elif source_exists and not target_exists:
            rename_map[source_column] = target_column

    if rename_map:
        result = result.rename(columns=rename_map)

    missing = [
        column for column in CANONICAL_SCOPE_COLUMNS if column not in result.columns
    ]
    if missing:
        scope_like = sorted(
            column
            for column in result.columns
            if "Strict" in column or "Scope" in column
        )
        raise AssertionError(
            f"{frame_label} is missing required scope columns after normalization: "
            f"{missing}\nAvailable scope-like columns: {scope_like}"
        )

    for column in CANONICAL_SCOPE_COLUMNS:
        result[column] = bool_series(result[column], f"{frame_label}.{column}")

    return result


def metric_record(actual: pd.Series, prediction: pd.Series) -> dict:
    actual_array = pd.to_numeric(actual, errors="raise").to_numpy(dtype=float)
    prediction_array = pd.to_numeric(prediction, errors="raise").to_numpy(dtype=float)

    if len(actual_array) != len(prediction_array):
        raise AssertionError("Actual and prediction lengths differ")
    if not np.isfinite(actual_array).all() or not np.isfinite(prediction_array).all():
        raise AssertionError("Non-finite values entered metric calculation")

    errors = prediction_array - actual_array
    absolute_errors = np.abs(errors)
    denominator = float(np.abs(actual_array).sum())

    return {
        "Observations": int(len(actual_array)),
        "ActualTotal": float(actual_array.sum()),
        "PredictedTotal": float(prediction_array.sum()),
        "MAE": float(absolute_errors.mean()) if len(actual_array) else np.nan,
        "RMSE": (
            float(np.sqrt(np.mean(errors ** 2))) if len(actual_array) else np.nan
        ),
        "WAPEPercentage": (
            float(100.0 * absolute_errors.sum() / denominator)
            if denominator != 0
            else np.nan
        ),
        "MeanBias": float(errors.mean()) if len(actual_array) else np.nan,
        "TotalBias": float(errors.sum()),
    }


def update_section(text: str, marker: str, heading: str, body: str) -> str:
    start = f"<!-- BEGIN {marker} -->"
    end = f"<!-- END {marker} -->"
    section = f"{start}\n## {heading}\n\n{body.rstrip()}\n{end}"

    if start in text and end in text:
        before = text.split(start, 1)[0].rstrip()
        after = text.split(end, 1)[1].lstrip()
        return before + "\n\n" + section + ("\n\n" + after if after else "") + "\n"

    return text.rstrip() + "\n\n" + section + "\n"


def stage_path(stage_root: Path, final_path: Path) -> Path:
    return stage_root / final_path.relative_to(EXT_ROOT)


def stage_text(stage_root: Path, final_path: Path, text: str) -> None:
    atomic_text(stage_path(stage_root, final_path), text)


def stage_csv(stage_root: Path, final_path: Path, frame: pd.DataFrame) -> None:
    atomic_csv(stage_path(stage_root, final_path), frame)


def stage_json(stage_root: Path, final_path: Path, payload: dict) -> None:
    atomic_json(stage_path(stage_root, final_path), payload)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


# =============================================================================
# 3. VERIFY DIRECTORIES, INPUT FILES, LOCKS AND OVERWRITE GUARDS
# =============================================================================

for directory, label in [
    (EXT_ROOT, "extension root"),
    (MEMORY_ROOT, "memory root"),
    (FEATURE_ROOT, "weekly feature directory"),
    (BASELINE_ROOT, "weekly baseline directory"),
    (VALIDATION_ROOT, "weekly validation directory"),
    (CHECKPOINT_ROOT, "checkpoint directory"),
    (LOG_ROOT, "log directory"),
]:
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing required {label}:\n{directory}")

for path, label in [
    (SELECTION_FEATURES, "Part 11F model-selection features"),
    (PART11E_PREDICTIONS, "Part 11E baseline predictions"),
    (PART11E_RANKING, "Part 11E baseline ranking"),
    (PART11G_PREDICTIONS, "Part 11G candidate predictions"),
    (PART11G_RANKING, "Part 11G candidate ranking"),
    (PART11G_MANIFEST, "Part 11G output hash manifest"),
]:
    require_file(path, label)

for key, path in MEMORY_FILES.items():
    require_file(path, f"memory file {key}")

if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise FileExistsError(f"Part 11H overwrite lock triggered:\n{LOCK_PATH}")

existing_outputs = [path for path in NEW_OUTPUTS if path.exists()]
if existing_outputs:
    raise FileExistsError(
        "Existing uncommitted Part 11H outputs found; no files changed:\n"
        + "\n".join(f"- {path}" for path in existing_outputs)
    )

for old_stage in EXT_ROOT.glob(".11H_staging_*"):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)

part11g_lock_hash = verify_sidecar(PART11G_LOCK, PART11G_LOCK_SHA, "Part 11G lock")
part11g_checkpoint_hash = verify_sidecar(
    PART11G_CHECKPOINT, PART11G_CHECKPOINT_SHA, "Part 11G checkpoint"
)
part11e_lock_hash = verify_sidecar(PART11E_LOCK, PART11E_LOCK_SHA, "Part 11E lock")

part11g_lock = json.loads(PART11G_LOCK.read_text(encoding="utf-8"))
part11e_lock = json.loads(PART11E_LOCK.read_text(encoding="utf-8"))

if part11g_lock.get("Status") != "PART_11G_COMPLETED_READY_FOR_11H":
    raise AssertionError(f"Unexpected Part 11G status: {part11g_lock.get('Status')}")
if part11g_lock.get("ReadyForPart11H") is not True:
    raise AssertionError("Part 11G lock does not authorise Part 11H")
if part11g_lock.get("Part11E", {}).get("LockSHA256") != part11e_lock_hash:
    raise AssertionError("Part 11G does not reference the current Part 11E lock")
if part11e_lock.get("Status") != "PART_11E_COMPLETED_READY_FOR_11F":
    raise AssertionError("Part 11E lock status changed unexpectedly")

for key, expected in {
    "ProtectedTargetVaultOpened": False,
    "ProtectedTargetVaultCopied": False,
    "OpenedMarch2026TargetsRead": False,
    "ProductIDUsedAsPrimaryDirectPredictor": False,
    "HyperparameterSearchPerformed": False,
    "FinalModelsRefitted": False,
    "FinalPredictionsCreated": False,
}.items():
    actual = part11g_lock.get("SafetyAssertions", {}).get(key)
    if actual is not expected:
        raise AssertionError(f"Invalid Part 11G safety assertion {key}: {actual}")

predictions_hash = sha256_file(PART11G_PREDICTIONS)
ranking_hash = sha256_file(PART11G_RANKING)

part11g_manifest = pd.read_csv(PART11G_MANIFEST, low_memory=False)
if not {"RelativePath", "SHA256"}.issubset(part11g_manifest.columns):
    raise AssertionError("Part 11G output manifest schema is invalid")


def manifest_expected_hash(path: Path) -> str:
    relative = str(path.relative_to(EXT_ROOT))
    rows = part11g_manifest.loc[
        part11g_manifest["RelativePath"].astype(str) == relative
    ]
    if len(rows) != 1:
        raise AssertionError(
            f"Part 11G manifest does not uniquely record {relative}"
        )
    return str(rows.iloc[0]["SHA256"])


if predictions_hash != manifest_expected_hash(PART11G_PREDICTIONS):
    raise AssertionError("Part 11G candidate-prediction hash changed")
if ranking_hash != manifest_expected_hash(PART11G_RANKING):
    raise AssertionError("Part 11G candidate-ranking hash changed")
if ranking_hash != part11g_lock.get("Screening", {}).get("CandidateRankingSHA256"):
    raise AssertionError("Part 11G lock ranking hash does not match the ranking file")
if sha256_file(PART11E_RANKING) != part11g_lock.get("Part11E", {}).get(
    "ReferenceBaselineRankingSHA256"
):
    raise AssertionError("Part 11E baseline-ranking hash changed after Part 11G")

# =============================================================================
# 4. LOAD INPUTS AND NORMALISE SCHEMAS
# =============================================================================

features = pd.read_csv(SELECTION_FEATURES, low_memory=False)
baseline_predictions = pd.read_csv(PART11E_PREDICTIONS, low_memory=False)
baseline_ranking = pd.read_csv(PART11E_RANKING, low_memory=False)
model_predictions = pd.read_csv(PART11G_PREDICTIONS, low_memory=False)
model_ranking = pd.read_csv(PART11G_RANKING, low_memory=False)

for frame, date_columns in [
    (features, ["WeekStartDate", "WeekEndDate"]),
    (baseline_predictions, ["WeekStartDate", "WeekEndDate"]),
    (model_predictions, ["WeekStartDate", "WeekEndDate"]),
]:
    for column in date_columns:
        frame[column] = pd.to_datetime(frame[column], errors="raise")

for frame in [features, baseline_predictions, model_predictions]:
    frame["CanonicalProductID"] = frame["CanonicalProductID"].astype(str)

features = normalise_scope_columns(features, "Part 11F features")
baseline_predictions = normalise_scope_columns(
    baseline_predictions,
    "Part 11E baseline predictions",
    aliases=BASELINE_SCOPE_ALIASES,
)
model_predictions = normalise_scope_columns(
    model_predictions,
    "Part 11G model predictions",
)

remaining_baseline_aliases = [
    column for column in BASELINE_SCOPE_ALIASES if column in baseline_predictions.columns
]
if remaining_baseline_aliases:
    raise AssertionError(
        "Part 11E scope aliases remained after normalization: "
        f"{remaining_baseline_aliases}"
    )

print("Scope-column normalization passed:")
print("- Part 11F features: " + ", ".join(CANONICAL_SCOPE_COLUMNS))
print("- Part 11E baseline predictions: InFoldStrict* converted to PriorStrict*")
print("- Part 11G model predictions: " + ", ".join(CANONICAL_SCOPE_COLUMNS))

required_feature_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
    "BulkDemandLag1",
    "BulkDemandSum4",
    "PriorProductContextCount",
    *CANONICAL_SCOPE_COLUMNS,
}
required_baseline_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "TargetID",
    "BaselineMethod",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
}
required_model_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "TargetID",
    "CandidateSystemID",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
}

for frame, required, label in [
    (features, required_feature_columns, "Part 11F feature dataset"),
    (baseline_predictions, required_baseline_columns, "Part 11E predictions"),
    (model_predictions, required_model_columns, "Part 11G predictions"),
]:
    missing = sorted(required - set(frame.columns))
    if missing:
        raise AssertionError(f"{label} missing required columns: {missing}")

if features.duplicated(["WeekStartDate", "CanonicalProductID"]).any():
    raise AssertionError("Duplicate Part 11F feature keys found")

forecast_weeks = sorted(model_predictions["WeekStartDate"].drop_duplicates())
if len(forecast_weeks) != 39:
    raise AssertionError(f"Expected 39 forecast weeks, found {len(forecast_weeks)}")

feature_columns_for_evaluation = [
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
    "BulkDemandLag1",
    "BulkDemandSum4",
    "PriorProductContextCount",
    *CANONICAL_SCOPE_COLUMNS,
]

evaluation_features = features.loc[
    features["WeekStartDate"].isin(forecast_weeks),
    feature_columns_for_evaluation,
].copy()

expected_evaluation_rows = 7212
if len(evaluation_features) != expected_evaluation_rows:
    raise AssertionError(
        f"Unexpected evaluation rows: {len(evaluation_features)} != "
        f"{expected_evaluation_rows}"
    )

for column in [
    "WeeklyNormalDemand",
    "WeeklyBulkDemand",
    "WeeklyTotalDemand",
    "BulkDemandLag1",
    "BulkDemandSum4",
    "PriorProductContextCount",
]:
    evaluation_features[column] = pd.to_numeric(
        evaluation_features[column], errors="coerce"
    )

if not np.allclose(
    evaluation_features["WeeklyNormalDemand"]
    + evaluation_features["WeeklyBulkDemand"],
    evaluation_features["WeeklyTotalDemand"],
    atol=1e-9,
):
    raise AssertionError("Normal demand plus bulk demand does not equal total demand")

# =============================================================================
# 5. BULK-DEMAND DESCRIPTIVE PROFILE
# =============================================================================

bulk_total = float(evaluation_features["WeeklyBulkDemand"].sum())
normal_total = float(evaluation_features["WeeklyNormalDemand"].sum())
total_total = float(evaluation_features["WeeklyTotalDemand"].sum())

bulk_product_profile = (
    evaluation_features.groupby(
        ["CanonicalProductID", "CanonicalProductName"], as_index=False
    )
    .agg(
        EvaluationWeeks=("WeekStartDate", "nunique"),
        TotalBulkDemand=("WeeklyBulkDemand", "sum"),
        AverageWeeklyBulkDemand=("WeeklyBulkDemand", "mean"),
        ActiveBulkWeeks=(
            "WeeklyBulkDemand",
            lambda series: int((series > 0).sum()),
        ),
        ZeroBulkWeeks=(
            "WeeklyBulkDemand",
            lambda series: int((series == 0).sum()),
        ),
        BulkDemandStdDevPopulation=(
            "WeeklyBulkDemand",
            lambda series: float(np.std(series, ddof=0)),
        ),
        MaximumWeeklyBulkDemand=("WeeklyBulkDemand", "max"),
    )
    .sort_values(
        ["TotalBulkDemand", "CanonicalProductID"],
        ascending=[False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

bulk_product_profile["ZeroBulkWeekPercentage"] = (
    100.0
    * bulk_product_profile["ZeroBulkWeeks"]
    / bulk_product_profile["EvaluationWeeks"]
)
bulk_product_profile["BulkDemandSharePercentage"] = np.where(
    bulk_total > 0,
    100.0 * bulk_product_profile["TotalBulkDemand"] / bulk_total,
    np.nan,
)
bulk_product_profile["BulkDemandRank"] = np.arange(
    1, len(bulk_product_profile) + 1
)
bulk_product_profile["CumulativeBulkDemandCoveragePercentage"] = np.where(
    bulk_total > 0,
    100.0 * bulk_product_profile["TotalBulkDemand"].cumsum() / bulk_total,
    np.nan,
)

bulk_week_summary = (
    evaluation_features.groupby(
        ["WeekStartDate", "WeekEndDate", "WeekID"], as_index=False
    )
    .agg(
        WeeklyNormalDemand=("WeeklyNormalDemand", "sum"),
        WeeklyBulkDemand=("WeeklyBulkDemand", "sum"),
        WeeklyTotalDemand=("WeeklyTotalDemand", "sum"),
        ProductsWithBulkDemand=(
            "WeeklyBulkDemand",
            lambda series: int((series > 0).sum()),
        ),
        ProductWeekRows=("CanonicalProductID", "size"),
    )
)
bulk_week_summary["BulkShareOfWeeklyTotalPercentage"] = np.where(
    bulk_week_summary["WeeklyTotalDemand"] > 0,
    100.0
    * bulk_week_summary["WeeklyBulkDemand"]
    / bulk_week_summary["WeeklyTotalDemand"],
    np.nan,
)

# =============================================================================
# 6. BULK-ONLY FALLBACK PREDICTIONS AND METRICS
# =============================================================================

bulk_base = evaluation_features.copy()
bulk_base["BulkPrediction_ZERO_BULK"] = 0.0
bulk_base["BulkPrediction_LAST_BULK_CONTEXT"] = (
    bulk_base["BulkDemandLag1"].fillna(0.0).clip(lower=0.0)
)

prior_contexts_for_mean4 = np.minimum(
    bulk_base["PriorProductContextCount"].fillna(0.0).to_numpy(dtype=float),
    4.0,
)
bulk_sum4 = bulk_base["BulkDemandSum4"].fillna(0.0).to_numpy(dtype=float)
bulk_base["BulkPrediction_MEAN_LAST_4_BULK_CONTEXTS"] = np.where(
    prior_contexts_for_mean4 > 0,
    bulk_sum4 / prior_contexts_for_mean4,
    0.0,
)

bulk_prediction_frames: list[pd.DataFrame] = []
for method in BULK_METHOD_COMPLEXITY:
    frame = bulk_base[
        [
            "WeekStartDate",
            "WeekEndDate",
            "WeekID",
            "CanonicalProductID",
            "CanonicalProductName",
            "WeeklyBulkDemand",
            *CANONICAL_SCOPE_COLUMNS,
        ]
    ].copy()
    frame["BulkMethodID"] = method
    frame["ActualBulkDemand"] = frame["WeeklyBulkDemand"].astype(float)
    frame["PredictedBulkDemand"] = bulk_base[
        f"BulkPrediction_{method}"
    ].to_numpy(dtype=float)
    frame["Error"] = frame["PredictedBulkDemand"] - frame["ActualBulkDemand"]
    frame["AbsoluteError"] = frame["Error"].abs()
    bulk_prediction_frames.append(frame)

bulk_fallback_predictions = pd.concat(bulk_prediction_frames, ignore_index=True)

bulk_metric_rows: list[dict] = []
for method, group in bulk_fallback_predictions.groupby("BulkMethodID", sort=True):
    for scope_id, flag_column in SCOPE_FLAGS.items():
        scoped = group if flag_column is None else group.loc[group[flag_column]]
        bulk_metric_rows.append(
            {
                "TargetID": "WEEKLY_BULK_DEMAND",
                "ScopeID": scope_id,
                "BulkMethodID": method,
                "ForecastWeeks": int(scoped["WeekStartDate"].nunique()),
                "Products": int(scoped["CanonicalProductID"].nunique()),
                **metric_record(
                    scoped["ActualBulkDemand"], scoped["PredictedBulkDemand"]
                ),
            }
        )

bulk_fallback_metrics = pd.DataFrame(bulk_metric_rows)
bulk_fallback_metrics["MethodComplexityOrder"] = bulk_fallback_metrics[
    "BulkMethodID"
].map(BULK_METHOD_COMPLEXITY)
bulk_fallback_metrics["AbsoluteTotalBias"] = bulk_fallback_metrics[
    "TotalBias"
].abs()
bulk_fallback_metrics = bulk_fallback_metrics.sort_values(
    [
        "ScopeID",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "MethodComplexityOrder",
        "BulkMethodID",
    ],
    kind="mergesort",
).reset_index(drop=True)
bulk_fallback_metrics["BulkMethodRank"] = (
    bulk_fallback_metrics.groupby("ScopeID").cumcount() + 1
)
bulk_fallback_metrics["IsBulkReferenceFallback"] = (
    bulk_fallback_metrics["BulkMethodRank"] == 1
)

selected_bulk_row = bulk_fallback_metrics.loc[
    (bulk_fallback_metrics["ScopeID"] == "ALL_PRODUCTS")
    & bulk_fallback_metrics["IsBulkReferenceFallback"]
]
if len(selected_bulk_row) != 1:
    raise AssertionError("Could not identify one all-products bulk fallback")
selected_bulk_method = str(selected_bulk_row.iloc[0]["BulkMethodID"])

# =============================================================================
# 7. PAIR NORMAL AND TOTAL FORECAST SOURCES
# =============================================================================

baseline_selected = baseline_predictions.loc[
    baseline_predictions["BaselineMethod"].astype(str) == "NAIVE_LAST_CONTEXT"
].copy()

base_keys = [
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    *CANONICAL_SCOPE_COLUMNS,
]

baseline_normal = baseline_selected.loc[
    baseline_selected["TargetID"] == "WEEKLY_NORMAL_DEMAND",
    base_keys + ["Actual", "Prediction"],
].rename(
    columns={
        "Actual": "ActualNormalDemand",
        "Prediction": "NormalPrediction",
    }
)

baseline_total = baseline_selected.loc[
    baseline_selected["TargetID"] == "WEEKLY_TOTAL_DEMAND",
    base_keys + ["Actual", "Prediction"],
].rename(
    columns={
        "Actual": "ActualTotalDemand",
        "Prediction": "DirectTotalPrediction",
    }
)

if baseline_normal.duplicated(base_keys).any() or baseline_total.duplicated(base_keys).any():
    raise AssertionError("Duplicate Part 11E baseline keys found after method filtering")

baseline_pair = baseline_normal.merge(
    baseline_total,
    on=base_keys,
    how="inner",
    validate="one_to_one",
)
baseline_pair["ForecastSourceID"] = "BASELINE__NAIVE_LAST_CONTEXT"

model_normal = model_predictions.loc[
    model_predictions["TargetID"] == "WEEKLY_NORMAL_DEMAND",
    base_keys + ["CandidateSystemID", "Actual", "Prediction"],
].rename(
    columns={
        "Actual": "ActualNormalDemand",
        "Prediction": "NormalPrediction",
    }
)

model_total = model_predictions.loc[
    model_predictions["TargetID"] == "WEEKLY_TOTAL_DEMAND",
    base_keys + ["CandidateSystemID", "Actual", "Prediction"],
].rename(
    columns={
        "Actual": "ActualTotalDemand",
        "Prediction": "DirectTotalPrediction",
    }
)

model_merge_keys = base_keys + ["CandidateSystemID"]
if model_normal.duplicated(model_merge_keys).any() or model_total.duplicated(
    model_merge_keys
).any():
    raise AssertionError("Duplicate Part 11G model keys found before target pairing")

model_pair = model_normal.merge(
    model_total,
    on=model_merge_keys,
    how="inner",
    validate="one_to_one",
)
model_pair["ForecastSourceID"] = "MODEL__" + model_pair[
    "CandidateSystemID"
].astype(str)
model_pair = model_pair.drop(columns=["CandidateSystemID"])

paired_sources = pd.concat([baseline_pair, model_pair], ignore_index=True)

bulk_merge_columns = [
    "WeekStartDate",
    "CanonicalProductID",
    "WeeklyBulkDemand",
    "BulkPrediction_ZERO_BULK",
    "BulkPrediction_LAST_BULK_CONTEXT",
    "BulkPrediction_MEAN_LAST_4_BULK_CONTEXTS",
]

paired_sources = paired_sources.merge(
    bulk_base[bulk_merge_columns],
    on=["WeekStartDate", "CanonicalProductID"],
    how="left",
    validate="many_to_one",
)

required_bulk_merge_values = [
    "WeeklyBulkDemand",
    "BulkPrediction_ZERO_BULK",
    "BulkPrediction_LAST_BULK_CONTEXT",
    "BulkPrediction_MEAN_LAST_4_BULK_CONTEXTS",
]
if paired_sources[required_bulk_merge_values].isna().any().any():
    raise AssertionError("Some paired forecast rows did not receive bulk features")

if not np.allclose(
    paired_sources["ActualNormalDemand"] + paired_sources["WeeklyBulkDemand"],
    paired_sources["ActualTotalDemand"],
    atol=1e-9,
):
    raise AssertionError("Paired normal, bulk and total actuals do not reconcile")

# =============================================================================
# 8. BUILD TARGET-STRATEGY PREDICTIONS AND METRICS
# =============================================================================

strategy_definitions = [
    (
        "DIRECT_WEEKLY_TOTAL_DEMAND",
        paired_sources["DirectTotalPrediction"].to_numpy(dtype=float),
        "DIRECT_TOTAL_TARGET",
    ),
    (
        "NORMAL_PLUS_ZERO_BULK",
        paired_sources["NormalPrediction"].to_numpy(dtype=float)
        + paired_sources["BulkPrediction_ZERO_BULK"].to_numpy(dtype=float),
        "ZERO_BULK",
    ),
    (
        "NORMAL_PLUS_LAST_BULK_CONTEXT",
        paired_sources["NormalPrediction"].to_numpy(dtype=float)
        + paired_sources["BulkPrediction_LAST_BULK_CONTEXT"].to_numpy(
            dtype=float
        ),
        "LAST_BULK_CONTEXT",
    ),
    (
        "NORMAL_PLUS_MEAN_LAST_4_BULK_CONTEXTS",
        paired_sources["NormalPrediction"].to_numpy(dtype=float)
        + paired_sources[
            "BulkPrediction_MEAN_LAST_4_BULK_CONTEXTS"
        ].to_numpy(dtype=float),
        "MEAN_LAST_4_BULK_CONTEXTS",
    ),
]

strategy_frames: list[pd.DataFrame] = []
for strategy_id, prediction_values, bulk_method in strategy_definitions:
    frame = paired_sources[
        base_keys
        + [
            "ForecastSourceID",
            "ActualNormalDemand",
            "WeeklyBulkDemand",
            "ActualTotalDemand",
        ]
    ].copy()
    frame["TargetStrategyID"] = strategy_id
    frame["BulkMethodID"] = bulk_method
    frame["Prediction"] = np.clip(prediction_values, 0.0, None)
    frame["Actual"] = frame["ActualTotalDemand"].astype(float)
    frame["Error"] = frame["Prediction"] - frame["Actual"]
    frame["AbsoluteError"] = frame["Error"].abs()
    strategy_frames.append(frame)

strategy_predictions = pd.concat(strategy_frames, ignore_index=True)
if not np.isfinite(strategy_predictions["Prediction"]).all():
    raise AssertionError("Non-finite Part 11H strategy predictions were created")
if (strategy_predictions["Prediction"] < 0).any():
    raise AssertionError("Negative Part 11H strategy predictions were created")

strategy_metric_rows: list[dict] = []
for (
    source_id,
    strategy_id,
    bulk_method,
), group in strategy_predictions.groupby(
    ["ForecastSourceID", "TargetStrategyID", "BulkMethodID"], sort=True
):
    for scope_id, flag_column in SCOPE_FLAGS.items():
        scoped = group if flag_column is None else group.loc[group[flag_column]]
        strategy_metric_rows.append(
            {
                "ForecastSourceID": source_id,
                "TargetStrategyID": strategy_id,
                "BulkMethodID": bulk_method,
                "ScopeID": scope_id,
                "ForecastWeeks": int(scoped["WeekStartDate"].nunique()),
                "Products": int(scoped["CanonicalProductID"].nunique()),
                **metric_record(scoped["Actual"], scoped["Prediction"]),
            }
        )

strategy_metrics = pd.DataFrame(strategy_metric_rows)
strategy_metrics["AbsoluteTotalBias"] = strategy_metrics["TotalBias"].abs()
strategy_metrics["StrategyComplexityOrder"] = strategy_metrics[
    "TargetStrategyID"
].map(STRATEGY_COMPLEXITY)
strategy_metrics = strategy_metrics.sort_values(
    [
        "ForecastSourceID",
        "ScopeID",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "StrategyComplexityOrder",
        "TargetStrategyID",
    ],
    kind="mergesort",
).reset_index(drop=True)
strategy_metrics["StrategyRankWithinSourceAndScope"] = (
    strategy_metrics.groupby(["ForecastSourceID", "ScopeID"]).cumcount() + 1
)
strategy_metrics["IsBestStrategyWithinSourceAndScope"] = (
    strategy_metrics["StrategyRankWithinSourceAndScope"] == 1
)

selected_normal_strategy_id = {
    "ZERO_BULK": "NORMAL_PLUS_ZERO_BULK",
    "LAST_BULK_CONTEXT": "NORMAL_PLUS_LAST_BULK_CONTEXT",
    "MEAN_LAST_4_BULK_CONTEXTS": "NORMAL_PLUS_MEAN_LAST_4_BULK_CONTEXTS",
}[selected_bulk_method]

paired_comparison = strategy_metrics.loc[
    strategy_metrics["TargetStrategyID"].isin(
        ["DIRECT_WEEKLY_TOTAL_DEMAND", selected_normal_strategy_id]
    )
].pivot_table(
    index=["ForecastSourceID", "ScopeID"],
    columns="TargetStrategyID",
    values=["WAPEPercentage", "MAE", "RMSE", "TotalBias"],
    aggfunc="first",
)
paired_comparison.columns = [
    f"{metric}__{strategy}" for metric, strategy in paired_comparison.columns
]
paired_comparison = paired_comparison.reset_index()

normal_wape_column = f"WAPEPercentage__{selected_normal_strategy_id}"
direct_wape_column = "WAPEPercentage__DIRECT_WEEKLY_TOTAL_DEMAND"
for required_column in [normal_wape_column, direct_wape_column]:
    if required_column not in paired_comparison.columns:
        raise AssertionError(
            f"Paired target comparison missing required column: {required_column}"
        )

paired_comparison["NormalPlusSelectedBulkWAPEAdvantagePercentagePoints"] = (
    paired_comparison[direct_wape_column] - paired_comparison[normal_wape_column]
)
paired_comparison["NormalPlusSelectedBulkWon"] = (
    paired_comparison[normal_wape_column] < paired_comparison[direct_wape_column]
)
paired_comparison["DirectTotalWon"] = (
    paired_comparison[direct_wape_column] < paired_comparison[normal_wape_column]
)
paired_comparison["Tie"] = np.isclose(
    paired_comparison[normal_wape_column],
    paired_comparison[direct_wape_column],
    atol=1e-12,
)

# =============================================================================
# 9. DIRECT NORMAL-TARGET VERSUS TOTAL-TARGET SCREENING COMPARISON
# =============================================================================

if "IsReferenceBaseline" not in baseline_ranking.columns:
    raise AssertionError("Part 11E baseline ranking lacks IsReferenceBaseline")

baseline_reference_mask = (
    baseline_ranking["IsReferenceBaseline"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1"])
)
baseline_reference = baseline_ranking.loc[baseline_reference_mask].copy()

baseline_target_compare = baseline_reference.pivot_table(
    index=["ScopeID", "BaselineMethod"],
    columns="TargetID",
    values="WAPEPercentage",
    aggfunc="first",
).reset_index()
baseline_target_compare["ForecastSourceID"] = "BASELINE__NAIVE_LAST_CONTEXT"
baseline_target_compare = baseline_target_compare.rename(
    columns={
        "WEEKLY_NORMAL_DEMAND": "NormalTargetWAPEPercentage",
        "WEEKLY_TOTAL_DEMAND": "TotalTargetWAPEPercentage",
    }
)

model_target_compare = model_ranking.pivot_table(
    index=["ScopeID", "CandidateSystemID"],
    columns="TargetID",
    values="WAPEPercentage",
    aggfunc="first",
).reset_index()
model_target_compare["ForecastSourceID"] = "MODEL__" + model_target_compare[
    "CandidateSystemID"
].astype(str)
model_target_compare = model_target_compare.rename(
    columns={
        "WEEKLY_NORMAL_DEMAND": "NormalTargetWAPEPercentage",
        "WEEKLY_TOTAL_DEMAND": "TotalTargetWAPEPercentage",
    }
).drop(columns=["CandidateSystemID"])

comparison_columns = [
    "ForecastSourceID",
    "ScopeID",
    "NormalTargetWAPEPercentage",
    "TotalTargetWAPEPercentage",
]
for frame, label in [
    (baseline_target_compare, "baseline target comparison"),
    (model_target_compare, "model target comparison"),
]:
    missing = sorted(set(comparison_columns) - set(frame.columns))
    if missing:
        raise AssertionError(f"{label} missing required columns: {missing}")

target_comparison = pd.concat(
    [
        baseline_target_compare[comparison_columns],
        model_target_compare[comparison_columns],
    ],
    ignore_index=True,
)
target_comparison["NormalTargetAdvantagePercentagePoints"] = (
    target_comparison["TotalTargetWAPEPercentage"]
    - target_comparison["NormalTargetWAPEPercentage"]
)
target_comparison["NormalTargetBetter"] = (
    target_comparison["NormalTargetWAPEPercentage"]
    < target_comparison["TotalTargetWAPEPercentage"]
)

# =============================================================================
# 10. DECISION LOGIC
# =============================================================================

normal_target_better_count = int(target_comparison["NormalTargetBetter"].sum())
target_pair_count = int(len(target_comparison))
normal_plus_bulk_win_count = int(
    paired_comparison["NormalPlusSelectedBulkWon"].sum()
)
direct_total_win_count = int(paired_comparison["DirectTotalWon"].sum())
combined_pair_count = int(len(paired_comparison))

bulk_positive_rows = int((evaluation_features["WeeklyBulkDemand"] > 0).sum())
bulk_zero_percentage = float(
    100.0 * (evaluation_features["WeeklyBulkDemand"] == 0).mean()
)
bulk_share_percentage = (
    float(100.0 * bulk_total / total_total) if total_total != 0 else np.nan
)
products_with_bulk = int(
    evaluation_features.loc[
        evaluation_features["WeeklyBulkDemand"] > 0, "CanonicalProductID"
    ].nunique()
)
weeks_with_bulk = int(
    evaluation_features.loc[
        evaluation_features["WeeklyBulkDemand"] > 0, "WeekStartDate"
    ].nunique()
)


def products_to_reach_coverage(profile: pd.DataFrame, threshold: float) -> int:
    if bulk_total <= 0:
        return 0
    reached = profile.index[
        profile["CumulativeBulkDemandCoveragePercentage"] >= threshold
    ]
    if len(reached) == 0:
        return len(profile)
    return int(reached[0] + 1)


bulk_top_80_count = products_to_reach_coverage(bulk_product_profile, 80.0)
bulk_top_95_count = products_to_reach_coverage(bulk_product_profile, 95.0)

normal_architecture_supported = (
    normal_target_better_count == target_pair_count
    and normal_plus_bulk_win_count >= direct_total_win_count
)

if normal_architecture_supported:
    primary_modelled_target = "WEEKLY_NORMAL_DEMAND"
    direct_total_status = "RETAIN_FOR_SENSITIVITY_ONLY"
    decision_status = "NORMAL_DEMAND_ARCHITECTURE_APPROVED_FOR_PART_11I"
else:
    primary_modelled_target = "BOTH_NORMAL_AND_TOTAL_REMAIN_CANDIDATES"
    direct_total_status = "RETAIN_AS_ACTIVE_CANDIDATE"
    decision_status = "TARGET_ARCHITECTURE_REMAINS_UNRESOLVED_FOR_PART_11I"

bulk_handling_rule = (
    "ADD_CONFIRMED_FUTURE_BULK_QUANTITIES_EXTERNALLY; WHEN NO CONFIRMED "
    f"BULK INFORMATION EXISTS, USE {selected_bulk_method}"
)

scope_summary = strategy_metrics.loc[
    (strategy_metrics["ForecastSourceID"] == "BASELINE__NAIVE_LAST_CONTEXT")
    & (strategy_metrics["TargetStrategyID"] == selected_normal_strategy_id)
].sort_values("WAPEPercentage")
scope_proceed_order = scope_summary["ScopeID"].tolist()

scopes_proceeding = [
    "FOLD_STRICT_80_PERCENT",
    "FOLD_STRICT_90_PERCENT",
    "FOLD_STRICT_95_PERCENT",
]

decision = {
    "StepID": STEP_ID,
    "CreatedLocal": NOW_LOCAL.isoformat(),
    "DecisionStatus": decision_status,
    "PrimaryModelledTargetForPart11I": primary_modelled_target,
    "DirectWeeklyTotalDemandStatus": direct_total_status,
    "SelectedUnconfirmedBulkFallback": selected_bulk_method,
    "BulkHandlingRule": bulk_handling_rule,
    "ConfirmedBulkPolicy": (
        "Confirmed future bulk quantities are operational inputs, not quantities "
        "that should be inferred from sparse historical POS patterns."
    ),
    "Evidence": {
        "NormalTargetBetterComparisons": normal_target_better_count,
        "NormalVsTotalComparisonCount": target_pair_count,
        "NormalPlusSelectedBulkWins": normal_plus_bulk_win_count,
        "DirectTotalWins": direct_total_win_count,
        "CombinedStrategyComparisonCount": combined_pair_count,
        "EvaluationNormalDemandUnits": normal_total,
        "EvaluationBulkDemandUnits": bulk_total,
        "EvaluationTotalDemandUnits": total_total,
        "BulkShareOfTotalPercentage": bulk_share_percentage,
        "BulkPositiveProductWeeks": bulk_positive_rows,
        "BulkZeroProductWeekPercentage": bulk_zero_percentage,
        "ProductsWithBulkDemand": products_with_bulk,
        "WeeksWithBulkDemand": weeks_with_bulk,
        "ProductsCovering80PercentOfBulkDemand": bulk_top_80_count,
        "ProductsCovering95PercentOfBulkDemand": bulk_top_95_count,
    },
    "ScopeCandidatesProceedingToPart11I": scopes_proceeding,
    "ScopeScreeningOrderByBaselineNormalPlusSelectedBulkWAPE": scope_proceed_order,
    "FinalMethodSelected": False,
    "FinalScopeSelected": False,
}

decision_table = pd.DataFrame(
    [
        {
            "DecisionField": "Primary modelled target",
            "Decision": primary_modelled_target,
            "Status": decision_status,
        },
        {
            "DecisionField": "Direct weekly total demand",
            "Decision": direct_total_status,
            "Status": decision_status,
        },
        {
            "DecisionField": "Unconfirmed bulk fallback",
            "Decision": selected_bulk_method,
            "Status": "LOCKED_FOR_PART_11I_COMPARISON",
        },
        {
            "DecisionField": "Confirmed future bulk",
            "Decision": "ADD_EXTERNALLY_TO_NORMAL_DEMAND_FORECAST",
            "Status": "OPERATIONAL_RULE",
        },
        {
            "DecisionField": "Scopes proceeding",
            "Decision": "80_PERCENT;90_PERCENT;95_PERCENT",
            "Status": "FINAL_SCOPE_NOT_YET_SELECTED",
        },
    ]
)

# =============================================================================
# 11. VALIDATION
# =============================================================================

expected_sources = 1 + int(model_predictions["CandidateSystemID"].nunique())
expected_strategy_metric_rows = expected_sources * len(SCOPE_FLAGS) * 4
expected_bulk_metric_rows = len(BULK_METHOD_COMPLEXITY) * len(SCOPE_FLAGS)
expected_target_comparison_rows = expected_sources * len(SCOPE_FLAGS)

validation_rows = [
    {
        "Check": "Part 11G lock and checkpoint verified",
        "Expected": True,
        "Actual": True,
        "Passed": True,
    },
    {
        "Check": "Part 11G references current Part 11E lock",
        "Expected": part11e_lock_hash,
        "Actual": part11g_lock.get("Part11E", {}).get("LockSHA256"),
        "Passed": part11g_lock.get("Part11E", {}).get("LockSHA256")
        == part11e_lock_hash,
    },
    {
        "Check": "Chronological evaluation weeks",
        "Expected": 39,
        "Actual": len(forecast_weeks),
        "Passed": len(forecast_weeks) == 39,
    },
    {
        "Check": "Evaluation product-week rows",
        "Expected": expected_evaluation_rows,
        "Actual": len(evaluation_features),
        "Passed": len(evaluation_features) == expected_evaluation_rows,
    },
    {
        "Check": "Normal plus bulk equals total actual",
        "Expected": True,
        "Actual": bool(
            np.allclose(
                evaluation_features["WeeklyNormalDemand"]
                + evaluation_features["WeeklyBulkDemand"],
                evaluation_features["WeeklyTotalDemand"],
                atol=1e-9,
            )
        ),
        "Passed": bool(
            np.allclose(
                evaluation_features["WeeklyNormalDemand"]
                + evaluation_features["WeeklyBulkDemand"],
                evaluation_features["WeeklyTotalDemand"],
                atol=1e-9,
            )
        ),
    },
    {
        "Check": "Bulk fallback metric groups",
        "Expected": expected_bulk_metric_rows,
        "Actual": len(bulk_fallback_metrics),
        "Passed": len(bulk_fallback_metrics) == expected_bulk_metric_rows,
    },
    {
        "Check": "Exactly one all-products bulk fallback selected",
        "Expected": 1,
        "Actual": len(selected_bulk_row),
        "Passed": len(selected_bulk_row) == 1,
    },
    {
        "Check": "Paired forecast sources",
        "Expected": expected_sources,
        "Actual": paired_sources["ForecastSourceID"].nunique(),
        "Passed": paired_sources["ForecastSourceID"].nunique()
        == expected_sources,
    },
    {
        "Check": "Target-strategy metric groups",
        "Expected": expected_strategy_metric_rows,
        "Actual": len(strategy_metrics),
        "Passed": len(strategy_metrics) == expected_strategy_metric_rows,
    },
    {
        "Check": "Direct normal-versus-total comparison groups",
        "Expected": expected_target_comparison_rows,
        "Actual": len(target_comparison),
        "Passed": len(target_comparison) == expected_target_comparison_rows,
    },
    {
        "Check": "Non-finite or negative strategy predictions",
        "Expected": 0,
        "Actual": int((~np.isfinite(strategy_predictions["Prediction"])).sum())
        + int((strategy_predictions["Prediction"] < 0).sum()),
        "Passed": bool(np.isfinite(strategy_predictions["Prediction"]).all())
        and bool((strategy_predictions["Prediction"] >= 0).all()),
    },
    {
        "Check": "Opened March 2026 targets read",
        "Expected": False,
        "Actual": False,
        "Passed": True,
    },
    {
        "Check": "Models fitted or refitted in Part 11H",
        "Expected": False,
        "Actual": False,
        "Passed": True,
    },
    {
        "Check": "Final method or scope selected",
        "Expected": False,
        "Actual": False,
        "Passed": True,
    },
]
validation = pd.DataFrame(validation_rows)

if not validation["Passed"].all():
    raise AssertionError(
        "Part 11H validation failed:\n"
        + validation.loc[~validation["Passed"]].to_string(index=False)
    )

# =============================================================================
# 12. STAGE OUTPUTS, MEMORY, CHECKPOINT, MANIFEST AND LOCK
# =============================================================================

stage_root = EXT_ROOT / f".11H_staging_{uuid.uuid4().hex}"
stage_root.mkdir(parents=True, exist_ok=False)

try:
    for path, frame in [
        (BULK_PRODUCT_PROFILE_PATH, bulk_product_profile),
        (BULK_WEEK_SUMMARY_PATH, bulk_week_summary),
        (BULK_FALLBACK_PREDICTIONS_PATH, bulk_fallback_predictions),
        (BULK_FALLBACK_METRICS_PATH, bulk_fallback_metrics),
        (STRATEGY_PREDICTIONS_PATH, strategy_predictions),
        (STRATEGY_METRICS_PATH, strategy_metrics),
        (PAIRED_COMPARISON_PATH, paired_comparison),
        (TARGET_COMPARISON_PATH, target_comparison),
        (DECISION_TABLE_PATH, decision_table),
        (VALIDATION_PATH, validation),
    ]:
        stage_csv(stage_root, path, frame)

    stage_json(stage_root, DECISION_PATH, decision)

    decisions_text = "\n".join(
        [
            f"- Decision date: {NOW_LOCAL.date().isoformat()}.",
            f"- Primary modelled target status: {primary_modelled_target}.",
            f"- Direct weekly total-demand status: {direct_total_status}.",
            (
                "- Unconfirmed bulk fallback selected from pre-holdout evidence: "
                f"{selected_bulk_method}."
            ),
            (
                "- Confirmed future bulk quantities must be added externally as "
                "known operational inputs."
            ),
            (
                "- 80%, 90% and 95% scopes all proceed to Part 11I; no final "
                "scope is selected here."
            ),
            "- No model was fitted or refitted in Part 11H.",
            "- March 2026 targets were not read.",
        ]
    )

    files_text = "\n".join(
        [
            f"- Bulk product profile: `{BULK_PRODUCT_PROFILE_PATH}`",
            f"- Bulk week summary: `{BULK_WEEK_SUMMARY_PATH}`",
            f"- Bulk fallback metrics: `{BULK_FALLBACK_METRICS_PATH}`",
            f"- Target-strategy metrics: `{STRATEGY_METRICS_PATH}`",
            (
                "- Direct-total versus normal-plus-bulk comparison: "
                f"`{PAIRED_COMPARISON_PATH}`"
            ),
            f"- Normal-versus-total comparison: `{TARGET_COMPARISON_PATH}`",
            f"- Decision contract: `{DECISION_PATH}`",
            f"- Validation: `{VALIDATION_PATH}`",
        ]
    )

    results_text = "\n".join(
        [
            f"- Evaluation normal demand: {normal_total:,.0f} units.",
            f"- Evaluation bulk demand: {bulk_total:,.0f} units.",
            f"- Evaluation total demand: {total_total:,.0f} units.",
            f"- Bulk share of evaluation total demand: {bulk_share_percentage:.6f}%.",
            (
                f"- Positive bulk product-weeks: {bulk_positive_rows:,} of "
                f"{len(evaluation_features):,}."
            ),
            f"- Zero-bulk product-week percentage: {bulk_zero_percentage:.6f}%.",
            f"- Products with any bulk demand: {products_with_bulk}.",
            f"- Weeks with any bulk demand: {weeks_with_bulk} of {len(forecast_weeks)}.",
            f"- Products covering 80% of bulk demand: {bulk_top_80_count}.",
            f"- Products covering 95% of bulk demand: {bulk_top_95_count}.",
            (
                f"- Normal target better than total target in "
                f"{normal_target_better_count}/{target_pair_count} matched comparisons."
            ),
            (
                f"- Normal plus selected bulk fallback won "
                f"{normal_plus_bulk_win_count}/{combined_pair_count} paired "
                "total-demand comparisons."
            ),
            f"- Selected unconfirmed bulk fallback: {selected_bulk_method}.",
            f"- Decision status: {decision_status}.",
        ]
    )

    for key, final_path in MEMORY_FILES.items():
        original = final_path.read_text(encoding="utf-8")

        if key == "PROJECT_CONTEXT":
            updated = update_section(
                original,
                "STEP_11H",
                "Part 11H target and bulk-demand decision",
                (
                    "Normal-demand and total-demand forecasting were compared, "
                    "bulk-demand sparsity was quantified, and an operational "
                    "bulk-treatment rule was frozen before final method selection."
                ),
            )
        elif key == "WORKFLOW":
            updated = update_section(
                original,
                "STEP_11H",
                "Part 11H workflow status",
                (
                    "Part 11H is complete. Part 11I will use the locked target "
                    "and bulk-treatment decision to select the final weekly method "
                    "and scope from pre-holdout chronological evidence only."
                ),
            )
        elif key == "DECISIONS":
            updated = update_section(
                original, "STEP_11H", "Part 11H decisions", decisions_text
            )
        elif key == "FILES_AND_PATHS":
            updated = update_section(
                original, "STEP_11H", "Part 11H files", files_text
            )
        elif key == "METRICS_AND_RESULTS":
            updated = update_section(
                original,
                "STEP_11H",
                "Part 11H metrics and results",
                results_text,
            )
        elif key == "CHAT_INDEX":
            row = (
                f"| {NOW_LOCAL.isoformat()} | 11H | Target comparison and "
                f"bulk-demand treatment decision | {STATUS} |"
            )
            updated = original if row in original else original.rstrip() + "\n" + row + "\n"
        elif key == "CURRENT_HANDOFF":
            updated = "\n".join(
                [
                    "# Current Handoff",
                    "",
                    f"- Current completed step: {STEP_ID}",
                    f"- Status: {STATUS}",
                    f"- Updated local time: {NOW_LOCAL.isoformat()}",
                    f"- Target and bulk decision: {DECISION_PATH}",
                    f"- Primary modelled target status: {primary_modelled_target}",
                    f"- Selected unconfirmed bulk fallback: {selected_bulk_method}",
                    "- Confirmed future bulk quantities must be added externally.",
                    "- Final weekly method and scope are not yet selected.",
                    "- March 2026 targets were not read.",
                    (
                        "- Next step: 11I final weekly method and scope selection "
                        "from pre-holdout evidence."
                    ),
                    "",
                ]
            )
        else:
            raise KeyError(key)

        stage_text(stage_root, final_path, updated)

    step_text = f"""# Step 11H — Target Comparison and Bulk-Demand Treatment

- **Step ID:** 11H
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## Purpose
Compare WeeklyNormalDemand and WeeklyTotalDemand approaches, quantify bulk-demand behaviour, and freeze the bulk-treatment rule before final method and scope selection.

## Technical actions
- Verified the Part 11G and Part 11E locks.
- Used the same 39 pre-holdout chronological forecast weeks.
- Normalised Part 11E `InFoldStrict*` scope names to the canonical Part 11F/11G `PriorStrict*` names.
- Profiled product-level and week-level bulk demand.
- Compared zero, previous-context and four-context-mean unconfirmed-bulk fallbacks.
- Combined every locked normal-demand forecast source with each bulk fallback.
- Compared those strategies with the corresponding direct WeeklyTotalDemand forecast.
- Compared matched normal-target and total-target screening WAPE results.
- Created a target-and-bulk decision contract without fitting or refitting models.

## Decisions
{decisions_text}

## Outputs
{files_text}

## Results
{results_text}

## Validation
All {len(validation)} Part 11H validation checks passed.

## Errors corrected
- Part 11E baseline predictions used `InFoldStrict80/90/95PctScope`; these are now normalised to `PriorStrict80/90/95PctScope` before analysis.
- The model normal/total merge now uses the valid pandas argument `validate="one_to_one"`.

## Limitations
- Confirmed future bulk-order information is not present in the POS dataset and cannot be backtested here.
- Part 11H selects an operational target architecture and bulk rule, not the final forecasting method or product scope.
- March 2026 remains excluded from weekly selection.
- A new untouched future period is still required for unbiased final weekly evaluation.

## Next action
Part 11I: select and lock the final weekly forecasting method and product scope using pre-holdout chronological evidence only.
"""
    stage_text(stage_root, STEP_MEMORY_PATH, step_text)

    checkpoint = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "Input": {
            "Part11GLockSHA256": part11g_lock_hash,
            "Part11GCheckpointSHA256": part11g_checkpoint_hash,
            "Part11ELockSHA256": part11e_lock_hash,
            "CandidatePredictionsSHA256": predictions_hash,
            "CandidateRankingSHA256": ranking_hash,
        },
        "BulkAnalysis": {
            "EvaluationRows": len(evaluation_features),
            "EvaluationWeeks": len(forecast_weeks),
            "NormalDemandUnits": normal_total,
            "BulkDemandUnits": bulk_total,
            "TotalDemandUnits": total_total,
            "BulkShareOfTotalPercentage": bulk_share_percentage,
            "PositiveBulkProductWeeks": bulk_positive_rows,
            "ZeroBulkProductWeekPercentage": bulk_zero_percentage,
            "SelectedUnconfirmedBulkFallback": selected_bulk_method,
        },
        "Decision": decision,
        "Safety": {
            "ProtectedTargetVaultOpened": False,
            "ProtectedTargetVaultCopied": False,
            "OpenedMarch2026TargetsRead": False,
            "ModelsFittedOrRefitted": False,
            "FinalMethodSelected": False,
            "FinalScopeSelected": False,
            "ForecastsRoundedBeforeScoring": False,
        },
        "ReadyForPart11I": True,
        "NextStep": "11I",
    }
    stage_json(stage_root, CHECKPOINT_PATH, checkpoint)

    checkpoint_hash = sha256_file(stage_path(stage_root, CHECKPOINT_PATH))
    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        f"{checkpoint_hash}  {CHECKPOINT_PATH.name}\n",
    )

    manifest_targets = [
        BULK_PRODUCT_PROFILE_PATH,
        BULK_WEEK_SUMMARY_PATH,
        BULK_FALLBACK_PREDICTIONS_PATH,
        BULK_FALLBACK_METRICS_PATH,
        STRATEGY_PREDICTIONS_PATH,
        STRATEGY_METRICS_PATH,
        PAIRED_COMPARISON_PATH,
        TARGET_COMPARISON_PATH,
        DECISION_PATH,
        DECISION_TABLE_PATH,
        VALIDATION_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]

    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(EXT_ROOT)),
                "Bytes": stage_path(stage_root, path).stat().st_size,
                "SHA256": sha256_file(stage_path(stage_root, path)),
            }
            for path in manifest_targets
        ]
    ).sort_values("RelativePath")
    stage_csv(stage_root, MANIFEST_PATH, manifest)
    manifest_hash = sha256_file(stage_path(stage_root, MANIFEST_PATH))

    lock = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "Part11G": {
            "LockSHA256": part11g_lock_hash,
            "CheckpointSHA256": part11g_checkpoint_hash,
            "CandidatePredictionsSHA256": predictions_hash,
            "CandidateRankingSHA256": ranking_hash,
        },
        "Part11E": {
            "LockSHA256": part11e_lock_hash,
        },
        "TargetAndBulkDecision": {
            "Path": str(DECISION_PATH),
            "SHA256": sha256_file(stage_path(stage_root, DECISION_PATH)),
            "DecisionStatus": decision_status,
            "PrimaryModelledTargetForPart11I": primary_modelled_target,
            "SelectedUnconfirmedBulkFallback": selected_bulk_method,
            "FinalMethodSelected": False,
            "FinalScopeSelected": False,
        },
        "OutputHashManifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_hash,
            "FilesListed": len(manifest),
        },
        "Checkpoint": {
            "Path": str(CHECKPOINT_PATH),
            "SHA256": checkpoint_hash,
        },
        "SafetyAssertions": checkpoint["Safety"],
        "ReadyForPart11I": True,
        "NextStep": "11I",
    }
    stage_json(stage_root, LOCK_PATH, lock)

    lock_hash = sha256_file(stage_path(stage_root, LOCK_PATH))
    stage_text(stage_root, LOCK_SHA_PATH, f"{lock_hash}  {LOCK_PATH.name}\n")

    stage_text(
        stage_root,
        LOG_PATH,
        "\n".join(
            [
                f"Status: {STATUS}",
                f"Run local: {NOW_LOCAL.isoformat()}",
                f"Part 11G lock SHA256: {part11g_lock_hash}",
                f"Evaluation rows: {len(evaluation_features)}",
                f"Evaluation weeks: {len(forecast_weeks)}",
                f"Normal demand units: {normal_total}",
                f"Bulk demand units: {bulk_total}",
                f"Total demand units: {total_total}",
                f"Bulk share percentage: {bulk_share_percentage}",
                f"Selected unconfirmed bulk fallback: {selected_bulk_method}",
                f"Primary modelled target status: {primary_modelled_target}",
                f"Checkpoint SHA256: {checkpoint_hash}",
                f"Part 11H lock SHA256: {lock_hash}",
                "",
            ]
        ),
    )

    staged_files = [path for path in stage_root.rglob("*") if path.is_file()]
    if not staged_files:
        raise AssertionError("Part 11H staging directory is empty")
    if any(
        "target_vault" in path.name.lower() or path.suffix.lower() == ".joblib"
        for path in staged_files
    ):
        raise PermissionError(
            "Forbidden target-vault or model file detected in staging"
        )

    for staged in sorted(staged_files):
        final = EXT_ROOT / staged.relative_to(stage_root)
        final.parent.mkdir(parents=True, exist_ok=True)
        if final.exists() and final not in MEMORY_FILES.values():
            raise FileExistsError(f"Refusing to overwrite Part 11H output: {final}")
        os.replace(staged, final)

    for path in NEW_OUTPUTS:
        make_read_only(path)

finally:
    if stage_root.exists():
        shutil.rmtree(stage_root)

# =============================================================================
# 13. FINAL TECHNICAL OUTPUT
# =============================================================================

print("=" * 100)
print("EDEN WEEKLY FORECASTING EXTENSION — PART 11H COMPLETE")
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Extension root: {EXT_ROOT}")
print(f"Local time: {NOW_LOCAL.isoformat()}")

print("\nBULK-DEMAND PROFILE")
print(f"Evaluation product-weeks: {len(evaluation_features)}")
print(f"Evaluation weeks: {len(forecast_weeks)}")
print(f"Normal demand units: {normal_total:.6f}")
print(f"Bulk demand units: {bulk_total:.6f}")
print(f"Total demand units: {total_total:.6f}")
print(f"Bulk share of total demand: {bulk_share_percentage:.6f}%")
print(f"Positive bulk product-weeks: {bulk_positive_rows}")
print(f"Zero-bulk product-week percentage: {bulk_zero_percentage:.6f}%")
print(f"Products with any bulk demand: {products_with_bulk}")
print(f"Products covering 80% of bulk demand: {bulk_top_80_count}")
print(f"Products covering 95% of bulk demand: {bulk_top_95_count}")

print("\nBULK FALLBACK RANKING")
print(
    bulk_fallback_metrics[
        [
            "ScopeID",
            "BulkMethodRank",
            "BulkMethodID",
            "Observations",
            "ActualTotal",
            "PredictedTotal",
            "MAE",
            "RMSE",
            "WAPEPercentage",
            "MeanBias",
            "TotalBias",
            "IsBulkReferenceFallback",
        ]
    ].to_string(index=False)
)

print("\nDIRECT NORMAL-TARGET VERSUS TOTAL-TARGET COMPARISON")
print(target_comparison.to_string(index=False))

print("\nDIRECT TOTAL VERSUS NORMAL PLUS SELECTED BULK FALLBACK")
print(paired_comparison.to_string(index=False))

print("\nPART 11H DECISION")
print(decision_table.to_string(index=False))
print(
    f"Normal target better comparisons: {normal_target_better_count}/"
    f"{target_pair_count}"
)
print(
    f"Normal plus selected bulk wins: {normal_plus_bulk_win_count}/"
    f"{combined_pair_count}"
)
print(f"Selected unconfirmed bulk fallback: {selected_bulk_method}")
print(f"Decision status: {decision_status}")

print("\nPART 11H VALIDATION")
print(validation.to_string(index=False))

print("\nCONTROL OUTPUTS")
print(f"- Bulk product profile: {BULK_PRODUCT_PROFILE_PATH}")
print(f"- Bulk week summary: {BULK_WEEK_SUMMARY_PATH}")
print(f"- Bulk fallback metrics: {BULK_FALLBACK_METRICS_PATH}")
print(f"- Target-strategy metrics: {STRATEGY_METRICS_PATH}")
print(f"- Paired target comparison: {PAIRED_COMPARISON_PATH}")
print(f"- Target and bulk decision: {DECISION_PATH}")
print(f"- Checkpoint: {CHECKPOINT_PATH}")
print(f"- Checkpoint SHA256: {checkpoint_hash}")
print(f"- Part 11H lock: {LOCK_PATH}")
print(f"- Part 11H lock SHA256: {lock_hash}")

print(
    "\nSAFETY: protected target vault opened/copied False/False; opened "
    "March 2026 targets read False; models fitted/refitted False; final "
    "method selected False; final scope selected False; forecasts rounded "
    "before scoring False."
)
print("=" * 100)

Scope-column normalization passed:
- Part 11F features: PriorStrict80PctScope, PriorStrict90PctScope, PriorStrict95PctScope
- Part 11E baseline predictions: InFoldStrict* converted to PriorStrict*
- Part 11G model predictions: PriorStrict80PctScope, PriorStrict90PctScope, PriorStrict95PctScope
EDEN WEEKLY FORECASTING EXTENSION — PART 11H COMPLETE
Status: PART_11H_COMPLETED_READY_FOR_11I
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-05T21:41:51.747738+01:00

BULK-DEMAND PROFILE
Evaluation product-weeks: 7212
Evaluation weeks: 39
Normal demand units: 87052.000000
Bulk demand units: 1972.000000
Total demand units: 89024.000000
Bulk share of total demand: 2.215133%
Positive bulk product-weeks: 24
Zero-bulk product-week percentage: 99.667221%
Products with any bulk demand: 11
Products covering 80% of bulk demand: 5
Products covering 95% of bulk demand: 8

BULK FALLBACK RANKING
               ScopeID  BulkMethodRank       

In [25]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import stat
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11I
# Final pre-holdout weekly method and scope selection and lock
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"
MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
MODEL_ROOT = EXT_ROOT / "06_weekly_models"
FINAL_MODEL_ROOT = MODEL_ROOT / "final"
BASELINE_ROOT = EXT_ROOT / "05_weekly_baselines"
VALIDATION_ROOT = EXT_ROOT / "07_weekly_validation"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
LOG_ROOT = EXT_ROOT / "12_logs"

PART11E_PREDICTIONS = BASELINE_ROOT / "11E_preholdout_weekly_baseline_predictions.csv"
PART11E_RANKING = BASELINE_ROOT / "11E_preholdout_baseline_ranking.csv"
PART11E_DESIGN_CONTRACT = VALIDATION_ROOT / "11E_baseline_design_contract.json"
PART11E_LOCK = CHECKPOINT_ROOT / "11E_weekly_baselines_lock.json"
PART11E_LOCK_SHA = CHECKPOINT_ROOT / "11E_weekly_baselines_lock.sha256"

PART11G_PREDICTIONS = VALIDATION_ROOT / "11G_candidate_weekly_predictions.csv"
PART11G_RANKING = VALIDATION_ROOT / "11G_candidate_model_ranking.csv"
PART11G_LOCK = CHECKPOINT_ROOT / "11G_weekly_model_screening_lock.json"
PART11G_LOCK_SHA = CHECKPOINT_ROOT / "11G_weekly_model_screening_lock.sha256"

PART11H_DECISION = VALIDATION_ROOT / "11H_target_and_bulk_decision.json"
PART11H_LOCK = CHECKPOINT_ROOT / "11H_target_and_bulk_decision_lock.json"
PART11H_LOCK_SHA = CHECKPOINT_ROOT / "11H_target_and_bulk_decision_lock.sha256"
PART11H_CHECKPOINT = CHECKPOINT_ROOT / "11H_checkpoint.json"
PART11H_CHECKPOINT_SHA = CHECKPOINT_ROOT / "11H_checkpoint.sha256"

METHOD_COMPARISON_PATH = VALIDATION_ROOT / "11I_final_method_comparison.csv"
METHOD_RECONCILIATION_PATH = VALIDATION_ROOT / "11I_published_metric_reconciliation.csv"
FOLD_METRICS_PATH = VALIDATION_ROOT / "11I_method_fold_metrics.csv"
FOLD_STABILITY_PATH = VALIDATION_ROOT / "11I_method_fold_stability.csv"
SCOPE_WINNERS_PATH = VALIDATION_ROOT / "11I_scope_method_winners.csv"
SCOPE_TRADEOFF_PATH = VALIDATION_ROOT / "11I_scope_tradeoff.csv"
SELECTED_PREDICTIONS_PATH = VALIDATION_ROOT / "11I_selected_preholdout_predictions.csv"
SELECTED_FOLD_METRICS_PATH = VALIDATION_ROOT / "11I_selected_preholdout_fold_metrics.csv"
SELECTION_EVIDENCE_PATH = VALIDATION_ROOT / "11I_final_selection_evidence.csv"
FINAL_DESIGN_PATH = FINAL_MODEL_ROOT / "11I_final_weekly_forecasting_design.json"
FINAL_DESIGN_TABLE_PATH = FINAL_MODEL_ROOT / "11I_final_weekly_forecasting_design.csv"
VALIDATION_PATH = VALIDATION_ROOT / "11I_validation.csv"
MANIFEST_PATH = VALIDATION_ROOT / "11I_output_hash_manifest.csv"
STEP_MEMORY_PATH = STEP_MEMORY_ROOT / "STEP_11I_FINAL_WEEKLY_SELECTION.md"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "11I_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "11I_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.sha256"
LOG_PATH = LOG_ROOT / "11I_final_weekly_selection_log.txt"

MEMORY_FILES = {
    "PROJECT_CONTEXT": MEMORY_ROOT / "PROJECT_CONTEXT.md",
    "WORKFLOW": MEMORY_ROOT / "WORKFLOW.md",
    "DECISIONS": MEMORY_ROOT / "DECISIONS.md",
    "FILES_AND_PATHS": MEMORY_ROOT / "FILES_AND_PATHS.md",
    "METRICS_AND_RESULTS": MEMORY_ROOT / "METRICS_AND_RESULTS.md",
    "CHAT_INDEX": MEMORY_ROOT / "CHAT_INDEX.md",
    "CURRENT_HANDOFF": MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

STEP_ID = "11I"
STATUS = "PART_11I_COMPLETED_READY_FOR_11J"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

FINAL_TARGET_REQUIRED = "WEEKLY_NORMAL_DEMAND"
FINAL_TARGET_COLUMN = "WeeklyNormalDemand"
EXPECTED_BULK_FALLBACK = "ZERO_BULK"
SCOPE_TOLERANCE_PERCENT = 5.0

SCOPE_FLAGS = {
    "ALL_PRODUCTS": None,
    "FOLD_STRICT_80_PERCENT": "PriorStrict80PctScope",
    "FOLD_STRICT_90_PERCENT": "PriorStrict90PctScope",
    "FOLD_STRICT_95_PERCENT": "PriorStrict95PctScope",
}

SCOPE_NOMINAL_COVERAGE = {
    "ALL_PRODUCTS": 100,
    "FOLD_STRICT_80_PERCENT": 80,
    "FOLD_STRICT_90_PERCENT": 90,
    "FOLD_STRICT_95_PERCENT": 95,
}

BASELINE_COMPLEXITY_ORDER = {
    "ZERO": 0,
    "NAIVE_LAST_CONTEXT": 1,
    "MEAN_LAST_4_CONTEXTS": 2,
    "MEDIAN_LAST_4_CONTEXTS": 3,
    "MEAN_LAST_8_CONTEXTS": 4,
    "EXPANDING_MEAN_CONTEXTS": 5,
}

MODEL_COMPLEXITY_ORDER = {
    "GLOBAL_RIDGE": 10,
    "GLOBAL_HISTGB": 11,
    "SEGMENT_FEATURE_RIDGE": 12,
    "GLOBAL_HURDLE_HISTGB": 13,
    "ROUTED_SEGMENT_HISTGB": 14,
}

CANONICAL_SCOPE_COLUMNS = [
    "PriorStrict80PctScope",
    "PriorStrict90PctScope",
    "PriorStrict95PctScope",
]

BASELINE_SCOPE_ALIASES = {
    "InFoldStrict80PctScope": "PriorStrict80PctScope",
    "InFoldStrict90PctScope": "PriorStrict90PctScope",
    "InFoldStrict95PctScope": "PriorStrict95PctScope",
}

NEW_OUTPUTS = [
    METHOD_COMPARISON_PATH,
    METHOD_RECONCILIATION_PATH,
    FOLD_METRICS_PATH,
    FOLD_STABILITY_PATH,
    SCOPE_WINNERS_PATH,
    SCOPE_TRADEOFF_PATH,
    SELECTED_PREDICTIONS_PATH,
    SELECTED_FOLD_METRICS_PATH,
    SELECTION_EVIDENCE_PATH,
    FINAL_DESIGN_PATH,
    FINAL_DESIGN_TABLE_PATH,
    VALIDATION_PATH,
    MANIFEST_PATH,
    STEP_MEMORY_PATH,
    CHECKPOINT_PATH,
    CHECKPOINT_SHA_PATH,
    LOCK_PATH,
    LOCK_SHA_PATH,
    LOG_PATH,
]

# =============================================================================
# HELPERS
# =============================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required {label}:\n{path}"
        )


def verify_sidecar(
    path: Path,
    sidecar: Path,
    label: str,
) -> str:
    require_file(path, label)
    require_file(sidecar, f"{label} SHA-256 sidecar")

    expected = (
        sidecar
        .read_text(encoding="utf-8")
        .strip()
        .split()[0]
        .lower()
    )

    actual = sha256_file(path)

    if len(expected) != 64 or expected != actual:
        raise AssertionError(
            f"{label} SHA-256 mismatch:\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.{uuid.uuid4().hex}.tmp"
    )

    try:
        temporary.write_text(
            text,
            encoding="utf-8",
        )

        os.replace(
            temporary,
            path,
        )

    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_csv(
    path: Path,
    frame: pd.DataFrame,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.{uuid.uuid4().hex}.tmp"
    )

    try:
        frame.to_csv(
            temporary,
            index=False,
        )

        os.replace(
            temporary,
            path,
        )

    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_json(
    path: Path,
    payload: dict,
) -> None:
    atomic_text(
        path,
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def bool_series(
    series: pd.Series,
    label: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    normalized = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    invalid = sorted(
        set(normalized)
        - set(mapping)
    )

    if invalid:
        raise ValueError(
            f"{label} has invalid Boolean values: "
            f"{invalid[:10]}"
        )

    return (
        normalized
        .map(mapping)
        .astype(bool)
    )


def normalize_scope_columns(
    frame: pd.DataFrame,
    frame_label: str,
    aliases: dict[str, str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    aliases = aliases or {}

    rename_map: dict[str, str] = {}

    for source, target in aliases.items():
        source_exists = source in result.columns
        target_exists = target in result.columns

        if source_exists and target_exists:
            source_values = bool_series(
                result[source],
                f"{frame_label}.{source}",
            )

            target_values = bool_series(
                result[target],
                f"{frame_label}.{target}",
            )

            if not source_values.equals(target_values):
                raise AssertionError(
                    f"{frame_label} contains both "
                    f"{source} and {target}, "
                    "but values differ."
                )

            result = result.drop(
                columns=[source]
            )

        elif source_exists and not target_exists:
            rename_map[source] = target

    if rename_map:
        result = result.rename(
            columns=rename_map
        )

    missing = [
        column
        for column in CANONICAL_SCOPE_COLUMNS
        if column not in result.columns
    ]

    if missing:
        scope_like = sorted(
            column
            for column in result.columns
            if (
                "Strict" in column
                or "Scope" in column
            )
        )

        raise AssertionError(
            f"{frame_label} is missing scope columns "
            f"after normalization: {missing}\n"
            f"Available scope-like columns: {scope_like}"
        )

    for column in CANONICAL_SCOPE_COLUMNS:
        result[column] = bool_series(
            result[column],
            f"{frame_label}.{column}",
        )

    return result


def metric_record(
    actual: pd.Series,
    prediction: pd.Series,
) -> dict:
    actual_array = (
        pd.to_numeric(
            actual,
            errors="raise",
        )
        .to_numpy(dtype=float)
    )

    prediction_array = (
        pd.to_numeric(
            prediction,
            errors="raise",
        )
        .to_numpy(dtype=float)
    )

    if len(actual_array) != len(prediction_array):
        raise AssertionError(
            "Actual and prediction lengths differ"
        )

    if (
        not np.isfinite(actual_array).all()
        or not np.isfinite(prediction_array).all()
    ):
        raise AssertionError(
            "Non-finite values entered metric calculation"
        )

    errors = (
        prediction_array
        - actual_array
    )

    absolute_errors = np.abs(errors)

    denominator = float(
        np.abs(actual_array).sum()
    )

    return {
        "Observations":
            int(len(actual_array)),

        "ActualTotal":
            float(actual_array.sum()),

        "PredictedTotal":
            float(prediction_array.sum()),

        "MAE":
            (
                float(absolute_errors.mean())
                if len(actual_array)
                else np.nan
            ),

        "RMSE":
            (
                float(
                    np.sqrt(
                        np.mean(errors ** 2)
                    )
                )
                if len(actual_array)
                else np.nan
            ),

        "WAPEPercentage":
            (
                float(
                    100.0
                    * absolute_errors.sum()
                    / denominator
                )
                if denominator != 0
                else np.nan
            ),

        "MeanBias":
            (
                float(errors.mean())
                if len(actual_array)
                else np.nan
            ),

        "TotalBias":
            float(errors.sum()),
    }


def update_section(
    text: str,
    marker: str,
    heading: str,
    body: str,
) -> str:
    start = f"<!-- BEGIN {marker} -->"
    end = f"<!-- END {marker} -->"

    section = (
        f"{start}\n"
        f"## {heading}\n\n"
        f"{body.rstrip()}\n"
        f"{end}"
    )

    if start in text and end in text:
        before = (
            text
            .split(start, 1)[0]
            .rstrip()
        )

        after = (
            text
            .split(end, 1)[1]
            .lstrip()
        )

        return (
            before
            + "\n\n"
            + section
            + (
                "\n\n" + after
                if after
                else ""
            )
            + "\n"
        )

    return (
        text.rstrip()
        + "\n\n"
        + section
        + "\n"
    )


def stage_path(
    stage_root: Path,
    final_path: Path,
) -> Path:
    return (
        stage_root
        / final_path.relative_to(EXT_ROOT)
    )


def stage_text(
    stage_root: Path,
    final_path: Path,
    text: str,
) -> None:
    atomic_text(
        stage_path(stage_root, final_path),
        text,
    )


def stage_csv(
    stage_root: Path,
    final_path: Path,
    frame: pd.DataFrame,
) -> None:
    atomic_csv(
        stage_path(stage_root, final_path),
        frame,
    )


def stage_json(
    stage_root: Path,
    final_path: Path,
    payload: dict,
) -> None:
    atomic_json(
        stage_path(stage_root, final_path),
        payload,
    )


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(
            stat.S_IRUSR
            | stat.S_IRGRP
            | stat.S_IROTH
        )


def method_complexity(
    method_id: str,
    method_class: str,
) -> int:
    if method_class == "BASELINE":
        return BASELINE_COMPLEXITY_ORDER.get(
            method_id,
            99,
        )

    return MODEL_COMPLEXITY_ORDER.get(
        method_id,
        199,
    )


# =============================================================================
# VERIFY INPUTS AND LOCK CHAIN
# =============================================================================

for directory, label in [
    (EXT_ROOT, "extension root"),
    (MEMORY_ROOT, "memory root"),
    (STEP_MEMORY_ROOT, "step-memory directory"),
    (MODEL_ROOT, "weekly model directory"),
    (BASELINE_ROOT, "weekly baseline directory"),
    (VALIDATION_ROOT, "weekly validation directory"),
    (CHECKPOINT_ROOT, "checkpoint directory"),
    (LOG_ROOT, "log directory"),
]:
    if not directory.is_dir():
        raise FileNotFoundError(
            f"Missing required {label}:\n"
            f"{directory}"
        )


for path, label in [
    (
        PART11E_PREDICTIONS,
        "Part 11E baseline predictions",
    ),
    (
        PART11E_RANKING,
        "Part 11E baseline ranking",
    ),
    (
        PART11E_DESIGN_CONTRACT,
        "Part 11E baseline design contract",
    ),
    (
        PART11G_PREDICTIONS,
        "Part 11G candidate predictions",
    ),
    (
        PART11G_RANKING,
        "Part 11G candidate ranking",
    ),
    (
        PART11H_DECISION,
        "Part 11H target and bulk decision",
    ),
]:
    require_file(
        path,
        label,
    )


for key, path in MEMORY_FILES.items():
    require_file(
        path,
        f"memory file {key}",
    )


if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise FileExistsError(
        "Part 11I overwrite lock triggered:\n"
        f"{LOCK_PATH}"
    )


existing_outputs = [
    path
    for path in NEW_OUTPUTS
    if path.exists()
]

if existing_outputs:
    raise FileExistsError(
        "Existing uncommitted Part 11I outputs "
        "found; no files changed:\n"
        + "\n".join(
            f"- {path}"
            for path in existing_outputs
        )
    )


for old_stage in EXT_ROOT.glob(
    ".11I_staging_*"
):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)


part11e_lock_hash = verify_sidecar(
    PART11E_LOCK,
    PART11E_LOCK_SHA,
    "Part 11E lock",
)

part11g_lock_hash = verify_sidecar(
    PART11G_LOCK,
    PART11G_LOCK_SHA,
    "Part 11G lock",
)

part11h_lock_hash = verify_sidecar(
    PART11H_LOCK,
    PART11H_LOCK_SHA,
    "Part 11H lock",
)

part11h_checkpoint_hash = verify_sidecar(
    PART11H_CHECKPOINT,
    PART11H_CHECKPOINT_SHA,
    "Part 11H checkpoint",
)


part11g_lock = json.loads(
    PART11G_LOCK.read_text(
        encoding="utf-8"
    )
)

part11h_lock = json.loads(
    PART11H_LOCK.read_text(
        encoding="utf-8"
    )
)

part11h_decision = json.loads(
    PART11H_DECISION.read_text(
        encoding="utf-8"
    )
)


if (
    part11h_lock.get("Status")
    != "PART_11H_COMPLETED_READY_FOR_11I"
):
    raise AssertionError(
        "Unexpected Part 11H status: "
        f"{part11h_lock.get('Status')}"
    )


if (
    part11h_lock.get("ReadyForPart11I")
    is not True
):
    raise AssertionError(
        "Part 11H lock does not authorise Part 11I"
    )


if (
    part11h_lock
    .get("Part11G", {})
    .get("LockSHA256")
    != part11g_lock_hash
):
    raise AssertionError(
        "Part 11H does not reference "
        "the current Part 11G lock"
    )


if (
    part11g_lock
    .get("Part11E", {})
    .get("LockSHA256")
    != part11e_lock_hash
):
    raise AssertionError(
        "Part 11G does not reference "
        "the current Part 11E lock"
    )


if (
    sha256_file(PART11H_DECISION)
    !=
    part11h_lock
    .get("TargetAndBulkDecision", {})
    .get("SHA256")
):
    raise AssertionError(
        "Part 11H decision hash changed"
    )


for key, expected in {
    "ProtectedTargetVaultOpened":
        False,

    "ProtectedTargetVaultCopied":
        False,

    "OpenedMarch2026TargetsRead":
        False,

    "ModelsFittedOrRefitted":
        False,

    "FinalMethodSelected":
        False,

    "FinalScopeSelected":
        False,

    "ForecastsRoundedBeforeScoring":
        False,
}.items():
    actual = (
        part11h_lock
        .get("SafetyAssertions", {})
        .get(key)
    )

    if actual is not expected:
        raise AssertionError(
            "Invalid Part 11H safety assertion "
            f"{key}: {actual}"
        )


if (
    part11h_decision.get(
        "PrimaryModelledTargetForPart11I"
    )
    != FINAL_TARGET_REQUIRED
):
    raise AssertionError(
        "Part 11H did not approve "
        "WEEKLY_NORMAL_DEMAND for Part 11I"
    )


if (
    part11h_decision.get(
        "SelectedUnconfirmedBulkFallback"
    )
    != EXPECTED_BULK_FALLBACK
):
    raise AssertionError(
        "Part 11H unconfirmed-bulk fallback "
        "is not ZERO_BULK"
    )


proceeding_scopes = list(
    part11h_decision.get(
        "ScopeCandidatesProceedingToPart11I",
        [],
    )
)

required_proceeding_scopes = [
    "FOLD_STRICT_80_PERCENT",
    "FOLD_STRICT_90_PERCENT",
    "FOLD_STRICT_95_PERCENT",
]

if (
    set(proceeding_scopes)
    != set(required_proceeding_scopes)
):
    raise AssertionError(
        "Unexpected Part 11H scope candidates: "
        f"{proceeding_scopes}"
    )


baseline_predictions_hash = sha256_file(
    PART11E_PREDICTIONS
)

baseline_ranking_hash = sha256_file(
    PART11E_RANKING
)

baseline_contract_hash = sha256_file(
    PART11E_DESIGN_CONTRACT
)

model_predictions_hash = sha256_file(
    PART11G_PREDICTIONS
)

model_ranking_hash = sha256_file(
    PART11G_RANKING
)


if (
    model_predictions_hash
    !=
    part11h_lock
    .get("Part11G", {})
    .get("CandidatePredictionsSHA256")
):
    raise AssertionError(
        "Part 11G candidate-prediction hash "
        "changed after Part 11H"
    )


if (
    model_ranking_hash
    !=
    part11h_lock
    .get("Part11G", {})
    .get("CandidateRankingSHA256")
):
    raise AssertionError(
        "Part 11G candidate-ranking hash "
        "changed after Part 11H"
    )


# =============================================================================
# LOAD AND NORMALIZE LOCKED PRE-HOLDOUT PREDICTIONS
# =============================================================================

baseline_predictions = pd.read_csv(
    PART11E_PREDICTIONS,
    low_memory=False,
)

baseline_ranking = pd.read_csv(
    PART11E_RANKING,
    low_memory=False,
)

model_predictions = pd.read_csv(
    PART11G_PREDICTIONS,
    low_memory=False,
)

model_ranking = pd.read_csv(
    PART11G_RANKING,
    low_memory=False,
)


for frame in [
    baseline_predictions,
    model_predictions,
]:
    for column in [
        "WeekStartDate",
        "WeekEndDate",
    ]:
        frame[column] = pd.to_datetime(
            frame[column],
            errors="raise",
        )

    frame["CanonicalProductID"] = (
        frame["CanonicalProductID"]
        .astype(str)
    )


baseline_predictions = normalize_scope_columns(
    baseline_predictions,
    "Part 11E baseline predictions",
    aliases=BASELINE_SCOPE_ALIASES,
)

model_predictions = normalize_scope_columns(
    model_predictions,
    "Part 11G candidate predictions",
)


required_baseline_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "TargetID",
    "BaselineMethod",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
}

required_model_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "TargetID",
    "CandidateSystemID",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
}


if not required_baseline_columns.issubset(
    baseline_predictions.columns
):
    raise AssertionError(
        "Part 11E baseline-prediction "
        "schema is incomplete"
    )


if not required_model_columns.issubset(
    model_predictions.columns
):
    raise AssertionError(
        "Part 11G model-prediction "
        "schema is incomplete"
    )


baseline_target = baseline_predictions.loc[
    baseline_predictions["TargetID"].astype(str)
    == FINAL_TARGET_REQUIRED
].copy()

model_target = model_predictions.loc[
    model_predictions["TargetID"].astype(str)
    == FINAL_TARGET_REQUIRED
].copy()


if baseline_target.empty or model_target.empty:
    raise AssertionError(
        "Normal-demand prediction rows are missing"
    )


baseline_target["MethodID"] = (
    baseline_target["BaselineMethod"]
    .astype(str)
)

baseline_target["MethodClass"] = "BASELINE"

model_target["MethodID"] = (
    model_target["CandidateSystemID"]
    .astype(str)
)

model_target["MethodClass"] = (
    "MACHINE_LEARNING"
)


common_columns = [
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "TargetID",
    "MethodID",
    "MethodClass",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
]


all_predictions = pd.concat(
    [
        baseline_target[common_columns],
        model_target[common_columns],
    ],
    ignore_index=True,
)


all_predictions["Actual"] = (
    pd.to_numeric(
        all_predictions["Actual"],
        errors="raise",
    )
    .astype(float)
)

all_predictions["Prediction"] = (
    pd.to_numeric(
        all_predictions["Prediction"],
        errors="raise",
    )
    .astype(float)
)


if not np.isfinite(
    all_predictions[
        ["Actual", "Prediction"]
    ].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "Non-finite values found in "
        "locked prediction files"
    )


if (
    all_predictions["Prediction"]
    < 0
).any():
    raise AssertionError(
        "Negative locked forecasts found"
    )


if all_predictions.duplicated(
    [
        "WeekStartDate",
        "CanonicalProductID",
        "MethodID",
    ]
).any():
    raise AssertionError(
        "Duplicate method/product/week rows found"
    )


forecast_weeks = sorted(
    all_predictions[
        "WeekStartDate"
    ].drop_duplicates()
)


if len(forecast_weeks) != 39:
    raise AssertionError(
        "Expected 39 pre-holdout forecast weeks, "
        f"found {len(forecast_weeks)}"
    )


method_counts = (
    all_predictions
    .groupby(
        [
            "MethodID",
            "MethodClass",
        ],
        as_index=False,
    )
    .agg(
        PredictionRows=(
            "Prediction",
            "size",
        ),
        ForecastWeeks=(
            "WeekStartDate",
            "nunique",
        ),
    )
)


if (
    method_counts["ForecastWeeks"]
    != 39
).any():
    raise AssertionError(
        "Not every method covers all "
        "39 forecast weeks"
    )


actual_reconciliation = (
    all_predictions
    .groupby(
        [
            "WeekStartDate",
            "CanonicalProductID",
        ],
        as_index=False,
    )
    .agg(
        ActualNunique=(
            "Actual",
            "nunique",
        ),

        Strict80Nunique=(
            "PriorStrict80PctScope",
            "nunique",
        ),

        Strict90Nunique=(
            "PriorStrict90PctScope",
            "nunique",
        ),

        Strict95Nunique=(
            "PriorStrict95PctScope",
            "nunique",
        ),
    )
)


actual_scope_mismatches = int(
    (
        actual_reconciliation[
            [
                "ActualNunique",
                "Strict80Nunique",
                "Strict90Nunique",
                "Strict95Nunique",
            ]
        ]
        != 1
    )
    .any(axis=1)
    .sum()
)


if actual_scope_mismatches:
    raise AssertionError(
        "Actual or scope membership differs "
        "between methods for "
        f"{actual_scope_mismatches} rows"
    )


unique_actual_rows = (
    all_predictions
    .sort_values(
        [
            "WeekStartDate",
            "CanonicalProductID",
            "MethodID",
        ]
    )
    .drop_duplicates(
        [
            "WeekStartDate",
            "CanonicalProductID",
        ]
    )
)


all_actual_total = float(
    unique_actual_rows["Actual"].sum()
)

all_observations = int(
    len(unique_actual_rows)
)


# =============================================================================
# RECOMPUTE ALL METHOD METRICS AND FOLD STABILITY
# =============================================================================

method_rows = []


for (
    method_id,
    method_class,
), group in all_predictions.groupby(
    [
        "MethodID",
        "MethodClass",
    ],
    sort=True,
):
    for (
        scope_id,
        flag,
    ) in SCOPE_FLAGS.items():
        scoped = (
            group
            if flag is None
            else group.loc[group[flag]]
        )

        metrics = metric_record(
            scoped["Actual"],
            scoped["Prediction"],
        )

        method_rows.append(
            {
                "TargetID":
                    FINAL_TARGET_REQUIRED,

                "TargetColumn":
                    FINAL_TARGET_COLUMN,

                "ScopeID":
                    scope_id,

                "NominalScopeCoveragePercentage":
                    SCOPE_NOMINAL_COVERAGE[
                        scope_id
                    ],

                "MethodID":
                    method_id,

                "MethodClass":
                    method_class,

                "ForecastWeeks":
                    int(
                        scoped[
                            "WeekStartDate"
                        ].nunique()
                    ),

                "Products":
                    int(
                        scoped[
                            "CanonicalProductID"
                        ].nunique()
                    ),

                "ObservationCoveragePercentage":
                    (
                        100.0
                        * metrics["Observations"]
                        / all_observations
                    ),

                "EvaluationActualDemandCoveragePercentage":
                    (
                        100.0
                        * metrics["ActualTotal"]
                        / all_actual_total
                        if all_actual_total
                        else np.nan
                    ),

                **metrics,
            }
        )


method_comparison = pd.DataFrame(
    method_rows
)

method_comparison[
    "AbsoluteTotalBias"
] = (
    method_comparison["TotalBias"]
    .abs()
)

method_comparison[
    "MethodComplexityOrder"
] = method_comparison.apply(
    lambda row:
        method_complexity(
            str(row["MethodID"]),
            str(row["MethodClass"]),
        ),
    axis=1,
)


method_comparison = (
    method_comparison
    .sort_values(
        [
            "ScopeID",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "AbsoluteTotalBias",
            "MethodComplexityOrder",
            "MethodID",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


method_comparison[
    "MethodRankWithinScope"
] = (
    method_comparison
    .groupby("ScopeID")
    .cumcount()
    + 1
)


method_comparison[
    "IsMethodWinnerWithinScope"
] = (
    method_comparison[
        "MethodRankWithinScope"
    ]
    == 1
)


fold_rows = []


for (
    method_id,
    method_class,
    week,
), group in all_predictions.groupby(
    [
        "MethodID",
        "MethodClass",
        "WeekStartDate",
    ],
    sort=True,
):
    for (
        scope_id,
        flag,
    ) in SCOPE_FLAGS.items():
        scoped = (
            group
            if flag is None
            else group.loc[group[flag]]
        )

        fold_rows.append(
            {
                "ForecastWeek":
                    week,

                "TargetID":
                    FINAL_TARGET_REQUIRED,

                "ScopeID":
                    scope_id,

                "MethodID":
                    method_id,

                "MethodClass":
                    method_class,

                **metric_record(
                    scoped["Actual"],
                    scoped["Prediction"],
                ),
            }
        )


fold_metrics = pd.DataFrame(
    fold_rows
)

fold_metrics[
    "AbsoluteTotalBias"
] = (
    fold_metrics["TotalBias"]
    .abs()
)

fold_metrics[
    "MethodComplexityOrder"
] = fold_metrics.apply(
    lambda row:
        method_complexity(
            str(row["MethodID"]),
            str(row["MethodClass"]),
        ),
    axis=1,
)


fold_metrics = (
    fold_metrics
    .sort_values(
        [
            "ForecastWeek",
            "ScopeID",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "AbsoluteTotalBias",
            "MethodComplexityOrder",
            "MethodID",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


fold_metrics[
    "WeeklyMethodRankWithinScope"
] = (
    fold_metrics
    .groupby(
        [
            "ForecastWeek",
            "ScopeID",
        ]
    )
    .cumcount()
    + 1
)


fold_metrics[
    "WonForecastWeek"
] = (
    fold_metrics[
        "WeeklyMethodRankWithinScope"
    ]
    == 1
)


fold_stability = (
    fold_metrics
    .groupby(
        [
            "TargetID",
            "ScopeID",
            "MethodID",
            "MethodClass",
        ],
        as_index=False,
    )
    .agg(
        ForecastWeeks=(
            "ForecastWeek",
            "nunique",
        ),

        MeanWeeklyWAPEPercentage=(
            "WAPEPercentage",
            "mean",
        ),

        MedianWeeklyWAPEPercentage=(
            "WAPEPercentage",
            "median",
        ),

        StdWeeklyWAPEPercentage=(
            "WAPEPercentage",
            "std",
        ),

        MinimumWeeklyWAPEPercentage=(
            "WAPEPercentage",
            "min",
        ),

        MaximumWeeklyWAPEPercentage=(
            "WAPEPercentage",
            "max",
        ),

        WeeksWon=(
            "WonForecastWeek",
            "sum",
        ),

        MeanWeeklyBias=(
            "MeanBias",
            "mean",
        ),

        CumulativeFoldBias=(
            "TotalBias",
            "sum",
        ),
    )
)


# =============================================================================
# RECONCILE RECOMPUTED METRICS WITH PUBLISHED 11E AND 11G RANKINGS
# =============================================================================

metric_columns = [
    "Observations",
    "ActualTotal",
    "PredictedTotal",
    "MAE",
    "RMSE",
    "WAPEPercentage",
    "MeanBias",
    "TotalBias",
]

required_metric_columns = {
    "TargetID",
    "ScopeID",
    *metric_columns,
}


if (
    not required_metric_columns.issubset(
        baseline_ranking.columns
    )
    or "BaselineMethod"
    not in baseline_ranking.columns
):
    raise AssertionError(
        "Part 11E ranking schema is incomplete"
    )


if (
    not required_metric_columns.issubset(
        model_ranking.columns
    )
    or "CandidateSystemID"
    not in model_ranking.columns
):
    raise AssertionError(
        "Part 11G ranking schema is incomplete"
    )


published_baseline = baseline_ranking.loc[
    baseline_ranking["TargetID"].astype(str)
    == FINAL_TARGET_REQUIRED
].copy()

published_baseline["MethodID"] = (
    published_baseline["BaselineMethod"]
    .astype(str)
)

published_baseline[
    "MethodClass"
] = "BASELINE"


published_model = model_ranking.loc[
    model_ranking["TargetID"].astype(str)
    == FINAL_TARGET_REQUIRED
].copy()

published_model["MethodID"] = (
    published_model["CandidateSystemID"]
    .astype(str)
)

published_model[
    "MethodClass"
] = "MACHINE_LEARNING"


published_metrics = pd.concat(
    [
        published_baseline,
        published_model,
    ],
    ignore_index=True,
)


reconciliation = published_metrics[
    [
        "TargetID",
        "ScopeID",
        "MethodID",
        "MethodClass",
        *metric_columns,
    ]
].merge(
    method_comparison[
        [
            "TargetID",
            "ScopeID",
            "MethodID",
            "MethodClass",
            *metric_columns,
        ]
    ],
    on=[
        "TargetID",
        "ScopeID",
        "MethodID",
        "MethodClass",
    ],
    how="outer",
    suffixes=(
        "__Published",
        "__Recomputed",
    ),
    indicator=True,
    validate="one_to_one",
)


for metric in metric_columns:
    reconciliation[
        f"{metric}AbsoluteDifference"
    ] = (
        pd.to_numeric(
            reconciliation[
                f"{metric}__Published"
            ],
            errors="coerce",
        )
        - pd.to_numeric(
            reconciliation[
                f"{metric}__Recomputed"
            ],
            errors="coerce",
        )
    ).abs()


reconciliation[
    "MaximumMetricAbsoluteDifference"
] = reconciliation[
    [
        f"{metric}AbsoluteDifference"
        for metric in metric_columns
    ]
].max(
    axis=1,
    skipna=True,
)


reconciliation[
    "Passed"
] = (
    (
        reconciliation["_merge"]
        == "both"
    )
    & (
        reconciliation[
            "MaximumMetricAbsoluteDifference"
        ]
        <= 1e-6
    )
)


metric_reconciliation_failures = int(
    (
        ~reconciliation["Passed"]
    ).sum()
)


if metric_reconciliation_failures:
    raise AssertionError(
        "Published/recomputed metric "
        "reconciliation failed:\n"
        + reconciliation.loc[
            ~reconciliation["Passed"]
        ]
        .head(30)
        .to_string(index=False)
    )


# =============================================================================
# SELECT METHOD AND SCOPE
# =============================================================================

scope_winners = (
    method_comparison.loc[
        method_comparison[
            "IsMethodWinnerWithinScope"
        ]
    ]
    .sort_values(
        "NominalScopeCoveragePercentage"
    )
    .reset_index(drop=True)
)


if len(scope_winners) != len(SCOPE_FLAGS):
    raise AssertionError(
        "Expected exactly one method "
        "winner per scope"
    )


scope_winner_methods = set(
    scope_winners[
        "MethodID"
    ].astype(str)
)


if len(scope_winner_methods) != 1:
    raise AssertionError(
        "Different methods won different scopes: "
        f"{sorted(scope_winner_methods)}"
    )


final_method_id = str(
    next(iter(scope_winner_methods))
)

final_method_class = str(
    scope_winners.iloc[0][
        "MethodClass"
    ]
)


scope_tradeoff = scope_winners.loc[
    scope_winners[
        "ScopeID"
    ].isin(proceeding_scopes)
].copy()


if len(scope_tradeoff) != 3:
    raise AssertionError(
        "Expected three Part 11H "
        "scope candidates"
    )


best_scope_wape = float(
    scope_tradeoff[
        "WAPEPercentage"
    ].min()
)


scope_tradeoff[
    "WAPEIncreasePercentagePointsFromBest"
] = (
    scope_tradeoff[
        "WAPEPercentage"
    ]
    - best_scope_wape
)


scope_tradeoff[
    "RelativeWAPEIncreaseFromBestPercentage"
] = np.where(
    best_scope_wape != 0,
    (
        100.0
        * scope_tradeoff[
            "WAPEIncreasePercentagePointsFromBest"
        ]
        / best_scope_wape
    ),
    np.nan,
)


scope_tradeoff[
    "WithinScopeTolerance"
] = (
    scope_tradeoff[
        "RelativeWAPEIncreaseFromBestPercentage"
    ]
    <= (
        SCOPE_TOLERANCE_PERCENT
        + 1e-12
    )
)


eligible_scopes = scope_tradeoff.loc[
    scope_tradeoff[
        "WithinScopeTolerance"
    ]
].copy()


if eligible_scopes.empty:
    raise AssertionError(
        "No scope satisfied the "
        "predeclared tolerance rule"
    )


eligible_scopes = (
    eligible_scopes
    .sort_values(
        [
            "NominalScopeCoveragePercentage",
            "WAPEPercentage",
            "MAE",
            "RMSE",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


selected_scope_row = (
    eligible_scopes.iloc[0]
)

final_scope_id = str(
    selected_scope_row["ScopeID"]
)

final_scope_flag = (
    SCOPE_FLAGS[final_scope_id]
)


scope_tradeoff[
    "SelectedFinalScope"
] = (
    scope_tradeoff["ScopeID"]
    == final_scope_id
)


scope_tradeoff = (
    scope_tradeoff
    .sort_values(
        "NominalScopeCoveragePercentage"
    )
    .reset_index(drop=True)
)


selected_method_rows = method_comparison.loc[
    (
        method_comparison["ScopeID"]
        == final_scope_id
    )
    & (
        method_comparison[
            "MethodID"
        ].astype(str)
        == final_method_id
    )
]


if len(selected_method_rows) != 1:
    raise AssertionError(
        "Could not identify one final "
        "method/scope metric row"
    )


selected_method_row = (
    selected_method_rows.iloc[0]
)


selected_prediction_rows = all_predictions.loc[
    all_predictions[
        "MethodID"
    ].astype(str)
    == final_method_id
].copy()


if final_scope_flag is not None:
    selected_prediction_rows = (
        selected_prediction_rows.loc[
            selected_prediction_rows[
                final_scope_flag
            ]
        ]
        .copy()
    )


selected_prediction_rows[
    "SelectedTargetID"
] = FINAL_TARGET_REQUIRED

selected_prediction_rows[
    "SelectedScopeID"
] = final_scope_id

selected_prediction_rows[
    "SelectedMethodID"
] = final_method_id

selected_prediction_rows[
    "NormalDemandActual"
] = selected_prediction_rows["Actual"]

selected_prediction_rows[
    "NormalDemandForecast"
] = selected_prediction_rows["Prediction"]

selected_prediction_rows[
    "Error"
] = (
    selected_prediction_rows[
        "NormalDemandForecast"
    ]
    - selected_prediction_rows[
        "NormalDemandActual"
    ]
)

selected_prediction_rows[
    "AbsoluteError"
] = (
    selected_prediction_rows["Error"]
    .abs()
)

selected_prediction_rows[
    "BulkIncludedInModelledTarget"
] = False

selected_prediction_rows[
    "UnconfirmedBulkForecast"
] = 0.0

selected_prediction_rows[
    "ConfirmedBulkHandling"
] = "ADD_EXTERNALLY_WHEN_KNOWN"

selected_prediction_rows[
    "OperationalRequiredQuantityRule"
] = (
    "NormalDemandForecast + "
    "ConfirmedBulkQuantity"
)


selected_prediction_rows = (
    selected_prediction_rows
    .sort_values(
        [
            "WeekStartDate",
            "CanonicalProductID",
        ]
    )
    .reset_index(drop=True)
)


selected_fold_metrics = fold_metrics.loc[
    (
        fold_metrics["ScopeID"]
        == final_scope_id
    )
    & (
        fold_metrics["MethodID"].astype(str)
        == final_method_id
    )
].sort_values(
    "ForecastWeek"
).reset_index(
    drop=True
)


if len(selected_fold_metrics) != 39:
    raise AssertionError(
        "Expected 39 selected fold metrics"
    )


selected_stability_rows = fold_stability.loc[
    (
        fold_stability["ScopeID"]
        == final_scope_id
    )
    & (
        fold_stability[
            "MethodID"
        ].astype(str)
        == final_method_id
    )
]


if len(selected_stability_rows) != 1:
    raise AssertionError(
        "Could not identify selected "
        "fold-stability row"
    )


selected_stability = (
    selected_stability_rows.iloc[0]
)


best_ml_rows = method_comparison.loc[
    (
        method_comparison["ScopeID"]
        == final_scope_id
    )
    & (
        method_comparison["MethodClass"]
        == "MACHINE_LEARNING"
    )
].sort_values(
    [
        "WAPEPercentage",
        "MAE",
        "RMSE",
    ]
)


if best_ml_rows.empty:
    raise AssertionError(
        "Could not identify best ML comparator"
    )


best_ml_selected_scope = (
    best_ml_rows.iloc[0]
)


best_accuracy_scope_row = (
    scope_tradeoff
    .sort_values(
        [
            "WAPEPercentage",
            "MAE",
            "RMSE",
        ]
    )
    .iloc[0]
)


scope_wape_cost_points = float(
    selected_method_row["WAPEPercentage"]
    - best_accuracy_scope_row[
        "WAPEPercentage"
    ]
)


scope_wape_cost_relative = float(
    100.0
    * scope_wape_cost_points
    / best_accuracy_scope_row[
        "WAPEPercentage"
    ]
)


scope_actual_coverage_gain = float(
    selected_method_row[
        "EvaluationActualDemandCoveragePercentage"
    ]
    - best_accuracy_scope_row[
        "EvaluationActualDemandCoveragePercentage"
    ]
)


selection_evidence = pd.DataFrame(
    [
        {
            "EvidenceID":
                "FINAL_TARGET",

            "Decision":
                FINAL_TARGET_REQUIRED,

            "Evidence":
                (
                    "Part 11H found normal demand "
                    "better than direct total demand "
                    "in every matched comparison."
                ),

            "Value":
                part11h_decision[
                    "Evidence"
                ][
                    "NormalTargetBetterComparisons"
                ],

            "ReferenceValue":
                part11h_decision[
                    "Evidence"
                ][
                    "NormalVsTotalComparisonCount"
                ],
        },
        {
            "EvidenceID":
                "FINAL_METHOD",

            "Decision":
                final_method_id,

            "Evidence":
                (
                    "Lowest pooled pre-holdout WAPE "
                    "within every evaluated scope."
                ),

            "Value":
                float(
                    selected_method_row[
                        "WAPEPercentage"
                    ]
                ),

            "ReferenceValue":
                float(
                    best_ml_selected_scope[
                        "WAPEPercentage"
                    ]
                ),
        },
        {
            "EvidenceID":
                "METHOD_ADVANTAGE_OVER_BEST_ML",

            "Decision":
                final_method_id,

            "Evidence":
                (
                    "WAPE advantage over the best "
                    "ML candidate at the selected "
                    "scope, in percentage points."
                ),

            "Value":
                float(
                    best_ml_selected_scope[
                        "WAPEPercentage"
                    ]
                    - selected_method_row[
                        "WAPEPercentage"
                    ]
                ),

            "ReferenceValue":
                str(
                    best_ml_selected_scope[
                        "MethodID"
                    ]
                ),
        },
        {
            "EvidenceID":
                "FINAL_SCOPE",

            "Decision":
                final_scope_id,

            "Evidence":
                (
                    "Broadest Part 11H scope whose "
                    "WAPE remains within 5% of the "
                    "best surviving scope."
                ),

            "Value":
                float(
                    selected_method_row[
                        "EvaluationActualDemandCoveragePercentage"
                    ]
                ),

            "ReferenceValue":
                SCOPE_TOLERANCE_PERCENT,
        },
        {
            "EvidenceID":
                "SCOPE_ACCURACY_COST",

            "Decision":
                final_scope_id,

            "Evidence":
                (
                    "WAPE percentage-point increase "
                    "from the best-accuracy scope "
                    "to the selected scope."
                ),

            "Value":
                scope_wape_cost_points,

            "ReferenceValue":
                scope_wape_cost_relative,
        },
        {
            "EvidenceID":
                "SCOPE_COVERAGE_GAIN",

            "Decision":
                final_scope_id,

            "Evidence":
                (
                    "Evaluation actual-demand coverage "
                    "gain over the best-accuracy scope, "
                    "in percentage points."
                ),

            "Value":
                scope_actual_coverage_gain,

            "ReferenceValue":
                str(
                    best_accuracy_scope_row[
                        "ScopeID"
                    ]
                ),
        },
        {
            "EvidenceID":
                "BULK_POLICY",

            "Decision":
                EXPECTED_BULK_FALLBACK,

            "Evidence":
                (
                    "Bulk is excluded from the modelled "
                    "target; confirmed preorders are "
                    "added externally."
                ),

            "Value":
                part11h_decision[
                    "Evidence"
                ][
                    "BulkShareOfTotalPercentage"
                ],

            "ReferenceValue":
                part11h_decision[
                    "Evidence"
                ][
                    "BulkZeroProductWeekPercentage"
                ],
        },
    ]
)


# =============================================================================
# FINAL DESIGN CONTRACT
# =============================================================================

final_design = {
    "StepID":
        STEP_ID,

    "CreatedUTC":
        NOW_UTC.isoformat(),

    "CreatedLocal":
        NOW_LOCAL.isoformat(),

    "SelectionStatus":
        "FINAL_PREHOLDOUT_WEEKLY_DESIGN_LOCKED",

    "FinalTarget":
        {
            "TargetID":
                FINAL_TARGET_REQUIRED,

            "TargetColumn":
                FINAL_TARGET_COLUMN,

            "BulkIncludedInModelledTarget":
                False,

            "Reason":
                (
                    "Ordinary uncertain restaurant "
                    "demand is forecast separately "
                    "from bulk preorders that are "
                    "normally known operationally."
                ),
        },

    "FinalMethod":
        {
            "MethodID":
                final_method_id,

            "MethodClass":
                final_method_class,

            "OperationalDefinition":
                (
                    "Use the locked Part 11E "
                    "NAIVE_LAST_CONTEXT rule for "
                    "weekly normal demand: the most "
                    "recent available product-level "
                    "weekly normal-demand context is "
                    "used as the next weekly forecast."
                ),

            "ImplementationAuthority":
                str(PART11E_DESIGN_CONTRACT),

            "ImplementationAuthoritySHA256":
                baseline_contract_hash,

            "HyperparameterSearchRequired":
                False,

            "FittedModelArtifactRequired":
                False,
        },

    "FinalScope":
        {
            "ScopeID":
                final_scope_id,

            "NominalCumulativeNormalDemandCoveragePercentage":
                int(
                    selected_method_row[
                        "NominalScopeCoveragePercentage"
                    ]
                ),

            "ScopeMembershipField":
                final_scope_flag,

            "DynamicScopeRule":
                (
                    "Before each forecast week, rank "
                    "products using cumulative normal "
                    "demand from prior weeks only and "
                    "retain the strict cumulative-"
                    "coverage scope. Membership is "
                    "recalculated for every forecast "
                    "week and is not a permanent "
                    "product list."
                ),

            "OutsideScopeTreatment":
                (
                    "Keep outside-scope products in "
                    "the audit dataset and manual-"
                    "review workflow; exclude them "
                    "from the primary operational "
                    "accuracy claim."
                ),
        },

    "ScopeSelectionRule":
        {
            "CandidateScopes":
                proceeding_scopes,

            "PrimaryMetric":
                "WAPEPercentage",

            "ToleranceType":
                "RELATIVE_TO_BEST_SCOPE_WAPE",

            "TolerancePercentage":
                SCOPE_TOLERANCE_PERCENT,

            "DecisionRule":
                (
                    "Choose the broadest candidate "
                    "scope whose pooled WAPE is no "
                    "more than 5% above the best "
                    "candidate-scope WAPE."
                ),

            "BestAccuracyScope":
                str(
                    best_accuracy_scope_row[
                        "ScopeID"
                    ]
                ),

            "SelectedScope":
                final_scope_id,

            "SelectedScopeRelativeWAPEIncreaseFromBestPercentage":
                scope_wape_cost_relative,
        },

    "BulkPolicy":
        {
            "HistoricalBulkForecastedByModel":
                False,

            "UnconfirmedBulkFallback":
                EXPECTED_BULK_FALLBACK,

            "UnconfirmedBulkQuantity":
                0.0,

            "ConfirmedBulkOrders":
                "ADD_EXTERNALLY_WHEN_KNOWN",

            "OperationalQuantityFormula":
                (
                    "RequiredQuantity = "
                    "ForecastNormalDemand + "
                    "ConfirmedBulkQuantity"
                ),
        },

    "LockedPreholdoutPerformance":
        {
            "ForecastWeeks":
                int(
                    selected_method_row[
                        "ForecastWeeks"
                    ]
                ),

            "Observations":
                int(
                    selected_method_row[
                        "Observations"
                    ]
                ),

            "Products":
                int(
                    selected_method_row[
                        "Products"
                    ]
                ),

            "ActualNormalDemand":
                float(
                    selected_method_row[
                        "ActualTotal"
                    ]
                ),

            "PredictedNormalDemand":
                float(
                    selected_method_row[
                        "PredictedTotal"
                    ]
                ),

            "MAE":
                float(
                    selected_method_row[
                        "MAE"
                    ]
                ),

            "RMSE":
                float(
                    selected_method_row[
                        "RMSE"
                    ]
                ),

            "WAPEPercentage":
                float(
                    selected_method_row[
                        "WAPEPercentage"
                    ]
                ),

            "MeanBias":
                float(
                    selected_method_row[
                        "MeanBias"
                    ]
                ),

            "TotalBias":
                float(
                    selected_method_row[
                        "TotalBias"
                    ]
                ),

            "EvaluationActualDemandCoveragePercentage":
                float(
                    selected_method_row[
                        "EvaluationActualDemandCoveragePercentage"
                    ]
                ),
        },

    "FoldStability":
        {
            "MeanWeeklyWAPEPercentage":
                float(
                    selected_stability[
                        "MeanWeeklyWAPEPercentage"
                    ]
                ),

            "MedianWeeklyWAPEPercentage":
                float(
                    selected_stability[
                        "MedianWeeklyWAPEPercentage"
                    ]
                ),

            "StdWeeklyWAPEPercentage":
                float(
                    selected_stability[
                        "StdWeeklyWAPEPercentage"
                    ]
                ),

            "WeeksWonAgainstAllComparedMethods":
                int(
                    selected_stability[
                        "WeeksWon"
                    ]
                ),
        },

    "EvaluationGovernance":
        {
            "SelectionData":
                (
                    "Pre-holdout chronological "
                    "validation only"
                ),

            "OpenedMarch2026TargetsUsed":
                False,

            "ForecastsRoundedBeforeScoring":
                False,

            "FinalUnbiasedEvaluationRequirement":
                (
                    "Evaluate the locked design on "
                    "a new untouched future period."
                ),
        },

    "NextStep":
        "11J",
}


final_design_table = pd.DataFrame(
    [
        {
            "DesignField":
                "Forecast target",

            "LockedValue":
                FINAL_TARGET_REQUIRED,

            "Status":
                "LOCKED",
        },
        {
            "DesignField":
                "Forecast method",

            "LockedValue":
                final_method_id,

            "Status":
                "LOCKED",
        },
        {
            "DesignField":
                "Primary product scope",

            "LockedValue":
                final_scope_id,

            "Status":
                "LOCKED_DYNAMIC_SCOPE",
        },
        {
            "DesignField":
                "Scope tolerance rule",

            "LockedValue":
                (
                    f"Broadest scope within "
                    f"{SCOPE_TOLERANCE_PERCENT:.1f}% "
                    "relative WAPE of best scope"
                ),

            "Status":
                "LOCKED",
        },
        {
            "DesignField":
                "Bulk in model target",

            "LockedValue":
                "EXCLUDED",

            "Status":
                "LOCKED",
        },
        {
            "DesignField":
                "Unknown future bulk",

            "LockedValue":
                EXPECTED_BULK_FALLBACK,

            "Status":
                "LOCKED",
        },
        {
            "DesignField":
                "Confirmed future bulk",

            "LockedValue":
                "ADD_EXTERNALLY",

            "Status":
                "OPERATIONAL_RULE",
        },
        {
            "DesignField":
                "Operational quantity formula",

            "LockedValue":
                (
                    "Normal-demand forecast + "
                    "confirmed bulk quantity"
                ),

            "Status":
                "LOCKED",
        },
    ]
)


# =============================================================================
# VALIDATION
# =============================================================================

expected_method_count = int(
    baseline_target["MethodID"].nunique()
    + model_target["MethodID"].nunique()
)

expected_method_metric_rows = (
    expected_method_count
    * len(SCOPE_FLAGS)
)

expected_fold_metric_rows = (
    expected_method_count
    * len(SCOPE_FLAGS)
    * len(forecast_weeks)
)


selected_recomputed = metric_record(
    selected_prediction_rows[
        "NormalDemandActual"
    ],
    selected_prediction_rows[
        "NormalDemandForecast"
    ],
)


selected_metric_difference = max(
    abs(
        float(
            selected_recomputed[metric]
        )
        - float(
            selected_method_row[metric]
        )
    )
    for metric in [
        "ActualTotal",
        "PredictedTotal",
        "MAE",
        "RMSE",
        "WAPEPercentage",
        "MeanBias",
        "TotalBias",
    ]
)


validation = pd.DataFrame(
    [
        {
            "Check":
                (
                    "Part 11H lock and "
                    "checkpoint verified"
                ),

            "Expected":
                True,

            "Actual":
                True,

            "Passed":
                True,
        },
        {
            "Check":
                (
                    "Part 11H references current "
                    "Part 11G lock"
                ),

            "Expected":
                part11g_lock_hash,

            "Actual":
                (
                    part11h_lock
                    .get("Part11G", {})
                    .get("LockSHA256")
                ),

            "Passed":
                (
                    part11h_lock
                    .get("Part11G", {})
                    .get("LockSHA256")
                    == part11g_lock_hash
                ),
        },
        {
            "Check":
                (
                    "Part 11G references current "
                    "Part 11E lock"
                ),

            "Expected":
                part11e_lock_hash,

            "Actual":
                (
                    part11g_lock
                    .get("Part11E", {})
                    .get("LockSHA256")
                ),

            "Passed":
                (
                    part11g_lock
                    .get("Part11E", {})
                    .get("LockSHA256")
                    == part11e_lock_hash
                ),
        },
        {
            "Check":
                "Final target from Part 11H",

            "Expected":
                FINAL_TARGET_REQUIRED,

            "Actual":
                part11h_decision.get(
                    "PrimaryModelledTargetForPart11I"
                ),

            "Passed":
                (
                    part11h_decision.get(
                        "PrimaryModelledTargetForPart11I"
                    )
                    == FINAL_TARGET_REQUIRED
                ),
        },
        {
            "Check":
                "Unconfirmed bulk fallback",

            "Expected":
                EXPECTED_BULK_FALLBACK,

            "Actual":
                part11h_decision.get(
                    "SelectedUnconfirmedBulkFallback"
                ),

            "Passed":
                (
                    part11h_decision.get(
                        "SelectedUnconfirmedBulkFallback"
                    )
                    == EXPECTED_BULK_FALLBACK
                ),
        },
        {
            "Check":
                "Chronological forecast weeks",

            "Expected":
                39,

            "Actual":
                len(forecast_weeks),

            "Passed":
                len(forecast_weeks) == 39,
        },
        {
            "Check":
                (
                    "Actual and scope mismatches "
                    "between methods"
                ),

            "Expected":
                0,

            "Actual":
                actual_scope_mismatches,

            "Passed":
                actual_scope_mismatches == 0,
        },
        {
            "Check":
                "Method comparison rows",

            "Expected":
                expected_method_metric_rows,

            "Actual":
                len(method_comparison),

            "Passed":
                (
                    len(method_comparison)
                    == expected_method_metric_rows
                ),
        },
        {
            "Check":
                "Fold metric rows",

            "Expected":
                expected_fold_metric_rows,

            "Actual":
                len(fold_metrics),

            "Passed":
                (
                    len(fold_metrics)
                    == expected_fold_metric_rows
                ),
        },
        {
            "Check":
                (
                    "Published metric "
                    "reconciliation failures"
                ),

            "Expected":
                0,

            "Actual":
                metric_reconciliation_failures,

            "Passed":
                (
                    metric_reconciliation_failures
                    == 0
                ),
        },
        {
            "Check":
                (
                    "Exactly one method winner "
                    "per scope"
                ),

            "Expected":
                len(SCOPE_FLAGS),

            "Actual":
                len(scope_winners),

            "Passed":
                (
                    len(scope_winners)
                    == len(SCOPE_FLAGS)
                ),
        },
        {
            "Check":
                (
                    "One method won all "
                    "evaluated scopes"
                ),

            "Expected":
                1,

            "Actual":
                len(scope_winner_methods),

            "Passed":
                (
                    len(scope_winner_methods)
                    == 1
                ),
        },
        {
            "Check":
                (
                    "Final scope satisfies 5% "
                    "relative WAPE tolerance"
                ),

            "Expected":
                True,

            "Actual":
                bool(
                    selected_scope_row[
                        "WithinScopeTolerance"
                    ]
                ),

            "Passed":
                bool(
                    selected_scope_row[
                        "WithinScopeTolerance"
                    ]
                ),
        },
        {
            "Check":
                (
                    "Final scope is broadest "
                    "eligible scope"
                ),

            "Expected":
                int(
                    eligible_scopes[
                        "NominalScopeCoveragePercentage"
                    ].max()
                ),

            "Actual":
                int(
                    selected_scope_row[
                        "NominalScopeCoveragePercentage"
                    ]
                ),

            "Passed":
                (
                    int(
                        selected_scope_row[
                            "NominalScopeCoveragePercentage"
                        ]
                    )
                    == int(
                        eligible_scopes[
                            "NominalScopeCoveragePercentage"
                        ].max()
                    )
                ),
        },
        {
            "Check":
                "Selected prediction rows",

            "Expected":
                int(
                    selected_method_row[
                        "Observations"
                    ]
                ),

            "Actual":
                len(selected_prediction_rows),

            "Passed":
                (
                    len(selected_prediction_rows)
                    == int(
                        selected_method_row[
                            "Observations"
                        ]
                    )
                ),
        },
        {
            "Check":
                (
                    "Selected prediction metric "
                    "maximum difference"
                ),

            "Expected":
                "<= 1e-9",

            "Actual":
                selected_metric_difference,

            "Passed":
                (
                    selected_metric_difference
                    <= 1e-9
                ),
        },
        {
            "Check":
                "Selected fold metric rows",

            "Expected":
                39,

            "Actual":
                len(selected_fold_metrics),

            "Passed":
                (
                    len(selected_fold_metrics)
                    == 39
                ),
        },
        {
            "Check":
                (
                    "Bulk included in modelled "
                    "forecast target"
                ),

            "Expected":
                False,

            "Actual":
                False,

            "Passed":
                True,
        },
        {
            "Check":
                (
                    "Opened March 2026 "
                    "targets read"
                ),

            "Expected":
                False,

            "Actual":
                False,

            "Passed":
                True,
        },
        {
            "Check":
                (
                    "Models fitted or refitted "
                    "in Part 11I"
                ),

            "Expected":
                False,

            "Actual":
                False,

            "Passed":
                True,
        },
        {
            "Check":
                (
                    "Final weekly method and "
                    "scope locked"
                ),

            "Expected":
                True,

            "Actual":
                True,

            "Passed":
                True,
        },
    ]
)


if not validation["Passed"].all():
    raise AssertionError(
        "Part 11I validation failed:\n"
        + validation.loc[
            ~validation["Passed"]
        ].to_string(index=False)
    )


# =============================================================================
# STAGE OUTPUTS, MEMORY, CHECKPOINT AND LOCK
# =============================================================================

FINAL_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

stage_root = (
    EXT_ROOT
    / f".11I_staging_{uuid.uuid4().hex}"
)

stage_root.mkdir(
    parents=True,
    exist_ok=False,
)


try:
    for path, frame in [
        (
            METHOD_COMPARISON_PATH,
            method_comparison,
        ),
        (
            METHOD_RECONCILIATION_PATH,
            reconciliation,
        ),
        (
            FOLD_METRICS_PATH,
            fold_metrics,
        ),
        (
            FOLD_STABILITY_PATH,
            fold_stability,
        ),
        (
            SCOPE_WINNERS_PATH,
            scope_winners,
        ),
        (
            SCOPE_TRADEOFF_PATH,
            scope_tradeoff,
        ),
        (
            SELECTED_PREDICTIONS_PATH,
            selected_prediction_rows,
        ),
        (
            SELECTED_FOLD_METRICS_PATH,
            selected_fold_metrics,
        ),
        (
            SELECTION_EVIDENCE_PATH,
            selection_evidence,
        ),
        (
            FINAL_DESIGN_TABLE_PATH,
            final_design_table,
        ),
        (
            VALIDATION_PATH,
            validation,
        ),
    ]:
        stage_csv(
            stage_root,
            path,
            frame,
        )

    stage_json(
        stage_root,
        FINAL_DESIGN_PATH,
        final_design,
    )


    decisions_text = "\n".join(
        [
            (
                f"- Decision date: "
                f"{NOW_LOCAL.date().isoformat()}."
            ),
            (
                "- The final modelled target is "
                "WeeklyNormalDemand; historical bulk "
                "is excluded from the forecast target."
            ),
            (
                "- The final weekly forecasting "
                f"method is {final_method_id}."
            ),
            (
                "- The final primary scope is "
                f"{final_scope_id}."
            ),
            (
                "- Scope membership is dynamic and "
                "recalculated from prior normal demand "
                "before every forecast week."
            ),
            (
                "- The scope rule selects the broadest "
                "scope within 5% relative WAPE of the "
                "best surviving scope."
            ),
            (
                "- Unknown future bulk defaults to zero; "
                "confirmed preorders are added externally."
            ),
            (
                "- Products outside the selected scope "
                "remain in audit/manual review and "
                "outside the primary accuracy claim."
            ),
            (
                "- No March 2026 target was read and "
                "no model was fitted or refitted."
            ),
        ]
    )


    files_text = "\n".join(
        [
            (
                "- Method comparison: "
                f"`{METHOD_COMPARISON_PATH}`"
            ),
            (
                "- Fold stability: "
                f"`{FOLD_STABILITY_PATH}`"
            ),
            (
                "- Scope trade-off: "
                f"`{SCOPE_TRADEOFF_PATH}`"
            ),
            (
                "- Selected pre-holdout predictions: "
                f"`{SELECTED_PREDICTIONS_PATH}`"
            ),
            (
                "- Final selection evidence: "
                f"`{SELECTION_EVIDENCE_PATH}`"
            ),
            (
                "- Final weekly forecasting design: "
                f"`{FINAL_DESIGN_PATH}`"
            ),
            (
                "- Validation: "
                f"`{VALIDATION_PATH}`"
            ),
        ]
    )


    results_text = "\n".join(
        [
            (
                f"- Final target: "
                f"{FINAL_TARGET_REQUIRED}."
            ),
            (
                f"- Final method: "
                f"{final_method_id}."
            ),
            (
                f"- Final scope: "
                f"{final_scope_id}."
            ),
            (
                f"- Forecast weeks: "
                f"{int(selected_method_row['ForecastWeeks'])}."
            ),
            (
                f"- Selected observations: "
                f"{int(selected_method_row['Observations']):,}."
            ),
            (
                "- Evaluation actual-demand coverage: "
                f"{float(selected_method_row['EvaluationActualDemandCoveragePercentage']):.6f}%."
            ),
            (
                f"- WAPE: "
                f"{float(selected_method_row['WAPEPercentage']):.6f}%."
            ),
            (
                f"- MAE: "
                f"{float(selected_method_row['MAE']):.6f}."
            ),
            (
                f"- RMSE: "
                f"{float(selected_method_row['RMSE']):.6f}."
            ),
            (
                f"- Total bias: "
                f"{float(selected_method_row['TotalBias']):.6f}."
            ),
            (
                "- WAPE advantage over best ML at "
                "selected scope: "
                f"{float(best_ml_selected_scope['WAPEPercentage'] - selected_method_row['WAPEPercentage']):.6f} "
                "percentage points."
            ),
            (
                "- Relative WAPE increase from "
                "best-accuracy scope: "
                f"{scope_wape_cost_relative:.6f}%."
            ),
            (
                "- Evaluation actual-demand coverage "
                "gain over best-accuracy scope: "
                f"{scope_actual_coverage_gain:.6f} "
                "percentage points."
            ),
        ]
    )


    for key, final_path in MEMORY_FILES.items():
        original = final_path.read_text(
            encoding="utf-8"
        )

        if key == "PROJECT_CONTEXT":
            updated = update_section(
                original,
                "STEP_11I",
                (
                    "Part 11I final weekly "
                    "forecasting design"
                ),
                (
                    "The final pre-holdout design was "
                    f"locked as {final_method_id} for "
                    f"{FINAL_TARGET_REQUIRED} within "
                    f"the dynamic {final_scope_id} "
                    "scope. Bulk preorders are excluded "
                    "from the modelled target and added "
                    "externally when confirmed."
                ),
            )

        elif key == "WORKFLOW":
            updated = update_section(
                original,
                "STEP_11I",
                "Part 11I workflow status",
                (
                    "Part 11I is complete. The weekly "
                    "target, method, dynamic scope and "
                    "bulk policy are locked. Part 11J "
                    "will assess ingredient-planning "
                    "readiness without changing this "
                    "forecasting design."
                ),
            )

        elif key == "DECISIONS":
            updated = update_section(
                original,
                "STEP_11I",
                "Part 11I decisions",
                decisions_text,
            )

        elif key == "FILES_AND_PATHS":
            updated = update_section(
                original,
                "STEP_11I",
                "Part 11I files",
                files_text,
            )

        elif key == "METRICS_AND_RESULTS":
            updated = update_section(
                original,
                "STEP_11I",
                (
                    "Part 11I final selection "
                    "metrics and results"
                ),
                results_text,
            )

        elif key == "CHAT_INDEX":
            row = (
                f"| {NOW_LOCAL.isoformat()} | "
                "11I | Final weekly method and "
                f"scope selection | {STATUS} |"
            )

            updated = (
                original
                if row in original
                else (
                    original.rstrip()
                    + "\n"
                    + row
                    + "\n"
                )
            )

        elif key == "CURRENT_HANDOFF":
            updated = "\n".join(
                [
                    "# Current Handoff",
                    "",
                    (
                        "- Current completed step: "
                        f"{STEP_ID}"
                    ),
                    (
                        f"- Status: {STATUS}"
                    ),
                    (
                        "- Updated local time: "
                        f"{NOW_LOCAL.isoformat()}"
                    ),
                    (
                        "- Final target: "
                        f"{FINAL_TARGET_REQUIRED}"
                    ),
                    (
                        "- Final method: "
                        f"{final_method_id}"
                    ),
                    (
                        "- Final scope: "
                        f"{final_scope_id}"
                    ),
                    (
                        "- Bulk policy: exclude bulk "
                        "from the model target; unknown "
                        "bulk is zero; confirmed bulk is "
                        "added externally."
                    ),
                    (
                        "- Final design contract: "
                        f"{FINAL_DESIGN_PATH}"
                    ),
                    (
                        "- March 2026 targets were "
                        "not read."
                    ),
                    (
                        "- Next step: 11J ingredient-"
                        "planning readiness assessment."
                    ),
                    "",
                ]
            )

        else:
            raise KeyError(key)

        stage_text(
            stage_root,
            final_path,
            updated,
        )


    step_text = f"""# Step 11I — Final Weekly Method and Scope Selection

- **Step ID:** 11I
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## Purpose
Select and lock the final weekly forecasting target, method, dynamic product scope and bulk-demand treatment using pre-holdout chronological evidence only.

## Technical actions
- Verified the Part 11E, Part 11G and Part 11H lock chain.
- Recomputed all normal-demand metrics directly from the locked Part 11E and Part 11G prediction files.
- Reconciled the recomputed metrics against all published Part 11E and Part 11G ranking rows.
- Ranked every baseline and machine-learning method within all-products, 80%, 90% and 95% scopes.
- Recomputed weekly fold metrics and stability across the same 39 chronological forecast weeks.
- Applied the scope rule: choose the broadest Part 11H candidate scope whose pooled WAPE remains within {SCOPE_TOLERANCE_PERCENT:.1f}% of the best candidate-scope WAPE.
- Locked the final normal-demand design without reading March 2026 targets or fitting a model.

## Final decisions
{decisions_text}

## Final performance
{results_text}

## Outputs
{files_text}

## Evaluation governance
- Part 11I performance is pre-holdout selection evidence, not an unbiased final test.
- March 2026 was not used because its daily targets had previously been opened.
- A new untouched future period is required for final unbiased evaluation.
- Forecast values were not rounded before metric calculation.

## Next action
Part 11J: assess whether the locked product-demand forecasts are ready to support ingredient planning and identify the recipe and ingredient data still required.
"""

    stage_text(
        stage_root,
        STEP_MEMORY_PATH,
        step_text,
    )


    checkpoint = {
        "StepID":
            STEP_ID,

        "Status":
            STATUS,

        "CreatedUTC":
            NOW_UTC.isoformat(),

        "CreatedLocal":
            NOW_LOCAL.isoformat(),

        "Input":
            {
                "Part11ELockSHA256":
                    part11e_lock_hash,

                "Part11GLockSHA256":
                    part11g_lock_hash,

                "Part11HLockSHA256":
                    part11h_lock_hash,

                "Part11HCheckpointSHA256":
                    part11h_checkpoint_hash,

                "Part11EBaselinePredictionsSHA256":
                    baseline_predictions_hash,

                "Part11EBaselineRankingSHA256":
                    baseline_ranking_hash,

                "Part11EBaselineDesignContractSHA256":
                    baseline_contract_hash,

                "Part11GCandidatePredictionsSHA256":
                    model_predictions_hash,

                "Part11GCandidateRankingSHA256":
                    model_ranking_hash,

                "Part11HTargetAndBulkDecisionSHA256":
                    sha256_file(
                        PART11H_DECISION
                    ),
            },

        "FinalSelection":
            {
                "TargetID":
                    FINAL_TARGET_REQUIRED,

                "MethodID":
                    final_method_id,

                "MethodClass":
                    final_method_class,

                "ScopeID":
                    final_scope_id,

                "ScopeTolerancePercentage":
                    SCOPE_TOLERANCE_PERCENT,

                "BulkIncludedInModelledTarget":
                    False,

                "UnconfirmedBulkFallback":
                    EXPECTED_BULK_FALLBACK,

                "ConfirmedBulkHandling":
                    "ADD_EXTERNALLY_WHEN_KNOWN",

                "Metrics":
                    final_design[
                        "LockedPreholdoutPerformance"
                    ],
            },

        "Safety":
            {
                "ProtectedTargetVaultOpened":
                    False,

                "ProtectedTargetVaultCopied":
                    False,

                "OpenedMarch2026TargetsRead":
                    False,

                "BulkIncludedInModelledTarget":
                    False,

                "ModelsFittedOrRefitted":
                    False,

                "FinalMethodSelected":
                    True,

                "FinalScopeSelected":
                    True,

                "FinalForecastDesignLocked":
                    True,

                "ForecastsRoundedBeforeScoring":
                    False,
            },

        "ReadyForPart11J":
            True,

        "NextStep":
            "11J",
    }


    stage_json(
        stage_root,
        CHECKPOINT_PATH,
        checkpoint,
    )


    checkpoint_hash = sha256_file(
        stage_path(
            stage_root,
            CHECKPOINT_PATH,
        )
    )


    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        (
            f"{checkpoint_hash}  "
            f"{CHECKPOINT_PATH.name}\n"
        ),
    )


    manifest_targets = [
        METHOD_COMPARISON_PATH,
        METHOD_RECONCILIATION_PATH,
        FOLD_METRICS_PATH,
        FOLD_STABILITY_PATH,
        SCOPE_WINNERS_PATH,
        SCOPE_TRADEOFF_PATH,
        SELECTED_PREDICTIONS_PATH,
        SELECTED_FOLD_METRICS_PATH,
        SELECTION_EVIDENCE_PATH,
        FINAL_DESIGN_PATH,
        FINAL_DESIGN_TABLE_PATH,
        VALIDATION_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]


    manifest = pd.DataFrame(
        [
            {
                "RelativePath":
                    str(
                        path.relative_to(
                            EXT_ROOT
                        )
                    ),

                "Bytes":
                    stage_path(
                        stage_root,
                        path,
                    ).stat().st_size,

                "SHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            path,
                        )
                    ),
            }
            for path in manifest_targets
        ]
    ).sort_values(
        "RelativePath"
    )


    stage_csv(
        stage_root,
        MANIFEST_PATH,
        manifest,
    )


    manifest_hash = sha256_file(
        stage_path(
            stage_root,
            MANIFEST_PATH,
        )
    )


    lock = {
        "StepID":
            STEP_ID,

        "Status":
            STATUS,

        "CreatedUTC":
            NOW_UTC.isoformat(),

        "LockChain":
            {
                "Part11ELockSHA256":
                    part11e_lock_hash,

                "Part11GLockSHA256":
                    part11g_lock_hash,

                "Part11HLockSHA256":
                    part11h_lock_hash,

                "Part11HCheckpointSHA256":
                    part11h_checkpoint_hash,
            },

        "FinalForecastingDesign":
            {
                "Path":
                    str(FINAL_DESIGN_PATH),

                "SHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            FINAL_DESIGN_PATH,
                        )
                    ),

                "TargetID":
                    FINAL_TARGET_REQUIRED,

                "MethodID":
                    final_method_id,

                "ScopeID":
                    final_scope_id,

                "BulkIncludedInModelledTarget":
                    False,

                "UnconfirmedBulkFallback":
                    EXPECTED_BULK_FALLBACK,

                "ConfirmedBulkHandling":
                    "ADD_EXTERNALLY_WHEN_KNOWN",
            },

        "SelectionEvidence":
            {
                "MethodComparisonPath":
                    str(METHOD_COMPARISON_PATH),

                "MethodComparisonSHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            METHOD_COMPARISON_PATH,
                        )
                    ),

                "ScopeTradeoffPath":
                    str(SCOPE_TRADEOFF_PATH),

                "ScopeTradeoffSHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            SCOPE_TRADEOFF_PATH,
                        )
                    ),

                "SelectedPredictionsPath":
                    str(SELECTED_PREDICTIONS_PATH),

                "SelectedPredictionsSHA256":
                    sha256_file(
                        stage_path(
                            stage_root,
                            SELECTED_PREDICTIONS_PATH,
                        )
                    ),
            },

        "OutputHashManifest":
            {
                "Path":
                    str(MANIFEST_PATH),

                "SHA256":
                    manifest_hash,

                "FilesListed":
                    len(manifest),
            },

        "Checkpoint":
            {
                "Path":
                    str(CHECKPOINT_PATH),

                "SHA256":
                    checkpoint_hash,
            },

        "SafetyAssertions":
            checkpoint["Safety"],

        "ReadyForPart11J":
            True,

        "NextStep":
            "11J",
    }


    stage_json(
        stage_root,
        LOCK_PATH,
        lock,
    )


    lock_hash = sha256_file(
        stage_path(
            stage_root,
            LOCK_PATH,
        )
    )


    stage_text(
        stage_root,
        LOCK_SHA_PATH,
        (
            f"{lock_hash}  "
            f"{LOCK_PATH.name}\n"
        ),
    )


    stage_text(
        stage_root,
        LOG_PATH,
        "\n".join(
            [
                f"Status: {STATUS}",
                (
                    f"Run local: "
                    f"{NOW_LOCAL.isoformat()}"
                ),
                (
                    f"Final target: "
                    f"{FINAL_TARGET_REQUIRED}"
                ),
                (
                    f"Final method: "
                    f"{final_method_id}"
                ),
                (
                    f"Final scope: "
                    f"{final_scope_id}"
                ),
                (
                    "Scope tolerance percentage: "
                    f"{SCOPE_TOLERANCE_PERCENT}"
                ),
                (
                    "Bulk included in model target: "
                    "False"
                ),
                (
                    "Unconfirmed bulk fallback: "
                    f"{EXPECTED_BULK_FALLBACK}"
                ),
                (
                    "Confirmed bulk handling: "
                    "ADD_EXTERNALLY_WHEN_KNOWN"
                ),
                (
                    "Opened March 2026 targets read: "
                    "False"
                ),
                (
                    "Models fitted/refitted: False"
                ),
                (
                    "Checkpoint SHA256: "
                    f"{checkpoint_hash}"
                ),
                (
                    "Part 11I lock SHA256: "
                    f"{lock_hash}"
                ),
                "",
            ]
        ),
    )


    staged_files = [
        path
        for path in stage_root.rglob("*")
        if path.is_file()
    ]


    if not staged_files:
        raise AssertionError(
            "Part 11I staging directory is empty"
        )


    if any(
        (
            "target_vault"
            in path.name.lower()
        )
        or (
            path.suffix.lower()
            == ".joblib"
        )
        for path in staged_files
    ):
        raise PermissionError(
            "Forbidden target-vault or "
            "persisted-model file detected "
            "in staging"
        )


    for staged in sorted(staged_files):
        final = (
            EXT_ROOT
            / staged.relative_to(stage_root)
        )

        final.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if (
            final.exists()
            and final
            not in MEMORY_FILES.values()
        ):
            raise FileExistsError(
                "Refusing to overwrite "
                f"Part 11I output: {final}"
            )

        os.replace(
            staged,
            final,
        )


    for path in NEW_OUTPUTS:
        make_read_only(path)


finally:
    if stage_root.exists():
        shutil.rmtree(stage_root)


# =============================================================================
# FINAL TECHNICAL OUTPUT
# =============================================================================

print("=" * 100)

print(
    "EDEN WEEKLY FORECASTING EXTENSION — "
    "PART 11I COMPLETE"
)

print("=" * 100)

print(f"Status: {STATUS}")

print(
    f"Extension root: {EXT_ROOT}"
)

print(
    f"Local time: {NOW_LOCAL.isoformat()}"
)


print(
    "\nFINAL WEEKLY FORECASTING DESIGN"
)

print(
    f"Target: {FINAL_TARGET_REQUIRED}"
)

print(
    f"Method: {final_method_id}"
)

print(
    f"Method class: {final_method_class}"
)

print(
    f"Scope: {final_scope_id}"
)

print(
    "Scope membership: dynamic and "
    "recalculated from prior normal demand "
    "before every forecast week"
)

print(
    "Bulk included in modelled target: False"
)

print(
    "Unknown future bulk forecast: 0"
)

print(
    "Confirmed bulk orders: add externally"
)

print(
    "Operational quantity formula: "
    "Normal-demand forecast + "
    "confirmed bulk quantity"
)


print(
    "\nMETHOD WINNERS BY SCOPE"
)

print(
    scope_winners[
        [
            "ScopeID",
            "NominalScopeCoveragePercentage",
            "MethodID",
            "MethodClass",
            "Observations",
            "ActualTotal",
            "PredictedTotal",
            "EvaluationActualDemandCoveragePercentage",
            "MAE",
            "RMSE",
            "WAPEPercentage",
            "MeanBias",
            "TotalBias",
        ]
    ].to_string(index=False)
)


print(
    "\nSCOPE TRADE-OFF AND FINAL SELECTION"
)

print(
    scope_tradeoff[
        [
            "ScopeID",
            "NominalScopeCoveragePercentage",
            "MethodID",
            "Observations",
            "EvaluationActualDemandCoveragePercentage",
            "WAPEPercentage",
            "WAPEIncreasePercentagePointsFromBest",
            "RelativeWAPEIncreaseFromBestPercentage",
            "WithinScopeTolerance",
            "SelectedFinalScope",
        ]
    ].to_string(index=False)
)


print(
    "\nFINAL SELECTED PRE-HOLDOUT PERFORMANCE"
)

print(
    pd.DataFrame(
        [
            {
                "TargetID":
                    FINAL_TARGET_REQUIRED,

                "MethodID":
                    final_method_id,

                "ScopeID":
                    final_scope_id,

                "ForecastWeeks":
                    int(
                        selected_method_row[
                            "ForecastWeeks"
                        ]
                    ),

                "Observations":
                    int(
                        selected_method_row[
                            "Observations"
                        ]
                    ),

                "Products":
                    int(
                        selected_method_row[
                            "Products"
                        ]
                    ),

                "ActualTotal":
                    float(
                        selected_method_row[
                            "ActualTotal"
                        ]
                    ),

                "PredictedTotal":
                    float(
                        selected_method_row[
                            "PredictedTotal"
                        ]
                    ),

                "MAE":
                    float(
                        selected_method_row[
                            "MAE"
                        ]
                    ),

                "RMSE":
                    float(
                        selected_method_row[
                            "RMSE"
                        ]
                    ),

                "WAPEPercentage":
                    float(
                        selected_method_row[
                            "WAPEPercentage"
                        ]
                    ),

                "MeanBias":
                    float(
                        selected_method_row[
                            "MeanBias"
                        ]
                    ),

                "TotalBias":
                    float(
                        selected_method_row[
                            "TotalBias"
                        ]
                    ),

                "EvaluationActualDemandCoveragePercentage":
                    float(
                        selected_method_row[
                            "EvaluationActualDemandCoveragePercentage"
                        ]
                    ),
            }
        ]
    ).to_string(index=False)
)


print(
    "\nFINAL METHOD VERSUS BEST ML "
    "AT SELECTED SCOPE"
)

print(
    pd.DataFrame(
        [
            {
                "MethodRole":
                    "FINAL_SELECTED_METHOD",

                "MethodID":
                    final_method_id,

                "WAPEPercentage":
                    float(
                        selected_method_row[
                            "WAPEPercentage"
                        ]
                    ),

                "MAE":
                    float(
                        selected_method_row[
                            "MAE"
                        ]
                    ),

                "RMSE":
                    float(
                        selected_method_row[
                            "RMSE"
                        ]
                    ),

                "TotalBias":
                    float(
                        selected_method_row[
                            "TotalBias"
                        ]
                    ),
            },
            {
                "MethodRole":
                    "BEST_MACHINE_LEARNING_COMPARATOR",

                "MethodID":
                    str(
                        best_ml_selected_scope[
                            "MethodID"
                        ]
                    ),

                "WAPEPercentage":
                    float(
                        best_ml_selected_scope[
                            "WAPEPercentage"
                        ]
                    ),

                "MAE":
                    float(
                        best_ml_selected_scope[
                            "MAE"
                        ]
                    ),

                "RMSE":
                    float(
                        best_ml_selected_scope[
                            "RMSE"
                        ]
                    ),

                "TotalBias":
                    float(
                        best_ml_selected_scope[
                            "TotalBias"
                        ]
                    ),
            },
        ]
    ).to_string(index=False)
)


print(
    "\nSELECTED METHOD FOLD STABILITY"
)

print(
    pd.DataFrame(
        [
            selected_stability.to_dict()
        ]
    ).to_string(index=False)
)


print(
    "\nPART 11I VALIDATION"
)

print(
    validation.to_string(index=False)
)


print(
    "\nCONTROL OUTPUTS"
)

print(
    "- Method comparison: "
    f"{METHOD_COMPARISON_PATH}"
)

print(
    "- Scope trade-off: "
    f"{SCOPE_TRADEOFF_PATH}"
)

print(
    "- Selected predictions: "
    f"{SELECTED_PREDICTIONS_PATH}"
)

print(
    "- Final selection evidence: "
    f"{SELECTION_EVIDENCE_PATH}"
)

print(
    "- Final weekly forecasting design: "
    f"{FINAL_DESIGN_PATH}"
)

print(
    f"- Checkpoint: {CHECKPOINT_PATH}"
)

print(
    "- Checkpoint SHA256: "
    f"{checkpoint_hash}"
)

print(
    f"- Part 11I lock: {LOCK_PATH}"
)

print(
    "- Part 11I lock SHA256: "
    f"{lock_hash}"
)


print(
    "\nSAFETY: protected target vault "
    "opened/copied False/False; opened "
    "March 2026 targets read False; bulk "
    "included in modelled target False; "
    "models fitted/refitted False; final "
    "method selected True; final scope "
    "selected True; final forecasting design "
    "locked True; forecasts rounded before "
    "scoring False."
)

print("=" * 100)

EDEN WEEKLY FORECASTING EXTENSION — PART 11I COMPLETE
Status: PART_11I_COMPLETED_READY_FOR_11J
Extension root: /Users/ryansmac/Desktop/Meng Project/eden_datasets/weekly_forecasting_extension
Local time: 2026-08-06T19:08:11.212225+01:00

FINAL WEEKLY FORECASTING DESIGN
Target: WEEKLY_NORMAL_DEMAND
Method: NAIVE_LAST_CONTEXT
Method class: BASELINE
Scope: FOLD_STRICT_95_PERCENT
Scope membership: dynamic and recalculated from prior normal demand before every forecast week
Bulk included in modelled target: False
Unknown future bulk forecast: 0
Confirmed bulk orders: add externally
Operational quantity formula: Normal-demand forecast + confirmed bulk quantity

METHOD WINNERS BY SCOPE
               ScopeID  NominalScopeCoveragePercentage           MethodID MethodClass  Observations  ActualTotal  PredictedTotal  EvaluationActualDemandCoveragePercentage       MAE      RMSE  WAPEPercentage  MeanBias  TotalBias
FOLD_STRICT_80_PERCENT                              80 NAIVE_LAST_CONTEXT    BASELINE

In [27]:
from __future__ import annotations

import hashlib
import importlib
import importlib.metadata
import inspect
import json
import os
import shutil
import stat
import sys
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11IA1
# Expanded direct-weekly model challenge
#
# Purpose
# -------
# Challenge the locked Part 11I weekly benchmark with stronger nonlinear,
# count-aware and categorical-aware models under the same leakage-safe,
# chronological pre-holdout evaluation design.
#
# This step DOES NOT replace or modify the Part 11I lock. It creates a separate
# challenger audit. Final replacement, if justified, must occur in a later
# controlled amendment after tuning and daily-to-weekly comparison.
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"

FEATURE_ROOT = EXT_ROOT / "04_weekly_features"
BASELINE_ROOT = EXT_ROOT / "05_weekly_baselines"
MODEL_ROOT = EXT_ROOT / "06_weekly_models"
VALIDATION_ROOT = EXT_ROOT / "07_weekly_validation"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
LOG_ROOT = EXT_ROOT / "12_logs"

PART11F_FEATURES = FEATURE_ROOT / "11F_model_selection_weekly_features.csv"
PART11E_PREDICTIONS = BASELINE_ROOT / "11E_preholdout_weekly_baseline_predictions.csv"
PART11I_DESIGN = MODEL_ROOT / "final" / "11I_final_weekly_forecasting_design.json"
PART11I_LOCK = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.json"
PART11I_LOCK_SHA = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.sha256"

OUTPUT_DIR = VALIDATION_ROOT / "11IA1_expanded_weekly_challenge"
PACKAGE_AUDIT_PATH = OUTPUT_DIR / "11IA1_package_audit.csv"
FEATURE_AUDIT_PATH = OUTPUT_DIR / "11IA1_feature_audit.csv"
FOLD_DESIGN_PATH = OUTPUT_DIR / "11IA1_chronological_fold_design.csv"
CANDIDATE_AUDIT_PATH = OUTPUT_DIR / "11IA1_candidate_audit.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_predictions.csv"
POOLED_METRICS_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_pooled_metrics.csv"
FOLD_METRICS_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_fold_metrics.csv"
RANKING_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_ranking.csv"
COMPARISON_PATH = OUTPUT_DIR / "11IA1_challenger_vs_locked_benchmark.csv"
VALIDATION_PATH = OUTPUT_DIR / "11IA1_validation.csv"
CONTRACT_PATH = OUTPUT_DIR / "11IA1_expanded_weekly_challenge_contract.json"
MANIFEST_PATH = OUTPUT_DIR / "11IA1_output_hash_manifest.csv"
STEP_MEMORY_PATH = STEP_MEMORY_ROOT / "STEP_11IA1_EXPANDED_WEEKLY_MODEL_CHALLENGE.md"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "11IA1_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "11IA1_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "11IA1_expanded_weekly_challenge_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "11IA1_expanded_weekly_challenge_lock.sha256"
LOG_PATH = LOG_ROOT / "11IA1_expanded_weekly_challenge_log.txt"

STEP_ID = "11IA1"
STATUS = "PART_11IA1_COMPLETED_READY_FOR_11IA2"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

TARGET_ID = "WEEKLY_NORMAL_DEMAND"
TARGET_COLUMN = "WeeklyNormalDemand"
LOCKED_BENCHMARK_METHOD = "NAIVE_LAST_CONTEXT"
PRIMARY_SCOPE_ID = "FOLD_STRICT_95_PERCENT"
MINIMUM_TRAINING_WEEKS = 8
RANDOM_SEED = 6073

SCOPE_FLAGS = {
    "ALL_PRODUCTS": None,
    "FOLD_STRICT_80_PERCENT": "PriorStrict80PctScope",
    "FOLD_STRICT_90_PERCENT": "PriorStrict90PctScope",
    "FOLD_STRICT_95_PERCENT": "PriorStrict95PctScope",
}

BASELINE_SCOPE_ALIASES = {
    "InFoldStrict80PctScope": "PriorStrict80PctScope",
    "InFoldStrict90PctScope": "PriorStrict90PctScope",
    "InFoldStrict95PctScope": "PriorStrict95PctScope",
}

CANONICAL_SCOPE_COLUMNS = [
    "PriorStrict80PctScope",
    "PriorStrict90PctScope",
    "PriorStrict95PctScope",
]

MEMORY_FILES = {
    "PROJECT_CONTEXT": MEMORY_ROOT / "PROJECT_CONTEXT.md",
    "WORKFLOW": MEMORY_ROOT / "WORKFLOW.md",
    "DECISIONS": MEMORY_ROOT / "DECISIONS.md",
    "FILES_AND_PATHS": MEMORY_ROOT / "FILES_AND_PATHS.md",
    "METRICS_AND_RESULTS": MEMORY_ROOT / "METRICS_AND_RESULTS.md",
    "CHAT_INDEX": MEMORY_ROOT / "CHAT_INDEX.md",
    "CURRENT_HANDOFF": MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

NEW_OUTPUTS = [
    PACKAGE_AUDIT_PATH,
    FEATURE_AUDIT_PATH,
    FOLD_DESIGN_PATH,
    CANDIDATE_AUDIT_PATH,
    PREDICTIONS_PATH,
    POOLED_METRICS_PATH,
    FOLD_METRICS_PATH,
    RANKING_PATH,
    COMPARISON_PATH,
    VALIDATION_PATH,
    CONTRACT_PATH,
    MANIFEST_PATH,
    STEP_MEMORY_PATH,
    CHECKPOINT_PATH,
    CHECKPOINT_SHA_PATH,
    LOCK_PATH,
    LOCK_SHA_PATH,
    LOG_PATH,
]

# =============================================================================
# GENERAL HELPERS
# =============================================================================


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def verify_sidecar(path: Path, sidecar: Path, label: str) -> str:
    require_file(path, label)
    require_file(sidecar, f"{label} SHA-256 sidecar")
    expected = sidecar.read_text(encoding="utf-8").strip().split()[0].lower()
    actual = sha256_file(path)
    if expected != actual:
        raise AssertionError(
            f"{label} SHA-256 mismatch:\nExpected: {expected}\nActual:   {actual}"
        )
    return actual


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_json(path: Path, payload: dict) -> None:
    atomic_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        frame.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def stage_path(stage_root: Path, final_path: Path) -> Path:
    return stage_root / final_path.relative_to(EXT_ROOT)


def stage_text(stage_root: Path, final_path: Path, text: str) -> None:
    atomic_text(stage_path(stage_root, final_path), text)


def stage_json(stage_root: Path, final_path: Path, payload: dict) -> None:
    atomic_json(stage_path(stage_root, final_path), payload)


def stage_csv(stage_root: Path, final_path: Path, frame: pd.DataFrame) -> None:
    atomic_csv(stage_path(stage_root, final_path), frame)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def bool_series(series: pd.Series, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    normalized = series.astype(str).str.strip().str.lower()
    invalid = sorted(set(normalized) - set(mapping))
    if invalid:
        raise ValueError(f"{label} has invalid Boolean values: {invalid[:10]}")
    return normalized.map(mapping).astype(bool)


def normalize_scope_columns(
    frame: pd.DataFrame,
    frame_label: str,
    aliases: dict[str, str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    aliases = aliases or {}
    rename_map: dict[str, str] = {}

    for source, target in aliases.items():
        if source in result.columns and target in result.columns:
            left = bool_series(result[source], f"{frame_label}.{source}")
            right = bool_series(result[target], f"{frame_label}.{target}")
            if not left.equals(right):
                raise AssertionError(
                    f"{frame_label} contains conflicting {source} and {target}"
                )
            result = result.drop(columns=[source])
        elif source in result.columns:
            rename_map[source] = target

    if rename_map:
        result = result.rename(columns=rename_map)

    missing = [c for c in CANONICAL_SCOPE_COLUMNS if c not in result.columns]
    if missing:
        raise AssertionError(
            f"{frame_label} is missing normalized scope columns: {missing}"
        )

    for column in CANONICAL_SCOPE_COLUMNS:
        result[column] = bool_series(result[column], f"{frame_label}.{column}")

    return result


def metric_record(actual: pd.Series, prediction: pd.Series) -> dict:
    actual_array = pd.to_numeric(actual, errors="raise").to_numpy(dtype=float)
    prediction_array = pd.to_numeric(prediction, errors="raise").to_numpy(dtype=float)

    if len(actual_array) != len(prediction_array):
        raise AssertionError("Actual and prediction lengths differ")
    if not np.isfinite(actual_array).all() or not np.isfinite(prediction_array).all():
        raise AssertionError("Non-finite values entered metric calculation")

    errors = prediction_array - actual_array
    absolute_errors = np.abs(errors)
    denominator = float(np.abs(actual_array).sum())

    return {
        "Observations": int(len(actual_array)),
        "ActualTotal": float(actual_array.sum()),
        "PredictedTotal": float(prediction_array.sum()),
        "MAE": float(absolute_errors.mean()) if len(actual_array) else np.nan,
        "RMSE": float(np.sqrt(np.mean(errors ** 2))) if len(actual_array) else np.nan,
        "WAPEPercentage": (
            float(100.0 * absolute_errors.sum() / denominator)
            if denominator != 0
            else np.nan
        ),
        "MeanBias": float(errors.mean()) if len(actual_array) else np.nan,
        "TotalBias": float(errors.sum()),
    }


def update_section(text: str, marker: str, heading: str, body: str) -> str:
    start = f"<!-- BEGIN {marker} -->"
    end = f"<!-- END {marker} -->"
    section = f"{start}\n## {heading}\n\n{body.rstrip()}\n{end}"

    if start in text and end in text:
        before = text.split(start, 1)[0].rstrip()
        after = text.split(end, 1)[1].lstrip()
        return before + "\n\n" + section + ("\n\n" + after if after else "") + "\n"

    return text.rstrip() + "\n\n" + section + "\n"


def package_version(distribution_name: str) -> str | None:
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return None


def make_ohe() -> OneHotEncoder:
    parameters = inspect.signature(OneHotEncoder).parameters
    kwargs = {"handle_unknown": "ignore"}
    if "sparse_output" in parameters:
        kwargs["sparse_output"] = False
    else:
        kwargs["sparse"] = False
    return OneHotEncoder(**kwargs)


def make_imputer() -> SimpleImputer:
    parameters = inspect.signature(SimpleImputer).parameters
    kwargs = {"strategy": "median"}
    if "keep_empty_features" in parameters:
        kwargs["keep_empty_features"] = True
    return SimpleImputer(**kwargs)


def candidate_complexity_order(candidate_id: str) -> int:
    order = {
        LOCKED_BENCHMARK_METHOD: 0,
        "HISTGB_POISSON": 10,
        "RANDOM_FOREST_SQUARED": 20,
        "RANDOM_FOREST_POISSON": 21,
        "EXTRA_TREES_SQUARED": 30,
        "EXTRA_TREES_POISSON": 31,
        "XGBOOST_SQUARED": 40,
        "XGBOOST_POISSON": 41,
        "LIGHTGBM_TWEEDIE": 50,
        "CATBOOST_RMSE": 60,
    }
    return order.get(candidate_id, 999)


def recursively_find_predictor_lists(obj, path: tuple[str, ...] = ()) -> list[tuple[str, list[str]]]:
    found: list[tuple[str, list[str]]] = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            found.extend(recursively_find_predictor_lists(value, path + (str(key),)))
    elif isinstance(obj, list) and obj and all(isinstance(item, str) for item in obj):
        joined = ".".join(path).lower()
        if "predict" in joined and "normal" in joined:
            found.append((".".join(path), obj))
    return found


def discover_normal_predictors(frame: pd.DataFrame) -> tuple[list[str], str, str | None]:
    contract_candidates = sorted(FEATURE_ROOT.glob("*11F*contract*.json"))
    contract_candidates += sorted(FEATURE_ROOT.glob("**/*11F*contract*.json"))
    contract_candidates = list(dict.fromkeys(contract_candidates))

    for contract_path in contract_candidates:
        try:
            payload = json.loads(contract_path.read_text(encoding="utf-8"))
        except Exception:
            continue

        candidates = recursively_find_predictor_lists(payload)
        candidates = [
            (name, values)
            for name, values in candidates
            if all(value in frame.columns for value in values)
        ]
        if candidates:
            candidates.sort(key=lambda item: (abs(len(item[1]) - 48), item[0]))
            selected_name, selected_values = candidates[0]
            return list(selected_values), f"CONTRACT:{selected_name}", str(contract_path)

    explicit_exclusions = {
        "WeekStartDate",
        "WeekEndDate",
        "WeekID",
        "CanonicalProductID",
        "CanonicalProductName",
        "TargetID",
        TARGET_COLUMN,
        "WeeklyTotalDemand",
        "WeeklyBulkDemand",
        "BulkDemand",
        "NormalDemand",
        "TotalDemand",
        "Actual",
        "Prediction",
        "WeeklyTuningEligible",
        "IsOpenedMarchHoldout",
        *CANONICAL_SCOPE_COLUMNS,
    }

    forbidden_exact_or_prefix = (
        "CurrentWeek",
        "Future",
        "ObservedTarget",
        "Protected",
    )

    predictors: list[str] = []
    for column in frame.columns:
        if column in explicit_exclusions:
            continue
        if column.startswith(forbidden_exact_or_prefix):
            continue
        if "Scope" in column:
            continue
        if "Target" in column and not any(
            token in column for token in ["Lag", "Mean", "Median", "Sum", "Std", "Rate", "Count"]
        ):
            continue
        if pd.api.types.is_numeric_dtype(frame[column]) or pd.api.types.is_bool_dtype(frame[column]):
            predictors.append(column)

    if not predictors:
        raise AssertionError("No fallback numeric predictor columns were discovered")

    return predictors, "FALLBACK_NUMERIC_SCHEMA", None


# =============================================================================
# INPUT AND LOCK VERIFICATION
# =============================================================================

for directory, label in [
    (EXT_ROOT, "extension root"),
    (FEATURE_ROOT, "weekly feature directory"),
    (BASELINE_ROOT, "weekly baseline directory"),
    (MODEL_ROOT, "weekly model directory"),
    (VALIDATION_ROOT, "weekly validation directory"),
    (CHECKPOINT_ROOT, "checkpoint directory"),
    (MEMORY_ROOT, "project memory directory"),
    (STEP_MEMORY_ROOT, "step memory directory"),
    (LOG_ROOT, "log directory"),
]:
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing required {label}:\n{directory}")

for path, label in [
    (PART11F_FEATURES, "Part 11F model-selection features"),
    (PART11E_PREDICTIONS, "Part 11E baseline predictions"),
    (PART11I_DESIGN, "Part 11I final design"),
]:
    require_file(path, label)

for key, path in MEMORY_FILES.items():
    require_file(path, f"memory file {key}")

if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise FileExistsError(f"Part 11IA1 overwrite lock triggered:\n{LOCK_PATH}")

existing_outputs = [path for path in NEW_OUTPUTS if path.exists()]
if existing_outputs:
    raise FileExistsError(
        "Existing Part 11IA1 outputs found; no files changed:\n"
        + "\n".join(f"- {path}" for path in existing_outputs)
    )

for old_stage in EXT_ROOT.glob(".11IA1_staging_*"):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)

part11i_lock_hash = verify_sidecar(PART11I_LOCK, PART11I_LOCK_SHA, "Part 11I lock")
part11i_lock = json.loads(PART11I_LOCK.read_text(encoding="utf-8"))
part11i_design = json.loads(PART11I_DESIGN.read_text(encoding="utf-8"))

if part11i_lock.get("Status") != "PART_11I_COMPLETED_READY_FOR_11J":
    raise AssertionError(f"Unexpected Part 11I status: {part11i_lock.get('Status')}")

locked_design = part11i_lock.get("FinalForecastingDesign", {})
if locked_design.get("TargetID") != TARGET_ID:
    raise AssertionError("Part 11I target is not WEEKLY_NORMAL_DEMAND")
if locked_design.get("MethodID") != LOCKED_BENCHMARK_METHOD:
    raise AssertionError("Unexpected locked Part 11I benchmark method")
if locked_design.get("ScopeID") != PRIMARY_SCOPE_ID:
    raise AssertionError("Unexpected locked Part 11I primary scope")
if locked_design.get("BulkIncludedInModelledTarget") is not False:
    raise AssertionError("Part 11I did not exclude bulk from the model target")

for key, expected in {
    "OpenedMarch2026TargetsRead": False,
    "BulkIncludedInModelledTarget": False,
    "ModelsFittedOrRefitted": False,
    "FinalForecastDesignLocked": True,
    "ForecastsRoundedBeforeScoring": False,
}.items():
    actual = part11i_lock.get("SafetyAssertions", {}).get(key)
    if actual is not expected:
        raise AssertionError(f"Invalid Part 11I safety assertion {key}: {actual}")

if sha256_file(PART11I_DESIGN) != locked_design.get("SHA256"):
    raise AssertionError("Part 11I final design hash changed")

# =============================================================================
# PACKAGE AUDIT AND CANDIDATE REGISTRY
# =============================================================================

package_rows = []
for distribution, import_name, role in [
    ("numpy", "numpy", "CORE"),
    ("pandas", "pandas", "CORE"),
    ("scikit-learn", "sklearn", "CORE"),
    ("xgboost", "xgboost", "OPTIONAL_STRONG_MODEL"),
    ("lightgbm", "lightgbm", "OPTIONAL_STRONG_MODEL"),
    ("catboost", "catboost", "OPTIONAL_STRONG_MODEL"),
]:
    version = package_version(distribution)
    import_ok = False
    import_error = ""
    if version is not None:
        try:
            importlib.import_module(import_name)
            import_ok = True
        except Exception as exc:
            import_error = f"{type(exc).__name__}: {exc}"
    package_rows.append(
        {
            "Distribution": distribution,
            "ImportName": import_name,
            "Role": role,
            "Installed": version is not None,
            "Version": version or "",
            "ImportSucceeded": import_ok,
            "ImportError": import_error,
        }
    )

package_audit = pd.DataFrame(package_rows)
package_lookup = package_audit.set_index("ImportName")["ImportSucceeded"].to_dict()

candidate_specs: list[dict] = []

candidate_specs.append(
    {
        "CandidateID": "HISTGB_POISSON",
        "CandidateFamily": "HISTOGRAM_GRADIENT_BOOSTING",
        "Package": "scikit-learn",
        "UsesProductID": True,
        "CountAware": True,
        "Available": True,
        "Estimator": HistGradientBoostingRegressor(
            loss="poisson",
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=RANDOM_SEED,
        ),
        "SpecialRunner": None,
        "Configuration": {
            "loss": "poisson",
            "learning_rate": 0.05,
            "max_iter": 300,
            "max_leaf_nodes": 31,
            "min_samples_leaf": 20,
            "l2_regularization": 1.0,
        },
    }
)

for candidate_id, criterion, count_aware in [
    ("RANDOM_FOREST_SQUARED", "squared_error", False),
    ("RANDOM_FOREST_POISSON", "poisson", True),
]:
    candidate_specs.append(
        {
            "CandidateID": candidate_id,
            "CandidateFamily": "RANDOM_FOREST",
            "Package": "scikit-learn",
            "UsesProductID": True,
            "CountAware": count_aware,
            "Available": True,
            "Estimator": RandomForestRegressor(
                n_estimators=300,
                criterion=criterion,
                max_depth=None,
                min_samples_split=4,
                min_samples_leaf=2,
                max_features=0.70,
                bootstrap=True,
                n_jobs=-1,
                random_state=RANDOM_SEED,
            ),
            "SpecialRunner": None,
            "Configuration": {
                "n_estimators": 300,
                "criterion": criterion,
                "min_samples_leaf": 2,
                "max_features": 0.70,
                "bootstrap": True,
            },
        }
    )

for candidate_id, criterion, count_aware in [
    ("EXTRA_TREES_SQUARED", "squared_error", False),
    ("EXTRA_TREES_POISSON", "poisson", True),
]:
    candidate_specs.append(
        {
            "CandidateID": candidate_id,
            "CandidateFamily": "EXTRA_TREES",
            "Package": "scikit-learn",
            "UsesProductID": True,
            "CountAware": count_aware,
            "Available": True,
            "Estimator": ExtraTreesRegressor(
                n_estimators=300,
                criterion=criterion,
                max_depth=None,
                min_samples_split=4,
                min_samples_leaf=2,
                max_features=0.85,
                bootstrap=False,
                n_jobs=-1,
                random_state=RANDOM_SEED,
            ),
            "SpecialRunner": None,
            "Configuration": {
                "n_estimators": 300,
                "criterion": criterion,
                "min_samples_leaf": 2,
                "max_features": 0.85,
                "bootstrap": False,
            },
        }
    )

if package_lookup.get("xgboost", False):
    from xgboost import XGBRegressor

    for candidate_id, objective, count_aware in [
        ("XGBOOST_SQUARED", "reg:squarederror", False),
        ("XGBOOST_POISSON", "count:poisson", True),
    ]:
        candidate_specs.append(
            {
                "CandidateID": candidate_id,
                "CandidateFamily": "XGBOOST",
                "Package": "xgboost",
                "UsesProductID": True,
                "CountAware": count_aware,
                "Available": True,
                "Estimator": XGBRegressor(
                    objective=objective,
                    n_estimators=500,
                    learning_rate=0.03,
                    max_depth=6,
                    min_child_weight=5,
                    subsample=0.80,
                    colsample_bytree=0.80,
                    reg_alpha=0.0,
                    reg_lambda=2.0,
                    n_jobs=-1,
                    random_state=RANDOM_SEED,
                    tree_method="hist",
                    verbosity=0,
                ),
                "SpecialRunner": None,
                "Configuration": {
                    "objective": objective,
                    "n_estimators": 500,
                    "learning_rate": 0.03,
                    "max_depth": 6,
                    "min_child_weight": 5,
                    "subsample": 0.80,
                    "colsample_bytree": 0.80,
                    "reg_lambda": 2.0,
                },
            }
        )

if package_lookup.get("lightgbm", False):
    from lightgbm import LGBMRegressor

    candidate_specs.append(
        {
            "CandidateID": "LIGHTGBM_TWEEDIE",
            "CandidateFamily": "LIGHTGBM",
            "Package": "lightgbm",
            "UsesProductID": True,
            "CountAware": True,
            "Available": True,
            "Estimator": LGBMRegressor(
                objective="tweedie",
                tweedie_variance_power=1.3,
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.0,
                reg_lambda=1.0,
                n_jobs=-1,
                random_state=RANDOM_SEED,
                verbosity=-1,
            ),
            "SpecialRunner": None,
            "Configuration": {
                "objective": "tweedie",
                "tweedie_variance_power": 1.3,
                "n_estimators": 500,
                "learning_rate": 0.03,
                "num_leaves": 31,
                "min_child_samples": 20,
                "subsample": 0.80,
                "colsample_bytree": 0.80,
            },
        }
    )

if package_lookup.get("catboost", False):
    candidate_specs.append(
        {
            "CandidateID": "CATBOOST_RMSE",
            "CandidateFamily": "CATBOOST",
            "Package": "catboost",
            "UsesProductID": True,
            "CountAware": False,
            "Available": True,
            "Estimator": None,
            "SpecialRunner": "CATBOOST",
            "Configuration": {
                "loss_function": "RMSE",
                "iterations": 500,
                "learning_rate": 0.03,
                "depth": 7,
                "l2_leaf_reg": 3.0,
                "random_strength": 1.0,
            },
        }
    )

# =============================================================================
# LOAD FEATURES AND BASELINE
# =============================================================================

features = pd.read_csv(PART11F_FEATURES, low_memory=False)
baseline_predictions = pd.read_csv(PART11E_PREDICTIONS, low_memory=False)

for frame, label in [(features, "Part 11F features"), (baseline_predictions, "Part 11E predictions")]:
    for column in ["WeekStartDate", "WeekEndDate"]:
        if column not in frame.columns:
            raise AssertionError(f"{label} is missing {column}")
        frame[column] = pd.to_datetime(frame[column], errors="raise")
    if "CanonicalProductID" not in frame.columns:
        raise AssertionError(f"{label} is missing CanonicalProductID")
    frame["CanonicalProductID"] = frame["CanonicalProductID"].astype(str)

features = normalize_scope_columns(features, "Part 11F features")
baseline_predictions = normalize_scope_columns(
    baseline_predictions,
    "Part 11E baseline predictions",
    aliases=BASELINE_SCOPE_ALIASES,
)

required_feature_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    TARGET_COLUMN,
    *CANONICAL_SCOPE_COLUMNS,
}
if not required_feature_columns.issubset(features.columns):
    raise AssertionError(
        "Part 11F feature schema is incomplete. Missing: "
        f"{sorted(required_feature_columns - set(features.columns))}"
    )

# The Part 11F file is already the pre-holdout model-selection universe.
# No March 2026 target file or opened holdout source is read here.
features[TARGET_COLUMN] = pd.to_numeric(features[TARGET_COLUMN], errors="raise").astype(float)
if (features[TARGET_COLUMN] < 0).any():
    raise AssertionError("Negative weekly normal-demand targets found")

predictor_columns, predictor_source, predictor_contract_path = discover_normal_predictors(features)

# Explicit leakage guard. Predictor contracts must contain only information
# available before the forecast week. Lagged and rolling demand features are
# allowed; current targets, scope flags and evaluation outputs are not.
forbidden_predictors = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductName",
    "TargetID",
    TARGET_COLUMN,
    "WeeklyTotalDemand",
    "WeeklyBulkDemand",
    "BulkDemand",
    "NormalDemand",
    "TotalDemand",
    "Actual",
    "Prediction",
    "WeeklyTuningEligible",
    "IsOpenedMarchHoldout",
    *CANONICAL_SCOPE_COLUMNS,
}
leakage_predictors = sorted(
    column
    for column in predictor_columns
    if column in forbidden_predictors or "Scope" in column
)
if leakage_predictors:
    raise AssertionError(
        "Leakage/evaluation columns entered the predictor list: "
        f"{leakage_predictors}"
    )

# Product identity is intentionally added as a direct categorical predictor in
# this challenger stage. It is encoded inside each training fold only.
categorical_columns = ["CanonicalProductID"]
numeric_predictor_columns = [
    column
    for column in predictor_columns
    if column != "CanonicalProductID"
]

# Force numeric conversion of contract/fallback predictors. Infinite values are
# converted to missing values and imputed from each fold's training data only.
for column in numeric_predictor_columns:
    features[column] = pd.to_numeric(features[column], errors="coerce")
features[numeric_predictor_columns] = features[numeric_predictor_columns].replace(
    [np.inf, -np.inf], np.nan
)

# Remove columns that are entirely non-finite over the complete pre-holdout file.
entirely_empty_predictors = [
    column
    for column in numeric_predictor_columns
    if not np.isfinite(features[column].to_numpy(dtype=float)).any()
]
numeric_predictor_columns = [
    column for column in numeric_predictor_columns if column not in entirely_empty_predictors
]

if not numeric_predictor_columns:
    raise AssertionError("No usable numeric predictors remain")

feature_audit = pd.DataFrame(
    [
        {
            "FeatureName": column,
            "FeatureRole": "CATEGORICAL_PRODUCT_ID",
            "Source": "DIRECT_CHALLENGER_ADDITION",
            "DType": str(features[column].dtype),
            "MissingCount": int(features[column].isna().sum()),
            "FiniteCount": int(features[column].notna().sum()),
            "Included": True,
        }
        for column in categorical_columns
    ]
    + [
        {
            "FeatureName": column,
            "FeatureRole": "NUMERIC_PREDICTOR",
            "Source": predictor_source,
            "DType": str(features[column].dtype),
            "MissingCount": int(features[column].isna().sum()),
            "FiniteCount": int(np.isfinite(features[column].to_numpy(dtype=float)).sum()),
            "Included": True,
        }
        for column in numeric_predictor_columns
    ]
    + [
        {
            "FeatureName": column,
            "FeatureRole": "ENTIRELY_EMPTY_REMOVED",
            "Source": predictor_source,
            "DType": str(features[column].dtype),
            "MissingCount": int(features[column].isna().sum()),
            "FiniteCount": 0,
            "Included": False,
        }
        for column in entirely_empty_predictors
    ]
)

# =============================================================================
# CHRONOLOGICAL FOLD DESIGN
# =============================================================================

weeks = sorted(features["WeekStartDate"].drop_duplicates())
if len(weeks) != 47:
    raise AssertionError(f"Expected 47 pre-holdout weeks, found {len(weeks)}")

forecast_weeks = weeks[MINIMUM_TRAINING_WEEKS:]
if len(forecast_weeks) != 39:
    raise AssertionError(f"Expected 39 forecast weeks, found {len(forecast_weeks)}")

fold_rows = []
for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
    training = features.loc[features["WeekStartDate"] < forecast_week]
    validation = features.loc[features["WeekStartDate"] == forecast_week]
    fold_rows.append(
        {
            "FoldNumber": fold_number,
            "ForecastWeek": forecast_week,
            "TrainingStartWeek": training["WeekStartDate"].min(),
            "TrainingEndWeek": training["WeekStartDate"].max(),
            "TrainingWeeks": int(training["WeekStartDate"].nunique()),
            "TrainingRows": int(len(training)),
            "ValidationRows": int(len(validation)),
            "ValidationProducts": int(validation["CanonicalProductID"].nunique()),
            "Strict95Rows": int(validation["PriorStrict95PctScope"].sum()),
        }
    )
fold_design = pd.DataFrame(fold_rows)

# Baseline rows used for exact same-fold comparison.
required_baseline_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "TargetID",
    "BaselineMethod",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
}
if not required_baseline_columns.issubset(baseline_predictions.columns):
    raise AssertionError("Part 11E baseline-prediction schema is incomplete")

locked_baseline = baseline_predictions.loc[
    (baseline_predictions["TargetID"].astype(str) == TARGET_ID)
    & (baseline_predictions["BaselineMethod"].astype(str) == LOCKED_BENCHMARK_METHOD)
].copy()

if locked_baseline.empty:
    raise AssertionError("Locked NAIVE_LAST_CONTEXT rows are missing")

locked_baseline["Actual"] = pd.to_numeric(locked_baseline["Actual"], errors="raise").astype(float)
locked_baseline["Prediction"] = pd.to_numeric(locked_baseline["Prediction"], errors="raise").astype(float)

# Rename the feature-side target before merging. Part 11E can also contain a
# WeeklyNormalDemand column, so relying on pandas' automatic suffixing would
# remove the unsuffixed TARGET_COLUMN name and make reconciliation ambiguous.
FEATURE_RECONCILIATION_TARGET = f"{TARGET_COLUMN}__Part11FFeatureTarget"

feature_keys = features[
    [
        "WeekStartDate",
        "CanonicalProductID",
        TARGET_COLUMN,
        *CANONICAL_SCOPE_COLUMNS,
    ]
].copy()

feature_keys = feature_keys.rename(
    columns={TARGET_COLUMN: FEATURE_RECONCILIATION_TARGET}
)

reconciled_baseline = locked_baseline.merge(
    feature_keys,
    on=["WeekStartDate", "CanonicalProductID"],
    how="inner",
    suffixes=("__Baseline", "__Feature"),
    validate="one_to_one",
)

if len(reconciled_baseline) != len(locked_baseline):
    raise AssertionError("Baseline and feature rows do not reconcile one-to-one")

if FEATURE_RECONCILIATION_TARGET not in reconciled_baseline.columns:
    raise AssertionError(
        "Feature-side normal-demand target is missing after baseline reconciliation. "
        f"Expected column: {FEATURE_RECONCILIATION_TARGET}"
    )

actual_difference = np.abs(
    reconciled_baseline["Actual"].to_numpy(dtype=float)
    - reconciled_baseline[FEATURE_RECONCILIATION_TARGET].to_numpy(dtype=float)
).max()
if actual_difference > 1e-9:
    raise AssertionError(f"Baseline actuals differ from Part 11F targets by {actual_difference}")

for scope_column in CANONICAL_SCOPE_COLUMNS:
    baseline_column = f"{scope_column}__Baseline"
    feature_column = f"{scope_column}__Feature"
    if baseline_column in reconciled_baseline.columns and feature_column in reconciled_baseline.columns:
        mismatch = int((reconciled_baseline[baseline_column] != reconciled_baseline[feature_column]).sum())
        if mismatch:
            raise AssertionError(f"Scope mismatch for {scope_column}: {mismatch}")

# =============================================================================
# PREPROCESSOR AND MODEL RUNNERS
# =============================================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "product",
            Pipeline([("one_hot", make_ohe())]),
            categorical_columns,
        ),
        (
            "numeric",
            Pipeline([("imputer", make_imputer())]),
            numeric_predictor_columns,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)


def run_standard_candidate(spec: dict, train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    estimator = clone(spec["Estimator"])
    pipeline = Pipeline(
        [
            ("preprocess", clone(preprocessor)),
            ("model", estimator),
        ]
    )
    x_train = train[categorical_columns + numeric_predictor_columns]
    y_train = train[TARGET_COLUMN].to_numpy(dtype=float)
    x_valid = valid[categorical_columns + numeric_predictor_columns]
    pipeline.fit(x_train, y_train)
    prediction = pipeline.predict(x_valid)
    return np.clip(np.asarray(prediction, dtype=float), 0.0, None)


def run_catboost_candidate(spec: dict, train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    from catboost import CatBoostRegressor

    usable_numeric = [
        column
        for column in numeric_predictor_columns
        if np.isfinite(train[column].to_numpy(dtype=float)).any()
    ]
    columns = categorical_columns + usable_numeric
    x_train = train[columns].copy()
    x_valid = valid[columns].copy()
    x_train["CanonicalProductID"] = x_train["CanonicalProductID"].astype(str)
    x_valid["CanonicalProductID"] = x_valid["CanonicalProductID"].astype(str)

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=500,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,
        random_strength=1.0,
        random_seed=RANDOM_SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    model.fit(
        x_train,
        train[TARGET_COLUMN].to_numpy(dtype=float),
        cat_features=[0],
    )
    prediction = model.predict(x_valid)
    return np.clip(np.asarray(prediction, dtype=float), 0.0, None)


# =============================================================================
# EXPANDING-WINDOW CHALLENGE
# =============================================================================

prediction_rows: list[pd.DataFrame] = []
candidate_audit_rows: list[dict] = []

for spec in candidate_specs:
    candidate_id = spec["CandidateID"]
    candidate_failed = False
    candidate_error = ""
    successful_folds = 0

    print("-" * 100)
    print(f"Running candidate: {candidate_id}")

    for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
        train = features.loc[features["WeekStartDate"] < forecast_week].copy()
        valid = features.loc[features["WeekStartDate"] == forecast_week].copy()

        if train["WeekStartDate"].nunique() < MINIMUM_TRAINING_WEEKS:
            raise AssertionError("Fold has insufficient training weeks")
        if valid.empty:
            raise AssertionError(f"Empty validation week: {forecast_week}")

        try:
            if spec["SpecialRunner"] == "CATBOOST":
                prediction = run_catboost_candidate(spec, train, valid)
            else:
                prediction = run_standard_candidate(spec, train, valid)
        except Exception as exc:
            candidate_failed = True
            candidate_error = (
                f"Fold {fold_number}, week {forecast_week.date()}: "
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()
            break

        if len(prediction) != len(valid):
            raise AssertionError(f"Prediction length mismatch for {candidate_id}")
        if not np.isfinite(prediction).all():
            raise AssertionError(f"Non-finite predictions from {candidate_id}")
        if (prediction < 0).any():
            raise AssertionError(f"Negative predictions from {candidate_id}")

        output = valid[
            [
                "WeekStartDate",
                "WeekEndDate",
                "WeekID",
                "CanonicalProductID",
                "CanonicalProductName",
                TARGET_COLUMN,
                *CANONICAL_SCOPE_COLUMNS,
            ]
        ].copy()
        output["FoldNumber"] = fold_number
        output["TargetID"] = TARGET_ID
        output["CandidateID"] = candidate_id
        output["CandidateFamily"] = spec["CandidateFamily"]
        output["Actual"] = output[TARGET_COLUMN].astype(float)
        output["Prediction"] = prediction
        output["Error"] = output["Prediction"] - output["Actual"]
        output["AbsoluteError"] = output["Error"].abs()
        output["ProductIDUsedAsDirectPredictor"] = bool(spec["UsesProductID"])
        output["BulkIncludedInModelTarget"] = False
        prediction_rows.append(output)
        successful_folds += 1

    candidate_audit_rows.append(
        {
            "CandidateID": candidate_id,
            "CandidateFamily": spec["CandidateFamily"],
            "Package": spec["Package"],
            "UsesProductID": spec["UsesProductID"],
            "CountAware": spec["CountAware"],
            "Available": spec["Available"],
            "SuccessfulFolds": successful_folds,
            "ExpectedFolds": len(forecast_weeks),
            "CompletedAllFolds": not candidate_failed and successful_folds == len(forecast_weeks),
            "FailureMessage": candidate_error,
            "ConfigurationJSON": json.dumps(spec["Configuration"], sort_keys=True),
        }
    )

candidate_audit = pd.DataFrame(candidate_audit_rows)
completed_candidate_ids = candidate_audit.loc[
    candidate_audit["CompletedAllFolds"], "CandidateID"
].astype(str).tolist()

if not completed_candidate_ids:
    raise RuntimeError("No expanded weekly candidate completed all 39 folds")

candidate_predictions = pd.concat(prediction_rows, ignore_index=True)
candidate_predictions = candidate_predictions.loc[
    candidate_predictions["CandidateID"].isin(completed_candidate_ids)
].copy()

# Add locked benchmark rows to a common comparison table, but keep them clearly
# labelled as the existing reference rather than a newly fitted candidate.
baseline_common = locked_baseline[
    [
        "WeekStartDate",
        "WeekEndDate",
        "WeekID",
        "CanonicalProductID",
        "CanonicalProductName",
        "Actual",
        "Prediction",
        *CANONICAL_SCOPE_COLUMNS,
    ]
].copy()
baseline_common["FoldNumber"] = baseline_common["WeekStartDate"].map(
    {week: index for index, week in enumerate(forecast_weeks, start=1)}
)
baseline_common["TargetID"] = TARGET_ID
baseline_common["CandidateID"] = LOCKED_BENCHMARK_METHOD
baseline_common["CandidateFamily"] = "LOCKED_REFERENCE_BASELINE"
baseline_common["Error"] = baseline_common["Prediction"] - baseline_common["Actual"]
baseline_common["AbsoluteError"] = baseline_common["Error"].abs()
baseline_common["ProductIDUsedAsDirectPredictor"] = False
baseline_common["BulkIncludedInModelTarget"] = False

comparison_predictions = pd.concat(
    [candidate_predictions, baseline_common],
    ignore_index=True,
    sort=False,
)

# =============================================================================
# METRICS, RANKING AND CHALLENGER DECISION
# =============================================================================

pooled_rows = []
for (candidate_id, candidate_family), group in comparison_predictions.groupby(
    ["CandidateID", "CandidateFamily"], sort=True
):
    for scope_id, scope_flag in SCOPE_FLAGS.items():
        scoped = group if scope_flag is None else group.loc[group[scope_flag]]
        pooled_rows.append(
            {
                "TargetID": TARGET_ID,
                "ScopeID": scope_id,
                "CandidateID": candidate_id,
                "CandidateFamily": candidate_family,
                "ForecastWeeks": int(scoped["WeekStartDate"].nunique()),
                "Products": int(scoped["CanonicalProductID"].nunique()),
                "ProductIDUsedAsDirectPredictor": bool(
                    scoped["ProductIDUsedAsDirectPredictor"].iloc[0]
                ),
                **metric_record(scoped["Actual"], scoped["Prediction"]),
            }
        )
pooled_metrics = pd.DataFrame(pooled_rows)
pooled_metrics["AbsoluteTotalBias"] = pooled_metrics["TotalBias"].abs()
pooled_metrics["ComplexityOrder"] = pooled_metrics["CandidateID"].map(candidate_complexity_order)
pooled_metrics = pooled_metrics.sort_values(
    [
        "ScopeID",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "ComplexityOrder",
        "CandidateID",
    ],
    kind="mergesort",
).reset_index(drop=True)
pooled_metrics["RankWithinScope"] = pooled_metrics.groupby("ScopeID").cumcount() + 1
pooled_metrics["IsScopeWinner"] = pooled_metrics["RankWithinScope"] == 1

fold_metric_rows = []
for (candidate_id, candidate_family, forecast_week), group in comparison_predictions.groupby(
    ["CandidateID", "CandidateFamily", "WeekStartDate"], sort=True
):
    for scope_id, scope_flag in SCOPE_FLAGS.items():
        scoped = group if scope_flag is None else group.loc[group[scope_flag]]
        fold_metric_rows.append(
            {
                "ForecastWeek": forecast_week,
                "TargetID": TARGET_ID,
                "ScopeID": scope_id,
                "CandidateID": candidate_id,
                "CandidateFamily": candidate_family,
                **metric_record(scoped["Actual"], scoped["Prediction"]),
            }
        )
fold_metrics = pd.DataFrame(fold_metric_rows)

primary_metrics = pooled_metrics.loc[pooled_metrics["ScopeID"] == PRIMARY_SCOPE_ID].copy()
primary_metrics = primary_metrics.sort_values(
    [
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "ComplexityOrder",
        "CandidateID",
    ],
    kind="mergesort",
).reset_index(drop=True)
primary_metrics["PrimaryScopeRank"] = np.arange(1, len(primary_metrics) + 1)

benchmark_row = primary_metrics.loc[
    primary_metrics["CandidateID"] == LOCKED_BENCHMARK_METHOD
]
if len(benchmark_row) != 1:
    raise AssertionError("Could not identify one locked benchmark metric row")
benchmark_row = benchmark_row.iloc[0]

challenger_rows = primary_metrics.loc[
    primary_metrics["CandidateID"] != LOCKED_BENCHMARK_METHOD
].copy()
if challenger_rows.empty:
    raise AssertionError("No completed challenger metric rows")
best_challenger = challenger_rows.iloc[0]

comparison = challenger_rows.copy()
comparison["BenchmarkMethodID"] = LOCKED_BENCHMARK_METHOD
comparison["BenchmarkWAPEPercentage"] = float(benchmark_row["WAPEPercentage"])
comparison["WAPEImprovementPercentagePoints"] = (
    comparison["BenchmarkWAPEPercentage"] - comparison["WAPEPercentage"]
)
comparison["RelativeWAPEImprovementPercentage"] = (
    100.0
    * comparison["WAPEImprovementPercentagePoints"]
    / comparison["BenchmarkWAPEPercentage"]
)
comparison["BenchmarkMAE"] = float(benchmark_row["MAE"])
comparison["MAEImprovement"] = comparison["BenchmarkMAE"] - comparison["MAE"]
comparison["BenchmarkRMSE"] = float(benchmark_row["RMSE"])
comparison["RMSEImprovement"] = comparison["BenchmarkRMSE"] - comparison["RMSE"]
comparison["BenchmarkTotalBias"] = float(benchmark_row["TotalBias"])
comparison["BeatBenchmarkOnWAPE"] = comparison["WAPEPercentage"] < comparison["BenchmarkWAPEPercentage"]
comparison["BeatBenchmarkOnMAE"] = comparison["MAE"] < comparison["BenchmarkMAE"]
comparison["BeatBenchmarkOnRMSE"] = comparison["RMSE"] < comparison["BenchmarkRMSE"]
comparison["CandidateReadyForNestedTuning"] = comparison["BeatBenchmarkOnWAPE"]

ranking = pooled_metrics.copy()
ranking["IsLockedPart11IBenchmark"] = ranking["CandidateID"] == LOCKED_BENCHMARK_METHOD
ranking["ProceedTo11IA2"] = False

# Advance up to the three best completed challengers, prioritising any that beat
# the benchmark on primary-scope WAPE. The next step performs controlled tuning
# and daily-to-weekly comparison; this step does not make the final replacement.
advance_pool = comparison.sort_values(
    [
        "BeatBenchmarkOnWAPE",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
    ],
    ascending=[False, True, True, True, True],
    kind="mergesort",
).head(3)
advance_ids = set(advance_pool["CandidateID"].astype(str))
ranking.loc[ranking["CandidateID"].isin(advance_ids), "ProceedTo11IA2"] = True

best_challenger_beats_benchmark = bool(
    best_challenger["WAPEPercentage"] < benchmark_row["WAPEPercentage"]
)

# =============================================================================
# VALIDATION
# =============================================================================

evaluation_feature_rows = int(
    features["WeekStartDate"].isin(forecast_weeks).sum()
)
expected_candidate_prediction_rows = (
    evaluation_feature_rows * len(completed_candidate_ids)
)
actual_candidate_prediction_rows = len(candidate_predictions)
expected_total_metric_groups = (len(completed_candidate_ids) + 1) * len(SCOPE_FLAGS)
expected_fold_metric_groups = (
    (len(completed_candidate_ids) + 1) * len(SCOPE_FLAGS) * len(forecast_weeks)
)

nonfinite_predictions = int(
    (~np.isfinite(candidate_predictions["Prediction"].to_numpy(dtype=float))).sum()
)
negative_predictions = int((candidate_predictions["Prediction"] < 0).sum())

validation = pd.DataFrame(
    [
        {
            "Check": "Part 11I lock verified",
            "Expected": True,
            "Actual": True,
            "Passed": True,
        },
        {
            "Check": "Part 11I target remains normal demand",
            "Expected": TARGET_ID,
            "Actual": locked_design.get("TargetID"),
            "Passed": locked_design.get("TargetID") == TARGET_ID,
        },
        {
            "Check": "Part 11I primary scope remains strict 95 percent",
            "Expected": PRIMARY_SCOPE_ID,
            "Actual": locked_design.get("ScopeID"),
            "Passed": locked_design.get("ScopeID") == PRIMARY_SCOPE_ID,
        },
        {
            "Check": "Bulk included in model target",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Pre-holdout feature weeks",
            "Expected": 47,
            "Actual": len(weeks),
            "Passed": len(weeks) == 47,
        },
        {
            "Check": "Chronological forecast weeks",
            "Expected": 39,
            "Actual": len(forecast_weeks),
            "Passed": len(forecast_weeks) == 39,
        },
        {
            "Check": "Minimum training weeks",
            "Expected": MINIMUM_TRAINING_WEEKS,
            "Actual": int(fold_design["TrainingWeeks"].min()),
            "Passed": int(fold_design["TrainingWeeks"].min()) == MINIMUM_TRAINING_WEEKS,
        },
        {
            "Check": "Completed challenger models",
            "Expected": ">= 1",
            "Actual": len(completed_candidate_ids),
            "Passed": len(completed_candidate_ids) >= 1,
        },
        {
            "Check": "Candidate prediction rows",
            "Expected": expected_candidate_prediction_rows,
            "Actual": actual_candidate_prediction_rows,
            "Passed": actual_candidate_prediction_rows == expected_candidate_prediction_rows,
        },
        {
            "Check": "Pooled metric groups",
            "Expected": expected_total_metric_groups,
            "Actual": len(pooled_metrics),
            "Passed": len(pooled_metrics) == expected_total_metric_groups,
        },
        {
            "Check": "Fold metric groups",
            "Expected": expected_fold_metric_groups,
            "Actual": len(fold_metrics),
            "Passed": len(fold_metrics) == expected_fold_metric_groups,
        },
        {
            "Check": "Non-finite challenger predictions",
            "Expected": 0,
            "Actual": nonfinite_predictions,
            "Passed": nonfinite_predictions == 0,
        },
        {
            "Check": "Negative challenger predictions",
            "Expected": 0,
            "Actual": negative_predictions,
            "Passed": negative_predictions == 0,
        },
        {
            "Check": "Product ID used by expanded candidates",
            "Expected": True,
            "Actual": bool(candidate_predictions["ProductIDUsedAsDirectPredictor"].all()),
            "Passed": bool(candidate_predictions["ProductIDUsedAsDirectPredictor"].all()),
        },
        {
            "Check": "Opened March 2026 targets read",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Existing Part 11I lock modified",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Final replacement method selected in Part 11IA1",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
    ]
)

if not validation["Passed"].all():
    raise AssertionError(
        "Part 11IA1 validation failed:\n"
        + validation.loc[~validation["Passed"]].to_string(index=False)
    )

# =============================================================================
# CONTRACT, MEMORY, CHECKPOINT AND LOCK
# =============================================================================

contract = {
    "StepID": STEP_ID,
    "CreatedUTC": NOW_UTC.isoformat(),
    "CreatedLocal": NOW_LOCAL.isoformat(),
    "Status": STATUS,
    "Purpose": (
        "Challenge the locked Part 11I direct-weekly benchmark with stronger "
        "tree-based, count-aware and categorical-aware models under the same "
        "39-fold chronological pre-holdout design."
    ),
    "LockedBenchmark": {
        "TargetID": TARGET_ID,
        "MethodID": LOCKED_BENCHMARK_METHOD,
        "ScopeID": PRIMARY_SCOPE_ID,
        "Part11ILockSHA256": part11i_lock_hash,
    },
    "FeatureDesign": {
        "FeatureSource": str(PART11F_FEATURES),
        "FeatureSourceSHA256": sha256_file(PART11F_FEATURES),
        "PredictorDiscovery": predictor_source,
        "PredictorContractPath": predictor_contract_path,
        "NumericPredictorCount": len(numeric_predictor_columns),
        "DirectCategoricalPredictors": categorical_columns,
        "EntirelyEmptyPredictorsRemoved": entirely_empty_predictors,
        "ProductIDEncoding": "ONE_HOT_WITHIN_EACH_TRAINING_FOLD_OR_NATIVE_CATBOOST",
    },
    "Evaluation": {
        "TrainingDesign": "EXPANDING_WINDOW",
        "MinimumTrainingWeeks": MINIMUM_TRAINING_WEEKS,
        "ForecastWeeks": len(forecast_weeks),
        "PrimaryScopeID": PRIMARY_SCOPE_ID,
        "PrimaryMetric": "WAPEPercentage",
        "OtherMetrics": ["MAE", "RMSE", "MeanBias", "TotalBias"],
        "PredictionsRoundedBeforeScoring": False,
    },
    "CandidateModels": [
        {
            "CandidateID": spec["CandidateID"],
            "CandidateFamily": spec["CandidateFamily"],
            "Package": spec["Package"],
            "UsesProductID": spec["UsesProductID"],
            "CountAware": spec["CountAware"],
            "Configuration": spec["Configuration"],
        }
        for spec in candidate_specs
    ],
    "CompletedCandidateIDs": completed_candidate_ids,
    "FailedCandidateIDs": candidate_audit.loc[
        ~candidate_audit["CompletedAllFolds"], "CandidateID"
    ].astype(str).tolist(),
    "BestExpandedChallenger": {
        "CandidateID": str(best_challenger["CandidateID"]),
        "WAPEPercentage": float(best_challenger["WAPEPercentage"]),
        "MAE": float(best_challenger["MAE"]),
        "RMSE": float(best_challenger["RMSE"]),
        "TotalBias": float(best_challenger["TotalBias"]),
        "BeatLockedBenchmarkOnWAPE": best_challenger_beats_benchmark,
    },
    "LockedBenchmarkPerformance": {
        "WAPEPercentage": float(benchmark_row["WAPEPercentage"]),
        "MAE": float(benchmark_row["MAE"]),
        "RMSE": float(benchmark_row["RMSE"]),
        "TotalBias": float(benchmark_row["TotalBias"]),
    },
    "ProceedingTo11IA2": sorted(advance_ids),
    "Governance": {
        "Part11ILockModified": False,
        "FinalReplacementSelected": False,
        "OpenedMarch2026TargetsRead": False,
        "BulkIncludedInModelTarget": False,
        "NextStep": (
            "Part 11IA2: controlled tuning of the strongest challengers and "
            "same-fold daily-to-weekly architecture comparison."
        ),
    },
}

stage_root = EXT_ROOT / f".11IA1_staging_{uuid.uuid4().hex}"
stage_root.mkdir(parents=True, exist_ok=False)

try:
    for path, frame in [
        (PACKAGE_AUDIT_PATH, package_audit),
        (FEATURE_AUDIT_PATH, feature_audit),
        (FOLD_DESIGN_PATH, fold_design),
        (CANDIDATE_AUDIT_PATH, candidate_audit),
        (PREDICTIONS_PATH, candidate_predictions),
        (POOLED_METRICS_PATH, pooled_metrics),
        (FOLD_METRICS_PATH, fold_metrics),
        (RANKING_PATH, ranking),
        (COMPARISON_PATH, comparison),
        (VALIDATION_PATH, validation),
    ]:
        stage_csv(stage_root, path, frame)

    stage_json(stage_root, CONTRACT_PATH, contract)

    decision_body = "\n".join(
        [
            "- Part 11I remains the locked benchmark and was not overwritten.",
            f"- Expanded weekly candidates were evaluated for {len(forecast_weeks)} chronological folds.",
            "- WeeklyNormalDemand remains the target; bulk preorders remain excluded.",
            "- CanonicalProductID was introduced as a direct predictor using fold-safe encoding.",
            f"- Best expanded challenger: {best_challenger['CandidateID']}.",
            f"- Best challenger WAPE at the 95% scope: {float(best_challenger['WAPEPercentage']):.6f}%.",
            f"- Locked benchmark WAPE at the 95% scope: {float(benchmark_row['WAPEPercentage']):.6f}%.",
            f"- Best challenger beat benchmark on WAPE: {best_challenger_beats_benchmark}.",
            f"- Candidates proceeding to 11IA2: {', '.join(sorted(advance_ids))}.",
            "- No final replacement decision was made in this screening step.",
        ]
    )

    files_body = "\n".join(
        [
            f"- Package audit: `{PACKAGE_AUDIT_PATH}`",
            f"- Feature audit: `{FEATURE_AUDIT_PATH}`",
            f"- Candidate audit: `{CANDIDATE_AUDIT_PATH}`",
            f"- Candidate predictions: `{PREDICTIONS_PATH}`",
            f"- Pooled metrics: `{POOLED_METRICS_PATH}`",
            f"- Challenger comparison: `{COMPARISON_PATH}`",
            f"- Challenge contract: `{CONTRACT_PATH}`",
        ]
    )

    for key, final_path in MEMORY_FILES.items():
        original = final_path.read_text(encoding="utf-8")

        if key == "PROJECT_CONTEXT":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 expanded weekly model challenge",
                (
                    "A separate challenger stage was added after Part 11I to test "
                    "stronger direct-weekly models, including random forests, extra "
                    "trees, count-aware boosting and any installed XGBoost, LightGBM "
                    "or CatBoost implementations. The Part 11I lock remains intact."
                ),
            )
        elif key == "WORKFLOW":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 workflow status",
                (
                    f"Part 11IA1 completed with status {STATUS}. Part 11IA2 should "
                    "tune the strongest challengers and compare them against a "
                    "leakage-safe daily-to-weekly architecture on the same folds."
                ),
            )
        elif key == "DECISIONS":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 decisions",
                decision_body,
            )
        elif key == "FILES_AND_PATHS":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 files",
                files_body,
            )
        elif key == "METRICS_AND_RESULTS":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 results",
                decision_body,
            )
        elif key == "CHAT_INDEX":
            row = (
                f"| {NOW_LOCAL.isoformat()} | 11IA1 | Expanded weekly model challenge | "
                f"{STATUS} |"
            )
            updated = original if row in original else original.rstrip() + "\n" + row + "\n"
        elif key == "CURRENT_HANDOFF":
            updated = "\n".join(
                [
                    "# Current Handoff",
                    "",
                    f"- Current completed step: {STEP_ID}",
                    f"- Status: {STATUS}",
                    f"- Updated local time: {NOW_LOCAL.isoformat()}",
                    f"- Locked benchmark remains: {LOCKED_BENCHMARK_METHOD}",
                    f"- Primary scope remains: {PRIMARY_SCOPE_ID}",
                    f"- Best expanded challenger: {best_challenger['CandidateID']}",
                    f"- Best challenger beat benchmark: {best_challenger_beats_benchmark}",
                    f"- Candidates proceeding: {', '.join(sorted(advance_ids))}",
                    "- Part 11I lock was not changed.",
                    "- Next step: 11IA2 controlled tuning and daily-to-weekly comparison.",
                    "",
                ]
            )
        else:
            raise KeyError(key)

        stage_text(stage_root, final_path, updated)

    step_text = f"""# Step 11IA1 — Expanded Weekly Model Challenge

- **Step ID:** {STEP_ID}
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## Purpose
Challenge the provisional Part 11I weekly benchmark with stronger direct-weekly models while preserving the same target, chronological folds, dynamic scopes, bulk policy and leakage controls.

## Models evaluated
{candidate_audit[['CandidateID', 'CandidateFamily', 'CompletedAllFolds', 'FailureMessage']].to_markdown(index=False)}

## Primary-scope result
- Locked benchmark: {LOCKED_BENCHMARK_METHOD}
- Locked benchmark WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%
- Best expanded challenger: {best_challenger['CandidateID']}
- Best challenger WAPE: {float(best_challenger['WAPEPercentage']):.6f}%
- Best challenger beat benchmark: {best_challenger_beats_benchmark}
- Candidates proceeding to 11IA2: {', '.join(sorted(advance_ids))}

## Governance
- Part 11I was not overwritten.
- March 2026 targets were not read.
- Bulk was not included in the modelled target.
- Product identity was encoded inside each training fold.
- This was a fixed-configuration screening step; it did not make a final replacement decision.

## Outputs
{files_body}

## Next step
Part 11IA2: controlled tuning of the strongest challengers and direct comparison with leakage-safe daily-to-weekly forecasting on the same 39 folds.
"""
    stage_text(stage_root, STEP_MEMORY_PATH, step_text)

    checkpoint = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "Part11ILockSHA256": part11i_lock_hash,
        "FeatureSourceSHA256": sha256_file(PART11F_FEATURES),
        "BaselinePredictionsSHA256": sha256_file(PART11E_PREDICTIONS),
        "CompletedCandidateIDs": completed_candidate_ids,
        "ProceedingTo11IA2": sorted(advance_ids),
        "BestExpandedChallenger": contract["BestExpandedChallenger"],
        "LockedBenchmarkPerformance": contract["LockedBenchmarkPerformance"],
        "Safety": {
            "Part11ILockModified": False,
            "OpenedMarch2026TargetsRead": False,
            "BulkIncludedInModelTarget": False,
            "ProductIDEncodedWithinTrainingFold": True,
            "ForecastsRoundedBeforeScoring": False,
            "FinalReplacementSelected": False,
        },
        "ReadyForPart11IA2": True,
    }
    stage_json(stage_root, CHECKPOINT_PATH, checkpoint)
    checkpoint_hash = sha256_file(stage_path(stage_root, CHECKPOINT_PATH))
    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        f"{checkpoint_hash}  {CHECKPOINT_PATH.name}\n",
    )

    manifest_targets = [
        PACKAGE_AUDIT_PATH,
        FEATURE_AUDIT_PATH,
        FOLD_DESIGN_PATH,
        CANDIDATE_AUDIT_PATH,
        PREDICTIONS_PATH,
        POOLED_METRICS_PATH,
        FOLD_METRICS_PATH,
        RANKING_PATH,
        COMPARISON_PATH,
        VALIDATION_PATH,
        CONTRACT_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]
    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(EXT_ROOT)),
                "Bytes": stage_path(stage_root, path).stat().st_size,
                "SHA256": sha256_file(stage_path(stage_root, path)),
            }
            for path in manifest_targets
        ]
    ).sort_values("RelativePath")
    stage_csv(stage_root, MANIFEST_PATH, manifest)
    manifest_hash = sha256_file(stage_path(stage_root, MANIFEST_PATH))

    lock = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "Part11ILockSHA256": part11i_lock_hash,
        "Contract": {
            "Path": str(CONTRACT_PATH),
            "SHA256": sha256_file(stage_path(stage_root, CONTRACT_PATH)),
        },
        "Checkpoint": {
            "Path": str(CHECKPOINT_PATH),
            "SHA256": checkpoint_hash,
        },
        "OutputHashManifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_hash,
            "FilesListed": int(len(manifest)),
        },
        "BestExpandedChallenger": contract["BestExpandedChallenger"],
        "ProceedingTo11IA2": sorted(advance_ids),
        "SafetyAssertions": checkpoint["Safety"],
        "ReadyForPart11IA2": True,
        "NextStep": "11IA2",
    }
    stage_json(stage_root, LOCK_PATH, lock)
    lock_hash = sha256_file(stage_path(stage_root, LOCK_PATH))
    stage_text(stage_root, LOCK_SHA_PATH, f"{lock_hash}  {LOCK_PATH.name}\n")

    log_text = "\n".join(
        [
            f"Status: {STATUS}",
            f"Run local: {NOW_LOCAL.isoformat()}",
            f"Predictor source: {predictor_source}",
            f"Numeric predictors: {len(numeric_predictor_columns)}",
            f"Completed candidates: {', '.join(completed_candidate_ids)}",
            f"Best challenger: {best_challenger['CandidateID']}",
            f"Best challenger WAPE: {float(best_challenger['WAPEPercentage']):.12f}",
            f"Benchmark WAPE: {float(benchmark_row['WAPEPercentage']):.12f}",
            f"Best challenger beat benchmark: {best_challenger_beats_benchmark}",
            f"Proceeding: {', '.join(sorted(advance_ids))}",
            f"Checkpoint SHA256: {checkpoint_hash}",
            f"Lock SHA256: {lock_hash}",
            "",
        ]
    )
    stage_text(stage_root, LOG_PATH, log_text)

    staged_files = [path for path in stage_root.rglob("*") if path.is_file()]
    if not staged_files:
        raise AssertionError("Part 11IA1 staging directory is empty")

    for staged in sorted(staged_files):
        final = EXT_ROOT / staged.relative_to(stage_root)
        final.parent.mkdir(parents=True, exist_ok=True)
        if final.exists() and final not in MEMORY_FILES.values():
            raise FileExistsError(f"Refusing to overwrite Part 11IA1 output: {final}")
        os.replace(staged, final)

    for path in NEW_OUTPUTS:
        make_read_only(path)

finally:
    if stage_root.exists():
        shutil.rmtree(stage_root)

# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("=" * 100)
print("EDEN WEEKLY FORECASTING EXTENSION — PART 11IA1 COMPLETE")
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Extension root: {EXT_ROOT}")
print(f"Local time: {NOW_LOCAL.isoformat()}")

print("\nPACKAGE AVAILABILITY")
print(
    package_audit[
        ["Distribution", "Version", "Installed", "ImportSucceeded", "ImportError"]
    ].to_string(index=False)
)

print("\nFEATURE DESIGN")
print(f"Predictor source: {predictor_source}")
print(f"Predictor contract: {predictor_contract_path}")
print(f"Numeric predictors used: {len(numeric_predictor_columns)}")
print(f"Direct categorical predictors: {categorical_columns}")
print(f"Entirely empty predictors removed: {entirely_empty_predictors}")

print("\nCANDIDATE EXECUTION AUDIT")
print(
    candidate_audit[
        [
            "CandidateID",
            "CandidateFamily",
            "Package",
            "CountAware",
            "SuccessfulFolds",
            "CompletedAllFolds",
            "FailureMessage",
        ]
    ].to_string(index=False)
)

print("\nPRIMARY 95% SCOPE RANKING")
print(
    primary_metrics[
        [
            "PrimaryScopeRank",
            "CandidateID",
            "CandidateFamily",
            "ForecastWeeks",
            "Observations",
            "ActualTotal",
            "PredictedTotal",
            "MAE",
            "RMSE",
            "WAPEPercentage",
            "MeanBias",
            "TotalBias",
            "ProductIDUsedAsDirectPredictor",
        ]
    ].to_string(index=False)
)

print("\nCHALLENGER VERSUS LOCKED BENCHMARK")
print(
    comparison[
        [
            "CandidateID",
            "WAPEPercentage",
            "BenchmarkWAPEPercentage",
            "WAPEImprovementPercentagePoints",
            "RelativeWAPEImprovementPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "BeatBenchmarkOnWAPE",
            "CandidateReadyForNestedTuning",
        ]
    ].to_string(index=False)
)

print("\nPART 11IA1 DECISION")
print(f"Locked benchmark retained without modification: {LOCKED_BENCHMARK_METHOD}")
print(f"Best expanded challenger: {best_challenger['CandidateID']}")
print(f"Best challenger WAPE: {float(best_challenger['WAPEPercentage']):.6f}%")
print(f"Locked benchmark WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%")
print(f"Best challenger beat benchmark: {best_challenger_beats_benchmark}")
print(f"Proceeding to Part 11IA2: {', '.join(sorted(advance_ids))}")
print("Final replacement selected in Part 11IA1: False")

print("\nPART 11IA1 VALIDATION")
print(validation.to_string(index=False))

print("\nCONTROL OUTPUTS")
print(f"- Package audit: {PACKAGE_AUDIT_PATH}")
print(f"- Candidate audit: {CANDIDATE_AUDIT_PATH}")
print(f"- Candidate predictions: {PREDICTIONS_PATH}")
print(f"- Pooled metrics: {POOLED_METRICS_PATH}")
print(f"- Challenger comparison: {COMPARISON_PATH}")
print(f"- Challenge contract: {CONTRACT_PATH}")
print(f"- Checkpoint: {CHECKPOINT_PATH}")
print(f"- Checkpoint SHA256: {checkpoint_hash}")
print(f"- Part 11IA1 lock: {LOCK_PATH}")
print(f"- Part 11IA1 lock SHA256: {lock_hash}")

print(
    "\nSAFETY: Part 11I lock modified False; opened March 2026 targets read False; "
    "bulk included in model target False; product identity encoded within each "
    "training fold True; forecasts rounded before scoring False; final replacement "
    "selected False."
)
print("=" * 100)

----------------------------------------------------------------------------------------------------
Running candidate: HISTGB_POISSON
----------------------------------------------------------------------------------------------------
Running candidate: RANDOM_FOREST_SQUARED
----------------------------------------------------------------------------------------------------
Running candidate: RANDOM_FOREST_POISSON
----------------------------------------------------------------------------------------------------
Running candidate: EXTRA_TREES_SQUARED
----------------------------------------------------------------------------------------------------
Running candidate: EXTRA_TREES_POISSON
----------------------------------------------------------------------------------------------------
Running candidate: XGBOOST_SQUARED
----------------------------------------------------------------------------------------------------
Running candidate: XGBOOST_POISSON
-----------------------------

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.

In [28]:
from __future__ import annotations

import hashlib
import importlib
import importlib.metadata
import inspect
import json
import os
import shutil
import stat
import sys
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11IA1
# Expanded direct-weekly model challenge
#
# Purpose
# -------
# Challenge the locked Part 11I weekly benchmark with stronger nonlinear,
# count-aware and categorical-aware models under the same leakage-safe,
# chronological pre-holdout evaluation design.
#
# This step DOES NOT replace or modify the Part 11I lock. It creates a separate
# challenger audit. Final replacement, if justified, must occur in a later
# controlled amendment after tuning and daily-to-weekly comparison.
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"

FEATURE_ROOT = EXT_ROOT / "04_weekly_features"
BASELINE_ROOT = EXT_ROOT / "05_weekly_baselines"
MODEL_ROOT = EXT_ROOT / "06_weekly_models"
VALIDATION_ROOT = EXT_ROOT / "07_weekly_validation"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
LOG_ROOT = EXT_ROOT / "12_logs"

PART11F_FEATURES = FEATURE_ROOT / "11F_model_selection_weekly_features.csv"
PART11E_PREDICTIONS = BASELINE_ROOT / "11E_preholdout_weekly_baseline_predictions.csv"
PART11I_DESIGN = MODEL_ROOT / "final" / "11I_final_weekly_forecasting_design.json"
PART11I_LOCK = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.json"
PART11I_LOCK_SHA = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.sha256"

OUTPUT_DIR = VALIDATION_ROOT / "11IA1_expanded_weekly_challenge"
PACKAGE_AUDIT_PATH = OUTPUT_DIR / "11IA1_package_audit.csv"
FEATURE_AUDIT_PATH = OUTPUT_DIR / "11IA1_feature_audit.csv"
FOLD_DESIGN_PATH = OUTPUT_DIR / "11IA1_chronological_fold_design.csv"
CANDIDATE_AUDIT_PATH = OUTPUT_DIR / "11IA1_candidate_audit.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_predictions.csv"
POOLED_METRICS_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_pooled_metrics.csv"
FOLD_METRICS_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_fold_metrics.csv"
RANKING_PATH = OUTPUT_DIR / "11IA1_expanded_candidate_ranking.csv"
COMPARISON_PATH = OUTPUT_DIR / "11IA1_challenger_vs_locked_benchmark.csv"
VALIDATION_PATH = OUTPUT_DIR / "11IA1_validation.csv"
CONTRACT_PATH = OUTPUT_DIR / "11IA1_expanded_weekly_challenge_contract.json"
MANIFEST_PATH = OUTPUT_DIR / "11IA1_output_hash_manifest.csv"
STEP_MEMORY_PATH = STEP_MEMORY_ROOT / "STEP_11IA1_EXPANDED_WEEKLY_MODEL_CHALLENGE.md"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "11IA1_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "11IA1_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "11IA1_expanded_weekly_challenge_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "11IA1_expanded_weekly_challenge_lock.sha256"
LOG_PATH = LOG_ROOT / "11IA1_expanded_weekly_challenge_log.txt"

STEP_ID = "11IA1"
STATUS = "PART_11IA1_COMPLETED_READY_FOR_11IA2"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

TARGET_ID = "WEEKLY_NORMAL_DEMAND"
TARGET_COLUMN = "WeeklyNormalDemand"
LOCKED_BENCHMARK_METHOD = "NAIVE_LAST_CONTEXT"
PRIMARY_SCOPE_ID = "FOLD_STRICT_95_PERCENT"
MINIMUM_TRAINING_WEEKS = 8
RANDOM_SEED = 6073

SCOPE_FLAGS = {
    "ALL_PRODUCTS": None,
    "FOLD_STRICT_80_PERCENT": "PriorStrict80PctScope",
    "FOLD_STRICT_90_PERCENT": "PriorStrict90PctScope",
    "FOLD_STRICT_95_PERCENT": "PriorStrict95PctScope",
}

BASELINE_SCOPE_ALIASES = {
    "InFoldStrict80PctScope": "PriorStrict80PctScope",
    "InFoldStrict90PctScope": "PriorStrict90PctScope",
    "InFoldStrict95PctScope": "PriorStrict95PctScope",
}

CANONICAL_SCOPE_COLUMNS = [
    "PriorStrict80PctScope",
    "PriorStrict90PctScope",
    "PriorStrict95PctScope",
]

MEMORY_FILES = {
    "PROJECT_CONTEXT": MEMORY_ROOT / "PROJECT_CONTEXT.md",
    "WORKFLOW": MEMORY_ROOT / "WORKFLOW.md",
    "DECISIONS": MEMORY_ROOT / "DECISIONS.md",
    "FILES_AND_PATHS": MEMORY_ROOT / "FILES_AND_PATHS.md",
    "METRICS_AND_RESULTS": MEMORY_ROOT / "METRICS_AND_RESULTS.md",
    "CHAT_INDEX": MEMORY_ROOT / "CHAT_INDEX.md",
    "CURRENT_HANDOFF": MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

NEW_OUTPUTS = [
    PACKAGE_AUDIT_PATH,
    FEATURE_AUDIT_PATH,
    FOLD_DESIGN_PATH,
    CANDIDATE_AUDIT_PATH,
    PREDICTIONS_PATH,
    POOLED_METRICS_PATH,
    FOLD_METRICS_PATH,
    RANKING_PATH,
    COMPARISON_PATH,
    VALIDATION_PATH,
    CONTRACT_PATH,
    MANIFEST_PATH,
    STEP_MEMORY_PATH,
    CHECKPOINT_PATH,
    CHECKPOINT_SHA_PATH,
    LOCK_PATH,
    LOCK_SHA_PATH,
    LOG_PATH,
]

# =============================================================================
# GENERAL HELPERS
# =============================================================================


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def verify_sidecar(path: Path, sidecar: Path, label: str) -> str:
    require_file(path, label)
    require_file(sidecar, f"{label} SHA-256 sidecar")
    expected = sidecar.read_text(encoding="utf-8").strip().split()[0].lower()
    actual = sha256_file(path)
    if expected != actual:
        raise AssertionError(
            f"{label} SHA-256 mismatch:\nExpected: {expected}\nActual:   {actual}"
        )
    return actual


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_json(path: Path, payload: dict) -> None:
    atomic_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        frame.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def stage_path(stage_root: Path, final_path: Path) -> Path:
    return stage_root / final_path.relative_to(EXT_ROOT)


def stage_text(stage_root: Path, final_path: Path, text: str) -> None:
    atomic_text(stage_path(stage_root, final_path), text)


def stage_json(stage_root: Path, final_path: Path, payload: dict) -> None:
    atomic_json(stage_path(stage_root, final_path), payload)


def stage_csv(stage_root: Path, final_path: Path, frame: pd.DataFrame) -> None:
    atomic_csv(stage_path(stage_root, final_path), frame)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def bool_series(series: pd.Series, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    normalized = series.astype(str).str.strip().str.lower()
    invalid = sorted(set(normalized) - set(mapping))
    if invalid:
        raise ValueError(f"{label} has invalid Boolean values: {invalid[:10]}")
    return normalized.map(mapping).astype(bool)


def normalize_scope_columns(
    frame: pd.DataFrame,
    frame_label: str,
    aliases: dict[str, str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    aliases = aliases or {}
    rename_map: dict[str, str] = {}

    for source, target in aliases.items():
        if source in result.columns and target in result.columns:
            left = bool_series(result[source], f"{frame_label}.{source}")
            right = bool_series(result[target], f"{frame_label}.{target}")
            if not left.equals(right):
                raise AssertionError(
                    f"{frame_label} contains conflicting {source} and {target}"
                )
            result = result.drop(columns=[source])
        elif source in result.columns:
            rename_map[source] = target

    if rename_map:
        result = result.rename(columns=rename_map)

    missing = [c for c in CANONICAL_SCOPE_COLUMNS if c not in result.columns]
    if missing:
        raise AssertionError(
            f"{frame_label} is missing normalized scope columns: {missing}"
        )

    for column in CANONICAL_SCOPE_COLUMNS:
        result[column] = bool_series(result[column], f"{frame_label}.{column}")

    return result


def metric_record(actual: pd.Series, prediction: pd.Series) -> dict:
    actual_array = pd.to_numeric(actual, errors="raise").to_numpy(dtype=float)
    prediction_array = pd.to_numeric(prediction, errors="raise").to_numpy(dtype=float)

    if len(actual_array) != len(prediction_array):
        raise AssertionError("Actual and prediction lengths differ")
    if not np.isfinite(actual_array).all() or not np.isfinite(prediction_array).all():
        raise AssertionError("Non-finite values entered metric calculation")

    errors = prediction_array - actual_array
    absolute_errors = np.abs(errors)
    denominator = float(np.abs(actual_array).sum())

    return {
        "Observations": int(len(actual_array)),
        "ActualTotal": float(actual_array.sum()),
        "PredictedTotal": float(prediction_array.sum()),
        "MAE": float(absolute_errors.mean()) if len(actual_array) else np.nan,
        "RMSE": float(np.sqrt(np.mean(errors ** 2))) if len(actual_array) else np.nan,
        "WAPEPercentage": (
            float(100.0 * absolute_errors.sum() / denominator)
            if denominator != 0
            else np.nan
        ),
        "MeanBias": float(errors.mean()) if len(actual_array) else np.nan,
        "TotalBias": float(errors.sum()),
    }


def update_section(text: str, marker: str, heading: str, body: str) -> str:
    start = f"<!-- BEGIN {marker} -->"
    end = f"<!-- END {marker} -->"
    section = f"{start}\n## {heading}\n\n{body.rstrip()}\n{end}"

    if start in text and end in text:
        before = text.split(start, 1)[0].rstrip()
        after = text.split(end, 1)[1].lstrip()
        return before + "\n\n" + section + ("\n\n" + after if after else "") + "\n"

    return text.rstrip() + "\n\n" + section + "\n"


def package_version(distribution_name: str) -> str | None:
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return None


def make_ohe() -> OneHotEncoder:
    parameters = inspect.signature(OneHotEncoder).parameters
    kwargs = {"handle_unknown": "ignore"}
    if "sparse_output" in parameters:
        kwargs["sparse_output"] = False
    else:
        kwargs["sparse"] = False
    return OneHotEncoder(**kwargs)


def make_imputer() -> SimpleImputer:
    parameters = inspect.signature(SimpleImputer).parameters
    kwargs = {"strategy": "median"}
    if "keep_empty_features" in parameters:
        kwargs["keep_empty_features"] = True
    return SimpleImputer(**kwargs)


def candidate_complexity_order(candidate_id: str) -> int:
    order = {
        LOCKED_BENCHMARK_METHOD: 0,
        "HISTGB_POISSON": 10,
        "RANDOM_FOREST_SQUARED": 20,
        "RANDOM_FOREST_POISSON": 21,
        "EXTRA_TREES_SQUARED": 30,
        "EXTRA_TREES_POISSON": 31,
        "XGBOOST_SQUARED": 40,
        "XGBOOST_POISSON": 41,
        "LIGHTGBM_TWEEDIE": 50,
        "CATBOOST_RMSE": 60,
    }
    return order.get(candidate_id, 999)


def recursively_find_predictor_lists(obj, path: tuple[str, ...] = ()) -> list[tuple[str, list[str]]]:
    found: list[tuple[str, list[str]]] = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            found.extend(recursively_find_predictor_lists(value, path + (str(key),)))
    elif isinstance(obj, list) and obj and all(isinstance(item, str) for item in obj):
        joined = ".".join(path).lower()
        if "predict" in joined and "normal" in joined:
            found.append((".".join(path), obj))
    return found


def discover_normal_predictors(frame: pd.DataFrame) -> tuple[list[str], str, str | None]:
    contract_candidates = sorted(FEATURE_ROOT.glob("*11F*contract*.json"))
    contract_candidates += sorted(FEATURE_ROOT.glob("**/*11F*contract*.json"))
    contract_candidates = list(dict.fromkeys(contract_candidates))

    for contract_path in contract_candidates:
        try:
            payload = json.loads(contract_path.read_text(encoding="utf-8"))
        except Exception:
            continue

        candidates = recursively_find_predictor_lists(payload)
        candidates = [
            (name, values)
            for name, values in candidates
            if all(value in frame.columns for value in values)
        ]
        if candidates:
            candidates.sort(key=lambda item: (abs(len(item[1]) - 48), item[0]))
            selected_name, selected_values = candidates[0]
            return list(selected_values), f"CONTRACT:{selected_name}", str(contract_path)

    explicit_exclusions = {
        "WeekStartDate",
        "WeekEndDate",
        "WeekID",
        "CanonicalProductID",
        "CanonicalProductName",
        "TargetID",
        TARGET_COLUMN,
        "WeeklyTotalDemand",
        "WeeklyBulkDemand",
        "BulkDemand",
        "NormalDemand",
        "TotalDemand",
        "Actual",
        "Prediction",
        "WeeklyTuningEligible",
        "IsOpenedMarchHoldout",
        *CANONICAL_SCOPE_COLUMNS,
    }

    forbidden_exact_or_prefix = (
        "CurrentWeek",
        "Future",
        "ObservedTarget",
        "Protected",
    )

    predictors: list[str] = []
    for column in frame.columns:
        if column in explicit_exclusions:
            continue
        if column.startswith(forbidden_exact_or_prefix):
            continue
        if "Scope" in column:
            continue
        if "Target" in column and not any(
            token in column for token in ["Lag", "Mean", "Median", "Sum", "Std", "Rate", "Count"]
        ):
            continue
        if pd.api.types.is_numeric_dtype(frame[column]) or pd.api.types.is_bool_dtype(frame[column]):
            predictors.append(column)

    if not predictors:
        raise AssertionError("No fallback numeric predictor columns were discovered")

    return predictors, "FALLBACK_NUMERIC_SCHEMA", None


# =============================================================================
# INPUT AND LOCK VERIFICATION
# =============================================================================

for directory, label in [
    (EXT_ROOT, "extension root"),
    (FEATURE_ROOT, "weekly feature directory"),
    (BASELINE_ROOT, "weekly baseline directory"),
    (MODEL_ROOT, "weekly model directory"),
    (VALIDATION_ROOT, "weekly validation directory"),
    (CHECKPOINT_ROOT, "checkpoint directory"),
    (MEMORY_ROOT, "project memory directory"),
    (STEP_MEMORY_ROOT, "step memory directory"),
    (LOG_ROOT, "log directory"),
]:
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing required {label}:\n{directory}")

for path, label in [
    (PART11F_FEATURES, "Part 11F model-selection features"),
    (PART11E_PREDICTIONS, "Part 11E baseline predictions"),
    (PART11I_DESIGN, "Part 11I final design"),
]:
    require_file(path, label)

for key, path in MEMORY_FILES.items():
    require_file(path, f"memory file {key}")

if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise FileExistsError(f"Part 11IA1 overwrite lock triggered:\n{LOCK_PATH}")

existing_outputs = [path for path in NEW_OUTPUTS if path.exists()]
if existing_outputs:
    raise FileExistsError(
        "Existing Part 11IA1 outputs found; no files changed:\n"
        + "\n".join(f"- {path}" for path in existing_outputs)
    )

for old_stage in EXT_ROOT.glob(".11IA1_staging_*"):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)

part11i_lock_hash = verify_sidecar(PART11I_LOCK, PART11I_LOCK_SHA, "Part 11I lock")
part11i_lock = json.loads(PART11I_LOCK.read_text(encoding="utf-8"))
part11i_design = json.loads(PART11I_DESIGN.read_text(encoding="utf-8"))

if part11i_lock.get("Status") != "PART_11I_COMPLETED_READY_FOR_11J":
    raise AssertionError(f"Unexpected Part 11I status: {part11i_lock.get('Status')}")

locked_design = part11i_lock.get("FinalForecastingDesign", {})
if locked_design.get("TargetID") != TARGET_ID:
    raise AssertionError("Part 11I target is not WEEKLY_NORMAL_DEMAND")
if locked_design.get("MethodID") != LOCKED_BENCHMARK_METHOD:
    raise AssertionError("Unexpected locked Part 11I benchmark method")
if locked_design.get("ScopeID") != PRIMARY_SCOPE_ID:
    raise AssertionError("Unexpected locked Part 11I primary scope")
if locked_design.get("BulkIncludedInModelledTarget") is not False:
    raise AssertionError("Part 11I did not exclude bulk from the model target")

for key, expected in {
    "OpenedMarch2026TargetsRead": False,
    "BulkIncludedInModelledTarget": False,
    "ModelsFittedOrRefitted": False,
    "FinalForecastDesignLocked": True,
    "ForecastsRoundedBeforeScoring": False,
}.items():
    actual = part11i_lock.get("SafetyAssertions", {}).get(key)
    if actual is not expected:
        raise AssertionError(f"Invalid Part 11I safety assertion {key}: {actual}")

if sha256_file(PART11I_DESIGN) != locked_design.get("SHA256"):
    raise AssertionError("Part 11I final design hash changed")

# =============================================================================
# PACKAGE AUDIT AND CANDIDATE REGISTRY
# =============================================================================

package_rows = []
for distribution, import_name, role in [
    ("numpy", "numpy", "CORE"),
    ("pandas", "pandas", "CORE"),
    ("scikit-learn", "sklearn", "CORE"),
    ("xgboost", "xgboost", "OPTIONAL_STRONG_MODEL"),
    ("lightgbm", "lightgbm", "OPTIONAL_STRONG_MODEL"),
    ("catboost", "catboost", "OPTIONAL_STRONG_MODEL"),
]:
    version = package_version(distribution)
    import_ok = False
    import_error = ""
    if version is not None:
        try:
            importlib.import_module(import_name)
            import_ok = True
        except Exception as exc:
            import_error = f"{type(exc).__name__}: {exc}"
    package_rows.append(
        {
            "Distribution": distribution,
            "ImportName": import_name,
            "Role": role,
            "Installed": version is not None,
            "Version": version or "",
            "ImportSucceeded": import_ok,
            "ImportError": import_error,
        }
    )

package_audit = pd.DataFrame(package_rows)
package_lookup = package_audit.set_index("ImportName")["ImportSucceeded"].to_dict()

candidate_specs: list[dict] = []

candidate_specs.append(
    {
        "CandidateID": "HISTGB_POISSON",
        "CandidateFamily": "HISTOGRAM_GRADIENT_BOOSTING",
        "Package": "scikit-learn",
        "UsesProductID": True,
        "CountAware": True,
        "Available": True,
        "Estimator": HistGradientBoostingRegressor(
            loss="poisson",
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=RANDOM_SEED,
        ),
        "SpecialRunner": None,
        "Configuration": {
            "loss": "poisson",
            "learning_rate": 0.05,
            "max_iter": 300,
            "max_leaf_nodes": 31,
            "min_samples_leaf": 20,
            "l2_regularization": 1.0,
        },
    }
)

for candidate_id, criterion, count_aware in [
    ("RANDOM_FOREST_SQUARED", "squared_error", False),
    ("RANDOM_FOREST_POISSON", "poisson", True),
]:
    candidate_specs.append(
        {
            "CandidateID": candidate_id,
            "CandidateFamily": "RANDOM_FOREST",
            "Package": "scikit-learn",
            "UsesProductID": True,
            "CountAware": count_aware,
            "Available": True,
            "Estimator": RandomForestRegressor(
                n_estimators=300,
                criterion=criterion,
                max_depth=None,
                min_samples_split=4,
                min_samples_leaf=2,
                max_features=0.70,
                bootstrap=True,
                n_jobs=-1,
                random_state=RANDOM_SEED,
            ),
            "SpecialRunner": None,
            "Configuration": {
                "n_estimators": 300,
                "criterion": criterion,
                "min_samples_leaf": 2,
                "max_features": 0.70,
                "bootstrap": True,
            },
        }
    )

for candidate_id, criterion, count_aware in [
    ("EXTRA_TREES_SQUARED", "squared_error", False),
    ("EXTRA_TREES_POISSON", "poisson", True),
]:
    candidate_specs.append(
        {
            "CandidateID": candidate_id,
            "CandidateFamily": "EXTRA_TREES",
            "Package": "scikit-learn",
            "UsesProductID": True,
            "CountAware": count_aware,
            "Available": True,
            "Estimator": ExtraTreesRegressor(
                n_estimators=300,
                criterion=criterion,
                max_depth=None,
                min_samples_split=4,
                min_samples_leaf=2,
                max_features=0.85,
                bootstrap=False,
                n_jobs=-1,
                random_state=RANDOM_SEED,
            ),
            "SpecialRunner": None,
            "Configuration": {
                "n_estimators": 300,
                "criterion": criterion,
                "min_samples_leaf": 2,
                "max_features": 0.85,
                "bootstrap": False,
            },
        }
    )

if package_lookup.get("xgboost", False):
    from xgboost import XGBRegressor

    for candidate_id, objective, count_aware in [
        ("XGBOOST_SQUARED", "reg:squarederror", False),
        ("XGBOOST_POISSON", "count:poisson", True),
    ]:
        candidate_specs.append(
            {
                "CandidateID": candidate_id,
                "CandidateFamily": "XGBOOST",
                "Package": "xgboost",
                "UsesProductID": True,
                "CountAware": count_aware,
                "Available": True,
                "Estimator": XGBRegressor(
                    objective=objective,
                    n_estimators=500,
                    learning_rate=0.03,
                    max_depth=6,
                    min_child_weight=5,
                    subsample=0.80,
                    colsample_bytree=0.80,
                    reg_alpha=0.0,
                    reg_lambda=2.0,
                    n_jobs=-1,
                    random_state=RANDOM_SEED,
                    tree_method="hist",
                    verbosity=0,
                ),
                "SpecialRunner": None,
                "Configuration": {
                    "objective": objective,
                    "n_estimators": 500,
                    "learning_rate": 0.03,
                    "max_depth": 6,
                    "min_child_weight": 5,
                    "subsample": 0.80,
                    "colsample_bytree": 0.80,
                    "reg_lambda": 2.0,
                },
            }
        )

if package_lookup.get("lightgbm", False):
    from lightgbm import LGBMRegressor

    candidate_specs.append(
        {
            "CandidateID": "LIGHTGBM_TWEEDIE",
            "CandidateFamily": "LIGHTGBM",
            "Package": "lightgbm",
            "UsesProductID": True,
            "CountAware": True,
            "Available": True,
            "Estimator": LGBMRegressor(
                objective="tweedie",
                tweedie_variance_power=1.3,
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.0,
                reg_lambda=1.0,
                n_jobs=-1,
                random_state=RANDOM_SEED,
                verbosity=-1,
            ),
            "SpecialRunner": None,
            "Configuration": {
                "objective": "tweedie",
                "tweedie_variance_power": 1.3,
                "n_estimators": 500,
                "learning_rate": 0.03,
                "num_leaves": 31,
                "min_child_samples": 20,
                "subsample": 0.80,
                "colsample_bytree": 0.80,
            },
        }
    )

if package_lookup.get("catboost", False):
    candidate_specs.append(
        {
            "CandidateID": "CATBOOST_RMSE",
            "CandidateFamily": "CATBOOST",
            "Package": "catboost",
            "UsesProductID": True,
            "CountAware": False,
            "Available": True,
            "Estimator": None,
            "SpecialRunner": "CATBOOST",
            "Configuration": {
                "loss_function": "RMSE",
                "iterations": 500,
                "learning_rate": 0.03,
                "depth": 7,
                "l2_leaf_reg": 3.0,
                "random_strength": 1.0,
            },
        }
    )

# =============================================================================
# LOAD FEATURES AND BASELINE
# =============================================================================

features = pd.read_csv(PART11F_FEATURES, low_memory=False)
baseline_predictions = pd.read_csv(PART11E_PREDICTIONS, low_memory=False)

for frame, label in [(features, "Part 11F features"), (baseline_predictions, "Part 11E predictions")]:
    for column in ["WeekStartDate", "WeekEndDate"]:
        if column not in frame.columns:
            raise AssertionError(f"{label} is missing {column}")
        frame[column] = pd.to_datetime(frame[column], errors="raise")
    if "CanonicalProductID" not in frame.columns:
        raise AssertionError(f"{label} is missing CanonicalProductID")
    frame["CanonicalProductID"] = frame["CanonicalProductID"].astype(str)

features = normalize_scope_columns(features, "Part 11F features")
baseline_predictions = normalize_scope_columns(
    baseline_predictions,
    "Part 11E baseline predictions",
    aliases=BASELINE_SCOPE_ALIASES,
)

required_feature_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    TARGET_COLUMN,
    *CANONICAL_SCOPE_COLUMNS,
}
if not required_feature_columns.issubset(features.columns):
    raise AssertionError(
        "Part 11F feature schema is incomplete. Missing: "
        f"{sorted(required_feature_columns - set(features.columns))}"
    )

# The Part 11F file is already the pre-holdout model-selection universe.
# No March 2026 target file or opened holdout source is read here.
features[TARGET_COLUMN] = pd.to_numeric(features[TARGET_COLUMN], errors="raise").astype(float)
if (features[TARGET_COLUMN] < 0).any():
    raise AssertionError("Negative weekly normal-demand targets found")

predictor_columns, predictor_source, predictor_contract_path = discover_normal_predictors(features)

# Explicit leakage guard. Predictor contracts must contain only information
# available before the forecast week. Lagged and rolling demand features are
# allowed; current targets, scope flags and evaluation outputs are not.
forbidden_predictors = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductName",
    "TargetID",
    TARGET_COLUMN,
    "WeeklyTotalDemand",
    "WeeklyBulkDemand",
    "BulkDemand",
    "NormalDemand",
    "TotalDemand",
    "Actual",
    "Prediction",
    "WeeklyTuningEligible",
    "IsOpenedMarchHoldout",
    *CANONICAL_SCOPE_COLUMNS,
}
leakage_predictors = sorted(
    column
    for column in predictor_columns
    if column in forbidden_predictors or "Scope" in column
)
if leakage_predictors:
    raise AssertionError(
        "Leakage/evaluation columns entered the predictor list: "
        f"{leakage_predictors}"
    )

# Product identity is intentionally added as a direct categorical predictor in
# this challenger stage. It is encoded inside each training fold only.
categorical_columns = ["CanonicalProductID"]
numeric_predictor_columns = [
    column
    for column in predictor_columns
    if column != "CanonicalProductID"
]

# Force numeric conversion of contract/fallback predictors. Infinite values are
# converted to missing values and imputed from each fold's training data only.
for column in numeric_predictor_columns:
    features[column] = pd.to_numeric(features[column], errors="coerce")
features[numeric_predictor_columns] = features[numeric_predictor_columns].replace(
    [np.inf, -np.inf], np.nan
)

# Remove columns that are entirely non-finite over the complete pre-holdout file.
entirely_empty_predictors = [
    column
    for column in numeric_predictor_columns
    if not np.isfinite(features[column].to_numpy(dtype=float)).any()
]
numeric_predictor_columns = [
    column for column in numeric_predictor_columns if column not in entirely_empty_predictors
]

if not numeric_predictor_columns:
    raise AssertionError("No usable numeric predictors remain")

feature_audit = pd.DataFrame(
    [
        {
            "FeatureName": column,
            "FeatureRole": "CATEGORICAL_PRODUCT_ID",
            "Source": "DIRECT_CHALLENGER_ADDITION",
            "DType": str(features[column].dtype),
            "MissingCount": int(features[column].isna().sum()),
            "FiniteCount": int(features[column].notna().sum()),
            "Included": True,
        }
        for column in categorical_columns
    ]
    + [
        {
            "FeatureName": column,
            "FeatureRole": "NUMERIC_PREDICTOR",
            "Source": predictor_source,
            "DType": str(features[column].dtype),
            "MissingCount": int(features[column].isna().sum()),
            "FiniteCount": int(np.isfinite(features[column].to_numpy(dtype=float)).sum()),
            "Included": True,
        }
        for column in numeric_predictor_columns
    ]
    + [
        {
            "FeatureName": column,
            "FeatureRole": "ENTIRELY_EMPTY_REMOVED",
            "Source": predictor_source,
            "DType": str(features[column].dtype),
            "MissingCount": int(features[column].isna().sum()),
            "FiniteCount": 0,
            "Included": False,
        }
        for column in entirely_empty_predictors
    ]
)

# =============================================================================
# CHRONOLOGICAL FOLD DESIGN
# =============================================================================

weeks = sorted(features["WeekStartDate"].drop_duplicates())
if len(weeks) != 47:
    raise AssertionError(f"Expected 47 pre-holdout weeks, found {len(weeks)}")

forecast_weeks = weeks[MINIMUM_TRAINING_WEEKS:]
if len(forecast_weeks) != 39:
    raise AssertionError(f"Expected 39 forecast weeks, found {len(forecast_weeks)}")

fold_rows = []
for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
    training = features.loc[features["WeekStartDate"] < forecast_week]
    validation = features.loc[features["WeekStartDate"] == forecast_week]
    fold_rows.append(
        {
            "FoldNumber": fold_number,
            "ForecastWeek": forecast_week,
            "TrainingStartWeek": training["WeekStartDate"].min(),
            "TrainingEndWeek": training["WeekStartDate"].max(),
            "TrainingWeeks": int(training["WeekStartDate"].nunique()),
            "TrainingRows": int(len(training)),
            "ValidationRows": int(len(validation)),
            "ValidationProducts": int(validation["CanonicalProductID"].nunique()),
            "Strict95Rows": int(validation["PriorStrict95PctScope"].sum()),
        }
    )
fold_design = pd.DataFrame(fold_rows)

# Baseline rows used for exact same-fold comparison.
required_baseline_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "TargetID",
    "BaselineMethod",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
}
if not required_baseline_columns.issubset(baseline_predictions.columns):
    raise AssertionError("Part 11E baseline-prediction schema is incomplete")

locked_baseline = baseline_predictions.loc[
    (baseline_predictions["TargetID"].astype(str) == TARGET_ID)
    & (baseline_predictions["BaselineMethod"].astype(str) == LOCKED_BENCHMARK_METHOD)
].copy()

if locked_baseline.empty:
    raise AssertionError("Locked NAIVE_LAST_CONTEXT rows are missing")

locked_baseline["Actual"] = pd.to_numeric(locked_baseline["Actual"], errors="raise").astype(float)
locked_baseline["Prediction"] = pd.to_numeric(locked_baseline["Prediction"], errors="raise").astype(float)

# Rename the feature-side target before merging. Part 11E can also contain a
# WeeklyNormalDemand column, so relying on pandas' automatic suffixing would
# remove the unsuffixed TARGET_COLUMN name and make reconciliation ambiguous.
FEATURE_RECONCILIATION_TARGET = f"{TARGET_COLUMN}__Part11FFeatureTarget"

feature_keys = features[
    [
        "WeekStartDate",
        "CanonicalProductID",
        TARGET_COLUMN,
        *CANONICAL_SCOPE_COLUMNS,
    ]
].copy()

feature_keys = feature_keys.rename(
    columns={TARGET_COLUMN: FEATURE_RECONCILIATION_TARGET}
)

reconciled_baseline = locked_baseline.merge(
    feature_keys,
    on=["WeekStartDate", "CanonicalProductID"],
    how="inner",
    suffixes=("__Baseline", "__Feature"),
    validate="one_to_one",
)

if len(reconciled_baseline) != len(locked_baseline):
    raise AssertionError("Baseline and feature rows do not reconcile one-to-one")

if FEATURE_RECONCILIATION_TARGET not in reconciled_baseline.columns:
    raise AssertionError(
        "Feature-side normal-demand target is missing after baseline reconciliation. "
        f"Expected column: {FEATURE_RECONCILIATION_TARGET}"
    )

actual_difference = np.abs(
    reconciled_baseline["Actual"].to_numpy(dtype=float)
    - reconciled_baseline[FEATURE_RECONCILIATION_TARGET].to_numpy(dtype=float)
).max()
if actual_difference > 1e-9:
    raise AssertionError(f"Baseline actuals differ from Part 11F targets by {actual_difference}")

for scope_column in CANONICAL_SCOPE_COLUMNS:
    baseline_column = f"{scope_column}__Baseline"
    feature_column = f"{scope_column}__Feature"
    if baseline_column in reconciled_baseline.columns and feature_column in reconciled_baseline.columns:
        mismatch = int((reconciled_baseline[baseline_column] != reconciled_baseline[feature_column]).sum())
        if mismatch:
            raise AssertionError(f"Scope mismatch for {scope_column}: {mismatch}")

# =============================================================================
# PREPROCESSOR AND MODEL RUNNERS
# =============================================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "product",
            Pipeline([("one_hot", make_ohe())]),
            categorical_columns,
        ),
        (
            "numeric",
            Pipeline([("imputer", make_imputer())]),
            numeric_predictor_columns,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)


def run_standard_candidate(spec: dict, train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    estimator = clone(spec["Estimator"])
    pipeline = Pipeline(
        [
            ("preprocess", clone(preprocessor)),
            ("model", estimator),
        ]
    )
    x_train = train[categorical_columns + numeric_predictor_columns]
    y_train = train[TARGET_COLUMN].to_numpy(dtype=float)
    x_valid = valid[categorical_columns + numeric_predictor_columns]
    pipeline.fit(x_train, y_train)
    prediction = pipeline.predict(x_valid)
    return np.clip(np.asarray(prediction, dtype=float), 0.0, None)


def run_catboost_candidate(spec: dict, train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    from catboost import CatBoostRegressor

    usable_numeric = [
        column
        for column in numeric_predictor_columns
        if np.isfinite(train[column].to_numpy(dtype=float)).any()
    ]
    columns = categorical_columns + usable_numeric
    x_train = train[columns].copy()
    x_valid = valid[columns].copy()
    x_train["CanonicalProductID"] = x_train["CanonicalProductID"].astype(str)
    x_valid["CanonicalProductID"] = x_valid["CanonicalProductID"].astype(str)

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=500,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,
        random_strength=1.0,
        random_seed=RANDOM_SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    model.fit(
        x_train,
        train[TARGET_COLUMN].to_numpy(dtype=float),
        cat_features=[0],
    )
    prediction = model.predict(x_valid)
    return np.clip(np.asarray(prediction, dtype=float), 0.0, None)


# =============================================================================
# EXPANDING-WINDOW CHALLENGE
# =============================================================================

prediction_rows: list[pd.DataFrame] = []
candidate_audit_rows: list[dict] = []

for spec in candidate_specs:
    candidate_id = spec["CandidateID"]
    candidate_failed = False
    candidate_error = ""
    successful_folds = 0

    print("-" * 100)
    print(f"Running candidate: {candidate_id}")

    for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
        train = features.loc[features["WeekStartDate"] < forecast_week].copy()
        valid = features.loc[features["WeekStartDate"] == forecast_week].copy()

        if train["WeekStartDate"].nunique() < MINIMUM_TRAINING_WEEKS:
            raise AssertionError("Fold has insufficient training weeks")
        if valid.empty:
            raise AssertionError(f"Empty validation week: {forecast_week}")

        try:
            if spec["SpecialRunner"] == "CATBOOST":
                prediction = run_catboost_candidate(spec, train, valid)
            else:
                prediction = run_standard_candidate(spec, train, valid)
        except Exception as exc:
            candidate_failed = True
            candidate_error = (
                f"Fold {fold_number}, week {forecast_week.date()}: "
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()
            break

        if len(prediction) != len(valid):
            raise AssertionError(f"Prediction length mismatch for {candidate_id}")
        if not np.isfinite(prediction).all():
            raise AssertionError(f"Non-finite predictions from {candidate_id}")
        if (prediction < 0).any():
            raise AssertionError(f"Negative predictions from {candidate_id}")

        output = valid[
            [
                "WeekStartDate",
                "WeekEndDate",
                "WeekID",
                "CanonicalProductID",
                "CanonicalProductName",
                TARGET_COLUMN,
                *CANONICAL_SCOPE_COLUMNS,
            ]
        ].copy()
        output["FoldNumber"] = fold_number
        output["TargetID"] = TARGET_ID
        output["CandidateID"] = candidate_id
        output["CandidateFamily"] = spec["CandidateFamily"]
        output["Actual"] = output[TARGET_COLUMN].astype(float)
        output["Prediction"] = prediction
        output["Error"] = output["Prediction"] - output["Actual"]
        output["AbsoluteError"] = output["Error"].abs()
        output["ProductIDUsedAsDirectPredictor"] = bool(spec["UsesProductID"])
        output["BulkIncludedInModelTarget"] = False
        prediction_rows.append(output)
        successful_folds += 1

    candidate_audit_rows.append(
        {
            "CandidateID": candidate_id,
            "CandidateFamily": spec["CandidateFamily"],
            "Package": spec["Package"],
            "UsesProductID": spec["UsesProductID"],
            "CountAware": spec["CountAware"],
            "Available": spec["Available"],
            "SuccessfulFolds": successful_folds,
            "ExpectedFolds": len(forecast_weeks),
            "CompletedAllFolds": not candidate_failed and successful_folds == len(forecast_weeks),
            "FailureMessage": candidate_error,
            "ConfigurationJSON": json.dumps(spec["Configuration"], sort_keys=True),
        }
    )

candidate_audit = pd.DataFrame(candidate_audit_rows)
completed_candidate_ids = candidate_audit.loc[
    candidate_audit["CompletedAllFolds"], "CandidateID"
].astype(str).tolist()

if not completed_candidate_ids:
    raise RuntimeError("No expanded weekly candidate completed all 39 folds")

candidate_predictions = pd.concat(prediction_rows, ignore_index=True)
candidate_predictions = candidate_predictions.loc[
    candidate_predictions["CandidateID"].isin(completed_candidate_ids)
].copy()

# Add locked benchmark rows to a common comparison table, but keep them clearly
# labelled as the existing reference rather than a newly fitted candidate.
baseline_common = locked_baseline[
    [
        "WeekStartDate",
        "WeekEndDate",
        "WeekID",
        "CanonicalProductID",
        "CanonicalProductName",
        "Actual",
        "Prediction",
        *CANONICAL_SCOPE_COLUMNS,
    ]
].copy()
baseline_common["FoldNumber"] = baseline_common["WeekStartDate"].map(
    {week: index for index, week in enumerate(forecast_weeks, start=1)}
)
baseline_common["TargetID"] = TARGET_ID
baseline_common["CandidateID"] = LOCKED_BENCHMARK_METHOD
baseline_common["CandidateFamily"] = "LOCKED_REFERENCE_BASELINE"
baseline_common["Error"] = baseline_common["Prediction"] - baseline_common["Actual"]
baseline_common["AbsoluteError"] = baseline_common["Error"].abs()
baseline_common["ProductIDUsedAsDirectPredictor"] = False
baseline_common["BulkIncludedInModelTarget"] = False

comparison_predictions = pd.concat(
    [candidate_predictions, baseline_common],
    ignore_index=True,
    sort=False,
)

# =============================================================================
# METRICS, RANKING AND CHALLENGER DECISION
# =============================================================================

pooled_rows = []
for (candidate_id, candidate_family), group in comparison_predictions.groupby(
    ["CandidateID", "CandidateFamily"], sort=True
):
    for scope_id, scope_flag in SCOPE_FLAGS.items():
        scoped = group if scope_flag is None else group.loc[group[scope_flag]]
        pooled_rows.append(
            {
                "TargetID": TARGET_ID,
                "ScopeID": scope_id,
                "CandidateID": candidate_id,
                "CandidateFamily": candidate_family,
                "ForecastWeeks": int(scoped["WeekStartDate"].nunique()),
                "Products": int(scoped["CanonicalProductID"].nunique()),
                "ProductIDUsedAsDirectPredictor": bool(
                    scoped["ProductIDUsedAsDirectPredictor"].iloc[0]
                ),
                **metric_record(scoped["Actual"], scoped["Prediction"]),
            }
        )
pooled_metrics = pd.DataFrame(pooled_rows)
pooled_metrics["AbsoluteTotalBias"] = pooled_metrics["TotalBias"].abs()
pooled_metrics["ComplexityOrder"] = pooled_metrics["CandidateID"].map(candidate_complexity_order)
pooled_metrics = pooled_metrics.sort_values(
    [
        "ScopeID",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "ComplexityOrder",
        "CandidateID",
    ],
    kind="mergesort",
).reset_index(drop=True)
pooled_metrics["RankWithinScope"] = pooled_metrics.groupby("ScopeID").cumcount() + 1
pooled_metrics["IsScopeWinner"] = pooled_metrics["RankWithinScope"] == 1

fold_metric_rows = []
for (candidate_id, candidate_family, forecast_week), group in comparison_predictions.groupby(
    ["CandidateID", "CandidateFamily", "WeekStartDate"], sort=True
):
    for scope_id, scope_flag in SCOPE_FLAGS.items():
        scoped = group if scope_flag is None else group.loc[group[scope_flag]]
        fold_metric_rows.append(
            {
                "ForecastWeek": forecast_week,
                "TargetID": TARGET_ID,
                "ScopeID": scope_id,
                "CandidateID": candidate_id,
                "CandidateFamily": candidate_family,
                **metric_record(scoped["Actual"], scoped["Prediction"]),
            }
        )
fold_metrics = pd.DataFrame(fold_metric_rows)

primary_metrics = pooled_metrics.loc[pooled_metrics["ScopeID"] == PRIMARY_SCOPE_ID].copy()
primary_metrics = primary_metrics.sort_values(
    [
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "ComplexityOrder",
        "CandidateID",
    ],
    kind="mergesort",
).reset_index(drop=True)
primary_metrics["PrimaryScopeRank"] = np.arange(1, len(primary_metrics) + 1)

benchmark_row = primary_metrics.loc[
    primary_metrics["CandidateID"] == LOCKED_BENCHMARK_METHOD
]
if len(benchmark_row) != 1:
    raise AssertionError("Could not identify one locked benchmark metric row")
benchmark_row = benchmark_row.iloc[0]

challenger_rows = primary_metrics.loc[
    primary_metrics["CandidateID"] != LOCKED_BENCHMARK_METHOD
].copy()
if challenger_rows.empty:
    raise AssertionError("No completed challenger metric rows")
best_challenger = challenger_rows.iloc[0]

comparison = challenger_rows.copy()
comparison["BenchmarkMethodID"] = LOCKED_BENCHMARK_METHOD
comparison["BenchmarkWAPEPercentage"] = float(benchmark_row["WAPEPercentage"])
comparison["WAPEImprovementPercentagePoints"] = (
    comparison["BenchmarkWAPEPercentage"] - comparison["WAPEPercentage"]
)
comparison["RelativeWAPEImprovementPercentage"] = (
    100.0
    * comparison["WAPEImprovementPercentagePoints"]
    / comparison["BenchmarkWAPEPercentage"]
)
comparison["BenchmarkMAE"] = float(benchmark_row["MAE"])
comparison["MAEImprovement"] = comparison["BenchmarkMAE"] - comparison["MAE"]
comparison["BenchmarkRMSE"] = float(benchmark_row["RMSE"])
comparison["RMSEImprovement"] = comparison["BenchmarkRMSE"] - comparison["RMSE"]
comparison["BenchmarkTotalBias"] = float(benchmark_row["TotalBias"])
comparison["BeatBenchmarkOnWAPE"] = comparison["WAPEPercentage"] < comparison["BenchmarkWAPEPercentage"]
comparison["BeatBenchmarkOnMAE"] = comparison["MAE"] < comparison["BenchmarkMAE"]
comparison["BeatBenchmarkOnRMSE"] = comparison["RMSE"] < comparison["BenchmarkRMSE"]
comparison["CandidateReadyForNestedTuning"] = comparison["BeatBenchmarkOnWAPE"]

ranking = pooled_metrics.copy()
ranking["IsLockedPart11IBenchmark"] = ranking["CandidateID"] == LOCKED_BENCHMARK_METHOD
ranking["ProceedTo11IA2"] = False

# Advance up to the three best completed challengers, prioritising any that beat
# the benchmark on primary-scope WAPE. The next step performs controlled tuning
# and daily-to-weekly comparison; this step does not make the final replacement.
advance_pool = comparison.sort_values(
    [
        "BeatBenchmarkOnWAPE",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
    ],
    ascending=[False, True, True, True, True],
    kind="mergesort",
).head(3)
advance_ids = set(advance_pool["CandidateID"].astype(str))
ranking.loc[ranking["CandidateID"].isin(advance_ids), "ProceedTo11IA2"] = True

best_challenger_beats_benchmark = bool(
    best_challenger["WAPEPercentage"] < benchmark_row["WAPEPercentage"]
)

# =============================================================================
# VALIDATION
# =============================================================================

evaluation_feature_rows = int(
    features["WeekStartDate"].isin(forecast_weeks).sum()
)
expected_candidate_prediction_rows = (
    evaluation_feature_rows * len(completed_candidate_ids)
)
actual_candidate_prediction_rows = len(candidate_predictions)
expected_total_metric_groups = (len(completed_candidate_ids) + 1) * len(SCOPE_FLAGS)
expected_fold_metric_groups = (
    (len(completed_candidate_ids) + 1) * len(SCOPE_FLAGS) * len(forecast_weeks)
)

nonfinite_predictions = int(
    (~np.isfinite(candidate_predictions["Prediction"].to_numpy(dtype=float))).sum()
)
negative_predictions = int((candidate_predictions["Prediction"] < 0).sum())

validation = pd.DataFrame(
    [
        {
            "Check": "Part 11I lock verified",
            "Expected": True,
            "Actual": True,
            "Passed": True,
        },
        {
            "Check": "Part 11I target remains normal demand",
            "Expected": TARGET_ID,
            "Actual": locked_design.get("TargetID"),
            "Passed": locked_design.get("TargetID") == TARGET_ID,
        },
        {
            "Check": "Part 11I primary scope remains strict 95 percent",
            "Expected": PRIMARY_SCOPE_ID,
            "Actual": locked_design.get("ScopeID"),
            "Passed": locked_design.get("ScopeID") == PRIMARY_SCOPE_ID,
        },
        {
            "Check": "Bulk included in model target",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Pre-holdout feature weeks",
            "Expected": 47,
            "Actual": len(weeks),
            "Passed": len(weeks) == 47,
        },
        {
            "Check": "Chronological forecast weeks",
            "Expected": 39,
            "Actual": len(forecast_weeks),
            "Passed": len(forecast_weeks) == 39,
        },
        {
            "Check": "Minimum training weeks",
            "Expected": MINIMUM_TRAINING_WEEKS,
            "Actual": int(fold_design["TrainingWeeks"].min()),
            "Passed": int(fold_design["TrainingWeeks"].min()) == MINIMUM_TRAINING_WEEKS,
        },
        {
            "Check": "Completed challenger models",
            "Expected": ">= 1",
            "Actual": len(completed_candidate_ids),
            "Passed": len(completed_candidate_ids) >= 1,
        },
        {
            "Check": "Candidate prediction rows",
            "Expected": expected_candidate_prediction_rows,
            "Actual": actual_candidate_prediction_rows,
            "Passed": actual_candidate_prediction_rows == expected_candidate_prediction_rows,
        },
        {
            "Check": "Pooled metric groups",
            "Expected": expected_total_metric_groups,
            "Actual": len(pooled_metrics),
            "Passed": len(pooled_metrics) == expected_total_metric_groups,
        },
        {
            "Check": "Fold metric groups",
            "Expected": expected_fold_metric_groups,
            "Actual": len(fold_metrics),
            "Passed": len(fold_metrics) == expected_fold_metric_groups,
        },
        {
            "Check": "Non-finite challenger predictions",
            "Expected": 0,
            "Actual": nonfinite_predictions,
            "Passed": nonfinite_predictions == 0,
        },
        {
            "Check": "Negative challenger predictions",
            "Expected": 0,
            "Actual": negative_predictions,
            "Passed": negative_predictions == 0,
        },
        {
            "Check": "Product ID used by expanded candidates",
            "Expected": True,
            "Actual": bool(candidate_predictions["ProductIDUsedAsDirectPredictor"].all()),
            "Passed": bool(candidate_predictions["ProductIDUsedAsDirectPredictor"].all()),
        },
        {
            "Check": "Opened March 2026 targets read",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Existing Part 11I lock modified",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Final replacement method selected in Part 11IA1",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
    ]
)

if not validation["Passed"].all():
    raise AssertionError(
        "Part 11IA1 validation failed:\n"
        + validation.loc[~validation["Passed"]].to_string(index=False)
    )

# =============================================================================
# CONTRACT, MEMORY, CHECKPOINT AND LOCK
# =============================================================================

contract = {
    "StepID": STEP_ID,
    "CreatedUTC": NOW_UTC.isoformat(),
    "CreatedLocal": NOW_LOCAL.isoformat(),
    "Status": STATUS,
    "Purpose": (
        "Challenge the locked Part 11I direct-weekly benchmark with stronger "
        "tree-based, count-aware and categorical-aware models under the same "
        "39-fold chronological pre-holdout design."
    ),
    "LockedBenchmark": {
        "TargetID": TARGET_ID,
        "MethodID": LOCKED_BENCHMARK_METHOD,
        "ScopeID": PRIMARY_SCOPE_ID,
        "Part11ILockSHA256": part11i_lock_hash,
    },
    "FeatureDesign": {
        "FeatureSource": str(PART11F_FEATURES),
        "FeatureSourceSHA256": sha256_file(PART11F_FEATURES),
        "PredictorDiscovery": predictor_source,
        "PredictorContractPath": predictor_contract_path,
        "NumericPredictorCount": len(numeric_predictor_columns),
        "DirectCategoricalPredictors": categorical_columns,
        "EntirelyEmptyPredictorsRemoved": entirely_empty_predictors,
        "ProductIDEncoding": "ONE_HOT_WITHIN_EACH_TRAINING_FOLD_OR_NATIVE_CATBOOST",
    },
    "Evaluation": {
        "TrainingDesign": "EXPANDING_WINDOW",
        "MinimumTrainingWeeks": MINIMUM_TRAINING_WEEKS,
        "ForecastWeeks": len(forecast_weeks),
        "PrimaryScopeID": PRIMARY_SCOPE_ID,
        "PrimaryMetric": "WAPEPercentage",
        "OtherMetrics": ["MAE", "RMSE", "MeanBias", "TotalBias"],
        "PredictionsRoundedBeforeScoring": False,
    },
    "CandidateModels": [
        {
            "CandidateID": spec["CandidateID"],
            "CandidateFamily": spec["CandidateFamily"],
            "Package": spec["Package"],
            "UsesProductID": spec["UsesProductID"],
            "CountAware": spec["CountAware"],
            "Configuration": spec["Configuration"],
        }
        for spec in candidate_specs
    ],
    "CompletedCandidateIDs": completed_candidate_ids,
    "FailedCandidateIDs": candidate_audit.loc[
        ~candidate_audit["CompletedAllFolds"], "CandidateID"
    ].astype(str).tolist(),
    "BestExpandedChallenger": {
        "CandidateID": str(best_challenger["CandidateID"]),
        "WAPEPercentage": float(best_challenger["WAPEPercentage"]),
        "MAE": float(best_challenger["MAE"]),
        "RMSE": float(best_challenger["RMSE"]),
        "TotalBias": float(best_challenger["TotalBias"]),
        "BeatLockedBenchmarkOnWAPE": best_challenger_beats_benchmark,
    },
    "LockedBenchmarkPerformance": {
        "WAPEPercentage": float(benchmark_row["WAPEPercentage"]),
        "MAE": float(benchmark_row["MAE"]),
        "RMSE": float(benchmark_row["RMSE"]),
        "TotalBias": float(benchmark_row["TotalBias"]),
    },
    "ProceedingTo11IA2": sorted(advance_ids),
    "Governance": {
        "Part11ILockModified": False,
        "FinalReplacementSelected": False,
        "OpenedMarch2026TargetsRead": False,
        "BulkIncludedInModelTarget": False,
        "NextStep": (
            "Part 11IA2: controlled tuning of the strongest challengers and "
            "same-fold daily-to-weekly architecture comparison."
        ),
    },
}

stage_root = EXT_ROOT / f".11IA1_staging_{uuid.uuid4().hex}"
stage_root.mkdir(parents=True, exist_ok=False)

try:
    for path, frame in [
        (PACKAGE_AUDIT_PATH, package_audit),
        (FEATURE_AUDIT_PATH, feature_audit),
        (FOLD_DESIGN_PATH, fold_design),
        (CANDIDATE_AUDIT_PATH, candidate_audit),
        (PREDICTIONS_PATH, candidate_predictions),
        (POOLED_METRICS_PATH, pooled_metrics),
        (FOLD_METRICS_PATH, fold_metrics),
        (RANKING_PATH, ranking),
        (COMPARISON_PATH, comparison),
        (VALIDATION_PATH, validation),
    ]:
        stage_csv(stage_root, path, frame)

    stage_json(stage_root, CONTRACT_PATH, contract)

    decision_body = "\n".join(
        [
            "- Part 11I remains the locked benchmark and was not overwritten.",
            f"- Expanded weekly candidates were evaluated for {len(forecast_weeks)} chronological folds.",
            "- WeeklyNormalDemand remains the target; bulk preorders remain excluded.",
            "- CanonicalProductID was introduced as a direct predictor using fold-safe encoding.",
            f"- Best expanded challenger: {best_challenger['CandidateID']}.",
            f"- Best challenger WAPE at the 95% scope: {float(best_challenger['WAPEPercentage']):.6f}%.",
            f"- Locked benchmark WAPE at the 95% scope: {float(benchmark_row['WAPEPercentage']):.6f}%.",
            f"- Best challenger beat benchmark on WAPE: {best_challenger_beats_benchmark}.",
            f"- Candidates proceeding to 11IA2: {', '.join(sorted(advance_ids))}.",
            "- No final replacement decision was made in this screening step.",
        ]
    )

    files_body = "\n".join(
        [
            f"- Package audit: `{PACKAGE_AUDIT_PATH}`",
            f"- Feature audit: `{FEATURE_AUDIT_PATH}`",
            f"- Candidate audit: `{CANDIDATE_AUDIT_PATH}`",
            f"- Candidate predictions: `{PREDICTIONS_PATH}`",
            f"- Pooled metrics: `{POOLED_METRICS_PATH}`",
            f"- Challenger comparison: `{COMPARISON_PATH}`",
            f"- Challenge contract: `{CONTRACT_PATH}`",
        ]
    )

    for key, final_path in MEMORY_FILES.items():
        original = final_path.read_text(encoding="utf-8")

        if key == "PROJECT_CONTEXT":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 expanded weekly model challenge",
                (
                    "A separate challenger stage was added after Part 11I to test "
                    "stronger direct-weekly models, including random forests, extra "
                    "trees, count-aware boosting and any installed XGBoost, LightGBM "
                    "or CatBoost implementations. The Part 11I lock remains intact."
                ),
            )
        elif key == "WORKFLOW":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 workflow status",
                (
                    f"Part 11IA1 completed with status {STATUS}. Part 11IA2 should "
                    "tune the strongest challengers and compare them against a "
                    "leakage-safe daily-to-weekly architecture on the same folds."
                ),
            )
        elif key == "DECISIONS":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 decisions",
                decision_body,
            )
        elif key == "FILES_AND_PATHS":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 files",
                files_body,
            )
        elif key == "METRICS_AND_RESULTS":
            updated = update_section(
                original,
                "STEP_11IA1",
                "Part 11IA1 results",
                decision_body,
            )
        elif key == "CHAT_INDEX":
            row = (
                f"| {NOW_LOCAL.isoformat()} | 11IA1 | Expanded weekly model challenge | "
                f"{STATUS} |"
            )
            updated = original if row in original else original.rstrip() + "\n" + row + "\n"
        elif key == "CURRENT_HANDOFF":
            updated = "\n".join(
                [
                    "# Current Handoff",
                    "",
                    f"- Current completed step: {STEP_ID}",
                    f"- Status: {STATUS}",
                    f"- Updated local time: {NOW_LOCAL.isoformat()}",
                    f"- Locked benchmark remains: {LOCKED_BENCHMARK_METHOD}",
                    f"- Primary scope remains: {PRIMARY_SCOPE_ID}",
                    f"- Best expanded challenger: {best_challenger['CandidateID']}",
                    f"- Best challenger beat benchmark: {best_challenger_beats_benchmark}",
                    f"- Candidates proceeding: {', '.join(sorted(advance_ids))}",
                    "- Part 11I lock was not changed.",
                    "- Next step: 11IA2 controlled tuning and daily-to-weekly comparison.",
                    "",
                ]
            )
        else:
            raise KeyError(key)

        stage_text(stage_root, final_path, updated)

    step_text = f"""# Step 11IA1 — Expanded Weekly Model Challenge

- **Step ID:** {STEP_ID}
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## Purpose
Challenge the provisional Part 11I weekly benchmark with stronger direct-weekly models while preserving the same target, chronological folds, dynamic scopes, bulk policy and leakage controls.

## Models evaluated
```text
{candidate_audit[['CandidateID', 'CandidateFamily', 'CompletedAllFolds', 'FailureMessage']].to_string(index=False)}
```

## Primary-scope result
- Locked benchmark: {LOCKED_BENCHMARK_METHOD}
- Locked benchmark WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%
- Best expanded challenger: {best_challenger['CandidateID']}
- Best challenger WAPE: {float(best_challenger['WAPEPercentage']):.6f}%
- Best challenger beat benchmark: {best_challenger_beats_benchmark}
- Candidates proceeding to 11IA2: {', '.join(sorted(advance_ids))}

## Governance
- Part 11I was not overwritten.
- March 2026 targets were not read.
- Bulk was not included in the modelled target.
- Product identity was encoded inside each training fold.
- This was a fixed-configuration screening step; it did not make a final replacement decision.

## Outputs
{files_body}

## Next step
Part 11IA2: controlled tuning of the strongest challengers and direct comparison with leakage-safe daily-to-weekly forecasting on the same 39 folds.
"""
    stage_text(stage_root, STEP_MEMORY_PATH, step_text)

    checkpoint = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "Part11ILockSHA256": part11i_lock_hash,
        "FeatureSourceSHA256": sha256_file(PART11F_FEATURES),
        "BaselinePredictionsSHA256": sha256_file(PART11E_PREDICTIONS),
        "CompletedCandidateIDs": completed_candidate_ids,
        "ProceedingTo11IA2": sorted(advance_ids),
        "BestExpandedChallenger": contract["BestExpandedChallenger"],
        "LockedBenchmarkPerformance": contract["LockedBenchmarkPerformance"],
        "Safety": {
            "Part11ILockModified": False,
            "OpenedMarch2026TargetsRead": False,
            "BulkIncludedInModelTarget": False,
            "ProductIDEncodedWithinTrainingFold": True,
            "ForecastsRoundedBeforeScoring": False,
            "FinalReplacementSelected": False,
        },
        "ReadyForPart11IA2": True,
    }
    stage_json(stage_root, CHECKPOINT_PATH, checkpoint)
    checkpoint_hash = sha256_file(stage_path(stage_root, CHECKPOINT_PATH))
    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        f"{checkpoint_hash}  {CHECKPOINT_PATH.name}\n",
    )

    manifest_targets = [
        PACKAGE_AUDIT_PATH,
        FEATURE_AUDIT_PATH,
        FOLD_DESIGN_PATH,
        CANDIDATE_AUDIT_PATH,
        PREDICTIONS_PATH,
        POOLED_METRICS_PATH,
        FOLD_METRICS_PATH,
        RANKING_PATH,
        COMPARISON_PATH,
        VALIDATION_PATH,
        CONTRACT_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]
    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(EXT_ROOT)),
                "Bytes": stage_path(stage_root, path).stat().st_size,
                "SHA256": sha256_file(stage_path(stage_root, path)),
            }
            for path in manifest_targets
        ]
    ).sort_values("RelativePath")
    stage_csv(stage_root, MANIFEST_PATH, manifest)
    manifest_hash = sha256_file(stage_path(stage_root, MANIFEST_PATH))

    lock = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "Part11ILockSHA256": part11i_lock_hash,
        "Contract": {
            "Path": str(CONTRACT_PATH),
            "SHA256": sha256_file(stage_path(stage_root, CONTRACT_PATH)),
        },
        "Checkpoint": {
            "Path": str(CHECKPOINT_PATH),
            "SHA256": checkpoint_hash,
        },
        "OutputHashManifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_hash,
            "FilesListed": int(len(manifest)),
        },
        "BestExpandedChallenger": contract["BestExpandedChallenger"],
        "ProceedingTo11IA2": sorted(advance_ids),
        "SafetyAssertions": checkpoint["Safety"],
        "ReadyForPart11IA2": True,
        "NextStep": "11IA2",
    }
    stage_json(stage_root, LOCK_PATH, lock)
    lock_hash = sha256_file(stage_path(stage_root, LOCK_PATH))
    stage_text(stage_root, LOCK_SHA_PATH, f"{lock_hash}  {LOCK_PATH.name}\n")

    log_text = "\n".join(
        [
            f"Status: {STATUS}",
            f"Run local: {NOW_LOCAL.isoformat()}",
            f"Predictor source: {predictor_source}",
            f"Numeric predictors: {len(numeric_predictor_columns)}",
            f"Completed candidates: {', '.join(completed_candidate_ids)}",
            f"Best challenger: {best_challenger['CandidateID']}",
            f"Best challenger WAPE: {float(best_challenger['WAPEPercentage']):.12f}",
            f"Benchmark WAPE: {float(benchmark_row['WAPEPercentage']):.12f}",
            f"Best challenger beat benchmark: {best_challenger_beats_benchmark}",
            f"Proceeding: {', '.join(sorted(advance_ids))}",
            f"Checkpoint SHA256: {checkpoint_hash}",
            f"Lock SHA256: {lock_hash}",
            "",
        ]
    )
    stage_text(stage_root, LOG_PATH, log_text)

    staged_files = [path for path in stage_root.rglob("*") if path.is_file()]
    if not staged_files:
        raise AssertionError("Part 11IA1 staging directory is empty")

    for staged in sorted(staged_files):
        final = EXT_ROOT / staged.relative_to(stage_root)
        final.parent.mkdir(parents=True, exist_ok=True)
        if final.exists() and final not in MEMORY_FILES.values():
            raise FileExistsError(f"Refusing to overwrite Part 11IA1 output: {final}")
        os.replace(staged, final)

    for path in NEW_OUTPUTS:
        make_read_only(path)

finally:
    if stage_root.exists():
        shutil.rmtree(stage_root)

# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("=" * 100)
print("EDEN WEEKLY FORECASTING EXTENSION — PART 11IA1 COMPLETE")
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Extension root: {EXT_ROOT}")
print(f"Local time: {NOW_LOCAL.isoformat()}")

print("\nPACKAGE AVAILABILITY")
print(
    package_audit[
        ["Distribution", "Version", "Installed", "ImportSucceeded", "ImportError"]
    ].to_string(index=False)
)

print("\nFEATURE DESIGN")
print(f"Predictor source: {predictor_source}")
print(f"Predictor contract: {predictor_contract_path}")
print(f"Numeric predictors used: {len(numeric_predictor_columns)}")
print(f"Direct categorical predictors: {categorical_columns}")
print(f"Entirely empty predictors removed: {entirely_empty_predictors}")

print("\nCANDIDATE EXECUTION AUDIT")
print(
    candidate_audit[
        [
            "CandidateID",
            "CandidateFamily",
            "Package",
            "CountAware",
            "SuccessfulFolds",
            "CompletedAllFolds",
            "FailureMessage",
        ]
    ].to_string(index=False)
)

print("\nPRIMARY 95% SCOPE RANKING")
print(
    primary_metrics[
        [
            "PrimaryScopeRank",
            "CandidateID",
            "CandidateFamily",
            "ForecastWeeks",
            "Observations",
            "ActualTotal",
            "PredictedTotal",
            "MAE",
            "RMSE",
            "WAPEPercentage",
            "MeanBias",
            "TotalBias",
            "ProductIDUsedAsDirectPredictor",
        ]
    ].to_string(index=False)
)

print("\nCHALLENGER VERSUS LOCKED BENCHMARK")
print(
    comparison[
        [
            "CandidateID",
            "WAPEPercentage",
            "BenchmarkWAPEPercentage",
            "WAPEImprovementPercentagePoints",
            "RelativeWAPEImprovementPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "BeatBenchmarkOnWAPE",
            "CandidateReadyForNestedTuning",
        ]
    ].to_string(index=False)
)

print("\nPART 11IA1 DECISION")
print(f"Locked benchmark retained without modification: {LOCKED_BENCHMARK_METHOD}")
print(f"Best expanded challenger: {best_challenger['CandidateID']}")
print(f"Best challenger WAPE: {float(best_challenger['WAPEPercentage']):.6f}%")
print(f"Locked benchmark WAPE: {float(benchmark_row['WAPEPercentage']):.6f}%")
print(f"Best challenger beat benchmark: {best_challenger_beats_benchmark}")
print(f"Proceeding to Part 11IA2: {', '.join(sorted(advance_ids))}")
print("Final replacement selected in Part 11IA1: False")

print("\nPART 11IA1 VALIDATION")
print(validation.to_string(index=False))

print("\nCONTROL OUTPUTS")
print(f"- Package audit: {PACKAGE_AUDIT_PATH}")
print(f"- Candidate audit: {CANDIDATE_AUDIT_PATH}")
print(f"- Candidate predictions: {PREDICTIONS_PATH}")
print(f"- Pooled metrics: {POOLED_METRICS_PATH}")
print(f"- Challenger comparison: {COMPARISON_PATH}")
print(f"- Challenge contract: {CONTRACT_PATH}")
print(f"- Checkpoint: {CHECKPOINT_PATH}")
print(f"- Checkpoint SHA256: {checkpoint_hash}")
print(f"- Part 11IA1 lock: {LOCK_PATH}")
print(f"- Part 11IA1 lock SHA256: {lock_hash}")

print(
    "\nSAFETY: Part 11I lock modified False; opened March 2026 targets read False; "
    "bulk included in model target False; product identity encoded within each "
    "training fold True; forecasts rounded before scoring False; final replacement "
    "selected False."
)
print("=" * 100)

----------------------------------------------------------------------------------------------------
Running candidate: HISTGB_POISSON
----------------------------------------------------------------------------------------------------
Running candidate: RANDOM_FOREST_SQUARED
----------------------------------------------------------------------------------------------------
Running candidate: RANDOM_FOREST_POISSON
----------------------------------------------------------------------------------------------------
Running candidate: EXTRA_TREES_SQUARED
----------------------------------------------------------------------------------------------------
Running candidate: EXTRA_TREES_POISSON
----------------------------------------------------------------------------------------------------
Running candidate: XGBOOST_SQUARED
----------------------------------------------------------------------------------------------------
Running candidate: XGBOOST_POISSON
-----------------------------

In [29]:
from __future__ import annotations

import hashlib
import importlib.metadata
import inspect
import json
import os
import shutil
import stat
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# =============================================================================
# EDEN WEEKLY FORECASTING EXTENSION — PART 11IA2
# Tuned, calibrated, blended and daily-to-weekly challenger evaluation
#
# This step:
# - preserves the Part 11I and Part 11IA1 locks;
# - performs leakage-safe prequential configuration selection;
# - calibrates candidate forecasts using prior validation weeks only;
# - tests adaptive blends with the locked weekly baseline;
# - reconstructs daily hurdle-plus-NAIVE5 forecasts at a true week-start origin;
# - separately reports an actual-within-week-update diagnostic;
# - makes a controlled final method amendment only when preset operational
#   acceptance conditions are satisfied.
# =============================================================================

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_ROOT = PROJECT_ROOT / "eden_datasets"
EXT_ROOT = EDEN_ROOT / "weekly_forecasting_extension"

SNAPSHOT_ROOT = EXT_ROOT / "01_source_snapshot"
SNAPSHOT_DATA_ROOT = SNAPSHOT_ROOT / "data"
SNAPSHOT_CONTRACT_ROOT = SNAPSHOT_ROOT / "contracts"
SNAPSHOT_MANIFEST_ROOT = SNAPSHOT_ROOT / "manifests"
FEATURE_ROOT = EXT_ROOT / "04_weekly_features"
BASELINE_ROOT = EXT_ROOT / "05_weekly_baselines"
MODEL_ROOT = EXT_ROOT / "06_weekly_models"
FINAL_MODEL_ROOT = MODEL_ROOT / "final"
VALIDATION_ROOT = EXT_ROOT / "07_weekly_validation"
CHECKPOINT_ROOT = EXT_ROOT / "11_checkpoints"
MEMORY_ROOT = EXT_ROOT / "00_project_memory"
STEP_MEMORY_ROOT = MEMORY_ROOT / "steps"
LOG_ROOT = EXT_ROOT / "12_logs"

PART11F_FEATURES = FEATURE_ROOT / "11F_model_selection_weekly_features.csv"
PART11E_PREDICTIONS = BASELINE_ROOT / "11E_preholdout_weekly_baseline_predictions.csv"
PART11I_DESIGN = FINAL_MODEL_ROOT / "11I_final_weekly_forecasting_design.json"
PART11I_LOCK = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.json"
PART11I_LOCK_SHA = CHECKPOINT_ROOT / "11I_final_weekly_selection_lock.sha256"

PART11IA1_OUTPUT_DIR = VALIDATION_ROOT / "11IA1_expanded_weekly_challenge"
PART11IA1_CANDIDATE_AUDIT = PART11IA1_OUTPUT_DIR / "11IA1_candidate_audit.csv"
PART11IA1_POOLED_METRICS = PART11IA1_OUTPUT_DIR / "11IA1_expanded_candidate_pooled_metrics.csv"
PART11IA1_COMPARISON = PART11IA1_OUTPUT_DIR / "11IA1_challenger_vs_locked_benchmark.csv"
PART11IA1_CONTRACT = PART11IA1_OUTPUT_DIR / "11IA1_expanded_weekly_challenge_contract.json"
PART11IA1_LOCK = CHECKPOINT_ROOT / "11IA1_expanded_weekly_challenge_lock.json"
PART11IA1_LOCK_SHA = CHECKPOINT_ROOT / "11IA1_expanded_weekly_challenge_lock.sha256"

PART11C_LOCK = CHECKPOINT_ROOT / "11C_weekly_dataset_lock.json"
PART11C_LOCK_SHA = CHECKPOINT_ROOT / "11C_weekly_dataset_lock.sha256"
PART11C_SOURCE_AMENDMENT_LOCK = SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_lock.json"
PART11C_SOURCE_AMENDMENT_LOCK_SHA = SNAPSHOT_MANIFEST_ROOT / "11C_source_amendment_lock.sha256"

DAILY_MODEL_PANEL = (
    SNAPSHOT_DATA_ROOT
    / "11C_forecasting_preparation_source"
    / "UL_EDEN_forecasting_preparation_final_model_dataset.csv"
)
CANONICAL_DAILY_SOURCE = (
    SNAPSHOT_DATA_ROOT / "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"
)
DAILY_FEATURE_CONTRACT = (
    SNAPSHOT_CONTRACT_ROOT
    / "11C_forecasting_preparation"
    / "17_frozen_model_feature_contract.csv"
)

OUTPUT_DIR = VALIDATION_ROOT / "11IA2_tuned_and_daily_challenge"
PACKAGE_AUDIT_PATH = OUTPUT_DIR / "11IA2_package_audit.csv"
WEEKLY_CONFIG_REGISTRY_PATH = OUTPUT_DIR / "11IA2_weekly_configuration_registry.csv"
WEEKLY_CONFIG_PREDICTIONS_PATH = OUTPUT_DIR / "11IA2_weekly_configuration_predictions.csv"
WEEKLY_SELECTION_AUDIT_PATH = OUTPUT_DIR / "11IA2_prequential_configuration_selection.csv"
WEEKLY_TUNED_PREDICTIONS_PATH = OUTPUT_DIR / "11IA2_weekly_tuned_predictions.csv"
CALIBRATION_AUDIT_PATH = OUTPUT_DIR / "11IA2_prequential_calibration_audit.csv"
BLEND_AUDIT_PATH = OUTPUT_DIR / "11IA2_prequential_blend_audit.csv"
DAILY_SOURCE_AUDIT_PATH = OUTPUT_DIR / "11IA2_daily_source_audit.csv"
DAILY_FEATURE_AUDIT_PATH = OUTPUT_DIR / "11IA2_daily_feature_audit.csv"
DAILY_FOLD_AUDIT_PATH = OUTPUT_DIR / "11IA2_daily_fold_execution_audit.csv"
DAILY_PREDICTIONS_PATH = OUTPUT_DIR / "11IA2_daily_to_weekly_predictions.csv"
ALL_PREDICTIONS_PATH = OUTPUT_DIR / "11IA2_all_method_predictions.csv"
POOLED_METRICS_PATH = OUTPUT_DIR / "11IA2_primary_scope_pooled_metrics.csv"
FOLD_METRICS_PATH = OUTPUT_DIR / "11IA2_primary_scope_fold_metrics.csv"
PAIRWISE_AUDIT_PATH = OUTPUT_DIR / "11IA2_pairwise_weekly_error_audit.csv"
FINAL_COMPARISON_PATH = OUTPUT_DIR / "11IA2_final_challenger_comparison.csv"
FINAL_DECISION_PATH = OUTPUT_DIR / "11IA2_final_method_decision.json"
FINAL_DESIGN_AMENDMENT_PATH = (
    FINAL_MODEL_ROOT / "11IA2_final_weekly_forecasting_design_amendment.json"
)
FINAL_DESIGN_TABLE_PATH = (
    FINAL_MODEL_ROOT / "11IA2_final_weekly_forecasting_design_amendment.csv"
)
VALIDATION_PATH = OUTPUT_DIR / "11IA2_validation.csv"
CONTRACT_PATH = OUTPUT_DIR / "11IA2_challenge_contract.json"
MANIFEST_PATH = OUTPUT_DIR / "11IA2_output_hash_manifest.csv"
STEP_MEMORY_PATH = STEP_MEMORY_ROOT / "STEP_11IA2_TUNED_AND_DAILY_WEEKLY_CHALLENGE.md"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "11IA2_checkpoint.json"
CHECKPOINT_SHA_PATH = CHECKPOINT_ROOT / "11IA2_checkpoint.sha256"
LOCK_PATH = CHECKPOINT_ROOT / "11IA2_final_weekly_challenger_amendment_lock.json"
LOCK_SHA_PATH = CHECKPOINT_ROOT / "11IA2_final_weekly_challenger_amendment_lock.sha256"
LOG_PATH = LOG_ROOT / "11IA2_tuned_and_daily_weekly_challenge_log.txt"

STEP_ID = "11IA2"
STATUS = "PART_11IA2_COMPLETED_READY_FOR_11J"
NOW_UTC = datetime.now(timezone.utc)
NOW_LOCAL = NOW_UTC.astimezone(ZoneInfo("Europe/Dublin"))

TARGET_ID = "WEEKLY_NORMAL_DEMAND"
TARGET_COLUMN = "WeeklyNormalDemand"
DAILY_TARGET_COLUMN = "DailyNormalDemand"
LOCKED_BENCHMARK_METHOD = "NAIVE_LAST_CONTEXT"
PRIMARY_SCOPE_ID = "FOLD_STRICT_95_PERCENT"
PRIMARY_SCOPE_FLAG = "PriorStrict95PctScope"
MINIMUM_TRAINING_WEEKS = 8
PREQUENTIAL_WARMUP_FOLDS = 4
RANDOM_SEED = 6073
HOLDOUT_START = pd.Timestamp("2026-03-02")

CALIBRATION_FACTOR_MIN = 0.75
CALIBRATION_FACTOR_MAX = 1.30
BLEND_WEIGHTS = [0.00, 0.25, 0.50, 0.75, 1.00]
DEFAULT_BLEND_WEIGHT = 0.50

MINIMUM_WAPE_IMPROVEMENT_POINTS = 0.50
MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE = 3.00
MAXIMUM_RMSE_RATIO_TO_BENCHMARK = 1.10
MINIMUM_PAIRWISE_WEEK_WINS = 20

BASELINE_SCOPE_ALIASES = {
    "InFoldStrict80PctScope": "PriorStrict80PctScope",
    "InFoldStrict90PctScope": "PriorStrict90PctScope",
    "InFoldStrict95PctScope": "PriorStrict95PctScope",
}
CANONICAL_SCOPE_COLUMNS = [
    "PriorStrict80PctScope",
    "PriorStrict90PctScope",
    "PriorStrict95PctScope",
]
KEY_COLUMNS = ["WeekStartDate", "CanonicalProductID"]

MEMORY_FILES = {
    "PROJECT_CONTEXT": MEMORY_ROOT / "PROJECT_CONTEXT.md",
    "WORKFLOW": MEMORY_ROOT / "WORKFLOW.md",
    "DECISIONS": MEMORY_ROOT / "DECISIONS.md",
    "FILES_AND_PATHS": MEMORY_ROOT / "FILES_AND_PATHS.md",
    "METRICS_AND_RESULTS": MEMORY_ROOT / "METRICS_AND_RESULTS.md",
    "CHAT_INDEX": MEMORY_ROOT / "CHAT_INDEX.md",
    "CURRENT_HANDOFF": MEMORY_ROOT / "CURRENT_HANDOFF.md",
}

NEW_OUTPUTS = [
    PACKAGE_AUDIT_PATH,
    WEEKLY_CONFIG_REGISTRY_PATH,
    WEEKLY_CONFIG_PREDICTIONS_PATH,
    WEEKLY_SELECTION_AUDIT_PATH,
    WEEKLY_TUNED_PREDICTIONS_PATH,
    CALIBRATION_AUDIT_PATH,
    BLEND_AUDIT_PATH,
    DAILY_SOURCE_AUDIT_PATH,
    DAILY_FEATURE_AUDIT_PATH,
    DAILY_FOLD_AUDIT_PATH,
    DAILY_PREDICTIONS_PATH,
    ALL_PREDICTIONS_PATH,
    POOLED_METRICS_PATH,
    FOLD_METRICS_PATH,
    PAIRWISE_AUDIT_PATH,
    FINAL_COMPARISON_PATH,
    FINAL_DECISION_PATH,
    FINAL_DESIGN_AMENDMENT_PATH,
    FINAL_DESIGN_TABLE_PATH,
    VALIDATION_PATH,
    CONTRACT_PATH,
    MANIFEST_PATH,
    STEP_MEMORY_PATH,
    CHECKPOINT_PATH,
    CHECKPOINT_SHA_PATH,
    LOCK_PATH,
    LOCK_SHA_PATH,
    LOG_PATH,
]

# =============================================================================
# GENERAL HELPERS
# =============================================================================


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def require_directory(path: Path, label: str) -> None:
    if not path.is_dir():
        raise FileNotFoundError(f"Missing required {label}:\n{path}")


def verify_sidecar(path: Path, sidecar: Path, label: str) -> str:
    require_file(path, label)
    require_file(sidecar, f"{label} SHA-256 sidecar")
    expected = sidecar.read_text(encoding="utf-8").strip().split()[0].lower()
    actual = sha256_file(path)
    if expected != actual:
        raise AssertionError(
            f"{label} SHA-256 mismatch:\nExpected: {expected}\nActual:   {actual}"
        )
    return actual


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        frame.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_json(path: Path, payload: dict) -> None:
    atomic_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def stage_path(stage_root: Path, final_path: Path) -> Path:
    return stage_root / final_path.relative_to(EXT_ROOT)


def stage_text(stage_root: Path, final_path: Path, text: str) -> None:
    atomic_text(stage_path(stage_root, final_path), text)


def stage_csv(stage_root: Path, final_path: Path, frame: pd.DataFrame) -> None:
    atomic_csv(stage_path(stage_root, final_path), frame)


def stage_json(stage_root: Path, final_path: Path, payload: dict) -> None:
    atomic_json(stage_path(stage_root, final_path), payload)


def make_read_only(path: Path) -> None:
    if path.is_file():
        path.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)


def update_section(text: str, marker: str, heading: str, body: str) -> str:
    start = f"<!-- BEGIN {marker} -->"
    end = f"<!-- END {marker} -->"
    section = f"{start}\n## {heading}\n\n{body.rstrip()}\n{end}"
    if start in text and end in text:
        before = text.split(start, 1)[0].rstrip()
        after = text.split(end, 1)[1].lstrip()
        return before + "\n\n" + section + ("\n\n" + after if after else "") + "\n"
    return text.rstrip() + "\n\n" + section + "\n"


def bool_series(series: pd.Series, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    normalized = series.astype(str).str.strip().str.lower()
    invalid = sorted(set(normalized) - set(mapping))
    if invalid:
        raise ValueError(f"{label} has invalid Boolean values: {invalid[:10]}")
    return normalized.map(mapping).astype(bool)


def normalize_scope_columns(
    frame: pd.DataFrame,
    frame_label: str,
    aliases: dict[str, str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    aliases = aliases or {}
    rename_map: dict[str, str] = {}
    for source, target in aliases.items():
        if source in result.columns and target in result.columns:
            source_values = bool_series(result[source], f"{frame_label}.{source}")
            target_values = bool_series(result[target], f"{frame_label}.{target}")
            if not source_values.equals(target_values):
                raise AssertionError(
                    f"{frame_label} has conflicting {source} and {target} values"
                )
            result = result.drop(columns=[source])
        elif source in result.columns:
            rename_map[source] = target
    if rename_map:
        result = result.rename(columns=rename_map)
    missing = [column for column in CANONICAL_SCOPE_COLUMNS if column not in result]
    if missing:
        raise AssertionError(
            f"{frame_label} is missing normalized scope columns: {missing}"
        )
    for column in CANONICAL_SCOPE_COLUMNS:
        result[column] = bool_series(result[column], f"{frame_label}.{column}")
    return result


def metric_record(actual: pd.Series | np.ndarray, prediction: pd.Series | np.ndarray) -> dict:
    actual_array = np.asarray(actual, dtype=float).reshape(-1)
    prediction_array = np.asarray(prediction, dtype=float).reshape(-1)
    if len(actual_array) != len(prediction_array):
        raise AssertionError("Actual and prediction lengths differ")
    if not np.isfinite(actual_array).all() or not np.isfinite(prediction_array).all():
        raise AssertionError("Non-finite values entered metric calculation")
    error = prediction_array - actual_array
    absolute_error = np.abs(error)
    denominator = float(np.abs(actual_array).sum())
    return {
        "Observations": int(len(actual_array)),
        "ActualTotal": float(actual_array.sum()),
        "PredictedTotal": float(prediction_array.sum()),
        "MAE": float(absolute_error.mean()) if len(actual_array) else np.nan,
        "RMSE": float(np.sqrt(np.mean(error ** 2))) if len(actual_array) else np.nan,
        "WAPEPercentage": (
            float(100.0 * absolute_error.sum() / denominator)
            if denominator != 0
            else np.nan
        ),
        "MeanBias": float(error.mean()) if len(actual_array) else np.nan,
        "TotalBias": float(error.sum()),
    }


def package_version(distribution_name: str) -> str | None:
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return None


def make_ohe() -> OneHotEncoder:
    parameters = inspect.signature(OneHotEncoder).parameters
    kwargs = {"handle_unknown": "ignore"}
    if "sparse_output" in parameters:
        kwargs["sparse_output"] = False
    else:
        kwargs["sparse"] = False
    return OneHotEncoder(**kwargs)


def make_imputer(strategy: str = "median", fill_value=None) -> SimpleImputer:
    parameters = inspect.signature(SimpleImputer).parameters
    kwargs = {"strategy": strategy}
    if fill_value is not None:
        kwargs["fill_value"] = fill_value
    if "keep_empty_features" in parameters:
        kwargs["keep_empty_features"] = True
    return SimpleImputer(**kwargs)


def recursively_find_predictor_lists(
    obj,
    path: tuple[str, ...] = (),
) -> list[tuple[str, list[str]]]:
    found: list[tuple[str, list[str]]] = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            found.extend(recursively_find_predictor_lists(value, path + (str(key),)))
    elif isinstance(obj, list) and obj and all(isinstance(item, str) for item in obj):
        joined = ".".join(path).lower()
        if "predict" in joined and "normal" in joined:
            found.append((".".join(path), obj))
    return found


def discover_normal_predictors(frame: pd.DataFrame) -> tuple[list[str], str, str | None]:
    candidates = sorted(FEATURE_ROOT.glob("*11F*contract*.json"))
    candidates += sorted(FEATURE_ROOT.glob("**/*11F*contract*.json"))
    candidates = list(dict.fromkeys(candidates))
    for contract_path in candidates:
        try:
            payload = json.loads(contract_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        lists = [
            (name, values)
            for name, values in recursively_find_predictor_lists(payload)
            if all(value in frame.columns for value in values)
        ]
        if lists:
            lists.sort(key=lambda item: (abs(len(item[1]) - 48), item[0]))
            selected_name, selected_values = lists[0]
            return list(selected_values), f"CONTRACT:{selected_name}", str(contract_path)
    excluded = {
        "WeekStartDate",
        "WeekEndDate",
        "WeekID",
        "CanonicalProductID",
        "CanonicalProductName",
        "TargetID",
        TARGET_COLUMN,
        "WeeklyTotalDemand",
        "WeeklyBulkDemand",
        "Actual",
        "Prediction",
        "WeeklyTuningEligible",
        "IsOpenedMarchHoldout",
        *CANONICAL_SCOPE_COLUMNS,
    }
    predictors = [
        column
        for column in frame.columns
        if column not in excluded
        and "Scope" not in column
        and (
            pd.api.types.is_numeric_dtype(frame[column])
            or pd.api.types.is_bool_dtype(frame[column])
        )
    ]
    if not predictors:
        raise AssertionError("No normal-demand weekly predictors were discovered")
    return predictors, "FALLBACK_NUMERIC_SCHEMA", None


def method_complexity(method_id: str) -> int:
    order = {
        LOCKED_BENCHMARK_METHOD: 0,
        "TUNED_RANDOM_FOREST_POISSON": 20,
        "CALIBRATED_RANDOM_FOREST_POISSON": 21,
        "BLEND_CALIBRATED_RANDOM_FOREST_POISSON_WITH_NAIVE": 22,
        "TUNED_XGBOOST_POISSON": 30,
        "CALIBRATED_XGBOOST_POISSON": 31,
        "BLEND_CALIBRATED_XGBOOST_POISSON_WITH_NAIVE": 32,
        "TUNED_CATBOOST_RMSE": 40,
        "CALIBRATED_CATBOOST_RMSE": 41,
        "BLEND_CALIBRATED_CATBOOST_RMSE_WITH_NAIVE": 42,
        "DAILY_NAIVE5_WEEKLY_ORIGIN": 50,
        "DAILY_HURDLE_NAIVE5_H75_WEEKLY_ORIGIN": 51,
        "CALIBRATED_DAILY_HURDLE_NAIVE5_H75_WEEKLY_ORIGIN": 52,
        "BLEND_CALIBRATED_DAILY_HURDLE_WITH_WEEKLY_NAIVE": 53,
        "DAILY_NAIVE5_ACTUAL_UPDATE_DIAGNOSTIC": 90,
        "DAILY_HURDLE_NAIVE5_H75_ACTUAL_UPDATE_DIAGNOSTIC": 91,
    }
    return order.get(method_id, 999)


# =============================================================================
# INPUT AND LOCK VERIFICATION
# =============================================================================

for directory, label in [
    (EXT_ROOT, "extension root"),
    (SNAPSHOT_ROOT, "source snapshot"),
    (FEATURE_ROOT, "weekly feature directory"),
    (BASELINE_ROOT, "weekly baseline directory"),
    (MODEL_ROOT, "weekly model directory"),
    (FINAL_MODEL_ROOT, "final model directory"),
    (VALIDATION_ROOT, "weekly validation directory"),
    (CHECKPOINT_ROOT, "checkpoint directory"),
    (MEMORY_ROOT, "project memory directory"),
    (STEP_MEMORY_ROOT, "step-memory directory"),
    (LOG_ROOT, "log directory"),
]:
    require_directory(directory, label)

for path, label in [
    (PART11F_FEATURES, "Part 11F weekly features"),
    (PART11E_PREDICTIONS, "Part 11E baseline predictions"),
    (PART11I_DESIGN, "Part 11I final design"),
    (PART11IA1_CANDIDATE_AUDIT, "Part 11IA1 candidate audit"),
    (PART11IA1_POOLED_METRICS, "Part 11IA1 pooled metrics"),
    (PART11IA1_COMPARISON, "Part 11IA1 challenger comparison"),
    (PART11IA1_CONTRACT, "Part 11IA1 contract"),
    (DAILY_MODEL_PANEL, "locked daily model panel"),
    (CANONICAL_DAILY_SOURCE, "locked canonical daily demand source"),
    (DAILY_FEATURE_CONTRACT, "locked daily feature contract"),
]:
    require_file(path, label)

for key, path in MEMORY_FILES.items():
    require_file(path, f"memory file {key}")

if LOCK_PATH.exists() or LOCK_SHA_PATH.exists():
    raise FileExistsError(f"Part 11IA2 overwrite lock triggered:\n{LOCK_PATH}")

existing_outputs = [path for path in NEW_OUTPUTS if path.exists()]
if existing_outputs:
    raise FileExistsError(
        "Existing Part 11IA2 outputs found; no files changed:\n"
        + "\n".join(f"- {path}" for path in existing_outputs)
    )

for old_stage in EXT_ROOT.glob(".11IA2_staging_*"):
    if old_stage.is_dir():
        shutil.rmtree(old_stage)

part11i_lock_hash = verify_sidecar(PART11I_LOCK, PART11I_LOCK_SHA, "Part 11I lock")
part11ia1_lock_hash = verify_sidecar(
    PART11IA1_LOCK,
    PART11IA1_LOCK_SHA,
    "Part 11IA1 lock",
)
part11c_lock_hash = verify_sidecar(PART11C_LOCK, PART11C_LOCK_SHA, "Part 11C lock")
part11c_amendment_hash = verify_sidecar(
    PART11C_SOURCE_AMENDMENT_LOCK,
    PART11C_SOURCE_AMENDMENT_LOCK_SHA,
    "Part 11C source-amendment lock",
)

part11i_lock = json.loads(PART11I_LOCK.read_text(encoding="utf-8"))
part11i_design = json.loads(PART11I_DESIGN.read_text(encoding="utf-8"))
part11ia1_lock = json.loads(PART11IA1_LOCK.read_text(encoding="utf-8"))
part11ia1_contract = json.loads(PART11IA1_CONTRACT.read_text(encoding="utf-8"))
part11c_lock = json.loads(PART11C_LOCK.read_text(encoding="utf-8"))

if part11i_lock.get("Status") != "PART_11I_COMPLETED_READY_FOR_11J":
    raise AssertionError(f"Unexpected Part 11I status: {part11i_lock.get('Status')}")
if part11ia1_lock.get("Status") != "PART_11IA1_COMPLETED_READY_FOR_11IA2":
    raise AssertionError(f"Unexpected Part 11IA1 status: {part11ia1_lock.get('Status')}")
if part11ia1_lock.get("ReadyForPart11IA2") is not True:
    raise AssertionError("Part 11IA1 lock does not authorise Part 11IA2")
if part11ia1_lock.get("Part11ILockSHA256") != part11i_lock_hash:
    raise AssertionError("Part 11IA1 does not reference the current Part 11I lock")
if part11c_lock.get("Status") != "PART_11C_COMPLETED_READY_FOR_11D":
    raise AssertionError(f"Unexpected Part 11C status: {part11c_lock.get('Status')}")

locked_design = part11i_lock.get("FinalForecastingDesign", {})
if locked_design.get("TargetID") != TARGET_ID:
    raise AssertionError("Part 11I target is not WEEKLY_NORMAL_DEMAND")
if locked_design.get("MethodID") != LOCKED_BENCHMARK_METHOD:
    raise AssertionError("Unexpected Part 11I benchmark method")
if locked_design.get("ScopeID") != PRIMARY_SCOPE_ID:
    raise AssertionError("Unexpected Part 11I primary scope")
if locked_design.get("BulkIncludedInModelledTarget") is not False:
    raise AssertionError("Part 11I did not exclude bulk")
if sha256_file(PART11I_DESIGN) != locked_design.get("SHA256"):
    raise AssertionError("Part 11I final-design hash changed")

proceeding = set(part11ia1_lock.get("ProceedingTo11IA2", []))
expected_proceeding = {
    "XGBOOST_POISSON",
    "RANDOM_FOREST_POISSON",
    "CATBOOST_RMSE",
}
if proceeding != expected_proceeding:
    raise AssertionError(
        f"Unexpected Part 11IA1 proceeding candidates: {sorted(proceeding)}"
    )

for key, expected in {
    "OpenedMarch2026TargetsRead": False,
    "BulkIncludedInModelTarget": False,
    "Part11ILockModified": False,
    "FinalReplacementSelected": False,
}.items():
    actual = part11ia1_lock.get("SafetyAssertions", {}).get(key)
    if actual is not expected:
        raise AssertionError(f"Invalid Part 11IA1 safety assertion {key}: {actual}")

# =============================================================================
# PACKAGE AUDIT
# =============================================================================

package_rows = []
for distribution, import_name, required in [
    ("numpy", "numpy", True),
    ("pandas", "pandas", True),
    ("scikit-learn", "sklearn", True),
    ("xgboost", "xgboost", True),
    ("catboost", "catboost", True),
]:
    version = package_version(distribution)
    import_succeeded = False
    import_error = ""
    if version is not None:
        try:
            __import__(import_name)
            import_succeeded = True
        except Exception as exc:
            import_error = f"{type(exc).__name__}: {exc}"
    package_rows.append(
        {
            "Distribution": distribution,
            "ImportName": import_name,
            "Required": required,
            "Installed": version is not None,
            "Version": version or "",
            "ImportSucceeded": import_succeeded,
            "ImportError": import_error,
        }
    )
package_audit = pd.DataFrame(package_rows)
if not package_audit.loc[package_audit["Required"], "ImportSucceeded"].all():
    raise ImportError(
        "Part 11IA2 required package audit failed:\n"
        + package_audit.loc[
            package_audit["Required"] & ~package_audit["ImportSucceeded"]
        ].to_string(index=False)
    )

from xgboost import XGBClassifier, XGBRegressor
from catboost import CatBoostRegressor

# =============================================================================
# LOAD WEEKLY FEATURES AND BASELINE
# =============================================================================

features = pd.read_csv(PART11F_FEATURES, low_memory=False)
baseline_predictions = pd.read_csv(PART11E_PREDICTIONS, low_memory=False)

for frame, label in [
    (features, "Part 11F features"),
    (baseline_predictions, "Part 11E predictions"),
]:
    frame["WeekStartDate"] = pd.to_datetime(frame["WeekStartDate"], errors="raise")
    if "WeekEndDate" in frame.columns:
        frame["WeekEndDate"] = pd.to_datetime(frame["WeekEndDate"], errors="raise")
    frame["CanonicalProductID"] = frame["CanonicalProductID"].astype(str)

features = normalize_scope_columns(features, "Part 11F features")
baseline_predictions = normalize_scope_columns(
    baseline_predictions,
    "Part 11E predictions",
    aliases=BASELINE_SCOPE_ALIASES,
)

required_feature_columns = {
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    TARGET_COLUMN,
    *CANONICAL_SCOPE_COLUMNS,
}
if not required_feature_columns.issubset(features.columns):
    raise AssertionError(
        "Part 11F feature schema is incomplete: "
        f"{sorted(required_feature_columns - set(features.columns))}"
    )

predictor_columns, predictor_source, predictor_contract_path = discover_normal_predictors(
    features
)
forbidden_predictors = {
    TARGET_COLUMN,
    "WeeklyTotalDemand",
    "WeeklyBulkDemand",
    "Actual",
    "Prediction",
    *CANONICAL_SCOPE_COLUMNS,
}
leakage_predictors = sorted(
    column
    for column in predictor_columns
    if column in forbidden_predictors or "Scope" in column
)
if leakage_predictors:
    raise AssertionError(f"Leakage predictors found: {leakage_predictors}")

numeric_predictors = [
    column
    for column in predictor_columns
    if column in features.columns
]
for column in numeric_predictors:
    features[column] = pd.to_numeric(features[column], errors="coerce")
features[numeric_predictors] = features[numeric_predictors].replace(
    [np.inf, -np.inf], np.nan
)
empty_weekly_predictors = [
    column for column in numeric_predictors if features[column].notna().sum() == 0
]
numeric_predictors = [
    column for column in numeric_predictors if column not in empty_weekly_predictors
]
if not numeric_predictors:
    raise AssertionError("No usable weekly predictors remain")

weeks = sorted(features["WeekStartDate"].drop_duplicates())
if len(weeks) != 47:
    raise AssertionError(f"Expected 47 pre-holdout weeks, found {len(weeks)}")
forecast_weeks = weeks[MINIMUM_TRAINING_WEEKS:]
if len(forecast_weeks) != 39:
    raise AssertionError(f"Expected 39 forecast weeks, found {len(forecast_weeks)}")
if max(forecast_weeks) >= HOLDOUT_START:
    raise AssertionError("Forecast-week design entered the opened March 2026 period")

locked_baseline = baseline_predictions.loc[
    (baseline_predictions["TargetID"].astype(str) == TARGET_ID)
    & (
        baseline_predictions["BaselineMethod"].astype(str)
        == LOCKED_BENCHMARK_METHOD
    )
    & baseline_predictions[PRIMARY_SCOPE_FLAG]
].copy()
if len(locked_baseline) != 3351:
    raise AssertionError(
        f"Expected 3,351 locked 95% baseline rows, found {len(locked_baseline)}"
    )
locked_baseline["Actual"] = pd.to_numeric(
    locked_baseline["Actual"], errors="raise"
).astype(float)
locked_baseline["Prediction"] = pd.to_numeric(
    locked_baseline["Prediction"], errors="raise"
).astype(float)
locked_baseline["MethodID"] = LOCKED_BENCHMARK_METHOD
locked_baseline["MethodFamily"] = "LOCKED_REFERENCE_BASELINE"
locked_baseline["SelectionEligible"] = True
locked_baseline["ForecastOrigin"] = "WEEK_START"
locked_baseline["ConfigurationID"] = "LOCKED_PART_11I"
locked_baseline["CalibrationFactor"] = 1.0
locked_baseline["BlendWeightOnCandidate"] = 0.0

feature_actual = features.loc[
    features["WeekStartDate"].isin(forecast_weeks) & features[PRIMARY_SCOPE_FLAG],
    ["WeekStartDate", "CanonicalProductID", TARGET_COLUMN],
].copy()
reconciliation = locked_baseline.merge(
    feature_actual.rename(columns={TARGET_COLUMN: "FeatureActual"}),
    on=KEY_COLUMNS,
    how="inner",
    validate="one_to_one",
)
if len(reconciliation) != len(locked_baseline):
    raise AssertionError("Baseline and feature rows do not reconcile one-to-one")
if np.abs(reconciliation["Actual"] - reconciliation["FeatureActual"]).max() > 1e-9:
    raise AssertionError("Part 11E baseline actuals differ from Part 11F targets")

weekly_preprocessor = ColumnTransformer(
    transformers=[
        (
            "product",
            Pipeline(
                [
                    ("imputer", make_imputer("constant", "__MISSING__")),
                    ("one_hot", make_ohe()),
                ]
            ),
            ["CanonicalProductID"],
        ),
        (
            "numeric",
            Pipeline([("imputer", make_imputer("median"))]),
            numeric_predictors,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

# =============================================================================
# WEEKLY CONFIGURATION REGISTRY
# =============================================================================

weekly_configs: list[dict] = []


def register_weekly_config(
    family: str,
    config_id: str,
    default: bool,
    complexity: int,
    parameters: dict,
) -> None:
    weekly_configs.append(
        {
            "Family": family,
            "ConfigurationID": config_id,
            "DefaultConfiguration": default,
            "ComplexityOrder": complexity,
            "Parameters": parameters,
        }
    )


register_weekly_config(
    "XGBOOST_POISSON",
    "XGB_P1_BASE",
    True,
    1,
    {
        "objective": "count:poisson",
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 6,
        "min_child_weight": 5,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.0,
        "reg_lambda": 2.0,
        "max_delta_step": 0.7,
    },
)
register_weekly_config(
    "XGBOOST_POISSON",
    "XGB_P2_SHALLOW",
    False,
    2,
    {
        "objective": "count:poisson",
        "n_estimators": 450,
        "learning_rate": 0.04,
        "max_depth": 4,
        "min_child_weight": 3,
        "subsample": 0.85,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 3.0,
        "max_delta_step": 0.7,
    },
)
register_weekly_config(
    "XGBOOST_POISSON",
    "XGB_P3_REGULARISED",
    False,
    3,
    {
        "objective": "count:poisson",
        "n_estimators": 650,
        "learning_rate": 0.025,
        "max_depth": 5,
        "min_child_weight": 8,
        "subsample": 0.75,
        "colsample_bytree": 0.75,
        "reg_alpha": 0.10,
        "reg_lambda": 5.0,
        "max_delta_step": 0.7,
    },
)
register_weekly_config(
    "XGBOOST_POISSON",
    "XGB_P4_DEEP",
    False,
    4,
    {
        "objective": "count:poisson",
        "n_estimators": 600,
        "learning_rate": 0.025,
        "max_depth": 7,
        "min_child_weight": 5,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.0,
        "reg_lambda": 4.0,
        "max_delta_step": 0.7,
    },
)

register_weekly_config(
    "RANDOM_FOREST_POISSON",
    "RF_P1_BASE",
    True,
    1,
    {
        "n_estimators": 400,
        "criterion": "poisson",
        "max_depth": None,
        "min_samples_split": 4,
        "min_samples_leaf": 2,
        "max_features": 0.85,
        "bootstrap": False,
    },
)
register_weekly_config(
    "RANDOM_FOREST_POISSON",
    "RF_P2_FINE_LEAVES",
    False,
    2,
    {
        "n_estimators": 450,
        "criterion": "poisson",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 0.70,
        "bootstrap": False,
    },
)
register_weekly_config(
    "RANDOM_FOREST_POISSON",
    "RF_P3_DEPTH_LIMITED",
    False,
    3,
    {
        "n_estimators": 450,
        "criterion": "poisson",
        "max_depth": 18,
        "min_samples_split": 4,
        "min_samples_leaf": 2,
        "max_features": 1.00,
        "bootstrap": False,
    },
)
register_weekly_config(
    "RANDOM_FOREST_POISSON",
    "RF_P4_SMOOTH",
    False,
    4,
    {
        "n_estimators": 500,
        "criterion": "poisson",
        "max_depth": None,
        "min_samples_split": 8,
        "min_samples_leaf": 4,
        "max_features": 0.60,
        "bootstrap": False,
    },
)

register_weekly_config(
    "CATBOOST_RMSE",
    "CAT_R1_BASE",
    True,
    1,
    {
        "loss_function": "RMSE",
        "iterations": 500,
        "learning_rate": 0.03,
        "depth": 7,
        "l2_leaf_reg": 3.0,
        "random_strength": 1.0,
    },
)
register_weekly_config(
    "CATBOOST_RMSE",
    "CAT_R2_SHALLOW",
    False,
    2,
    {
        "loss_function": "RMSE",
        "iterations": 700,
        "learning_rate": 0.025,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 0.5,
    },
)
register_weekly_config(
    "CATBOOST_RMSE",
    "CAT_R3_DEEP",
    False,
    3,
    {
        "loss_function": "RMSE",
        "iterations": 450,
        "learning_rate": 0.035,
        "depth": 8,
        "l2_leaf_reg": 6.0,
        "random_strength": 1.0,
    },
)
register_weekly_config(
    "CATBOOST_RMSE",
    "CAT_R4_COMPACT",
    False,
    4,
    {
        "loss_function": "RMSE",
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 5,
        "l2_leaf_reg": 4.0,
        "random_strength": 0.8,
    },
)

weekly_config_registry = pd.DataFrame(
    [
        {
            "Family": spec["Family"],
            "ConfigurationID": spec["ConfigurationID"],
            "DefaultConfiguration": spec["DefaultConfiguration"],
            "ComplexityOrder": spec["ComplexityOrder"],
            "ParametersJSON": json.dumps(spec["Parameters"], sort_keys=True),
        }
        for spec in weekly_configs
    ]
)
if weekly_config_registry.groupby("Family")["DefaultConfiguration"].sum().ne(1).any():
    raise AssertionError("Each weekly family must have exactly one default configuration")


def build_weekly_estimator(spec: dict):
    parameters = dict(spec["Parameters"])
    family = spec["Family"]
    if family == "XGBOOST_POISSON":
        return XGBRegressor(
            **parameters,
            n_jobs=-1,
            random_state=RANDOM_SEED,
            tree_method="hist",
            verbosity=0,
        )
    if family == "RANDOM_FOREST_POISSON":
        return RandomForestRegressor(
            **parameters,
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )
    raise KeyError(f"Standard estimator requested for unsupported family {family}")


def run_weekly_standard(spec: dict, train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    pipeline = Pipeline(
        [
            ("preprocess", clone(weekly_preprocessor)),
            ("model", build_weekly_estimator(spec)),
        ]
    )
    columns = ["CanonicalProductID"] + numeric_predictors
    pipeline.fit(train[columns], train[TARGET_COLUMN].to_numpy(dtype=float))
    return np.clip(np.asarray(pipeline.predict(valid[columns]), dtype=float), 0.0, None)


def run_weekly_catboost(spec: dict, train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    usable_numeric = [
        column
        for column in numeric_predictors
        if np.isfinite(train[column].to_numpy(dtype=float)).any()
    ]
    columns = ["CanonicalProductID"] + usable_numeric
    x_train = train[columns].copy()
    x_valid = valid[columns].copy()
    x_train["CanonicalProductID"] = x_train["CanonicalProductID"].astype(str)
    x_valid["CanonicalProductID"] = x_valid["CanonicalProductID"].astype(str)
    model = CatBoostRegressor(
        **spec["Parameters"],
        random_seed=RANDOM_SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    model.fit(
        x_train,
        train[TARGET_COLUMN].to_numpy(dtype=float),
        cat_features=[0],
    )
    return np.clip(np.asarray(model.predict(x_valid), dtype=float), 0.0, None)

# =============================================================================
# RUN ALL WEEKLY CONFIGURATIONS ON ALL 39 OUTER FOLDS
# =============================================================================

weekly_config_prediction_rows: list[pd.DataFrame] = []
weekly_config_failure_rows: list[dict] = []

for spec in weekly_configs:
    print("-" * 100)
    print(
        f"Running weekly configuration: {spec['Family']} / "
        f"{spec['ConfigurationID']}"
    )
    successful_folds = 0
    failure = ""
    for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
        train = features.loc[features["WeekStartDate"] < forecast_week].copy()
        valid = features.loc[features["WeekStartDate"] == forecast_week].copy()
        if train["WeekStartDate"].nunique() < MINIMUM_TRAINING_WEEKS:
            raise AssertionError("Weekly outer fold has insufficient training weeks")
        try:
            if spec["Family"] == "CATBOOST_RMSE":
                prediction = run_weekly_catboost(spec, train, valid)
            else:
                prediction = run_weekly_standard(spec, train, valid)
        except Exception as exc:
            failure = (
                f"Fold {fold_number}, {forecast_week.date()}: "
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()
            break
        if len(prediction) != len(valid):
            raise AssertionError("Weekly configuration prediction length mismatch")
        output = valid.loc[
            valid[PRIMARY_SCOPE_FLAG],
            [
                "WeekStartDate",
                "WeekEndDate",
                "WeekID",
                "CanonicalProductID",
                "CanonicalProductName",
                TARGET_COLUMN,
            ],
        ].copy()
        output["FoldNumber"] = fold_number
        output["Family"] = spec["Family"]
        output["ConfigurationID"] = spec["ConfigurationID"]
        output["DefaultConfiguration"] = spec["DefaultConfiguration"]
        output["ConfigurationComplexityOrder"] = spec["ComplexityOrder"]
        output["Actual"] = output[TARGET_COLUMN].astype(float)
        output["Prediction"] = prediction[valid[PRIMARY_SCOPE_FLAG].to_numpy()]
        output["Error"] = output["Prediction"] - output["Actual"]
        output["AbsoluteError"] = output["Error"].abs()
        weekly_config_prediction_rows.append(output)
        successful_folds += 1
    weekly_config_failure_rows.append(
        {
            "Family": spec["Family"],
            "ConfigurationID": spec["ConfigurationID"],
            "SuccessfulFolds": successful_folds,
            "ExpectedFolds": len(forecast_weeks),
            "CompletedAllFolds": successful_folds == len(forecast_weeks) and not failure,
            "FailureMessage": failure,
        }
    )

weekly_config_execution = pd.DataFrame(weekly_config_failure_rows)
if not weekly_config_execution["CompletedAllFolds"].all():
    raise RuntimeError(
        "One or more Part 11IA2 weekly configurations failed:\n"
        + weekly_config_execution.loc[
            ~weekly_config_execution["CompletedAllFolds"]
        ].to_string(index=False)
    )

weekly_config_predictions = pd.concat(
    weekly_config_prediction_rows,
    ignore_index=True,
)
if not np.isfinite(weekly_config_predictions["Prediction"]).all():
    raise AssertionError("Weekly configuration predictions contain non-finite values")
if (weekly_config_predictions["Prediction"] < 0).any():
    raise AssertionError("Weekly configuration predictions contain negative values")

# =============================================================================
# PREQUENTIAL CONFIGURATION SELECTION
# =============================================================================

selection_audit_rows: list[dict] = []
weekly_tuned_rows: list[pd.DataFrame] = []

for family, family_specs in weekly_config_registry.groupby("Family", sort=True):
    default_config = str(
        family_specs.loc[
            family_specs["DefaultConfiguration"], "ConfigurationID"
        ].iloc[0]
    )
    for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
        prior = weekly_config_predictions.loc[
            (weekly_config_predictions["Family"] == family)
            & (weekly_config_predictions["WeekStartDate"] < forecast_week)
        ].copy()
        prior_weeks = int(prior["WeekStartDate"].nunique())
        score_rows = []
        if prior_weeks >= PREQUENTIAL_WARMUP_FOLDS:
            for config_id, group in prior.groupby("ConfigurationID", sort=True):
                metrics = metric_record(group["Actual"], group["Prediction"])
                complexity = int(
                    family_specs.loc[
                        family_specs["ConfigurationID"] == config_id,
                        "ComplexityOrder",
                    ].iloc[0]
                )
                score_rows.append(
                    {
                        "ConfigurationID": config_id,
                        "PriorWeeks": prior_weeks,
                        "PriorWAPEPercentage": metrics["WAPEPercentage"],
                        "PriorMAE": metrics["MAE"],
                        "PriorRMSE": metrics["RMSE"],
                        "PriorAbsoluteTotalBias": abs(metrics["TotalBias"]),
                        "ComplexityOrder": complexity,
                    }
                )
            score_table = pd.DataFrame(score_rows).sort_values(
                [
                    "PriorWAPEPercentage",
                    "PriorMAE",
                    "PriorRMSE",
                    "PriorAbsoluteTotalBias",
                    "ComplexityOrder",
                    "ConfigurationID",
                ],
                kind="mergesort",
            )
            selected_config = str(score_table.iloc[0]["ConfigurationID"])
            selection_reason = "LOWEST_PRIOR_PREQUENTIAL_WAPE"
            selected_prior_metrics = score_table.iloc[0].to_dict()
        else:
            selected_config = default_config
            selection_reason = "WARMUP_DEFAULT_CONFIGURATION"
            selected_prior_metrics = {
                "PriorWAPEPercentage": np.nan,
                "PriorMAE": np.nan,
                "PriorRMSE": np.nan,
                "PriorAbsoluteTotalBias": np.nan,
            }
        current = weekly_config_predictions.loc[
            (weekly_config_predictions["Family"] == family)
            & (weekly_config_predictions["ConfigurationID"] == selected_config)
            & (weekly_config_predictions["WeekStartDate"] == forecast_week)
        ].copy()
        expected_rows = int(
            features.loc[
                (features["WeekStartDate"] == forecast_week)
                & features[PRIMARY_SCOPE_FLAG]
            ].shape[0]
        )
        if len(current) != expected_rows:
            raise AssertionError(
                f"Prequential selection row mismatch for {family}, {forecast_week.date()}"
            )
        method_id = f"TUNED_{family}"
        current["MethodID"] = method_id
        current["MethodFamily"] = family
        current["SelectionEligible"] = True
        current["ForecastOrigin"] = "WEEK_START"
        current["CalibrationFactor"] = 1.0
        current["BlendWeightOnCandidate"] = 1.0
        weekly_tuned_rows.append(current)
        selection_audit_rows.append(
            {
                "Family": family,
                "FoldNumber": fold_number,
                "ForecastWeek": forecast_week,
                "PriorForecastWeeksAvailable": prior_weeks,
                "SelectedConfigurationID": selected_config,
                "SelectionReason": selection_reason,
                "SelectedPriorWAPEPercentage": selected_prior_metrics[
                    "PriorWAPEPercentage"
                ],
                "SelectedPriorMAE": selected_prior_metrics["PriorMAE"],
                "SelectedPriorRMSE": selected_prior_metrics["PriorRMSE"],
                "SelectedPriorAbsoluteTotalBias": selected_prior_metrics[
                    "PriorAbsoluteTotalBias"
                ],
            }
        )

weekly_selection_audit = pd.DataFrame(selection_audit_rows)
weekly_tuned_predictions = pd.concat(weekly_tuned_rows, ignore_index=True)

# =============================================================================
# GENERIC PREQUENTIAL CALIBRATION AND BLENDING
# =============================================================================


def prequential_calibrate(
    raw_predictions: pd.DataFrame,
    raw_method_id: str,
    calibrated_method_id: str,
    family: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    output_rows = []
    audit_rows = []
    raw = raw_predictions.loc[raw_predictions["MethodID"] == raw_method_id].copy()
    for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
        prior = raw.loc[raw["WeekStartDate"] < forecast_week]
        prior_weeks = int(prior["WeekStartDate"].nunique())
        if prior_weeks >= PREQUENTIAL_WARMUP_FOLDS:
            denominator = float(prior["Prediction"].sum())
            raw_factor = (
                float(prior["Actual"].sum()) / denominator
                if denominator > 0
                else 1.0
            )
            factor = float(
                np.clip(raw_factor, CALIBRATION_FACTOR_MIN, CALIBRATION_FACTOR_MAX)
            )
            reason = "PRIOR_POOLED_ACTUAL_TO_PREDICTED_RATIO"
        else:
            raw_factor = 1.0
            factor = 1.0
            reason = "WARMUP_NO_CALIBRATION"
        current = raw.loc[raw["WeekStartDate"] == forecast_week].copy()
        current["Prediction"] = np.clip(
            current["Prediction"].to_numpy(dtype=float) * factor,
            0.0,
            None,
        )
        current["Error"] = current["Prediction"] - current["Actual"]
        current["AbsoluteError"] = current["Error"].abs()
        current["MethodID"] = calibrated_method_id
        current["MethodFamily"] = family
        current["SelectionEligible"] = True
        current["ForecastOrigin"] = "WEEK_START"
        current["CalibrationFactor"] = factor
        current["BlendWeightOnCandidate"] = 1.0
        output_rows.append(current)
        audit_rows.append(
            {
                "RawMethodID": raw_method_id,
                "CalibratedMethodID": calibrated_method_id,
                "FoldNumber": fold_number,
                "ForecastWeek": forecast_week,
                "PriorForecastWeeksAvailable": prior_weeks,
                "RawCalibrationFactor": raw_factor,
                "AppliedCalibrationFactor": factor,
                "CalibrationReason": reason,
                "FactorWasClipped": not np.isclose(raw_factor, factor),
            }
        )
    return pd.concat(output_rows, ignore_index=True), pd.DataFrame(audit_rows)


def prequential_blend(
    candidate_predictions: pd.DataFrame,
    candidate_method_id: str,
    blend_method_id: str,
    family: str,
    baseline: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    candidate = candidate_predictions.loc[
        candidate_predictions["MethodID"] == candidate_method_id
    ].copy()
    base = baseline[
        [
            "WeekStartDate",
            "WeekEndDate",
            "WeekID",
            "CanonicalProductID",
            "CanonicalProductName",
            "Actual",
            "Prediction",
        ]
    ].rename(columns={"Prediction": "BaselinePrediction"})
    paired = candidate.merge(
        base,
        on=[
            "WeekStartDate",
            "WeekEndDate",
            "WeekID",
            "CanonicalProductID",
            "CanonicalProductName",
            "Actual",
        ],
        how="inner",
        validate="one_to_one",
    )
    if len(paired) != len(candidate):
        raise AssertionError(f"Blend pairing failed for {candidate_method_id}")
    output_rows = []
    audit_rows = []
    for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
        prior = paired.loc[paired["WeekStartDate"] < forecast_week]
        prior_weeks = int(prior["WeekStartDate"].nunique())
        weight_scores = []
        if prior_weeks >= PREQUENTIAL_WARMUP_FOLDS:
            for weight in BLEND_WEIGHTS:
                prediction = (
                    weight * prior["Prediction"].to_numpy(dtype=float)
                    + (1.0 - weight)
                    * prior["BaselinePrediction"].to_numpy(dtype=float)
                )
                metrics = metric_record(prior["Actual"], prediction)
                weight_scores.append(
                    {
                        "CandidateWeight": weight,
                        "PriorWAPEPercentage": metrics["WAPEPercentage"],
                        "PriorMAE": metrics["MAE"],
                        "PriorRMSE": metrics["RMSE"],
                        "PriorAbsoluteTotalBias": abs(metrics["TotalBias"]),
                        "DistanceFromBalancedBlend": abs(weight - 0.50),
                    }
                )
            score_table = pd.DataFrame(weight_scores).sort_values(
                [
                    "PriorWAPEPercentage",
                    "PriorMAE",
                    "PriorRMSE",
                    "PriorAbsoluteTotalBias",
                    "DistanceFromBalancedBlend",
                    "CandidateWeight",
                ],
                kind="mergesort",
            )
            selected_weight = float(score_table.iloc[0]["CandidateWeight"])
            reason = "LOWEST_PRIOR_PREQUENTIAL_BLEND_WAPE"
            prior_wape = float(score_table.iloc[0]["PriorWAPEPercentage"])
        else:
            selected_weight = DEFAULT_BLEND_WEIGHT
            reason = "WARMUP_DEFAULT_BALANCED_BLEND"
            prior_wape = np.nan
        current = paired.loc[paired["WeekStartDate"] == forecast_week].copy()
        current["Prediction"] = np.clip(
            selected_weight * current["Prediction"].to_numpy(dtype=float)
            + (1.0 - selected_weight)
            * current["BaselinePrediction"].to_numpy(dtype=float),
            0.0,
            None,
        )
        current["Error"] = current["Prediction"] - current["Actual"]
        current["AbsoluteError"] = current["Error"].abs()
        current["MethodID"] = blend_method_id
        current["MethodFamily"] = family
        current["SelectionEligible"] = True
        current["ForecastOrigin"] = "WEEK_START"
        current["BlendWeightOnCandidate"] = selected_weight
        output_rows.append(current.drop(columns=["BaselinePrediction"]))
        audit_rows.append(
            {
                "CandidateMethodID": candidate_method_id,
                "BlendMethodID": blend_method_id,
                "FoldNumber": fold_number,
                "ForecastWeek": forecast_week,
                "PriorForecastWeeksAvailable": prior_weeks,
                "SelectedCandidateWeight": selected_weight,
                "SelectedBaselineWeight": 1.0 - selected_weight,
                "SelectedPriorWAPEPercentage": prior_wape,
                "BlendSelectionReason": reason,
            }
        )
    return pd.concat(output_rows, ignore_index=True), pd.DataFrame(audit_rows)


calibrated_weekly_frames = []
weekly_calibration_audits = []
weekly_blend_frames = []
weekly_blend_audits = []

for family in sorted(weekly_tuned_predictions["MethodFamily"].unique()):
    raw_method = f"TUNED_{family}"
    calibrated_method = f"CALIBRATED_{family}"
    blend_method = f"BLEND_CALIBRATED_{family}_WITH_NAIVE"
    calibrated, calibration_audit = prequential_calibrate(
        weekly_tuned_predictions,
        raw_method,
        calibrated_method,
        family,
    )
    blended, blend_audit = prequential_blend(
        calibrated,
        calibrated_method,
        blend_method,
        family,
        locked_baseline,
    )
    calibrated_weekly_frames.append(calibrated)
    weekly_calibration_audits.append(calibration_audit)
    weekly_blend_frames.append(blended)
    weekly_blend_audits.append(blend_audit)

weekly_calibrated_predictions = pd.concat(calibrated_weekly_frames, ignore_index=True)
weekly_blended_predictions = pd.concat(weekly_blend_frames, ignore_index=True)
weekly_calibration_audit = pd.concat(weekly_calibration_audits, ignore_index=True)
weekly_blend_audit = pd.concat(weekly_blend_audits, ignore_index=True)

# =============================================================================
# DAILY SOURCE LOADING AND NORMAL-DEMAND FEATURE RECONSTRUCTION
# =============================================================================

# The canonical source is loaded and immediately restricted to pre-holdout dates.
# March 2026 target rows are never retained, merged, scored or used for selection.
daily_panel = pd.read_csv(DAILY_MODEL_PANEL, low_memory=False)
canonical_daily = pd.read_csv(
    CANONICAL_DAILY_SOURCE,
    usecols=["Date", "CanonicalProductID", "NormalDemand", "BulkDemand", "TotalDemand"],
    low_memory=False,
)
feature_contract = pd.read_csv(DAILY_FEATURE_CONTRACT).sort_values("ColumnOrder")

for frame, label in [
    (daily_panel, "daily model panel"),
    (canonical_daily, "canonical daily source"),
]:
    frame["Date"] = pd.to_datetime(frame["Date"], errors="raise")
    frame["CanonicalProductID"] = frame["CanonicalProductID"].astype(str)

march_rows_loaded = int((canonical_daily["Date"] >= HOLDOUT_START).sum())
canonical_daily = canonical_daily.loc[canonical_daily["Date"] < HOLDOUT_START].copy()
daily_panel = daily_panel.loc[daily_panel["Date"] < HOLDOUT_START].copy()

required_daily_panel_columns = {
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "OperatingDaySequence",
    "TotalDemand",
}
if not required_daily_panel_columns.issubset(daily_panel.columns):
    raise AssertionError(
        "Daily model panel is missing columns: "
        f"{sorted(required_daily_panel_columns - set(daily_panel.columns))}"
    )

normal_lookup = canonical_daily[
    ["Date", "CanonicalProductID", "NormalDemand", "BulkDemand", "TotalDemand"]
].copy()
for column in ["NormalDemand", "BulkDemand", "TotalDemand"]:
    normal_lookup[column] = pd.to_numeric(normal_lookup[column], errors="raise").astype(float)
if normal_lookup.duplicated(["Date", "CanonicalProductID"]).any():
    raise AssertionError("Canonical daily source has duplicate product-date rows")

# Avoid a target-name collision with the model panel's TotalDemand.
daily_panel = daily_panel.merge(
    normal_lookup.rename(
        columns={
            "NormalDemand": DAILY_TARGET_COLUMN,
            "BulkDemand": "DailyBulkDemand",
            "TotalDemand": "CanonicalDailyTotalDemand",
        }
    ),
    on=["Date", "CanonicalProductID"],
    how="left",
    validate="one_to_one",
)

# A missing component is permitted only where the model-panel total is zero.
missing_normal = daily_panel[DAILY_TARGET_COLUMN].isna()
positive_missing = missing_normal & (
    pd.to_numeric(daily_panel["TotalDemand"], errors="raise") > 0
)
if positive_missing.any():
    raise AssertionError(
        "Positive daily model rows are missing normal-demand components: "
        f"{int(positive_missing.sum())}"
    )
daily_panel[DAILY_TARGET_COLUMN] = daily_panel[DAILY_TARGET_COLUMN].fillna(0.0)
daily_panel["DailyBulkDemand"] = daily_panel["DailyBulkDemand"].fillna(0.0)
daily_panel["CanonicalDailyTotalDemand"] = daily_panel[
    "CanonicalDailyTotalDemand"
].fillna(0.0)

component_difference = np.abs(
    daily_panel["CanonicalDailyTotalDemand"].to_numpy(dtype=float)
    - (
        daily_panel[DAILY_TARGET_COLUMN].to_numpy(dtype=float)
        + daily_panel["DailyBulkDemand"].to_numpy(dtype=float)
    )
).max()
if component_difference > 1e-9:
    raise AssertionError("Daily normal plus bulk does not reconcile to total")

# Direct model inputs are taken from the frozen contract. Historical demand
# features are regenerated from DailyNormalDemand, not from TotalDemand.
direct_allowed = bool_series(
    feature_contract["DirectModelInputAllowed"],
    "daily feature contract.DirectModelInputAllowed",
)
historical_allowed = bool_series(
    feature_contract["IsHistoricalDemandFeature"],
    "daily feature contract.IsHistoricalDemandFeature",
)
direct_contract = feature_contract.loc[direct_allowed].copy()
daily_direct_inputs = direct_contract["Column"].astype(str).tolist()
daily_historical_inputs = feature_contract.loc[
    direct_allowed & historical_allowed, "Column"
].astype(str).tolist()
daily_known_inputs = [
    column for column in daily_direct_inputs if column not in daily_historical_inputs
]
missing_contract_inputs = [
    column for column in daily_known_inputs if column not in daily_panel.columns
]
if missing_contract_inputs:
    raise AssertionError(
        f"Daily known-ahead contract inputs are missing: {missing_contract_inputs}"
    )

# Recompute historical features from normal demand using actual prior rows only.
daily_panel = daily_panel.sort_values(
    ["CanonicalProductID", "OperatingDaySequence", "Date"]
).reset_index(drop=True)

grouped_target = daily_panel.groupby("CanonicalProductID", sort=False)[
    DAILY_TARGET_COLUMN
]
shifted = grouped_target.shift(1)
for lag in [1, 2, 3, 5, 10, 20]:
    daily_panel[f"TotalDemandLag_{lag}"] = grouped_target.shift(lag)

for window in [3, 5, 10, 20]:
    rolling = shifted.groupby(daily_panel["CanonicalProductID"], sort=False).rolling(
        window=window,
        min_periods=1,
    )
    daily_panel[f"PastDemandRollingMean_{window}"] = (
        rolling.mean().reset_index(level=0, drop=True)
    )
    daily_panel[f"PastDemandRollingMedian_{window}"] = (
        rolling.median().reset_index(level=0, drop=True)
    )
    daily_panel[f"PastDemandRollingStd_{window}"] = (
        rolling.std(ddof=0).reset_index(level=0, drop=True)
    )
    daily_panel[f"PastDemandRollingSum_{window}"] = (
        rolling.sum().reset_index(level=0, drop=True)
    )
    zero_indicator = (daily_panel[DAILY_TARGET_COLUMN] == 0).astype(float)
    positive_indicator = (daily_panel[DAILY_TARGET_COLUMN] > 0).astype(float)
    shifted_zero = zero_indicator.groupby(daily_panel["CanonicalProductID"]).shift(1)
    shifted_positive = positive_indicator.groupby(
        daily_panel["CanonicalProductID"]
    ).shift(1)
    daily_panel[f"PastZeroDemandRate_{window}"] = (
        shifted_zero.groupby(daily_panel["CanonicalProductID"], sort=False)
        .rolling(window=window, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )
    daily_panel[f"PastPositiveDemandCount_{window}"] = (
        shifted_positive.groupby(daily_panel["CanonicalProductID"], sort=False)
        .rolling(window=window, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

daily_panel["ExpandingPastMeanDemand"] = (
    shifted.groupby(daily_panel["CanonicalProductID"], sort=False)
    .expanding(min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
positive_indicator = (daily_panel[DAILY_TARGET_COLUMN] > 0).astype(float)
daily_panel["ExpandingPastPositiveDemandRate"] = (
    positive_indicator.groupby(daily_panel["CanonicalProductID"], sort=False)
    .shift(1)
    .groupby(daily_panel["CanonicalProductID"], sort=False)
    .expanding(min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
positive_sequence = daily_panel["OperatingDaySequence"].where(
    daily_panel[DAILY_TARGET_COLUMN] > 0
)
previous_positive_sequence = (
    positive_sequence.groupby(daily_panel["CanonicalProductID"], sort=False)
    .shift(1)
    .groupby(daily_panel["CanonicalProductID"], sort=False)
    .ffill()
)
daily_panel["OperatingDaysSincePreviousPositiveDemand"] = (
    daily_panel["OperatingDaySequence"] - previous_positive_sequence
)

missing_historical_contract = [
    column for column in daily_historical_inputs if column not in daily_panel.columns
]
if missing_historical_contract:
    raise AssertionError(
        f"Failed to regenerate daily historical inputs: {missing_historical_contract}"
    )

for column in daily_direct_inputs:
    if column in daily_panel.columns and (
        pd.api.types.is_numeric_dtype(daily_panel[column])
        or pd.api.types.is_bool_dtype(daily_panel[column])
    ):
        daily_panel[column] = pd.to_numeric(daily_panel[column], errors="coerce")
daily_panel[daily_historical_inputs] = daily_panel[daily_historical_inputs].replace(
    [np.inf, -np.inf], np.nan
)

# Determine categorical versus numeric fields from actual pre-holdout types.
daily_categorical_inputs = ["CanonicalProductID"]
for column in daily_known_inputs:
    if column in daily_panel.columns and (
        pd.api.types.is_object_dtype(daily_panel[column])
        or pd.api.types.is_string_dtype(daily_panel[column])
        or pd.api.types.is_categorical_dtype(daily_panel[column])
    ):
        daily_categorical_inputs.append(column)
daily_categorical_inputs = list(dict.fromkeys(daily_categorical_inputs))
daily_numeric_inputs = [
    column
    for column in daily_direct_inputs
    if column not in daily_categorical_inputs
]
empty_daily_numeric_inputs = [
    column
    for column in daily_numeric_inputs
    if column in daily_panel.columns and daily_panel[column].notna().sum() == 0
]
daily_numeric_inputs = [
    column for column in daily_numeric_inputs if column not in empty_daily_numeric_inputs
]

for column in daily_categorical_inputs:
    if column != "CanonicalProductID" and column not in daily_panel.columns:
        raise AssertionError(f"Daily categorical input is missing: {column}")
for column in daily_numeric_inputs:
    if column not in daily_panel.columns:
        raise AssertionError(f"Daily numeric input is missing: {column}")

calendar_columns = [
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "IsWeekend",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay",
]
calendar_reference = (
    daily_panel[["Date", *calendar_columns]]
    .drop_duplicates("Date")
    .sort_values("Date")
    .set_index("Date")
)
if calendar_reference.index.duplicated().any():
    raise AssertionError("Daily calendar reference contains duplicate dates")

product_static_candidates = [
    "CanonicalProductName",
    "ProductFirstObservedDate",
    "SourcePLUCount",
    "SourceGroupCodes",
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration",
    "IsMultiPLUCanonicalProduct",
]
product_static_candidates = [
    column for column in product_static_candidates if column in daily_panel.columns
]

# =============================================================================
# DAILY MODEL AND RECURSIVE FEATURE HELPERS
# =============================================================================


def build_daily_preprocessor(train: pd.DataFrame) -> tuple[ColumnTransformer, list[str], list[str]]:
    usable_numeric = [
        column
        for column in daily_numeric_inputs
        if column in train.columns and train[column].notna().sum() > 0
    ]
    usable_categorical = [
        column
        for column in daily_categorical_inputs
        if column in train.columns
    ]
    transformer = ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", make_imputer("constant", "__MISSING__")),
                        ("one_hot", make_ohe()),
                    ]
                ),
                usable_categorical,
            ),
            (
                "numeric",
                Pipeline([("imputer", make_imputer("median"))]),
                usable_numeric,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )
    return transformer, usable_categorical, usable_numeric


def fit_daily_hurdle_models(train: pd.DataFrame):
    preprocessor, categorical, numeric = build_daily_preprocessor(train)
    columns = categorical + numeric
    y = train[DAILY_TARGET_COLUMN].to_numpy(dtype=float)
    occurrence = (y > 0).astype(int)
    if occurrence.min() == occurrence.max():
        raise AssertionError("Daily occurrence training target has only one class")
    classifier = Pipeline(
        [
            ("preprocess", clone(preprocessor)),
            (
                "model",
                XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    n_estimators=250,
                    learning_rate=0.04,
                    max_depth=5,
                    min_child_weight=5,
                    subsample=0.80,
                    colsample_bytree=0.80,
                    reg_alpha=0.0,
                    reg_lambda=3.0,
                    n_jobs=-1,
                    random_state=RANDOM_SEED,
                    tree_method="hist",
                    verbosity=0,
                ),
            ),
        ]
    )
    classifier.fit(train[columns], occurrence)
    positive_train = train.loc[train[DAILY_TARGET_COLUMN] > 0].copy()
    if len(positive_train) < 20:
        raise AssertionError("Insufficient positive daily rows for quantity model")
    quantity = Pipeline(
        [
            ("preprocess", clone(preprocessor)),
            (
                "model",
                XGBRegressor(
                    objective="reg:squarederror",
                    n_estimators=300,
                    learning_rate=0.035,
                    max_depth=5,
                    min_child_weight=4,
                    subsample=0.80,
                    colsample_bytree=0.80,
                    reg_alpha=0.0,
                    reg_lambda=3.0,
                    n_jobs=-1,
                    random_state=RANDOM_SEED,
                    tree_method="hist",
                    verbosity=0,
                ),
            ),
        ]
    )
    quantity.fit(
        positive_train[columns],
        positive_train[DAILY_TARGET_COLUMN].to_numpy(dtype=float),
    )
    return classifier, quantity, columns


def history_feature_values(
    demand_history: list[float],
    sequence_history: list[float],
    current_sequence: float,
) -> dict:
    values: dict[str, float] = {}
    n = len(demand_history)
    for lag in [1, 2, 3, 5, 10, 20]:
        values[f"TotalDemandLag_{lag}"] = (
            float(demand_history[-lag]) if n >= lag else np.nan
        )
    array = np.asarray(demand_history, dtype=float)
    for window in [3, 5, 10, 20]:
        recent = array[-window:] if n else np.asarray([], dtype=float)
        if len(recent):
            values[f"PastDemandRollingMean_{window}"] = float(recent.mean())
            values[f"PastDemandRollingMedian_{window}"] = float(np.median(recent))
            values[f"PastDemandRollingStd_{window}"] = float(recent.std(ddof=0))
            values[f"PastDemandRollingSum_{window}"] = float(recent.sum())
            values[f"PastZeroDemandRate_{window}"] = float((recent == 0).mean())
            values[f"PastPositiveDemandCount_{window}"] = float((recent > 0).sum())
        else:
            values[f"PastDemandRollingMean_{window}"] = np.nan
            values[f"PastDemandRollingMedian_{window}"] = np.nan
            values[f"PastDemandRollingStd_{window}"] = np.nan
            values[f"PastDemandRollingSum_{window}"] = np.nan
            values[f"PastZeroDemandRate_{window}"] = np.nan
            values[f"PastPositiveDemandCount_{window}"] = np.nan
    if n:
        values["ExpandingPastMeanDemand"] = float(array.mean())
        values["ExpandingPastPositiveDemandRate"] = float((array > 0).mean())
        positive_indices = np.flatnonzero(array > 0)
        if len(positive_indices):
            previous_sequence = float(sequence_history[int(positive_indices[-1])])
            values["OperatingDaysSincePreviousPositiveDemand"] = float(
                current_sequence - previous_sequence
            )
        else:
            values["OperatingDaysSincePreviousPositiveDemand"] = np.nan
    else:
        values["ExpandingPastMeanDemand"] = np.nan
        values["ExpandingPastPositiveDemandRate"] = np.nan
        values["OperatingDaysSincePreviousPositiveDemand"] = np.nan
    return values


def build_daily_forecast_rows(
    products: list[str],
    forecast_date: pd.Timestamp,
    product_metadata: pd.DataFrame,
    first_sequence_lookup: dict[str, float],
    histories: dict[str, list[float]],
    sequence_histories: dict[str, list[float]],
) -> pd.DataFrame:
    if forecast_date not in calendar_reference.index:
        raise AssertionError(f"Forecast date is absent from operating calendar: {forecast_date}")
    calendar = calendar_reference.loc[forecast_date]
    current_sequence = float(calendar["OperatingDaySequence"])
    rows = []
    for product_id in products:
        if product_id not in product_metadata.index:
            raise AssertionError(
                f"No pre-week product metadata for daily forecast product {product_id}"
            )
        metadata = product_metadata.loc[product_id]
        row = {"Date": forecast_date, "CanonicalProductID": product_id}
        for column in product_static_candidates:
            row[column] = metadata.get(column, np.nan)
        for column in calendar_columns:
            row[column] = calendar[column]
        first_sequence = first_sequence_lookup.get(product_id, np.nan)
        row["ProductAgeOperatingDays"] = (
            max(0.0, current_sequence - first_sequence)
            if np.isfinite(first_sequence)
            else np.nan
        )
        row.update(
            history_feature_values(
                histories.get(product_id, []),
                sequence_histories.get(product_id, []),
                current_sequence,
            )
        )
        rows.append(row)
    return pd.DataFrame(rows)


def forecast_daily_week(
    classifier,
    quantity_model,
    model_columns: list[str],
    products: list[str],
    operating_dates: list[pd.Timestamp],
    product_metadata: pd.DataFrame,
    first_sequence_lookup: dict[str, float],
    starting_histories: dict[str, list[float]],
    starting_sequence_histories: dict[str, list[float]],
    actual_lookup: dict[tuple[pd.Timestamp, str], float],
    update_mode: str,
) -> pd.DataFrame:
    histories = {key: list(value) for key, value in starting_histories.items()}
    sequences = {key: list(value) for key, value in starting_sequence_histories.items()}
    rows = []
    for forecast_date in operating_dates:
        feature_rows = build_daily_forecast_rows(
            products,
            forecast_date,
            product_metadata,
            first_sequence_lookup,
            histories,
            sequences,
        )
        probability = classifier.predict_proba(feature_rows[model_columns])[:, 1]
        positive_quantity = np.clip(
            np.asarray(quantity_model.predict(feature_rows[model_columns]), dtype=float),
            0.0,
            None,
        )
        hurdle = np.clip(probability * positive_quantity, 0.0, None)
        naive5 = np.asarray(
            [
                float(np.mean(histories.get(product_id, [])[-5:]))
                if len(histories.get(product_id, []))
                else 0.0
                for product_id in products
            ],
            dtype=float,
        )
        blend = np.clip(0.75 * hurdle + 0.25 * naive5, 0.0, None)
        current_sequence = float(
            calendar_reference.loc[forecast_date, "OperatingDaySequence"]
        )
        for index, product_id in enumerate(products):
            actual = float(actual_lookup.get((forecast_date, product_id), 0.0))
            rows.append(
                {
                    "Date": forecast_date,
                    "CanonicalProductID": product_id,
                    "ActualDailyNormalDemand": actual,
                    "Naive5Prediction": float(naive5[index]),
                    "HurdlePrediction": float(hurdle[index]),
                    "HurdleNaive5H75Prediction": float(blend[index]),
                    "OccurrenceProbability": float(probability[index]),
                    "PositiveQuantityPrediction": float(positive_quantity[index]),
                    "UpdateMode": update_mode,
                }
            )
            if update_mode == "WEEKLY_ORIGIN_RECURSIVE":
                appended = float(blend[index])
            elif update_mode == "ACTUAL_WITHIN_WEEK_UPDATE_DIAGNOSTIC":
                appended = actual
            else:
                raise KeyError(update_mode)
            histories.setdefault(product_id, []).append(appended)
            sequences.setdefault(product_id, []).append(current_sequence)
    return pd.DataFrame(rows)

# =============================================================================
# RUN DAILY-TO-WEEKLY CHALLENGERS
# =============================================================================

daily_source_audit = pd.DataFrame(
    [
        {
            "Metric": "PreholdoutDailyModelRows",
            "Value": int(len(daily_panel)),
        },
        {
            "Metric": "PreholdoutDailyProducts",
            "Value": int(daily_panel["CanonicalProductID"].nunique()),
        },
        {
            "Metric": "PreholdoutOperatingDates",
            "Value": int(daily_panel["Date"].nunique()),
        },
        {
            "Metric": "PreholdoutDateMinimum",
            "Value": daily_panel["Date"].min().strftime("%Y-%m-%d"),
        },
        {
            "Metric": "PreholdoutDateMaximum",
            "Value": daily_panel["Date"].max().strftime("%Y-%m-%d"),
        },
        {
            "Metric": "PreholdoutNormalDemandUnits",
            "Value": float(daily_panel[DAILY_TARGET_COLUMN].sum()),
        },
        {
            "Metric": "MarchRowsDiscardedBeforeFeatureOrModelUse",
            "Value": march_rows_loaded,
        },
        {
            "Metric": "NormalBulkTotalMaximumDifference",
            "Value": component_difference,
        },
    ]
)

daily_feature_audit = pd.DataFrame(
    [
        {
            "FeatureRole": "CATEGORICAL",
            "Column": column,
            "Included": True,
            "MissingCount": int(daily_panel[column].isna().sum()),
        }
        for column in daily_categorical_inputs
    ]
    + [
        {
            "FeatureRole": "NUMERIC",
            "Column": column,
            "Included": column not in empty_daily_numeric_inputs,
            "MissingCount": int(daily_panel[column].isna().sum()),
        }
        for column in daily_numeric_inputs + empty_daily_numeric_inputs
    ]
)

actual_daily_lookup = {
    (row.Date, str(row.CanonicalProductID)): float(row.DailyNormalDemand)
    for row in daily_panel[
        ["Date", "CanonicalProductID", DAILY_TARGET_COLUMN]
    ].itertuples(index=False)
}

first_sequence_lookup_global = (
    daily_panel.groupby("CanonicalProductID")["OperatingDaySequence"].min().astype(float).to_dict()
)

daily_weekly_rows: list[pd.DataFrame] = []
daily_fold_audit_rows: list[dict] = []

for fold_number, forecast_week in enumerate(forecast_weeks, start=1):
    print("-" * 100)
    print(f"Running daily-to-weekly fold {fold_number}/39: {forecast_week.date()}")
    weekly_valid = features.loc[
        (features["WeekStartDate"] == forecast_week) & features[PRIMARY_SCOPE_FLAG],
        [
            "WeekStartDate",
            "WeekEndDate",
            "WeekID",
            "CanonicalProductID",
            "CanonicalProductName",
            TARGET_COLUMN,
        ],
    ].copy()
    products = weekly_valid["CanonicalProductID"].astype(str).tolist()
    training_daily = daily_panel.loc[daily_panel["Date"] < forecast_week].copy()
    if training_daily.empty:
        raise AssertionError("Daily fold has no training rows")
    operating_dates = [
        date
        for date in calendar_reference.index
        if forecast_week <= date <= forecast_week + pd.Timedelta(days=6)
    ]
    if not operating_dates:
        raise AssertionError(f"No operating dates in forecast week {forecast_week.date()}")
    # Metadata is restricted to rows known before the forecast week.
    metadata_rows = (
        training_daily.sort_values("Date")
        .groupby("CanonicalProductID", as_index=False)
        .tail(1)
        .set_index("CanonicalProductID")
    )
    missing_metadata_products = sorted(set(products) - set(metadata_rows.index.astype(str)))
    if missing_metadata_products:
        raise AssertionError(
            "The dynamic 95% scope contains products without pre-week daily metadata: "
            f"{missing_metadata_products[:10]}"
        )
    histories: dict[str, list[float]] = {}
    sequence_histories: dict[str, list[float]] = {}
    for product_id, group in training_daily.loc[
        training_daily["CanonicalProductID"].isin(products)
    ].groupby("CanonicalProductID", sort=False):
        ordered = group.sort_values("OperatingDaySequence")
        histories[str(product_id)] = ordered[DAILY_TARGET_COLUMN].astype(float).tolist()
        sequence_histories[str(product_id)] = ordered["OperatingDaySequence"].astype(float).tolist()
    classifier, quantity_model, daily_model_columns = fit_daily_hurdle_models(
        training_daily
    )
    recursive_daily = forecast_daily_week(
        classifier,
        quantity_model,
        daily_model_columns,
        products,
        operating_dates,
        metadata_rows,
        first_sequence_lookup_global,
        histories,
        sequence_histories,
        actual_daily_lookup,
        "WEEKLY_ORIGIN_RECURSIVE",
    )
    actual_update_daily = forecast_daily_week(
        classifier,
        quantity_model,
        daily_model_columns,
        products,
        operating_dates,
        metadata_rows,
        first_sequence_lookup_global,
        histories,
        sequence_histories,
        actual_daily_lookup,
        "ACTUAL_WITHIN_WEEK_UPDATE_DIAGNOSTIC",
    )
    expected_daily_rows = len(products) * len(operating_dates)
    if len(recursive_daily) != expected_daily_rows or len(actual_update_daily) != expected_daily_rows:
        raise AssertionError("Daily fold prediction row count mismatch")

    daily_actual_aggregate = (
        recursive_daily.groupby("CanonicalProductID", as_index=False)
        .agg(DailyActualNormalDemand=("ActualDailyNormalDemand", "sum"))
    )
    daily_weekly_reconciliation = weekly_valid.merge(
        daily_actual_aggregate,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one",
    )
    daily_weekly_reconciliation["DailyActualNormalDemand"] = (
        daily_weekly_reconciliation["DailyActualNormalDemand"].fillna(0.0)
    )
    maximum_daily_weekly_actual_difference = float(
        np.abs(
            daily_weekly_reconciliation[TARGET_COLUMN].to_numpy(dtype=float)
            - daily_weekly_reconciliation["DailyActualNormalDemand"].to_numpy(dtype=float)
        ).max()
    )
    if maximum_daily_weekly_actual_difference > 1e-9:
        raise AssertionError(
            "Daily normal-demand actuals do not reconcile to the weekly target "
            f"for {forecast_week.date()}: {maximum_daily_weekly_actual_difference}"
        )

    def aggregate_daily_method(
        daily_predictions: pd.DataFrame,
        prediction_column: str,
        method_id: str,
        selection_eligible: bool,
        forecast_origin: str,
    ) -> pd.DataFrame:
        aggregated = (
            daily_predictions.groupby("CanonicalProductID", as_index=False)
            .agg(Prediction=(prediction_column, "sum"))
        )
        result = weekly_valid.merge(
            aggregated,
            on="CanonicalProductID",
            how="left",
            validate="one_to_one",
        )
        result["Prediction"] = result["Prediction"].fillna(0.0)
        result["Actual"] = result[TARGET_COLUMN].astype(float)
        result["FoldNumber"] = fold_number
        result["MethodID"] = method_id
        result["MethodFamily"] = "DAILY_TO_WEEKLY"
        result["SelectionEligible"] = selection_eligible
        result["ForecastOrigin"] = forecast_origin
        result["ConfigurationID"] = "DAILY_RECONSTRUCTED_FIXED_CONFIG"
        result["CalibrationFactor"] = 1.0
        result["BlendWeightOnCandidate"] = 1.0
        result["OperatingDatesForecast"] = len(operating_dates)
        result["Error"] = result["Prediction"] - result["Actual"]
        result["AbsoluteError"] = result["Error"].abs()
        return result

    daily_weekly_rows.extend(
        [
            aggregate_daily_method(
                recursive_daily,
                "Naive5Prediction",
                "DAILY_NAIVE5_WEEKLY_ORIGIN",
                True,
                "WEEK_START_RECURSIVE",
            ),
            aggregate_daily_method(
                recursive_daily,
                "HurdleNaive5H75Prediction",
                "DAILY_HURDLE_NAIVE5_H75_WEEKLY_ORIGIN",
                True,
                "WEEK_START_RECURSIVE",
            ),
            aggregate_daily_method(
                actual_update_daily,
                "Naive5Prediction",
                "DAILY_NAIVE5_ACTUAL_UPDATE_DIAGNOSTIC",
                False,
                "DAILY_ACTUAL_UPDATE_DIAGNOSTIC",
            ),
            aggregate_daily_method(
                actual_update_daily,
                "HurdleNaive5H75Prediction",
                "DAILY_HURDLE_NAIVE5_H75_ACTUAL_UPDATE_DIAGNOSTIC",
                False,
                "DAILY_ACTUAL_UPDATE_DIAGNOSTIC",
            ),
        ]
    )
    daily_fold_audit_rows.append(
        {
            "FoldNumber": fold_number,
            "ForecastWeek": forecast_week,
            "TrainingDailyRows": int(len(training_daily)),
            "TrainingOperatingDates": int(training_daily["Date"].nunique()),
            "TrainingProducts": int(training_daily["CanonicalProductID"].nunique()),
            "ForecastScopeProducts": len(products),
            "ForecastOperatingDates": len(operating_dates),
            "DailyPredictionRowsPerOrigin": expected_daily_rows,
            "MaximumDailyToWeeklyActualDifference": maximum_daily_weekly_actual_difference,
            "WeeklyOriginCompleted": True,
            "ActualUpdateDiagnosticCompleted": True,
        }
    )

daily_predictions = pd.concat(daily_weekly_rows, ignore_index=True)
daily_fold_audit = pd.DataFrame(daily_fold_audit_rows)

# Prequential calibration and blend of the selection-eligible daily hurdle method.
daily_calibrated, daily_calibration_audit = prequential_calibrate(
    daily_predictions,
    "DAILY_HURDLE_NAIVE5_H75_WEEKLY_ORIGIN",
    "CALIBRATED_DAILY_HURDLE_NAIVE5_H75_WEEKLY_ORIGIN",
    "DAILY_TO_WEEKLY",
)
daily_blended, daily_blend_audit = prequential_blend(
    daily_calibrated,
    "CALIBRATED_DAILY_HURDLE_NAIVE5_H75_WEEKLY_ORIGIN",
    "BLEND_CALIBRATED_DAILY_HURDLE_WITH_WEEKLY_NAIVE",
    "DAILY_TO_WEEKLY",
    locked_baseline,
)

calibration_audit = pd.concat(
    [weekly_calibration_audit, daily_calibration_audit],
    ignore_index=True,
)
blend_audit = pd.concat(
    [weekly_blend_audit, daily_blend_audit],
    ignore_index=True,
)

# =============================================================================
# COMMON PREDICTION BANK AND METRICS
# =============================================================================

common_columns = [
    "WeekStartDate",
    "WeekEndDate",
    "WeekID",
    "CanonicalProductID",
    "CanonicalProductName",
    "Actual",
    "Prediction",
    "MethodID",
    "MethodFamily",
    "SelectionEligible",
    "ForecastOrigin",
    "ConfigurationID",
    "CalibrationFactor",
    "BlendWeightOnCandidate",
]

prediction_frames = [
    locked_baseline,
    weekly_tuned_predictions,
    weekly_calibrated_predictions,
    weekly_blended_predictions,
    daily_predictions,
    daily_calibrated,
    daily_blended,
]
all_predictions = pd.concat(
    [frame.reindex(columns=common_columns) for frame in prediction_frames],
    ignore_index=True,
)
all_predictions["Actual"] = pd.to_numeric(all_predictions["Actual"], errors="raise").astype(float)
all_predictions["Prediction"] = pd.to_numeric(
    all_predictions["Prediction"], errors="raise"
).astype(float)
all_predictions["Error"] = all_predictions["Prediction"] - all_predictions["Actual"]
all_predictions["AbsoluteError"] = all_predictions["Error"].abs()
all_predictions["TargetID"] = TARGET_ID
all_predictions["ScopeID"] = PRIMARY_SCOPE_ID
all_predictions["BulkIncludedInModelTarget"] = False

if all_predictions.duplicated([*KEY_COLUMNS, "MethodID"]).any():
    duplicate = all_predictions.loc[
        all_predictions.duplicated([*KEY_COLUMNS, "MethodID"], keep=False)
    ].head(20)
    raise AssertionError(
        "Duplicate method/product/week predictions found:\n"
        + duplicate.to_string(index=False)
    )
if not np.isfinite(all_predictions[["Actual", "Prediction"]].to_numpy()).all():
    raise AssertionError("The common prediction bank contains non-finite values")
if (all_predictions["Prediction"] < 0).any():
    raise AssertionError("The common prediction bank contains negative forecasts")

method_counts = all_predictions.groupby("MethodID").agg(
    Rows=("Prediction", "size"),
    Weeks=("WeekStartDate", "nunique"),
)
if (method_counts["Rows"] != len(locked_baseline)).any() or (method_counts["Weeks"] != 39).any():
    raise AssertionError(
        "Not every method has the same 3,351 rows and 39 weeks:\n"
        + method_counts.to_string()
    )

pooled_rows = []
for (method_id, method_family, eligible, origin), group in all_predictions.groupby(
    ["MethodID", "MethodFamily", "SelectionEligible", "ForecastOrigin"],
    sort=True,
):
    metrics = metric_record(group["Actual"], group["Prediction"])
    pooled_rows.append(
        {
            "TargetID": TARGET_ID,
            "ScopeID": PRIMARY_SCOPE_ID,
            "MethodID": method_id,
            "MethodFamily": method_family,
            "SelectionEligible": bool(eligible),
            "ForecastOrigin": origin,
            "ForecastWeeks": int(group["WeekStartDate"].nunique()),
            "Products": int(group["CanonicalProductID"].nunique()),
            **metrics,
        }
    )
pooled_metrics = pd.DataFrame(pooled_rows)
pooled_metrics["AbsoluteTotalBias"] = pooled_metrics["TotalBias"].abs()
pooled_metrics["AbsoluteBiasPercentage"] = np.where(
    pooled_metrics["ActualTotal"] != 0,
    100.0 * pooled_metrics["AbsoluteTotalBias"] / pooled_metrics["ActualTotal"],
    np.nan,
)
pooled_metrics["ComplexityOrder"] = pooled_metrics["MethodID"].map(method_complexity)
pooled_metrics = pooled_metrics.sort_values(
    [
        "SelectionEligible",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "ComplexityOrder",
        "MethodID",
    ],
    ascending=[False, True, True, True, True, True, True],
    kind="mergesort",
).reset_index(drop=True)

fold_metric_rows = []
for (method_id, forecast_week), group in all_predictions.groupby(
    ["MethodID", "WeekStartDate"], sort=True
):
    row = pooled_metrics.loc[pooled_metrics["MethodID"] == method_id].iloc[0]
    fold_metric_rows.append(
        {
            "MethodID": method_id,
            "MethodFamily": row["MethodFamily"],
            "SelectionEligible": bool(row["SelectionEligible"]),
            "ForecastOrigin": row["ForecastOrigin"],
            "ForecastWeek": forecast_week,
            **metric_record(group["Actual"], group["Prediction"]),
        }
    )
fold_metrics = pd.DataFrame(fold_metric_rows)

benchmark_metrics = pooled_metrics.loc[
    pooled_metrics["MethodID"] == LOCKED_BENCHMARK_METHOD
]
if len(benchmark_metrics) != 1:
    raise AssertionError("Could not identify one benchmark metric row")
benchmark_metrics = benchmark_metrics.iloc[0]

benchmark_fold_error = (
    all_predictions.loc[all_predictions["MethodID"] == LOCKED_BENCHMARK_METHOD]
    .groupby("WeekStartDate", as_index=False)
    .agg(BenchmarkAbsoluteError=("AbsoluteError", "sum"))
)
pairwise_rows = []
for method_id, group in all_predictions.groupby("MethodID", sort=True):
    weekly_error = (
        group.groupby("WeekStartDate", as_index=False)
        .agg(CandidateAbsoluteError=("AbsoluteError", "sum"))
        .merge(benchmark_fold_error, on="WeekStartDate", validate="one_to_one")
    )
    weekly_error["MethodID"] = method_id
    weekly_error["CandidateWonWeek"] = (
        weekly_error["CandidateAbsoluteError"]
        < weekly_error["BenchmarkAbsoluteError"] - 1e-12
    )
    weekly_error["BenchmarkWonWeek"] = (
        weekly_error["CandidateAbsoluteError"]
        > weekly_error["BenchmarkAbsoluteError"] + 1e-12
    )
    weekly_error["Tie"] = ~(
        weekly_error["CandidateWonWeek"] | weekly_error["BenchmarkWonWeek"]
    )
    pairwise_rows.append(weekly_error)
pairwise_audit = pd.concat(pairwise_rows, ignore_index=True)
pairwise_summary = (
    pairwise_audit.groupby("MethodID", as_index=False)
    .agg(
        PairwiseWeeksWon=("CandidateWonWeek", "sum"),
        PairwiseWeeksLost=("BenchmarkWonWeek", "sum"),
        PairwiseWeeksTied=("Tie", "sum"),
    )
)

final_comparison = pooled_metrics.merge(
    pairwise_summary,
    on="MethodID",
    how="left",
    validate="one_to_one",
)
final_comparison["BenchmarkWAPEPercentage"] = float(
    benchmark_metrics["WAPEPercentage"]
)
final_comparison["WAPEImprovementPercentagePoints"] = (
    final_comparison["BenchmarkWAPEPercentage"]
    - final_comparison["WAPEPercentage"]
)
final_comparison["RelativeWAPEImprovementPercentage"] = (
    100.0
    * final_comparison["WAPEImprovementPercentagePoints"]
    / final_comparison["BenchmarkWAPEPercentage"]
)
final_comparison["BenchmarkMAE"] = float(benchmark_metrics["MAE"])
final_comparison["BenchmarkRMSE"] = float(benchmark_metrics["RMSE"])
final_comparison["RMSERatioToBenchmark"] = (
    final_comparison["RMSE"] / final_comparison["BenchmarkRMSE"]
)
final_comparison["BeatBenchmarkOnWAPE"] = (
    final_comparison["WAPEImprovementPercentagePoints"] > 0
)
final_comparison["PassedMinimumWAPEImprovement"] = (
    final_comparison["WAPEImprovementPercentagePoints"]
    >= MINIMUM_WAPE_IMPROVEMENT_POINTS
)
final_comparison["PassedMAERequirement"] = (
    final_comparison["MAE"] <= final_comparison["BenchmarkMAE"] + 1e-12
)
final_comparison["PassedBiasRequirement"] = (
    final_comparison["AbsoluteBiasPercentage"]
    <= MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE
)
final_comparison["PassedRMSERequirement"] = (
    final_comparison["RMSERatioToBenchmark"]
    <= MAXIMUM_RMSE_RATIO_TO_BENCHMARK
)
final_comparison["PassedWeekWinRequirement"] = (
    final_comparison["PairwiseWeeksWon"] >= MINIMUM_PAIRWISE_WEEK_WINS
)
final_comparison["QualifiedToReplaceBenchmark"] = (
    final_comparison["SelectionEligible"]
    & (final_comparison["MethodID"] != LOCKED_BENCHMARK_METHOD)
    & final_comparison["PassedMinimumWAPEImprovement"]
    & final_comparison["PassedMAERequirement"]
    & final_comparison["PassedBiasRequirement"]
    & final_comparison["PassedRMSERequirement"]
    & final_comparison["PassedWeekWinRequirement"]
)
final_comparison = final_comparison.sort_values(
    [
        "QualifiedToReplaceBenchmark",
        "SelectionEligible",
        "WAPEPercentage",
        "MAE",
        "RMSE",
        "AbsoluteTotalBias",
        "ComplexityOrder",
        "MethodID",
    ],
    ascending=[False, False, True, True, True, True, True, True],
    kind="mergesort",
).reset_index(drop=True)
final_comparison["FinalComparisonRank"] = np.arange(1, len(final_comparison) + 1)

qualified = final_comparison.loc[final_comparison["QualifiedToReplaceBenchmark"]].copy()
if qualified.empty:
    final_method_id = LOCKED_BENCHMARK_METHOD
    replacement_selected = False
    final_reason = (
        "No selection-eligible challenger satisfied all predeclared WAPE, MAE, "
        "bias, RMSE and pairwise-week requirements."
    )
else:
    selected = qualified.sort_values(
        [
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "AbsoluteTotalBias",
            "ComplexityOrder",
            "MethodID",
        ],
        kind="mergesort",
    ).iloc[0]
    final_method_id = str(selected["MethodID"])
    replacement_selected = True
    final_reason = (
        "The challenger satisfied every predeclared operational acceptance "
        "condition and had the lowest WAPE among qualifying methods."
    )

final_metric_row = final_comparison.loc[
    final_comparison["MethodID"] == final_method_id
].iloc[0]

# Derive frozen deployment parameters using all pre-holdout selection evidence.
deployment_parameters: dict = {
    "MethodID": final_method_id,
    "TargetID": TARGET_ID,
    "ScopeID": PRIMARY_SCOPE_ID,
}
if final_method_id.startswith("TUNED_") or "CALIBRATED_" in final_method_id or "BLEND_" in final_method_id:
    family_match = None
    for family in ["XGBOOST_POISSON", "RANDOM_FOREST_POISSON", "CATBOOST_RMSE"]:
        if family in final_method_id:
            family_match = family
            break
    if family_match is not None:
        config_scores = []
        family_config_predictions = weekly_config_predictions.loc[
            weekly_config_predictions["Family"] == family_match
        ]
        for config_id, group in family_config_predictions.groupby("ConfigurationID"):
            metrics = metric_record(group["Actual"], group["Prediction"])
            complexity = int(
                weekly_config_registry.loc[
                    weekly_config_registry["ConfigurationID"] == config_id,
                    "ComplexityOrder",
                ].iloc[0]
            )
            config_scores.append(
                {
                    "ConfigurationID": config_id,
                    "WAPEPercentage": metrics["WAPEPercentage"],
                    "MAE": metrics["MAE"],
                    "RMSE": metrics["RMSE"],
                    "AbsoluteTotalBias": abs(metrics["TotalBias"]),
                    "ComplexityOrder": complexity,
                }
            )
        selected_config = pd.DataFrame(config_scores).sort_values(
            [
                "WAPEPercentage",
                "MAE",
                "RMSE",
                "AbsoluteTotalBias",
                "ComplexityOrder",
                "ConfigurationID",
            ],
            kind="mergesort",
        ).iloc[0]
        selected_config_id = str(selected_config["ConfigurationID"])
        selected_parameters_json = weekly_config_registry.loc[
            weekly_config_registry["ConfigurationID"] == selected_config_id,
            "ParametersJSON",
        ].iloc[0]
        deployment_parameters["WeeklyModelFamily"] = family_match
        deployment_parameters["ConfigurationID"] = selected_config_id
        deployment_parameters["Configuration"] = json.loads(selected_parameters_json)
        raw_method = f"TUNED_{family_match}"
        raw_all = weekly_tuned_predictions.loc[
            weekly_tuned_predictions["MethodID"] == raw_method
        ]
        deployment_parameters["CalibrationFactor"] = float(
            np.clip(
                raw_all["Actual"].sum() / raw_all["Prediction"].sum(),
                CALIBRATION_FACTOR_MIN,
                CALIBRATION_FACTOR_MAX,
            )
        )
        if final_method_id.startswith("BLEND_"):
            calibrated_method = f"CALIBRATED_{family_match}"
            calibrated_all = weekly_calibrated_predictions.loc[
                weekly_calibrated_predictions["MethodID"] == calibrated_method
            ].merge(
                locked_baseline[KEY_COLUMNS + ["Prediction"]].rename(
                    columns={"Prediction": "BaselinePrediction"}
                ),
                on=KEY_COLUMNS,
                validate="one_to_one",
            )
            weight_scores = []
            for weight in BLEND_WEIGHTS:
                pred = (
                    weight * calibrated_all["Prediction"]
                    + (1.0 - weight) * calibrated_all["BaselinePrediction"]
                )
                metrics = metric_record(calibrated_all["Actual"], pred)
                weight_scores.append(
                    {
                        "Weight": weight,
                        "WAPEPercentage": metrics["WAPEPercentage"],
                        "MAE": metrics["MAE"],
                        "RMSE": metrics["RMSE"],
                        "AbsoluteTotalBias": abs(metrics["TotalBias"]),
                    }
                )
            deployment_weight = pd.DataFrame(weight_scores).sort_values(
                ["WAPEPercentage", "MAE", "RMSE", "AbsoluteTotalBias", "Weight"],
                kind="mergesort",
            ).iloc[0]
            deployment_parameters["BlendWeightOnCandidate"] = float(
                deployment_weight["Weight"]
            )
            deployment_parameters["BlendWeightOnWeeklyNaive"] = float(
                1.0 - deployment_weight["Weight"]
            )

if "DAILY_HURDLE" in final_method_id:
    raw_daily = daily_predictions.loc[
        daily_predictions["MethodID"]
        == "DAILY_HURDLE_NAIVE5_H75_WEEKLY_ORIGIN"
    ]
    deployment_parameters.update(
        {
            "Architecture": "DAILY_HURDLE_PLUS_NAIVE5_AGGREGATED_TO_WEEK",
            "ForecastOrigin": "WEEK_START_RECURSIVE",
            "HurdleWeight": 0.75,
            "Naive5Weight": 0.25,
            "CalibrationFactor": float(
                np.clip(
                    raw_daily["Actual"].sum() / raw_daily["Prediction"].sum(),
                    CALIBRATION_FACTOR_MIN,
                    CALIBRATION_FACTOR_MAX,
                )
            ),
        }
    )
    if final_method_id.startswith("BLEND_"):
        calibrated_daily_all = daily_calibrated.merge(
            locked_baseline[KEY_COLUMNS + ["Prediction"]].rename(
                columns={"Prediction": "BaselinePrediction"}
            ),
            on=KEY_COLUMNS,
            validate="one_to_one",
        )
        weight_scores = []
        for weight in BLEND_WEIGHTS:
            pred = (
                weight * calibrated_daily_all["Prediction"]
                + (1.0 - weight) * calibrated_daily_all["BaselinePrediction"]
            )
            metrics = metric_record(calibrated_daily_all["Actual"], pred)
            weight_scores.append(
                {
                    "Weight": weight,
                    "WAPEPercentage": metrics["WAPEPercentage"],
                    "MAE": metrics["MAE"],
                    "RMSE": metrics["RMSE"],
                    "AbsoluteTotalBias": abs(metrics["TotalBias"]),
                }
            )
        deployment_weight = pd.DataFrame(weight_scores).sort_values(
            ["WAPEPercentage", "MAE", "RMSE", "AbsoluteTotalBias", "Weight"],
            kind="mergesort",
        ).iloc[0]
        deployment_parameters["BlendWeightOnDailyCandidate"] = float(
            deployment_weight["Weight"]
        )
        deployment_parameters["BlendWeightOnWeeklyNaive"] = float(
            1.0 - deployment_weight["Weight"]
        )

final_decision = {
    "StepID": STEP_ID,
    "CreatedUTC": NOW_UTC.isoformat(),
    "CreatedLocal": NOW_LOCAL.isoformat(),
    "Status": STATUS,
    "PreviousLockedBenchmark": LOCKED_BENCHMARK_METHOD,
    "FinalAuthoritativeMethodID": final_method_id,
    "ReplacementSelected": replacement_selected,
    "DecisionReason": final_reason,
    "TargetID": TARGET_ID,
    "ScopeID": PRIMARY_SCOPE_ID,
    "BulkIncludedInModelledTarget": False,
    "UnknownBulkFallback": "ZERO_BULK",
    "ConfirmedBulkHandling": "ADD_EXTERNALLY_WHEN_KNOWN",
    "AcceptanceCriteria": {
        "MinimumWAPEImprovementPercentagePoints": MINIMUM_WAPE_IMPROVEMENT_POINTS,
        "MaximumAbsoluteBiasPercentage": MAXIMUM_ABSOLUTE_BIAS_PERCENTAGE,
        "MaximumRMSERatioToBenchmark": MAXIMUM_RMSE_RATIO_TO_BENCHMARK,
        "MinimumPairwiseWeeksWon": MINIMUM_PAIRWISE_WEEK_WINS,
        "MAEMustNotExceedBenchmark": True,
    },
    "FinalPerformance": {
        key: (
            bool(final_metric_row[key])
            if isinstance(final_metric_row[key], (bool, np.bool_))
            else float(final_metric_row[key])
            if isinstance(final_metric_row[key], (int, float, np.integer, np.floating))
            else str(final_metric_row[key])
        )
        for key in [
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "MeanBias",
            "TotalBias",
            "AbsoluteBiasPercentage",
            "PairwiseWeeksWon",
            "PairwiseWeeksLost",
            "WAPEImprovementPercentagePoints",
            "RelativeWAPEImprovementPercentage",
        ]
    },
    "DeploymentParameters": deployment_parameters,
    "DiagnosticPolicy": {
        "DailyActualUpdateMethodsSelectionEligible": False,
        "Reason": (
            "These methods use actual demand observed earlier inside the same "
            "week and are not valid Monday-origin weekly purchasing forecasts."
        ),
    },
    "FinalUntouchedEvaluationStillRequired": True,
}

final_design_amendment = {
    "StepID": STEP_ID,
    "AmendmentType": (
        "METHOD_COMPONENT_SUPERSEDED"
        if replacement_selected
        else "BENCHMARK_CONFIRMED_NO_METHOD_CHANGE"
    ),
    "CreatedUTC": NOW_UTC.isoformat(),
    "Part11ILockSHA256": part11i_lock_hash,
    "Part11IA1LockSHA256": part11ia1_lock_hash,
    "AuthoritativeForecastDesign": {
        "TargetID": TARGET_ID,
        "MethodID": final_method_id,
        "ScopeID": PRIMARY_SCOPE_ID,
        "BulkIncludedInModelledTarget": False,
        "UnknownBulkFallback": "ZERO_BULK",
        "ConfirmedBulkHandling": "ADD_EXTERNALLY_WHEN_KNOWN",
        "OperationalQuantityFormula": (
            "ForecastNormalDemand + ConfirmedBulkQuantity"
        ),
    },
    "DeploymentParameters": deployment_parameters,
    "SelectionEvidence": str(FINAL_COMPARISON_PATH),
    "FinalUntouchedEvaluationRequired": True,
}

final_design_table = pd.DataFrame(
    [
        {
            "DesignField": "Forecast target",
            "Part11IValue": TARGET_ID,
            "Part11IA2AuthoritativeValue": TARGET_ID,
            "Changed": False,
        },
        {
            "DesignField": "Forecast method",
            "Part11IValue": LOCKED_BENCHMARK_METHOD,
            "Part11IA2AuthoritativeValue": final_method_id,
            "Changed": replacement_selected,
        },
        {
            "DesignField": "Primary scope",
            "Part11IValue": PRIMARY_SCOPE_ID,
            "Part11IA2AuthoritativeValue": PRIMARY_SCOPE_ID,
            "Changed": False,
        },
        {
            "DesignField": "Bulk in model target",
            "Part11IValue": "EXCLUDED",
            "Part11IA2AuthoritativeValue": "EXCLUDED",
            "Changed": False,
        },
        {
            "DesignField": "Confirmed bulk handling",
            "Part11IValue": "ADD_EXTERNALLY",
            "Part11IA2AuthoritativeValue": "ADD_EXTERNALLY",
            "Changed": False,
        },
    ]
)

# =============================================================================
# VALIDATION
# =============================================================================

expected_weekly_config_rows = len(weekly_configs) * len(locked_baseline)
expected_tuned_rows = 3 * len(locked_baseline)
expected_daily_methods = 4
expected_daily_rows = expected_daily_methods * len(locked_baseline)
selection_eligible_methods = int(
    pooled_metrics.loc[pooled_metrics["SelectionEligible"], "MethodID"].nunique()
)
actual_update_methods = pooled_metrics.loc[
    pooled_metrics["ForecastOrigin"] == "DAILY_ACTUAL_UPDATE_DIAGNOSTIC"
]

validation = pd.DataFrame(
    [
        {
            "Check": "Part 11I lock verified",
            "Expected": True,
            "Actual": True,
            "Passed": True,
        },
        {
            "Check": "Part 11IA1 lock verified",
            "Expected": True,
            "Actual": True,
            "Passed": True,
        },
        {
            "Check": "Part 11IA1 references current Part 11I lock",
            "Expected": part11i_lock_hash,
            "Actual": part11ia1_lock.get("Part11ILockSHA256"),
            "Passed": part11ia1_lock.get("Part11ILockSHA256") == part11i_lock_hash,
        },
        {
            "Check": "Part 11C lock and daily source amendment verified",
            "Expected": True,
            "Actual": True,
            "Passed": True,
        },
        {
            "Check": "Final target remains weekly normal demand",
            "Expected": TARGET_ID,
            "Actual": final_decision["TargetID"],
            "Passed": final_decision["TargetID"] == TARGET_ID,
        },
        {
            "Check": "Final scope remains strict 95 percent",
            "Expected": PRIMARY_SCOPE_ID,
            "Actual": final_decision["ScopeID"],
            "Passed": final_decision["ScopeID"] == PRIMARY_SCOPE_ID,
        },
        {
            "Check": "Bulk included in model target",
            "Expected": False,
            "Actual": final_decision["BulkIncludedInModelledTarget"],
            "Passed": final_decision["BulkIncludedInModelledTarget"] is False,
        },
        {
            "Check": "Chronological outer forecast weeks",
            "Expected": 39,
            "Actual": len(forecast_weeks),
            "Passed": len(forecast_weeks) == 39,
        },
        {
            "Check": "Weekly configurations completed",
            "Expected": len(weekly_configs),
            "Actual": int(weekly_config_execution["CompletedAllFolds"].sum()),
            "Passed": int(weekly_config_execution["CompletedAllFolds"].sum())
            == len(weekly_configs),
        },
        {
            "Check": "Weekly configuration prediction rows",
            "Expected": expected_weekly_config_rows,
            "Actual": len(weekly_config_predictions),
            "Passed": len(weekly_config_predictions) == expected_weekly_config_rows,
        },
        {
            "Check": "Prequential tuned weekly rows",
            "Expected": expected_tuned_rows,
            "Actual": len(weekly_tuned_predictions),
            "Passed": len(weekly_tuned_predictions) == expected_tuned_rows,
        },
        {
            "Check": "Daily fold executions",
            "Expected": 39,
            "Actual": len(daily_fold_audit),
            "Passed": len(daily_fold_audit) == 39,
        },
        {
            "Check": "Daily-to-weekly base prediction rows",
            "Expected": expected_daily_rows,
            "Actual": len(daily_predictions),
            "Passed": len(daily_predictions) == expected_daily_rows,
        },
        {
            "Check": "Actual-update diagnostic methods selection eligible",
            "Expected": False,
            "Actual": bool(actual_update_methods["SelectionEligible"].any()),
            "Passed": not bool(actual_update_methods["SelectionEligible"].any()),
        },
        {
            "Check": "Selection-eligible methods evaluated",
            "Expected": ">= 10",
            "Actual": selection_eligible_methods,
            "Passed": selection_eligible_methods >= 10,
        },
        {
            "Check": "Every method has 3,351 rows",
            "Expected": True,
            "Actual": bool((method_counts["Rows"] == 3351).all()),
            "Passed": bool((method_counts["Rows"] == 3351).all()),
        },
        {
            "Check": "Every method has 39 forecast weeks",
            "Expected": True,
            "Actual": bool((method_counts["Weeks"] == 39).all()),
            "Passed": bool((method_counts["Weeks"] == 39).all()),
        },
        {
            "Check": "Non-finite predictions",
            "Expected": 0,
            "Actual": int((~np.isfinite(all_predictions["Prediction"])).sum()),
            "Passed": int((~np.isfinite(all_predictions["Prediction"])).sum()) == 0,
        },
        {
            "Check": "Negative predictions",
            "Expected": 0,
            "Actual": int((all_predictions["Prediction"] < 0).sum()),
            "Passed": int((all_predictions["Prediction"] < 0).sum()) == 0,
        },
        {
            "Check": "March 2026 target rows retained or scored",
            "Expected": 0,
            "Actual": int((daily_panel["Date"] >= HOLDOUT_START).sum()),
            "Passed": int((daily_panel["Date"] >= HOLDOUT_START).sum()) == 0,
        },
        {
            "Check": "Existing Part 11I lock modified",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Existing Part 11IA1 lock modified",
            "Expected": False,
            "Actual": False,
            "Passed": True,
        },
        {
            "Check": "Final method decision produced",
            "Expected": True,
            "Actual": final_method_id in set(pooled_metrics["MethodID"]),
            "Passed": final_method_id in set(pooled_metrics["MethodID"]),
        },
        {
            "Check": "Replacement obeys all acceptance criteria",
            "Expected": True,
            "Actual": (
                True
                if not replacement_selected
                else bool(
                    final_comparison.loc[
                        final_comparison["MethodID"] == final_method_id,
                        "QualifiedToReplaceBenchmark",
                    ].iloc[0]
                )
            ),
            "Passed": (
                True
                if not replacement_selected
                else bool(
                    final_comparison.loc[
                        final_comparison["MethodID"] == final_method_id,
                        "QualifiedToReplaceBenchmark",
                    ].iloc[0]
                )
            ),
        },
    ]
)
if not validation["Passed"].all():
    raise AssertionError(
        "Part 11IA2 validation failed:\n"
        + validation.loc[~validation["Passed"]].to_string(index=False)
    )

# =============================================================================
# CONTRACT, MEMORY, CHECKPOINT, MANIFEST AND LOCK
# =============================================================================

contract = {
    "StepID": STEP_ID,
    "Status": STATUS,
    "CreatedUTC": NOW_UTC.isoformat(),
    "CreatedLocal": NOW_LOCAL.isoformat(),
    "Purpose": (
        "Perform prequential tuning, calibration, blending and a same-fold "
        "daily-to-weekly architecture comparison before confirming or amending "
        "the Part 11I weekly method."
    ),
    "LockChain": {
        "Part11ILockSHA256": part11i_lock_hash,
        "Part11IA1LockSHA256": part11ia1_lock_hash,
        "Part11CLockSHA256": part11c_lock_hash,
        "Part11CSourceAmendmentLockSHA256": part11c_amendment_hash,
    },
    "EvaluationDesign": {
        "TargetID": TARGET_ID,
        "ScopeID": PRIMARY_SCOPE_ID,
        "ForecastWeeks": len(forecast_weeks),
        "MinimumTrainingWeeks": MINIMUM_TRAINING_WEEKS,
        "ConfigurationSelection": (
            "For each outer forecast week, choose the configuration using only "
            "completed prior outer-fold predictions; use the IA1 default during "
            "the first four folds."
        ),
        "Calibration": (
            "Multiplicative actual-to-predicted ratio estimated only from prior "
            "forecast weeks and clipped to the declared range."
        ),
        "Blending": (
            "Candidate weight selected from a fixed grid using only prior "
            "forecast weeks."
        ),
        "DailyWeeklyOrigin": (
            "Fit at week start and recursively use forecasts for later daily "
            "lags inside the week."
        ),
        "DailyActualUpdateDiagnostic": (
            "Uses actual demand observed earlier inside the forecast week and "
            "is explicitly ineligible for final weekly-method selection."
        ),
    },
    "AcceptanceCriteria": final_decision["AcceptanceCriteria"],
    "FinalDecision": final_decision,
}

stage_root = EXT_ROOT / f".11IA2_staging_{uuid.uuid4().hex}"
stage_root.mkdir(parents=True, exist_ok=False)

try:
    output_frames = [
        (PACKAGE_AUDIT_PATH, package_audit),
        (WEEKLY_CONFIG_REGISTRY_PATH, weekly_config_registry),
        (WEEKLY_CONFIG_PREDICTIONS_PATH, weekly_config_predictions),
        (WEEKLY_SELECTION_AUDIT_PATH, weekly_selection_audit),
        (WEEKLY_TUNED_PREDICTIONS_PATH, weekly_tuned_predictions),
        (CALIBRATION_AUDIT_PATH, calibration_audit),
        (BLEND_AUDIT_PATH, blend_audit),
        (DAILY_SOURCE_AUDIT_PATH, daily_source_audit),
        (DAILY_FEATURE_AUDIT_PATH, daily_feature_audit),
        (DAILY_FOLD_AUDIT_PATH, daily_fold_audit),
        (DAILY_PREDICTIONS_PATH, daily_predictions),
        (ALL_PREDICTIONS_PATH, all_predictions),
        (POOLED_METRICS_PATH, pooled_metrics),
        (FOLD_METRICS_PATH, fold_metrics),
        (PAIRWISE_AUDIT_PATH, pairwise_audit),
        (FINAL_COMPARISON_PATH, final_comparison),
        (FINAL_DESIGN_TABLE_PATH, final_design_table),
        (VALIDATION_PATH, validation),
    ]
    for path, frame in output_frames:
        stage_csv(stage_root, path, frame)
    stage_json(stage_root, FINAL_DECISION_PATH, final_decision)
    stage_json(stage_root, FINAL_DESIGN_AMENDMENT_PATH, final_design_amendment)
    stage_json(stage_root, CONTRACT_PATH, contract)

    decision_body = "\n".join(
        [
            f"- Final authoritative target: `{TARGET_ID}`.",
            f"- Final authoritative scope: `{PRIMARY_SCOPE_ID}`.",
            f"- Previous Part 11I method: `{LOCKED_BENCHMARK_METHOD}`.",
            f"- Part 11IA2 authoritative method: `{final_method_id}`.",
            f"- Method replacement selected: `{replacement_selected}`.",
            f"- Decision reason: {final_reason}",
            "- Bulk remains excluded from the modelled target.",
            "- Unknown bulk remains zero and confirmed preorders are added externally.",
            "- Daily actual-update results are diagnostic only and cannot support a Monday-origin weekly purchasing claim.",
            "- A new untouched future period remains required for final unbiased evaluation.",
        ]
    )
    results_body = "\n".join(
        [
            f"- Final method WAPE: `{float(final_metric_row['WAPEPercentage']):.6f}%`.",
            f"- Final method MAE: `{float(final_metric_row['MAE']):.6f}`.",
            f"- Final method RMSE: `{float(final_metric_row['RMSE']):.6f}`.",
            f"- Final method total bias: `{float(final_metric_row['TotalBias']):.6f}`.",
            f"- Final method absolute bias percentage: `{float(final_metric_row['AbsoluteBiasPercentage']):.6f}%`.",
            f"- WAPE improvement over Part 11I: `{float(final_metric_row['WAPEImprovementPercentagePoints']):.6f}` percentage points.",
            f"- Pairwise weeks won against Part 11I: `{int(final_metric_row['PairwiseWeeksWon'])}` of 39.",
        ]
    )
    files_body = "\n".join(
        [
            f"- Final comparison: `{FINAL_COMPARISON_PATH}`",
            f"- Final decision: `{FINAL_DECISION_PATH}`",
            f"- Final design amendment: `{FINAL_DESIGN_AMENDMENT_PATH}`",
            f"- Weekly tuning audit: `{WEEKLY_SELECTION_AUDIT_PATH}`",
            f"- Calibration audit: `{CALIBRATION_AUDIT_PATH}`",
            f"- Blend audit: `{BLEND_AUDIT_PATH}`",
            f"- Daily-to-weekly predictions: `{DAILY_PREDICTIONS_PATH}`",
            f"- Validation: `{VALIDATION_PATH}`",
        ]
    )

    for key, final_path in MEMORY_FILES.items():
        original = final_path.read_text(encoding="utf-8")
        if key == "PROJECT_CONTEXT":
            updated = update_section(
                original,
                "STEP_11IA2",
                "Part 11IA2 tuned and daily-to-weekly challenge",
                (
                    "Part 11IA2 completed the expanded weekly modelling challenge "
                    "with prequential tuning, calibration, blending and a true "
                    "week-start daily-to-weekly reconstruction. "
                    f"The authoritative method is `{final_method_id}`."
                ),
            )
        elif key == "WORKFLOW":
            updated = update_section(
                original,
                "STEP_11IA2",
                "Part 11IA2 workflow status",
                (
                    "The extended modelling challenge is complete. The target, "
                    "scope and bulk policy remain locked. The final method decision "
                    "has been recorded in a separate amendment lock. Part 11J may "
                    "now assess ingredient-planning readiness."
                ),
            )
        elif key == "DECISIONS":
            updated = update_section(
                original,
                "STEP_11IA2",
                "Part 11IA2 decisions",
                decision_body,
            )
        elif key == "FILES_AND_PATHS":
            updated = update_section(
                original,
                "STEP_11IA2",
                "Part 11IA2 files",
                files_body,
            )
        elif key == "METRICS_AND_RESULTS":
            updated = update_section(
                original,
                "STEP_11IA2",
                "Part 11IA2 metrics and results",
                results_body,
            )
        elif key == "CHAT_INDEX":
            row = (
                f"| {NOW_LOCAL.isoformat()} | 11IA2 | Tuned, calibrated and "
                f"daily-to-weekly challenge | {STATUS} |"
            )
            updated = original if row in original else original.rstrip() + "\n" + row + "\n"
        elif key == "CURRENT_HANDOFF":
            updated = "\n".join(
                [
                    "# Current Handoff",
                    "",
                    f"- Current completed step: {STEP_ID}",
                    f"- Status: {STATUS}",
                    f"- Updated local time: {NOW_LOCAL.isoformat()}",
                    f"- Final target: {TARGET_ID}",
                    f"- Final authoritative method: {final_method_id}",
                    f"- Final scope: {PRIMARY_SCOPE_ID}",
                    f"- Method replacement selected: {replacement_selected}",
                    "- Bulk policy: forecast normal demand only; add confirmed bulk externally.",
                    f"- Final amendment: {FINAL_DESIGN_AMENDMENT_PATH}",
                    "- Next step: 11J ingredient-planning readiness.",
                    "",
                ]
            )
        else:
            raise KeyError(key)
        stage_text(stage_root, final_path, updated)

    candidate_summary_text = final_comparison[
        [
            "FinalComparisonRank",
            "MethodID",
            "SelectionEligible",
            "ForecastOrigin",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "AbsoluteBiasPercentage",
            "PairwiseWeeksWon",
            "QualifiedToReplaceBenchmark",
        ]
    ].to_string(index=False)

    step_text = f"""# Step 11IA2 — Tuned, Calibrated and Daily-to-Weekly Challenge

- **Step ID:** {STEP_ID}
- **Date and time:** {NOW_LOCAL.isoformat()}
- **Status:** {STATUS}

## Purpose
Complete the extended weekly modelling challenge before ingredient planning by applying prequential configuration selection, leakage-safe bias calibration, adaptive baseline blending and a true week-start daily-to-weekly reconstruction on the same 39 chronological folds and dynamic 95% scope.

## Final decision
{decision_body}

## Final result
{results_body}

## Candidate comparison
```text
{candidate_summary_text}
```

## Forecast-origin interpretation
The week-start daily-to-weekly methods recursively use their own predictions for later daily lag features and are selection eligible. The actual-update diagnostic uses actual demand observed earlier in the same week; it demonstrates the value of daily information updates but is not a valid Monday-origin weekly purchasing forecast.

## Outputs
{files_body}
"""
    stage_text(stage_root, STEP_MEMORY_PATH, step_text)

    checkpoint = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "CreatedLocal": NOW_LOCAL.isoformat(),
        "InputLocks": {
            "Part11ILockSHA256": part11i_lock_hash,
            "Part11IA1LockSHA256": part11ia1_lock_hash,
            "Part11CLockSHA256": part11c_lock_hash,
            "Part11CSourceAmendmentLockSHA256": part11c_amendment_hash,
        },
        "FinalDecision": final_decision,
        "Safety": {
            "ExistingPart11ILockModified": False,
            "ExistingPart11IA1LockModified": False,
            "March2026TargetsRetainedOrUsedForSelection": False,
            "BulkIncludedInModelTarget": False,
            "DailyActualUpdateDiagnosticSelectionEligible": False,
            "ForecastsRoundedBeforeScoring": False,
            "FinalMethodDecisionSelected": True,
        },
        "ReadyForPart11J": True,
        "NextStep": "11J",
    }
    stage_json(stage_root, CHECKPOINT_PATH, checkpoint)
    checkpoint_hash = sha256_file(stage_path(stage_root, CHECKPOINT_PATH))
    stage_text(
        stage_root,
        CHECKPOINT_SHA_PATH,
        f"{checkpoint_hash}  {CHECKPOINT_PATH.name}\n",
    )

    manifest_targets = [
        *[path for path, _ in output_frames],
        FINAL_DECISION_PATH,
        FINAL_DESIGN_AMENDMENT_PATH,
        CONTRACT_PATH,
        STEP_MEMORY_PATH,
        CHECKPOINT_PATH,
        CHECKPOINT_SHA_PATH,
        *MEMORY_FILES.values(),
    ]
    manifest = pd.DataFrame(
        [
            {
                "RelativePath": str(path.relative_to(EXT_ROOT)),
                "Bytes": stage_path(stage_root, path).stat().st_size,
                "SHA256": sha256_file(stage_path(stage_root, path)),
            }
            for path in manifest_targets
        ]
    ).sort_values("RelativePath")
    stage_csv(stage_root, MANIFEST_PATH, manifest)
    manifest_hash = sha256_file(stage_path(stage_root, MANIFEST_PATH))

    lock = {
        "StepID": STEP_ID,
        "Status": STATUS,
        "CreatedUTC": NOW_UTC.isoformat(),
        "LockChain": checkpoint["InputLocks"],
        "FinalDecision": {
            "Path": str(FINAL_DECISION_PATH),
            "SHA256": sha256_file(stage_path(stage_root, FINAL_DECISION_PATH)),
            "FinalAuthoritativeMethodID": final_method_id,
            "ReplacementSelected": replacement_selected,
        },
        "FinalDesignAmendment": {
            "Path": str(FINAL_DESIGN_AMENDMENT_PATH),
            "SHA256": sha256_file(
                stage_path(stage_root, FINAL_DESIGN_AMENDMENT_PATH)
            ),
        },
        "OutputHashManifest": {
            "Path": str(MANIFEST_PATH),
            "SHA256": manifest_hash,
            "FilesListed": int(len(manifest)),
        },
        "Checkpoint": {
            "Path": str(CHECKPOINT_PATH),
            "SHA256": checkpoint_hash,
        },
        "SafetyAssertions": checkpoint["Safety"],
        "ReadyForPart11J": True,
        "NextStep": "11J",
    }
    stage_json(stage_root, LOCK_PATH, lock)
    lock_hash = sha256_file(stage_path(stage_root, LOCK_PATH))
    stage_text(stage_root, LOCK_SHA_PATH, f"{lock_hash}  {LOCK_PATH.name}\n")

    log_text = "\n".join(
        [
            f"Status: {STATUS}",
            f"Run local: {NOW_LOCAL.isoformat()}",
            f"Previous method: {LOCKED_BENCHMARK_METHOD}",
            f"Final authoritative method: {final_method_id}",
            f"Replacement selected: {replacement_selected}",
            f"Final WAPE: {float(final_metric_row['WAPEPercentage']):.12f}",
            f"Final MAE: {float(final_metric_row['MAE']):.12f}",
            f"Final RMSE: {float(final_metric_row['RMSE']):.12f}",
            f"Final total bias: {float(final_metric_row['TotalBias']):.12f}",
            f"Checkpoint SHA256: {checkpoint_hash}",
            f"Lock SHA256: {lock_hash}",
            "",
        ]
    )
    stage_text(stage_root, LOG_PATH, log_text)

    staged_files = [path for path in stage_root.rglob("*") if path.is_file()]
    if not staged_files:
        raise AssertionError("Part 11IA2 staging directory is empty")
    for staged in sorted(staged_files):
        final = EXT_ROOT / staged.relative_to(stage_root)
        final.parent.mkdir(parents=True, exist_ok=True)
        if final.exists() and final not in MEMORY_FILES.values():
            raise FileExistsError(f"Refusing to overwrite Part 11IA2 output: {final}")
        os.replace(staged, final)
    for path in NEW_OUTPUTS:
        make_read_only(path)
finally:
    if stage_root.exists():
        shutil.rmtree(stage_root)

# =============================================================================
# FINAL TECHNICAL OUTPUT
# =============================================================================

print("=" * 100)
print("EDEN WEEKLY FORECASTING EXTENSION — PART 11IA2 COMPLETE")
print("=" * 100)
print(f"Status: {STATUS}")
print(f"Extension root: {EXT_ROOT}")
print(f"Local time: {NOW_LOCAL.isoformat()}")

print("\nPACKAGE AVAILABILITY")
print(
    package_audit[
        ["Distribution", "Version", "Installed", "ImportSucceeded", "ImportError"]
    ].to_string(index=False)
)

print("\nWEEKLY PREQUENTIAL CONFIGURATION SELECTION SUMMARY")
print(
    weekly_selection_audit.groupby(
        ["Family", "SelectedConfigurationID"], as_index=False
    ).agg(WeeksSelected=("ForecastWeek", "count")).sort_values(
        ["Family", "WeeksSelected"], ascending=[True, False]
    ).to_string(index=False)
)

print("\nFINAL PRIMARY-SCOPE METHOD RANKING")
print(
    final_comparison[
        [
            "FinalComparisonRank",
            "MethodID",
            "SelectionEligible",
            "ForecastOrigin",
            "WAPEPercentage",
            "WAPEImprovementPercentagePoints",
            "MAE",
            "RMSE",
            "TotalBias",
            "AbsoluteBiasPercentage",
            "PairwiseWeeksWon",
            "QualifiedToReplaceBenchmark",
        ]
    ].to_string(index=False)
)

print("\nDAILY FORECAST-ORIGIN COMPARISON")
print(
    pooled_metrics.loc[
        pooled_metrics["MethodFamily"] == "DAILY_TO_WEEKLY",
        [
            "MethodID",
            "SelectionEligible",
            "ForecastOrigin",
            "WAPEPercentage",
            "MAE",
            "RMSE",
            "TotalBias",
            "AbsoluteBiasPercentage",
        ],
    ].sort_values("WAPEPercentage").to_string(index=False)
)

print("\nPART 11IA2 FINAL DECISION")
print(f"Previous Part 11I method: {LOCKED_BENCHMARK_METHOD}")
print(f"Final authoritative method: {final_method_id}")
print(f"Replacement selected: {replacement_selected}")
print(f"Decision reason: {final_reason}")
print(f"Final WAPE: {float(final_metric_row['WAPEPercentage']):.6f}%")
print(f"Final MAE: {float(final_metric_row['MAE']):.6f}")
print(f"Final RMSE: {float(final_metric_row['RMSE']):.6f}")
print(f"Final total bias: {float(final_metric_row['TotalBias']):.6f}")
print(
    "WAPE improvement over Part 11I: "
    f"{float(final_metric_row['WAPEImprovementPercentagePoints']):.6f} percentage points"
)
print(
    "Pairwise forecast weeks won against Part 11I: "
    f"{int(final_metric_row['PairwiseWeeksWon'])}/39"
)
print("Target: WEEKLY_NORMAL_DEMAND")
print("Scope: FOLD_STRICT_95_PERCENT")
print("Bulk included in modelled target: False")
print("Confirmed bulk handling: add externally")

print("\nPART 11IA2 VALIDATION")
print(validation.to_string(index=False))

print("\nCONTROL OUTPUTS")
print(f"- Final comparison: {FINAL_COMPARISON_PATH}")
print(f"- Final decision: {FINAL_DECISION_PATH}")
print(f"- Final design amendment: {FINAL_DESIGN_AMENDMENT_PATH}")
print(f"- Weekly tuning audit: {WEEKLY_SELECTION_AUDIT_PATH}")
print(f"- Daily-to-weekly predictions: {DAILY_PREDICTIONS_PATH}")
print(f"- Checkpoint: {CHECKPOINT_PATH}")
print(f"- Checkpoint SHA256: {checkpoint_hash}")
print(f"- Part 11IA2 lock: {LOCK_PATH}")
print(f"- Part 11IA2 lock SHA256: {lock_hash}")

print(
    "\nSAFETY: existing Part 11I lock modified False; existing Part 11IA1 lock "
    "modified False; March 2026 target rows retained or used for selection False; "
    "bulk included in model target False; daily actual-update diagnostics selection "
    "eligible False; forecasts rounded before scoring False."
)
print("=" * 100)

----------------------------------------------------------------------------------------------------
Running weekly configuration: XGBOOST_POISSON / XGB_P1_BASE
----------------------------------------------------------------------------------------------------
Running weekly configuration: XGBOOST_POISSON / XGB_P2_SHALLOW
----------------------------------------------------------------------------------------------------
Running weekly configuration: XGBOOST_POISSON / XGB_P3_REGULARISED
----------------------------------------------------------------------------------------------------
Running weekly configuration: XGBOOST_POISSON / XGB_P4_DEEP
----------------------------------------------------------------------------------------------------
Running weekly configuration: RANDOM_FOREST_POISSON / RF_P1_BASE
----------------------------------------------------------------------------------------------------
Running weekly configuration: RANDOM_FOREST_POISSON / RF_P2_FINE_LEAVES
-------

/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_59566/3973846732.py:1684: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  or pd.api.types.is_categorical_dtype(daily_panel[column])


----------------------------------------------------------------------------------------------------
Running daily-to-weekly fold 1/39: 2025-05-26
----------------------------------------------------------------------------------------------------
Running daily-to-weekly fold 2/39: 2025-06-02
----------------------------------------------------------------------------------------------------
Running daily-to-weekly fold 3/39: 2025-06-09
----------------------------------------------------------------------------------------------------
Running daily-to-weekly fold 4/39: 2025-06-16
----------------------------------------------------------------------------------------------------
Running daily-to-weekly fold 5/39: 2025-06-23
----------------------------------------------------------------------------------------------------
Running daily-to-weekly fold 6/39: 2025-06-30
----------------------------------------------------------------------------------------------------
Running daily-to-